
# NLP Analysis of Diplomatic Discourse

## 1. Baseline Lexical Similarity

### Research objective

Quantify the annual lexical similarity among Iran, the United States, and China in United Nations General Debate speeches from 1946 to 2025.

### Method

A common TF-IDF feature space is constructed from all 233 speeches, using unigrams and bigrams.

Cosine similarity is calculated between speeches delivered in the same calendar year.

The three comparisons are:

- Iran–United States
- China–United States
- Iran–China

### Interpretation

TF-IDF cosine distance measures differences in weighted lexical content. It does not directly measure diplomatic hostility, disagreement, political alignment, or rhetorical framing.

The analysis provides an initial measurement baseline for comparison with topic-based and semantic methods.

### Methodological safeguards

- All source speeches are preserved.
- Missing country-year pairs remain missing.
- All countries share the same fitted vocabulary and IDF weights.
- No country-specific vocabulary is fitted.
- Unigrams and bigrams are retained for exploratory lexical measurement.
- No statistical significance or causal interpretation is assigned to the resulting distances.

Model specifications and preprocessing choices will be evaluated through sensitivity analyses before substantive conclusions are drawn.


In [ ]:
# ============================================================
# STEP 0 — CENTRALIZED LIBRARY IMPORTS
# ============================================================

# 1. Standard Python Libraries
# ------------------------------------------------------------
import re
import textwrap
import warnings

from pathlib import Path
from collections import Counter
from itertools import combinations


# 2. Numerical Computing & Data Manipulation
# ------------------------------------------------------------
import numpy as np
import pandas as pd


# 3. Statistical Analysis & Hypothesis Testing
# ------------------------------------------------------------
import statsmodels.api as sm

from scipy.stats import norm, spearmanr
from scipy.optimize import linear_sum_assignment
from scipy.spatial.distance import jensenshannon
from scipy.sparse import csr_matrix
from scipy import sparse
from statsmodels.stats.multitest import multipletests



# 4. Machine Learning & Topic Modeling
# ------------------------------------------------------------
from sklearn.decomposition import NMF
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity
from sklearn.preprocessing import normalize
from sklearn.exceptions import ConvergenceWarning


# 5. Natural Language Processing (NLP)
# ------------------------------------------------------------
import spacy


# 6. Data Visualization & Image Processing
# ------------------------------------------------------------
import matplotlib.pyplot as plt

from matplotlib.lines import Line2D
from PIL import Image


# 7. Jupyter Notebook Display Utilities
# ------------------------------------------------------------
from IPython.display import display, Markdown


# ============================================================
# END OF CENTRALIZED LIBRARY IMPORTS
# ============================================================

In [ ]:

# ============================================================
# NOTEBOOK 03: NLP ANALYSIS
# STEP 01: BASELINE TF-IDF LEXICAL SIMILARITY
# ============================================================


# ------------------------------------------------------------
# 1. Load the validated analytical corpus
# ------------------------------------------------------------

PROJECT_DIR = Path.home() / "Projects" / "7-masoud"

DATA_PATH = (
    PROJECT_DIR / "data" / "processed"
    / "speeches_master.parquet"
)

df_nlp = pd.read_parquet(DATA_PATH)

df_nlp = df_nlp.sort_values(
    ["year", "country_code"]
).reset_index(drop=True)

assert len(df_nlp) == 233
assert df_nlp["archive_path"].is_unique
assert df_nlp["text_clean"].notna().all()

# ------------------------------------------------------------
# 2. Configure a common TF-IDF feature space
# ------------------------------------------------------------

tfidf_vectorizer = TfidfVectorizer(
    lowercase=True,
    strip_accents=None,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=3,
    max_df=0.95,
    sublinear_tf=True,
    norm="l2",
    dtype=np.float64
)

tfidf_matrix = tfidf_vectorizer.fit_transform(
    df_nlp["text_clean"]
)

# ------------------------------------------------------------
# 3. Validate the document-feature matrix
# ------------------------------------------------------------

assert tfidf_matrix.shape[0] == len(df_nlp)
assert tfidf_matrix.shape[1] > 0

document_norms = np.sqrt(
    tfidf_matrix.multiply(tfidf_matrix)
    .sum(axis=1)
).A1

assert np.allclose(document_norms, 1.0)

print("=" * 70)
print("BASELINE TF-IDF FEATURE SPACE")
print("=" * 70)

print("Documents:", tfidf_matrix.shape[0])
print("Vocabulary features:", tfidf_matrix.shape[1])
print("Nonzero feature values:", tfidf_matrix.nnz)

# ------------------------------------------------------------
# 4. Define matched-year country pairs
# ------------------------------------------------------------

PAIRS = [
    ("IRN", "USA"),
    ("CHN", "USA"),
    ("IRN", "CHN")
]

document_lookup = {
    (row.country_code, int(row.year)): index
    for index, row in df_nlp.iterrows()
}

assert len(document_lookup) == len(df_nlp)

distance_records = []

for first, second in PAIRS:

    common_years = sorted(
        set(
            df_nlp.loc[
                df_nlp["country_code"] == first,
                "year"
            ]
        ).intersection(
            df_nlp.loc[
                df_nlp["country_code"] == second,
                "year"
            ]
        )
    )

    for year in common_years:

        first_index = document_lookup[(first, year)]
        second_index = document_lookup[(second, year)]

        similarity = float(
            cosine_similarity(
                tfidf_matrix[first_index],
                tfidf_matrix[second_index]
            )[0, 0]
        )

        distance_records.append({
            "country_1": first,
            "country_2": second,
            "year": int(year),
            "cosine_similarity": similarity,
            "cosine_distance": 1.0 - similarity
        })

tfidf_distances = pd.DataFrame(distance_records)

# ------------------------------------------------------------
# 5. Validate the paired comparisons
# ------------------------------------------------------------

expected_pairs = {
    ("IRN", "USA"): 74,
    ("CHN", "USA"): 79,
    ("IRN", "CHN"): 73
}

for pair, expected_n in expected_pairs.items():

    actual_n = len(
        tfidf_distances.loc[
            (tfidf_distances["country_1"] == pair[0])
            & (tfidf_distances["country_2"] == pair[1])
        ]
    )

    assert actual_n == expected_n

assert tfidf_distances["cosine_similarity"].between(
    -1e-10, 1 + 1e-10
).all()

assert tfidf_distances["cosine_distance"].between(
    -1e-10, 1 + 1e-10
).all()

# ------------------------------------------------------------
# 6. Summarize the baseline distances
# ------------------------------------------------------------

distance_summary = (
    tfidf_distances
    .groupby(["country_1", "country_2"])
    .agg(
        n_years=("year", "size"),
        mean_distance=("cosine_distance", "mean"),
        median_distance=("cosine_distance", "median"),
        sd_distance=("cosine_distance", "std"),
        min_distance=("cosine_distance", "min"),
        max_distance=("cosine_distance", "max")
    )
    .reset_index()
)

print("\nMatched-year lexical distance summary:")
display(distance_summary.round(4))

print("\nSample annual distances:")
display(tfidf_distances.head(12).round(4))

print("\nBASELINE TF-IDF VALIDATION: PASSED")



## 2. Temporal Dynamics of Lexical Distance

### Objective

Examine the historical evolution of TF-IDF cosine distance among Iran, the United States, and China.

### Analytical procedures

1. Identify calendar years with complete three-country coverage.
2. Calculate comparable pairwise distance summaries over identical years.
3. Construct annual distance series for all three country pairs.
4. Calculate centered five-year rolling averages for descriptive visualization.
5. Preserve missing country-year comparisons without interpolation.

### Statistical considerations

Pairwise cosine distances share speech documents and are not statistically independent.

Temporal autocorrelation and potential measurement drift must be considered before formal inference.

Rolling averages are descriptive and are not evidence of structural breaks.

The TF-IDF feature space was fitted to the complete retrospective corpus. Results are therefore conditional on the selected corpus, vocabulary, and weighting configuration.

No causal or geopolitical interpretation is assigned to the observed distance patterns at this stage.


In [ ]:

# ============================================================
# STEP 02: COMMON-YEAR LEXICAL DISTANCE COMPARISON
# ============================================================

PAIR_LABELS = {
    ("IRN", "USA"): "Iran–United States",
    ("CHN", "USA"): "China–United States",
    ("IRN", "CHN"): "Iran–China"
}

# ------------------------------------------------------------
# 1. Identify complete three-country years
# ------------------------------------------------------------

country_year_matrix = (
    df_nlp.assign(present=True)
    .pivot(
        index="year",
        columns="country_code",
        values="present"
    )
    .reindex(
        index=range(1946, 2026),
        columns=["IRN", "USA", "CHN"]
    )
    .fillna(False)
    .astype(bool)
)

common_years = country_year_matrix.index[
    country_year_matrix.all(axis=1)
].tolist()

assert len(common_years) == 73

# ------------------------------------------------------------
# 2. Restrict all comparisons to identical years
# ------------------------------------------------------------

matched_distances = tfidf_distances.loc[
    tfidf_distances["year"].isin(common_years)
].copy()

matched_distances["pair"] = [
    PAIR_LABELS[(a, b)]
    for a, b in zip(
        matched_distances["country_1"],
        matched_distances["country_2"]
    )
]

assert len(matched_distances) == 73 * 3

assert (
    matched_distances.groupby("pair")["year"]
    .nunique()
    .eq(73)
    .all()
)

# ------------------------------------------------------------
# 3. Create a paired year-level matrix
# ------------------------------------------------------------

distance_wide = matched_distances.pivot(
    index="year",
    columns="pair",
    values="cosine_distance"
)

assert distance_wide.shape == (73, 3)
assert distance_wide.notna().all().all()

# ------------------------------------------------------------
# 4. Calculate comparable descriptive statistics
# ------------------------------------------------------------

common_year_summary = (
    matched_distances
    .groupby("pair")
    .agg(
        n_years=("year", "size"),
        mean_distance=("cosine_distance", "mean"),
        median_distance=("cosine_distance", "median"),
        sd_distance=("cosine_distance", "std"),
        min_distance=("cosine_distance", "min"),
        max_distance=("cosine_distance", "max")
    )
    .reset_index()
)

print("=" * 70)
print("COMMON-YEAR LEXICAL DISTANCE COMPARISON")
print("=" * 70)

print(f"Complete three-country years: {len(common_years)}")
print(f"First common year: {min(common_years)}")
print(f"Last common year: {max(common_years)}")

display(common_year_summary.round(4))

print("\nPaired annual distance correlations:")
display(distance_wide.corr().round(3))

print("\nCommon-year comparison: PASSED")


In [ ]:

# ============================================================
# STEP 02.1: ANNUAL LEXICAL DISTANCE SERIES
# ============================================================

pair_order = [
    "Iran–United States",
    "China–United States",
    "Iran–China"
]

annual_distances = tfidf_distances.copy()

annual_distances["pair"] = [
    PAIR_LABELS[(a, b)]
    for a, b in zip(
        annual_distances["country_1"],
        annual_distances["country_2"]
    )
]

calendar = pd.MultiIndex.from_product(
    [pair_order, range(1946, 2026)],
    names=["pair", "year"]
)

distance_temporal = (
    annual_distances
    .set_index(["pair", "year"])
    .reindex(calendar)
    .reset_index()
    .sort_values(["pair", "year"])
)

distance_temporal["rolling_mean_5y"] = (
    distance_temporal
    .groupby("pair")["cosine_distance"]
    .transform(
        lambda x: x.rolling(
            window=5,
            center=True,
            min_periods=3
        ).mean()
    )
)

distance_temporal["observed"] = (
    distance_temporal["cosine_distance"].notna()
)

assert len(distance_temporal) == 240
assert distance_temporal["observed"].sum() == 226

print("=" * 70)
print("ANNUAL LEXICAL DISTANCE PREPARATION")
print("=" * 70)

display(
    distance_temporal.groupby("pair")
    .agg(
        observed_years=("observed", "sum"),
        smoothed_years=("rolling_mean_5y", "count")
    )
)

print("\nTemporal distance preparation: PASSED")


In [ ]:

# ============================================================
# REVISED VISUALIZATION: ANNUAL TF-IDF LEXICAL DISTANCE
# Publication-quality layout with external legend
# ============================================================


fig_distance_final, axes = plt.subplots(
    nrows=3,
    ncols=1,
    figsize=(12, 9),
    sharex=True,
    sharey=True,
    constrained_layout=False
)

for ax, pair in zip(axes, pair_order):

    subset = distance_temporal.loc[
        distance_temporal["pair"] == pair
    ].sort_values("year")

    observed = subset["observed"]

    ax.scatter(
        subset.loc[observed, "year"],
        subset.loc[observed, "cosine_distance"],
        s=24,
        alpha=0.5,
        color="#8C8C8C",
        edgecolors="none",
        zorder=2
    )

    smoothed = subset["rolling_mean_5y"].where(
        observed
    )

    ax.plot(
        subset["year"],
        smoothed,
        linewidth=2.2,
        color="#1B1B1B",
        zorder=3
    )

    ax.set_title(
        pair,
        loc="left",
        fontsize=12,
        fontweight="bold",
        pad=10
    )

    ax.set_ylabel("Cosine distance", fontsize=10)
    ax.grid(alpha=0.15, linewidth=0.7)
    ax.set_axisbelow(True)
    ax.tick_params(labelsize=9)

axes[-1].set_xlabel("Year", fontsize=11)
axes[-1].set_xlim(1945, 2026)

legend_handles = [
    Line2D(
        [0], [0],
        marker="o",
        linestyle="None",
        color="#8C8C8C",
        markersize=6,
        label="Annual observation"
    ),
    Line2D(
        [0], [0],
        color="#1B1B1B",
        linewidth=2.2,
        label="Centered 5-year rolling mean"
    )
]

fig_distance_final.legend(
    handles=legend_handles,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.945),
    ncol=2,
    frameon=False,
    fontsize=10
)

fig_distance_final.suptitle(
    "Historical Variation in TF-IDF Lexical Distance",
    fontsize=15,
    fontweight="bold",
    y=0.985
)

fig_distance_final.subplots_adjust(
    left=0.09,
    right=0.98,
    bottom=0.07,
    top=0.88,
    hspace=0.25
)

plt.show()



## 3. TF-IDF Specification Sensitivity Analysis

### Objective

Evaluate the robustness of annual lexical distance estimates to alternative TF-IDF feature specifications.

### Model specifications

- **M0 — Baseline:** Unigrams and bigrams, English stopword removal, minimum document frequency of 3, maximum document frequency of 0.95, and sublinear term frequency.
- **M1 — Unigrams only:** Same as the baseline, using unigrams only.
- **M2 — No stopword removal:** Retain English function words and other common vocabulary.
- **M3 — Broader vocabulary:** Reduce the minimum document frequency to 2.
- **M4 — Linear term frequency:** Disable sublinear term-frequency scaling.

### Evaluation criteria

1. Compare mean lexical distances over the same 73 complete three-country years.
2. Assess Spearman rank correlations between annual baseline and alternative distance estimates.
3. Calculate mean absolute and maximum absolute deviations from the baseline.
4. Compare the descriptive ordering of country pairs across model specifications.

### Statistical considerations

The same corpus is used for all model specifications. These analyses evaluate measurement sensitivity rather than out-of-sample predictive performance.

Each model is fitted independently, so absolute cosine distances are conditional on its feature space.

Rank correlations assess the stability of temporal ordering, not the equivalence of distance magnitudes.

Sensitivity analyses do not establish causal effects, political alignment, or statistical significance.

No source documents are removed or modified.


In [ ]:

# ============================================================
# STEP 03: TF-IDF SPECIFICATION SENSITIVITY ANALYSIS
# ============================================================


# ------------------------------------------------------------
# 1. Define model specifications
# ------------------------------------------------------------

MODEL_CONFIGS = {
    "M0_baseline": {
        "ngram_range": (1, 2),
        "stop_words": "english",
        "min_df": 3,
        "max_df": 0.95,
        "sublinear_tf": True
    },
    "M1_unigrams": {
        "ngram_range": (1, 1),
        "stop_words": "english",
        "min_df": 3,
        "max_df": 0.95,
        "sublinear_tf": True
    },
    "M2_no_stopwords": {
        "ngram_range": (1, 2),
        "stop_words": None,
        "min_df": 3,
        "max_df": 0.95,
        "sublinear_tf": True
    },
    "M3_broad_vocabulary": {
        "ngram_range": (1, 2),
        "stop_words": "english",
        "min_df": 2,
        "max_df": 0.95,
        "sublinear_tf": True
    },
    "M4_linear_tf": {
        "ngram_range": (1, 2),
        "stop_words": "english",
        "min_df": 3,
        "max_df": 0.95,
        "sublinear_tf": False
    }
}

# ------------------------------------------------------------
# 2. Establish the common-year comparison keys
# ------------------------------------------------------------

COMMON_PAIRS = [
    ("IRN", "USA"),
    ("CHN", "USA"),
    ("IRN", "CHN")
]

comparison_keys = [
    (first, second, int(year))
    for first, second in COMMON_PAIRS
    for year in common_years
]

assert len(comparison_keys) == 219
assert len(set(comparison_keys)) == 219

first_indices = np.array([
    document_lookup[(first, year)]
    for first, second, year in comparison_keys
])

second_indices = np.array([
    document_lookup[(second, year)]
    for first, second, year in comparison_keys
])

# ------------------------------------------------------------
# 3. Fit each model and compute matched-year distances
# ------------------------------------------------------------

model_records = []
feature_records = []

for model_name, config in MODEL_CONFIGS.items():

    vectorizer = TfidfVectorizer(
        lowercase=True,
        strip_accents=None,
        norm="l2",
        dtype=np.float64,
        **config
    )

    X = vectorizer.fit_transform(
        df_nlp["text_clean"]
    )

    # Sparse row-wise dot products of normalized vectors.
    similarity = np.asarray(
        X[first_indices]
        .multiply(X[second_indices])
        .sum(axis=1)
    ).ravel()

    distance = 1.0 - similarity

    assert len(distance) == 219
    assert np.isfinite(distance).all()
    assert np.all(distance >= -1e-10)
    assert np.all(distance <= 1 + 1e-10)

    for key, value in zip(comparison_keys, distance):

        first, second, year = key

        model_records.append({
            "model": model_name,
            "country_1": first,
            "country_2": second,
            "year": year,
            "cosine_distance": float(value)
        })

    feature_records.append({
        "model": model_name,
        "n_features": X.shape[1],
        "nonzero_values": X.nnz
    })

sensitivity_df = pd.DataFrame(model_records)
feature_summary = pd.DataFrame(feature_records)

# ------------------------------------------------------------
# 4. Validate baseline reproducibility
# ------------------------------------------------------------

baseline_original = (
    matched_distances
    .set_index(["country_1", "country_2", "year"])
    ["cosine_distance"]
    .sort_index()
)

baseline_recomputed = (
    sensitivity_df.loc[
        sensitivity_df["model"] == "M0_baseline"
    ]
    .set_index(["country_1", "country_2", "year"])
    ["cosine_distance"]
    .sort_index()
)

assert baseline_original.index.equals(
    baseline_recomputed.index
)

assert np.allclose(
    baseline_original.to_numpy(),
    baseline_recomputed.to_numpy(),
    atol=1e-10,
    rtol=1e-10
), "Baseline model reproducibility failed."

# ------------------------------------------------------------
# 5. Summarize distances across specifications
# ------------------------------------------------------------

model_distance_summary = (
    sensitivity_df
    .groupby(["model", "country_1", "country_2"])
    .agg(
        n_years=("year", "size"),
        mean_distance=("cosine_distance", "mean"),
        sd_distance=("cosine_distance", "std"),
        median_distance=("cosine_distance", "median")
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Report outputs
# ------------------------------------------------------------

print("=" * 70)
print("TF-IDF SPECIFICATION SENSITIVITY")
print("=" * 70)

print("\nModel feature spaces:")
display(feature_summary)

print("\nMatched-year distance summary:")
display(model_distance_summary.round(4))

print("\nBaseline reproducibility: PASSED")
print("Specification sensitivity calculations: PASSED")


In [ ]:

# ============================================================
# STEP 03.1: TF-IDF DISTANCE STABILITY DIAGNOSTICS
# ============================================================

# ------------------------------------------------------------
# 1. Create an aligned model comparison matrix
# ------------------------------------------------------------

comparison_matrix = sensitivity_df.pivot(
    index=["country_1", "country_2", "year"],
    columns="model",
    values="cosine_distance"
).sort_index()

assert comparison_matrix.shape == (219, 5)
assert comparison_matrix.notna().all().all()

# ------------------------------------------------------------
# 2. Compare annual distances with the baseline
# ------------------------------------------------------------

stability_records = []

for first, second in COMMON_PAIRS:

    pair_data = comparison_matrix.loc[
        (first, second)
    ]

    reference = pair_data["M0_baseline"].to_numpy()

    for model_name in MODEL_CONFIGS:

        alternative = pair_data[model_name].to_numpy()

        rho, _ = spearmanr(
            reference,
            alternative
        )

        absolute_difference = np.abs(
            reference - alternative
        )

        stability_records.append({
            "pair": f"{first}-{second}",
            "model": model_name,
            "spearman_rho": float(rho),
            "mean_absolute_difference": float(
                absolute_difference.mean()
            ),
            "max_absolute_difference": float(
                absolute_difference.max()
            )
        })

stability_summary = pd.DataFrame(
    stability_records
)

# ------------------------------------------------------------
# 3. Compare descriptive rankings of country pairs
# ------------------------------------------------------------

ranking_records = []

for model_name, group in model_distance_summary.groupby(
    "model"
):

    ranked = group.sort_values(
        "mean_distance",
        ascending=False
    )

    ranking_records.append({
        "model": model_name,
        "descending_distance_order": " > ".join(
            ranked["country_1"]
            + "-"
            + ranked["country_2"]
        )
    })

model_rankings = pd.DataFrame(ranking_records)

# ------------------------------------------------------------
# 4. Validate diagnostics
# ------------------------------------------------------------

assert stability_summary.shape[0] == 15

assert stability_summary["spearman_rho"].between(
    -1, 1
).all()

assert stability_summary[
    "mean_absolute_difference"
].ge(0).all()

# ------------------------------------------------------------
# 5. Display diagnostic results
# ------------------------------------------------------------

print("=" * 70)
print("TF-IDF ROBUSTNESS AND STABILITY")
print("=" * 70)

print("\nAnnual rank-correlation stability:")

display(
    stability_summary.pivot(
        index="model",
        columns="pair",
        values="spearman_rho"
    ).round(4)
)

print("\nMean absolute deviation from baseline:")

display(
    stability_summary.pivot(
        index="model",
        columns="pair",
        values="mean_absolute_difference"
    ).round(4)
)

print("\nCountry-pair distance ordering:")

display(model_rankings)

print("\nTF-IDF robustness diagnostics: PASSED")



## 4. Temporal Baseline Diagnostics

### Objective

Assess whether same-year diplomatic speeches are lexically closer than speeches delivered by the same country pair in nearby, non-matching years.

### Analytical design

For each country pair and complete three-country year:

1. Calculate the observed same-year TF-IDF cosine distance.
2. Construct an alternative-year reference using speeches from the comparison country within a ±5-year calendar window.
3. Exclude the matching year from the reference distribution.
4. Calculate the mean alternative-year distance.
5. Calculate the within-year contrast between observed and reference distances.

### Interpretation

A negative contrast indicates that same-year speeches are lexically closer than their local alternative-year reference.

Alternative-year comparisons are not independent observations. The analysis is exploratory and does not provide a causal estimate of common historical effects.

No inferential p-values are computed. The analysis uses the previously fitted baseline TF-IDF representation.


In [ ]:

# ============================================================
# STEP 04: LOCAL TEMPORAL BASELINE DIAGNOSTICS
# ============================================================

TEMPORAL_WINDOW = 5

temporal_baseline_records = []

for first, second in COMMON_PAIRS:

    for year in common_years:

        first_idx = document_lookup[(first, year)]
        second_idx = document_lookup[(second, year)]

        observed_similarity = float(
            tfidf_matrix[first_idx]
            .multiply(tfidf_matrix[second_idx])
            .sum()
        )

        observed_distance = 1.0 - observed_similarity

        alternative_years = [
            other_year
            for other_year in range(
                year - TEMPORAL_WINDOW,
                year + TEMPORAL_WINDOW + 1
            )
            if (
                other_year != year
                and (second, other_year) in document_lookup
            )
        ]

        reference_distances = []

        for other_year in alternative_years:

            comparison_idx = document_lookup[
                (second, other_year)
            ]

            similarity = float(
                tfidf_matrix[first_idx]
                .multiply(tfidf_matrix[comparison_idx])
                .sum()
            )

            reference_distances.append(
                1.0 - similarity
            )

        reference_mean = (
            float(np.mean(reference_distances))
            if reference_distances
            else np.nan
        )

        temporal_baseline_records.append({
            "country_1": first,
            "country_2": second,
            "year": year,
            "observed_distance": observed_distance,
            "reference_mean_distance": reference_mean,
            "n_reference_years": len(reference_distances),
            "distance_contrast": (
                observed_distance - reference_mean
                if np.isfinite(reference_mean)
                else np.nan
            )
        })

temporal_baseline_df = pd.DataFrame(
    temporal_baseline_records
)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

assert len(temporal_baseline_df) == 219

assert temporal_baseline_df[
    "n_reference_years"
].ge(1).all()

assert temporal_baseline_df[
    "distance_contrast"
].notna().all()

observed_keyed = (
    temporal_baseline_df
    .set_index(["country_1", "country_2", "year"])
    ["observed_distance"]
    .sort_index()
)

assert np.allclose(
    observed_keyed.to_numpy(),
    baseline_original.to_numpy(),
    atol=1e-10
)

# ------------------------------------------------------------
# Summary
# ------------------------------------------------------------

temporal_baseline_summary = (
    temporal_baseline_df
    .groupby(["country_1", "country_2"])
    .agg(
        n_years=("year", "size"),
        mean_observed=("observed_distance", "mean"),
        mean_reference=("reference_mean_distance", "mean"),
        mean_contrast=("distance_contrast", "mean"),
        median_contrast=("distance_contrast", "median"),
        sd_contrast=("distance_contrast", "std"),
        negative_contrast_pct=(
            "distance_contrast",
            lambda x: 100 * x.lt(0).mean()
        )
    )
    .reset_index()
)

print("=" * 70)
print("LOCAL TEMPORAL BASELINE DIAGNOSTICS")
print("=" * 70)

display(temporal_baseline_summary.round(4))

print("\nReference-year availability:")

display(
    temporal_baseline_df.groupby(
        ["country_1", "country_2"]
    )["n_reference_years"].describe().round(2)
)

print("\nTemporal baseline validation: PASSED")



## 5. Symmetric Temporal Baseline Sensitivity

### Objective

Evaluate the sensitivity of same-year lexical distance contrasts to temporal window width and the direction of cross-year comparisons.

### Analytical design

For each country pair and common calendar year:

1. Compute the same-year cosine distance.
2. Compare the first country's speech with nearby speeches from the second country.
3. Reverse the comparison direction.
4. Average the two direction-specific reference distances.
5. Repeat the procedure for temporal windows of ±3, ±5, and ±10 years.
6. Compare the resulting temporal contrasts across specifications.

### Methodological considerations

The symmetric reference reduces dependence on the arbitrarily chosen direction of comparison but does not eliminate historical confounding.

Reference speeches may be reused across multiple years, generating dependence among observations.

Broader windows may introduce greater historical heterogeneity.

The analysis is descriptive. Formal uncertainty estimation will require methods that accommodate temporal dependence.

The comparison does not establish that greater lexical similarity implies diplomatic agreement.


In [ ]:

# ============================================================
# STEP 05: SYMMETRIC TEMPORAL BASELINE SENSITIVITY
# ============================================================

WINDOWS = [3, 5, 10]
MIN_REFERENCE_YEARS_PER_DIRECTION = 2

# Reuse the fitted baseline TF-IDF matrix.
# All documents remain unchanged.

symmetric_records = []

def cross_year_distances(anchor_code, comparison_code, year, window):
    """Return available cross-year lexical distances."""

    anchor_idx = document_lookup[(anchor_code, year)]

    eligible_years = [
        other_year
        for other_year in range(year - window, year + window + 1)
        if (
            other_year != year
            and (comparison_code, other_year) in document_lookup
        )
    ]

    values = []

    for other_year in eligible_years:

        comparison_idx = document_lookup[
            (comparison_code, other_year)
        ]

        similarity = float(
            tfidf_matrix[anchor_idx]
            .multiply(tfidf_matrix[comparison_idx])
            .sum()
        )

        values.append(1.0 - similarity)

    return np.asarray(values, dtype=float)


for first, second in COMMON_PAIRS:

    for year in common_years:

        idx_first = document_lookup[(first, year)]
        idx_second = document_lookup[(second, year)]

        observed = 1.0 - float(
            tfidf_matrix[idx_first]
            .multiply(tfidf_matrix[idx_second])
            .sum()
        )

        for window in WINDOWS:

            forward = cross_year_distances(
                first, second, year, window
            )

            reverse = cross_year_distances(
                second, first, year, window
            )

            valid_reference = (
                len(forward) >= MIN_REFERENCE_YEARS_PER_DIRECTION
                and len(reverse) >= MIN_REFERENCE_YEARS_PER_DIRECTION
            )

            forward_mean = (
                float(forward.mean()) if len(forward) else np.nan
            )

            reverse_mean = (
                float(reverse.mean()) if len(reverse) else np.nan
            )

            symmetric_reference = (
                (forward_mean + reverse_mean) / 2
                if valid_reference else np.nan
            )

            symmetric_records.append({
                "country_1": first,
                "country_2": second,
                "year": int(year),
                "window": window,
                "observed_distance": observed,
                "forward_reference": forward_mean,
                "reverse_reference": reverse_mean,
                "symmetric_reference": symmetric_reference,
                "n_forward": len(forward),
                "n_reverse": len(reverse),
                "contrast": (
                    observed - symmetric_reference
                    if valid_reference else np.nan
                )
            })

symmetric_baseline_df = pd.DataFrame(symmetric_records)

# ------------------------------------------------------------
# 2. Validation
# ------------------------------------------------------------

assert len(symmetric_baseline_df) == 219 * len(WINDOWS)

assert symmetric_baseline_df["observed_distance"].between(
    0, 1
).all()

assert (
    symmetric_baseline_df.loc[
        symmetric_baseline_df["contrast"].notna(),
        ["n_forward", "n_reverse"]
    ].ge(MIN_REFERENCE_YEARS_PER_DIRECTION).all().all()
)

# ------------------------------------------------------------
# 3. Summarize sensitivity results
# ------------------------------------------------------------

symmetric_summary = (
    symmetric_baseline_df
    .groupby(["country_1", "country_2", "window"])
    .agg(
        n_years=("contrast", "count"),
        mean_contrast=("contrast", "mean"),
        median_contrast=("contrast", "median"),
        sd_contrast=("contrast", "std"),
        negative_contrast_pct=(
            "contrast",
            lambda x: (
                100 * x.dropna().lt(0).mean()
                if x.notna().any() else np.nan
            )
        ),
        mean_forward_reference=(
            "forward_reference", "mean"
        ),
        mean_reverse_reference=(
            "reverse_reference", "mean"
        )
    )
    .reset_index()
)

print("=" * 70)
print("SYMMETRIC TEMPORAL BASELINE SENSITIVITY")
print("=" * 70)

display(symmetric_summary.round(4))

print("\nIncomplete reference comparisons:")

display(
    symmetric_baseline_df.loc[
        symmetric_baseline_df["contrast"].isna(),
        [
            "country_1", "country_2", "year",
            "window", "n_forward", "n_reverse"
        ]
    ]
)

print("\nSymmetric baseline diagnostics: PASSED")



## 6. Common-Sample and Directional Robustness Diagnostics

### Objective

Assess temporal baseline robustness while holding the analyzed calendar years constant across window specifications.

### Analytical procedures

1. Identify country-pair years with valid symmetric references for all three temporal windows.
2. Compare mean and median contrasts using the common analytical sample.
3. Evaluate the agreement between forward and reverse reference distances.
4. Quantify the proportion of years with negative contrasts in each direction.
5. Verify that the conclusions are not driven by differences in sample composition.

### Methodological considerations

The common-sample comparison removes variation in sample composition but does not address serial dependence.

Forward and reverse reference estimates can differ because the countries have different speech-length distributions, temporal coverage, and lexical trajectories.

The three window specifications are sensitivity analyses, not independent replications.

No formal hypothesis testing is conducted at this stage.


In [ ]:

# ============================================================
# STEP 06: CORRECTED COMMON-SAMPLE TEMPORAL ROBUSTNESS
# ============================================================
# ------------------------------------------------------------
# 1. Identify valid years within each country pair
# ------------------------------------------------------------

valid_matrix = (
    symmetric_baseline_df
    .assign(valid=lambda x: x["contrast"].notna())
    .pivot(
        index=["country_1", "country_2", "year"],
        columns="window",
        values="valid"
    )
)

assert set(WINDOWS).issubset(valid_matrix.columns)

common_valid_keys = valid_matrix.index[
    valid_matrix[WINDOWS].fillna(False).all(axis=1)
]

common_sample = (
    symmetric_baseline_df
    .set_index(["country_1", "country_2", "year"])
    .loc[common_valid_keys]
    .reset_index()
    .copy()
)

# ------------------------------------------------------------
# 2. Validate within-pair common-year consistency
# ------------------------------------------------------------

expected_common_counts = {
    ("CHN", "USA"): 73,
    ("IRN", "CHN"): 71,
    ("IRN", "USA"): 71
}

common_counts = (
    common_sample
    .groupby(["country_1", "country_2", "window"])
    .size()
)

for pair, expected_n in expected_common_counts.items():

    for window in WINDOWS:

        actual_n = int(
            common_counts.loc[
                (pair[0], pair[1], window)
            ]
        )

        assert actual_n == expected_n, (
            f"Unexpected sample size for {pair}, "
            f"window={window}: {actual_n}"
        )

assert common_sample["contrast"].notna().all()

# ------------------------------------------------------------
# 3. Calculate direction-specific contrasts
# ------------------------------------------------------------

common_sample["forward_contrast"] = (
    common_sample["observed_distance"]
    - common_sample["forward_reference"]
)

common_sample["reverse_contrast"] = (
    common_sample["observed_distance"]
    - common_sample["reverse_reference"]
)

common_sample["directional_gap"] = (
    common_sample["forward_reference"]
    - common_sample["reverse_reference"]
)

assert np.allclose(
    common_sample["contrast"],
    (
        common_sample["forward_contrast"]
        + common_sample["reverse_contrast"]
    ) / 2
)

# ------------------------------------------------------------
# 4. Calculate common-sample descriptive statistics
# ------------------------------------------------------------

common_sample_summary = (
    common_sample
    .groupby(["country_1", "country_2", "window"])
    .agg(
        n_years=("year", "size"),
        mean_contrast=("contrast", "mean"),
        median_contrast=("contrast", "median"),
        sd_contrast=("contrast", "std"),
        negative_symmetric_pct=(
            "contrast",
            lambda x: 100 * x.lt(0).mean()
        ),
        negative_forward_pct=(
            "forward_contrast",
            lambda x: 100 * x.lt(0).mean()
        ),
        negative_reverse_pct=(
            "reverse_contrast",
            lambda x: 100 * x.lt(0).mean()
        ),
        mean_absolute_directional_gap=(
            "directional_gap",
            lambda x: x.abs().mean()
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 5. Evaluate directional agreement
# ------------------------------------------------------------

directional_records = []

for (first, second, window), group in common_sample.groupby(
    ["country_1", "country_2", "window"]
):

    correlation = group["forward_reference"].corr(
        group["reverse_reference"]
    )

    sign_agreement = (
        np.sign(group["forward_contrast"])
        == np.sign(group["reverse_contrast"])
    ).mean()

    directional_records.append({
        "country_1": first,
        "country_2": second,
        "window": window,
        "directional_reference_correlation": correlation,
        "same_contrast_sign_pct": 100 * sign_agreement
    })

directional_summary = pd.DataFrame(
    directional_records
)

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

expected_total_rows = (
    sum(expected_common_counts.values()) * len(WINDOWS)
)

assert len(common_sample) == expected_total_rows

assert len(common_sample_summary) == 9
assert len(directional_summary) == 9

assert np.isfinite(
    common_sample["contrast"].to_numpy()
).all()

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("=" * 70)
print("CORRECTED COMMON-SAMPLE TEMPORAL ROBUSTNESS")
print("=" * 70)

print("\nCommon valid years by country pair:")

display(
    common_sample.groupby(
        ["country_1", "country_2", "window"]
    ).size().rename("n_years").to_frame()
)

print("\nCommon-sample contrast summary:")

display(common_sample_summary.round(4))

print("\nDirectional agreement diagnostics:")

display(directional_summary.round(4))

print("\nCommon-sample robustness: PASSED")



## 7. Temporal Dependence and Autocorrelation Diagnostics

### Objective

Assess serial dependence in the annual symmetric temporal-baseline contrasts before selecting an inferential procedure.

### Primary diagnostic series

The five-year symmetric temporal baseline is treated as the primary descriptive specification.

Three country-pair series are assessed separately:

- Iran–United States
- China–United States
- Iran–China

### Diagnostic procedures

1. Verify chronological ordering and missing calendar years.
2. Calculate calendar-aligned autocorrelations at lags 1–5.
3. Identify the longest consecutively observed segment.
4. Examine autocorrelation within that segment.
5. Evaluate implications for heteroskedasticity and autocorrelation-consistent inference.

### Statistical safeguards

Calendar-aligned autocorrelations are calculated only from observed year pairs. Missing years are never treated as adjacent observations.

The diagnostics are descriptive. Autocorrelation estimates may be unstable in short series, and failure to detect autocorrelation does not establish independence.

Selection of an inference method must account for temporal dependence, overlapping reference windows, and the shared-document structure of the three country-pair comparisons.


In [ ]:

# ============================================================
# STEP 07: TEMPORAL AUTOCORRELATION DIAGNOSTICS
# ============================================================

PRIMARY_WINDOW = 5
MAX_LAG = 5

# ------------------------------------------------------------
# 1. Select the primary symmetric baseline
# ------------------------------------------------------------

autocorr_data = (
    symmetric_baseline_df.loc[
        (symmetric_baseline_df["window"] == PRIMARY_WINDOW)
        & symmetric_baseline_df["contrast"].notna()
    ]
    .copy()
    .sort_values(["country_1", "country_2", "year"])
)

autocorr_data["pair"] = (
    autocorr_data["country_1"]
    + "-"
    + autocorr_data["country_2"]
)

assert len(autocorr_data) == 219

# ------------------------------------------------------------
# 2. Calculate calendar-aligned lag correlations
# ------------------------------------------------------------

autocorr_records = []

for pair, group in autocorr_data.groupby("pair"):

    series = (
        group.set_index("year")["contrast"]
        .reindex(range(1946, 2026))
    )

    for lag in range(1, MAX_LAG + 1):

        paired = pd.DataFrame({
            "current": series,
            "lagged": series.shift(lag)
        }).dropna()

        correlation = (
            paired["current"].corr(paired["lagged"])
            if len(paired) >= 3
            else np.nan
        )

        autocorr_records.append({
            "pair": pair,
            "lag": lag,
            "n_valid_pairs": len(paired),
            "autocorrelation": correlation
        })

autocorr_summary = pd.DataFrame(autocorr_records)

# ------------------------------------------------------------
# 3. Identify the longest consecutive segment
# ------------------------------------------------------------

segment_records = []

for pair, group in autocorr_data.groupby("pair"):

    observed_years = np.sort(
        group["year"].astype(int).unique()
    )

    segment_id = np.cumsum(
        np.r_[True, np.diff(observed_years) != 1]
    )

    segments = [
        observed_years[segment_id == sid]
        for sid in np.unique(segment_id)
    ]

    longest = max(segments, key=len)

    segment_records.append({
        "pair": pair,
        "total_observed_years": len(observed_years),
        "longest_segment_start": int(longest.min()),
        "longest_segment_end": int(longest.max()),
        "longest_segment_n": len(longest),
        "number_of_segments": len(segments)
    })

segment_summary = pd.DataFrame(segment_records)

# ------------------------------------------------------------
# 4. Validate diagnostic structure
# ------------------------------------------------------------

assert len(autocorr_summary) == 3 * MAX_LAG
assert len(segment_summary) == 3

assert autocorr_summary["n_valid_pairs"].ge(3).all()

assert autocorr_summary[
    "autocorrelation"
].dropna().between(-1, 1).all()

# ------------------------------------------------------------
# 5. Report results
# ------------------------------------------------------------

print("=" * 70)
print("TEMPORAL AUTOCORRELATION DIAGNOSTICS")
print("=" * 70)

print("\nCalendar-aligned lag correlations:")

acf_matrix = autocorr_summary.pivot(
    index="lag",
    columns="pair",
    values="autocorrelation"
)

display(acf_matrix.round(4))

print("\nNumber of valid year pairs:")

display(
    autocorr_summary.pivot(
        index="lag",
        columns="pair",
        values="n_valid_pairs"
    )
)

print("\nConsecutive observation segments:")

display(segment_summary)

print("\nTemporal dependence diagnostics: PASSED")


In [ ]:

# ============================================================
# STEP 07.1: AUTOCORRELATION DIAGNOSTIC VISUALIZATION
# ============================================================

fig_acf, axes = plt.subplots(
    1, 3,
    figsize=(12, 3.8),
    sharey=True,
    constrained_layout=True
)

pair_titles = {
    "IRN-USA": "Iran–United States",
    "CHN-USA": "China–United States",
    "IRN-CHN": "Iran–China"
}

for ax, (pair, title) in zip(
    axes,
    pair_titles.items()
):

    subset = autocorr_summary.loc[
        autocorr_summary["pair"] == pair
    ].sort_values("lag")

    ax.axhline(
        0,
        color="black",
        linewidth=0.9
    )

    ax.vlines(
        subset["lag"],
        0,
        subset["autocorrelation"],
        color="#404040",
        linewidth=2.2
    )

    ax.scatter(
        subset["lag"],
        subset["autocorrelation"],
        color="#202020",
        s=30,
        zorder=3
    )

    ax.set_title(title, fontsize=11, pad=10)
    ax.set_xlabel("Calendar lag (years)")
    ax.set_xticks(range(1, MAX_LAG + 1))
    ax.set_xlim(0.5, MAX_LAG + 0.5)
    ax.set_ylim(-1, 1)
    ax.grid(axis="y", alpha=0.15)

axes[0].set_ylabel("Lagged correlation")

fig_acf.suptitle(
    "Temporal Dependence in Symmetric Lexical Distance Contrasts",
    fontsize=12
)

plt.show()



## 8. HAC Inference for Same-Year Lexical Proximity

### Objective

Estimate mean same-year versus local cross-year lexical distance contrasts with uncertainty estimates robust to heteroskedasticity and serial correlation.

### Primary analysis

- Period: 1971–2025, a continuous 55-year interval.
- Country pairs: Iran–United States, China–United States, and Iran–China.
- Primary temporal reference: symmetric ±5-year window.
- Estimand: mean annual same-year minus local cross-year cosine distance.

### Statistical method

An intercept-only OLS model estimates the mean contrast, with Newey–West heteroskedasticity and autocorrelation-consistent covariance.

HAC bandwidth sensitivity is examined using 3, 5, and 10 annual lags.

Two-sided 95% confidence intervals and exploratory p-values are reported.

### Interpretation and limitations

A negative contrast indicates closer lexical proximity for same-year speeches than for the selected local cross-year reference.

This is a model-based descriptive contrast, not a causal effect or direct measure of diplomatic alignment.

HAC inference relies on asymptotic approximations and may remain imperfect with 55 annual observations, overlapping reference windows, nonstationarity, and historical changes in text production.

The three country-pair estimates share source speeches. They are not independent estimates.

These exploratory p-values are not adjusted for multiple comparisons and must not be interpreted as confirmatory evidence.


In [ ]:

# ============================================================
# STEP 08: HAC INFERENCE FOR TEMPORAL LEXICAL CONTRASTS
# ============================================================

PRIMARY_WINDOW = 5
INFERENCE_START = 1971
INFERENCE_END = 2025

HAC_LAGS = [3, 5, 10]

# ------------------------------------------------------------
# 1. Construct the continuous inference sample
# ------------------------------------------------------------

inference_df = symmetric_baseline_df.loc[
    (symmetric_baseline_df["window"] == PRIMARY_WINDOW)
    & symmetric_baseline_df["year"].between(
        INFERENCE_START, INFERENCE_END
    )
].copy()

inference_df["pair"] = (
    inference_df["country_1"]
    + "-"
    + inference_df["country_2"]
)

inference_df = inference_df.sort_values(
    ["pair", "year"]
).reset_index(drop=True)

expected_years = np.arange(
    INFERENCE_START, INFERENCE_END + 1
)

for pair, group in inference_df.groupby("pair"):

    assert np.array_equal(
        group["year"].to_numpy(),
        expected_years
    )

    assert group["contrast"].notna().all()

assert len(inference_df) == 55 * 3

# ------------------------------------------------------------
# 2. Estimate intercept-only HAC models
# ------------------------------------------------------------

hac_records = []
hac_models = {}

for pair, group in inference_df.groupby("pair"):

    y = group["contrast"].to_numpy(dtype=float)

    X = np.ones((len(y), 1), dtype=float)

    for lag in HAC_LAGS:

        result = sm.OLS(y, X).fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": lag,
                "use_correction": True
            }
        )

        ci = result.conf_int(alpha=0.05)[0]

        hac_records.append({
            "pair": pair,
            "hac_lag": lag,
            "n_years": len(y),
            "mean_contrast": float(result.params[0]),
            "hac_se": float(result.bse[0]),
            "ci_lower": float(ci[0]),
            "ci_upper": float(ci[1]),
            "p_value": float(result.pvalues[0])
        })

        hac_models[(pair, lag)] = result

hac_results = pd.DataFrame(hac_records)

# ------------------------------------------------------------
# 3. Validate estimates
# ------------------------------------------------------------

assert len(hac_results) == 9
assert hac_results["hac_se"].gt(0).all()
assert hac_results["p_value"].between(0, 1).all()

assert (
    hac_results["ci_lower"]
    <= hac_results["mean_contrast"]
).all()

assert (
    hac_results["mean_contrast"]
    <= hac_results["ci_upper"]
).all()

for pair, group in inference_df.groupby("pair"):

    expected_mean = group["contrast"].mean()

    estimated_means = hac_results.loc[
        hac_results["pair"] == pair,
        "mean_contrast"
    ]

    assert np.allclose(
        estimated_means,
        expected_mean,
        atol=1e-12
    )

# ------------------------------------------------------------
# 4. Display inference results
# ------------------------------------------------------------

print("=" * 70)
print("HAC INFERENCE: SYMMETRIC TEMPORAL BASELINE")
print("=" * 70)

print(
    f"Inference period: {INFERENCE_START}-{INFERENCE_END}"
)
print(f"Primary temporal window: ±{PRIMARY_WINDOW} years")
print("Covariance estimator: Newey-West HAC")

print("\nHAC bandwidth sensitivity:")
display(hac_results.round(6))

print("\nPrimary specification: HAC lag 5")

primary_hac = hac_results.loc[
    hac_results["hac_lag"] == 5
].copy()

display(primary_hac.round(6))

print("\nHAC inference calculations: PASSED")



## 9. Pairwise Differences in Temporal Lexical Proximity

### Objective

Evaluate whether mean same-year lexical proximity contrasts differ across the three country pairs during 1971–2025.

### Statistical design

Annual symmetric temporal contrasts are aligned by calendar year using the primary ±5-year reference specification.

Three comparisons are evaluated:

1. Iran–United States versus China–United States
2. Iran–United States versus Iran–China
3. China–United States versus Iran–China

### Estimation

For each comparison, a within-year difference series is constructed.

An intercept-only OLS model with Newey–West HAC covariance estimates the mean difference and its 95% confidence interval.

HAC bandwidth sensitivity is evaluated using 3, 5, and 10 lags.

### Multiple testing

Exploratory p-values from the three primary comparisons are adjusted using the Holm procedure.

### Interpretation

A difference in temporal lexical proximity contrasts does not establish a difference in diplomatic agreement or political relationships.

The analysis is conditional on the TF-IDF specification, temporal reference definition, and selected historical period.

Inference is exploratory and relies on asymptotic HAC approximations.


In [ ]:

# ============================================================
# STEP 09: PAIRED HAC COMPARISONS OF TEMPORAL CONTRASTS
# ============================================================

# ------------------------------------------------------------
# 1. Align the three contrast series by year
# ------------------------------------------------------------

contrast_wide = inference_df.pivot(
    index="year",
    columns="pair",
    values="contrast"
).sort_index()

EXPECTED_PAIRS = [
    "IRN-USA",
    "CHN-USA",
    "IRN-CHN"
]

contrast_wide = contrast_wide[EXPECTED_PAIRS]

assert contrast_wide.shape == (55, 3)
assert contrast_wide.notna().all().all()

assert np.array_equal(
    contrast_wide.index.to_numpy(),
    np.arange(1971, 2026)
)

# ------------------------------------------------------------
# 2. Define comparisons
# ------------------------------------------------------------

COMPARISONS = [
    ("IRN-USA", "CHN-USA"),
    ("IRN-USA", "IRN-CHN"),
    ("CHN-USA", "IRN-CHN")
]

HAC_LAGS = [3, 5, 10]

comparison_records = []

# ------------------------------------------------------------
# 3. Fit HAC models to within-year differences
# ------------------------------------------------------------

for first, second in COMPARISONS:

    differences = (
        contrast_wide[first] - contrast_wide[second]
    ).to_numpy(dtype=float)

    X = np.ones((len(differences), 1))

    for lag in HAC_LAGS:

        result = sm.OLS(differences, X).fit(
            cov_type="HAC",
            cov_kwds={
                "maxlags": lag,
                "use_correction": True
            }
        )

        ci = result.conf_int(alpha=0.05)[0]

        comparison_records.append({
            "comparison": f"{first} minus {second}",
            "hac_lag": lag,
            "n_years": len(differences),
            "mean_difference": float(result.params[0]),
            "hac_se": float(result.bse[0]),
            "ci_lower": float(ci[0]),
            "ci_upper": float(ci[1]),
            "p_value": float(result.pvalues[0])
        })

paired_hac_results = pd.DataFrame(comparison_records)

# ------------------------------------------------------------
# 4. Apply Holm correction to primary comparisons
# ------------------------------------------------------------

primary_mask = paired_hac_results["hac_lag"].eq(5)

raw_pvalues = paired_hac_results.loc[
    primary_mask, "p_value"
].to_numpy()

reject, adjusted_p, _, _ = multipletests(
    raw_pvalues,
    alpha=0.05,
    method="holm"
)

paired_hac_results["holm_adjusted_p"] = np.nan
paired_hac_results["holm_reject_005"] = pd.NA

paired_hac_results.loc[
    primary_mask, "holm_adjusted_p"
] = adjusted_p

paired_hac_results.loc[
    primary_mask, "holm_reject_005"
] = reject.tolist()

# ------------------------------------------------------------
# 5. Validate results
# ------------------------------------------------------------

assert len(paired_hac_results) == 9

assert paired_hac_results["n_years"].eq(55).all()

assert paired_hac_results["hac_se"].gt(0).all()

for first, second in COMPARISONS:

    expected_mean = (
        contrast_wide[first] - contrast_wide[second]
    ).mean()

    observed_means = paired_hac_results.loc[
        paired_hac_results["comparison"]
        == f"{first} minus {second}",
        "mean_difference"
    ]

    assert np.allclose(
        observed_means,
        expected_mean,
        atol=1e-12
    )

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 70)
print("PAIRED HAC COMPARISONS OF TEMPORAL CONTRASTS")
print("=" * 70)

print("\nHAC bandwidth sensitivity:")

display(
    paired_hac_results[
        [
            "comparison", "hac_lag", "n_years",
            "mean_difference", "hac_se",
            "ci_lower", "ci_upper", "p_value"
        ]
    ].round(6)
)

print("\nPrimary comparisons with Holm adjustment:")

primary_pairwise = paired_hac_results.loc[
    primary_mask
].copy()

display(
    primary_pairwise[
        [
            "comparison",
            "mean_difference",
            "ci_lower",
            "ci_upper",
            "p_value",
            "holm_adjusted_p",
            "holm_reject_005"
        ]
    ].round(6)
)

print("\nPaired HAC comparisons: PASSED")



## 10. Lexical Baseline: Interim Findings and Reporting

### Summary of evidence

The TF-IDF analysis provides a retrospective lexical-distance baseline for three country pairs.

Key findings:

- Same-year speeches exhibited lower lexical distances than nearby cross-year references on average.
- This pattern was descriptively robust across symmetric temporal windows.
- Mean contrasts were negative during 1971–2025 under the primary HAC specification.
- Paired HAC comparisons did not identify statistically significant differences in mean contrasts among country pairs after Holm adjustment.

### Scope of inference

These results characterize similarity in TF-IDF-weighted lexical representations rather than diplomatic alignment, thematic agreement, or political hostility.

The results are exploratory and conditional on preprocessing, corpus composition, reference-window definition, and model assumptions.

### Next analytical phase

The subsequent analysis will estimate thematic structure and examine country-level changes in topic prevalence. Model quality, interpretability, temporal comparability, and robustness will be evaluated before substantive interpretation.


In [ ]:

# # ============================================================
# # TABLE EXPORT: SUPPLEMENTARY TABLE S04
# # TF-IDF Temporal Baseline and HAC Results
# # ============================================================

# TABLE_PATH = (
#     PROJECT_DIR
#     / "results"
#     / "TableS04_TFIDF_Temporal_Baseline_Robustness.xlsx"
# )

# with pd.ExcelWriter(
#     TABLE_PATH,
#     engine="openpyxl"
# ) as writer:

#     common_year_summary.to_excel(
#         writer,
#         sheet_name="Lexical_Distances",
#         index=False
#     )

#     stability_summary.to_excel(
#         writer,
#         sheet_name="TFIDF_Model_Stability",
#         index=False
#     )

#     symmetric_summary.to_excel(
#         writer,
#         sheet_name="Temporal_Window_Sensitivity",
#         index=False
#     )

#     primary_hac.to_excel(
#         writer,
#         sheet_name="Primary_HAC_Inference",
#         index=False
#     )

#     primary_pairwise.to_excel(
#         writer,
#         sheet_name="Paired_HAC_Holm",
#         index=False
#     )

# assert TABLE_PATH.is_file()

# print("=" * 65)
# print("SUPPLEMENTARY TABLE S04 EXPORT")
# print("=" * 65)

# print(f"Output: {TABLE_PATH}")
# print("Number of worksheets: 5")
# print("Statistical table export: PASSED")



## 11. Topic Modeling Design Audit

### Objective

Establish the analytical configuration for exploratory thematic modeling of the three-country UN General Debate corpus.

### Candidate method

Non-negative Matrix Factorization (NMF) applied to a shared document-term TF-IDF representation.

### Design principles

- All three countries are modeled in a shared thematic feature space.
- Each speech is treated as one document.
- The original text data remain unchanged.
- Topic counts are evaluated over a predefined candidate range.
- Random initialization and reproducibility are controlled.
- Model interpretability and stability are assessed before selecting a final specification.
- Historical and country-level topic prevalence are not interpreted until model validation is complete.

### Limitations

NMF components are mathematical patterns of word co-occurrence and do not automatically correspond to substantive diplomatic issues.

Short phrases, names, institutional vocabulary, and historical changes in terminology may affect the extracted components.

Comparability across the full historical period requires additional checks, including sensitivity to China's change in UN representation in 1971.


In [ ]:

# ============================================================
# STEP 11: TOPIC MODELING CONFIGURATION AUDIT
# ============================================================


TOPIC_CANDIDATES = [5, 7, 10, 12, 15]
TOPIC_RANDOM_SEEDS = [11, 42, 73]

TOPIC_VECTORIZER_CONFIG = {
    "lowercase": True,
    "stop_words": "english",
    "ngram_range": (1, 2),
    "min_df": 3,
    "max_df": 0.90,
    "sublinear_tf": True,
    "norm": "l2",
    "dtype": np.float64
}

# ------------------------------------------------------------
# 1. Build a shared thematic vocabulary
# ------------------------------------------------------------

topic_vectorizer = TfidfVectorizer(
    **TOPIC_VECTORIZER_CONFIG
)

X_topics = topic_vectorizer.fit_transform(
    df_nlp["text_clean"]
)

topic_features = topic_vectorizer.get_feature_names_out()

# ------------------------------------------------------------
# 2. Validate document-term representation
# ------------------------------------------------------------

assert X_topics.shape[0] == 233
assert X_topics.shape[1] > max(TOPIC_CANDIDATES)

assert np.isfinite(X_topics.data).all()
assert (X_topics.data >= 0).all()

document_feature_counts = np.diff(
    X_topics.indptr
)

assert np.all(document_feature_counts > 0)

# ------------------------------------------------------------
# 3. Report modeling readiness
# ------------------------------------------------------------

print("=" * 70)
print("TOPIC MODELING DESIGN AUDIT")
print("=" * 70)

print(f"Documents: {X_topics.shape[0]}")
print(f"Vocabulary features: {X_topics.shape[1]:,}")
print(f"Nonzero entries: {X_topics.nnz:,}")

print("\nCandidate topic counts:", TOPIC_CANDIDATES)
print("Random seeds:", TOPIC_RANDOM_SEEDS)

print("\nFeatures per document:")

display(
    pd.Series(
        document_feature_counts,
        name="n_active_features"
    ).describe().to_frame()
)

print("\nCountry-level document composition:")

display(
    df_nlp["country_code"]
    .value_counts()
    .rename("n_documents")
    .to_frame()
)

print("\nTopic modeling configuration: PASSED")



## 12. NMF Topic Model Fitting and Stability Assessment

### Objective

Evaluate candidate Non-negative Matrix Factorization (NMF) models for identifying latent thematic patterns in the three-country UN General Debate corpus.

### Candidate configurations

- Topic counts: 5, 7, 10, 12, and 15
- Random seeds: 11, 42, and 73
- Initialization: NNDSVD with randomized zero filling
- Optimization: Coordinate descent
- Loss function: Frobenius reconstruction loss

### Evaluation criteria

1. **Reconstruction error:** Evaluate how accurately each model approximates the document-term representation.
2. **Convergence:** Record whether the optimization reaches its stopping criterion.
3. **Topic diversity:** Quantify the proportion of unique terms among the top ten terms of all topics.
4. **Topic redundancy:** Evaluate similarity among components within the same model.
5. **Cross-seed stability:** Match topic components across random seeds using cosine similarity and optimal assignment.

### Model selection principles

Increasing the number of topics generally improves in-sample reconstruction, so reconstruction error alone is insufficient for selecting the final model.

High stability does not automatically establish semantic validity.

Final model selection will additionally require topic coherence diagnostics, examination of representative documents, historical interpretability, and sensitivity analyses.

No political labels are assigned to topics at this stage.


In [ ]:

# ============================================================
# STEP 12: NMF CANDIDATE MODEL FITTING
# ============================================================



# ------------------------------------------------------------
# 1. Define experimental configuration
# ------------------------------------------------------------

NMF_TOPIC_COUNTS = [5, 7, 10, 12, 15]
NMF_RANDOM_SEEDS = [11, 42, 73]

NMF_MAX_ITER = 600
NMF_TOL = 1e-4
TOP_TERMS = 10

# ------------------------------------------------------------
# 2. Validate the input feature matrix
# ------------------------------------------------------------

assert X_topics.shape[0] == len(df_nlp) == 233
assert X_topics.shape[1] == len(topic_features)

assert np.isfinite(X_topics.data).all()
assert np.all(X_topics.data >= 0)

matrix_norm = float(
    np.sqrt(X_topics.multiply(X_topics).sum())
)

assert matrix_norm > 0

# ------------------------------------------------------------
# 3. Fit candidate NMF models
# ------------------------------------------------------------

nmf_model_bank = {}
nmf_fit_records = []

for k in NMF_TOPIC_COUNTS:

    for seed in NMF_RANDOM_SEEDS:

        model = NMF(
            n_components=k,
            init="nndsvdar",
            solver="cd",
            beta_loss="frobenius",
            max_iter=NMF_MAX_ITER,
            tol=NMF_TOL,
            random_state=seed
        )

        W = model.fit_transform(X_topics)
        H = model.components_

        # Validate factorization.
        assert W.shape == (233, k)
        assert H.shape == (k, X_topics.shape[1])

        assert np.isfinite(W).all()
        assert np.isfinite(H).all()

        assert np.all(W >= 0)
        assert np.all(H >= 0)

        # Scale reconstruction error by the input norm.
        relative_error = (
            model.reconstruction_err_ / matrix_norm
        )

        converged = model.n_iter_ < NMF_MAX_ITER

        nmf_model_bank[(k, seed)] = {
            "model": model,
            "W": W,
            "H": H
        }

        nmf_fit_records.append({
            "n_topics": k,
            "random_seed": seed,
            "n_iterations": model.n_iter_,
            "converged": converged,
            "reconstruction_error": model.reconstruction_err_,
            "relative_reconstruction_error": relative_error
        })

        status = "OK" if converged else "REVIEW"

        print(
            f"[{status}] "
            f"K={k:02d} | "
            f"Seed={seed} | "
            f"Iterations={model.n_iter_:03d} | "
            f"Relative error={relative_error:.5f}"
        )

# ------------------------------------------------------------
# 4. Validate experiment completeness
# ------------------------------------------------------------

nmf_fit_df = pd.DataFrame(nmf_fit_records)

assert len(nmf_fit_df) == 15
assert len(nmf_model_bank) == 15

assert not nmf_fit_df.duplicated(
    ["n_topics", "random_seed"]
).any()

# ------------------------------------------------------------
# 5. Summarize fitted models
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NMF CANDIDATE MODEL FITTING SUMMARY")
print("=" * 70)

fit_summary = (
    nmf_fit_df
    .groupby("n_topics")
    .agg(
        n_runs=("random_seed", "size"),
        mean_relative_error=(
            "relative_reconstruction_error", "mean"
        ),
        sd_relative_error=(
            "relative_reconstruction_error", "std"
        ),
        max_iterations=("n_iterations", "max"),
        all_converged=("converged", "all")
    )
    .reset_index()
)

display(fit_summary.round(5))

print("\nCompleted model fits:", len(nmf_model_bank))

if nmf_fit_df["converged"].all():
    print("NMF fitting and convergence: PASSED")
else:
    print("NMF fitting: REVIEW NON-CONVERGED MODELS")


In [ ]:

# ============================================================
# STEP 12.1: NMF TOPIC STABILITY AND DIVERSITY
# ============================================================



# ------------------------------------------------------------
# 1. Define model-level diagnostic functions
# ------------------------------------------------------------

def calculate_topic_diversity(H, top_n=10):
    """Calculate top-term uniqueness across topic components."""

    top_indices = np.argsort(
        H, axis=1
    )[:, -top_n:]

    total_terms = top_indices.size
    unique_terms = np.unique(top_indices).size

    return unique_terms / total_terms


def calculate_topic_redundancy(H):
    """Calculate mean maximum cross-topic cosine similarity."""

    similarity = cosine_similarity(H)

    np.fill_diagonal(similarity, -np.inf)

    return float(
        np.max(similarity, axis=1).mean()
    )


def calculate_aligned_topic_stability(H1, H2):
    """Align NMF components using optimal cosine assignment."""

    similarity = cosine_similarity(H1, H2)

    row_ind, col_ind = linear_sum_assignment(
        -similarity
    )

    matched = similarity[row_ind, col_ind]

    return {
        "mean_stability": float(matched.mean()),
        "median_stability": float(np.median(matched)),
        "min_stability": float(matched.min())
    }


# ------------------------------------------------------------
# 2. Compute diversity and within-model redundancy
# ------------------------------------------------------------

quality_records = []

for (k, seed), fitted in nmf_model_bank.items():

    H = fitted["H"]

    quality_records.append({
        "n_topics": k,
        "random_seed": seed,
        "topic_diversity": calculate_topic_diversity(
            H, TOP_TERMS
        ),
        "topic_redundancy": calculate_topic_redundancy(H)
    })

nmf_quality_df = pd.DataFrame(quality_records)

# ------------------------------------------------------------
# 3. Compute pairwise stability across random seeds
# ------------------------------------------------------------

stability_records = []

for k in NMF_TOPIC_COUNTS:

    for seed_1, seed_2 in combinations(
        NMF_RANDOM_SEEDS, 2
    ):

        H1 = nmf_model_bank[(k, seed_1)]["H"]
        H2 = nmf_model_bank[(k, seed_2)]["H"]

        stability = calculate_aligned_topic_stability(
            H1, H2
        )

        stability_records.append({
            "n_topics": k,
            "seed_1": seed_1,
            "seed_2": seed_2,
            **stability
        })

nmf_stability_df = pd.DataFrame(stability_records)

# ------------------------------------------------------------
# 4. Aggregate candidate model diagnostics
# ------------------------------------------------------------

quality_summary = (
    nmf_quality_df
    .groupby("n_topics")
    .agg(
        mean_topic_diversity=("topic_diversity", "mean"),
        mean_topic_redundancy=("topic_redundancy", "mean")
    )
    .reset_index()
)

stability_summary_nmf = (
    nmf_stability_df
    .groupby("n_topics")
    .agg(
        mean_stability=("mean_stability", "mean"),
        sd_stability=("mean_stability", "std"),
        worst_matched_topic=("min_stability", "min")
    )
    .reset_index()
)

nmf_diagnostics = (
    fit_summary
    .merge(
        quality_summary,
        on="n_topics",
        validate="one_to_one"
    )
    .merge(
        stability_summary_nmf,
        on="n_topics",
        validate="one_to_one"
    )
)

# ------------------------------------------------------------
# 5. Validate diagnostic results
# ------------------------------------------------------------

assert len(nmf_quality_df) == 15
assert len(nmf_stability_df) == 15
assert len(nmf_diagnostics) == 5

assert nmf_diagnostics[
    "mean_topic_diversity"
].between(0, 1).all()

assert nmf_diagnostics[
    "mean_stability"
].between(0, 1).all()

assert nmf_diagnostics[
    "mean_topic_redundancy"
].between(0, 1).all()

# ------------------------------------------------------------
# 6. Display results
# ------------------------------------------------------------

print("=" * 70)
print("NMF TOPIC MODEL QUALITY AND STABILITY")
print("=" * 70)

display(nmf_diagnostics.round(4))

print("\nSeed-pair stability details:")
display(nmf_stability_df.round(4))

print("\nNMF topic diagnostics: PASSED")



## 12.2. Convergence Refinement of the 15-Topic Model

### Objective

Resolve the convergence warnings observed for two candidate 15-topic NMF models without modifying previously converged models.

### Procedure

1. Identify non-converged model runs.
2. Refit only the affected configurations using a higher iteration limit.
3. Preserve the original initialization, random seeds, optimization tolerance, and input feature matrix.
4. Compare reconstruction errors and convergence status.
5. Update the experimental model bank only when convergence is achieved without deterioration in reconstruction quality.

### Methodological considerations

Convergence is a necessary numerical diagnostic but does not establish topic interpretability, coherence, or generalizability.

The apparent stability under NNDSVD-based initialization must subsequently be evaluated using meaningfully different initializations.

Non-converged models will remain flagged and will not be treated as validated candidates.


In [ ]:

# ============================================================
# STEP 12.2: TARGETED NMF CONVERGENCE REFINEMENT
# ============================================================



REFIT_MAX_ITER = 3000
REFIT_TOL = 1e-4

# ------------------------------------------------------------
# 1. Identify non-converged configurations
# ------------------------------------------------------------

non_converged = nmf_fit_df.loc[
    ~nmf_fit_df["converged"]
].copy()

print("=" * 70)
print("NMF CONVERGENCE REFINEMENT")
print("=" * 70)

print("\nModels requiring refinement:")
display(non_converged)

assert set(
    zip(
        non_converged["n_topics"],
        non_converged["random_seed"]
    )
) == {(15, 11), (15, 42)}

# ------------------------------------------------------------
# 2. Refit affected models
# ------------------------------------------------------------

refit_records = []

for row in non_converged.itertuples(index=False):

    k = int(row.n_topics)
    seed = int(row.random_seed)

    old_error = float(row.reconstruction_error)

    refined_model = NMF(
        n_components=k,
        init="nndsvdar",
        solver="cd",
        beta_loss="frobenius",
        max_iter=REFIT_MAX_ITER,
        tol=REFIT_TOL,
        random_state=seed
    )

    with warnings.catch_warnings(record=True) as caught:

        warnings.simplefilter(
            "always",
            ConvergenceWarning
        )

        W_new = refined_model.fit_transform(X_topics)

    H_new = refined_model.components_

    warning_detected = any(
        issubclass(w.category, ConvergenceWarning)
        for w in caught
    )

    converged = (
        refined_model.n_iter_ < REFIT_MAX_ITER
        and not warning_detected
    )

    new_error = float(
        refined_model.reconstruction_err_
    )

    relative_error = new_error / matrix_norm

    assert np.isfinite(W_new).all()
    assert np.isfinite(H_new).all()
    assert np.all(W_new >= 0)
    assert np.all(H_new >= 0)

    # Accept only converged models without material
    # deterioration in reconstruction quality.
    accepted = (
        converged
        and new_error <= old_error + 1e-6
    )

    if accepted:

        nmf_model_bank[(k, seed)] = {
            "model": refined_model,
            "W": W_new,
            "H": H_new
        }

        mask = (
            (nmf_fit_df["n_topics"] == k)
            & (nmf_fit_df["random_seed"] == seed)
        )

        nmf_fit_df.loc[
            mask, "n_iterations"
        ] = refined_model.n_iter_

        nmf_fit_df.loc[
            mask, "converged"
        ] = True

        nmf_fit_df.loc[
            mask, "reconstruction_error"
        ] = new_error

        nmf_fit_df.loc[
            mask, "relative_reconstruction_error"
        ] = relative_error

    refit_records.append({
        "n_topics": k,
        "random_seed": seed,
        "old_error": old_error,
        "new_error": new_error,
        "relative_error": relative_error,
        "iterations": refined_model.n_iter_,
        "converged": converged,
        "accepted": accepted
    })

# ------------------------------------------------------------
# 3. Summarize refinement outcomes
# ------------------------------------------------------------

nmf_refit_summary = pd.DataFrame(refit_records)

print("\nConvergence refinement outcomes:")
display(nmf_refit_summary.round(6))

# ------------------------------------------------------------
# 4. Refresh the candidate fitting summary
# ------------------------------------------------------------

fit_summary = (
    nmf_fit_df
    .groupby("n_topics")
    .agg(
        n_runs=("random_seed", "size"),
        mean_relative_error=(
            "relative_reconstruction_error", "mean"
        ),
        sd_relative_error=(
            "relative_reconstruction_error", "std"
        ),
        max_iterations=("n_iterations", "max"),
        all_converged=("converged", "all")
    )
    .reset_index()
)

print("\nUpdated NMF fitting summary:")
display(fit_summary.round(6))

# ------------------------------------------------------------
# 5. Final status
# ------------------------------------------------------------

if nmf_fit_df["converged"].all():
    print("\nNMF CONVERGENCE: ALL MODELS PASSED")
else:
    print("\nNMF CONVERGENCE: FURTHER REVIEW REQUIRED")



## 12.3. NMF Robustness to Random Initialization

### Objective

Evaluate the sensitivity of NMF topic solutions to substantially different initialization conditions.

### Experimental design

Five candidate topic counts (5, 7, 10, 12, and 15) are fitted using three independently seeded random initializations.

All alternative models use the same input document-term matrix, objective function, solver, convergence tolerance, and feature vocabulary.

### Evaluation criteria

1. Numerical convergence of each random-initialized model.
2. Relative reconstruction error compared with the best converged NNDSVDAR-initialized solution.
3. Optimal cosine-based topic alignment between random and reference solutions.
4. Pairwise topic stability across random initializations.
5. Identification of candidate topic counts with initialization-sensitive solutions.

### Methodological safeguards

Topic numbers are permutation-invariant and must be optimally aligned before comparing solutions.

High cosine similarity between aligned topic vectors indicates numerical consistency, not necessarily semantic coherence.

Random initialization stability does not establish robustness to document sampling, historical coverage, or preprocessing choices.

Final model selection requires additional coherence and qualitative interpretability assessments.

No model is selected solely on reconstruction error or initialization stability.


In [ ]:

# ============================================================
# STEP 12.3: NMF RANDOM INITIALIZATION ROBUSTNESS
# ============================================================



RANDOM_INIT_SEEDS = [11, 42, 73]
RANDOM_INIT_MAX_ITER = 3000
RANDOM_INIT_TOL = 1e-4

# ------------------------------------------------------------
# 1. Define permutation-invariant topic matching
# ------------------------------------------------------------

def aligned_topic_similarity(H1, H2):
    """Match topic vectors using optimal cosine assignment."""

    assert H1.shape == H2.shape

    similarity = cosine_similarity(H1, H2)

    rows, cols = linear_sum_assignment(-similarity)

    matched = similarity[rows, cols]

    return {
        "mean_similarity": float(matched.mean()),
        "median_similarity": float(np.median(matched)),
        "minimum_similarity": float(matched.min())
    }


# Unit test: permuting topic order must preserve similarity.
test_topics = np.eye(3)

test_result = aligned_topic_similarity(
    test_topics,
    test_topics[[2, 0, 1]]
)

assert np.isclose(test_result["mean_similarity"], 1.0)

# ------------------------------------------------------------
# 2. Verify the converged reference models
# ------------------------------------------------------------

assert nmf_fit_df["converged"].all()
assert len(nmf_model_bank) == 15

reference_models = {}

for k in NMF_TOPIC_COUNTS:

    candidate_rows = nmf_fit_df.loc[
        (nmf_fit_df["n_topics"] == k)
        & nmf_fit_df["converged"]
    ]

    best_row = candidate_rows.sort_values(
        "reconstruction_error"
    ).iloc[0]

    reference_seed = int(best_row["random_seed"])

    reference_models[k] = {
        "seed": reference_seed,
        "H": nmf_model_bank[(k, reference_seed)]["H"],
        "relative_error": float(
            best_row["relative_reconstruction_error"]
        )
    }

# ------------------------------------------------------------
# 3. Fit alternative random-initialized models
# ------------------------------------------------------------

nmf_random_bank = {}
random_fit_records = []

for k in NMF_TOPIC_COUNTS:

    for seed in RANDOM_INIT_SEEDS:

        model = NMF(
            n_components=k,
            init="random",
            solver="cd",
            beta_loss="frobenius",
            max_iter=RANDOM_INIT_MAX_ITER,
            tol=RANDOM_INIT_TOL,
            random_state=seed
        )

        with warnings.catch_warnings(record=True) as caught:

            warnings.simplefilter(
                "always",
                ConvergenceWarning
            )

            W_random = model.fit_transform(X_topics)

        H_random = model.components_

        warning_detected = any(
            issubclass(w.category, ConvergenceWarning)
            for w in caught
        )

        converged = (
            model.n_iter_ < RANDOM_INIT_MAX_ITER
            and not warning_detected
        )

        relative_error = (
            model.reconstruction_err_ / matrix_norm
        )

        reference = reference_models[k]

        alignment = aligned_topic_similarity(
            H_random,
            reference["H"]
        )

        assert np.isfinite(W_random).all()
        assert np.isfinite(H_random).all()
        assert np.all(W_random >= 0)
        assert np.all(H_random >= 0)

        nmf_random_bank[(k, seed)] = {
            "H": H_random.copy(),
            "converged": converged,
            "relative_error": relative_error
        }

        random_fit_records.append({
            "n_topics": k,
            "seed": seed,
            "iterations": model.n_iter_,
            "converged": converged,
            "relative_error": relative_error,
            "error_gap_vs_reference": (
                relative_error - reference["relative_error"]
            ),
            "mean_reference_alignment": alignment[
                "mean_similarity"
            ],
            "minimum_reference_alignment": alignment[
                "minimum_similarity"
            ]
        })

        status = "OK" if converged else "REVIEW"

        print(
            f"[{status}] "
            f"K={k:02d} | "
            f"Seed={seed} | "
            f"Iterations={model.n_iter_:04d} | "
            f"Error={relative_error:.5f} | "
            f"Alignment={alignment['mean_similarity']:.4f}"
        )

random_fit_df = pd.DataFrame(random_fit_records)

# ------------------------------------------------------------
# 4. Evaluate stability among random initializations
# ------------------------------------------------------------

random_pair_records = []

for k in NMF_TOPIC_COUNTS:

    for seed_1, seed_2 in combinations(
        RANDOM_INIT_SEEDS, 2
    ):

        first = nmf_random_bank[(k, seed_1)]
        second = nmf_random_bank[(k, seed_2)]

        # Only converged solutions enter stability summaries.
        if not (first["converged"] and second["converged"]):
            continue

        alignment = aligned_topic_similarity(
            first["H"],
            second["H"]
        )

        random_pair_records.append({
            "n_topics": k,
            "seed_1": seed_1,
            "seed_2": seed_2,
            "mean_random_stability": alignment[
                "mean_similarity"
            ],
            "minimum_random_stability": alignment[
                "minimum_similarity"
            ]
        })

random_stability_df = pd.DataFrame(random_pair_records)

# ------------------------------------------------------------
# 5. Aggregate random-initialization diagnostics
# ------------------------------------------------------------

random_summary = (
    random_fit_df
    .groupby("n_topics")
    .agg(
        n_runs=("seed", "size"),
        converged_runs=("converged", "sum"),
        mean_relative_error=("relative_error", "mean"),
        mean_error_gap=("error_gap_vs_reference", "mean"),
        mean_reference_alignment=(
            "mean_reference_alignment", "mean"
        ),
        worst_reference_alignment=(
            "minimum_reference_alignment", "min"
        )
    )
    .reset_index()
)

if not random_stability_df.empty:

    random_pair_summary = (
        random_stability_df
        .groupby("n_topics")
        .agg(
            n_converged_pairs=("mean_random_stability", "size"),
            mean_random_stability=(
                "mean_random_stability", "mean"
            ),
            worst_random_stability=(
                "minimum_random_stability", "min"
            )
        )
        .reset_index()
    )

    random_summary = random_summary.merge(
        random_pair_summary,
        on="n_topics",
        how="left",
        validate="one_to_one"
    )

# ------------------------------------------------------------
# 6. Validate experiment completeness
# ------------------------------------------------------------

assert len(random_fit_df) == 15
assert len(nmf_random_bank) == 15
assert len(random_summary) == 5

assert random_fit_df["relative_error"].gt(0).all()

assert random_fit_df[
    "mean_reference_alignment"
].between(0, 1).all()

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("NMF RANDOM INITIALIZATION ROBUSTNESS")
print("=" * 70)

print("\nRandom-initialization summary:")
display(random_summary.round(5))

print("\nIndividual random-start diagnostics:")
display(random_fit_df.round(5))

print("\nConvergence status:")

if random_fit_df["converged"].all():
    print("ALL RANDOM INITIALIZATIONS CONVERGED")
else:
    print("REVIEW NON-CONVERGED RANDOM INITIALIZATIONS")

print("\nRandom initialization experiment: COMPLETED")



## 12.4. Topic Interpretability and Representative Speech Audit

### Objective

Evaluate whether stable NMF components correspond to coherent and substantively interpretable themes in UN General Debate speeches.

### Models under initial review

- K = 7: Parsimonious candidate
- K = 10: Initialization-stable candidate
- K = 12: More granular candidate

### Evaluation procedures

1. Extract the twelve highest-weighted terms from each NMF component.
2. Identify the three most representative speeches for each component.
3. Examine whether topics are dominated by individual countries or historical periods.
4. Identify potentially redundant, generic, or historically specific topics.
5. Flag components requiring qualitative interpretation.

### Document representation

Document-level component contributions are calculated from the NMF reconstruction factors.

Relative contributions are used only to identify representative documents. They are not calibrated probabilities of topic membership.

### Interpretation safeguards

Top-weighted words do not independently establish a topic's substantive meaning.

Representative documents require examination of their full content before assigning topic labels.

Topic quality will subsequently be assessed using sliding-window NPMI coherence and further robustness checks.

No political topic labels or final model selection decisions are made automatically.


In [ ]:

# ============================================================
# STEP 12.4: TOPIC INTERPRETABILITY AND DOCUMENT AUDIT
# ============================================================


CANDIDATE_K = [7, 10, 12]
N_TOP_TERMS = 12
N_REPRESENTATIVE_DOCS = 3

topic_inventory_records = []
representative_document_records = []
selected_topic_models = {}

# ------------------------------------------------------------
# 1. Select the best converged reference run for each K
# ------------------------------------------------------------

for k in CANDIDATE_K:

    candidates = nmf_fit_df.loc[
        (nmf_fit_df["n_topics"] == k)
        & (nmf_fit_df["converged"])
    ]

    assert not candidates.empty

    best_run = candidates.sort_values(
        ["reconstruction_error", "random_seed"]
    ).iloc[0]

    seed = int(best_run["random_seed"])

    fitted = nmf_model_bank[(k, seed)]

    W = fitted["W"]
    H = fitted["H"]

    selected_topic_models[k] = {
        "seed": seed,
        "W": W,
        "H": H
    }

    assert W.shape == (len(df_nlp), k)
    assert H.shape == (k, len(topic_features))

    # --------------------------------------------------------
    # 2. Calculate relative reconstruction contributions
    # --------------------------------------------------------

    component_mass = H.sum(axis=1)

    document_contributions = (
        W * component_mass[np.newaxis, :]
    )

    contribution_totals = document_contributions.sum(
        axis=1,
        keepdims=True
    )

    relative_contributions = np.divide(
        document_contributions,
        contribution_totals,
        out=np.zeros_like(document_contributions),
        where=contribution_totals > 0
    )

    valid_documents = contribution_totals[:, 0] > 0

    dominant_topics = np.argmax(
        relative_contributions,
        axis=1
    )

    # --------------------------------------------------------
    # 3. Inspect every topic component
    # --------------------------------------------------------

    for topic_idx in range(k):

        top_indices = np.argsort(
            H[topic_idx]
        )[::-1][:N_TOP_TERMS]

        top_terms = topic_features[top_indices].tolist()

        n_dominant = int(
            np.sum(
                (dominant_topics == topic_idx)
                & valid_documents
            )
        )

        topic_inventory_records.append({
            "n_topics": k,
            "reference_seed": seed,
            "topic_id": topic_idx + 1,
            "top_terms": ", ".join(top_terms),
            "dominant_speeches": n_dominant,
            "mean_relative_contribution": float(
                relative_contributions[:, topic_idx].mean()
            )
        })

        # ----------------------------------------------------
        # 4. Identify representative speeches
        # ----------------------------------------------------

        ranked_indices = np.argsort(
            relative_contributions[:, topic_idx]
        )[::-1]

        ranked_indices = [
            idx for idx in ranked_indices
            if valid_documents[idx]
        ][:N_REPRESENTATIVE_DOCS]

        for rank, doc_idx in enumerate(
            ranked_indices,
            start=1
        ):

            document = df_nlp.iloc[doc_idx]

            representative_document_records.append({
                "n_topics": k,
                "topic_id": topic_idx + 1,
                "rank": rank,
                "country_code": document["country_code"],
                "year": int(document["year"]),
                "archive_path": document["archive_path"],
                "relative_contribution": float(
                    relative_contributions[
                        doc_idx, topic_idx
                    ]
                )
            })

# ------------------------------------------------------------
# 5. Construct diagnostic tables
# ------------------------------------------------------------

topic_inventory_df = pd.DataFrame(
    topic_inventory_records
)

representative_docs_df = pd.DataFrame(
    representative_document_records
)

# ------------------------------------------------------------
# 6. Validate the results
# ------------------------------------------------------------

assert len(topic_inventory_df) == sum(CANDIDATE_K)

assert len(representative_docs_df) == (
    sum(CANDIDATE_K) * N_REPRESENTATIVE_DOCS
)

assert topic_inventory_df[
    "mean_relative_contribution"
].between(0, 1).all()

assert representative_docs_df[
    "relative_contribution"
].between(0, 1).all()

assert not representative_docs_df.duplicated(
    ["n_topics", "topic_id", "archive_path"]
).any()

# ------------------------------------------------------------
# 7. Display interpretability diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("NMF TOPIC INTERPRETABILITY AUDIT")
print("=" * 70)

for k in CANDIDATE_K:

    print(f"\n{'=' * 50}")
    print(f"MODEL: K={k}")
    print(
        "Reference seed:",
        selected_topic_models[k]["seed"]
    )

    print("\nTopic vocabulary:")
    display(
        topic_inventory_df.loc[
            topic_inventory_df["n_topics"] == k
        ].round(4)
    )

    print("\nRepresentative speeches:")
    display(
        representative_docs_df.loc[
            representative_docs_df["n_topics"] == k,
            [
                "topic_id",
                "rank",
                "country_code",
                "year",
                "archive_path",
                "relative_contribution"
            ]
        ].round(4)
    )

print("\nTopic interpretability diagnostics: PASSED")



## 12.5. Country and Historical Period Confounding Audit

### Objective

Evaluate the association of NMF components with country identity and historical periods before treating the components as substantive diplomatic themes.

### Analytical approach

For each candidate model (K = 7, 10, 12):

1. Assign the dominant reconstruction component to each speech.
2. Calculate the country composition of speeches assigned to each component.
3. Calculate the historical-period composition of each component.
4. Identify components dominated by a single country or historical period.
5. Inspect the historical representation of China in the original speech metadata.

### Historical considerations

The representation of China at the United Nations changed in October 1971. Pre-transition and post-transition speeches must not automatically be attributed to a single continuous government.

Historical periods are descriptive categories, not estimates of political change points.

### Methodological limitations

Country and period concentration may reflect substantive historical vocabulary as well as corpus-specific identity markers.

A country-specific topic is not necessarily invalid; however, it cannot automatically be interpreted as a shared diplomatic issue.

Dominant-component assignments simplify a mixed-factor representation and are used only for diagnostic purposes.

No original documents are modified or excluded.


In [ ]:

# ============================================================
# STEP 12.5: COUNTRY AND HISTORICAL PERIOD TOPIC AUDIT
# ============================================================



PERIOD_BINS = [1945, 1971, 1990, 2010, 2025]

PERIOD_LABELS = [
    "1946-1971",
    "1972-1990",
    "1991-2010",
    "2011-2025"
]

topic_confounding_records = []
document_topic_records = []

# ------------------------------------------------------------
# 1. Prepare historical metadata
# ------------------------------------------------------------

document_metadata = df_nlp[
    ["country_code", "year", "archive_path"]
].copy()

document_metadata["historical_period"] = pd.cut(
    document_metadata["year"],
    bins=PERIOD_BINS,
    labels=PERIOD_LABELS,
    include_lowest=True
)

assert document_metadata["historical_period"].notna().all()

# ------------------------------------------------------------
# 2. Examine each candidate model
# ------------------------------------------------------------

for k in CANDIDATE_K:

    W = selected_topic_models[k]["W"]
    H = selected_topic_models[k]["H"]

    component_mass = H.sum(axis=1)

    contributions = (
        W * component_mass[np.newaxis, :]
    )

    totals = contributions.sum(axis=1)

    valid = totals > 0

    assert valid.all(), (
        f"Zero reconstruction mass detected for K={k}"
    )

    relative_contributions = (
        contributions / totals[:, np.newaxis]
    )

    dominant = np.argmax(
        relative_contributions,
        axis=1
    ) + 1

    # --------------------------------------------------------
    # 3. Build document-level diagnostic assignments
    # --------------------------------------------------------

    assigned = document_metadata.copy()

    assigned["n_topics"] = k
    assigned["dominant_topic"] = dominant

    assigned["dominant_contribution"] = (
        relative_contributions.max(axis=1)
    )

    document_topic_records.append(assigned)

    # --------------------------------------------------------
    # 4. Examine country and period concentration
    # --------------------------------------------------------

    for topic_id in range(1, k + 1):

        subset = assigned.loc[
            assigned["dominant_topic"] == topic_id
        ]

        n_documents = len(subset)

        if n_documents == 0:
            continue

        country_counts = (
            subset["country_code"].value_counts()
        )

        period_counts = (
            subset["historical_period"].value_counts()
        )

        leading_country = country_counts.idxmax()
        leading_period = period_counts.idxmax()

        topic_confounding_records.append({
            "n_topics": k,
            "topic_id": topic_id,
            "n_dominant_speeches": n_documents,
            "n_countries": subset[
                "country_code"
            ].nunique(),
            "leading_country": leading_country,
            "leading_country_pct": (
                100 * country_counts.max() / n_documents
            ),
            "leading_period": str(leading_period),
            "leading_period_pct": (
                100 * period_counts.max() / n_documents
            ),
            "first_year": int(subset["year"].min()),
            "last_year": int(subset["year"].max()),
            "mean_dominant_contribution": float(
                subset["dominant_contribution"].mean()
            )
        })

# ------------------------------------------------------------
# 5. Construct diagnostic datasets
# ------------------------------------------------------------

topic_confounding_df = pd.DataFrame(
    topic_confounding_records
)

document_topic_df = pd.concat(
    document_topic_records,
    ignore_index=True
)

# ------------------------------------------------------------
# 6. Validate results
# ------------------------------------------------------------

assert len(document_topic_df) == (
    len(df_nlp) * len(CANDIDATE_K)
)

assert (
    document_topic_df.groupby("n_topics").size()
    == len(df_nlp)
).all()

assert document_topic_df[
    "dominant_contribution"
].between(0, 1).all()

assert (
    topic_confounding_df["leading_country_pct"]
    .between(0, 100)
    .all()
)

# ------------------------------------------------------------
# 7. Display country and historical concentration
# ------------------------------------------------------------

print("=" * 70)
print("COUNTRY AND HISTORICAL PERIOD TOPIC AUDIT")
print("=" * 70)

for k in CANDIDATE_K:

    print(f"\nMODEL K={k}")

    display(
        topic_confounding_df.loc[
            topic_confounding_df["n_topics"] == k
        ].round(2)
    )

# ------------------------------------------------------------
# 8. Inspect complete topic vocabulary for K=10
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPLETE TOPIC VOCABULARY: K=10")
print("=" * 70)

for row in topic_inventory_df.loc[
    topic_inventory_df["n_topics"] == 10
].sort_values("topic_id").itertuples(index=False):

    print(f"\nTopic {row.topic_id:02d}")

    print(
        textwrap.fill(
            row.top_terms,
            width=90
        )
    )

# ------------------------------------------------------------
# 9. Inspect historical China records
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CHINA REPRESENTATION: HISTORICAL SOURCE AUDIT")
print("=" * 70)

china_transition = df_nlp.loc[
    (df_nlp["country_code"] == "CHN")
    & df_nlp["year"].between(1968, 1974)
].copy()

available_columns = [
    col for col in [
        "year",
        "country_code",
        "speaker_name",
        "speaker_post",
        "archive_path"
    ]
    if col in china_transition.columns
]

display(
    china_transition[
        available_columns
    ].sort_values("year")
)

china_1971 = china_transition.loc[
    china_transition["year"] == 1971
]

if not china_1971.empty:

    print("\nCHN 1971 speech opening excerpt:")

    print(
        str(china_1971.iloc[0]["text_raw"])[:600]
    )

else:

    print("\nNo CHN speech found for 1971.")

print("\nTopic confounding audit: COMPUTATION PASSED")
print("Historical and thematic interpretation: PENDING REVIEW")



## 12.6. Historical Cohort and Country-Identity Ablation

### Objective

Assess whether the country-dominated structure of the initial NMF solutions persists after restricting the corpus to 1972–2025 and removing selected country-identity features.

### Experimental design

Two NMF models with ten components are fitted to an identical, balanced set of country-year speeches.

- Model A: Original thematic TF-IDF features.
- Model B: The same TF-IDF representation with selected country-identity features excluded and document vectors renormalized.

### Evaluation

1. Verify complete country-year coverage.
2. Audit the number of excluded vocabulary features.
3. Compare dominant-topic country concentration.
4. Compare country concentration using all nonzero topic contributions.
5. Inspect the highest-weighted words in each model.

### Safeguards

The original documents and all previously fitted models remain unchanged.

The feature exclusion list is a limited sensitivity specification, not a claim that all country-specific information has been removed.

Topic IDs are not directly comparable across the two models.

Reconstruction errors from feature spaces with different dimensions are not used as direct evidence of superior substantive model quality.

No final thematic model is selected in this step.


In [ ]:

# ============================================================
# STEP 12.6: HISTORICAL COHORT AND IDENTITY ABLATION
# ============================================================



# ------------------------------------------------------------
# 1. Construct a balanced historical cohort
# ------------------------------------------------------------

FOCAL_COUNTRIES = ["IRN", "USA", "CHN"]

focal_df = (
    df_nlp.loc[
        df_nlp["country_code"].isin(FOCAL_COUNTRIES)
        & df_nlp["year"].between(1972, 2025)
    ]
    .sort_values(["year", "country_code"])
    .reset_index(drop=True)
    .copy()
)

cohort_counts = (
    focal_df.groupby("country_code")["year"]
    .agg(["count", "nunique", "min", "max"])
)

assert len(focal_df) == 162
assert focal_df["text_clean"].notna().all()
assert focal_df["country_code"].value_counts().eq(54).all()

assert not focal_df.duplicated(
    ["country_code", "year"]
).any()

assert set(focal_df["year"]) == set(range(1972, 2026))

print("=" * 70)
print("BALANCED HISTORICAL COHORT")
print("=" * 70)

display(cohort_counts)

# ------------------------------------------------------------
# 2. Fit TF-IDF on the restricted cohort
# ------------------------------------------------------------

focal_vectorizer = TfidfVectorizer(
    **TOPIC_VECTORIZER_CONFIG
)

X_focal_original = focal_vectorizer.fit_transform(
    focal_df["text_clean"]
)

focal_features = focal_vectorizer.get_feature_names_out()

# ------------------------------------------------------------
# 3. Define a limited country-identity feature exclusion
# ------------------------------------------------------------

COUNTRY_IDENTITY_TOKENS = {
    "iran",
    "iranian",
    "iranians",
    "china",
    "chinese",
    "america",
    "american",
    "americans",
    "usa",
    "prc"
}

COUNTRY_IDENTITY_PHRASES = {
    "united states",
    "islamic republic"
}

def is_country_identity_feature(term):

    tokens = term.lower().split()

    return (
        term.lower() in COUNTRY_IDENTITY_PHRASES
        or any(
            token in COUNTRY_IDENTITY_TOKENS
            for token in tokens
        )
    )

excluded_mask = np.array(
    [
        is_country_identity_feature(term)
        for term in focal_features
    ],
    dtype=bool
)

retained_mask = ~excluded_mask

excluded_features = focal_features[excluded_mask]

X_focal_ablated = normalize(
    X_focal_original[:, retained_mask],
    norm="l2",
    axis=1
)

ablated_features = focal_features[retained_mask]

assert X_focal_original.shape[0] == 162
assert X_focal_ablated.shape[0] == 162
assert X_focal_ablated.shape[1] > 10
assert (np.diff(X_focal_ablated.indptr) > 0).all()

print("\n" + "=" * 70)
print("IDENTITY FEATURE ABLATION AUDIT")
print("=" * 70)

print("Original vocabulary:", len(focal_features))
print("Excluded features:", len(excluded_features))
print("Retained features:", len(ablated_features))

print("\nSample excluded features:")
print(excluded_features[:30].tolist())

# ------------------------------------------------------------
# 4. Fit both experimental NMF models
# ------------------------------------------------------------

ABLATION_K = 10
ABLATION_SEED = 11

experimental_spaces = {
    "A_original": (
        X_focal_original,
        focal_features
    ),
    "B_identity_ablated": (
        X_focal_ablated,
        ablated_features
    )
}

nmf_ablation_bank = {}
ablation_topic_records = []
ablation_model_records = []

country_array = focal_df["country_code"].to_numpy()

for model_name, (X_input, terms) in experimental_spaces.items():

    nmf = NMF(
        n_components=ABLATION_K,
        init="nndsvdar",
        solver="cd",
        beta_loss="frobenius",
        max_iter=3000,
        tol=1e-4,
        random_state=ABLATION_SEED
    )

    with warnings.catch_warnings(record=True) as caught:

        warnings.simplefilter(
            "always",
            ConvergenceWarning
        )

        W = nmf.fit_transform(X_input)

    H = nmf.components_

    warned = any(
        issubclass(w.category, ConvergenceWarning)
        for w in caught
    )

    converged = (
        nmf.n_iter_ < 3000 and not warned
    )

    # Relative reconstruction contribution by component.
    component_mass = H.sum(axis=1)

    contributions = W * component_mass[np.newaxis, :]

    totals = contributions.sum(axis=1, keepdims=True)

    assert (totals > 0).all()

    shares = contributions / totals

    dominant = np.argmax(shares, axis=1)

    nmf_ablation_bank[model_name] = {
        "model": nmf,
        "W": W,
        "H": H,
        "shares": shares,
        "dominant": dominant,
        "terms": terms,
        "converged": converged
    }

    ablation_model_records.append({
        "model": model_name,
        "documents": X_input.shape[0],
        "features": X_input.shape[1],
        "iterations": nmf.n_iter_,
        "converged": converged
    })

    for topic_idx in range(ABLATION_K):

        selected = dominant == topic_idx

        n_dominant = int(selected.sum())

        dominant_countries = (
            pd.Series(country_array[selected])
            .value_counts()
        )

        dominant_majority_pct = (
            100 * dominant_countries.max() / n_dominant
            if n_dominant else np.nan
        )

        # Unlike dominant assignments, these values use
        # contributions from every speech.
        country_mass = np.array([
            shares[
                country_array == code,
                topic_idx
            ].sum()
            for code in FOCAL_COUNTRIES
        ])

        country_mass_pct = (
            100 * country_mass / country_mass.sum()
        )

        top_indices = np.argsort(
            H[topic_idx]
        )[::-1][:10]

        ablation_topic_records.append({
            "model": model_name,
            "topic_id": topic_idx + 1,
            "n_dominant": n_dominant,
            "n_dominant_countries": len(dominant_countries),
            "dominant_majority_pct": dominant_majority_pct,
            "soft_majority_country_pct": float(
                country_mass_pct.max()
            ),
            "top_soft_country": FOCAL_COUNTRIES[
                int(country_mass_pct.argmax())
            ],
            "top_terms": ", ".join(
                terms[top_indices]
            )
        })

# ------------------------------------------------------------
# 5. Construct final diagnostic tables
# ------------------------------------------------------------

ablation_models_df = pd.DataFrame(
    ablation_model_records
)

ablation_topics_df = pd.DataFrame(
    ablation_topic_records
)

ablation_summary = (
    ablation_topics_df.groupby("model")
    .agg(
        n_topics=("topic_id", "size"),
        single_country_dominant_topics=(
            "n_dominant_countries",
            lambda x: int((x == 1).sum())
        ),
        mean_soft_majority_pct=(
            "soft_majority_country_pct", "mean"
        ),
        median_soft_majority_pct=(
            "soft_majority_country_pct", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 6. Validate experimental outputs
# ------------------------------------------------------------

assert len(ablation_models_df) == 2
assert len(ablation_topics_df) == 20

assert ablation_topics_df[
    "soft_majority_country_pct"
].between(0, 100).all()

assert (
    ablation_topics_df.groupby("model")["n_dominant"]
    .sum()
    .eq(162)
    .all()
)

# ------------------------------------------------------------
# 7. Display results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CONTROLLED NMF IDENTITY ABLATION RESULTS")
print("=" * 70)

print("\nModel fitting status:")
display(ablation_models_df)

print("\nCountry concentration comparison:")
display(ablation_summary.round(2))

print("\nTopic-level diagnostics:")

for model_name in experimental_spaces:

    print(f"\nMODEL: {model_name}")

    display(
        ablation_topics_df.loc[
            ablation_topics_df["model"] == model_name
        ].round(2)
    )

print("\nIdentity ablation experiment: COMPLETED")

if ablation_models_df["converged"].all():
    print("All experimental models converged: YES")
else:
    print("Model convergence requires review.")



## 12.7. Sentence-Aware Speech Segmentation and Integrity Audit

### Objective

Construct shorter textual units for exploratory segment-level topic modeling while preserving the complete normalized text of every original speech.

### Segmentation strategy

- Source: Balanced three-country corpus, 1972–2025.
- Target segment length: Approximately 250 whitespace-delimited words.
- Segmentation method: spaCy English sentence boundary detection.
- Adjacent sentences are grouped into non-overlapping segments.
- Short final segments are merged with the preceding segment.
- All normalized source text is retained.

### Data integrity requirements

1. Preserve all 162 original speeches.
2. Maintain country, year, and source-document identifiers.
3. Verify exact reconstruction of each normalized speech from its ordered segments.
4. Verify that word counts remain unchanged.
5. Report segment-length distributions and country composition.
6. Identify unusually long segments for further inspection.

### Statistical considerations

Segments from the same speech are correlated and must not be treated as independent country-year observations.

Segment-level topic representations will eventually be aggregated to the original speech level before inferential comparisons.

Segment length, document weighting, and historical composition will be evaluated before selecting the final thematic model.

No topic modeling is performed in this step.


In [ ]:

# ============================================================
# STEP 12.7: LOSSLESS SENTENCE-AWARE SPEECH SEGMENTATION
# ============================================================



# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

TARGET_SEGMENT_WORDS = 250
MIN_SEGMENT_WORDS = 100
LONG_SEGMENT_THRESHOLD = 400

EXPECTED_COUNTRIES = {"IRN", "USA", "CHN"}
EXPECTED_SPEECHES = 162

# ------------------------------------------------------------
# 2. Validate the source dataset
# ------------------------------------------------------------

required_columns = {
    "country_code",
    "year",
    "archive_path",
    "text_clean"
}

assert required_columns.issubset(focal_df.columns)

assert len(focal_df) == EXPECTED_SPEECHES

assert set(
    focal_df["country_code"].unique()
) == EXPECTED_COUNTRIES

assert focal_df["text_clean"].notna().all()

assert focal_df["text_clean"].map(
    lambda x: isinstance(x, str) and bool(x.split())
).all()

assert not focal_df.duplicated(
    ["country_code", "year"]
).any()

source_texts = focal_df[
    "text_clean"
].tolist()

# ------------------------------------------------------------
# 3. Initialize sentence boundary detection
# ------------------------------------------------------------

sentence_nlp = spacy.blank("en")

sentence_nlp.add_pipe("sentencizer")

sentence_nlp.max_length = max(
    1_000_000,
    max(len(text) for text in source_texts) + 1000
)

# ------------------------------------------------------------
# 4. Define a lossless segmentation function
# ------------------------------------------------------------

def segment_speech(
    text,
    sentence_doc,
    target_words=TARGET_SEGMENT_WORDS,
    min_words=MIN_SEGMENT_WORDS
):
    """
    Divide a speech into sentence-aware, non-overlapping
    segments while preserving its exact character sequence.

    Only sentence boundaries following whitespace are used
    as potential split points.

    Short final segments are merged with the preceding
    segment without modifying or removing any characters.

    Returns
    -------
    list[str]
        Ordered, non-overlapping text segments.
    """

    if sentence_doc.text != text:
        raise ValueError(
            "spaCy document differs from source text."
        )

    if not text.split():
        raise ValueError(
            "Cannot segment an empty speech."
        )

    segments = []
    current_start = 0

    # --------------------------------------------------------
    # Identify safe sentence boundaries
    # --------------------------------------------------------

    for sentence in sentence_doc.sents:

        boundary = sentence.start_char

        if not (0 < boundary < len(text)):
            continue

        # A split is allowed only after whitespace.
        if not text[boundary - 1].isspace():
            continue

        current_text = text[current_start:boundary]

        current_words = len(
            current_text.split()
        )

        next_sentence_words = len(
            sentence.text.split()
        )

        should_split = (
            current_words >= min_words
            and current_words + next_sentence_words
            > target_words
        )

        if should_split:

            segments.append(
                text[current_start:boundary]
            )

            current_start = boundary

    # --------------------------------------------------------
    # Append all remaining source characters
    # --------------------------------------------------------

    segments.append(
        text[current_start:]
    )

    # --------------------------------------------------------
    # Safely merge a short final segment
    # --------------------------------------------------------

    if len(segments) > 1:

        final_word_count = len(
            segments[-1].split()
        )

        if final_word_count < min_words:

            final_segment = segments.pop()

            segments[-1] = (
                segments[-1] + final_segment
            )

    # --------------------------------------------------------
    # Strict reconstruction checks
    # --------------------------------------------------------

    reconstructed = "".join(segments)

    if reconstructed != text:
        raise AssertionError(
            "Exact character reconstruction failed."
        )

    original_words = len(text.split())

    segmented_words = sum(
        len(segment.split())
        for segment in segments
    )

    if original_words != segmented_words:
        raise AssertionError(
            f"Word count mismatch: "
            f"original={original_words}, "
            f"segmented={segmented_words}"
        )

    if any(
        not segment.split()
        for segment in segments
    ):
        raise AssertionError(
            "An empty segment was generated."
        )

    if any(
        not segment[-1].isspace()
        for segment in segments[:-1]
    ):
        raise AssertionError(
            "An unsafe segment boundary was generated."
        )

    return segments


# ------------------------------------------------------------
# 5. Independent segmentation unit tests
# ------------------------------------------------------------

test_text = (
    "Alpha beta gamma delta epsilon. "
    "Zeta eta theta iota kappa. "
    "Lambda mu nu xi omicron. "
    "Pi rho."
)

test_segments = segment_speech(
    test_text,
    sentence_nlp(test_text),
    target_words=6,
    min_words=4
)

assert len(test_segments) == 3

assert "".join(test_segments) == test_text

assert sum(
    len(segment.split())
    for segment in test_segments
) == len(test_text.split())

# Verify irregular spacing and missing spaces after punctuation.
test_text_2 = (
    "This is important.The next sentence follows. "
    "Another sentence is here.\n\n"
    "The final sentence completes the example."
)

test_segments_2 = segment_speech(
    test_text_2,
    sentence_nlp(test_text_2),
    target_words=8,
    min_words=3
)

assert "".join(test_segments_2) == test_text_2

assert sum(
    len(segment.split())
    for segment in test_segments_2
) == len(test_text_2.split())

print("Segmentation unit tests: PASSED")

# ------------------------------------------------------------
# 6. Segment the complete historical cohort
# ------------------------------------------------------------

segment_records = []
speech_audit_records = []

processed_documents = sentence_nlp.pipe(
    source_texts,
    batch_size=8,
    n_process=1
)

for speech_id, sentence_doc in enumerate(
    processed_documents
):

    speech = focal_df.iloc[speech_id]

    original_text = source_texts[speech_id]

    try:

        speech_segments = segment_speech(
            original_text,
            sentence_doc
        )

    except Exception as exc:

        raise RuntimeError(
            f"Segmentation failed for "
            f"speech_id={speech_id}, "
            f"country={speech['country_code']}, "
            f"year={speech['year']}, "
            f"source={speech['archive_path']}"
        ) from exc

    # --------------------------------------------------------
    # Track character offsets and document metadata
    # --------------------------------------------------------

    character_cursor = 0

    for segment_id, segment_text in enumerate(
        speech_segments,
        start=1
    ):

        start_char = character_cursor

        end_char = (
            start_char + len(segment_text)
        )

        assert (
            original_text[start_char:end_char]
            == segment_text
        )

        segment_records.append({
            "speech_id": speech_id,
            "segment_id": segment_id,
            "country_code": speech["country_code"],
            "year": int(speech["year"]),
            "archive_path": speech["archive_path"],
            "start_char": start_char,
            "end_char": end_char,
            "segment_text": segment_text,
            "segment_words": len(
                segment_text.split()
            )
        })

        character_cursor = end_char

    assert character_cursor == len(original_text)

    # --------------------------------------------------------
    # Record speech-level preservation checks
    # --------------------------------------------------------

    original_word_count = len(
        original_text.split()
    )

    segmented_word_count = sum(
        len(segment.split())
        for segment in speech_segments
    )

    speech_audit_records.append({
        "speech_id": speech_id,
        "country_code": speech["country_code"],
        "year": int(speech["year"]),
        "archive_path": speech["archive_path"],
        "original_words": original_word_count,
        "segmented_words": segmented_word_count,
        "original_characters": len(original_text),
        "segmented_characters": sum(
            len(segment)
            for segment in speech_segments
        ),
        "n_segments": len(speech_segments),
        "exact_text_preserved": (
            "".join(speech_segments)
            == original_text
        )
    })

# ------------------------------------------------------------
# 7. Construct analytical datasets
# ------------------------------------------------------------

segments_df = pd.DataFrame(
    segment_records
)

speech_segmentation_audit = pd.DataFrame(
    speech_audit_records
)

segments_df = segments_df.sort_values(
    ["speech_id", "segment_id"]
).reset_index(drop=True)

speech_segmentation_audit = (
    speech_segmentation_audit
    .sort_values("speech_id")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 8. Validate global dataset integrity
# ------------------------------------------------------------

assert len(speech_segmentation_audit) == 162

assert segments_df[
    "speech_id"
].nunique() == 162

assert not segments_df.duplicated(
    ["speech_id", "segment_id"]
).any()

assert speech_segmentation_audit[
    "exact_text_preserved"
].all()

assert (
    speech_segmentation_audit["original_words"]
    == speech_segmentation_audit["segmented_words"]
).all()

assert (
    speech_segmentation_audit["original_characters"]
    == speech_segmentation_audit["segmented_characters"]
).all()

# Verify speech-level word-count reconciliation.
segment_word_totals = (
    segments_df
    .groupby("speech_id")["segment_words"]
    .sum()
    .sort_index()
)

original_word_totals = (
    speech_segmentation_audit
    .set_index("speech_id")["original_words"]
    .sort_index()
)

assert segment_word_totals.equals(
    original_word_totals.rename("segment_words")
)

# Independently reconstruct every speech.
reconstructed_speeches = (
    segments_df
    .groupby("speech_id", sort=True)["segment_text"]
    .agg("".join)
)

assert all(
    reconstructed_speeches.loc[speech_id]
    == source_texts[speech_id]
    for speech_id in range(len(source_texts))
)

# ------------------------------------------------------------
# 9. Calculate segment-length diagnostics
# ------------------------------------------------------------

length_summary = (
    segments_df["segment_words"]
    .describe(
        percentiles=[
            0.05,
            0.25,
            0.50,
            0.75,
            0.95
        ]
    )
    .to_frame("segment_words")
)

# ------------------------------------------------------------
# 10. Calculate country-level segment composition
# ------------------------------------------------------------

country_segment_summary = (
    speech_segmentation_audit
    .groupby("country_code")
    .agg(
        n_speeches=("speech_id", "size"),
        n_segments=("n_segments", "sum"),
        total_words=("original_words", "sum"),
        mean_segments_per_speech=(
            "n_segments", "mean"
        ),
        median_segments_per_speech=(
            "n_segments", "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 11. Identify unusually long segments
# ------------------------------------------------------------

long_segments = (
    segments_df.loc[
        segments_df["segment_words"]
        > LONG_SEGMENT_THRESHOLD,
        [
            "speech_id",
            "segment_id",
            "country_code",
            "year",
            "segment_words",
            "start_char",
            "end_char"
        ]
    ]
    .sort_values(
        "segment_words",
        ascending=False
    )
)

# ------------------------------------------------------------
# 12. Display segmentation results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SENTENCE-AWARE SEGMENTATION AUDIT")
print("=" * 70)

print(f"Original speeches: {len(focal_df)}")
print(f"Generated segments: {len(segments_df)}")
print(f"Target segment size: {TARGET_SEGMENT_WORDS} words")

print("\nSegment length distribution:")
display(length_summary.round(2))

print("\nCountry-level segment composition:")
display(country_segment_summary.round(2))

print("\nLong segments requiring inspection:")
print("Number of long segments:", len(long_segments))

display(long_segments.head(15))

# ------------------------------------------------------------
# 13. Display sample segment previews
# ------------------------------------------------------------

print("\nSample segment previews:")

preview = segments_df.head(6).copy()

preview["text_preview"] = (
    preview["segment_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 140)
)

display(
    preview[
        [
            "country_code",
            "year",
            "segment_id",
            "segment_words",
            "text_preview"
        ]
    ]
)

# ------------------------------------------------------------
# 14. Final integrity report
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SEGMENTATION INTEGRITY VALIDATION")
print("=" * 70)

print(
    "Original speeches preserved:",
    speech_segmentation_audit.shape[0] == 162
)

print(
    "Exact text reconstruction:",
    speech_segmentation_audit[
        "exact_text_preserved"
    ].all()
)

print(
    "Word counts preserved:",
    (
        speech_segmentation_audit["original_words"]
        == speech_segmentation_audit["segmented_words"]
    ).all()
)

print(
    "Character counts preserved:",
    (
        speech_segmentation_audit["original_characters"]
        == speech_segmentation_audit["segmented_characters"]
    ).all()
)

print(
    "Unique speech-segment identifiers:",
    not segments_df.duplicated(
        ["speech_id", "segment_id"]
    ).any()
)

print("\nSEGMENTATION INTEGRITY: PASSED")



## 12.8. Speech-Balanced Segment-Level NMF

### Objective

Evaluate whether sentence-aware segmentation improves the recovery of shared thematic components in UN General Debate speeches.

### Experimental design

- Corpus: 162 speeches from Iran, China, and the United States, 1972–2025.
- Segmentation: 2,344 non-overlapping, sentence-aware text segments.
- Pilot topic count: K = 10.
- Vocabulary and IDF: Fitted on the original 162 speeches.
- Feature transformation: Apply the speech-level TF-IDF vectorizer to individual segments.
- Optimization: Non-negative Matrix Factorization with speech-balanced reconstruction weights.

### Speech balancing

Each speech contributes equal total weight to the NMF reconstruction objective, irrespective of its number of segments.

The weighted matrix is constructed by multiplying each segment vector by the square root of its assigned reconstruction weight.

### Speech-level aggregation

Segment-level component contributions are normalized for descriptive purposes and aggregated using segment word counts.

The resulting speech-level representations are not calibrated topic probabilities.

### Evaluation

Compare the original full-speech baseline with the segment-level model using:

1. Country concentration of component contributions.
2. Number of countries represented among dominant-topic speeches.
3. Topic vocabulary and qualitative interpretability.
4. Numerical convergence.

### Limitations

K = 10 is a diagnostic pilot configuration, not a selected final topic count.

Differences in reconstruction error across differently constructed training matrices are not directly interpreted as improvements in thematic validity.

Further multi-seed stability, coherence, segment-length sensitivity, and qualitative validation are required.


In [ ]:

# ============================================================
# STEP 12.8: SPEECH-BALANCED SEGMENT-LEVEL NMF
# ============================================================



# ------------------------------------------------------------
# 1. Experimental configuration
# ------------------------------------------------------------

SEGMENT_NMF_K = 10
SEGMENT_NMF_SEED = 11
SEGMENT_NMF_MAX_ITER = 3000

N_SPEECHES = len(focal_df)
N_SEGMENTS = len(segments_df)

COUNTRY_CODES = ["IRN", "USA", "CHN"]

# ------------------------------------------------------------
# 2. Validate speech-segment alignment
# ------------------------------------------------------------

assert N_SPEECHES == 162
assert N_SEGMENTS == 2344

speech_ids = segments_df[
    "speech_id"
].to_numpy(dtype=int)

assert speech_ids.min() == 0
assert speech_ids.max() == N_SPEECHES - 1

assert np.array_equal(
    segments_df["country_code"].to_numpy(),
    focal_df.iloc[speech_ids][
        "country_code"
    ].to_numpy()
)

assert np.array_equal(
    segments_df["year"].to_numpy(),
    focal_df.iloc[speech_ids]["year"].to_numpy()
)

# ------------------------------------------------------------
# 3. Transform segments using the speech-level vocabulary
# ------------------------------------------------------------

X_segment_tfidf = focal_vectorizer.transform(
    segments_df["segment_text"]
).tocsr()

segment_features = (
    focal_vectorizer.get_feature_names_out()
)

assert X_segment_tfidf.shape == (
    N_SEGMENTS,
    X_focal_original.shape[1]
)

assert np.array_equal(
    segment_features,
    focal_features
)

assert np.isfinite(
    X_segment_tfidf.data
).all()

assert np.all(
    X_segment_tfidf.data >= 0
)

nonzero_features_per_segment = np.diff(
    X_segment_tfidf.indptr
)

zero_feature_segments = int(
    np.sum(nonzero_features_per_segment == 0)
)

print("=" * 70)
print("SEGMENT-LEVEL FEATURE AUDIT")
print("=" * 70)

print("Original speeches:", N_SPEECHES)
print("Text segments:", N_SEGMENTS)
print("Shared vocabulary:", X_segment_tfidf.shape[1])
print("Nonzero feature values:", X_segment_tfidf.nnz)
print("Zero-feature segments:", zero_feature_segments)

if zero_feature_segments:
    raise ValueError(
        "Some segments have no features in the shared "
        "vocabulary. Review them before model fitting."
    )

# ------------------------------------------------------------
# 4. Assign equal total reconstruction weight per speech
# ------------------------------------------------------------

segments_per_speech = np.bincount(
    speech_ids,
    minlength=N_SPEECHES
)

assert np.all(segments_per_speech > 0)

segment_weights = (
    1.0 / segments_per_speech[speech_ids]
)

speech_weight_totals = np.bincount(
    speech_ids,
    weights=segment_weights,
    minlength=N_SPEECHES
)

assert np.allclose(
    speech_weight_totals,
    1.0,
    atol=1e-12
)

assert np.isclose(
    segment_weights.sum(),
    N_SPEECHES
)

# Weighted Frobenius reconstruction:
# Each row is multiplied by sqrt(1 / segments_per_speech).
X_segment_weighted = (
    X_segment_tfidf.multiply(
        np.sqrt(segment_weights)[:, np.newaxis]
    )
    .tocsr()
)

# ------------------------------------------------------------
# 5. Fit the weighted segment-level NMF
# ------------------------------------------------------------

segment_nmf_model = NMF(
    n_components=SEGMENT_NMF_K,
    init="nndsvdar",
    solver="cd",
    beta_loss="frobenius",
    max_iter=SEGMENT_NMF_MAX_ITER,
    tol=1e-4,
    random_state=SEGMENT_NMF_SEED
)

with warnings.catch_warnings(record=True) as caught:

    warnings.simplefilter(
        "always",
        ConvergenceWarning
    )

    W_weighted = segment_nmf_model.fit_transform(
        X_segment_weighted
    )

H_segment = segment_nmf_model.components_

convergence_warning = any(
    issubclass(w.category, ConvergenceWarning)
    for w in caught
)

segment_nmf_converged = (
    segment_nmf_model.n_iter_ < SEGMENT_NMF_MAX_ITER
    and not convergence_warning
)

weighted_matrix_norm = float(
    np.sqrt(
        X_segment_weighted.multiply(
            X_segment_weighted
        ).sum()
    )
)

segment_relative_error = (
    segment_nmf_model.reconstruction_err_
    / weighted_matrix_norm
)

print("\n" + "=" * 70)
print("SPEECH-BALANCED SEGMENT NMF FITTING")
print("=" * 70)

print("Number of topics:", SEGMENT_NMF_K)
print("Iterations:", segment_nmf_model.n_iter_)
print("Converged:", segment_nmf_converged)
print(
    "Weighted relative reconstruction error:",
    round(segment_relative_error, 6)
)

# ------------------------------------------------------------
# 6. Obtain segment representations on the original scale
# ------------------------------------------------------------

# The model was fitted on weighted TF-IDF rows.
# Unweighted rows are transformed using the fitted components.

W_segment = segment_nmf_model.transform(
    X_segment_tfidf
)

assert W_segment.shape == (
    N_SEGMENTS,
    SEGMENT_NMF_K
)

assert np.isfinite(W_segment).all()
assert np.all(W_segment >= 0)

assert np.isfinite(H_segment).all()
assert np.all(H_segment >= 0)

# ------------------------------------------------------------
# 7. Calculate relative reconstruction contributions
# ------------------------------------------------------------

component_mass = H_segment.sum(axis=1)

segment_contributions = (
    W_segment * component_mass[np.newaxis, :]
)

segment_contribution_totals = (
    segment_contributions.sum(axis=1, keepdims=True)
)

assert np.all(
    segment_contribution_totals > 0
)

segment_topic_shares = (
    segment_contributions
    / segment_contribution_totals
)

assert np.allclose(
    segment_topic_shares.sum(axis=1),
    1.0,
    atol=1e-8
)

# ------------------------------------------------------------
# 8. Aggregate segment contributions to speeches
# ------------------------------------------------------------

segment_word_counts = segments_df[
    "segment_words"
].to_numpy(dtype=float)

speech_topic_mass = np.zeros(
    (N_SPEECHES, SEGMENT_NMF_K),
    dtype=float
)

np.add.at(
    speech_topic_mass,
    speech_ids,
    segment_topic_shares
    * segment_word_counts[:, np.newaxis]
)

speech_word_totals = np.bincount(
    speech_ids,
    weights=segment_word_counts,
    minlength=N_SPEECHES
)

speech_topic_shares = (
    speech_topic_mass
    / speech_word_totals[:, np.newaxis]
)

assert speech_topic_shares.shape == (
    N_SPEECHES,
    SEGMENT_NMF_K
)

assert np.isfinite(speech_topic_shares).all()

assert np.allclose(
    speech_topic_shares.sum(axis=1),
    1.0,
    atol=1e-8
)

# ------------------------------------------------------------
# 9. Define country concentration diagnostics
# ------------------------------------------------------------

def country_topic_diagnostics(
    topic_shares,
    country_codes,
    model_name
):
    """
    Evaluate country concentration using speech-level
    normalized reconstruction contributions.
    """

    records = []

    dominant_topics = np.argmax(
        topic_shares,
        axis=1
    )

    for topic_idx in range(topic_shares.shape[1]):

        assigned = dominant_topics == topic_idx

        assigned_countries = np.unique(
            country_codes[assigned]
        )

        country_mass = np.array([
            topic_shares[
                country_codes == code,
                topic_idx
            ].sum()
            for code in COUNTRY_CODES
        ])

        country_mass_pct = (
            100 * country_mass / country_mass.sum()
        )

        leading_idx = int(
            np.argmax(country_mass_pct)
        )

        records.append({
            "model": model_name,
            "topic_id": topic_idx + 1,
            "n_dominant_speeches": int(assigned.sum()),
            "n_dominant_countries": len(
                assigned_countries
            ),
            "leading_country": COUNTRY_CODES[
                leading_idx
            ],
            "soft_majority_pct": float(
                country_mass_pct[leading_idx]
            ),
            "iran_mass_pct": float(country_mass_pct[0]),
            "usa_mass_pct": float(country_mass_pct[1]),
            "china_mass_pct": float(country_mass_pct[2])
        })

    return pd.DataFrame(records)


# ------------------------------------------------------------
# 10. Compare with the original speech-level NMF
# ------------------------------------------------------------

country_array = focal_df[
    "country_code"
].to_numpy()

original_speech_shares = (
    nmf_ablation_bank["A_original"]["shares"]
)

assert original_speech_shares.shape == (
    N_SPEECHES,
    SEGMENT_NMF_K
)

original_topic_diagnostics = (
    country_topic_diagnostics(
        original_speech_shares,
        country_array,
        "A_original_speech_NMF"
    )
)

segment_topic_diagnostics = (
    country_topic_diagnostics(
        speech_topic_shares,
        country_array,
        "C_balanced_segment_NMF"
    )
)

comparison_df = pd.concat(
    [
        original_topic_diagnostics,
        segment_topic_diagnostics
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 11. Summarize country concentration
# ------------------------------------------------------------

model_comparison_summary = (
    comparison_df
    .groupby("model")
    .agg(
        n_topics=("topic_id", "size"),
        single_country_dominant_topics=(
            "n_dominant_countries",
            lambda x: int((x == 1).sum())
        ),
        topics_dominant_in_all_three=(
            "n_dominant_countries",
            lambda x: int((x == 3).sum())
        ),
        mean_soft_majority_pct=(
            "soft_majority_pct",
            "mean"
        ),
        median_soft_majority_pct=(
            "soft_majority_pct",
            "median"
        )
    )
    .reset_index()
)

# ------------------------------------------------------------
# 12. Inspect the segment-level thematic vocabulary
# ------------------------------------------------------------

segment_topic_vocabulary = []

for topic_idx in range(SEGMENT_NMF_K):

    top_indices = np.argsort(
        H_segment[topic_idx]
    )[::-1][:12]

    segment_topic_vocabulary.append({
        "topic_id": topic_idx + 1,
        "top_terms": ", ".join(
            segment_features[top_indices]
        )
    })

segment_vocabulary_df = pd.DataFrame(
    segment_topic_vocabulary
)

# ------------------------------------------------------------
# 13. Validate diagnostic tables
# ------------------------------------------------------------

assert len(comparison_df) == 20
assert len(model_comparison_summary) == 2
assert len(segment_vocabulary_df) == 10

assert comparison_df[
    "soft_majority_pct"
].between(0, 100).all()

for model_name, group in comparison_df.groupby("model"):

    assert group[
        "n_dominant_speeches"
    ].sum() == N_SPEECHES

    country_pct_sum = (
        group[
            [
                "iran_mass_pct",
                "usa_mass_pct",
                "china_mass_pct"
            ]
        ].sum(axis=1)
    )

    assert np.allclose(
        country_pct_sum,
        100.0,
        atol=1e-8
    )

# ------------------------------------------------------------
# 14. Display final diagnostic outputs
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPEECH-LEVEL VS SEGMENT-LEVEL NMF COMPARISON")
print("=" * 70)

display(
    model_comparison_summary.round(2)
)

print("\nSegment-level country concentration:")

display(
    segment_topic_diagnostics.round(2)
)

print("\n" + "=" * 70)
print("SEGMENT-LEVEL TOPIC VOCABULARY")
print("=" * 70)

for row in segment_vocabulary_df.itertuples(index=False):

    print(f"\nTopic {row.topic_id:02d}")
    print(row.top_terms)

print("\n" + "=" * 70)
print("FINAL PILOT STATUS")
print("=" * 70)

if segment_nmf_converged:
    print("SEGMENT-LEVEL NMF: NUMERICAL CONVERGENCE PASSED")
else:
    print("SEGMENT-LEVEL NMF: CONVERGENCE REVIEW REQUIRED")

print("Speech-level aggregation: PASSED")
print("Country concentration diagnostics: PASSED")
print("Semantic validation: PENDING")



## 12.9. Topic Coverage and Cross-Country Validation

### Objective

Evaluate the distribution and substantive relevance of the ten segment-level NMF components across Iran, the United States, and China.

### Analytical strategy

1. Examine all ten topic vocabularies without output truncation.
2. Quantify country-specific mean topic contributions at the speech level.
3. Estimate topic prevalence using descriptive contribution thresholds.
4. Compare speech-level dominant-topic assignments with segment-level assignments.
5. Calculate normalized country-distribution entropy for each topic.
6. Inspect representative text segments from different countries.

### Interpretation

Normalized country entropy ranges from zero to one, with higher values indicating a more balanced distribution of topic contributions across countries.

A threshold of 0.10 represents the uniform-share benchmark for a ten-component model, not a validated probability cutoff.

A topic can be substantively important even if it is not the dominant component of any complete speech.

### Statistical safeguards

All country comparisons use aggregated speech-level representations.

Segments remain nested within speeches and are not treated as independent observations for statistical inference.

Country balance and thematic coherence are different properties and must be evaluated separately.

No final topic labels, inferential claims, or model-selection decisions are made in this step.


In [ ]:

# ============================================================
# STEP 12.9: CROSS-COUNTRY TOPIC COVERAGE AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

COUNTRY_CODES = ["IRN", "USA", "CHN"]

TOPIC_SHARE_THRESHOLD = 0.10
SEGMENT_SHARE_THRESHOLD = 0.20

N_TOPICS = speech_topic_shares.shape[1]

assert N_TOPICS == 10
assert speech_topic_shares.shape == (162, 10)
assert segment_topic_shares.shape == (2344, 10)

assert np.allclose(
    speech_topic_shares.sum(axis=1),
    1.0,
    atol=1e-8
)

assert np.allclose(
    segment_topic_shares.sum(axis=1),
    1.0,
    atol=1e-8
)

# ------------------------------------------------------------
# 2. Prepare country and topic assignments
# ------------------------------------------------------------

speech_countries = focal_df[
    "country_code"
].to_numpy()

segment_countries = segments_df[
    "country_code"
].to_numpy()

speech_primary_topics = np.argmax(
    speech_topic_shares,
    axis=1
)

segment_primary_topics = np.argmax(
    segment_topic_shares,
    axis=1
)

# ------------------------------------------------------------
# 3. Calculate country distribution entropy
# ------------------------------------------------------------

def normalized_country_entropy(country_proportions):
    """
    Calculate normalized Shannon entropy across countries.

    Returns a value between 0 and 1.
    """

    p = np.asarray(
        country_proportions,
        dtype=float
    )

    assert np.isclose(p.sum(), 1.0)

    positive = p[p > 0]

    entropy = -np.sum(
        positive * np.log(positive)
    )

    return float(
        entropy / np.log(len(p))
    )


# ------------------------------------------------------------
# 4. Calculate topic-level coverage
# ------------------------------------------------------------

coverage_records = []

for topic_idx in range(N_TOPICS):

    speech_shares = speech_topic_shares[
        :, topic_idx
    ]

    segment_shares = segment_topic_shares[
        :, topic_idx
    ]

    country_topic_mass = np.array([
        speech_shares[
            speech_countries == country
        ].sum()
        for country in COUNTRY_CODES
    ])

    country_proportions = (
        country_topic_mass
        / country_topic_mass.sum()
    )

    country_entropy = normalized_country_entropy(
        country_proportions
    )

    record = {
        "topic_id": topic_idx + 1,
        "n_primary_speeches": int(
            np.sum(
                speech_primary_topics == topic_idx
            )
        ),
        "n_primary_segments": int(
            np.sum(
                segment_primary_topics == topic_idx
            )
        ),
        "n_speeches_share_ge_10pct": int(
            np.sum(
                speech_shares >= TOPIC_SHARE_THRESHOLD
            )
        ),
        "n_segments_share_ge_20pct": int(
            np.sum(
                segment_shares >= SEGMENT_SHARE_THRESHOLD
            )
        ),
        "country_entropy": country_entropy,
        "effective_countries": float(
            3 ** country_entropy
        ),
        "country_majority_pct": float(
            100 * country_proportions.max()
        )
    }

    for country in COUNTRY_CODES:

        mask = speech_countries == country

        country_values = speech_shares[mask]

        record[
            f"{country}_mean_share"
        ] = float(country_values.mean())

        record[
            f"{country}_speeches_ge_10pct"
        ] = int(
            np.sum(
                country_values >= TOPIC_SHARE_THRESHOLD
            )
        )

    coverage_records.append(record)

topic_coverage_df = pd.DataFrame(
    coverage_records
)

# ------------------------------------------------------------
# 5. Validate country-level topic summaries
# ------------------------------------------------------------

assert len(topic_coverage_df) == N_TOPICS

assert topic_coverage_df[
    "country_entropy"
].between(0, 1).all()

assert topic_coverage_df[
    "effective_countries"
].between(1, 3).all()

assert (
    topic_coverage_df["n_primary_speeches"].sum()
    == len(focal_df)
)

assert (
    topic_coverage_df["n_primary_segments"].sum()
    == len(segments_df)
)

for country in COUNTRY_CODES:

    assert np.isclose(
        topic_coverage_df[
            f"{country}_mean_share"
        ].sum(),
        1.0
    )

# ------------------------------------------------------------
# 6. Display overall topic coverage
# ------------------------------------------------------------

print("=" * 70)
print("SEGMENT-LEVEL NMF: TOPIC COVERAGE AUDIT")
print("=" * 70)

summary_columns = [
    "topic_id",
    "n_primary_speeches",
    "n_primary_segments",
    "n_speeches_share_ge_10pct",
    "n_segments_share_ge_20pct",
    "country_entropy",
    "effective_countries",
    "country_majority_pct"
]

display(
    topic_coverage_df[
        summary_columns
    ].round(3)
)

# ------------------------------------------------------------
# 7. Display country-level mean contributions
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COUNTRY-LEVEL MEAN TOPIC CONTRIBUTIONS")
print("=" * 70)

country_columns = [
    "topic_id",
    "IRN_mean_share",
    "USA_mean_share",
    "CHN_mean_share",
    "country_entropy"
]

display(
    topic_coverage_df[
        country_columns
    ].round(4)
)

# ------------------------------------------------------------
# 8. Display speech-level topic prevalence
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("SPEECH-LEVEL TOPIC PREVALENCE")
print("=" * 70)

prevalence_columns = [
    "topic_id",
    "IRN_speeches_ge_10pct",
    "USA_speeches_ge_10pct",
    "CHN_speeches_ge_10pct"
]

display(
    topic_coverage_df[
        prevalence_columns
    ]
)

# ------------------------------------------------------------
# 9. Display all ten topic vocabularies
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPLETE SEGMENT-LEVEL TOPIC VOCABULARY")
print("=" * 70)

for topic_idx in range(N_TOPICS):

    top_indices = np.argsort(
        H_segment[topic_idx]
    )[::-1][:15]

    top_terms = segment_features[
        top_indices
    ].tolist()

    print(f"\nTOPIC {topic_idx + 1:02d}")

    print(
        ", ".join(top_terms)
    )

# ------------------------------------------------------------
# 10. Identify topics for qualitative inspection
# ------------------------------------------------------------

INSPECTION_TOPICS = [4, 6, 8, 9]

print("\n" + "=" * 70)
print("CROSS-COUNTRY REPRESENTATIVE SEGMENTS")
print("=" * 70)

for topic_id in INSPECTION_TOPICS:

    topic_idx = topic_id - 1

    print(f"\n{'=' * 60}")
    print(f"TOPIC {topic_id:02d}")
    print("=" * 60)

    for country in COUNTRY_CODES:

        country_indices = np.flatnonzero(
            segment_countries == country
        )

        country_topic_scores = (
            segment_topic_shares[
                country_indices,
                topic_idx
            ]
        )

        best_local_idx = int(
            np.argmax(country_topic_scores)
        )

        best_global_idx = int(
            country_indices[best_local_idx]
        )

        best_segment = segments_df.iloc[
            best_global_idx
        ]

        best_score = float(
            country_topic_scores[best_local_idx]
        )

        segment_excerpt = " ".join(
            str(
                best_segment["segment_text"]
            ).split()
        )

        print(
            f"\nCountry: {country}"
            f" | Year: {int(best_segment['year'])}"
            f" | Speech: {int(best_segment['speech_id'])}"
            f" | Segment: {int(best_segment['segment_id'])}"
        )

        print(
            f"Relative topic contribution: "
            f"{best_score:.4f}"
        )

        print(
            "Excerpt:",
            segment_excerpt[:500]
        )

        if len(segment_excerpt) > 500:
            print("[Excerpt truncated]")

# ------------------------------------------------------------
# 11. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("TOPIC COVERAGE VALIDATION")
print("=" * 70)

print("Topics evaluated:", len(topic_coverage_df))

print(
    "Speech assignments:",
    topic_coverage_df[
        "n_primary_speeches"
    ].sum()
)

print(
    "Segment assignments:",
    topic_coverage_df[
        "n_primary_segments"
    ].sum()
)

print("\nTopic coverage calculations: PASSED")
print("Qualitative interpretation: PENDING REVIEW")



### 12.9.1. Country-Level Topic Contribution Heatmap

This diagnostic heatmap compares the mean relative contribution of each NMF component across Iran, the United States, and China.

Each country has 54 speeches, and topic contributions are aggregated at the original speech level.

The visualization is exploratory. Topic labels and the final model specification remain subject to coherence, stability, and substantive validation.


In [ ]:

# ============================================================
# STEP 12.9.1: COUNTRY-TOPIC CONTRIBUTION HEATMAP
# ============================================================


# ------------------------------------------------------------
# 1. Prepare the country-topic matrix
# ------------------------------------------------------------

country_columns = [
    "IRN_mean_share",
    "USA_mean_share",
    "CHN_mean_share"
]

topic_heatmap_df = (
    topic_coverage_df
    .sort_values("topic_id")
    .set_index("topic_id")[country_columns]
    .mul(100)
)

topic_heatmap_df.columns = [
    "Iran",
    "United States",
    "China"
]

assert topic_heatmap_df.shape == (10, 3)
assert np.isfinite(topic_heatmap_df.to_numpy()).all()

# ------------------------------------------------------------
# 2. Generate descriptive topic labels
# ------------------------------------------------------------

topic_labels = []

for topic_idx in range(H_segment.shape[0]):

    top_indices = np.argsort(
        H_segment[topic_idx]
    )[::-1][:3]

    keywords = ", ".join(
        segment_features[top_indices]
    )

    topic_labels.append(
        f"T{topic_idx + 1:02d} | {keywords}"
    )

# ------------------------------------------------------------
# 3. Draw the diagnostic figure
# ------------------------------------------------------------

fig, ax = plt.subplots(
    figsize=(10.5, 7.0),
    dpi=150
)

values = topic_heatmap_df.to_numpy()

image = ax.imshow(
    values,
    cmap="viridis",
    aspect="auto",
    vmin=0,
    vmax=np.ceil(values.max() / 5) * 5
)

ax.set_xticks(np.arange(3))
ax.set_xticklabels(
    topic_heatmap_df.columns,
    fontsize=11
)

ax.set_yticks(np.arange(10))
ax.set_yticklabels(
    topic_labels,
    fontsize=9
)

ax.set_title(
    "Country-Level NMF Topic Contributions",
    fontsize=14,
    fontweight="bold",
    pad=16
)

ax.set_xlabel(
    "Country",
    fontsize=11,
    labelpad=10
)

ax.set_ylabel(
    "NMF component and leading terms",
    fontsize=11,
    labelpad=10
)

# ------------------------------------------------------------
# 4. Annotate each cell
# ------------------------------------------------------------

scale_max = image.norm.vmax

for row_idx in range(values.shape[0]):

    for col_idx in range(values.shape[1]):

        value = values[row_idx, col_idx]

        annotation_color = (
            "white"
            if value < 0.55 * scale_max
            else "black"
        )

        ax.text(
            col_idx,
            row_idx,
            f"{value:.1f}%",
            ha="center",
            va="center",
            fontsize=9,
            color=annotation_color
        )

# ------------------------------------------------------------
# 5. Add an independent colorbar
# ------------------------------------------------------------

colorbar = fig.colorbar(
    image,
    ax=ax,
    fraction=0.035,
    pad=0.035
)

colorbar.set_label(
    "Mean speech-level topic contribution (%)",
    fontsize=10
)

ax.tick_params(
    axis="both",
    length=0
)

fig.subplots_adjust(
    left=0.32,
    right=0.91,
    top=0.91,
    bottom=0.11
)

plt.show()

print("Diagnostic heatmap: GENERATED")
print("Publication export: DEFERRED")



## 12.10. Topic Coherence Evaluation Using Window-Based NPMI

### Objective

Quantify the lexical coherence of the ten-component segment-level NMF model using normalized pointwise mutual information (NPMI).

### Primary coherence specification

- Corpus: 162 balanced country-year speeches, 1972–2025.
- Evaluation units: Sentence-aware text segments.
- Co-occurrence window: 20 analyzer tokens, sliding by one token.
- Topic representation: Ten highest-weighted unigram features per component.
- Tokenization: Consistent with the fitted TF-IDF vectorizer.
- Window weighting: Equal total probability mass per original speech.

### Methodological rationale

Unigram-only primary coherence avoids mechanically inflated associations between overlapping unigram and bigram features.

The fitted NMF model remains unchanged and continues to incorporate its original unigram and bigram feature space.

Zero-co-occurrence term pairs receive an NPMI score of -1. Unsupported terms are flagged rather than silently smoothed.

### Interpretation safeguards

High coherence does not establish substantive diplomatic relevance.

Country-specific vocabulary may also exhibit high coherence.

Coherence must be evaluated jointly with country concentration, topic prevalence, initialization stability, and qualitative inspection.

This step evaluates the existing K=10 pilot model only. It does not select the final number of topics.


In [ ]:

# ============================================================
# STEP 12.10: SPEECH-BALANCED WINDOW-BASED NPMI
# ============================================================

# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

NPMI_WINDOW_SIZE = 20
NPMI_TOP_WORDS = 10

N_TOPICS = H_segment.shape[0]
N_SPEECHES = len(focal_df)

assert N_TOPICS == 10
assert N_SPEECHES == 162
assert len(segments_df) == 2344

assert focal_vectorizer.analyzer == "word"
assert focal_vectorizer.ngram_range[0] == 1

# ------------------------------------------------------------
# 2. Select the highest-weighted unigram features
# ------------------------------------------------------------

topic_unigrams = {}
all_candidate_terms = set()

for topic_idx in range(N_TOPICS):

    ranked_indices = np.argsort(
        H_segment[topic_idx]
    )[::-1]

    selected_terms = []

    for feature_idx in ranked_indices:

        term = str(
            segment_features[feature_idx]
        )

        # Primary NPMI uses unigram features only.
        if " " in term:
            continue

        selected_terms.append(term)

        if len(selected_terms) == NPMI_TOP_WORDS:
            break

    assert len(selected_terms) == NPMI_TOP_WORDS

    topic_unigrams[topic_idx + 1] = selected_terms

    all_candidate_terms.update(selected_terms)

candidate_terms = sorted(all_candidate_terms)

term_to_index = {
    term: idx
    for idx, term in enumerate(candidate_terms)
}

# ------------------------------------------------------------
# 3. Reproduce the fitted vectorizer's tokenization
# ------------------------------------------------------------

preprocessor = focal_vectorizer.build_preprocessor()
tokenizer = focal_vectorizer.build_tokenizer()
analyzer = focal_vectorizer.build_analyzer()

stop_words = focal_vectorizer.get_stop_words()

if stop_words is None:
    stop_words = set()
else:
    stop_words = set(stop_words)


def tokenize_for_coherence(text):
    """
    Reproduce the unigram token sequence generated by
    the fitted word-level TF-IDF analyzer.
    """

    tokens = tokenizer(
        preprocessor(text)
    )

    return [
        token
        for token in tokens
        if token not in stop_words
    ]


# Validate tokenizer compatibility on sample segments.
for text in segments_df["segment_text"].head(20):

    expected = {
        term
        for term in analyzer(text)
        if (
            " " not in term
            and term in term_to_index
        )
    }

    observed = (
        set(tokenize_for_coherence(text))
        & set(candidate_terms)
    )

    assert expected == observed, (
        "Tokenizer and fitted analyzer are inconsistent."
    )

print("Analyzer compatibility checks: PASSED")

# ------------------------------------------------------------
# 4. Tokenize the complete segment corpus
# ------------------------------------------------------------

tokenized_segments = [
    tokenize_for_coherence(text)
    for text in segments_df["segment_text"]
]

segment_speech_ids = segments_df[
    "speech_id"
].to_numpy(dtype=int)

assert segment_speech_ids.min() == 0
assert segment_speech_ids.max() == N_SPEECHES - 1

assert all(
    len(tokens) > 0
    for tokens in tokenized_segments
)

# ------------------------------------------------------------
# 5. Calculate window counts and balancing weights
# ------------------------------------------------------------

segment_window_counts = np.array(
    [
        max(
            1,
            len(tokens) - NPMI_WINDOW_SIZE + 1
        )
        for tokens in tokenized_segments
    ],
    dtype=int
)

windows_per_speech = np.bincount(
    segment_speech_ids,
    weights=segment_window_counts,
    minlength=N_SPEECHES
)

assert np.all(windows_per_speech > 0)

expected_windows = int(
    segment_window_counts.sum()
)

# ------------------------------------------------------------
# 6. Build sparse window-term presence matrix
# ------------------------------------------------------------

# A term receives a value of one if it occurs at least
# once within a given sliding window.
#
# A rolling counter avoids repeatedly rebuilding the
# complete token set of every overlapping window.

row_indices = []
column_indices = []
window_weights = []

window_id = 0

for segment_idx, tokens in enumerate(
    tokenized_segments
):

    speech_id = segment_speech_ids[
        segment_idx
    ]

    n_windows = int(
        segment_window_counts[segment_idx]
    )

    effective_length = min(
        NPMI_WINDOW_SIZE,
        len(tokens)
    )

    active_terms = Counter()

    # Initialize the first window.
    for token in tokens[:effective_length]:

        term_idx = term_to_index.get(token)

        if term_idx is not None:
            active_terms[term_idx] += 1

    # Every window from a speech receives a share
    # of that speech's total unit weight.
    speech_window_weight = (
        1.0 / windows_per_speech[speech_id]
    )

    for start_idx in range(n_windows):

        # Record unique terms present in the window.
        active_indices = list(active_terms.keys())

        row_indices.extend(
            [window_id] * len(active_indices)
        )

        column_indices.extend(
            active_indices
        )

        window_weights.append(
            speech_window_weight
        )

        window_id += 1

        # Update the rolling window.
        if start_idx + 1 < n_windows:

            outgoing_token = tokens[start_idx]

            incoming_token = tokens[
                start_idx + NPMI_WINDOW_SIZE
            ]

            outgoing_idx = term_to_index.get(
                outgoing_token
            )

            incoming_idx = term_to_index.get(
                incoming_token
            )

            if outgoing_idx is not None:

                active_terms[outgoing_idx] -= 1

                if active_terms[outgoing_idx] == 0:
                    del active_terms[outgoing_idx]

            if incoming_idx is not None:

                active_terms[incoming_idx] += 1

# ------------------------------------------------------------
# 7. Validate the window matrix
# ------------------------------------------------------------

assert window_id == expected_windows

window_weights = np.asarray(
    window_weights,
    dtype=np.float64
)

assert np.isclose(
    window_weights.sum(),
    N_SPEECHES,
    atol=1e-8
)

window_presence = csr_matrix(
    (
        np.ones(
            len(row_indices),
            dtype=np.float64
        ),
        (
            np.asarray(row_indices),
            np.asarray(column_indices)
        )
    ),
    shape=(
        window_id,
        len(candidate_terms)
    )
)

assert window_presence.max() <= 1

# ------------------------------------------------------------
# 8. Estimate speech-balanced co-occurrence probabilities
# ------------------------------------------------------------

weighted_presence = (
    window_presence
    .multiply(
        window_weights[:, np.newaxis]
    )
    .tocsr()
)

joint_probabilities = (
    window_presence.T
    @ weighted_presence
).toarray() / window_weights.sum()

unigram_probabilities = np.diag(
    joint_probabilities
).copy()

assert np.all(
    unigram_probabilities >= 0
)

assert np.all(
    unigram_probabilities <= 1 + 1e-8
)

assert np.allclose(
    joint_probabilities,
    joint_probabilities.T,
    atol=1e-8
)

# ------------------------------------------------------------
# 9. Calculate NPMI without artificial smoothing
# ------------------------------------------------------------

def calculate_npmi(p_i, p_j, p_ij):
    """
    Return window-based normalized PMI.

    Zero co-occurrence is scored as -1.
    Undefined probability cases return NaN.
    """

    if p_i <= 0 or p_j <= 0:
        return np.nan

    if p_ij <= 0:
        return -1.0

    if p_ij >= 1:
        return np.nan

    numerator = np.log(
        p_ij / (p_i * p_j)
    )

    denominator = -np.log(p_ij)

    result = numerator / denominator

    return float(
        np.clip(result, -1.0, 1.0)
    )


# ------------------------------------------------------------
# 10. Calculate topic-level coherence
# ------------------------------------------------------------

topic_npmi_records = []
pairwise_npmi_records = []

for topic_id, terms in topic_unigrams.items():

    topic_pair_scores = []
    zero_cooccurrence_pairs = 0

    topic_term_indices = [
        term_to_index[term]
        for term in terms
    ]

    term_probabilities = (
        unigram_probabilities[
            topic_term_indices
        ]
    )

    for term_a, term_b in combinations(
        terms, 2
    ):

        idx_a = term_to_index[term_a]
        idx_b = term_to_index[term_b]

        p_a = unigram_probabilities[idx_a]
        p_b = unigram_probabilities[idx_b]

        p_ab = joint_probabilities[
            idx_a, idx_b
        ]

        score = calculate_npmi(
            p_a, p_b, p_ab
        )

        if p_ab <= 0:
            zero_cooccurrence_pairs += 1

        topic_pair_scores.append(score)

        pairwise_npmi_records.append({
            "topic_id": topic_id,
            "term_1": term_a,
            "term_2": term_b,
            "npmi": score,
            "joint_probability": float(p_ab)
        })

    topic_pair_scores = np.asarray(
        topic_pair_scores,
        dtype=float
    )

    all_defined = bool(
        np.isfinite(topic_pair_scores).all()
    )

    topic_npmi_records.append({
        "topic_id": topic_id,
        "n_terms": len(terms),
        "n_pairs": len(topic_pair_scores),
        "mean_npmi": (
            float(topic_pair_scores.mean())
            if all_defined else np.nan
        ),
        "median_npmi": (
            float(np.median(topic_pair_scores))
            if all_defined else np.nan
        ),
        "min_npmi": (
            float(topic_pair_scores.min())
            if all_defined else np.nan
        ),
        "zero_cooccurrence_pairs": (
            zero_cooccurrence_pairs
        ),
        "min_term_window_prevalence_pct": float(
            100 * term_probabilities.min()
        ),
        "top_unigrams": ", ".join(terms)
    })

topic_npmi_df = pd.DataFrame(
    topic_npmi_records
)

pairwise_npmi_df = pd.DataFrame(
    pairwise_npmi_records
)

# ------------------------------------------------------------
# 11. Combine coherence and topic coverage diagnostics
# ------------------------------------------------------------

topic_quality_df = topic_npmi_df.merge(
    topic_coverage_df[
        [
            "topic_id",
            "n_primary_speeches",
            "n_primary_segments",
            "n_speeches_share_ge_10pct",
            "country_entropy",
            "country_majority_pct"
        ]
    ],
    on="topic_id",
    how="left",
    validate="one_to_one"
)

# ------------------------------------------------------------
# 12. Validate coherence calculations
# ------------------------------------------------------------

assert len(topic_npmi_df) == N_TOPICS

assert len(pairwise_npmi_df) == (
    N_TOPICS
    * NPMI_TOP_WORDS
    * (NPMI_TOP_WORDS - 1)
    // 2
)

assert topic_npmi_df[
    "n_pairs"
].eq(45).all()

assert topic_quality_df[
    "country_entropy"
].notna().all()

valid_npmi = topic_npmi_df[
    "mean_npmi"
].dropna()

assert valid_npmi.between(
    -1, 1
).all()

# ------------------------------------------------------------
# 13. Display the coherence results
# ------------------------------------------------------------

print("=" * 70)
print("WINDOW-BASED NPMI COHERENCE AUDIT")
print("=" * 70)

print("Original speeches:", N_SPEECHES)
print("Segments:", len(segments_df))
print("Sliding window size:", NPMI_WINDOW_SIZE)
print("Total windows:", window_id)
print("Distinct evaluated terms:", len(candidate_terms))
print("Unigrams per topic:", NPMI_TOP_WORDS)

print("\nTopic coherence summary:")

display(
    topic_npmi_df[
        [
            "topic_id",
            "mean_npmi",
            "median_npmi",
            "min_npmi",
            "zero_cooccurrence_pairs",
            "min_term_window_prevalence_pct"
        ]
    ].round(4)
)

print("\nCoherence and country concentration:")

display(
    topic_quality_df[
        [
            "topic_id",
            "mean_npmi",
            "country_entropy",
            "country_majority_pct",
            "n_primary_speeches",
            "n_speeches_share_ge_10pct"
        ]
    ].round(4)
)

print("\nWeakest-supported evaluated terms:")

term_support_df = pd.DataFrame({
    "term": candidate_terms,
    "window_prevalence_pct": (
        100 * unigram_probabilities
    )
})

display(
    term_support_df.sort_values(
        "window_prevalence_pct"
    ).head(15).round(5)
)

print("\n" + "=" * 70)
print("NPMI VALIDATION")
print("=" * 70)

print(
    "All topic scores defined:",
    topic_npmi_df["mean_npmi"].notna().all()
)

print(
    "All pair scores within [-1, 1]:",
    pairwise_npmi_df["npmi"]
    .dropna()
    .between(-1, 1)
    .all()
)

print(
    "Window-weight normalization:",
    np.isclose(
        window_weights.sum(),
        N_SPEECHES
    )
)

if topic_npmi_df["mean_npmi"].notna().all():
    print("NPMI COMPUTATION: PASSED")
else:
    print("NPMI COMPUTATION: REVIEW UNDEFINED SCORES")

print("Final model selection: PENDING")



## 12.11. Multi-K and Multi-Seed NMF Model Selection

### Objective

Evaluate the sensitivity of speech-balanced segment-level NMF to the number of components and random initialization.

### Experimental configuration

- Corpus: 162 speeches, 1972–2025.
- Analytical units: 2,344 sentence-aware text segments.
- Candidate topic counts: 5, 7, 10, 12, 15.
- Random seeds: 11, 42, 73.
- Initialization: Random non-negative initialization.
- Optimization: Coordinate descent with Frobenius reconstruction loss.
- Weighting: Equal total reconstruction weight per original speech.
- Vocabulary and IDF: Fixed across all candidate models.

### Evaluation criteria

1. Numerical convergence and reconstruction error.
2. Within-K topic stability after optimal component matching.
3. Speech-balanced, window-based NPMI coherence.
4. Number of low-coverage and non-dominant components.
5. Substantive interpretability and country/period confounding.

### Interpretation

Lower reconstruction error alone does not imply superior thematic quality.

Higher NPMI alone does not imply substantive political relevance.

Random-seed stability is evaluated only among models with the same number of components.

Final model selection requires joint consideration of quantitative diagnostics and qualitative topic inspection.


In [ ]:

# ============================================================
# STEP 12.11A: MULTI-K AND MULTI-SEED NMF FITTING
# ============================================================


# ------------------------------------------------------------
# 1. Model selection configuration
# ------------------------------------------------------------

K_CANDIDATES = [5, 7, 10, 12, 15]
RANDOM_SEEDS = [11, 42, 73]

MAX_ITERATIONS = 2500
CONVERGENCE_TOL = 1e-4
TOPIC_PREVALENCE_THRESHOLD = 0.10

X_selection = X_segment_weighted.tocsr()

speech_ids_selection = segments_df[
    "speech_id"
].to_numpy(dtype=int)

segment_words_selection = segments_df[
    "segment_words"
].to_numpy(dtype=float)

N_SPEECHES_SELECTION = len(focal_df)
N_SEGMENTS_SELECTION = len(segments_df)

assert N_SPEECHES_SELECTION == 162
assert N_SEGMENTS_SELECTION == 2344

assert X_selection.shape == X_segment_tfidf.shape
assert X_selection.shape[1] == len(segment_features)

assert np.isfinite(X_selection.data).all()
assert np.all(X_selection.data >= 0)

# ------------------------------------------------------------
# 2. Calculate a common reconstruction-error denominator
# ------------------------------------------------------------

matrix_norm = float(
    np.sqrt(
        X_selection.multiply(X_selection).sum()
    )
)

assert matrix_norm > 0

# ------------------------------------------------------------
# 3. Initialize model storage
# ------------------------------------------------------------

nmf_selection_bank = {}
nmf_selection_records = []

# ------------------------------------------------------------
# 4. Fit candidate models
# ------------------------------------------------------------

for n_topics in K_CANDIDATES:

    for seed in RANDOM_SEEDS:

        model = NMF(
            n_components=n_topics,
            init="random",
            solver="cd",
            beta_loss="frobenius",
            max_iter=MAX_ITERATIONS,
            tol=CONVERGENCE_TOL,
            random_state=seed
        )

        with warnings.catch_warnings(
            record=True
        ) as caught_warnings:

            warnings.simplefilter(
                "always",
                ConvergenceWarning
            )

            W_weighted_candidate = model.fit_transform(
                X_selection
            )

        convergence_warning = any(
            issubclass(
                warning.category,
                ConvergenceWarning
            )
            for warning in caught_warnings
        )

        converged = (
            model.n_iter_ < MAX_ITERATIONS
            and not convergence_warning
        )

        H_candidate = model.components_.copy()

        relative_error = float(
            model.reconstruction_err_ / matrix_norm
        )

        # ----------------------------------------------------
        # 5. Transform the unweighted segment vectors
        # ----------------------------------------------------

        W_candidate = model.transform(
            X_segment_tfidf
        )

        component_mass = H_candidate.sum(axis=1)

        contributions = (
            W_candidate
            * component_mass[np.newaxis, :]
        )

        contribution_totals = contributions.sum(
            axis=1,
            keepdims=True
        )

        zero_mass_segments = int(
            np.sum(contribution_totals[:, 0] <= 0)
        )

        if zero_mass_segments:
            raise RuntimeError(
                f"K={n_topics}, seed={seed}: "
                f"{zero_mass_segments} segments have zero "
                "total topic contribution."
            )

        segment_shares_candidate = (
            contributions / contribution_totals
        )

        # ----------------------------------------------------
        # 6. Aggregate contributions to the speech level
        # ----------------------------------------------------

        speech_topic_mass = np.zeros(
            (N_SPEECHES_SELECTION, n_topics),
            dtype=float
        )

        np.add.at(
            speech_topic_mass,
            speech_ids_selection,
            segment_shares_candidate
            * segment_words_selection[:, np.newaxis]
        )

        speech_word_totals = np.bincount(
            speech_ids_selection,
            weights=segment_words_selection,
            minlength=N_SPEECHES_SELECTION
        )

        speech_shares_candidate = (
            speech_topic_mass
            / speech_word_totals[:, np.newaxis]
        )

        assert np.allclose(
            speech_shares_candidate.sum(axis=1),
            1.0,
            atol=1e-8
        )

        # ----------------------------------------------------
        # 7. Diagnostic indicators of weak topic utilization
        # ----------------------------------------------------

        topic_speech_coverage = (
            speech_shares_candidate
            >= TOPIC_PREVALENCE_THRESHOLD
        ).sum(axis=0)

        n_low_coverage_topics = int(
            np.sum(topic_speech_coverage <= 2)
        )

        dominant_segment_counts = np.bincount(
            np.argmax(
                segment_shares_candidate,
                axis=1
            ),
            minlength=n_topics
        )

        n_non_dominant_topics = int(
            np.sum(dominant_segment_counts == 0)
        )

        # ----------------------------------------------------
        # 8. Store the fitted model
        # ----------------------------------------------------

        nmf_selection_bank[(n_topics, seed)] = {
            "model": model,
            "H": H_candidate,
            "speech_shares": speech_shares_candidate,
            "topic_speech_coverage": topic_speech_coverage,
            "converged": converged
        }

        nmf_selection_records.append({
            "n_topics": n_topics,
            "seed": seed,
            "iterations": int(model.n_iter_),
            "converged": bool(converged),
            "relative_error": relative_error,
            "low_coverage_topics": n_low_coverage_topics,
            "non_dominant_segment_topics": (
                n_non_dominant_topics
            )
        })

        status = "OK" if converged else "REVIEW"

        print(
            f"[{status}] "
            f"K={n_topics:02d} | "
            f"Seed={seed:02d} | "
            f"Iterations={model.n_iter_:04d} | "
            f"Error={relative_error:.5f} | "
            f"Low-coverage={n_low_coverage_topics}"
        )

# ------------------------------------------------------------
# 9. Construct the fitting summary
# ------------------------------------------------------------

nmf_selection_runs = pd.DataFrame(
    nmf_selection_records
)

assert len(nmf_selection_runs) == 15
assert len(nmf_selection_bank) == 15

nmf_fitting_summary = (
    nmf_selection_runs
    .groupby("n_topics")
    .agg(
        n_runs=("seed", "size"),
        converged_runs=("converged", "sum"),
        mean_relative_error=(
            "relative_error", "mean"
        ),
        sd_relative_error=(
            "relative_error", "std"
        ),
        mean_low_coverage_topics=(
            "low_coverage_topics", "mean"
        ),
        mean_non_dominant_topics=(
            "non_dominant_segment_topics", "mean"
        )
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("MULTI-K NMF FITTING SUMMARY")
print("=" * 70)

display(nmf_fitting_summary.round(5))

print("\nCompleted fits:", len(nmf_selection_runs))

print(
    "Converged fits:",
    int(nmf_selection_runs["converged"].sum())
)

if nmf_selection_runs["converged"].all():
    print("MULTI-K FITTING: ALL MODELS CONVERGED")
else:
    print("MULTI-K FITTING: CONVERGENCE REVIEW REQUIRED")


In [ ]:

# ============================================================
# STEP 12.11B: NPMI COHERENCE AND TOPIC STABILITY
# ============================================================


# ------------------------------------------------------------
# 1. Configuration and dependency validation
# ------------------------------------------------------------

SELECTION_TOP_WORDS = 10
SELECTION_WINDOW_SIZE = 20

assert len(nmf_selection_bank) == 15
assert len(tokenized_segments) == len(segments_df)

assert SELECTION_WINDOW_SIZE == NPMI_WINDOW_SIZE

assert np.all(
    nmf_selection_runs["converged"]
), "Resolve non-converged models before scoring."

# ------------------------------------------------------------
# 2. Select top unigram words for every fitted model
# ------------------------------------------------------------

model_topic_words = {}
candidate_vocabulary = set()

for (n_topics, seed), result in nmf_selection_bank.items():

    H = result["H"]

    model_terms = []

    for topic_idx in range(n_topics):

        ranked_indices = np.argsort(
            H[topic_idx]
        )[::-1]

        words = []

        for feature_idx in ranked_indices:

            term = str(
                segment_features[feature_idx]
            )

            if " " in term:
                continue

            words.append(term)

            if len(words) == SELECTION_TOP_WORDS:
                break

        assert len(words) == SELECTION_TOP_WORDS

        model_terms.append(words)
        candidate_vocabulary.update(words)

    model_topic_words[(n_topics, seed)] = model_terms

selection_terms = sorted(candidate_vocabulary)

selection_term_lookup = {
    term: idx
    for idx, term in enumerate(selection_terms)
}

# ------------------------------------------------------------
# 3. Build a shared window-term presence matrix
# ------------------------------------------------------------

row_indices = []
column_indices = []
selection_window_weights = []

window_id = 0

for segment_idx, tokens in enumerate(tokenized_segments):

    speech_id = int(
        segment_speech_ids[segment_idx]
    )

    n_windows = int(
        segment_window_counts[segment_idx]
    )

    initial_length = min(
        SELECTION_WINDOW_SIZE,
        len(tokens)
    )

    active = Counter()

    for token in tokens[:initial_length]:

        idx = selection_term_lookup.get(token)

        if idx is not None:
            active[idx] += 1

    weight = float(
        1.0 / windows_per_speech[speech_id]
    )

    for start in range(n_windows):

        active_indices = list(active.keys())

        row_indices.extend(
            [window_id] * len(active_indices)
        )

        column_indices.extend(active_indices)

        selection_window_weights.append(weight)

        window_id += 1

        if start + 1 < n_windows:

            outgoing = selection_term_lookup.get(
                tokens[start]
            )

            incoming = selection_term_lookup.get(
                tokens[start + SELECTION_WINDOW_SIZE]
            )

            if outgoing is not None:

                active[outgoing] -= 1

                if active[outgoing] == 0:
                    del active[outgoing]

            if incoming is not None:
                active[incoming] += 1

# ------------------------------------------------------------
# 4. Estimate window-based probabilities
# ------------------------------------------------------------

selection_window_weights = np.asarray(
    selection_window_weights,
    dtype=np.float64
)

assert window_id == int(
    segment_window_counts.sum()
)

assert np.isclose(
    selection_window_weights.sum(),
    len(focal_df)
)

selection_presence = csr_matrix(
    (
        np.ones(len(row_indices), dtype=np.float64),
        (
            np.asarray(row_indices),
            np.asarray(column_indices)
        )
    ),
    shape=(window_id, len(selection_terms))
)

selection_weighted_presence = (
    selection_presence
    .multiply(
        selection_window_weights[:, np.newaxis]
    )
    .tocsr()
)

selection_joint = (
    selection_presence.T
    @ selection_weighted_presence
).toarray() / selection_window_weights.sum()

selection_marginal = np.diag(
    selection_joint
).copy()

assert np.allclose(
    selection_joint,
    selection_joint.T,
    atol=1e-8
)

assert np.all(
    selection_marginal >= 0
)

# ------------------------------------------------------------
# 5. Define the NPMI scoring function
# ------------------------------------------------------------

def selection_npmi(p_i, p_j, p_ij):

    if p_i <= 0 or p_j <= 0:
        return np.nan

    if p_ij <= 0:
        return -1.0

    if p_ij >= 1.0:
        return np.nan

    score = (
        np.log(p_ij / (p_i * p_j))
        / -np.log(p_ij)
    )

    return float(np.clip(score, -1.0, 1.0))


# ------------------------------------------------------------
# 6. Score the coherence of every candidate model
# ------------------------------------------------------------

coherence_records = []
topic_coherence_records = []

for (n_topics, seed), topic_lists in model_topic_words.items():

    model_scores = []

    for topic_idx, words in enumerate(topic_lists):

        pair_scores = []
        zero_pairs = 0

        for word_a, word_b in combinations(words, 2):

            idx_a = selection_term_lookup[word_a]
            idx_b = selection_term_lookup[word_b]

            p_a = selection_marginal[idx_a]
            p_b = selection_marginal[idx_b]
            p_ab = selection_joint[idx_a, idx_b]

            score = selection_npmi(
                p_a, p_b, p_ab
            )

            pair_scores.append(score)

            if p_ab <= 0:
                zero_pairs += 1

        pair_scores = np.asarray(
            pair_scores,
            dtype=float
        )

        topic_score = (
            float(pair_scores.mean())
            if np.isfinite(pair_scores).all()
            else np.nan
        )

        model_scores.append(topic_score)

        topic_coherence_records.append({
            "n_topics": n_topics,
            "seed": seed,
            "topic_id": topic_idx + 1,
            "mean_npmi": topic_score,
            "zero_cooccurrence_pairs": zero_pairs
        })

    model_scores = np.asarray(
        model_scores,
        dtype=float
    )

    coherence_records.append({
        "n_topics": n_topics,
        "seed": seed,
        "mean_npmi": (
            float(model_scores.mean())
            if np.isfinite(model_scores).all()
            else np.nan
        ),
        "median_topic_npmi": float(
            np.nanmedian(model_scores)
        ),
        "min_topic_npmi": float(
            np.nanmin(model_scores)
        ),
        "negative_npmi_topics": int(
            np.sum(model_scores < 0)
        ),
        "undefined_npmi_topics": int(
            np.sum(~np.isfinite(model_scores))
        )
    })

nmf_coherence_runs = pd.DataFrame(
    coherence_records
)

nmf_topic_coherence = pd.DataFrame(
    topic_coherence_records
)

assert len(nmf_coherence_runs) == 15

assert nmf_coherence_runs[
    "undefined_npmi_topics"
].eq(0).all(), (
    "Undefined coherence scores require inspection."
)

# ------------------------------------------------------------
# 7. Evaluate initialization stability within each K
# ------------------------------------------------------------

stability_records = []

for n_topics in K_CANDIDATES:

    for seed_a, seed_b in combinations(
        RANDOM_SEEDS, 2
    ):

        H_a = nmf_selection_bank[
            (n_topics, seed_a)
        ]["H"]

        H_b = nmf_selection_bank[
            (n_topics, seed_b)
        ]["H"]

        similarity = cosine_similarity(
            H_a, H_b
        )

        row_ind, col_ind = linear_sum_assignment(
            -similarity
        )

        matched_similarities = similarity[
            row_ind, col_ind
        ]

        stability_records.append({
            "n_topics": n_topics,
            "seed_a": seed_a,
            "seed_b": seed_b,
            "mean_topic_similarity": float(
                matched_similarities.mean()
            ),
            "min_topic_similarity": float(
                matched_similarities.min()
            ),
            "median_topic_similarity": float(
                np.median(matched_similarities)
            )
        })

nmf_stability_pairs = pd.DataFrame(
    stability_records
)

assert len(nmf_stability_pairs) == 15

assert nmf_stability_pairs[
    "mean_topic_similarity"
].between(0, 1).all()

# ------------------------------------------------------------
# 8. Construct the joint model-selection summary
# ------------------------------------------------------------

run_quality = nmf_selection_runs.merge(
    nmf_coherence_runs,
    on=["n_topics", "seed"],
    validate="one_to_one"
)

coherence_summary = (
    run_quality
    .groupby("n_topics")
    .agg(
        mean_npmi=("mean_npmi", "mean"),
        sd_npmi=("mean_npmi", "std"),
        min_run_npmi=("mean_npmi", "min"),
        max_run_npmi=("mean_npmi", "max"),
        mean_negative_topics=(
            "negative_npmi_topics", "mean"
        )
    )
    .reset_index()
)

stability_summary = (
    nmf_stability_pairs
    .groupby("n_topics")
    .agg(
        mean_topic_stability=(
            "mean_topic_similarity", "mean"
        ),
        worst_matched_topic=(
            "min_topic_similarity", "min"
        )
    )
    .reset_index()
)

nmf_model_selection_summary = (
    nmf_fitting_summary
    .merge(
        coherence_summary,
        on="n_topics",
        validate="one_to_one"
    )
    .merge(
        stability_summary,
        on="n_topics",
        validate="one_to_one"
    )
    .sort_values("n_topics")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 9. Validate the combined results
# ------------------------------------------------------------

assert len(nmf_model_selection_summary) == 5

assert np.isfinite(
    nmf_model_selection_summary.select_dtypes(
        include=[np.number]
    ).to_numpy()
).all()

# ------------------------------------------------------------
# 10. Display the evaluation tables
# ------------------------------------------------------------

print("=" * 70)
print("MULTI-K NMF MODEL SELECTION")
print("=" * 70)

display(
    nmf_model_selection_summary[
        [
            "n_topics",
            "converged_runs",
            "mean_relative_error",
            "mean_npmi",
            "sd_npmi",
            "mean_topic_stability",
            "worst_matched_topic",
            "mean_low_coverage_topics",
            "mean_negative_topics"
        ]
    ].round(4)
)

print("\nInitialization stability details:")

display(
    nmf_stability_pairs.round(4)
)

print("\nIndividual model coherence results:")

display(
    run_quality[
        [
            "n_topics",
            "seed",
            "converged",
            "relative_error",
            "mean_npmi",
            "negative_npmi_topics",
            "low_coverage_topics"
        ]
    ].round(4)
)

print("\n" + "=" * 70)
print("MODEL SELECTION VALIDATION")
print("=" * 70)

print("Candidate models:", len(nmf_selection_bank))
print("Coherence evaluations:", len(nmf_coherence_runs))
print("Seed-pair comparisons:", len(nmf_stability_pairs))

print("NPMI evaluations: PASSED")
print("Topic matching: PASSED")
print("Final model choice: PENDING QUALITATIVE REVIEW")


In [ ]:

# ============================================================
# STEP 12.11C: MODEL COHERENCE SENSITIVITY PLOT
# ============================================================

plot_df = (
    nmf_model_selection_summary
    .sort_values("n_topics")
    .copy()
)

x = plot_df["n_topics"].to_numpy()

mean_npmi = plot_df["mean_npmi"].to_numpy()
minimum_npmi = plot_df["min_run_npmi"].to_numpy()
maximum_npmi = plot_df["max_run_npmi"].to_numpy()

fig, ax = plt.subplots(
    figsize=(8.5, 5.5),
    dpi=150
)

ax.plot(
    x,
    mean_npmi,
    marker="o",
    linewidth=2,
    markersize=7,
    label="Mean topic NPMI"
)

ax.fill_between(
    x,
    minimum_npmi,
    maximum_npmi,
    alpha=0.18,
    label="Range across random seeds"
)

ax.axhline(
    0,
    color="gray",
    linewidth=1,
    linestyle="--"
)

ax.set_xticks(x)

ax.set_xlabel(
    "Number of NMF topics (K)",
    fontsize=11
)

ax.set_ylabel(
    "Mean window-based NPMI",
    fontsize=11
)

ax.set_title(
    "Topic Coherence Across Model Specifications",
    fontsize=13,
    fontweight="bold",
    pad=14
)

ax.grid(
    axis="y",
    alpha=0.2
)

ax.legend(
    loc="best",
    frameon=False
)

fig.tight_layout()

plt.show()

print("Model coherence diagnostic: GENERATED")
print("Publication export: DEFERRED")



## 12.12. Topic Interpretability and Historical Confounding Audit

### Objective

Compare the thematic interpretability of K=5 and K=7 NMF solutions and investigate the sensitivity of the K=7 solution to random initialization.

### Reference models

- K=5, seed=11
- K=7, seed=11
- K=7, seed=73 for instability diagnostics

### Evaluation

1. Full top-term inspection for every component.
2. Country-specific mean component contributions.
3. Dominant-topic representation across countries.
4. Topic prevalence using a K-specific uniform-share benchmark.
5. Historical-period variation in component contributions.
6. Identification of unstable components through optimal topic matching.
7. Contextual examination of representative text segments.

### Methodological safeguards

The threshold 1/K is a descriptive uniform-share reference, not an estimated probability of topic membership.

Country identity, historical period, and substantive thematic content may be confounded.

Component coherence and numerical convergence do not establish historical or political validity.

Final topic labels will be assigned only after contextual review of source speeches.


In [ ]:

# ============================================================
# STEP 12.12A: K=5 VS K=7 INTERPRETABILITY AUDIT
# ============================================================



# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REFERENCE_SEED = 11
CANDIDATE_K = [5, 7]

COUNTRIES = ["IRN", "USA", "CHN"]

country_array = focal_df[
    "country_code"
].to_numpy()

year_array = focal_df[
    "year"
].to_numpy(dtype=int)

assert len(focal_df) == 162

# ------------------------------------------------------------
# 2. Construct historical period labels
# ------------------------------------------------------------

period_array = np.select(
    [
        (year_array >= 1972) & (year_array <= 1990),
        (year_array >= 1991) & (year_array <= 2010),
        (year_array >= 2011) & (year_array <= 2025)
    ],
    [
        "1972-1990",
        "1991-2010",
        "2011-2025"
    ],
    default="OUTSIDE"
)

assert not np.any(period_array == "OUTSIDE")

PERIODS = [
    "1972-1990",
    "1991-2010",
    "2011-2025"
]

# ------------------------------------------------------------
# 3. Define topic concentration measures
# ------------------------------------------------------------

def calculate_country_entropy(values):

    values = np.asarray(values, dtype=float)

    assert np.all(values >= 0)
    assert values.sum() > 0

    probabilities = values / values.sum()

    positive = probabilities[
        probabilities > 0
    ]

    entropy = -np.sum(
        positive * np.log(positive)
    )

    return float(
        entropy / np.log(len(values))
    )


def get_topic_terms(H, topic_idx, n_terms=12):

    indices = np.argsort(
        H[topic_idx]
    )[::-1][:n_terms]

    return ", ".join(
        segment_features[indices]
    )

# ------------------------------------------------------------
# 4. Construct topic-level audit tables
# ------------------------------------------------------------

interpretability_records = []
historical_records = []

for k in CANDIDATE_K:

    model_result = nmf_selection_bank[
        (k, REFERENCE_SEED)
    ]

    H = model_result["H"]

    speech_shares = model_result[
        "speech_shares"
    ]

    assert H.shape[0] == k
    assert speech_shares.shape == (162, k)

    primary_topics = np.argmax(
        speech_shares,
        axis=1
    )

    uniform_reference = 1.0 / k

    for topic_idx in range(k):

        shares = speech_shares[
            :, topic_idx
        ]

        country_means = {
            country: float(
                shares[
                    country_array == country
                ].mean()
            )
            for country in COUNTRIES
        }

        country_mass = np.array([
            shares[
                country_array == country
            ].sum()
            for country in COUNTRIES
        ])

        country_entropy = (
            calculate_country_entropy(
                country_mass
            )
        )

        country_primary_counts = {
            country: int(
                np.sum(
                    (country_array == country)
                    & (primary_topics == topic_idx)
                )
            )
            for country in COUNTRIES
        }

        record = {
            "K": k,
            "topic_id": topic_idx + 1,
            "top_terms": get_topic_terms(
                H, topic_idx
            ),
            "country_entropy": country_entropy,
            "n_primary_speeches": int(
                np.sum(
                    primary_topics == topic_idx
                )
            ),
            "n_primary_countries": int(
                sum(
                    value > 0
                    for value in country_primary_counts.values()
                )
            ),
            "n_speeches_ge_uniform": int(
                np.sum(
                    shares >= uniform_reference
                )
            )
        }

        for country in COUNTRIES:

            record[
                f"{country}_mean"
            ] = country_means[country]

            record[
                f"{country}_primary"
            ] = country_primary_counts[country]

        interpretability_records.append(record)

        for period in PERIODS:

            mask = period_array == period

            historical_records.append({
                "K": k,
                "topic_id": topic_idx + 1,
                "period": period,
                "n_speeches": int(mask.sum()),
                "mean_contribution": float(
                    shares[mask].mean()
                )
            })

topic_interpretability_df = pd.DataFrame(
    interpretability_records
)

topic_historical_df = pd.DataFrame(
    historical_records
)

# ------------------------------------------------------------
# 5. Validate outputs
# ------------------------------------------------------------

assert len(topic_interpretability_df) == 12

assert len(topic_historical_df) == 36

assert topic_interpretability_df[
    "country_entropy"
].between(0, 1).all()

for k in CANDIDATE_K:

    subset = topic_interpretability_df.loc[
        topic_interpretability_df["K"] == k
    ]

    assert subset[
        "n_primary_speeches"
    ].sum() == 162

# ------------------------------------------------------------
# 6. Display topic vocabularies
# ------------------------------------------------------------

print("=" * 70)
print("K=5 AND K=7 TOPIC INTERPRETABILITY")
print("=" * 70)

for k in CANDIDATE_K:

    print(f"\n{'=' * 50}")
    print(f"MODEL K={k}")
    print("=" * 50)

    subset = topic_interpretability_df.loc[
        topic_interpretability_df["K"] == k
    ]

    for row in subset.itertuples(index=False):

        print(f"\nTOPIC {row.topic_id:02d}")
        print(row.top_terms)

# ------------------------------------------------------------
# 7. Display country concentration diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COUNTRY AND TOPIC COVERAGE COMPARISON")
print("=" * 70)

display(
    topic_interpretability_df[
        [
            "K",
            "topic_id",
            "country_entropy",
            "n_primary_speeches",
            "n_primary_countries",
            "n_speeches_ge_uniform",
            "IRN_mean",
            "USA_mean",
            "CHN_mean"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 8. Display historical contributions for K=7
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HISTORICAL TOPIC CONTRIBUTIONS: K=7")
print("=" * 70)

historical_k7 = (
    topic_historical_df
    .loc[
        topic_historical_df["K"] == 7
    ]
    .pivot(
        index="topic_id",
        columns="period",
        values="mean_contribution"
    )
    .reindex(columns=PERIODS)
)

display(
    historical_k7.round(4)
)

print("\nINTERPRETABILITY AUDIT: PASSED")
print("Historical interpretation: PENDING")


In [ ]:

# ============================================================
# STEP 12.12B: K=7 TOPIC INSTABILITY INVESTIGATION
# ============================================================



# ------------------------------------------------------------
# 1. Retrieve reference models
# ------------------------------------------------------------

K = 7

H_reference = nmf_selection_bank[
    (K, 11)
]["H"]

H_alternative = nmf_selection_bank[
    (K, 73)
]["H"]

assert H_reference.shape == H_alternative.shape

# ------------------------------------------------------------
# 2. Calculate component similarities
# ------------------------------------------------------------

similarity_matrix = cosine_similarity(
    H_reference,
    H_alternative
)

reference_indices, alternative_indices = (
    linear_sum_assignment(
        -similarity_matrix
    )
)

# ------------------------------------------------------------
# 3. Construct matched-topic diagnostics
# ------------------------------------------------------------

alignment_records = []

for ref_idx, alt_idx in zip(
    reference_indices,
    alternative_indices
):

    similarity = float(
        similarity_matrix[
            ref_idx, alt_idx
        ]
    )

    alignment_records.append({
        "reference_topic": ref_idx + 1,
        "seed73_topic": alt_idx + 1,
        "cosine_similarity": similarity,
        "reference_terms": get_topic_terms(
            H_reference,
            ref_idx,
            n_terms=10
        ),
        "seed73_terms": get_topic_terms(
            H_alternative,
            alt_idx,
            n_terms=10
        )
    })

k7_alignment_df = pd.DataFrame(
    alignment_records
).sort_values(
    "cosine_similarity"
).reset_index(drop=True)

# ------------------------------------------------------------
# 4. Validate the component matching
# ------------------------------------------------------------

assert len(k7_alignment_df) == K

assert k7_alignment_df[
    "reference_topic"
].is_unique

assert k7_alignment_df[
    "seed73_topic"
].is_unique

assert k7_alignment_df[
    "cosine_similarity"
].between(0, 1).all()

# ------------------------------------------------------------
# 5. Display the instability diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("K=7 RANDOM INITIALIZATION SENSITIVITY")
print("=" * 70)

with pd.option_context(
    "display.max_colwidth", None
):
    display(
        k7_alignment_df
    )

print("\nMean matched similarity:")
print(
    round(
        k7_alignment_df[
            "cosine_similarity"
        ].mean(),
        4
    )
)

print("\nMinimum matched similarity:")
print(
    round(
        k7_alignment_df[
            "cosine_similarity"
        ].min(),
        4
    )
)

print("\nTOPIC INSTABILITY INVESTIGATION: PASSED")
print("Unstable components require substantive review.")


In [ ]:

# ============================================================
# STEP 12.12C: REPRESENTATIVE TOPIC CONTEXT INSPECTION
# ============================================================


# ------------------------------------------------------------
# 1. Select a topic for substantive inspection
# ------------------------------------------------------------

INSPECT_K = 7
INSPECT_SEED = 11
INSPECT_TOPIC_ID = 1

EXAMPLES_PER_COUNTRY = 2
MAX_EXCERPT_CHARACTERS = 600

assert 1 <= INSPECT_TOPIC_ID <= INSPECT_K

# ------------------------------------------------------------
# 2. Retrieve the selected NMF model
# ------------------------------------------------------------

selected_result = nmf_selection_bank[
    (INSPECT_K, INSPECT_SEED)
]

selected_model = selected_result["model"]

selected_H = selected_result["H"]

# ------------------------------------------------------------
# 3. Recover normalized segment contributions
# ------------------------------------------------------------

selected_W = selected_model.transform(
    X_segment_tfidf
)

component_mass = selected_H.sum(axis=1)

component_contributions = (
    selected_W * component_mass[np.newaxis, :]
)

contribution_totals = (
    component_contributions.sum(
        axis=1,
        keepdims=True
    )
)

assert np.all(contribution_totals > 0)

selected_segment_shares = (
    component_contributions
    / contribution_totals
)

assert np.allclose(
    selected_segment_shares.sum(axis=1),
    1.0
)

# ------------------------------------------------------------
# 4. Identify representative segments
# ------------------------------------------------------------

topic_idx = INSPECT_TOPIC_ID - 1

inspection_df = segments_df.copy()

inspection_df["topic_share"] = (
    selected_segment_shares[
        :, topic_idx
    ]
)

selected_rows = []

for country in ["IRN", "USA", "CHN"]:

    candidates = (
        inspection_df
        .loc[
            inspection_df["country_code"]
            == country
        ]
        .sort_values(
            "topic_share",
            ascending=False
        )
        .drop_duplicates(
            subset=["speech_id"],
            keep="first"
        )
        .head(EXAMPLES_PER_COUNTRY)
    )

    selected_rows.append(candidates)

representative_segments_df = pd.concat(
    selected_rows,
    ignore_index=True
)

# ------------------------------------------------------------
# 5. Display representative contexts
# ------------------------------------------------------------

print("=" * 70)
print("REPRESENTATIVE SEGMENT INSPECTION")
print("=" * 70)

print("K:", INSPECT_K)
print("Seed:", INSPECT_SEED)
print("Topic:", INSPECT_TOPIC_ID)

print("\nLeading vocabulary:")

print(
    get_topic_terms(
        selected_H,
        topic_idx,
        n_terms=15
    )
)

for row in representative_segments_df.itertuples():

    clean_excerpt = " ".join(
        str(row.segment_text).split()
    )

    print("\n" + "-" * 70)

    print(
        f"Country: {row.country_code}"
        f" | Year: {row.year}"
        f" | Speech ID: {row.speech_id}"
        f" | Segment: {row.segment_id}"
    )

    print(
        f"Relative contribution: "
        f"{row.topic_share:.4f}"
    )

    print(
        f"Source: {row.archive_path}"
    )

    print("\nText excerpt:")

    print(
        clean_excerpt[
            :MAX_EXCERPT_CHARACTERS
        ]
    )

    if len(clean_excerpt) > MAX_EXCERPT_CHARACTERS:
        print("[Excerpt truncated]")

print("\nCONTEXT EXTRACTION: PASSED")
print("Human semantic validation: PENDING")



## 12.13. Identity-Feature Ablation and Country–Period Validation

### Objectives

1. Evaluate the influence of explicit country-identity vocabulary on the seven-component NMF solution.
2. Compare the original and identity-ablated models using optimal component alignment.
3. Assess initialization stability following vocabulary ablation.
4. Examine temporal topic contributions separately for each country.

### Controlled design

- Reference model: K=7, seed=11.
- Ablation models: K=7, seeds 11, 42, and 73.
- Original analytical sample: 162 speeches and 2,344 segments.
- Country-year coverage: Balanced, 1972–2025.
- Feature exclusion: Explicit country names, demonyms, and selected country-identifying expressions.
- TF-IDF: Original fitted feature weights retained for the surviving features, followed by row normalization.
- Speech weights: Preserved from the original weighted NMF experiment.
- Model initialization and optimization: Identical to the preceding multi-seed experiment.

### Evaluation criteria

- Component similarity after Hungarian matching.
- Changes in speech-level topic composition.
- Stability across random initializations.
- Country-specific and historical contribution patterns.
- Qualitative preservation of substantive themes.

### Limitations

Removing identity terms may remove substantive information alongside country cues. The experiment does not establish causal effects or guarantee that all country proxies have been eliminated.

The ablated models are sensitivity specifications, not automatic replacements for the original model.


In [ ]:

# ============================================================
# STEP 12.13A: CONTROLLED COUNTRY-IDENTITY FEATURE ABLATION
# ============================================================



# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

ABLATION_K = 7
ABLATION_SEEDS = [11, 42, 73]

ABLATION_MAX_ITER = 2500
ABLATION_TOL = 1e-4

identity_unigrams = {
    "iran", "iranian", "iranians",
    "persia", "persian", "persians",
    "china", "chinese",
    "america", "american", "americans",
    "usa"
}

identity_phrases = {
    "united states",
    "islamic republic",
    "people republic china"
}

# ------------------------------------------------------------
# 2. Identify explicit identity features
# ------------------------------------------------------------

def is_identity_feature(feature):

    feature = str(feature).lower().strip()

    tokens = set(
        re.findall(r"[a-z]+", feature)
    )

    if tokens.intersection(identity_unigrams):
        return True

    padded_feature = f" {feature} "

    return any(
        f" {phrase} " in padded_feature
        for phrase in identity_phrases
    )


feature_names = np.asarray(
    segment_features,
    dtype=str
)

excluded_mask = np.array(
    [
        is_identity_feature(feature)
        for feature in feature_names
    ],
    dtype=bool
)

retained_indices = np.flatnonzero(
    ~excluded_mask
)

excluded_features = feature_names[
    excluded_mask
]

retained_features = feature_names[
    retained_indices
]

assert excluded_mask.any()
assert len(retained_features) > ABLATION_K

# ------------------------------------------------------------
# 3. Recover and validate original segment weights
# ------------------------------------------------------------

X_original_raw = X_segment_tfidf.tocsr()

X_original_weighted = (
    X_segment_weighted.tocsr()
)

assert (
    X_original_raw.shape
    == X_original_weighted.shape
)

raw_norms = np.sqrt(
    np.asarray(
        X_original_raw.power(2).sum(axis=1)
    ).ravel()
)

weighted_norms = np.sqrt(
    np.asarray(
        X_original_weighted.power(2).sum(axis=1)
    ).ravel()
)

assert np.all(raw_norms > 0)
assert np.all(weighted_norms > 0)

original_segment_weights = (
    weighted_norms / raw_norms
)

reconstructed_weighted = (
    X_original_raw
    .multiply(
        original_segment_weights[:, np.newaxis]
    )
    .tocsr()
)

weighting_difference = (
    X_original_weighted
    - reconstructed_weighted
)

relative_weighting_difference = (
    np.linalg.norm(weighting_difference.data)
    / np.linalg.norm(X_original_weighted.data)
)

assert relative_weighting_difference < 1e-8, (
    "Original weighting is not row-scalar. "
    "Review the original weighting specification."
)

# ------------------------------------------------------------
# 4. Construct ablated matrices
# ------------------------------------------------------------

X_ablated_unscaled = (
    X_original_raw[:, retained_indices]
    .tocsr()
)

retained_row_norms = np.sqrt(
    np.asarray(
        X_ablated_unscaled.power(2).sum(axis=1)
    ).ravel()
)

zero_feature_segments = int(
    np.sum(retained_row_norms == 0)
)

assert zero_feature_segments == 0, (
    "At least one segment lost all features. "
    "Review the ablation dictionary."
)

X_ablated_raw = normalize(
    X_ablated_unscaled,
    norm="l2",
    axis=1,
    copy=True
).tocsr()

X_ablated_weighted = (
    X_ablated_raw
    .multiply(
        original_segment_weights[:, np.newaxis]
    )
    .tocsr()
)

assert np.isfinite(
    X_ablated_weighted.data
).all()

assert np.all(
    X_ablated_weighted.data >= 0
)

# ------------------------------------------------------------
# 5. Validate preservation of speech weights
# ------------------------------------------------------------

ablation_speech_ids = segments_df[
    "speech_id"
].to_numpy(dtype=int)

ablation_segment_words = segments_df[
    "segment_words"
].to_numpy(dtype=float)

assert len(ablation_speech_ids) == 2344
assert ablation_speech_ids.min() == 0
assert ablation_speech_ids.max() == 161

assert np.all(
    ablation_segment_words > 0
)

# ------------------------------------------------------------
# 6. Fit ablated models
# ------------------------------------------------------------

ablation_models = {}
ablation_fitting_records = []

for seed in ABLATION_SEEDS:

    model = NMF(
        n_components=ABLATION_K,
        init="random",
        solver="cd",
        beta_loss="frobenius",
        max_iter=ABLATION_MAX_ITER,
        tol=ABLATION_TOL,
        random_state=seed
    )

    with warnings.catch_warnings(
        record=True
    ) as fit_warnings:

        warnings.simplefilter(
            "always",
            ConvergenceWarning
        )

        W_weighted = model.fit_transform(
            X_ablated_weighted
        )

    warned = any(
        issubclass(
            warning.category,
            ConvergenceWarning
        )
        for warning in fit_warnings
    )

    converged = (
        model.n_iter_ < ABLATION_MAX_ITER
        and not warned
    )

    H_ablated = model.components_.copy()

    # Obtain segment-level contributions on the
    # corresponding unweighted, normalized matrix.
    W_ablated = model.transform(
        X_ablated_raw
    )

    topic_mass = H_ablated.sum(axis=1)

    contributions = (
        W_ablated * topic_mass[np.newaxis, :]
    )

    totals = contributions.sum(
        axis=1,
        keepdims=True
    )

    assert np.all(totals > 0)

    segment_shares = (
        contributions / totals
    )

    # Aggregate segments using their original word counts.
    speech_mass = np.zeros(
        (162, ABLATION_K),
        dtype=float
    )

    np.add.at(
        speech_mass,
        ablation_speech_ids,
        segment_shares
        * ablation_segment_words[:, np.newaxis]
    )

    speech_words = np.bincount(
        ablation_speech_ids,
        weights=ablation_segment_words,
        minlength=162
    )

    speech_shares = (
        speech_mass
        / speech_words[:, np.newaxis]
    )

    assert np.allclose(
        speech_shares.sum(axis=1),
        1.0,
        atol=1e-8
    )

    ablation_models[seed] = {
        "model": model,
        "H": H_ablated,
        "speech_shares": speech_shares,
        "segment_shares": segment_shares,
        "converged": converged
    }

    ablation_fitting_records.append({
        "seed": seed,
        "iterations": int(model.n_iter_),
        "converged": bool(converged),
        "reconstruction_error": float(
            model.reconstruction_err_
        )
    })

    print(
        f"[{'OK' if converged else 'REVIEW'}] "
        f"K=7 | Seed={seed} | "
        f"Iterations={model.n_iter_}"
    )

# ------------------------------------------------------------
# 7. Summarize the ablation experiment
# ------------------------------------------------------------

ablation_fit_df = pd.DataFrame(
    ablation_fitting_records
)

print("\n" + "=" * 70)
print("IDENTITY FEATURE ABLATION AUDIT")
print("=" * 70)

print("Original features:", len(feature_names))
print("Excluded features:", len(excluded_features))
print("Retained features:", len(retained_features))
print("Zero-feature segments:", zero_feature_segments)

print("\nSample excluded features:")

print(
    sorted(excluded_features.tolist())[:30]
)

print("\nAblated model fitting:")

display(ablation_fit_df)

if ablation_fit_df["converged"].all():
    print("\nIDENTITY ABLATION FITTING: PASSED")
else:
    print("\nIDENTITY ABLATION FITTING: CONVERGENCE REVIEW")


In [ ]:

# ============================================================
# STEP 12.13B: POST-ABLATION TOPIC ALIGNMENT
# ============================================================



assert all(
    result["converged"]
    for result in ablation_models.values()
), "Resolve any non-converged models first."


def ablated_top_terms(H, topic_idx, n_terms=12):
    indices = np.argsort(
        H[topic_idx]
    )[::-1][:n_terms]

    return ", ".join(
        retained_features[indices]
    )


# ------------------------------------------------------------
# 1. Compare ablated and original models by matching topics
# ------------------------------------------------------------

alignment_records = []
model_shift_records = []

for seed in ABLATION_SEEDS:

    original_result = nmf_selection_bank[
        (ABLATION_K, seed)
    ]

    ablated_result = ablation_models[seed]

    H_original = original_result["H"][
        :, retained_indices
    ]

    H_ablated = ablated_result["H"]

    similarity = cosine_similarity(
        H_original,
        H_ablated
    )

    ref_indices, alt_indices = linear_sum_assignment(
        -similarity
    )

    original_shares = original_result[
        "speech_shares"
    ]

    ablated_shares = ablated_result[
        "speech_shares"
    ]

    aligned_shares = np.zeros_like(
        ablated_shares
    )

    for ref_idx, alt_idx in zip(
        ref_indices,
        alt_indices
    ):

        aligned_shares[:, ref_idx] = (
            ablated_shares[:, alt_idx]
        )

        topic_similarity = float(
            similarity[ref_idx, alt_idx]
        )

        mean_share_shift = float(
            np.mean(
                np.abs(
                    original_shares[:, ref_idx]
                    - aligned_shares[:, ref_idx]
                )
            )
        )

        alignment_records.append({
            "seed": seed,
            "original_topic": int(ref_idx + 1),
            "ablated_topic": int(alt_idx + 1),
            "cosine_similarity": topic_similarity,
            "mean_absolute_share_shift": mean_share_shift,
            "original_terms": get_topic_terms(
                original_result["H"],
                ref_idx,
                n_terms=10
            ),
            "ablated_terms": ablated_top_terms(
                H_ablated,
                alt_idx,
                n_terms=10
            )
        })

    # Jensen-Shannon distance summarizes changes in
    # the complete speech-level topic composition.
    js_distances = np.array([
        jensenshannon(
            original_shares[i],
            aligned_shares[i],
            base=2
        )
        for i in range(len(original_shares))
    ])

    model_shift_records.append({
        "seed": seed,
        "mean_topic_cosine": float(
            similarity[
                ref_indices, alt_indices
            ].mean()
        ),
        "minimum_topic_cosine": float(
            similarity[
                ref_indices, alt_indices
            ].min()
        ),
        "mean_absolute_share_shift": float(
            np.abs(
                original_shares - aligned_shares
            ).mean()
        ),
        "mean_js_distance": float(
            js_distances.mean()
        ),
        "max_js_distance": float(
            js_distances.max()
        )
    })

ablation_alignment_df = pd.DataFrame(
    alignment_records
)

ablation_model_shift_df = pd.DataFrame(
    model_shift_records
)

# ------------------------------------------------------------
# 2. Assess stability among ablated initializations
# ------------------------------------------------------------

ablation_stability_records = []

for seed_a, seed_b in combinations(
    ABLATION_SEEDS, 2
):

    H_a = ablation_models[seed_a]["H"]
    H_b = ablation_models[seed_b]["H"]

    similarity = cosine_similarity(
        H_a, H_b
    )

    idx_a, idx_b = linear_sum_assignment(
        -similarity
    )

    matched = similarity[idx_a, idx_b]

    ablation_stability_records.append({
        "seed_a": seed_a,
        "seed_b": seed_b,
        "mean_similarity": float(
            matched.mean()
        ),
        "minimum_similarity": float(
            matched.min()
        ),
        "median_similarity": float(
            np.median(matched)
        )
    })

ablation_stability_df = pd.DataFrame(
    ablation_stability_records
)

# ------------------------------------------------------------
# 3. Compare random-start stability with the original model
# ------------------------------------------------------------

original_stability = (
    nmf_stability_pairs.loc[
        nmf_stability_pairs["n_topics"] == 7,
        [
            "seed_a",
            "seed_b",
            "mean_topic_similarity",
            "min_topic_similarity"
        ]
    ]
    .rename(
        columns={
            "mean_topic_similarity": "original_mean",
            "min_topic_similarity": "original_min"
        }
    )
)

stability_comparison_df = original_stability.merge(
    ablation_stability_df[
        [
            "seed_a",
            "seed_b",
            "mean_similarity",
            "minimum_similarity"
        ]
    ],
    on=["seed_a", "seed_b"],
    validate="one_to_one"
).rename(
    columns={
        "mean_similarity": "ablated_mean",
        "minimum_similarity": "ablated_min"
    }
)

# ------------------------------------------------------------
# 4. Validate results
# ------------------------------------------------------------

assert len(ablation_alignment_df) == 21
assert len(ablation_model_shift_df) == 3
assert len(ablation_stability_df) == 3

assert ablation_alignment_df[
    "cosine_similarity"
].between(0, 1).all()

assert ablation_model_shift_df[
    "mean_js_distance"
].between(0, 1).all()

# ------------------------------------------------------------
# 5. Display numerical diagnostics
# ------------------------------------------------------------

print("=" * 70)
print("CONTROLLED IDENTITY ABLATION COMPARISON")
print("=" * 70)

print("\nOverall model changes:")
display(
    ablation_model_shift_df.round(4)
)

print("\nTopic matching for reference seed 11:")

seed11_alignment = (
    ablation_alignment_df.loc[
        ablation_alignment_df["seed"] == 11
    ]
    .sort_values("original_topic")
)

with pd.option_context(
    "display.max_colwidth", None
):
    display(
        seed11_alignment[
            [
                "original_topic",
                "ablated_topic",
                "cosine_similarity",
                "mean_absolute_share_shift",
                "original_terms",
                "ablated_terms"
            ]
        ].round(4)
    )

print("\nInitialization stability comparison:")

display(
    stability_comparison_df.round(4)
)

print("\nIDENTITY ABLATION ALIGNMENT: PASSED")
print("Substantive interpretation: PENDING")


In [ ]:

# ============================================================
# STEP 12.13C: COUNTRY-STRATIFIED HISTORICAL TOPIC AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Reference model and balanced historical periods
# ------------------------------------------------------------

HISTORICAL_K = 7
HISTORICAL_SEED = 11

reference_shares = nmf_selection_bank[
    (HISTORICAL_K, HISTORICAL_SEED)
]["speech_shares"]

historical_country = focal_df[
    "country_code"
].to_numpy()

historical_year = focal_df[
    "year"
].to_numpy(dtype=int)

period_definitions = {
    "1972-1990": (1972, 1990),
    "1991-2010": (1991, 2010),
    "2011-2025": (2011, 2025)
}

# ------------------------------------------------------------
# 2. Construct country-period topic contributions
# ------------------------------------------------------------

stratified_records = []

for country in ["IRN", "USA", "CHN"]:

    for period, (start_year, end_year) in (
        period_definitions.items()
    ):

        mask = (
            (historical_country == country)
            & (historical_year >= start_year)
            & (historical_year <= end_year)
        )

        expected_years = (
            end_year - start_year + 1
        )

        assert mask.sum() == expected_years, (
            f"Incomplete balanced cohort: "
            f"{country}, {period}"
        )

        selected_shares = reference_shares[
            mask
        ]

        for topic_idx in range(HISTORICAL_K):

            stratified_records.append({
                "country_code": country,
                "period": period,
                "topic_id": topic_idx + 1,
                "n_speeches": int(mask.sum()),
                "mean_topic_share": float(
                    selected_shares[
                        :, topic_idx
                    ].mean()
                ),
                "sd_topic_share": float(
                    selected_shares[
                        :, topic_idx
                    ].std(ddof=1)
                ),
                "median_topic_share": float(
                    np.median(
                        selected_shares[
                            :, topic_idx
                        ]
                    )
                )
            })

country_period_topic_df = pd.DataFrame(
    stratified_records
)

# ------------------------------------------------------------
# 3. Validate compositional consistency
# ------------------------------------------------------------

assert len(country_period_topic_df) == 63

share_sums = (
    country_period_topic_df
    .groupby(
        ["country_code", "period"]
    )["mean_topic_share"]
    .sum()
)

assert np.allclose(
    share_sums.to_numpy(),
    1.0,
    atol=1e-8
)

# ------------------------------------------------------------
# 4. Calculate descriptive early-to-late changes
# ------------------------------------------------------------

comparison_records = []

for country in ["IRN", "USA", "CHN"]:

    for topic_id in range(1, HISTORICAL_K + 1):

        subset = country_period_topic_df.loc[
            (country_period_topic_df["country_code"] == country)
            & (country_period_topic_df["topic_id"] == topic_id)
        ].set_index("period")

        early = float(
            subset.loc[
                "1972-1990",
                "mean_topic_share"
            ]
        )

        late = float(
            subset.loc[
                "2011-2025",
                "mean_topic_share"
            ]
        )

        comparison_records.append({
            "country_code": country,
            "topic_id": topic_id,
            "early_mean": early,
            "late_mean": late,
            "absolute_change": late - early,
            "change_percentage_points": (
                100 * (late - early)
            )
        })

country_period_change_df = pd.DataFrame(
    comparison_records
)

# ------------------------------------------------------------
# 5. Display country-specific historical patterns
# ------------------------------------------------------------

print("=" * 70)
print("COUNTRY-STRATIFIED HISTORICAL TOPIC VALIDATION")
print("=" * 70)

for country in ["IRN", "USA", "CHN"]:

    print(f"\nCOUNTRY: {country}")

    country_table = (
        country_period_topic_df
        .loc[
            country_period_topic_df[
                "country_code"
            ] == country
        ]
        .pivot(
            index="topic_id",
            columns="period",
            values="mean_topic_share"
        )
        .reindex(
            columns=[
                "1972-1990",
                "1991-2010",
                "2011-2025"
            ]
        )
    )

    display(
        country_table.round(4)
    )

print("\nEarly-to-late descriptive changes:")

display(
    country_period_change_df[
        [
            "country_code",
            "topic_id",
            "early_mean",
            "late_mean",
            "change_percentage_points"
        ]
    ].round(4)
)

print("\nCOUNTRY-PERIOD VALIDATION: PASSED")
print("Temporal inference: NOT YET PERFORMED")



## 12.14. Historical Topic Robustness Across Seeds and Identity Ablation

### Objective

Determine whether observed historical changes in NMF topic contributions remain consistent across initialization seeds and identity-feature ablation.

### Common reference

The original K=7, seed=11 solution defines the reference topic numbering.

All alternative models are aligned to this reference through optimal one-to-one component matching using cosine similarity.

### Analytical sample

- Three countries: Iran, United States, and China.
- Annual coverage: 1972–2025.
- 54 speeches per country.
- Original and identity-ablated K=7 models.
- Three random seeds per specification.

### Outcomes

1. Matched component similarity across specifications.
2. Country-specific changes in mean topic contribution between historical periods.
3. Sensitivity of historical changes to initialization and ablation.
4. Topic-specific country concentration before and after ablation.

### Interpretation safeguards

A high cosine similarity establishes similarity of component loading vectors, not equivalence of their political meanings.

Poorly aligned topics must not be treated as substantively identical.

Historical changes are descriptive until temporal inference is explicitly conducted.

Country-level topic concentration is not by itself evidence of model bias or of meaningful differences in diplomatic priorities.


In [ ]:

# ============================================================
# STEP 12.14A: CROSS-SEED HISTORICAL ROBUSTNESS AUDIT
# ============================================================



# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

K_ROBUST = 7
REFERENCE_SEED = 11
SEEDS_ROBUST = [11, 42, 73]

COUNTRY_ORDER = ["IRN", "USA", "CHN"]
MIN_ALIGNMENT_SIMILARITY = 0.75

PERIOD_DEFINITIONS = {
    "1972-1990": (1972, 1990),
    "1991-2010": (1991, 2010),
    "2011-2025": (2011, 2025)
}

# This threshold is a diagnostic flag, not a validated
# statistical or substantive equivalence criterion.

# ------------------------------------------------------------
# 2. Validate the common balanced cohort
# ------------------------------------------------------------

cohort_robust = focal_df[
    ["country_code", "year"]
].reset_index(drop=True).copy()

cohort_robust["year"] = (
    cohort_robust["year"].astype(int)
)

assert len(cohort_robust) == 162

assert not cohort_robust.duplicated(
    ["country_code", "year"]
).any()

assert set(
    cohort_robust["country_code"]
) == set(COUNTRY_ORDER)

for country in COUNTRY_ORDER:

    observed_years = set(
        cohort_robust.loc[
            cohort_robust["country_code"] == country,
            "year"
        ]
    )

    assert observed_years == set(
        range(1972, 2026)
    )

# ------------------------------------------------------------
# 3. Define a common-reference alignment function
# ------------------------------------------------------------

reference_result = nmf_selection_bank[
    (K_ROBUST, REFERENCE_SEED)
]

H_reference_full = reference_result["H"]

reference_shares_robust = (
    reference_result["speech_shares"]
)

assert reference_shares_robust.shape == (162, 7)


def align_topics_to_reference(
    H_candidate,
    speech_shares_candidate,
    is_ablated=False
):

    H_reference = (
        H_reference_full[:, retained_indices]
        if is_ablated
        else H_reference_full
    )

    assert (
        H_reference.shape
        == H_candidate.shape
    )

    assert speech_shares_candidate.shape == (162, 7)

    similarity = cosine_similarity(
        H_reference,
        H_candidate
    )

    ref_indices, candidate_indices = (
        linear_sum_assignment(-similarity)
    )

    aligned_shares = np.zeros_like(
        speech_shares_candidate
    )

    aligned_cosines = np.zeros(
        K_ROBUST,
        dtype=float
    )

    aligned_topic_ids = np.zeros(
        K_ROBUST,
        dtype=int
    )

    for reference_idx, candidate_idx in zip(
        ref_indices, candidate_indices
    ):

        aligned_shares[:, reference_idx] = (
            speech_shares_candidate[:, candidate_idx]
        )

        aligned_cosines[reference_idx] = (
            similarity[
                reference_idx, candidate_idx
            ]
        )

        aligned_topic_ids[reference_idx] = (
            candidate_idx + 1
        )

    assert np.allclose(
        aligned_shares.sum(axis=1),
        1.0,
        atol=1e-8
    )

    return (
        aligned_shares,
        aligned_cosines,
        aligned_topic_ids
    )


# ------------------------------------------------------------
# 4. Align all six specifications to one topic reference
# ------------------------------------------------------------

aligned_model_shares = {}
alignment_quality_records = []

for specification in ["original", "ablated"]:

    for seed in SEEDS_ROBUST:

        if specification == "original":

            result = nmf_selection_bank[
                (K_ROBUST, seed)
            ]

            H_candidate = result["H"]
            shares_candidate = result["speech_shares"]

        else:

            result = ablation_models[seed]

            H_candidate = result["H"]
            shares_candidate = result["speech_shares"]

        (
            aligned_shares,
            aligned_cosines,
            aligned_topic_ids
        ) = align_topics_to_reference(
            H_candidate,
            shares_candidate,
            is_ablated=(specification == "ablated")
        )

        model_key = (
            f"{specification}_seed{seed}"
        )

        aligned_model_shares[
            model_key
        ] = aligned_shares

        for topic_idx in range(K_ROBUST):

            alignment_quality_records.append({
                "model": model_key,
                "reference_topic": topic_idx + 1,
                "matched_topic": int(
                    aligned_topic_ids[topic_idx]
                ),
                "cosine_similarity": float(
                    aligned_cosines[topic_idx]
                ),
                "alignment_review": bool(
                    aligned_cosines[topic_idx]
                    < MIN_ALIGNMENT_SIMILARITY
                )
            })

alignment_quality_df = pd.DataFrame(
    alignment_quality_records
)

assert len(alignment_quality_df) == 42
assert len(aligned_model_shares) == 6

# ------------------------------------------------------------
# 5. Calculate country-specific historical contributions
# ------------------------------------------------------------

historical_robustness_records = []

country_values = cohort_robust[
    "country_code"
].to_numpy()

year_values = cohort_robust[
    "year"
].to_numpy()

for model_key, shares in aligned_model_shares.items():

    model_alignment = (
        alignment_quality_df
        .loc[
            alignment_quality_df["model"] == model_key
        ]
        .set_index("reference_topic")
    )

    for country in COUNTRY_ORDER:

        for topic_idx in range(K_ROBUST):

            period_means = {}

            for period, bounds in (
                PERIOD_DEFINITIONS.items()
            ):

                start_year, end_year = bounds

                mask = (
                    (country_values == country)
                    & (year_values >= start_year)
                    & (year_values <= end_year)
                )

                assert mask.sum() == (
                    end_year - start_year + 1
                )

                period_means[period] = float(
                    shares[
                        mask, topic_idx
                    ].mean()
                )

            historical_robustness_records.append({
                "model": model_key,
                "country_code": country,
                "topic_id": topic_idx + 1,
                "alignment_cosine": float(
                    model_alignment.loc[
                        topic_idx + 1,
                        "cosine_similarity"
                    ]
                ),
                "early_mean": period_means[
                    "1972-1990"
                ],
                "middle_mean": period_means[
                    "1991-2010"
                ],
                "late_mean": period_means[
                    "2011-2025"
                ],
                "early_to_late_pp": (
                    period_means["2011-2025"]
                    - period_means["1972-1990"]
                ) * 100
            })

historical_robustness_df = pd.DataFrame(
    historical_robustness_records
)

assert len(historical_robustness_df) == 126

# ------------------------------------------------------------
# 6. Summarize historical sensitivity to ablation
# ------------------------------------------------------------

baseline_changes = (
    historical_robustness_df
    .loc[
        historical_robustness_df["model"]
        == "original_seed11",
        [
            "country_code",
            "topic_id",
            "early_to_late_pp"
        ]
    ]
    .rename(
        columns={
            "early_to_late_pp": "original_change_pp"
        }
    )
)

ablated_changes = (
    historical_robustness_df
    .loc[
        historical_robustness_df[
            "model"
        ].str.startswith("ablated_")
    ]
    .copy()
)

ablated_change_summary = (
    ablated_changes
    .groupby(
        ["country_code", "topic_id"]
    )
    .agg(
        mean_ablated_change_pp=(
            "early_to_late_pp", "mean"
        ),
        min_ablated_change_pp=(
            "early_to_late_pp", "min"
        ),
        max_ablated_change_pp=(
            "early_to_late_pp", "max"
        ),
        min_alignment_cosine=(
            "alignment_cosine", "min"
        ),
        n_reliable_alignments=(
            "alignment_cosine",
            lambda x: int(
                (x >= MIN_ALIGNMENT_SIMILARITY).sum()
            )
        )
    )
    .reset_index()
)

historical_sensitivity_summary = (
    baseline_changes
    .merge(
        ablated_change_summary,
        on=["country_code", "topic_id"],
        validate="one_to_one"
    )
)

historical_sensitivity_summary[
    "max_ablation_spread_pp"
] = (
    historical_sensitivity_summary[
        "max_ablated_change_pp"
    ]
    - historical_sensitivity_summary[
        "min_ablated_change_pp"
    ]
)

# ------------------------------------------------------------
# 7. Calculate country concentration before/after ablation
# ------------------------------------------------------------

def country_concentration_table(
    shares,
    model_name
):

    country_mass = np.vstack([
        shares[
            country_values == country
        ].sum(axis=0)
        for country in COUNTRY_ORDER
    ]).T

    probabilities = (
        country_mass
        / country_mass.sum(
            axis=1, keepdims=True
        )
    )

    safe_probabilities = np.where(
        probabilities > 0,
        probabilities,
        1.0
    )

    entropy = -np.sum(
        probabilities * np.log(
            safe_probabilities
        ),
        axis=1
    ) / np.log(3)

    return pd.DataFrame({
        "model": model_name,
        "topic_id": np.arange(1, 8),
        "country_majority_pct": (
            probabilities.max(axis=1) * 100
        ),
        "country_entropy": entropy,
        "leading_country": [
            COUNTRY_ORDER[idx]
            for idx in probabilities.argmax(axis=1)
        ]
    })


country_concentration_df = pd.concat(
    [
        country_concentration_table(
            aligned_model_shares[
                "original_seed11"
            ],
            "Original"
        ),
        country_concentration_table(
            aligned_model_shares[
                "ablated_seed11"
            ],
            "Ablated"
        )
    ],
    ignore_index=True
)

# ------------------------------------------------------------
# 8. Display compact, interpretable outputs
# ------------------------------------------------------------

print("=" * 70)
print("CROSS-SEED COMPONENT ALIGNMENT")
print("=" * 70)

display(
    alignment_quality_df[
        [
            "model",
            "reference_topic",
            "matched_topic",
            "cosine_similarity",
            "alignment_review"
        ]
    ].round(4)
)

print("\n" + "=" * 70)
print("HISTORICAL CHANGE ROBUSTNESS BY COUNTRY")
print("=" * 70)

display(
    historical_sensitivity_summary[
        [
            "country_code",
            "topic_id",
            "original_change_pp",
            "mean_ablated_change_pp",
            "min_ablated_change_pp",
            "max_ablated_change_pp",
            "n_reliable_alignments",
            "max_ablation_spread_pp"
        ]
    ].round(3)
)

print("\n" + "=" * 70)
print("COUNTRY CONCENTRATION: ORIGINAL VS ABLATED")
print("=" * 70)

display(
    country_concentration_df.round(4)
)

print("\nAlignment matches requiring review:")

display(
    alignment_quality_df.loc[
        alignment_quality_df["alignment_review"]
    ].round(4)
)

print("\n" + "=" * 70)
print("ROBUSTNESS VALIDATION")
print("=" * 70)

print(
    "Aligned models:",
    len(aligned_model_shares)
)

print(
    "Historical comparisons:",
    len(historical_robustness_df)
)

print(
    "Low-similarity matches:",
    int(
        alignment_quality_df[
            "alignment_review"
        ].sum()
    )
)

print("Computational validation: PASSED")
print("Substantive robustness: PENDING REVIEW")


In [ ]:

# ============================================================
# STEP 12.14B: HISTORICAL ABLATION SENSITIVITY FIGURE
# ============================================================

# ------------------------------------------------------------
# 1. Figure configuration
# ------------------------------------------------------------

PLOT_TOPICS = [1, 7]

TOPIC_LABELS = {
    1: "Economic Development",
    7: "Nuclear Disarmament"
}

COUNTRY_LABELS = {
    "IRN": "Iran",
    "USA": "United States",
    "CHN": "China"
}

ROLLING_WINDOW = 5

original_shares_plot = aligned_model_shares[
    "original_seed11"
]

ablated_shares_plot = aligned_model_shares[
    "ablated_seed11"
]

# Both specifications must use identical reference topics.
assert original_shares_plot.shape == (162, 7)
assert ablated_shares_plot.shape == (162, 7)

# ------------------------------------------------------------
# 2. Create the diagnostic figure
# ------------------------------------------------------------

fig, axes = plt.subplots(
    nrows=2,
    ncols=3,
    figsize=(14, 7.5),
    sharex=True,
    dpi=150
)

for row_idx, topic_id in enumerate(PLOT_TOPICS):

    topic_idx = topic_id - 1

    for col_idx, country in enumerate(
        COUNTRY_ORDER
    ):

        ax = axes[row_idx, col_idx]

        mask = (
            country_values == country
        )

        country_years = year_values[mask]

        order = np.argsort(
            country_years
        )

        years_sorted = country_years[
            order
        ]

        original_values = (
            original_shares_plot[
                mask, topic_idx
            ][order] * 100
        )

        ablated_values = (
            ablated_shares_plot[
                mask, topic_idx
            ][order] * 100
        )

        original_smooth = (
            pd.Series(original_values)
            .rolling(
                window=ROLLING_WINDOW,
                center=True,
                min_periods=3
            )
            .mean()
            .to_numpy()
        )

        ablated_smooth = (
            pd.Series(ablated_values)
            .rolling(
                window=ROLLING_WINDOW,
                center=True,
                min_periods=3
            )
            .mean()
            .to_numpy()
        )

        ax.plot(
            years_sorted,
            original_smooth,
            color="#1f2937",
            linewidth=2.0,
            label="Original model"
        )

        ax.plot(
            years_sorted,
            ablated_smooth,
            color="#c17b38",
            linewidth=1.8,
            linestyle="--",
            label="Identity-ablated model"
        )

        ax.set_xlim(1972, 2025)
        ax.set_ylim(0, 100)

        ax.grid(
            alpha=0.15
        )

        if row_idx == 0:
            ax.set_title(
                COUNTRY_LABELS[country],
                fontsize=12,
                fontweight="bold",
                pad=9
            )

        if col_idx == 0:
            ax.set_ylabel(
                f"T{topic_id:02d}: "
                f"{TOPIC_LABELS[topic_id]}\n"
                "Topic share (%)",
                fontsize=10
            )

        if row_idx == 1:
            ax.set_xlabel(
                "Year",
                fontsize=10
            )

# ------------------------------------------------------------
# 3. Place the shared legend outside plotting panels
# ------------------------------------------------------------

handles, labels = axes[0, 0].get_legend_handles_labels()

fig.suptitle(
    "Historical Topic Patterns: Original vs Identity-Ablated NMF",
    fontsize=15,
    fontweight="bold",
    y=0.985
)

fig.legend(
    handles,
    labels,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.935),
    ncol=2,
    frameon=False,
    fontsize=10
)

fig.subplots_adjust(
    left=0.09,
    right=0.98,
    bottom=0.09,
    top=0.85,
    wspace=0.20,
    hspace=0.32
)

plt.show()

print("Historical ablation diagnostic: GENERATED")
print("Publication figure export: DEFERRED")



## 12.15. HAC Inference for Historical Differences Between Countries

### Research question

Do changes in relative topic contributions between 1972–1990 and 2011–2025 differ across Iran, the United States, and China?

### Estimand

For each topic and country pair:

\[
\Delta_{A,B} =
(\bar{p}_{A,\mathrm{late}}-\bar{p}_{B,\mathrm{late}})
-
(\bar{p}_{A,\mathrm{early}}-\bar{p}_{B,\mathrm{early}})
\]

where \(p\) denotes the estimated speech-level topic contribution.

### Design

- Balanced annual panel: 1972–2025.
- Three countries and 54 observed speeches per country.
- Common-reference topic alignment across NMF specifications.
- Topics with inadequate cross-specification alignment excluded from the primary comparison.
- Primary covariance estimator: Newey–West HAC, maximum lag 5.
- Sensitivity analysis: HAC lags 3 and 10.
- Family-wise multiplicity adjustment: Holm procedure across all primary country-pair/topic comparisons.

### Statistical interpretation

The analysis tests differences in historical changes in relative topic contributions, not causal effects.

Pairwise annual differencing removes common additive annual components of the estimated topic shares.

HAC inference addresses serial dependence in the paired annual differences, subject to the assumptions of the estimator.

Results remain conditional on the estimated NMF representation and should be evaluated alongside cross-seed, ablation, and qualitative robustness.


In [ ]:

# ============================================================
# STEP 12.15A: PAIRED HISTORICAL CHANGE INFERENCE WITH HAC
# ============================================================



# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

PRIMARY_HAC_LAG = 5
HAC_LAGS = [3, 5, 10]

ALIGNMENT_THRESHOLD = 0.75

COUNTRY_PAIRS = [
    ("IRN", "USA"),
    ("IRN", "CHN"),
    ("USA", "CHN")
]

EXPECTED_YEARS = np.arange(1972, 2026)

COUNTRY_NAMES = {
    "IRN": "Iran",
    "USA": "United States",
    "CHN": "China"
}

# ------------------------------------------------------------
# 2. Identify topics with adequate component alignment
# ------------------------------------------------------------

minimum_alignment = (
    alignment_quality_df
    .groupby("reference_topic")[
        "cosine_similarity"
    ]
    .min()
)

eligible_topics = sorted(
    minimum_alignment[
        minimum_alignment >= ALIGNMENT_THRESHOLD
    ].index.astype(int).tolist()
)

excluded_topics = sorted(
    set(range(1, 8)) - set(eligible_topics)
)

print("=" * 70)
print("HAC INFERENCE DESIGN AUDIT")
print("=" * 70)

print("Eligible topic IDs:", eligible_topics)
print("Excluded topic IDs:", excluded_topics)
print("Country pairs:", len(COUNTRY_PAIRS))
print("Annual observations per country:", 54)
print("Primary HAC lag:", PRIMARY_HAC_LAG)

assert len(eligible_topics) > 0

# ------------------------------------------------------------
# 3. Validate the balanced annual sample
# ------------------------------------------------------------

country_vector = cohort_robust[
    "country_code"
].to_numpy()

year_vector = cohort_robust[
    "year"
].to_numpy(dtype=int)

original_topic_shares = aligned_model_shares[
    "original_seed11"
]

assert original_topic_shares.shape == (162, 7)

assert np.allclose(
    original_topic_shares.sum(axis=1),
    1.0,
    atol=1e-8
)

# ------------------------------------------------------------
# 4. Retrieve calendar-aligned annual topic shares
# ------------------------------------------------------------

def country_topic_series(shares, country, topic_id):

    indices = np.flatnonzero(
        country_vector == country
    )

    indices = indices[
        np.argsort(year_vector[indices])
    ]

    observed_years = year_vector[indices]

    assert np.array_equal(
        observed_years,
        EXPECTED_YEARS
    ), f"Incomplete historical series: {country}"

    return shares[
        indices,
        topic_id - 1
    ]


# ------------------------------------------------------------
# 5. Define the HAC period-contrast estimator
# ------------------------------------------------------------

def fit_historical_hac(series, max_lag):

    series = np.asarray(
        series, dtype=float
    )

    assert series.shape == (54,)
    assert np.isfinite(series).all()

    middle_indicator = (
        (EXPECTED_YEARS >= 1991)
        & (EXPECTED_YEARS <= 2010)
    ).astype(float)

    late_indicator = (
        EXPECTED_YEARS >= 2011
    ).astype(float)

    design = np.column_stack([
        np.ones(54),
        middle_indicator,
        late_indicator
    ])

    assert np.linalg.matrix_rank(design) == 3

    fitted = sm.OLS(
        series,
        design
    ).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": max_lag,
            "use_correction": True
        }
    )

    confidence_interval = np.asarray(
        fitted.conf_int(alpha=0.05)
    )[2]

    return {
        "estimate": float(fitted.params[2]),
        "hac_se": float(fitted.bse[2]),
        "ci_lower": float(confidence_interval[0]),
        "ci_upper": float(confidence_interval[1]),
        "p_value": float(fitted.pvalues[2])
    }


# ------------------------------------------------------------
# 6. Estimate country-pair changes
# ------------------------------------------------------------

hac_records = []

early_mask = (
    EXPECTED_YEARS <= 1990
)

late_mask = (
    EXPECTED_YEARS >= 2011
)

for country_a, country_b in COUNTRY_PAIRS:

    pair_name = (
        f"{country_a}-{country_b}"
    )

    for topic_id in eligible_topics:

        series_a = country_topic_series(
            original_topic_shares,
            country_a,
            topic_id
        )

        series_b = country_topic_series(
            original_topic_shares,
            country_b,
            topic_id
        )

        # Convert the topic-share difference to
        # percentage points before estimation.
        annual_gap = (
            series_a - series_b
        ) * 100.0

        early_gap = float(
            annual_gap[early_mask].mean()
        )

        late_gap = float(
            annual_gap[late_mask].mean()
        )

        for lag in HAC_LAGS:

            estimate = fit_historical_hac(
                annual_gap,
                max_lag=lag
            )

            hac_records.append({
                "pair": pair_name,
                "topic_id": topic_id,
                "hac_lag": lag,
                "n_years": len(EXPECTED_YEARS),
                "early_gap_pp": early_gap,
                "late_gap_pp": late_gap,
                "change_in_gap_pp": estimate[
                    "estimate"
                ],
                "hac_se": estimate["hac_se"],
                "ci_lower_pp": estimate["ci_lower"],
                "ci_upper_pp": estimate["ci_upper"],
                "p_value": estimate["p_value"]
            })

hac_inference_df = pd.DataFrame(
    hac_records
)

# ------------------------------------------------------------
# 7. Validate effect estimates
# ------------------------------------------------------------

expected_comparisons = (
    len(COUNTRY_PAIRS) * len(eligible_topics)
)

assert len(hac_inference_df) == (
    expected_comparisons * len(HAC_LAGS)
)

assert hac_inference_df[
    "p_value"
].between(0, 1).all()

assert np.allclose(
    hac_inference_df["change_in_gap_pp"],
    hac_inference_df["late_gap_pp"]
    - hac_inference_df["early_gap_pp"],
    atol=1e-8
)

# ------------------------------------------------------------
# 8. Apply Holm adjustment to the primary test family
# ------------------------------------------------------------

primary_hac_df = (
    hac_inference_df.loc[
        hac_inference_df["hac_lag"]
        == PRIMARY_HAC_LAG
    ]
    .copy()
    .reset_index(drop=True)
)

reject, adjusted_p, _, _ = multipletests(
    primary_hac_df["p_value"].to_numpy(),
    alpha=0.05,
    method="holm"
)

primary_hac_df["holm_adjusted_p"] = adjusted_p
primary_hac_df["holm_reject_005"] = reject

assert len(primary_hac_df) == expected_comparisons

# ------------------------------------------------------------
# 9. Display primary inferential results
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("PRIMARY PAIRED HAC INFERENCE")
print("=" * 70)

print(
    "Tested contrasts:",
    len(primary_hac_df)
)

print(
    "Multiplicity correction: Holm"
)

primary_display = primary_hac_df[
    [
        "pair",
        "topic_id",
        "early_gap_pp",
        "late_gap_pp",
        "change_in_gap_pp",
        "ci_lower_pp",
        "ci_upper_pp",
        "p_value",
        "holm_adjusted_p",
        "holm_reject_005"
    ]
].copy()

for column in [
    "early_gap_pp",
    "late_gap_pp",
    "change_in_gap_pp",
    "ci_lower_pp",
    "ci_upper_pp"
]:
    primary_display[column] = (
        primary_display[column].round(3)
    )

for column in [
    "p_value",
    "holm_adjusted_p"
]:
    primary_display[column] = (
        primary_display[column]
        .map(lambda value: f"{value:.3e}")
    )

display(primary_display)

# ------------------------------------------------------------
# 10. Examine HAC bandwidth sensitivity
# ------------------------------------------------------------

bandwidth_sensitivity_df = (
    hac_inference_df
    .pivot(
        index=["pair", "topic_id"],
        columns="hac_lag",
        values="hac_se"
    )
    .rename(
        columns={
            3: "se_lag3",
            5: "se_lag5",
            10: "se_lag10"
        }
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("HAC BANDWIDTH SENSITIVITY")
print("=" * 70)

display(
    bandwidth_sensitivity_df.round(4)
)

print("\nHAC INFERENCE COMPUTATION: PASSED")
print("Confirmatory interpretation: NOT WARRANTED")


In [ ]:

# ============================================================
# STEP 12.15B: CROSS-MODEL ROBUSTNESS OF COUNTRY COMPARISONS
# ============================================================

# ------------------------------------------------------------
# 1. Construct model-specific country changes
# ------------------------------------------------------------

country_change_matrix = (
    historical_robustness_df
    .loc[
        historical_robustness_df[
            "topic_id"
        ].isin(eligible_topics)
    ]
    .pivot(
        index=["model", "topic_id"],
        columns="country_code",
        values="early_to_late_pp"
    )
)

assert country_change_matrix.notna().all().all()

# ------------------------------------------------------------
# 2. Derive the paired historical contrasts
# ------------------------------------------------------------

model_contrast_records = []

for (model_name, topic_id), row in (
    country_change_matrix.iterrows()
):

    for country_a, country_b in COUNTRY_PAIRS:

        model_contrast_records.append({
            "model": model_name,
            "topic_id": int(topic_id),
            "pair": f"{country_a}-{country_b}",
            "effect_pp": float(
                row[country_a] - row[country_b]
            )
        })

model_contrast_df = pd.DataFrame(
    model_contrast_records
)

expected_rows = (
    6 * len(COUNTRY_PAIRS) * len(eligible_topics)
)

assert len(model_contrast_df) == expected_rows

# ------------------------------------------------------------
# 3. Verify consistency with the primary HAC estimates
# ------------------------------------------------------------

reference_contrasts = (
    model_contrast_df.loc[
        model_contrast_df["model"]
        == "original_seed11"
    ]
    .merge(
        primary_hac_df[
            ["pair", "topic_id", "change_in_gap_pp"]
        ],
        on=["pair", "topic_id"],
        validate="one_to_one"
    )
)

assert np.allclose(
    reference_contrasts["effect_pp"],
    reference_contrasts["change_in_gap_pp"],
    atol=1e-8
)

# ------------------------------------------------------------
# 4. Summarize model sensitivity
# ------------------------------------------------------------

robustness_records = []

for (pair, topic_id), group in (
    model_contrast_df.groupby(
        ["pair", "topic_id"]
    )
):

    reference_effect = float(
        group.loc[
            group["model"] == "original_seed11",
            "effect_pp"
        ].iloc[0]
    )

    effects = group[
        "effect_pp"
    ].to_numpy()

    same_direction = np.all(
        np.sign(effects) == np.sign(reference_effect)
    )

    robustness_records.append({
        "pair": pair,
        "topic_id": topic_id,
        "reference_effect_pp": reference_effect,
        "min_model_effect_pp": float(
            effects.min()
        ),
        "max_model_effect_pp": float(
            effects.max()
        ),
        "model_effect_range_pp": float(
            np.ptp(effects)
        ),
        "same_direction_all_models": bool(
            same_direction
        ),
        "n_models": len(effects)
    })

comparative_robustness_df = (
    pd.DataFrame(robustness_records)
    .sort_values(["pair", "topic_id"])
    .reset_index(drop=True)
)

assert (
    comparative_robustness_df["n_models"] == 6
).all()

# ------------------------------------------------------------
# 5. Display the main robustness summary
# ------------------------------------------------------------

print("=" * 70)
print("CROSS-MODEL COMPARATIVE ROBUSTNESS")
print("=" * 70)

display(
    comparative_robustness_df.round(3)
)

# ------------------------------------------------------------
# 6. Inspect selected substantive topics
# ------------------------------------------------------------

FOCUS_TOPICS = [
    topic_id
    for topic_id in [1, 3, 7]
    if topic_id in eligible_topics
]

print("\n" + "=" * 70)
print("MODEL-SPECIFIC EFFECTS FOR SELECTED TOPICS")
print("=" * 70)

selected_model_effects = (
    model_contrast_df.loc[
        model_contrast_df[
            "topic_id"
        ].isin(FOCUS_TOPICS)
    ]
    .pivot(
        index=["pair", "topic_id"],
        columns="model",
        values="effect_pp"
    )
)

display(
    selected_model_effects.round(3)
)

# ------------------------------------------------------------
# 7. Final diagnostic summary
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("COMPARATIVE ROBUSTNESS VALIDATION")
print("=" * 70)

print(
    "Country-pair/topic comparisons:",
    len(comparative_robustness_df)
)

print(
    "Directionally consistent comparisons:",
    int(
        comparative_robustness_df[
            "same_direction_all_models"
        ].sum()
    )
)

print(
    "Comparisons requiring directional review:",
    int(
        (
            ~comparative_robustness_df[
                "same_direction_all_models"
            ]
        ).sum()
    )
)

print("\nCROSS-MODEL ROBUSTNESS: COMPUTATION PASSED")
print("Final substantive conclusions: PENDING REVIEW")


In [ ]:

# ============================================================
# STEP 12.16A: MODEL AND HAC BANDWIDTH ROBUSTNESS
# ============================================================



# ------------------------------------------------------------
# 1. Validate upstream analysis objects
# ------------------------------------------------------------

required_objects = [
    "aligned_model_shares",
    "alignment_quality_df",
    "eligible_topics",
    "COUNTRY_PAIRS",
    "EXPECTED_YEARS",
    "country_topic_series",
    "fit_historical_hac",
    "primary_hac_df"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        "Missing upstream objects: "
        + ", ".join(missing_objects)
    )

MODEL_HAC_LAGS = (3, 5, 10)
PRIMARY_LAG = 5
ALPHA = 0.05
MIN_ALIGNMENT = 0.75

assert len(aligned_model_shares) == 6
assert len(eligible_topics) == 6
assert len(COUNTRY_PAIRS) == 3

# ------------------------------------------------------------
# 2. Re-estimate historical contrasts
# ------------------------------------------------------------

model_hac_records = []

for model_name, shares in sorted(
    aligned_model_shares.items()
):

    assert shares.shape == (162, 7)

    model_alignment = (
        alignment_quality_df.loc[
            alignment_quality_df["model"] == model_name
        ]
        .set_index("reference_topic")
    )

    for topic_id in eligible_topics:

        similarity = float(
            model_alignment.loc[
                topic_id,
                "cosine_similarity"
            ]
        )

        assert similarity >= MIN_ALIGNMENT, (
            f"Unreliable alignment: "
            f"{model_name}, topic {topic_id}"
        )

        for country_a, country_b in COUNTRY_PAIRS:

            series_a = country_topic_series(
                shares,
                country_a,
                topic_id
            )

            series_b = country_topic_series(
                shares,
                country_b,
                topic_id
            )

            annual_gap_pp = (
                series_a - series_b
            ) * 100.0

            for lag in MODEL_HAC_LAGS:

                result = fit_historical_hac(
                    annual_gap_pp,
                    max_lag=lag
                )

                model_hac_records.append({
                    "model": model_name,
                    "pair": f"{country_a}-{country_b}",
                    "topic_id": int(topic_id),
                    "hac_lag": int(lag),
                    "effect_pp": result["estimate"],
                    "hac_se": result["hac_se"],
                    "ci_lower_pp": result["ci_lower"],
                    "ci_upper_pp": result["ci_upper"],
                    "p_value": result["p_value"]
                })

model_hac_df = pd.DataFrame(
    model_hac_records
)

assert len(model_hac_df) == 324
assert np.isfinite(
    model_hac_df[
        ["effect_pp", "hac_se", "p_value"]
    ].to_numpy(dtype=float)
).all()

# ------------------------------------------------------------
# 3. Apply Holm adjustment within each specification
# ------------------------------------------------------------

model_hac_df["holm_adjusted_p"] = np.nan
model_hac_df["holm_reject_005"] = False

for (model_name, lag), group in model_hac_df.groupby(
    ["model", "hac_lag"]
):

    assert len(group) == 18

    rejected, adjusted, _, _ = multipletests(
        group["p_value"].to_numpy(),
        alpha=ALPHA,
        method="holm"
    )

    model_hac_df.loc[
        group.index,
        "holm_adjusted_p"
    ] = adjusted

    model_hac_df.loc[
        group.index,
        "holm_reject_005"
    ] = rejected

assert model_hac_df[
    "holm_adjusted_p"
].notna().all()

# ------------------------------------------------------------
# 4. Verify exact reproduction of primary results
# ------------------------------------------------------------

reproduced_primary = (
    model_hac_df.loc[
        (model_hac_df["model"] == "original_seed11")
        & (model_hac_df["hac_lag"] == PRIMARY_LAG)
    ]
    .merge(
        primary_hac_df[
            [
                "pair",
                "topic_id",
                "change_in_gap_pp",
                "holm_adjusted_p"
            ]
        ],
        on=["pair", "topic_id"],
        suffixes=("_new", "_previous"),
        validate="one_to_one"
    )
)

assert len(reproduced_primary) == 18

assert np.allclose(
    reproduced_primary["effect_pp"],
    reproduced_primary["change_in_gap_pp"],
    atol=1e-8
)

assert np.allclose(
    reproduced_primary["holm_adjusted_p_new"],
    reproduced_primary["holm_adjusted_p_previous"],
    atol=1e-8
)

# ------------------------------------------------------------
# 5. Summarize sensitivity across models and bandwidths
# ------------------------------------------------------------

robustness_summary_records = []

for (pair, topic_id), group in model_hac_df.groupby(
    ["pair", "topic_id"]
):

    reference_row = group.loc[
        (group["model"] == "original_seed11")
        & (group["hac_lag"] == PRIMARY_LAG)
    ].iloc[0]

    primary_lag_rows = group.loc[
        group["hac_lag"] == PRIMARY_LAG
    ]

    effects = primary_lag_rows[
        "effect_pp"
    ].to_numpy(dtype=float)

    reference_effect = float(
        reference_row["effect_pp"]
    )

    robustness_summary_records.append({
        "pair": pair,
        "topic_id": int(topic_id),
        "reference_effect_pp": reference_effect,
        "min_effect_pp": float(effects.min()),
        "max_effect_pp": float(effects.max()),
        "all_models_same_direction": bool(
            np.all(
                np.sign(effects)
                == np.sign(reference_effect)
            )
        ),
        "holm_significant_models_lag5": int(
            primary_lag_rows[
                "holm_reject_005"
            ].sum()
        ),
        "holm_significant_of_18": int(
            group["holm_reject_005"].sum()
        ),
        "min_adjusted_p": float(
            group["holm_adjusted_p"].min()
        ),
        "max_adjusted_p": float(
            group["holm_adjusted_p"].max()
        )
    })

model_hac_robustness_summary = (
    pd.DataFrame(robustness_summary_records)
    .sort_values(["pair", "topic_id"])
    .reset_index(drop=True)
)

assert len(model_hac_robustness_summary) == 18

# ------------------------------------------------------------
# 6. Display the robustness summary
# ------------------------------------------------------------

print("=" * 70)
print("MODEL AND HAC ROBUSTNESS SUMMARY")
print("=" * 70)

display(
    model_hac_robustness_summary.round(4)
)

print("\n" + "=" * 70)
print("PRIMARY SIGNIFICANT CONTRASTS: MODEL SENSITIVITY")
print("=" * 70)

reference_significant = primary_hac_df.loc[
    primary_hac_df["holm_reject_005"],
    ["pair", "topic_id"]
]

significant_robustness = (
    reference_significant
    .merge(
        model_hac_robustness_summary,
        on=["pair", "topic_id"],
        validate="one_to_one"
    )
)

display(
    significant_robustness.round(4)
)

print("\n" + "=" * 70)
print("ROBUSTNESS AUDIT")
print("=" * 70)

print(
    "Total HAC fits:",
    len(model_hac_df)
)

print(
    "Primary significant contrasts:",
    len(reference_significant)
)

print(
    "Contrasts significant in every model at lag 5:",
    int(
        (
            model_hac_robustness_summary[
                "holm_significant_models_lag5"
            ] == 6
        ).sum()
    )
)

print(
    "Contrasts significant in all 18 settings:",
    int(
        (
            model_hac_robustness_summary[
                "holm_significant_of_18"
            ] == 18
        ).sum()
    )
)

print("\nMODEL AND HAC ROBUSTNESS: COMPUTATION PASSED")
print("Scientific interpretation: PENDING")


In [ ]:

# ============================================================
# STEP 12.16B: HISTORICAL PERIOD BOUNDARY SENSITIVITY
# ============================================================

# ------------------------------------------------------------
# 1. Define alternative historical period boundaries
# ------------------------------------------------------------

PERIOD_SCENARIOS = {
    "primary": {
        "early": (1972, 1990),
        "late": (2011, 2025)
    },
    "narrow": {
        "early": (1972, 1987),
        "late": (2014, 2025)
    },
    "broad": {
        "early": (1972, 1993),
        "late": (2009, 2025)
    }
}

period_sensitivity_records = []

# ------------------------------------------------------------
# 2. Estimate contrasts across all models and periods
# ------------------------------------------------------------

for model_name, shares in sorted(
    aligned_model_shares.items()
):

    for country_a, country_b in COUNTRY_PAIRS:

        pair_name = f"{country_a}-{country_b}"

        for topic_id in eligible_topics:

            annual_gap_pp = (
                country_topic_series(
                    shares, country_a, topic_id
                )
                - country_topic_series(
                    shares, country_b, topic_id
                )
            ) * 100.0

            for scenario, boundaries in (
                PERIOD_SCENARIOS.items()
            ):

                early_start, early_end = boundaries["early"]
                late_start, late_end = boundaries["late"]

                assert early_end < late_start

                early_mask = (
                    (EXPECTED_YEARS >= early_start)
                    & (EXPECTED_YEARS <= early_end)
                )

                late_mask = (
                    (EXPECTED_YEARS >= late_start)
                    & (EXPECTED_YEARS <= late_end)
                )

                assert early_mask.sum() >= 10
                assert late_mask.sum() >= 10

                early_mean = float(
                    annual_gap_pp[early_mask].mean()
                )

                late_mean = float(
                    annual_gap_pp[late_mask].mean()
                )

                period_sensitivity_records.append({
                    "model": model_name,
                    "pair": pair_name,
                    "topic_id": topic_id,
                    "scenario": scenario,
                    "n_early": int(early_mask.sum()),
                    "n_late": int(late_mask.sum()),
                    "early_gap_pp": early_mean,
                    "late_gap_pp": late_mean,
                    "effect_pp": late_mean - early_mean
                })

period_sensitivity_df = pd.DataFrame(
    period_sensitivity_records
)

assert len(period_sensitivity_df) == 324

# ------------------------------------------------------------
# 3. Confirm primary-period reproducibility
# ------------------------------------------------------------

primary_reproduced = (
    period_sensitivity_df.loc[
        (period_sensitivity_df["model"] == "original_seed11")
        & (period_sensitivity_df["scenario"] == "primary")
    ]
    .merge(
        primary_hac_df[
            ["pair", "topic_id", "change_in_gap_pp"]
        ],
        on=["pair", "topic_id"],
        validate="one_to_one"
    )
)

assert len(primary_reproduced) == 18

assert np.allclose(
    primary_reproduced["effect_pp"],
    primary_reproduced["change_in_gap_pp"],
    atol=1e-8
)

# ------------------------------------------------------------
# 4. Compare historical period definitions
# ------------------------------------------------------------

reference_period_table = (
    period_sensitivity_df.loc[
        period_sensitivity_df["model"] == "original_seed11"
    ]
    .pivot(
        index=["pair", "topic_id"],
        columns="scenario",
        values="effect_pp"
    )
    .reset_index()
)

reference_period_table.columns.name = None

reference_period_table = reference_period_table[
    [
        "pair",
        "topic_id",
        "primary",
        "narrow",
        "broad"
    ]
]

print("=" * 70)
print("HISTORICAL PERIOD SENSITIVITY: REFERENCE MODEL")
print("=" * 70)

display(
    reference_period_table.round(3)
)

# ------------------------------------------------------------
# 5. Evaluate all model-period combinations
# ------------------------------------------------------------

period_robustness_records = []

for (pair, topic_id), group in (
    period_sensitivity_df.groupby(
        ["pair", "topic_id"]
    )
):

    baseline = float(
        group.loc[
            (group["model"] == "original_seed11")
            & (group["scenario"] == "primary"),
            "effect_pp"
        ].iloc[0]
    )

    all_effects = group[
        "effect_pp"
    ].to_numpy(dtype=float)

    matching_signs = (
        np.sign(all_effects)
        == np.sign(baseline)
    )

    period_robustness_records.append({
        "pair": pair,
        "topic_id": topic_id,
        "baseline_effect_pp": baseline,
        "minimum_effect_pp": float(
            all_effects.min()
        ),
        "maximum_effect_pp": float(
            all_effects.max()
        ),
        "n_matching_directions": int(
            matching_signs.sum()
        ),
        "n_specifications": len(all_effects),
        "all_directions_consistent": bool(
            matching_signs.all()
        )
    })

period_robustness_summary = (
    pd.DataFrame(period_robustness_records)
    .sort_values(["pair", "topic_id"])
    .reset_index(drop=True)
)

assert len(period_robustness_summary) == 18

assert (
    period_robustness_summary["n_specifications"]
    == 18
).all()

print("\n" + "=" * 70)
print("CROSS-MODEL AND PERIOD ROBUSTNESS")
print("=" * 70)

display(
    period_robustness_summary.round(3)
)

# ------------------------------------------------------------
# 6. Summarize sensitivity of primary significant findings
# ------------------------------------------------------------

primary_significant_keys = (
    primary_hac_df.loc[
        primary_hac_df["holm_reject_005"],
        ["pair", "topic_id"]
    ]
)

significant_period_robustness = (
    primary_significant_keys
    .merge(
        period_robustness_summary,
        on=["pair", "topic_id"],
        validate="one_to_one"
    )
)

print("\n" + "=" * 70)
print("PERIOD ROBUSTNESS OF PRIMARY SIGNIFICANT EFFECTS")
print("=" * 70)

display(
    significant_period_robustness.round(3)
)

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("HISTORICAL PERIOD VALIDATION")
print("=" * 70)

print(
    "Total sensitivity estimates:",
    len(period_sensitivity_df)
)

print(
    "Fully directionally consistent comparisons:",
    int(
        period_robustness_summary[
            "all_directions_consistent"
        ].sum()
    )
)

print(
    "Primary significant comparisons stable across all periods:",
    int(
        significant_period_robustness[
            "all_directions_consistent"
        ].sum()
    ),
    "/",
    len(significant_period_robustness)
)

print("\nPERIOD SENSITIVITY: COMPUTATION PASSED")
print("Historical interpretation: PENDING REVIEW")



## Step 12.17 — Publication-Ready Statistical Results

### Objective
To consolidate the primary inferential findings and identify historical changes in topic prevalence that remain statistically significant across alternative model specifications.

### Analytical Framework
The analysis compares Iran, the United States, and China using a balanced corpus of 162 United Nations General Debate speeches covering 1972–2025.

Historical changes in bilateral topic-share differences are evaluated using:

- Newey–West heteroskedasticity and autocorrelation-consistent (HAC) inference.
- Holm correction for multiple hypothesis testing across 18 primary contrasts.
- Six NMF model specifications, including original and identity-ablated models.
- HAC bandwidth sensitivity at lags 3, 5, and 10.
- Sensitivity analyses using alternative historical period definitions.

### Robustness Criterion
A finding is classified as robust when it is statistically significant after Holm adjustment in all 18 model–bandwidth combinations and maintains a consistent effect direction across the evaluated historical-period specifications.

### Outputs
- **Table 03:** Primary robust historical topic contrasts.
- **Figure 04:** Forest plot displaying effect estimates and 95% HAC confidence intervals.
- **Supplementary Table S01:** Full results for all 18 primary comparisons.

### Interpretation and Limitations
The reported estimates describe changes in between-country differences in average topic shares, measured in percentage points. They should not be interpreted as causal effects.

The displayed 95% confidence intervals are HAC-based but are not adjusted for multiplicity. Robustness classification is exploratory and does not substitute for external validation.

Substantive interpretations of topic labels remain provisional pending final qualitative validation.


In [ ]:

# ============================================================
# STEP 12.17A
# PUBLICATION RESULTS TABLE PREPARATION
# No files are saved in this cell.
# ============================================================

required = [
    "model_hac_df",
    "model_hac_robustness_summary",
    "period_robustness_summary"
]

missing = [x for x in required if x not in globals()]

if missing:
    raise RuntimeError(
        "Missing upstream objects: " + ", ".join(missing)
    )

TOPIC_LABELS = {
    1: "Economic development and cooperation",
    2: "China and peaceful development",
    3: "Kampuchea and regional conflicts",
    4: "Iran and Islamic Republic",
    5: "Human rights and freedom discourse",
    7: "Nuclear arms and disarmament"
}

PAIR_LABELS = {
    "IRN-USA": "Iran - United States",
    "IRN-CHN": "Iran - China",
    "USA-CHN": "United States - China"
}

# Primary reference specification
reference = model_hac_df.loc[
    (model_hac_df["model"] == "original_seed11")
    & (model_hac_df["hac_lag"] == 5)
].copy()

assert len(reference) == 18

# Attach robustness results
paper_results_all18 = (
    reference[
        [
            "pair", "topic_id", "effect_pp",
            "hac_se", "ci_lower_pp", "ci_upper_pp",
            "p_value", "holm_adjusted_p",
            "holm_reject_005"
        ]
    ]
    .merge(
        model_hac_robustness_summary[
            [
                "pair", "topic_id",
                "holm_significant_of_18",
                "holm_significant_models_lag5",
                "all_models_same_direction",
                "min_effect_pp", "max_effect_pp"
            ]
        ],
        on=["pair", "topic_id"],
        validate="one_to_one"
    )
    .merge(
        period_robustness_summary[
            [
                "pair", "topic_id",
                "n_matching_directions",
                "n_specifications",
                "all_directions_consistent"
            ]
        ],
        on=["pair", "topic_id"],
        validate="one_to_one"
    )
)

paper_results_all18["topic_label"] = (
    paper_results_all18["topic_id"]
    .map(TOPIC_LABELS)
)

paper_results_all18["pair_label"] = (
    paper_results_all18["pair"]
    .map(PAIR_LABELS)
)

assert paper_results_all18["topic_label"].notna().all()
assert paper_results_all18["pair_label"].notna().all()

# Evidence classification
paper_results_all18["evidence_category"] = np.select(
    [
        (
            paper_results_all18["holm_reject_005"]
            & paper_results_all18[
                "holm_significant_of_18"
            ].eq(18)
        ),
        paper_results_all18["holm_reject_005"]
    ],
    [
        "Robust across all 18 specifications",
        "Primary significant; model-sensitive"
    ],
    default="Not significant in primary model"
)

paper_results_all18["robust_primary"] = (
    paper_results_all18["evidence_category"]
    == "Robust across all 18 specifications"
)

paper_results_all18 = (
    paper_results_all18
    .sort_values(["pair", "topic_id"])
    .reset_index(drop=True)
)

# Main publication candidate: five robust contrasts
paper_results_main5 = (
    paper_results_all18.loc[
        paper_results_all18["robust_primary"]
    ]
    .copy()
    .reset_index(drop=True)
)

assert len(paper_results_all18) == 18
assert len(paper_results_main5) == 5

assert (
    paper_results_main5["all_directions_consistent"]
).all()

assert (
    paper_results_main5["holm_significant_of_18"]
    == 18
).all()

assert (
    paper_results_main5["ci_lower_pp"]
    <= paper_results_main5["effect_pp"]
).all()

assert (
    paper_results_main5["ci_upper_pp"]
    >= paper_results_main5["effect_pp"]
).all()

# Display fields for article review
cols = [
    "pair_label",
    "topic_id",
    "topic_label",
    "effect_pp",
    "ci_lower_pp",
    "ci_upper_pp",
    "holm_adjusted_p",
    "holm_significant_of_18",
    "n_matching_directions"
]

print("=" * 72)
print("PRIMARY ROBUST FINDINGS: PUBLICATION CANDIDATES")
print("=" * 72)

display(paper_results_main5[cols].round(5))

print("\n" + "=" * 72)
print("FULL INFERENTIAL RESULTS: ALL 18 COMPARISONS")
print("=" * 72)

display(
    paper_results_all18[
        cols + ["evidence_category"]
    ].round(5)
)

print("\nPublication table preparation: PASSED")
print("Files saved: NONE")


In [ ]:

# ============================================================
# STEP 12.17B
# PUBLICATION FIGURE: ROBUST HISTORICAL TOPIC CONTRASTS
# No files are saved in this cell.
# ============================================================

assert "paper_results_main5" in globals()

plot_df = paper_results_main5.copy()

pair_order = {
    "IRN-USA": 0,
    "IRN-CHN": 1,
    "USA-CHN": 2
}

plot_df["pair_order"] = (
    plot_df["pair"].map(pair_order)
)

plot_df = (
    plot_df
    .sort_values(["pair_order", "topic_id"])
    .reset_index(drop=True)
)

short_topic_names = {
    3: "Regional conflicts / Kampuchea",
    5: "Human rights and freedom",
    7: "Nuclear arms and disarmament"
}

topic_colors = {
    3: "#2878A5",
    5: "#7553A4",
    7: "#C77B2B"
}

fig_main, ax = plt.subplots(
    figsize=(12.4, 6.4),
    dpi=160
)

n = len(plot_df)
y_positions = np.arange(n)[::-1]

labels = []

for i, row in plot_df.iterrows():

    y = y_positions[i]

    color = topic_colors[int(row["topic_id"])]

    # HAC 95% confidence interval
    ax.hlines(
        y,
        row["ci_lower_pp"],
        row["ci_upper_pp"],
        color=color,
        linewidth=2.4,
        zorder=2
    )

    # Confidence interval caps
    ax.vlines(
        [row["ci_lower_pp"], row["ci_upper_pp"]],
        y - 0.10,
        y + 0.10,
        color=color,
        linewidth=1.7,
        zorder=2
    )

    # Point estimate
    ax.scatter(
        row["effect_pp"],
        y,
        s=85,
        color=color,
        edgecolor="white",
        linewidth=1,
        zorder=3
    )

    pair_label = row["pair_label"]
    topic_id = int(row["topic_id"])

    labels.append(
        f"{pair_label}  |  T{topic_id:02d}: "
        f"{short_topic_names[topic_id]}"
    )

    # Place adjusted p-values outside the plotting area
    p = float(row["holm_adjusted_p"])

    if p < 0.0001:
        p_label = "p < 0.0001"
    else:
        p_label = f"p = {p:.4f}"

    ax.text(
        1.025,
        y,
        p_label,
        transform=ax.get_yaxis_transform(),
        ha="left",
        va="center",
        fontsize=10,
        clip_on=False
    )

ax.set_yticks(y_positions)
ax.set_yticklabels(labels, fontsize=10)

# Zero-effect reference line
ax.axvline(
    0,
    color="#5C626A",
    linestyle="--",
    linewidth=1.2,
    alpha=0.85
)

x_min = min(
    0,
    float(plot_df["ci_lower_pp"].min())
) - 5

x_max = max(
    0,
    float(plot_df["ci_upper_pp"].max())
) + 5

ax.set_xlim(x_min, x_max)
ax.set_ylim(-0.6, n - 0.4)

ax.set_xlabel(
    "Change in between-country topic-share gap "
    "(percentage points)",
    fontsize=11,
    labelpad=12
)

ax.set_title(
    "Robust Historical Changes in Bilateral Topic Gaps",
    fontsize=15,
    fontweight="bold",
    pad=20
)

ax.grid(
    axis="x",
    linestyle="-",
    linewidth=0.5,
    alpha=0.23
)

ax.set_axisbelow(True)

for spine in ["top", "right", "left"]:
    ax.spines[spine].set_visible(False)

ax.tick_params(axis="y", length=0)

fig_main.subplots_adjust(
    left=0.405,
    right=0.815,
    top=0.80,
    bottom=0.24
)

fig_main.text(
    0.05,
    0.125,
    "Dots indicate estimated changes; horizontal lines "
    "represent 95% HAC confidence intervals.",
    fontsize=9.5
)

fig_main.text(
    0.05,
    0.085,
    "Reported p-values are Holm-adjusted across "
    "18 primary contrasts; confidence intervals are not "
    "multiplicity-adjusted.",
    fontsize=9.5
)

fig_main.text(
    0.05,
    0.045,
    "All five findings retain adjusted significance across "
    "six NMF solutions and three HAC bandwidths.",
    fontsize=9.5
)

plt.show()

print("Publication candidate figure generated.")
print("Files saved: NONE")


In [ ]:

# # ============================================================
# # SAVE FIGURE 04 ONLY
# # ============================================================


# FIGURE_NUMBER = "04"

# processed_dir = Path(
#     globals().get("DATA_PROCESSED_DIR", "Data_Processed")
# )

# figure_dir = processed_dir / "Article_Outputs" / "Figures"
# figure_dir.mkdir(parents=True, exist_ok=True)

# figure_name = (
#     f"Figure_{FIGURE_NUMBER}_"
#     "Robust_Historical_Topic_Contrasts"
# )

# png_path = figure_dir / f"{figure_name}.png"
# pdf_path = figure_dir / f"{figure_name}.pdf"

# fig_main.savefig(
#     png_path,
#     dpi=600,
#     bbox_inches="tight",
#     facecolor="white"
# )

# fig_main.savefig(
#     pdf_path,
#     bbox_inches="tight",
#     facecolor="white"
# )

# print("Figure saved successfully:")
# print(png_path.resolve())
# print(pdf_path.resolve())


In [ ]:

# # ============================================================
# # SAVE TABLE 03 ONLY
# # ============================================================


# TABLE_NUMBER = "03"

# processed_dir = Path(
#     globals().get("DATA_PROCESSED_DIR", "Data_Processed")
# )

# table_dir = processed_dir / "Article_Outputs" / "Tables"
# table_dir.mkdir(parents=True, exist_ok=True)

# table_path = table_dir / (
#     f"Table_{TABLE_NUMBER}_"
#     "Robust_Historical_Topic_Contrasts.csv"
# )

# paper_results_main5.to_csv(
#     table_path,
#     index=False,
#     encoding="utf-8-sig"
# )

# print("Main article table saved:")
# print(table_path.resolve())


In [ ]:

# # ============================================================
# # SAVE SUPPLEMENTARY TABLE ONLY
# # ============================================================


# processed_dir = Path(
#     globals().get("DATA_PROCESSED_DIR", "Data_Processed")
# )

# supp_dir = processed_dir / "Article_Outputs" / "Supplementary"
# supp_dir.mkdir(parents=True, exist_ok=True)

# supp_path = supp_dir / (
#     "Supplementary_Table_S01_"
#     "Complete_HAC_and_Robustness_Results.csv"
# )

# paper_results_all18.to_csv(
#     supp_path,
#     index=False,
#     encoding="utf-8-sig"
# )

# print("Supplementary table saved:")
# print(supp_path.resolve())



## Step 12.18 — Leave-One-Year-Out Influence Diagnostics

### Objective

To investigate whether the primary historical findings are disproportionately influenced by individual calendar years.

### Analytical Strategy

A leave-one-year-out (LOYO) analysis will be conducted using the reference NMF model (K = 7, seed = 11).

For each calendar year from 1972 to 2025:

1. Remove the corresponding observations for all three countries.
2. Re-estimate the 18 prespecified bilateral topic contrasts.
3. Calculate Newey–West HAC standard errors using bandwidth 5.
4. Apply Holm multiplicity adjustment across the 18 contrasts.
5. Compare the resulting estimates and statistical decisions with the full-sample results.

The regression specification includes an intercept, a middle-period indicator (1991–2010), and a late-period indicator (2011–2025), using 1972–1990 as the reference period.

### Primary Diagnostics

- Maximum absolute change in the estimated effect.
- Historical year producing the largest change.
- Range of leave-one-year-out effect estimates.
- Number of exclusions preserving effect direction.
- Number of exclusions retaining Holm-adjusted statistical significance.
- Maximum Holm-adjusted p-value across exclusions.

### Interpretation

The analysis evaluates the influence of individual years, rather than testing alternative geopolitical explanations.

Stable estimates provide evidence that the findings are not driven by a single calendar year. However, this analysis does not establish causality, topic validity, or robustness to the simultaneous removal of multiple years.

All analyses use the original observed data without permanently deleting any speeches.


In [ ]:

# ============================================================
# STEP 12.18A — CORRECTED
# REFERENCE NMF TOPIC SHARES AND SOURCE VALIDATION
# ============================================================


print("=" * 65)
print("REFERENCE NMF SOURCE RECONSTRUCTION")
print("=" * 65)

loyo_annual_df = None

# ------------------------------------------------------------
# 1. Verify required notebook objects
# ------------------------------------------------------------

required_objects = [
    "focal_df",
    "segments_df",
    "selected_result",
    "paper_results_all18",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing notebook objects: {missing_objects}"
)

assert isinstance(selected_result, dict)

print("\nAvailable selected-result keys:")
print(list(selected_result.keys()))

assert "speech_shares" in selected_result
assert "model" in selected_result

model_18 = selected_result["model"]

print("\nReference model information:")
print("Number of topics:",
      getattr(model_18, "n_components", "Unknown"))
print("Random state:",
      getattr(model_18, "random_state", "Unknown"))

assert getattr(model_18, "n_components", None) == 7, (
    "Selected model is not K=7."
)

model_seed = getattr(model_18, "random_state", None)

if isinstance(model_seed, (int, np.integer)):
    assert int(model_seed) == 11, (
        "Selected model is not reference seed 11."
    )

# ------------------------------------------------------------
# 2. Recover speech-level topic shares
# ------------------------------------------------------------

speech_shares_18 = np.asarray(
    selected_result["speech_shares"],
    dtype=float
)

print("\nSpeech-share matrix shape:")
print(speech_shares_18.shape)

assert speech_shares_18.shape == (162, 7), (
    "Expected a 162 x 7 speech-topic matrix."
)

assert np.isfinite(speech_shares_18).all()
assert (speech_shares_18 >= -1e-10).all()

speech_totals = speech_shares_18.sum(axis=1)

print("\nSpeech-level share normalization:")
print("Minimum total:", speech_totals.min())
print("Maximum total:", speech_totals.max())

assert np.allclose(
    speech_totals,
    1.0,
    rtol=0,
    atol=1e-5
), "Speech-level topic shares do not sum to one."

# ------------------------------------------------------------
# 3. Validate speech metadata and positional alignment
# ------------------------------------------------------------

metadata_18 = focal_df.reset_index(drop=True)[
    ["country_code", "year"]
].copy()

metadata_18["country_code"] = (
    metadata_18["country_code"]
    .astype(str)
    .str.upper()
    .str.strip()
)

metadata_18["year"] = pd.to_numeric(
    metadata_18["year"],
    errors="raise"
).astype(int)

assert len(metadata_18) == 162

assert set(metadata_18["country_code"]) == {
    "IRN", "USA", "CHN"
}

assert not metadata_18.duplicated(
    ["country_code", "year"]
).any()

country_year_counts = metadata_18.groupby(
    "country_code"
)["year"].agg(["count", "nunique", "min", "max"])

print("\nCountry-year metadata:")
display(country_year_counts)

assert country_year_counts["count"].eq(54).all()
assert country_year_counts["nunique"].eq(54).all()
assert country_year_counts["min"].eq(1972).all()
assert country_year_counts["max"].eq(2025).all()

# Verify that speech IDs from segmentation correspond to
# the same row positions in focal_df.

segment_meta_18 = (
    segments_df.groupby("speech_id")[
        ["country_code", "year"]
    ]
    .first()
    .sort_index()
)

expected_ids = np.arange(162)

assert np.array_equal(
    segment_meta_18.index.to_numpy(),
    expected_ids
), "Unexpected segment speech-ID indexing."

segment_countries = (
    segment_meta_18["country_code"]
    .astype(str).str.upper().str.strip()
    .to_numpy()
)

segment_years = (
    segment_meta_18["year"]
    .astype(int)
    .to_numpy()
)

assert np.array_equal(
    metadata_18["country_code"].to_numpy(),
    segment_countries
), "Country alignment between segments and speeches failed."

assert np.array_equal(
    metadata_18["year"].to_numpy(),
    segment_years
), "Year alignment between segments and speeches failed."

print("\nSpeech-to-segment metadata alignment: PASSED")

# ------------------------------------------------------------
# 4. Construct the annual long-format topic dataset
# ------------------------------------------------------------

candidate_annual_18 = pd.DataFrame({
    "country_code": np.repeat(
        metadata_18["country_code"].to_numpy(), 7
    ),
    "year": np.repeat(
        metadata_18["year"].to_numpy(), 7
    ),
    "topic_id": np.tile(
        np.arange(1, 8), len(metadata_18)
    ),
    "topic_share": speech_shares_18.reshape(-1)
})

candidate_annual_18 = candidate_annual_18.sort_values(
    ["year", "country_code", "topic_id"]
).reset_index(drop=True)

assert len(candidate_annual_18) == 1134

assert not candidate_annual_18.duplicated(
    ["country_code", "year", "topic_id"]
).any()

print("\nReconstructed annual topic records:",
      len(candidate_annual_18))

print("\nSample annual records:")
display(candidate_annual_18.head(12))

# ------------------------------------------------------------
# 5. Reproduce Step 12.17 historical effects
# ------------------------------------------------------------

pair_codes_18 = {
    "Iran - China": ("IRN", "CHN"),
    "Iran - United States": ("IRN", "USA"),
    "United States - China": ("USA", "CHN"),
}

required_result_cols = {
    "pair_label",
    "topic_id",
    "effect_pp",
}

assert required_result_cols.issubset(
    paper_results_all18.columns
)

assert len(paper_results_all18) == 18

annual_matrix_18 = candidate_annual_18.pivot(
    index=["country_code", "year"],
    columns="topic_id",
    values="topic_share"
)

reproduction_records = []

for _, row in paper_results_all18.iterrows():

    pair_label = row["pair_label"]
    topic_id = int(row["topic_id"])

    assert pair_label in pair_codes_18, (
        f"Unknown country pair: {pair_label}"
    )

    country_a, country_b = pair_codes_18[pair_label]

    country_a_series = annual_matrix_18.loc[
        country_a, topic_id
    ]

    country_b_series = annual_matrix_18.loc[
        country_b, topic_id
    ]

    bilateral_gap = (
        country_a_series - country_b_series
    )

    early_mean = bilateral_gap.loc[
        1972:1990
    ].mean()

    late_mean = bilateral_gap.loc[
        2011:2025
    ].mean()

    reconstructed_effect = (
        late_mean - early_mean
    ) * 100

    original_effect = float(row["effect_pp"])

    reproduction_records.append({
        "pair_label": pair_label,
        "topic_id": topic_id,
        "published_effect_pp": original_effect,
        "reconstructed_effect_pp": reconstructed_effect,
        "absolute_difference_pp": abs(
            reconstructed_effect - original_effect
        )
    })

loyo_reproduction_df = pd.DataFrame(
    reproduction_records
)

print("\n" + "=" * 65)
print("PUBLICATION EFFECT REPRODUCIBILITY")
print("=" * 65)

display(loyo_reproduction_df.round(6))

maximum_difference = (
    loyo_reproduction_df["absolute_difference_pp"].max()
)

print("\nMaximum absolute discrepancy (pp):",
      round(maximum_difference, 8))

# Accept only discrepancies compatible with rounding.
assert maximum_difference <= 0.05, (
    "Reconstructed effects do not reproduce Step 12.17. "
    "Do not proceed to 12.18B."
)

# Commit the validated dataset only after every check.
loyo_annual_df = candidate_annual_18.copy()

print("\n" + "=" * 65)
print("FINAL SOURCE VALIDATION")
print("=" * 65)

print("Original speeches:", len(metadata_18))
print("Annual topic records:", len(loyo_annual_df))
print("Topics per speech: 7")
print("Country-year coverage: 1972-2025")
print("Speech alignment: PASSED")
print("Effect reproduction: PASSED")
print("REFERENCE DATA VALIDATION: PASSED")
print("Files saved: NONE")


In [ ]:

# ============================================================
# STEP 12.18B
# LEAVE-ONE-YEAR-OUT INFLUENCE DIAGNOSTICS
# ============================================================

assert "loyo_annual_df" in globals()
assert loyo_annual_df is not None, (
    "Step 12.18A must pass source validation first."
)

HAC_LAG_18 = 5
ELIGIBLE_TOPICS_18 = (1, 2, 3, 4, 5, 7)

# ------------------------------------------------------------
# Restore bilateral comparisons and historical coverage
# ------------------------------------------------------------

PAIR_CODES_18 = {
    "Iran - China": ("IRN", "CHN"),
    "Iran - United States": ("IRN", "USA"),
    "United States - China": ("USA", "CHN"),
}

YEARS_18 = np.arange(1972, 2026, dtype=int)

# Validate source coverage
assert set(loyo_annual_df["country_code"].unique()) == {
    "IRN", "USA", "CHN"
}

assert set(loyo_annual_df["year"].astype(int).unique()) == set(YEARS_18)

assert len(PAIR_CODES_18) == 3
assert len(YEARS_18) == 54
assert len(ELIGIBLE_TOPICS_18) == 6

print("STEP 12.18B — Configuration restored")
print("Country comparisons:", len(PAIR_CODES_18))
print("Historical years:", len(YEARS_18))
print("Eligible topics:", len(ELIGIBLE_TOPICS_18))
print("Expected contrasts:", 3 * 6)
print("CONFIGURATION VALIDATION: PASSED")


# ============================================================
# STEP 12.18B — REPLACE ONLY THE ESTIMATOR SECTION
# Replace from "def estimate_historical_effect(years, series):"
# through its original closing "}" (before topic_matrix_18).
# Do not change other parts of Step 12.18B.
# ============================================================

from scipy.stats import norm


def calendar_hac_core_18(years_observed, y_observed, lag=5):
    """OLS three-period contrast with HAC on ORIGINAL calendar lags.

    Missing years contribute zero to calendar-aligned score products,
    but are excluded from the fitted model and the finite-sample factor.
    The estimand is (2011–2025 average minus 1972–1990 average)
    for the first-minus-second country annual topic-share gap.
    """
    years_observed = np.asarray(years_observed, dtype=int)
    y_observed = np.asarray(y_observed, dtype=float)
    calendar = np.asarray(YEARS_18, dtype=int)

    assert years_observed.ndim == y_observed.ndim == 1
    assert years_observed.size == y_observed.size
    assert years_observed.size >= 4
    assert np.unique(years_observed).size == years_observed.size
    assert np.isin(years_observed, calendar).all()
    assert np.isfinite(y_observed).all()
    assert isinstance(lag, (int, np.integer)) and 0 <= int(lag) < len(calendar)

    observed = pd.Series(y_observed, index=years_observed).reindex(calendar)
    keep = observed.notna().to_numpy()
    y = observed.to_numpy(dtype=float)[keep]

    early = ((calendar >= 1972) & (calendar <= 1990)).astype(float)
    middle = ((calendar >= 1991) & (calendar <= 2010)).astype(float)
    late = ((calendar >= 2011) & (calendar <= 2025)).astype(float)

    x_full = np.column_stack((early, middle, late))
    x = x_full[keep]
    n, p = x.shape
    assert p == 3 and n > p and np.linalg.matrix_rank(x) == p

    xtx_inv = np.linalg.inv(x.T @ x)
    beta = xtx_inv @ (x.T @ y)
    residuals = y - x @ beta

    # The score array keeps all 54 original calendar positions.
    scores = np.zeros((len(calendar), p), dtype=float)
    scores[keep, :] = x * residuals[:, None]

    meat = scores.T @ scores
    for h in range(1, int(lag) + 1):
        w = 1.0 - h / (int(lag) + 1.0)
        forward = scores[h:].T @ scores[:-h]
        meat += w * (forward + forward.T)

    covariance = (n / (n - p)) * xtx_inv @ meat @ xtx_inv
    contrast = np.array([-1.0, 0.0, 1.0])
    effect = float(contrast @ beta)
    variance = float(contrast @ covariance @ contrast)
    assert np.isfinite(variance) and variance > 0
    se = float(np.sqrt(variance))
    z = effect / se
    p_value = float(2.0 * norm.sf(abs(z)))
    critical = float(norm.ppf(0.975))

    return {
        "effect_pp": effect * 100.0,
        "hac_se_pp": se * 100.0,
        "ci_lower_pp": (effect - critical * se) * 100.0,
        "ci_upper_pp": (effect + critical * se) * 100.0,
        "p_value": p_value,
        "n_years": int(n),
        "n_early": int(early[keep].sum()),
        "n_middle": int(middle[keep].sum()),
        "n_late": int(late[keep].sum()),
    }


def estimate_historical_effect(years, series):
    # Maintains the previous Step 12.18B return-value contract.
    result = calendar_hac_core_18(years, series, lag=HAC_LAG_18)
    return {
        name: result[name]
        for name in ("effect_pp", "ci_lower_pp", "ci_upper_pp", "p_value")
    }


topic_matrix_18 = loyo_annual_df.pivot(
    index=["country_code", "year"],
    columns="topic_id",
    values="topic_share",
)

# Build annual bilateral topic-gap series.
gap_series_18 = {}

for label, (first, second) in PAIR_CODES_18.items():
    for topic_id in ELIGIBLE_TOPICS_18:
        first_values = topic_matrix_18.loc[
            first, topic_id
        ].reindex(YEARS_18)

        second_values = topic_matrix_18.loc[
            second, topic_id
        ].reindex(YEARS_18)

        gap = first_values - second_values

        assert gap.notna().all()

        gap_series_18[(label, topic_id)] = gap


def calculate_all_contrasts(excluded_year=None):
    records = []

    for (pair_label, topic_id), gap in gap_series_18.items():
        if excluded_year is None:
            selected = gap
        else:
            selected = gap.drop(index=excluded_year)

        fit = estimate_historical_effect(
            selected.index.to_numpy(),
            selected.to_numpy(),
        )

        records.append({
            "excluded_year": excluded_year,
            "pair_label": pair_label,
            "topic_id": topic_id,
            "n_years": len(selected),
            **fit,
        })

    result = pd.DataFrame(records)

    _, adjusted_p, _, _ = multipletests(
        result["p_value"].to_numpy(),
        alpha=0.05,
        method="holm",
    )

    result["holm_adjusted_p"] = adjusted_p
    result["holm_significant"] = adjusted_p < 0.05

    return result


# ------------------------------------------------------------
# 1. Full-sample reproducibility check
# ------------------------------------------------------------

loyo_baseline_df = calculate_all_contrasts()

comparison = loyo_baseline_df.merge(
    paper_results_all18[
        [
            "pair_label",
            "topic_id",
            "effect_pp",
            "holm_adjusted_p",
        ]
    ],
    on=["pair_label", "topic_id"],
    suffixes=("_recomputed", "_published"),
    validate="one_to_one",
)

assert len(comparison) == 18

max_effect_error = (
    comparison["effect_pp_recomputed"]
    - comparison["effect_pp_published"]
).abs().max()

max_p_error = (
    comparison["holm_adjusted_p_recomputed"]
    - comparison["holm_adjusted_p_published"]
).abs().max()

print("=" * 65)
print("FULL-SAMPLE HAC REPRODUCIBILITY")
print("=" * 65)

print("Maximum effect discrepancy (pp):",
      round(max_effect_error, 8))

print("Maximum Holm p-value discrepancy:",
      round(max_p_error, 8))

assert max_effect_error < 0.05, (
    "The baseline effect estimates were not reproduced."
)

assert max_p_error < 0.001, (
    "The baseline HAC inference was not reproduced."
)

print("BASELINE REPRODUCIBILITY: PASSED")


# ------------------------------------------------------------
# 2. Leave out each calendar year
# ------------------------------------------------------------

loyo_results_df = pd.concat(
    [
        calculate_all_contrasts(excluded_year=int(year))
        for year in YEARS_18
    ],
    ignore_index=True,
)

assert len(loyo_results_df) == 54 * 18

assert loyo_results_df.groupby(
    ["pair_label", "topic_id"]
)["excluded_year"].nunique().eq(54).all()

assert loyo_results_df["n_years"].eq(53).all()

print("\n" + "=" * 65)
print("LEAVE-ONE-YEAR-OUT ANALYSIS")
print("=" * 65)

print("Excluded calendar years:", 54)
print("Contrasts per exclusion:", 18)
print("Total refitted contrasts:", len(loyo_results_df))


# ------------------------------------------------------------
# 3. Summarize the five primary robust results
# ------------------------------------------------------------

primary_keys = paper_results_main5[
    ["pair_label", "topic_id"]
].drop_duplicates()

selected_loyo = loyo_results_df.merge(
    primary_keys,
    on=["pair_label", "topic_id"],
    how="inner",
    validate="many_to_one",
)

selected_baseline = loyo_baseline_df.merge(
    primary_keys,
    on=["pair_label", "topic_id"],
    how="inner",
    validate="one_to_one",
)

assert len(selected_loyo) == 54 * 5
assert len(selected_baseline) == 5

loyo_summary_records = []

for baseline in selected_baseline.itertuples(index=False):
    group = selected_loyo.loc[
        (selected_loyo["pair_label"] == baseline.pair_label)
        & (selected_loyo["topic_id"] == baseline.topic_id)
    ].copy()

    group["effect_shift_pp"] = (
        group["effect_pp"] - baseline.effect_pp
    )

    group["absolute_shift_pp"] = (
        group["effect_shift_pp"].abs()
    )

    group["same_direction"] = (
        np.sign(group["effect_pp"])
        == np.sign(baseline.effect_pp)
    )

    influential = group.loc[
        group["absolute_shift_pp"].idxmax()
    ]

    loyo_summary_records.append({
        "pair_label": baseline.pair_label,
        "topic_id": baseline.topic_id,
        "baseline_effect_pp": baseline.effect_pp,
        "minimum_effect_pp": group["effect_pp"].min(),
        "maximum_effect_pp": group["effect_pp"].max(),
        "maximum_absolute_shift_pp":
            group["absolute_shift_pp"].max(),
        "most_influential_year":
            int(influential["excluded_year"]),
        "same_direction_years":
            int(group["same_direction"].sum()),
        "holm_significant_years":
            int(group["holm_significant"].sum()),
        "maximum_holm_adjusted_p":
            group["holm_adjusted_p"].max(),
        "loyo_stable":
            bool(
                group["same_direction"].all()
                and group["holm_significant"].all()
            ),
    })

loyo_summary_df = pd.DataFrame(
    loyo_summary_records
).sort_values(
    ["pair_label", "topic_id"]
).reset_index(drop=True)

print("\n" + "=" * 65)
print("PRIMARY FINDINGS: INFLUENCE SUMMARY")
print("=" * 65)

display(loyo_summary_df.round(5))


# ------------------------------------------------------------
# 4. Inspect the largest changes
# ------------------------------------------------------------

selected_loyo = selected_loyo.merge(
    selected_baseline[
        ["pair_label", "topic_id", "effect_pp"]
    ].rename(
        columns={"effect_pp": "baseline_effect_pp"}
    ),
    on=["pair_label", "topic_id"],
    validate="many_to_one",
)

selected_loyo["effect_shift_pp"] = (
    selected_loyo["effect_pp"]
    - selected_loyo["baseline_effect_pp"]
)

selected_loyo["absolute_shift_pp"] = (
    selected_loyo["effect_shift_pp"].abs()
)

largest_influences_df = selected_loyo.sort_values(
    "absolute_shift_pp",
    ascending=False,
).head(15)

print("\nLargest individual-year influences:")

display(
    largest_influences_df[
        [
            "excluded_year",
            "pair_label",
            "topic_id",
            "baseline_effect_pp",
            "effect_pp",
            "effect_shift_pp",
            "holm_adjusted_p",
            "holm_significant",
        ]
    ].round(5)
)


# ------------------------------------------------------------
# 5. Final diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("LEAVE-ONE-YEAR-OUT VALIDATION")
print("=" * 65)

print("Primary findings evaluated:", len(loyo_summary_df))

print(
    "Findings retaining Holm significance in all exclusions:",
    int(loyo_summary_df["holm_significant_years"].eq(54).sum())
)

print(
    "Findings retaining direction in all exclusions:",
    int(loyo_summary_df["same_direction_years"].eq(54).sum())
)

print(
    "Findings meeting both stability criteria:",
    int(loyo_summary_df["loyo_stable"].sum())
)

print("\nLOYO COMPUTATION: PASSED")
print("Scientific interpretation: PENDING REVIEW")
print("Files saved: NONE")




### Step 12.19 — Five-Year Block Deletion Sensitivity Analysis

**Objective:** Evaluate whether the principal historical topic-gap findings depend disproportionately on any contiguous five-year historical period.

**Analytical design:**
- Study period: 1972–2025 (54 matched years).
- Reference model: Speech-balanced NMF, K = 7, seed = 11.
- Historical comparison: 1972–1990 versus 2011–2025.
- Sensitivity procedure: Sequential deletion of non-overlapping five-year calendar blocks, with a final four-year block.
- Scope: All 18 bilateral topic contrasts, with detailed assessment of the five primary robust findings.

**Primary diagnostics:**
1. Directional consistency across deleted blocks.
2. Maximum absolute change in estimated effect size.
3. Minimum and maximum effect estimates.
4. Identification of the most influential historical block.

This step evaluates the stability of point estimates. Statistical significance is not reassessed in this step; HAC confidence intervals and Holm-adjusted significance were previously examined using leave-one-year-out analysis.

No figures or publication files are automatically saved.


In [ ]:

# ============================================================
# STEP 12.19
# FIVE-YEAR BLOCK DELETION SENSITIVITY
# ============================================================

print("=" * 65)
print("FIVE-YEAR BLOCK DELETION SENSITIVITY")
print("=" * 65)

# ------------------------------------------------------------
# 1. Validate required objects
# ------------------------------------------------------------

assert "loyo_annual_df" in globals()
assert isinstance(loyo_annual_df, pd.DataFrame)
assert "paper_results_all18" in globals()

annual = loyo_annual_df.copy()
reference = paper_results_all18.copy()

required_annual = {
    "country_code", "year", "topic_id", "topic_share"
}
required_reference = {
    "pair_label", "topic_id", "effect_pp"
}

assert required_annual.issubset(annual.columns)
assert required_reference.issubset(reference.columns)

assert len(annual) == 1134
assert len(reference) == 18
assert not annual.duplicated(
    ["country_code", "year", "topic_id"]
).any()

assert annual["topic_share"].notna().all()
assert np.isfinite(annual["topic_share"]).all()

expected_years = set(range(1972, 2026))

for country in ["IRN", "USA", "CHN"]:
    subset = annual[annual["country_code"] == country]
    assert set(subset["year"]) == expected_years
    assert len(subset) == 54 * 7

pair_codes = {
    "Iran - China": ("IRN", "CHN"),
    "Iran - United States": ("IRN", "USA"),
    "United States - China": ("USA", "CHN"),
}

assert set(reference["pair_label"]).issubset(pair_codes)

matrix = annual.pivot(
    index=["country_code", "year"],
    columns="topic_id",
    values="topic_share"
).sort_index()

# ------------------------------------------------------------
# 2. Define historical blocks
# ------------------------------------------------------------

historical_blocks = []

for start in range(1972, 2026, 5):
    end = min(start + 4, 2025)

    historical_blocks.append({
        "block_start": start,
        "block_end": end,
        "block_label": f"{start}-{end}",
        "years": list(range(start, end + 1))
    })

assert len(historical_blocks) == 11

print("\nHistorical deletion blocks:")

for block in historical_blocks:
    print(
        block["block_label"],
        "| Years:",
        len(block["years"])
    )

# ------------------------------------------------------------
# 3. Reconstruct historical effects
# ------------------------------------------------------------

def historical_gap_effect(gap_series):
    early = gap_series[
        (gap_series.index >= 1972) &
        (gap_series.index <= 1990)
    ]

    late = gap_series[
        (gap_series.index >= 2011) &
        (gap_series.index <= 2025)
    ]

    assert len(early) >= 10
    assert len(late) >= 10

    effect = 100 * (late.mean() - early.mean())

    return float(effect), len(early), len(late)


block_records = []
baseline_checks = []

for _, ref in reference.iterrows():

    pair_label = ref["pair_label"]
    topic_id = int(ref["topic_id"])

    country_a, country_b = pair_codes[pair_label]

    series_a = matrix.xs(
        country_a, level="country_code"
    )[topic_id]

    series_b = matrix.xs(
        country_b, level="country_code"
    )[topic_id]

    gap = (series_a - series_b).sort_index()

    baseline_effect, _, _ = historical_gap_effect(gap)
    expected_effect = float(ref["effect_pp"])

    baseline_checks.append(
        abs(baseline_effect - expected_effect)
    )

    for block in historical_blocks:

        retained = gap.loc[
            ~gap.index.isin(block["years"])
        ]

        effect, n_early, n_late = (
            historical_gap_effect(retained)
        )

        block_records.append({
            "pair_label": pair_label,
            "topic_id": topic_id,
            "deleted_block": block["block_label"],
            "baseline_effect_pp": expected_effect,
            "block_deleted_effect_pp": effect,
            "effect_shift_pp": effect - expected_effect,
            "absolute_shift_pp": abs(
                effect - expected_effect
            ),
            "same_direction": bool(
                np.sign(effect) ==
                np.sign(expected_effect)
            ),
            "n_early": n_early,
            "n_late": n_late
        })

assert max(baseline_checks) < 1e-7, (
    "Baseline effects differ from Step 12.17."
)

block_deletion_all18 = pd.DataFrame(block_records)

assert len(block_deletion_all18) == 198
assert block_deletion_all18[
    "block_deleted_effect_pp"
].notna().all()

print("\nBaseline reproducibility: PASSED")
print("Historical blocks:", len(historical_blocks))
print("Bilateral topic contrasts:", len(reference))
print("Total sensitivity estimates:",
      len(block_deletion_all18))

# ------------------------------------------------------------
# 4. Summarize influence across all 18 contrasts
# ------------------------------------------------------------

summary_records = []

for (pair_label, topic_id), group in (
    block_deletion_all18.groupby(
        ["pair_label", "topic_id"],
        sort=False
    )
):

    influential = group.loc[
        group["absolute_shift_pp"].idxmax()
    ]

    summary_records.append({
        "pair_label": pair_label,
        "topic_id": topic_id,
        "baseline_effect_pp":
            group["baseline_effect_pp"].iloc[0],
        "minimum_effect_pp":
            group["block_deleted_effect_pp"].min(),
        "maximum_effect_pp":
            group["block_deleted_effect_pp"].max(),
        "maximum_absolute_shift_pp":
            group["absolute_shift_pp"].max(),
        "most_influential_block":
            influential["deleted_block"],
        "same_direction_blocks":
            int(group["same_direction"].sum()),
        "n_blocks": len(group),
        "block_direction_stable":
            bool(group["same_direction"].all())
    })

block_deletion_summary = pd.DataFrame(summary_records)

assert len(block_deletion_summary) == 18
assert block_deletion_summary["n_blocks"].eq(11).all()

# ------------------------------------------------------------
# 5. Identify five previously robust findings
# ------------------------------------------------------------

if "evidence_category" in reference.columns:

    primary = reference.loc[
        reference["evidence_category"]
        .astype(str)
        .str.contains(
            "Robust across all 18 specifications",
            case=False,
            regex=False
        )
    ].copy()

else:

    required_robust = {
        "holm_significant_of_18",
        "n_matching_directions"
    }

    assert required_robust.issubset(
        reference.columns
    ), "Cannot identify the five robust findings."

    primary = reference.loc[
        (reference["holm_significant_of_18"] == 18) &
        (reference["n_matching_directions"] == 18)
    ].copy()

assert len(primary) == 5, (
    "Expected exactly five previously robust findings."
)

primary_columns = ["pair_label", "topic_id"]

if "topic_label" in primary.columns:
    primary_columns.append("topic_label")

block_deletion_primary5 = primary[
    primary_columns
].merge(
    block_deletion_summary,
    on=["pair_label", "topic_id"],
    how="inner",
    validate="one_to_one"
)

assert len(block_deletion_primary5) == 5

# ------------------------------------------------------------
# 6. Display primary results
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PRIMARY FINDINGS: BLOCK DELETION SUMMARY")
print("=" * 65)

display(
    block_deletion_primary5.round(4)
)

print("\n" + "=" * 65)
print("MOST INFLUENTIAL HISTORICAL BLOCKS")
print("=" * 65)

primary_keys = primary[
    ["pair_label", "topic_id"]
]

primary_detail = block_deletion_all18.merge(
    primary_keys,
    on=["pair_label", "topic_id"],
    how="inner",
    validate="many_to_one"
)

display(
    primary_detail.sort_values(
        "absolute_shift_pp",
        ascending=False
    ).head(15).round(4)
)

# ------------------------------------------------------------
# 7. Final checks
# ------------------------------------------------------------

n_stable = int(
    block_deletion_primary5[
        "block_direction_stable"
    ].sum()
)

max_shift = (
    block_deletion_primary5[
        "maximum_absolute_shift_pp"
    ].max()
)

print("\n" + "=" * 65)
print("BLOCK DELETION VALIDATION")
print("=" * 65)

print("Primary findings:", len(primary))
print("Deletion blocks:", len(historical_blocks))
print("Stable directional findings:",
      f"{n_stable}/5")
print("Largest absolute effect shift (pp):",
      round(float(max_shift), 4))

if n_stable == 5:
    print("PRIMARY DIRECTIONAL STABILITY: PASSED")
else:
    print("PRIMARY DIRECTIONAL STABILITY: REVIEW REQUIRED")

print("Point-estimate sensitivity: COMPLETED")
print("HAC significance under block deletion: NOT TESTED")
print("Files saved: NONE")



### Step 12.20A — Reference HAC Specification Diagnosis

**Objective:** Identify why the reconstructed historical topic effects agree exactly with the reference estimates while their confidence intervals and Holm-adjusted p-values differ.

**Diagnostic specifications:**
1. Two-period HAC regression using only early and late observations.
2. Three-period HAC regression retaining the historical middle period.
3. Separate-period HAC variance estimation for the difference in means.

Finite-sample covariance correction and normal-versus-t inference are evaluated where applicable.

**Acceptance criterion:** Agreement with the original 18 reference contrasts in point estimates, 95% confidence intervals, and Holm-adjusted p-values.

This is a diagnostic analysis only. It does not modify the primary findings, select a new inferential model, or execute block deletion. Any apparent match must be verified against the original HAC implementation before further inference.


In [ ]:

# ============================================================
# STEP 12.20A
# REFERENCE HAC COVARIANCE SPECIFICATION DIAGNOSIS
# ============================================================

print("=" * 65)
print("REFERENCE HAC SPECIFICATION DIAGNOSIS")
print("=" * 65)

# ------------------------------------------------------------
# 1. Validate sources
# ------------------------------------------------------------

assert "loyo_annual_df" in globals()
assert "paper_results_all18" in globals()

audit_annual = loyo_annual_df.copy()
audit_ref = paper_results_all18.copy()

required_annual = {
    "country_code", "year", "topic_id", "topic_share"
}

required_ref = {
    "pair_label", "topic_id", "effect_pp",
    "ci_lower_pp", "ci_upper_pp", "holm_adjusted_p"
}

assert required_annual.issubset(audit_annual.columns)
assert required_ref.issubset(audit_ref.columns)

assert len(audit_annual) == 1134
assert len(audit_ref) == 18

audit_wide = audit_annual.pivot(
    index=["country_code", "year"],
    columns="topic_id",
    values="topic_share"
).sort_index()

audit_pairs = {
    "Iran - China": ("IRN", "CHN"),
    "Iran - United States": ("IRN", "USA"),
    "United States - China": ("USA", "CHN")
}

audit_gaps = {}

for pair_label, (a, b) in audit_pairs.items():

    series_a = audit_wide.xs(
        a, level="country_code"
    )

    series_b = audit_wide.xs(
        b, level="country_code"
    )

    assert series_a.index.equals(series_b.index)

    for topic_id in sorted(audit_ref["topic_id"].unique()):

        audit_gaps[(pair_label, int(topic_id))] = (
            series_a[topic_id] - series_b[topic_id]
        ).sort_index()

print("\nSource validation: PASSED")
print("Reference contrasts:", len(audit_ref))

# ------------------------------------------------------------
# 2. Alternative HAC specifications
# ------------------------------------------------------------

def diagnostic_hac(
    gap,
    specification,
    correction=False,
    use_t=False,
    lag=5
):

    years = gap.index.to_numpy(dtype=int)
    y_full = gap.to_numpy(dtype=float)

    early = (years >= 1972) & (years <= 1990)
    middle = (years >= 1991) & (years <= 2010)
    late = (years >= 2011) & (years <= 2025)

    if specification == "two_period":

        keep = early | late

        y = y_full[keep]

        x = sm.add_constant(
            late[keep].astype(float),
            has_constant="add"
        )

        contrast = np.array([0.0, 1.0])

    elif specification == "three_period":

        keep = early | middle | late

        y = y_full[keep]

        x = np.column_stack([
            early[keep].astype(float),
            middle[keep].astype(float),
            late[keep].astype(float)
        ])

        contrast = np.array([-1.0, 0.0, 1.0])

    elif specification == "separate_periods":

        period_means = []
        period_variances = []

        for mask in [early, late]:

            yy = y_full[mask]

            xx = np.ones((len(yy), 1))

            fitted = sm.OLS(yy, xx).fit(
                cov_type="HAC",
                cov_kwds={
                    "maxlags": lag,
                    "use_correction": correction
                },
                use_t=False
            )

            period_means.append(
                float(fitted.params[0])
            )

            period_variances.append(
                float(fitted.cov_params()[0, 0])
            )

        effect = (
            period_means[1] - period_means[0]
        )

        se = np.sqrt(sum(period_variances))

        z = effect / se

        p = float(2 * norm.sf(abs(z)))

        lower = effect - 1.95996398454 * se
        upper = effect + 1.95996398454 * se

        return {
            "effect_pp": effect * 100,
            "ci_lower_pp": lower * 100,
            "ci_upper_pp": upper * 100,
            "p_value": p
        }

    else:
        raise ValueError("Unknown specification")

    fitted = sm.OLS(y, x).fit(
        cov_type="HAC",
        cov_kwds={
            "maxlags": lag,
            "use_correction": correction
        },
        use_t=use_t
    )

    test = fitted.t_test(contrast)

    effect = float(
        np.asarray(test.effect).item()
    )

    ci = np.asarray(test.conf_int(alpha=0.05))

    return {
        "effect_pp": effect * 100,
        "ci_lower_pp": float(ci[0, 0]) * 100,
        "ci_upper_pp": float(ci[0, 1]) * 100,
        "p_value": float(
            np.asarray(test.pvalue).item()
        )
    }

# ------------------------------------------------------------
# 3. Evaluate candidate methods
# ------------------------------------------------------------

candidate_specs = []

for specification in ["two_period", "three_period"]:
    for correction in [False, True]:
        for use_t in [False, True]:
            candidate_specs.append(
                (specification, correction, use_t)
            )

for correction in [False, True]:
    candidate_specs.append(
        ("separate_periods", correction, False)
    )

audit_summaries = []
audit_candidate_tables = {}

for specification, correction, use_t in candidate_specs:

    rows = []

    for _, ref_row in audit_ref.iterrows():

        pair_label = ref_row["pair_label"]
        topic_id = int(ref_row["topic_id"])

        fitted = diagnostic_hac(
            audit_gaps[(pair_label, topic_id)],
            specification=specification,
            correction=correction,
            use_t=use_t,
            lag=5
        )

        rows.append({
            "pair_label": pair_label,
            "topic_id": topic_id,
            **fitted
        })

    candidate = pd.DataFrame(rows)

    candidate["holm_adjusted_p"] = multipletests(
        candidate["p_value"].to_numpy(),
        method="holm"
    )[1]

    compared = audit_ref.merge(
        candidate,
        on=["pair_label", "topic_id"],
        suffixes=("_reference", "_candidate"),
        validate="one_to_one"
    )

    effect_error = np.max(np.abs(
        compared["effect_pp_reference"] -
        compared["effect_pp_candidate"]
    ))

    ci_error = max(
        np.max(np.abs(
            compared["ci_lower_pp_reference"] -
            compared["ci_lower_pp_candidate"]
        )),
        np.max(np.abs(
            compared["ci_upper_pp_reference"] -
            compared["ci_upper_pp_candidate"]
        ))
    )

    p_error = np.max(np.abs(
        compared["holm_adjusted_p_reference"] -
        compared["holm_adjusted_p_candidate"]
    ))

    candidate_name = (
        f"{specification}"
        f"_correction_{correction}"
        f"_t_{use_t}"
    )

    audit_candidate_tables[candidate_name] = compared

    audit_summaries.append({
        "candidate": candidate_name,
        "specification": specification,
        "correction": correction,
        "use_t": use_t,
        "max_effect_error_pp": effect_error,
        "max_ci_error_pp": ci_error,
        "max_adjusted_p_error": p_error
    })

# ------------------------------------------------------------
# 4. Show overall comparison
# ------------------------------------------------------------

hac_diagnostic_summary = pd.DataFrame(
    audit_summaries
)

hac_diagnostic_summary = (
    hac_diagnostic_summary.sort_values(
        [
            "max_ci_error_pp",
            "max_adjusted_p_error"
        ]
    ).reset_index(drop=True)
)

print("\n" + "=" * 65)
print("CANDIDATE HAC SPECIFICATION COMPARISON")
print("=" * 65)

display(hac_diagnostic_summary.round(8))

# ------------------------------------------------------------
# 5. Inspect the closest candidate
# ------------------------------------------------------------

best_candidate_name = (
    hac_diagnostic_summary.iloc[0]["candidate"]
)

best_comparison = audit_candidate_tables[
    best_candidate_name
].copy()

best_comparison["ci_lower_error_pp"] = (
    best_comparison["ci_lower_pp_candidate"] -
    best_comparison["ci_lower_pp_reference"]
)

best_comparison["ci_upper_error_pp"] = (
    best_comparison["ci_upper_pp_candidate"] -
    best_comparison["ci_upper_pp_reference"]
)

best_comparison["adjusted_p_error"] = (
    best_comparison["holm_adjusted_p_candidate"] -
    best_comparison["holm_adjusted_p_reference"]
)

print("\n" + "=" * 65)
print("CLOSEST SPECIFICATION: CONTRAST-LEVEL AUDIT")
print("=" * 65)

print("Candidate:", best_candidate_name)

display(
    best_comparison[
        [
            "pair_label",
            "topic_id",
            "ci_lower_error_pp",
            "ci_upper_error_pp",
            "adjusted_p_error"
        ]
    ].round(7)
)

# ------------------------------------------------------------
# 6. Final diagnostic decision
# ------------------------------------------------------------

best_row = hac_diagnostic_summary.iloc[0]

close_match = bool(
    best_row["max_effect_error_pp"] < 1e-6
    and best_row["max_ci_error_pp"] < 0.02
    and best_row["max_adjusted_p_error"] < 0.002
)

print("\n" + "=" * 65)
print("FINAL HAC DIAGNOSTIC")
print("=" * 65)

print("Closest candidate:", best_candidate_name)
print(
    "Maximum CI discrepancy (pp):",
    round(float(best_row["max_ci_error_pp"]), 8)
)
print(
    "Maximum adjusted p discrepancy:",
    round(float(best_row["max_adjusted_p_error"]), 8)
)

if close_match:
    print("NUMERICAL REPRODUCTION: PASSED")
    print("Original implementation verification: REQUIRED")
else:
    print("NUMERICAL REPRODUCTION: NOT YET ACHIEVED")
    print("Original HAC implementation: REQUIRED")

print("Block deletion executed: NO")
print("Existing publication results modified: NO")
print("Files saved: NONE")



### Step 12.20 — Calendar-Aware Moving Five-Year Block Deletion

**Objective:** Evaluate whether the primary historical topic-gap findings are sensitive to the exclusion of any five consecutive calendar years.

**Reference model:**
- Three historical periods: 1972–1990, 1991–2010, and 2011–2025.
- Early-to-late contrast estimated with Newey–West HAC covariance.
- Primary HAC bandwidth: 5.
- Finite-sample correction enabled; normal-based inference.
- Holm adjustment across all 18 contrasts for each deletion.

**Sensitivity design:**
- 50 overlapping five-year deletion windows.
- All 18 contrasts re-estimated after each deletion.
- Calendar-year distances preserved in HAC covariance estimation.
- Five primary robust findings evaluated for direction and adjusted significance.

**Important:** The NMF topics and document-level topic shares remain fixed. This evaluates the sensitivity of historical statistical inference to excluded years, not the sensitivity of NMF topic estimation to removing documents.

No publication files are generated until the diagnostic outputs have been reviewed.


In [ ]:

# ============================================================
# STEP 12.20
# MOVING FIVE-YEAR BLOCK DELETION: HAC ROBUSTNESS
# ============================================================

print("=" * 68)
print("MOVING FIVE-YEAR BLOCK DELETION: HAC ROBUSTNESS")
print("=" * 68)

# ------------------------------------------------------------
# 1. Source validation
# ------------------------------------------------------------

assert "loyo_annual_df" in globals()
assert "paper_results_all18" in globals()

annual_1220 = loyo_annual_df.copy()
reference_1220 = paper_results_all18.copy()

YEARS_1220 = np.arange(1972, 2026)
HAC_LAG_1220 = 5

PAIR_MAP_1220 = {
    "Iran - China": ("IRN", "CHN"),
    "Iran - United States": ("IRN", "USA"),
    "United States - China": ("USA", "CHN")
}

assert len(annual_1220) == 1134
assert len(reference_1220) == 18

assert not annual_1220.duplicated(
    ["country_code", "year", "topic_id"]
).any()

assert set(annual_1220["country_code"]) == {
    "IRN", "USA", "CHN"
}

assert np.isfinite(
    annual_1220["topic_share"].to_numpy(dtype=float)
).all()

topic_matrix_1220 = annual_1220.pivot(
    index=["year", "topic_id"],
    columns="country_code",
    values="topic_share"
).sort_index()

assert not topic_matrix_1220.isna().any().any()

all_topics_1220 = sorted(
    annual_1220["topic_id"].unique()
)

assert len(all_topics_1220) == 7

expected_index_1220 = pd.MultiIndex.from_product(
    [YEARS_1220, all_topics_1220],
    names=["year", "topic_id"]
)

assert topic_matrix_1220.index.equals(
    expected_index_1220
)

assert len(reference_1220) == 18

# ------------------------------------------------------------
# 2. Calendar-aware three-period HAC estimator
# ------------------------------------------------------------

# ============================================================
# STEP 12.20 — REPLACE ONLY THE calendar_hac_1220 FUNCTION
# Replace its entire definition (from def calendar_hac_1220(...)
# down to the closing return-dictionary) with this wrapper.
# Keep all subsequent sections of Step 12.20 unchanged.
# ============================================================

def calendar_hac_1220(y_full, excluded_years=None, lag=5):
    """Delegates to the sole calendar-aware HAC implementation."""
    assert "calendar_hac_core_18" in globals(), (
        "Run the corrected Step 12.18B before Step 12.20."
    )
    assert np.array_equal(YEARS_1220, YEARS_18)
    y_full = np.asarray(y_full, dtype=float)
    assert y_full.shape == YEARS_1220.shape
    assert np.isfinite(y_full).all()

    if excluded_years is None:
        excluded_years = []
    excluded_years = np.asarray(list(excluded_years), dtype=int)
    assert np.isin(excluded_years, YEARS_1220).all()
    keep = ~np.isin(YEARS_1220, excluded_years)
    return calendar_hac_core_18(
        YEARS_1220[keep], y_full[keep], lag=lag
    )


# ------------------------------------------------------------
# 3. Prepare all reference contrasts
# ------------------------------------------------------------

reference_keys_1220 = list(
    reference_1220[
        ["pair_label", "topic_id"]
    ].itertuples(index=False, name=None)
)

assert len(set(reference_keys_1220)) == 18

gap_series_1220 = {}

for pair_label, topic_id in reference_keys_1220:

    country_a, country_b = PAIR_MAP_1220[
        pair_label
    ]

    topic_slice = topic_matrix_1220.xs(
        int(topic_id),
        level="topic_id"
    ).reindex(YEARS_1220)

    gap = (
        topic_slice[country_a] -
        topic_slice[country_b]
    )

    assert gap.notna().all()

    gap_series_1220[
        (pair_label, int(topic_id))
    ] = gap.to_numpy(dtype=float)


# ------------------------------------------------------------
# 4. Reproduce original HAC inference
# ------------------------------------------------------------

baseline_rows_1220 = []

for pair_label, topic_id in reference_keys_1220:

    result = calendar_hac_1220(
        gap_series_1220[
            (pair_label, int(topic_id))
        ],
        excluded_years=None,
        lag=HAC_LAG_1220
    )

    baseline_rows_1220.append({
        "pair_label": pair_label,
        "topic_id": int(topic_id),
        **result
    })

baseline_1220 = pd.DataFrame(
    baseline_rows_1220
)

baseline_1220["holm_adjusted_p"] = (
    multipletests(
        baseline_1220["p_value"],
        method="holm"
    )[1]
)

baseline_audit_1220 = reference_1220.merge(
    baseline_1220,
    on=["pair_label", "topic_id"],
    suffixes=("_reference", "_new"),
    validate="one_to_one"
)

effect_error_1220 = np.max(np.abs(
    baseline_audit_1220["effect_pp_reference"] -
    baseline_audit_1220["effect_pp_new"]
))

ci_error_1220 = max(
    np.max(np.abs(
        baseline_audit_1220["ci_lower_pp_reference"] -
        baseline_audit_1220["ci_lower_pp_new"]
    )),
    np.max(np.abs(
        baseline_audit_1220["ci_upper_pp_reference"] -
        baseline_audit_1220["ci_upper_pp_new"]
    ))
)

p_error_1220 = np.max(np.abs(
    baseline_audit_1220[
        "holm_adjusted_p_reference"
    ] -
    baseline_audit_1220[
        "holm_adjusted_p_new"
    ]
))

print("\nREFERENCE HAC REPRODUCTION")
print("-" * 68)

print("Effect discrepancy:", effect_error_1220)
print("CI discrepancy:", ci_error_1220)
print("Holm p discrepancy:", p_error_1220)

assert effect_error_1220 < 1e-6
assert ci_error_1220 < 1e-6
assert p_error_1220 < 1e-7

print("REFERENCE HAC REPRODUCTION: PASSED")


# ------------------------------------------------------------
# 5. Generate moving five-year windows
# ------------------------------------------------------------

WINDOW_SIZE_1220 = 5

block_starts_1220 = np.arange(
    1972,
    2025 - WINDOW_SIZE_1220 + 2
)

assert len(block_starts_1220) == 50

block_results_1220 = []

for start_year in block_starts_1220:

    end_year = start_year + WINDOW_SIZE_1220 - 1

    deleted_years = np.arange(
        start_year,
        end_year + 1
    )

    block_label = (
        f"{start_year}-{end_year}"
    )

    temporary_rows = []

    for pair_label, topic_id in reference_keys_1220:

        result = calendar_hac_1220(
            gap_series_1220[
                (pair_label, int(topic_id))
            ],
            excluded_years=deleted_years,
            lag=HAC_LAG_1220
        )

        temporary_rows.append({
            "deleted_block": block_label,
            "block_start": int(start_year),
            "block_end": int(end_year),
            "pair_label": pair_label,
            "topic_id": int(topic_id),
            **result
        })

    temporary_df = pd.DataFrame(
        temporary_rows
    )

    # Holm adjustment across all 18 contrasts
    temporary_df["holm_adjusted_p"] = (
        multipletests(
            temporary_df["p_value"],
            method="holm"
        )[1]
    )

    temporary_df["holm_significant"] = (
        temporary_df["holm_adjusted_p"] < 0.05
    )

    block_results_1220.append(
        temporary_df
    )

moving_block_hac_df = pd.concat(
    block_results_1220,
    ignore_index=True
)

assert len(moving_block_hac_df) == 900

assert (
    moving_block_hac_df.groupby(
        "deleted_block"
    ).size() == 18
).all()

assert (
    moving_block_hac_df["n_years"] == 49
).all()

print("\n" + "=" * 68)
print("MOVING BLOCK DELETION DESIGN")
print("=" * 68)

print("Calendar years:", len(YEARS_1220))
print("Moving five-year windows:", len(block_starts_1220))
print("Contrasts per deletion:", 18)
print("Total HAC estimates:", len(moving_block_hac_df))

print("First deleted block: 1972-1976")
print("Last deleted block: 2021-2025")


# ------------------------------------------------------------
# 6. Five primary robust findings
# ------------------------------------------------------------

primary_keys_1220 = [
    ("Iran - China", 3),
    ("Iran - China", 7),
    ("Iran - United States", 7),
    ("United States - China", 3),
    ("United States - China", 5)
]

summary_rows_1220 = []

for pair_label, topic_id in primary_keys_1220:

    subset = moving_block_hac_df.loc[
        (moving_block_hac_df["pair_label"] == pair_label) &
        (moving_block_hac_df["topic_id"] == topic_id)
    ].copy()

    assert len(subset) == 50

    base = baseline_1220.loc[
        (baseline_1220["pair_label"] == pair_label) &
        (baseline_1220["topic_id"] == topic_id),
        "effect_pp"
    ].iloc[0]

    subset["absolute_shift_pp"] = np.abs(
        subset["effect_pp"] - base
    )

    influential = subset.loc[
        subset["absolute_shift_pp"].idxmax()
    ]

    same_direction = (
        np.sign(subset["effect_pp"]) ==
        np.sign(base)
    )

    n_direction = int(
        same_direction.sum()
    )

    n_significant = int(
        subset["holm_significant"].sum()
    )

    summary_rows_1220.append({
        "pair_label": pair_label,
        "topic_id": topic_id,
        "baseline_effect_pp": base,
        "minimum_effect_pp": subset["effect_pp"].min(),
        "maximum_effect_pp": subset["effect_pp"].max(),
        "maximum_absolute_shift_pp": (
            subset["absolute_shift_pp"].max()
        ),
        "most_influential_block": (
            influential["deleted_block"]
        ),
        "same_direction_blocks": n_direction,
        "holm_significant_blocks": n_significant,
        "n_blocks": len(subset),
        "block_robust": (
            n_direction == 50 and
            n_significant == 50
        )
    })

moving_block_primary_summary = pd.DataFrame(
    summary_rows_1220
)

print("\n" + "=" * 68)
print("PRIMARY FINDINGS: MOVING BLOCK ROBUSTNESS")
print("=" * 68)

display(
    moving_block_primary_summary.round(5)
)


# ------------------------------------------------------------
# 7. Largest shifts across all 18 contrasts
# ------------------------------------------------------------

moving_block_detailed_1220 = (
    moving_block_hac_df.merge(
        baseline_1220[
            ["pair_label", "topic_id", "effect_pp"]
        ].rename(
            columns={"effect_pp": "baseline_effect_pp"}
        ),
        on=["pair_label", "topic_id"],
        how="left",
        validate="many_to_one"
    )
)

moving_block_detailed_1220["effect_shift_pp"] = (
    moving_block_detailed_1220["effect_pp"] -
    moving_block_detailed_1220["baseline_effect_pp"]
)

moving_block_detailed_1220["absolute_shift_pp"] = (
    moving_block_detailed_1220["effect_shift_pp"].abs()
)

largest_shifts_1220 = (
    moving_block_detailed_1220.sort_values(
        "absolute_shift_pp",
        ascending=False
    ).head(15)
)

print("\n" + "=" * 68)
print("LARGEST HISTORICAL BLOCK INFLUENCES")
print("=" * 68)

display(
    largest_shifts_1220[
        [
            "deleted_block",
            "pair_label",
            "topic_id",
            "baseline_effect_pp",
            "effect_pp",
            "effect_shift_pp",
            "holm_adjusted_p"
        ]
    ].round(5)
)


# ------------------------------------------------------------
# 8. Final validation and decision
# ------------------------------------------------------------

n_direction_stable_1220 = int(
    (
        moving_block_primary_summary[
            "same_direction_blocks"
        ] == 50
    ).sum()
)

n_significance_stable_1220 = int(
    (
        moving_block_primary_summary[
            "holm_significant_blocks"
        ] == 50
    ).sum()
)

n_robust_1220 = int(
    moving_block_primary_summary[
        "block_robust"
    ].sum()
)

print("\n" + "=" * 68)
print("MOVING BLOCK DELETION VALIDATION")
print("=" * 68)

print("Primary findings evaluated: 5")

print(
    "Directionally stable across all 50 windows:",
    f"{n_direction_stable_1220}/5"
)

print(
    "Holm significant across all 50 windows:",
    f"{n_significance_stable_1220}/5"
)

print(
    "Meeting both stability criteria:",
    f"{n_robust_1220}/5"
)

print("\nREFERENCE INFERENCE: REPRODUCED")
print("MOVING BLOCK COMPUTATION: PASSED")
print("Scientific interpretation: PENDING REVIEW")
print("Files saved: NONE")




### Step 12.21 — Investigation of Block-Deletion Exceptions

**Objective:** Identify and characterize the deletion windows in which the primary findings lose multiplicity-adjusted statistical significance.

This audit will:

1. Identify all non-significant windows among the five primary findings.
2. Compare their estimated effects and 95% HAC confidence intervals with the full-sample estimates.
3. Examine adjacent deletion windows to assess local sensitivity.
4. Distinguish directional stability from Holm-adjusted statistical significance.
5. Prepare a reproducible robustness summary for potential supplementary reporting.

**Statistical specifications:** Three-period regression, calendar-aware Newey–West HAC (lag 5), finite-sample correction, and Holm adjustment across 18 contrasts.

The analysis does not retrain the NMF model or generate publication figures. Final scientific interpretation remains pending review.


In [ ]:

# ============================================================
# STEP 12.21
# INVESTIGATION OF BLOCK-DELETION EXCEPTIONS
# ============================================================

print("=" * 68)
print("BLOCK-DELETION EXCEPTION INVESTIGATION")
print("=" * 68)

# ------------------------------------------------------------
# 1. Validate previous results
# ------------------------------------------------------------

required_objects = [
    "moving_block_hac_df",
    "moving_block_primary_summary",
    "baseline_1220"
]

for name in required_objects:
    assert name in globals(), f"Missing object: {name}"

primary_keys = moving_block_primary_summary[
    ["pair_label", "topic_id"]
].drop_duplicates()

primary_block = moving_block_hac_df.merge(
    primary_keys,
    on=["pair_label", "topic_id"],
    how="inner",
    validate="many_to_one"
).copy()

assert len(primary_block) == 250
assert primary_block["deleted_block"].nunique() == 50

assert (
    primary_block.groupby(
        ["pair_label", "topic_id"]
    ).size() == 50
).all()

assert (
    primary_block["holm_significant"] ==
    primary_block["holm_adjusted_p"].lt(0.05)
).all()

# ------------------------------------------------------------
# 2. Join full-sample estimates
# ------------------------------------------------------------

baseline_columns = baseline_1220[
    [
        "pair_label",
        "topic_id",
        "effect_pp",
        "ci_lower_pp",
        "ci_upper_pp",
        "holm_adjusted_p"
    ]
].rename(columns={
    "effect_pp": "baseline_effect_pp",
    "ci_lower_pp": "baseline_ci_lower_pp",
    "ci_upper_pp": "baseline_ci_upper_pp",
    "holm_adjusted_p": "baseline_adjusted_p"
})

primary_block = primary_block.merge(
    baseline_columns,
    on=["pair_label", "topic_id"],
    validate="many_to_one"
)

primary_block["effect_shift_pp"] = (
    primary_block["effect_pp"] -
    primary_block["baseline_effect_pp"]
)

primary_block["direction_preserved"] = (
    np.sign(primary_block["effect_pp"]) ==
    np.sign(primary_block["baseline_effect_pp"])
)

primary_block["unadjusted_ci_excludes_zero"] = (
    (primary_block["ci_lower_pp"] > 0) |
    (primary_block["ci_upper_pp"] < 0)
)

primary_block["holm_significant"] = (
    primary_block["holm_adjusted_p"] < 0.05
)

# ------------------------------------------------------------
# 3. Identify exceptional windows
# ------------------------------------------------------------

block_exception_df = primary_block.loc[
    ~primary_block["holm_significant"]
].sort_values(
    ["pair_label", "topic_id", "block_start"]
).copy()

print("\nNON-SIGNIFICANT DELETION WINDOWS")
print("-" * 68)

print("Total exceptions:", len(block_exception_df))

exception_columns = [
    "pair_label",
    "topic_id",
    "deleted_block",
    "effect_pp",
    "effect_shift_pp",
    "hac_se_pp",
    "ci_lower_pp",
    "ci_upper_pp",
    "p_value",
    "holm_adjusted_p",
    "direction_preserved",
    "unadjusted_ci_excludes_zero"
]

with pd.option_context(
    "display.max_columns", None,
    "display.width", 180,
    "display.float_format", "{:.7g}".format
):
    display(
        block_exception_df[
            exception_columns
        ].reset_index(drop=True)
    )

# ------------------------------------------------------------
# 4. Study neighboring deletion windows
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("LOCAL NEIGHBORHOOD OF EXCEPTIONS")
print("=" * 68)

neighborhood_frames = []

for _, exception in block_exception_df.iterrows():

    start = int(exception["block_start"])

    neighborhood = primary_block.loc[
        (
            primary_block["pair_label"] ==
            exception["pair_label"]
        ) &
        (
            primary_block["topic_id"] ==
            exception["topic_id"]
        ) &
        (
            primary_block["block_start"].between(
                start - 2,
                start + 2
            )
        )
    ].copy()

    neighborhood_frames.append(neighborhood)

if neighborhood_frames:

    exception_neighborhood_df = (
        pd.concat(
            neighborhood_frames,
            ignore_index=True
        )
        .drop_duplicates(
            ["pair_label", "topic_id", "deleted_block"]
        )
        .sort_values(
            ["pair_label", "topic_id", "block_start"]
        )
    )

    with pd.option_context(
        "display.float_format", "{:.7g}".format
    ):
        display(
            exception_neighborhood_df[
                [
                    "pair_label",
                    "topic_id",
                    "deleted_block",
                    "effect_pp",
                    "ci_lower_pp",
                    "ci_upper_pp",
                    "holm_adjusted_p",
                    "holm_significant"
                ]
            ].reset_index(drop=True)
        )

else:
    exception_neighborhood_df = pd.DataFrame()
    print("No exceptions detected.")

# ------------------------------------------------------------
# 5. Full primary robustness summary
# ------------------------------------------------------------

summary_rows = []

for (pair, topic), group in primary_block.groupby(
    ["pair_label", "topic_id"],
    sort=False
):

    worst_p_row = group.loc[
        group["holm_adjusted_p"].idxmax()
    ]

    summary_rows.append({
        "pair_label": pair,
        "topic_id": topic,
        "baseline_effect_pp": (
            group["baseline_effect_pp"].iloc[0]
        ),
        "min_effect_pp": group["effect_pp"].min(),
        "max_effect_pp": group["effect_pp"].max(),
        "direction_preserved_n": int(
            group["direction_preserved"].sum()
        ),
        "holm_significant_n": int(
            group["holm_significant"].sum()
        ),
        "non_significant_n": int(
            (~group["holm_significant"]).sum()
        ),
        "maximum_adjusted_p": (
            group["holm_adjusted_p"].max()
        ),
        "worst_p_block": worst_p_row["deleted_block"]
    })

block_robustness_report = pd.DataFrame(
    summary_rows
)

print("\n" + "=" * 68)
print("PRIMARY FINDINGS: DETAILED ROBUSTNESS SUMMARY")
print("=" * 68)

with pd.option_context(
    "display.max_columns", None,
    "display.float_format", "{:.7g}".format
):
    display(block_robustness_report)

# ------------------------------------------------------------
# 6. Diagnose period-level sample sizes
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("EXCEPTION WINDOW SAMPLE COMPOSITION")
print("=" * 68)

if len(block_exception_df) > 0:

    display(
        block_exception_df[
            [
                "pair_label",
                "topic_id",
                "deleted_block",
                "n_years",
                "n_early",
                "n_middle",
                "n_late"
            ]
        ].reset_index(drop=True)
    )

else:
    print("All primary findings retain significance.")

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

direction_stable = int(
    (
        block_robustness_report[
            "direction_preserved_n"
        ] == 50
    ).sum()
)

significance_stable = int(
    (
        block_robustness_report[
            "holm_significant_n"
        ] == 50
    ).sum()
)

print("\n" + "=" * 68)
print("EXCEPTION AUDIT VALIDATION")
print("=" * 68)

print("Primary findings:", len(primary_keys))
print("Total primary estimates:", len(primary_block))
print("Exceptions:", len(block_exception_df))

print(
    "Direction stable in all windows:",
    f"{direction_stable}/5"
)

print(
    "Holm significance stable in all windows:",
    f"{significance_stable}/5"
)

print("\nBLOCK EXCEPTION AUDIT: PASSED")
print("Scientific interpretation: PENDING REVIEW")
print("Files saved: NONE")



### Step 12.22 — Publication-Ready Moving-Block Robustness

**Objective:** Summarize the sensitivity of the five primary findings to moving five-year block deletion and visualize the only finding with incomplete multiplicity-adjusted significance stability.

The analysis uses 50 overlapping deletion windows (1972–1976 through 2021–2025), with HAC inference recalculated after each deletion and Holm correction applied across the 18 primary contrasts.

**Reporting principles:**
- Distinguish effect-direction stability from adjusted statistical significance.
- Report pointwise 95% HAC confidence intervals separately from Holm-adjusted p-values.
- Treat overlapping deletion windows as sensitivity checks, not independent replications.
- Preserve the five original publication findings without redefining the primary analysis.

The resulting table and figure are candidates for supplementary material, subject to visual and scientific review. No files will be saved in this step.


In [ ]:

# ============================================================
# STEP 12.22A — SUPPLEMENTARY ROBUSTNESS TABLE
# ============================================================


assert "primary_block" in globals()
assert "block_exception_df" in globals()
assert "block_robustness_report" in globals()

topic_names = {
    3: "Regional conflicts / Kampuchea",
    5: "Human rights and freedom",
    7: "Nuclear arms and disarmament"
}

rows = []

for (pair, topic), g in primary_block.groupby(
    ["pair_label", "topic_id"], sort=False
):
    g = g.sort_values("block_start")

    assert len(g) == 50
    assert g["deleted_block"].nunique() == 50

    rows.append({
        "Country comparison": pair,
        "Topic": f"T{int(topic):02d}",
        "Topic label": topic_names.get(int(topic), ""),
        "Baseline effect (pp)": g["baseline_effect_pp"].iloc[0],
        "Minimum effect (pp)": g["effect_pp"].min(),
        "Maximum effect (pp)": g["effect_pp"].max(),
        "Maximum absolute shift (pp)": (
            g["effect_shift_pp"].abs().max()
        ),
        "Direction stable": (
            f"{int(g['direction_preserved'].sum())}/50"
        ),
        "Holm significant": (
            f"{int(g['holm_significant'].sum())}/50"
        ),
        "Maximum adjusted p": g["holm_adjusted_p"].max(),
        "Non-significant windows": ", ".join(
            g.loc[
                ~g["holm_significant"],
                "deleted_block"
            ].astype(str)
        ) or "None"
    })

supplementary_moving_block_table = pd.DataFrame(rows)

print("=" * 68)
print("SUPPLEMENTARY TABLE — MOVING-BLOCK ROBUSTNESS")
print("=" * 68)

with pd.option_context(
    "display.max_columns", None,
    "display.width", 200,
    "display.float_format", "{:.4f}".format
):
    display(supplementary_moving_block_table)

assert len(supplementary_moving_block_table) == 5
assert primary_block["direction_preserved"].all()
assert primary_block["holm_significant"].sum() == 248

print("\nTABLE VALIDATION: PASSED")
print("Files saved: NONE")


In [ ]:

# ============================================================
# STEP 12.22B — MOVING-BLOCK SENSITIVITY FIGURE
# ============================================================

plot_data = primary_block.loc[
    (primary_block["pair_label"] == "Iran - United States") &
    (primary_block["topic_id"] == 7)
].sort_values("block_start").copy()

assert len(plot_data) == 50
assert plot_data["block_start"].is_unique

x = plot_data["block_start"].to_numpy(dtype=int)
effect = plot_data["effect_pp"].to_numpy(dtype=float)
lower = plot_data["ci_lower_pp"].to_numpy(dtype=float)
upper = plot_data["ci_upper_pp"].to_numpy(dtype=float)
p_adj = plot_data["holm_adjusted_p"].to_numpy(dtype=float)

exceptions = ~plot_data["holm_significant"].to_numpy(dtype=bool)

baseline = float(
    plot_data["baseline_effect_pp"].iloc[0]
)

fig_moving_block_robustness, axes = plt.subplots(
    2, 1,
    figsize=(11.5, 7.5),
    sharex=True
)

ax1, ax2 = axes

main_color = "#26384B"
exception_color = "#C86E22"

# ------------------------------------------------------------
# Panel A — Effect estimates and confidence intervals
# ------------------------------------------------------------

ax1.fill_between(
    x, lower, upper,
    color="#BAC8D3",
    alpha=0.50,
    linewidth=0,
    label="95% pointwise HAC CI"
)

ax1.plot(
    x, effect,
    color=main_color,
    linewidth=1.9,
    marker="o",
    markersize=3.0,
    label="Block-deleted estimate"
)

ax1.axhline(
    baseline,
    color="#637D73",
    linestyle="--",
    linewidth=1.7,
    label="Full-sample estimate"
)

ax1.scatter(
    x[exceptions],
    effect[exceptions],
    s=80,
    color=exception_color,
    edgecolor="white",
    linewidth=1.0,
    zorder=5,
    label="Holm non-significant window"
)

ax1.axhline(
    0,
    color="gray",
    linestyle=":",
    linewidth=0.9
)

ax1.set_ylabel("Change in bilateral topic gap (pp)")
ax1.set_title(
    "A. Effect estimates and pointwise HAC uncertainty",
    loc="left",
    fontsize=11,
    fontweight="bold"
)

# ------------------------------------------------------------
# Panel B — Holm-adjusted p-values
# ------------------------------------------------------------

ax2.plot(
    x, p_adj,
    color=main_color,
    linewidth=1.9,
    marker="o",
    markersize=3.0
)

ax2.scatter(
    x[exceptions],
    p_adj[exceptions],
    s=80,
    color=exception_color,
    edgecolor="white",
    linewidth=1.0,
    zorder=5
)

ax2.axhline(
    0.05,
    color="#9D4545",
    linestyle="--",
    linewidth=1.5,
    label="Adjusted significance threshold (0.05)"
)

ax2.set_ylim(
    0,
    max(0.11, float(np.max(p_adj)) * 1.15)
)

ax2.set_ylabel("Holm-adjusted p-value")
ax2.set_xlabel("First year of deleted five-year block")

ax2.set_title(
    "B. Multiplicity-adjusted statistical significance",
    loc="left",
    fontsize=11,
    fontweight="bold"
)

# ------------------------------------------------------------
# Shared formatting
# ------------------------------------------------------------

for ax in axes:
    ax.grid(
        axis="y",
        alpha=0.20,
        linewidth=0.7
    )
    ax.set_axisbelow(True)
    ax.spines["top"].set_visible(False)
    ax.spines["right"].set_visible(False)

ax2.set_xticks(np.arange(1972, 2022, 5))
ax2.set_xlim(1971, 2022)

# Place the legend OUTSIDE the plotting panels.
handles1, labels1 = ax1.get_legend_handles_labels()
handles2, labels2 = ax2.get_legend_handles_labels()

fig_moving_block_robustness.suptitle(
    "Moving Five-Year Block Deletion: Iran–United States\n"
    "Nuclear Arms and Disarmament (Topic 7)",
    fontsize=14,
    fontweight="bold",
    y=0.985
)

fig_moving_block_robustness.legend(
    handles1 + handles2,
    labels1 + labels2,
    loc="upper center",
    bbox_to_anchor=(0.5, 0.900),
    ncol=3,
    frameon=False,
    fontsize=9
)

fig_moving_block_robustness.subplots_adjust(
    left=0.11,
    right=0.97,
    top=0.77,
    bottom=0.15,
    hspace=0.27
)

fig_moving_block_robustness.text(
    0.11,
    0.045,
    "Note: Confidence intervals are pointwise and not "
    "multiplicity-adjusted. Holm correction covers 18 contrasts. "
    "Deletion windows overlap.",
    fontsize=8.5
)

plt.show()

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("=" * 68)
print("SUPPLEMENTARY FIGURE VALIDATION")
print("=" * 68)

print("Deletion windows:", len(plot_data))
print("Holm-significant windows:", int((~exceptions).sum()))
print("Non-significant windows:", int(exceptions.sum()))

print("\nException windows:")
print(
    plot_data.loc[
        exceptions,
        ["deleted_block", "effect_pp", "holm_adjusted_p"]
    ].to_string(index=False)
)

print("\nFIGURE GENERATION: PASSED")
print("Files saved: NONE")



### Step 12.23 — Export of Publication-Ready Supplementary Results

**Objective:** Export the validated moving-block deletion robustness table and supplementary sensitivity figure.

**Supplementary Table S1:** Sensitivity of five primary findings across 50 overlapping five-year deletion windows, reporting effect ranges, directional consistency, and Holm-adjusted statistical significance.

**Supplementary Figure S1:** Moving-block deletion sensitivity of the Iran–United States nuclear arms and disarmament contrast (Topic 7).

**Export specifications:**
- Table: CSV with unrounded numerical values and a formatted TSV version.
- Figure: vector PDF, high-resolution PNG, and 600-dpi TIFF.
- No statistical models will be refitted.
- Original analysis results and notebook objects will remain unchanged.

**Interpretive limitation:** The 95% HAC confidence intervals are pointwise rather than multiplicity-adjusted. Overlapping deletion windows do not represent independent statistical replications.


In [ ]:

# ============================================================
# STEP 12.23A — EXPORT SUPPLEMENTARY TABLE S1
# ============================================================

output_dir = Path.cwd() / "publication_outputs" / "supplementary"
output_dir.mkdir(parents=True, exist_ok=True)

assert "supplementary_moving_block_table" in globals()

raw_table = supplementary_moving_block_table.copy()

assert len(raw_table) == 5

# Preserve numerical precision in the research dataset
csv_path = output_dir / "Supplementary_Table_S1_Robustness.csv"

raw_table.to_csv(
    csv_path,
    index=False,
    encoding="utf-8-sig",
    float_format="%.12g"
)

# Prepare a manuscript-friendly version
formatted_table = raw_table.copy()

numeric_columns = [
    "Baseline effect (pp)",
    "Minimum effect (pp)",
    "Maximum effect (pp)",
    "Maximum absolute shift (pp)"
]

for col in numeric_columns:
    formatted_table[col] = formatted_table[col].map(
        lambda x: f"{x:.2f}"
    )

formatted_table["Maximum adjusted p"] = (
    formatted_table["Maximum adjusted p"].map(
        lambda x: "<0.0001" if x < 0.0001
        else f"{x:.4f}"
    )
)

tsv_path = output_dir / "Supplementary_Table_S1_Formatted.tsv"

formatted_table.to_csv(
    tsv_path,
    sep="\t",
    index=False,
    encoding="utf-8-sig"
)

# # Save reporting notes
# table_note = (
#     "Supplementary Table S1. Moving five-year block deletion "
#     "robustness of five primary topic contrasts.\n\n"
#     "Fifty overlapping five-year deletion windows were assessed. "
#     "Effect estimates are expressed in percentage points (pp). "
#     "Significance was evaluated using Newey-West HAC inference "
#     "with Holm adjustment across 18 contrasts within each "
#     "deletion analysis. Reported effect ranges are sensitivity "
#     "ranges, not confidence intervals. Deletion windows overlap "
#     "and are not independent replications.\n"
# )

# note_path = output_dir / "Supplementary_Table_S1_Note.txt"
# note_path.write_text(table_note, encoding="utf-8")

# assert csv_path.exists()
# assert tsv_path.exists()
# assert note_path.exists()

# reloaded = pd.read_csv(csv_path)
# assert len(reloaded) == 5

print("=" * 65)
print("SUPPLEMENTARY TABLE EXPORT")
print("=" * 65)

# print("\nSaved files:")
# for path in [csv_path, tsv_path, note_path]:
#     print(f" - {path.resolve()}")

print("\nFormatted table:")
display(formatted_table)

print("\nTABLE EXPORT VALIDATION: PASSED")


In [ ]:

# # ============================================================
# # STEP 12.23B — EXPORT SUPPLEMENTARY FIGURE S1
# # ============================================================

# assert "fig_moving_block_robustness" in globals()

# fig = fig_moving_block_robustness

# assert len(fig.axes) == 2, (
#     "Expected two-panel supplementary figure."
# )

# output_dir = Path.cwd() / "publication_outputs" / "supplementary"
# output_dir.mkdir(parents=True, exist_ok=True)

# base = "Supplementary_Figure_S1_Moving_Block_T07"

# pdf_path = output_dir / f"{base}.pdf"
# png_path = output_dir / f"{base}.png"
# tiff_path = output_dir / f"{base}.tiff"

# # Ensure the figure canvas is fully rendered
# fig.canvas.draw()

# # Vector PDF — suitable for journal production
# fig.savefig(
#     pdf_path,
#     format="pdf",
#     bbox_inches="tight",
#     pad_inches=0.18,
#     facecolor="white"
# )

# # High-resolution raster preview
# fig.savefig(
#     png_path,
#     format="png",
#     dpi=600,
#     bbox_inches="tight",
#     pad_inches=0.18,
#     facecolor="white"
# )

# # Print-quality TIFF with lossless compression
# fig.savefig(
#     tiff_path,
#     format="tiff",
#     dpi=600,
#     bbox_inches="tight",
#     pad_inches=0.18,
#     facecolor="white",
#     pil_kwargs={"compression": "tiff_lzw"}
# )

# # Publication caption
# caption = (
#     "Supplementary Figure S1. Moving five-year block deletion "
#     "sensitivity for the Iran–United States contrast in nuclear "
#     "arms and disarmament (Topic 7). Panel A presents changes "
#     "in the bilateral topic-share gap after deleting each "
#     "consecutive five-year window, alongside pointwise 95% "
#     "Newey–West HAC confidence intervals and the full-sample "
#     "estimate. Panel B presents Holm-adjusted p-values across "
#     "18 contrasts, with the 0.05 significance threshold. "
#     "The effect direction was preserved in all 50 windows, "
#     "while adjusted significance was retained in 48. "
#     "The windows 1976–1980 and 1977–1981 were the two "
#     "exceptions. Deletion windows overlap, and the confidence "
#     "intervals are not multiplicity-adjusted."
# )

# caption_path = output_dir / f"{base}_Caption.txt"
# caption_path.write_text(caption, encoding="utf-8")

# # ------------------------------------------------------------
# # File validation
# # ------------------------------------------------------------

# for path in [pdf_path, png_path, tiff_path, caption_path]:
#     assert path.exists()
#     assert path.stat().st_size > 0

# with Image.open(png_path) as im:
#     png_size = im.size
#     assert min(png_size) >= 2000

# with Image.open(tiff_path) as im:
#     tiff_size = im.size
#     assert min(tiff_size) >= 2000
#     assert im.info.get("dpi") is not None

# print("=" * 65)
# print("SUPPLEMENTARY FIGURE EXPORT")
# print("=" * 65)

# print("\nSaved files:")

# for path in [pdf_path, png_path, tiff_path, caption_path]:
#     print(
#         f" - {path.name} "
#         f"({path.stat().st_size / 1024 / 1024:.2f} MB)"
#     )

# print("\nPNG dimensions:", png_size)
# print("TIFF dimensions:", tiff_size)
# print("Output directory:", output_dir.resolve())

# print("\nFIGURE EXPORT VALIDATION: PASSED")
# print("SUPPLEMENTARY MATERIAL EXPORT: COMPLETED")



### Step 12.24 — Final Statistical Evidence Classification

**Objective:** Integrate the 18 primary HAC contrasts with the validated moving-block sensitivity results to establish a transparent hierarchy of statistical evidence.

The primary analysis uses a balanced three-country cohort covering 1972–2025, with six eligible NMF topics, three bilateral comparisons, Newey–West HAC inference, and Holm correction across 18 contrasts.

Findings will be classified into four categories:

1. **Fully block-stable:** Primary significance, previously established cross-specification robustness, and Holm-adjusted significance in all 50 moving-block deletion windows.
2. **Partially block-sensitive:** Primary and cross-specification significance, but loss of adjusted significance in one or more deletion windows.
3. **Model-sensitive:** Significant in the primary analysis but not consistently significant across the previously evaluated model and HAC specifications.
4. **Not significant:** Does not meet the primary Holm-adjusted significance threshold.

Pointwise confidence intervals must not be described as multiplicity-adjusted.

**Scope:** This is an evidence-classification and reproducibility audit, not a new inferential analysis. Topic interpretations remain subject to qualitative validation.

No publication files will be saved before review.


In [ ]:

# ============================================================
# STEP 12.24A — FINAL STATISTICAL EVIDENCE AUDIT
# ============================================================

# ------------------------------------------------------------
# 1. Verify existing analytical sources
# ------------------------------------------------------------

assert "reference" in globals(), (
    "The 18-row reference HAC results are required."
)

assert "supplementary_moving_block_table" in globals(), (
    "Run the validated Step 12.22A first."
)

ref = reference.copy()
block = supplementary_moving_block_table.copy()

print("=" * 68)
print("FINAL STATISTICAL EVIDENCE AUDIT")
print("=" * 68)

print("\nSource dimensions:")
print("Reference HAC contrasts:", ref.shape)
print("Block sensitivity findings:", block.shape)

# ------------------------------------------------------------
# 2. Resolve reference schema without changing source data
# ------------------------------------------------------------

def select_column(frame, candidates):
    found = [c for c in candidates if c in frame.columns]
    if not found:
        raise ValueError(
            f"Missing expected column. Tried: {candidates}\n"
            f"Available: {frame.columns.tolist()}"
        )
    return found[0]

columns = {
    "pair": select_column(ref, ["pair", "pair_label"]),
    "topic_id": select_column(ref, ["topic_id"]),
    "effect": select_column(ref, ["effect_pp"]),
    "ci_low": select_column(
        ref, ["ci_lower_pp", "ci_lower"]
    ),
    "ci_high": select_column(
        ref, ["ci_upper_pp", "ci_upper"]
    ),
    "adjusted_p": select_column(
        ref, ["holm_adjusted_p", "p_holm", "holm_p"]
    )
}

audit = ref[
    list(columns.values())
].copy()

audit.columns = list(columns.keys())

def normalize_pair(value):
    text = str(value).upper()

    text = re.sub(
        r"UNITED[\s\-]+STATES", "USA", text
    )
    text = re.sub(r"\bIRAN\b", "IRN", text)
    text = re.sub(r"\bCHINA\b", "CHN", text)

    codes = re.findall(
        r"\b(?:IRN|USA|CHN)\b", text
    )

    if len(codes) != 2:
        raise ValueError(
            f"Unrecognized country pair: {value}"
        )

    return "-".join(codes)

audit["pair"] = audit["pair"].map(normalize_pair)
audit["topic_id"] = audit["topic_id"].astype(int)

for col in ["effect", "ci_low", "ci_high", "adjusted_p"]:
    audit[col] = pd.to_numeric(
        audit[col], errors="raise"
    )

# Reference specification validation
assert len(audit) == 18
assert not audit.duplicated(
    ["pair", "topic_id"]
).any()

assert set(audit["pair"]) == {
    "IRN-CHN", "IRN-USA", "USA-CHN"
}

assert set(audit["topic_id"]) == {
    1, 2, 3, 4, 5, 7
}

assert (
    audit["ci_low"] <= audit["effect"]
).all()

assert (
    audit["effect"] <= audit["ci_high"]
).all()

assert audit["adjusted_p"].between(0, 1).all()

if "hac_lag" in ref.columns:
    assert (ref["hac_lag"] == 5).all()

# ------------------------------------------------------------
# 3. Match the five robust findings
# ------------------------------------------------------------

block["pair"] = block[
    "Country comparison"
].map(normalize_pair)

block["topic_id"] = (
    block["Topic"]
    .astype(str)
    .str.extract(r"(\d+)")[0]
    .astype(int)
)

assert len(block) == 5
assert not block.duplicated(
    ["pair", "topic_id"]
).any()

audit = audit.merge(
    block[
        [
            "pair",
            "topic_id",
            "Baseline effect (pp)",
            "Direction stable",
            "Holm significant",
            "Non-significant windows"
        ]
    ],
    on=["pair", "topic_id"],
    how="left",
    validate="one_to_one"
)

# Verify original point estimates are reproduced
selected = audit[
    audit["Baseline effect (pp)"].notna()
].copy()

assert len(selected) == 5

assert np.allclose(
    selected["effect"].to_numpy(),
    selected["Baseline effect (pp)"].to_numpy(),
    atol=1e-6,
    rtol=0
), "Baseline effect mismatch."

# ------------------------------------------------------------
# 4. Classify statistical evidence
# ------------------------------------------------------------

def classify_evidence(row):
    significant = row["adjusted_p"] < 0.05
    robust_candidate = pd.notna(
        row["Baseline effect (pp)"]
    )

    if not significant:
        return "Not significant"

    if not robust_candidate:
        return "Primary significant; model-sensitive"

    if row["Holm significant"] == "50/50":
        return "Fully block-stable"

    return "Partially block-sensitive"

audit["evidence_class"] = audit.apply(
    classify_evidence, axis=1
)

topic_labels = {
    1: "Economic development and cooperation",
    2: "China and peaceful development",
    3: "Kampuchea and regional conflicts",
    4: "Iran and Islamic Republic",
    5: "Human rights and freedom discourse",
    7: "Nuclear arms and disarmament"
}

pair_labels = {
    "IRN-CHN": "Iran - China",
    "IRN-USA": "Iran - United States",
    "USA-CHN": "United States - China"
}

audit["comparison"] = audit["pair"].map(
    pair_labels
)

audit["topic_label"] = audit["topic_id"].map(
    topic_labels
)

audit["primary_significant"] = (
    audit["adjusted_p"] < 0.05
)

# ------------------------------------------------------------
# 5. Validate evidence counts
# ------------------------------------------------------------

counts = audit["evidence_class"].value_counts()

n_primary = int(audit["primary_significant"].sum())
n_full = int(
    (audit["evidence_class"] == "Fully block-stable").sum()
)
n_partial = int(
    (audit["evidence_class"] == "Partially block-sensitive").sum()
)
n_sensitive = int(
    (audit["evidence_class"] ==
     "Primary significant; model-sensitive").sum()
)
n_nonsig = int(
    (audit["evidence_class"] == "Not significant").sum()
)

assert n_primary == 10, (
    "Primary significance count differs from prior results."
)
assert n_full == 4
assert n_partial == 1
assert n_sensitive == 5
assert n_nonsig == 8

assert n_full + n_partial + n_sensitive + n_nonsig == 18

# ------------------------------------------------------------
# 6. Display publication evidence table
# ------------------------------------------------------------

final_evidence_audit_df = audit.copy()

display_columns = [
    "comparison",
    "topic_id",
    "topic_label",
    "effect",
    "ci_low",
    "ci_high",
    "adjusted_p",
    "evidence_class"
]

print("\n" + "=" * 68)
print("ALL 18 PRIMARY CONTRASTS")
print("=" * 68)

with pd.option_context(
    "display.max_rows", None,
    "display.max_columns", None,
    "display.float_format", "{:.6f}".format
):
    display(
        final_evidence_audit_df[
            display_columns
        ].sort_values(
            ["comparison", "topic_id"]
        )
    )

# ------------------------------------------------------------
# 7. Final evidence hierarchy
# ------------------------------------------------------------

summary = pd.DataFrame({
    "Evidence category": [
        "Fully block-stable",
        "Partially block-sensitive",
        "Primary significant; model-sensitive",
        "Not significant"
    ],
    "Number of contrasts": [
        n_full,
        n_partial,
        n_sensitive,
        n_nonsig
    ]
})

print("\n" + "=" * 68)
print("FINAL EVIDENCE HIERARCHY")
print("=" * 68)

display(summary)

print("\nPrimary Holm-significant contrasts:", n_primary)
print("Cross-specification robust candidates:", len(selected))
print("Fully block-stable candidates:", n_full)
print("Partially block-sensitive candidates:", n_partial)

print("\n" + "=" * 68)
print("VALIDATION")
print("=" * 68)

print("Reference contrasts: PASSED")
print("Primary significance count: PASSED")
print("Block sensitivity linkage: PASSED")
print("Evidence classification: PASSED")
print("FINAL EVIDENCE AUDIT: PASSED")
print("Files saved: NONE")



### Step 12.25 — Manuscript-Ready Statistical Results

**Objective:** Generate a reproducible, publication-oriented statistical Results summary from the validated primary HAC estimates and robustness classifications.

The analysis uses 162 speeches from Iran, the United States, and China (54 speeches per country, 1972–2025). Historical changes are evaluated across the early (1972–1990) and late (2011–2025) periods.

Reporting principles:

- Report estimated changes in bilateral topic-share differences in percentage points.
- Present 95% pointwise Newey–West HAC confidence intervals alongside Holm-adjusted p-values.
- Distinguish primary significance, cross-specification robustness, and moving-block deletion stability.
- Report the two non-significant deletion windows for Iran–United States, Topic 7.
- Avoid interpreting signed changes in bilateral gaps automatically as convergence or divergence.
- Treat topic labels as provisional until qualitative semantic validation is completed.

**Outputs:** An integrated five-finding statistical table, a four-category evidence summary, and reproducible English Results paragraphs.

No files will be saved until the generated material is reviewed.


In [ ]:

# ============================================================
# STEP 12.25A — MANUSCRIPT-READY STATISTICAL RESULTS
# ============================================================

assert "final_evidence_audit_df" in globals(), (
    "Run Step 12.24A first."
)

evidence = final_evidence_audit_df.copy()

required = [
    "pair", "topic_id", "topic_label",
    "effect", "ci_low", "ci_high",
    "adjusted_p", "evidence_class",
    "primary_significant",
    "Holm significant"
]

assert set(required).issubset(evidence.columns)

assert len(evidence) == 18
assert evidence["primary_significant"].sum() == 10

# ------------------------------------------------------------
# 1. Formatting functions
# ------------------------------------------------------------

def format_p(p):
    p = float(p)
    if not np.isfinite(p) or not 0 <= p <= 1:
        raise ValueError("Invalid adjusted p-value.")
    if p < 0.0001:
        return "<0.0001"
    return f"{p:.3g}"

def format_effect(value):
    return f"{float(value):+.2f}"

def format_ci(row):
    return (
        f"[{row['ci_low']:.2f}, "
        f"{row['ci_high']:.2f}]"
    )

pair_names = {
    "IRN-CHN": "Iran–China",
    "IRN-USA": "Iran–United States",
    "USA-CHN": "United States–China"
}

# ------------------------------------------------------------
# 2. Validate the five cross-specification robust findings
# ------------------------------------------------------------

robust = evidence[
    evidence["evidence_class"].isin([
        "Fully block-stable",
        "Partially block-sensitive"
    ])
].copy()

assert len(robust) == 5
assert robust["primary_significant"].all()

assert (
    robust["evidence_class"] == "Fully block-stable"
).sum() == 4

assert (
    robust["evidence_class"] == "Partially block-sensitive"
).sum() == 1

robust["Comparison"] = robust["pair"].map(pair_names)

robust["Topic"] = robust["topic_id"].map(
    lambda x: f"T{x:02d}"
)

robust["Effect (pp)"] = robust["effect"].map(
    format_effect
)

robust["95% HAC CI (pp)"] = robust.apply(
    format_ci, axis=1
)

robust["Holm-adjusted p"] = robust[
    "adjusted_p"
].map(format_p)

robust["Block significance"] = robust[
    "Holm significant"
]

robust["Evidence status"] = robust[
    "evidence_class"
]

publication_summary_df = robust[
    [
        "Comparison",
        "Topic",
        "topic_label",
        "Effect (pp)",
        "95% HAC CI (pp)",
        "Holm-adjusted p",
        "Block significance",
        "Evidence status"
    ]
].rename(
    columns={"topic_label": "Topic label"}
).sort_values(
    ["Comparison", "Topic"]
).reset_index(drop=True)

print("=" * 68)
print("PUBLICATION STATISTICAL SUMMARY")
print("=" * 68)

display(publication_summary_df)

# ------------------------------------------------------------
# 3. Evidence hierarchy
# ------------------------------------------------------------

categories = [
    "Fully block-stable",
    "Partially block-sensitive",
    "Primary significant; model-sensitive",
    "Not significant"
]

evidence_hierarchy_df = pd.DataFrame({
    "Evidence category": categories,
    "Number of contrasts": [
        int((evidence["evidence_class"] == c).sum())
        for c in categories
    ]
})

print("\n" + "=" * 68)
print("EVIDENCE HIERARCHY")
print("=" * 68)

display(evidence_hierarchy_df)

# ------------------------------------------------------------
# 4. Generate manuscript Results text
# ------------------------------------------------------------

def describe_contrast(row):
    pair = pair_names[row["pair"]]
    topic = f"T{int(row['topic_id']):02d}"
    label = row["topic_label"]

    return (
        f"{pair}, {topic} ({label}): "
        f"{format_effect(row['effect'])} percentage points "
        f"(95% pointwise HAC CI "
        f"{format_ci(row)}; "
        f"Holm-adjusted p {format_p(row['adjusted_p'])})"
    )

stable = robust[
    robust["evidence_class"] == "Fully block-stable"
].sort_values(
    ["pair", "topic_id"]
)

partial = robust[
    robust["evidence_class"] ==
    "Partially block-sensitive"
]

stable_descriptions = [
    describe_contrast(row)
    for _, row in stable.iterrows()
]

partial_row = partial.iloc[0]

assert partial_row["pair"] == "IRN-USA"
assert partial_row["topic_id"] == 7
assert partial_row["Holm significant"] == "48/50"

paragraph_1 = (
    "The balanced historical cohort comprised 162 "
    "UN General Debate speeches from Iran, the "
    "United States, and China (54 annual observations "
    "per country, 1972–2025). Across six eligible "
    "NMF topics and three bilateral country comparisons, "
    "18 early-to-late contrasts were evaluated using "
    "Newey–West HAC inference. Ten contrasts were "
    "statistically significant following Holm adjustment "
    "across the 18 primary tests. Five findings retained "
    "adjusted significance across the previously examined "
    "six NMF model solutions and three HAC bandwidths."
)

paragraph_2 = (
    "Four of these five findings remained "
    "Holm-significant across all 50 overlapping "
    "five-year block-deletion windows. The estimated "
    "early-to-late changes in signed bilateral "
    "topic-share differences were: "
    + "; ".join(stable_descriptions)
    + "."
)

paragraph_3 = (
    "The remaining cross-specification robust finding "
    "was "
    + describe_contrast(partial_row)
    + ". Its estimated direction was preserved in "
    "all 50 moving-block deletion windows, but "
    "Holm-adjusted significance was retained in "
    "48 of 50 windows. Significance was lost when "
    "the deleted windows covered 1976–1980 and "
    "1977–1981. Five other primary-significant "
    "contrasts did not meet the cross-specification "
    "significance criterion, whereas eight contrasts "
    "were not significant in the primary analysis."
)

limitations = (
    "Statistical reporting note: Confidence intervals "
    "are pointwise 95% HAC intervals and are not "
    "multiplicity-adjusted; the reported p-values "
    "are Holm-adjusted across 18 primary contrasts. "
    "The overlapping block deletions constitute a "
    "sensitivity analysis rather than independent "
    "replications. Topic labels remain provisional "
    "pending qualitative semantic validation. "
    "The estimated contrasts are descriptive "
    "historical associations, not causal effects."
)

manuscript_results_text = "\n\n".join([
    paragraph_1,
    paragraph_2,
    paragraph_3
])

print("\n" + "=" * 68)
print("ENGLISH MANUSCRIPT RESULTS")
print("=" * 68)

display(Markdown(
    "### Statistical Results — Draft\n\n"
    + manuscript_results_text
    + "\n\n**Reporting note.** "
    + limitations
))

# ------------------------------------------------------------
# 5. Final validation
# ------------------------------------------------------------

assert evidence_hierarchy_df[
    "Number of contrasts"
].tolist() == [4, 1, 5, 8]

assert len(publication_summary_df) == 5
assert "1976–1980" in paragraph_3
assert "1977–1981" in paragraph_3

print("\n" + "=" * 68)
print("RESULTS REPORTING VALIDATION")
print("=" * 68)

print("Primary contrasts validated: 18")
print("Holm-significant contrasts: 10")
print("Cross-specification robust findings: 5")
print("Fully block-stable findings: 4")
print("Partially block-sensitive findings: 1")
print("English Results generation: PASSED")
print("Publication summary table: PASSED")
print("Files saved: NONE")
print("Scientific interpretation: PENDING REVIEW")



### Step 12.26 — Publication Tables and Statistical Results Export

**Objective:** Export the validated statistical findings into reproducible, manuscript-ready tables and a Results draft.

This step produces:

1. **Main Table 1:** Five cross-specification robust historical findings, including effect estimates, pointwise 95% HAC confidence intervals, Holm-adjusted p-values, and moving-block significance stability.
2. **Supplementary Table S2:** All 18 primary bilateral topic comparisons, including non-significant and model-sensitive results.
3. **Machine-readable evidence:** Unrounded statistical estimates for independent verification.
4. **Statistical Results draft:** The validated English Results text with appropriate reporting notes.

The main analysis covers 162 speeches from three countries (1972–2025). The two historical periods are 1972–1990 and 2011–2025. Statistical inference uses Newey–West HAC covariance estimation, with Holm adjustment across 18 primary comparisons.

**Important:** Statistical significance, stability across model specifications, and stability under historical block deletion are distinct evidentiary criteria. The exported tables must preserve these distinctions.

Topic names remain provisional pending semantic validation. No additional statistical tests or figures are introduced in this step.


In [ ]:

# ============================================================
# STEP 12.26A — PUBLICATION TABLES AND RESULTS EXPORT
# ============================================================

# ------------------------------------------------------------
# 1. Validate source objects
# ------------------------------------------------------------

assert "final_evidence_audit_df" in globals()
assert "manuscript_results_text" in globals()
assert "limitations" in globals()

source = final_evidence_audit_df.copy()

required = {
    "pair", "topic_id", "topic_label",
    "effect", "ci_low", "ci_high",
    "adjusted_p", "evidence_class",
    "primary_significant", "Holm significant"
}

assert required.issubset(source.columns), (
    f"Missing columns: {required - set(source.columns)}"
)

numeric_cols = [
    "effect", "ci_low", "ci_high", "adjusted_p"
]

for col in numeric_cols:
    source[col] = pd.to_numeric(
        source[col], errors="raise"
    )

assert len(source) == 18
assert not source.duplicated(["pair", "topic_id"]).any()
assert np.isfinite(source[numeric_cols].to_numpy()).all()
assert (source["ci_low"] < source["ci_high"]).all()
assert source["adjusted_p"].between(0, 1).all()

assert int(source["primary_significant"].sum()) == 10

categories = {
    "Fully block-stable": 4,
    "Partially block-sensitive": 1,
    "Primary significant; model-sensitive": 5,
    "Not significant": 8
}

for category, expected in categories.items():
    actual = int(
        source["evidence_class"].eq(category).sum()
    )
    assert actual == expected, (
        f"{category}: expected {expected}, got {actual}"
    )

# ------------------------------------------------------------
# 2. Labels and publication formatting
# ------------------------------------------------------------

pair_map = {
    "IRN-CHN": "Iran–China",
    "IRN-USA": "Iran–United States",
    "USA-CHN": "United States–China"
}

pair_order = {
    "IRN-CHN": 0,
    "IRN-USA": 1,
    "USA-CHN": 2
}

assert source["pair"].isin(pair_map).all()

source["_pair_order"] = source["pair"].map(pair_order)

source = source.sort_values(
    ["_pair_order", "topic_id"]
).reset_index(drop=True)

def publication_p(p):
    p = float(p)
    return "<0.0001" if p < 0.0001 else f"{p:.3g}"

def publication_ci(lower, upper):
    return f"[{lower:.2f}, {upper:.2f}]"

def publication_effect(value):
    return f"{value:+.2f}"

def topic_code(topic_id):
    return f"T{int(topic_id):02d}"

# ------------------------------------------------------------
# 3. Main Table 1 — robust findings
# ------------------------------------------------------------

robust_classes = [
    "Fully block-stable",
    "Partially block-sensitive"
]

main_source = source[
    source["evidence_class"].isin(robust_classes)
].copy()

assert len(main_source) == 5
assert main_source["primary_significant"].all()

main_table_1 = pd.DataFrame({
    "Country comparison":
        main_source["pair"].map(pair_map),

    "Topic":
        main_source["topic_id"].map(topic_code),

    "Topic label":
        main_source["topic_label"],

    "Effect (pp)":
        main_source["effect"].map(publication_effect),

    "95% HAC CI (pp)":
        [
            publication_ci(lo, hi)
            for lo, hi in zip(
                main_source["ci_low"],
                main_source["ci_high"]
            )
        ],

    "Holm-adjusted p":
        main_source["adjusted_p"].map(publication_p),

    "Significant deletion windows":
        main_source["Holm significant"],

    "Evidence classification":
        main_source["evidence_class"]
})

main_table_1 = main_table_1.reset_index(drop=True)

print("=" * 65)
print("MAIN TABLE 1 — ROBUST HISTORICAL FINDINGS")
print("=" * 65)

display(main_table_1)

# ------------------------------------------------------------
# 4. Supplementary Table S2 — all 18 comparisons
# ------------------------------------------------------------

supplementary_table_s2 = pd.DataFrame({
    "Country comparison":
        source["pair"].map(pair_map),

    "Topic":
        source["topic_id"].map(topic_code),

    "Topic label":
        source["topic_label"],

    "Effect (pp)":
        source["effect"].map(publication_effect),

    "95% HAC CI (pp)":
        [
            publication_ci(lo, hi)
            for lo, hi in zip(
                source["ci_low"],
                source["ci_high"]
            )
        ],

    "Holm-adjusted p":
        source["adjusted_p"].map(publication_p),

    "Primary significant":
        source["primary_significant"].map({
            True: "Yes",
            False: "No"
        }),

    "Evidence classification":
        source["evidence_class"]
})

assert len(supplementary_table_s2) == 18

print("\n" + "=" * 65)
print("SUPPLEMENTARY TABLE S2 — ALL PRIMARY CONTRASTS")
print("=" * 65)

display(supplementary_table_s2)

# ------------------------------------------------------------
# 5. Machine-readable numerical results
# ------------------------------------------------------------

machine_readable_results = source[
    [
        "pair", "topic_id", "topic_label",
        "effect", "ci_low", "ci_high",
        "adjusted_p", "primary_significant",
        "evidence_class", "Holm significant"
    ]
].copy()

assert len(machine_readable_results) == 18

# ------------------------------------------------------------
# 6. Statistical reporting notes
# ------------------------------------------------------------

table_notes = (
    "TABLE REPORTING NOTES\n\n"
    "Effects represent changes in signed between-country "
    "topic-share differences from 1972–1990 to 2011–2025, "
    "expressed in percentage points.\n\n"
    "Confidence intervals are pointwise 95% Newey–West HAC "
    "confidence intervals and are not multiplicity-adjusted.\n\n"
    "P-values are Holm-adjusted across all 18 primary "
    "bilateral topic comparisons.\n\n"
    "Significant deletion windows indicate the number of "
    "overlapping five-year deletion windows in which "
    "Holm-adjusted significance was retained, out of 50.\n\n"
    "Fully block-stable findings retain significance in "
    "all 50 windows. The partially block-sensitive finding "
    "retains significance in 48 of 50 windows.\n\n"
    "Six NMF model solutions and three HAC bandwidths "
    "were evaluated in the cross-specification robustness "
    "analysis.\n\n"
    "Topic names are provisional pending qualitative "
    "semantic validation. Signed changes should not "
    "automatically be interpreted as convergence, "
    "divergence, or causal effects.\n"
)

results_document = (
    "STATISTICAL RESULTS — DRAFT\n\n"
    + manuscript_results_text
    + "\n\nREPORTING NOTES\n\n"
    + limitations
    + "\n"
)

# ------------------------------------------------------------
# 7. Export to notebook publication directory
# ------------------------------------------------------------

output_dir = (
    Path.cwd() / "publication_outputs" / "tables"
)

output_dir.mkdir(
    parents=True,
    exist_ok=True
)

paths = {
    "Main_Table_1": (
        output_dir /
        "Main_Table_1_Robust_Findings.csv"
    ),
    "Supplementary_Table_S2": (
        output_dir /
        "Supplementary_Table_S2_All_Contrasts.csv"
    ),
    "Machine_Readable_Results": (
        output_dir /
        "Primary_Results_Unrounded.csv"
    ),
    "Results_Draft": (
        output_dir /
        "Statistical_Results_Draft.txt"
    ),
    "Table_Notes": (
        output_dir /
        "Statistical_Table_Notes.txt"
    )
}

main_table_1.to_csv(
    paths["Main_Table_1"],
    index=False,
    encoding="utf-8-sig"
)

supplementary_table_s2.to_csv(
    paths["Supplementary_Table_S2"],
    index=False,
    encoding="utf-8-sig"
)

machine_readable_results.to_csv(
    paths["Machine_Readable_Results"],
    index=False,
    encoding="utf-8"
)

paths["Results_Draft"].write_text(
    results_document,
    encoding="utf-8"
)

paths["Table_Notes"].write_text(
    table_notes,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 8. Export integrity validation
# ------------------------------------------------------------

reloaded_main = pd.read_csv(
    paths["Main_Table_1"]
)

reloaded_supp = pd.read_csv(
    paths["Supplementary_Table_S2"]
)

reloaded_numeric = pd.read_csv(
    paths["Machine_Readable_Results"]
)

assert len(reloaded_main) == 5
assert len(reloaded_supp) == 18
assert len(reloaded_numeric) == 18

for col in numeric_cols:
    np.testing.assert_allclose(
        reloaded_numeric[col].to_numpy(),
        machine_readable_results[col].to_numpy(),
        rtol=1e-12,
        atol=1e-12
    )

assert all(
    path.exists() and path.stat().st_size > 0
    for path in paths.values()
)

print("\n" + "=" * 65)
print("PUBLICATION EXPORT VALIDATION")
print("=" * 65)

print("Main table rows:", len(main_table_1))
print("Supplementary rows:", len(supplementary_table_s2))
print("Unrounded numerical results:", len(reloaded_numeric))
print("Evidence categories:", categories)
print("Numeric round-trip integrity: PASSED")
print("Publication table export: PASSED")

print("\nSaved files:")
for name, path in paths.items():
    print(f" - {name}: {path}")

print("\nSTEP 12.26A: COMPLETED")
print("Semantic interpretation: PENDING REVIEW")



### Step 12.27A — Stratified Semantic Validation: Evidence Sampling

**Objective:** Construct a reproducible, historically stratified evidence set for qualitative validation of the provisional NMF topic labels.

**Priority topics:**
- T03 — Kampuchea and regional conflicts
- T05 — Human rights and freedom discourse
- T07 — Nuclear arms and disarmament

**Sampling design**

For each topic, country (Iran, United States, China), and historical period (1972–1990, 1991–2010, 2011–2025), select:

1. One speech with the highest reference-model topic contribution.
2. One independently selected comparison speech from the remaining country-period observations.

Extract one randomly selected original sentence-aware segment from each sampled speech. Preserve the source document, year, segment identifier, topic contribution, and complete extracted segment text.

The expected audit contains 54 topic-specific review records. Sampling uses a fixed random seed for reproducibility.

**Interpretation safeguards**

High topic contribution is a sampling criterion, not evidence that a proposed label is semantically correct. Random comparison excerpts provide contextual checks, not a formal negative control. Reviewers must assess the actual meanings expressed in the excerpts, including possible historical shifts in topic meaning.

Automated extraction and integrity validation do not constitute human semantic validation. Topic labels remain provisional until qualitative review is completed.

**Output:** A review dataset, a reviewer coding template, and an auditable sampling key. No new inferential tests or publication figures are generated.


In [ ]:

# ============================================================
# STEP 12.27A — STRATIFIED SEMANTIC EVIDENCE SAMPLING
# ANALYSIS AND VALIDATION ONLY — NO FILE EXPORT
# ============================================================


RANDOM_SEED = 42
rng = np.random.default_rng(RANDOM_SEED)

PRIORITY_TOPICS = {
    3: "Kampuchea and regional conflicts",
    5: "Human rights and freedom discourse",
    7: "Nuclear arms and disarmament"
}

COUNTRIES = ["IRN", "USA", "CHN"]

PERIODS = [
    ("1972-1990", 1972, 1990),
    ("1991-2010", 1991, 2010),
    ("2011-2025", 2011, 2025)
]

# ------------------------------------------------------------
# 1. Validate reference NMF and historical metadata
# ------------------------------------------------------------

for name in ["selected_result", "focal_df", "segments_df"]:
    assert name in globals(), f"Missing object: {name}"

assert isinstance(selected_result, dict)
assert "speech_shares" in selected_result

speech_shares = np.asarray(
    selected_result["speech_shares"],
    dtype=float
)

metadata = focal_df.reset_index(drop=True).copy()

assert speech_shares.shape == (162, 7)
assert len(metadata) == 162
assert np.isfinite(speech_shares).all()
assert (speech_shares >= -1e-10).all()

np.testing.assert_allclose(
    speech_shares.sum(axis=1),
    np.ones(162),
    atol=1e-6
)

metadata["speech_id"] = np.arange(len(metadata))

assert {
    "country_code", "year", "archive_path"
}.issubset(metadata.columns)

assert set(metadata["country_code"]) == set(COUNTRIES)

metadata["year"] = pd.to_numeric(
    metadata["year"],
    errors="raise"
).astype(int)

assert metadata["year"].between(1972, 2025).all()

coverage = metadata.groupby("country_code").agg(
    n_speeches=("year", "size"),
    n_unique_years=("year", "nunique"),
    first_year=("year", "min"),
    last_year=("year", "max")
)

assert (coverage["n_speeches"] == 54).all()
assert (coverage["n_unique_years"] == 54).all()

print("=" * 65)
print("SEMANTIC VALIDATION — REFERENCE COHORT")
print("=" * 65)

display(coverage)

# ------------------------------------------------------------
# 2. Validate speech-to-segment alignment
# ------------------------------------------------------------

segments = segments_df.copy()

required_columns = {
    "speech_id", "segment_id",
    "country_code", "year"
}

assert required_columns.issubset(segments.columns)

segments["speech_id"] = pd.to_numeric(
    segments["speech_id"], errors="raise"
).astype(int)

segments["year"] = pd.to_numeric(
    segments["year"], errors="raise"
).astype(int)

assert len(segments) == 2344
assert segments["speech_id"].between(0, 161).all()

segment_metadata = (
    segments.groupby("speech_id")[
        ["country_code", "year"]
    ]
    .first()
    .reindex(range(162))
)

assert segment_metadata.notna().all().all()

assert (
    segment_metadata["country_code"].to_numpy()
    == metadata["country_code"].to_numpy()
).all(), "Country alignment failed."

assert (
    segment_metadata["year"].to_numpy()
    == metadata["year"].to_numpy()
).all(), "Year alignment failed."

# ------------------------------------------------------------
# 3. Identify original segment text
# ------------------------------------------------------------

preferred_text_columns = [
    "segment_text",
    "text",
    "raw_text",
    "segment_content",
    "content",
    "text_segment"
]

text_column = next(
    (
        col for col in preferred_text_columns
        if col in segments.columns
        and segments[col].notna().any()
    ),
    None
)

if text_column is None:

    candidates = []

    for col in segments.select_dtypes(
        include=["object", "string"]
    ).columns:

        if any(
            word in col.lower()
            for word in [
                "path", "file", "country",
                "preview", "id"
            ]
        ):
            continue

        lengths = (
            segments[col]
            .dropna()
            .astype(str)
            .str.len()
        )

        if len(lengths) and lengths.median() > 150:
            candidates.append(
                (col, float(lengths.median()))
            )

    if candidates:
        text_column = max(
            candidates,
            key=lambda item: item[1]
        )[0]

if text_column is None:
    raise ValueError(
        "Original segment text column not identified."
    )

assert segments[text_column].notna().all()

print("\nText source:", text_column)
print("Speech-to-segment alignment: PASSED")

# ------------------------------------------------------------
# 4. Construct reference speech-level topic shares
# ------------------------------------------------------------

reference_speeches = metadata[
    [
        "speech_id",
        "country_code",
        "year",
        "archive_path"
    ]
].copy()

for topic_id in range(1, 8):
    reference_speeches[f"T{topic_id:02d}"] = (
        speech_shares[:, topic_id - 1]
    )

# ------------------------------------------------------------
# 5. Stratified evidence sampling
# ------------------------------------------------------------

records = []

for topic_id, proposed_label in PRIORITY_TOPICS.items():

    topic_column = f"T{topic_id:02d}"

    for country in COUNTRIES:

        for period_name, start_year, end_year in PERIODS:

            group = reference_speeches.loc[
                (reference_speeches["country_code"] == country)
                & reference_speeches["year"].between(
                    start_year, end_year
                )
            ].copy()

            assert len(group) >= 2

            ranked = group.sort_values(
                [topic_column, "year"],
                ascending=[False, True]
            )

            top = ranked.iloc[0]

            remaining = group.loc[
                group["speech_id"] != top["speech_id"]
            ]

            random_row = remaining.iloc[
                int(rng.integers(0, len(remaining)))
            ]

            selections = [
                ("Highest topic contribution", top),
                ("Random comparison", random_row)
            ]

            for selection_type, speech in selections:

                speech_id = int(speech["speech_id"])

                available_segments = segments.loc[
                    segments["speech_id"] == speech_id
                ].sort_values("segment_id")

                assert len(available_segments) > 0

                chosen_idx = int(
                    rng.integers(
                        0, len(available_segments)
                    )
                )

                chosen = available_segments.iloc[
                    chosen_idx
                ]

                original_excerpt = str(
                    chosen[text_column]
                )

                records.append({
                    "topic_id": topic_id,
                    "provisional_topic_label": proposed_label,
                    "country_code": country,
                    "period": period_name,
                    "selection_type": selection_type,
                    "year": int(speech["year"]),
                    "speech_id": speech_id,
                    "speech_topic_share": float(
                        speech[topic_column]
                    ),
                    "segment_id": chosen["segment_id"],
                    "archive_path": speech["archive_path"],
                    "segment_word_count": len(
                        original_excerpt.split()
                    ),
                    "segment_text": original_excerpt
                })

semantic_evidence_df = pd.DataFrame(records)

assert len(semantic_evidence_df) == 54

assert not semantic_evidence_df[
    [
        "topic_id",
        "country_code",
        "period",
        "selection_type"
    ]
].duplicated().any()

semantic_evidence_df.insert(
    0,
    "review_id",
    [
        f"SV-{i:03d}"
        for i in range(1, len(semantic_evidence_df) + 1)
    ]
)

# ------------------------------------------------------------
# 6. Sampling diagnostics
# ------------------------------------------------------------

sampling_summary = (
    semantic_evidence_df
    .groupby(
        ["topic_id", "country_code", "period"]
    )
    .agg(
        selected_records=("review_id", "size"),
        maximum_sample_share=(
            "speech_topic_share", "max"
        ),
        minimum_sample_share=(
            "speech_topic_share", "min"
        )
    )
    .reset_index()
)

assert (
    sampling_summary["selected_records"] == 2
).all()

print("\n" + "=" * 65)
print("SEMANTIC VALIDATION — SAMPLING AUDIT")
print("=" * 65)

display(sampling_summary.round(4))

print("\nTotal review records:",
      len(semantic_evidence_df))

print("Unique source speeches:",
      semantic_evidence_df["speech_id"].nunique())

# ------------------------------------------------------------
# 7. Prepare in-memory human-review template
# ------------------------------------------------------------

semantic_review_template = semantic_evidence_df[
    [
        "review_id",
        "topic_id",
        "country_code",
        "period",
        "year",
        "speech_id",
        "segment_id",
        "archive_path",
        "segment_text"
    ]
].copy()

review_fields = [
    "Observed semantic theme",
    "Provisional label fits (Yes/Partial/No)",
    "Context sufficient (Yes/No)",
    "Supporting quotation",
    "Alternative topic label",
    "Historical interpretation notes",
    "Reviewer ID"
]

for col in review_fields:
    semantic_review_template[col] = ""

# ------------------------------------------------------------
# 8. Output inspection
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("EXAMPLE REVIEW RECORDS")
print("=" * 65)

preview = semantic_evidence_df[
    [
        "review_id",
        "topic_id",
        "country_code",
        "period",
        "selection_type",
        "year",
        "speech_topic_share",
        "segment_word_count"
    ]
].copy()

display(preview.head(18).round(4))

# ------------------------------------------------------------
# 9. Final validation — no file writing
# ------------------------------------------------------------

assert len(semantic_review_template) == 54
assert semantic_evidence_df["segment_text"].notna().all()

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Reference NMF alignment: PASSED")
print("Original text retrieval: PASSED")
print("Stratified sampling: PASSED")
print("Review records:", len(semantic_evidence_df))
print("Human semantic validation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27A: COMPLETED")


In [ ]:

# # ============================================================
# # STEP 12.27A — INDEPENDENT EXPORT CELL
# # ============================================================


# required_objects = [
#     "semantic_evidence_df",
#     "semantic_review_template",
#     "sampling_summary"
# ]

# for name in required_objects:
#     assert name in globals(), f"Missing object: {name}"

# output_dir = (
#     Path.cwd()
#     / "publication_outputs"
#     / "semantic_validation"
# )

# output_dir.mkdir(parents=True, exist_ok=True)

# export_files = {
#     "Semantic_Evidence_Packet.csv": semantic_evidence_df,
#     "Human_Semantic_Review_Template.csv": semantic_review_template,
#     "Semantic_Sampling_Audit.csv": sampling_summary
# }

# print("=" * 65)
# print("SEMANTIC VALIDATION — INDEPENDENT EXPORT")
# print("=" * 65)

# for filename, dataframe in export_files.items():

#     filepath = output_dir / filename

#     dataframe.to_csv(
#         filepath,
#         index=False,
#         encoding="utf-8-sig"
#     )

#     assert filepath.exists()
#     assert filepath.stat().st_size > 0

#     saved_rows = len(pd.read_csv(filepath))

#     assert saved_rows == len(dataframe)

#     print(f"\nSAVED: {filename}")
#     print(f"Rows: {saved_rows}")
#     print(f"Path: {filepath}")

# print("\nEXPORT VALIDATION: PASSED")
# print("FILES SAVED:", len(export_files))
# print("STEP 12.27A EXPORT: COMPLETED")



### Step 12.27B — Qualitative Semantic Inspection of Topic 03

**Objective:** Evaluate the substantive meaning and historical consistency of Topic 03, provisionally labeled "Kampuchea and regional conflicts."

The analysis examines six original text excerpts from Chinese diplomatic speeches, stratified across three historical periods (1972–1990, 1991–2010, and 2011–2025).

Two excerpts are evaluated per period: one from the speech with the highest Topic 03 contribution and one from a randomly selected comparison speech.

The review considers thematic relevance, historical context, consistency of topic interpretation, and possible alternative topic labels.

**Methodological limitation:** The extracted segments are not necessarily the segments with the highest topic contribution within each speech. Keyword matches serve only as descriptive diagnostics.

**Status:** Human semantic validation pending.


In [ ]:

# ============================================================
# STEP 12.27B — HISTORICAL SEMANTIC INSPECTION
# TOPIC 03: CHINESE DIPLOMATIC DISCOURSE
# ANALYSIS ONLY — NO FILE EXPORT
# ============================================================



# ------------------------------------------------------------
# 1. Validate source objects
# ------------------------------------------------------------

assert "semantic_evidence_df" in globals(), (
    "Run Step 12.27A first."
)

evidence = semantic_evidence_df.copy()

required_columns = {
    "review_id",
    "topic_id",
    "country_code",
    "period",
    "selection_type",
    "year",
    "speech_id",
    "speech_topic_share",
    "segment_id",
    "archive_path",
    "segment_text"
}

assert required_columns.issubset(evidence.columns)

# ------------------------------------------------------------
# 2. Select Chinese Topic 03 evidence
# ------------------------------------------------------------

TARGET_TOPIC = 3
TARGET_COUNTRY = "CHN"

PERIOD_ORDER = [
    "1972-1990",
    "1991-2010",
    "2011-2025"
]

topic03_china = evidence.loc[
    (evidence["topic_id"] == TARGET_TOPIC)
    & (evidence["country_code"] == TARGET_COUNTRY)
].copy()

topic03_china["period"] = pd.Categorical(
    topic03_china["period"],
    categories=PERIOD_ORDER,
    ordered=True
)

topic03_china = (
    topic03_china
    .sort_values(["period", "selection_type", "year"])
    .reset_index(drop=True)
)

assert len(topic03_china) == 6

assert (
    topic03_china.groupby(
        "period", observed=True
    ).size() == 2
).all()

assert (
    topic03_china["segment_text"]
    .fillna("")
    .astype(str)
    .str.strip()
    .str.len() > 0
).all()

# ------------------------------------------------------------
# 3. Historical sampling inventory
# ------------------------------------------------------------

print("=" * 70)
print("TOPIC 03 — CHINESE HISTORICAL SEMANTIC VALIDATION")
print("=" * 70)

print("\nProvisional topic label:")
print("Kampuchea and regional conflicts")

print("\nNumber of records:", len(topic03_china))
print("Historical periods:", len(PERIOD_ORDER))

inventory_columns = [
    "review_id",
    "period",
    "selection_type",
    "year",
    "speech_topic_share",
    "segment_word_count"
]

display(
    topic03_china[inventory_columns].round(4)
)

# ------------------------------------------------------------
# 4. Print complete original text excerpts
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("ORIGINAL SOURCE EXCERPTS")
print("=" * 70)

for period in PERIOD_ORDER:

    period_records = topic03_china.loc[
        topic03_china["period"] == period
    ]

    print("\n" + "#" * 70)
    print("HISTORICAL PERIOD:", period)
    print("#" * 70)

    for _, record in period_records.iterrows():

        print("\n" + "-" * 70)

        print("Review ID:", record["review_id"])
        print("Year:", int(record["year"]))
        print("Selection:", record["selection_type"])

        print(
            "Speech-level Topic 03 share:",
            f'{float(record["speech_topic_share"]):.4f}'
        )

        print("Speech ID:", record["speech_id"])
        print("Segment ID:", record["segment_id"])
        print("Source:", record["archive_path"])

        text = str(record["segment_text"])

        print("Segment word count:", len(text.split()))

        print("\nCOMPLETE SEGMENT TEXT:\n")
        print(text)

        print("\n" + "-" * 70)

# ------------------------------------------------------------
# 5. Prepare a qualitative review matrix
# ------------------------------------------------------------

semantic_review_27b = topic03_china[
    [
        "review_id",
        "period",
        "selection_type",
        "year",
        "speech_topic_share",
        "archive_path",
        "segment_text"
    ]
].copy()

semantic_review_27b["observed_theme"] = ""
semantic_review_27b["regional_conflict_relevance"] = ""
semantic_review_27b["historical_context_fit"] = ""
semantic_review_27b["provisional_label_fit"] = ""
semantic_review_27b["alternative_topic_label"] = ""
semantic_review_27b["supporting_quotation"] = ""
semantic_review_27b["reviewer_notes"] = ""
semantic_review_27b["reviewer_id"] = ""

# ------------------------------------------------------------
# 6. Review coverage diagnostics
# ------------------------------------------------------------

review_coverage = (
    topic03_china
    .groupby("period", observed=True)
    .agg(
        n_records=("review_id", "size"),
        n_unique_speeches=("speech_id", "nunique"),
        min_year=("year", "min"),
        max_year=("year", "max"),
        mean_topic_share=("speech_topic_share", "mean")
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("HISTORICAL COVERAGE DIAGNOSTICS")
print("=" * 70)

display(review_coverage.round(4))

# ------------------------------------------------------------
# 7. Human-review readiness checks
# ------------------------------------------------------------

assert len(semantic_review_27b) == 6

assert (
    semantic_review_27b["review_id"].nunique() == 6
)

assert set(
    topic03_china["period"].astype(str)
) == set(PERIOD_ORDER)

assert set(
    topic03_china["selection_type"]
) == {
    "Highest topic contribution",
    "Random comparison"
}

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Reference evidence retrieval: PASSED")
print("Historical stratification: PASSED")
print("Complete text extraction: PASSED")
print("Review template preparation: PASSED")
print("Human semantic validation: PENDING")

print("\nFiles saved: NONE")
print("STEP 12.27B: COMPUTATION COMPLETED")



### Step 12.27C — Targeted Segment-Level Semantic Validation

**Objective:** Evaluate the historical and substantive validity of Topic 03 using segment-level NMF topic contributions rather than randomly selected segments.

**Methodology**

1. Retrieve the original segment-topic weight matrix from the reference NMF model (K = 7, seed = 11).
2. Identify the highest-contributing Topic 03 segment within each of the six previously selected Chinese speeches.
3. Independently retrieve two high-contribution segments per historical period from other Chinese speeches.
4. Examine topic dominance, relative contribution, original source text, and historical context.
5. Construct a structured human-review dataset without automatically assigning semantic validity.

**Historical periods:** 1972–1990, 1991–2010, and 2011–2025.

**Important:** High NMF contribution is not equivalent to validated semantic relevance. The evidence will be assessed qualitatively before confirming or revising the topic label.

**Output:** Segment-level diagnostics, twelve candidate evidence records, and a qualitative review template.

**File exports:** Performed exclusively in a separate code cell.


In [ ]:

# ============================================================
# STEP 12.27C — REFERENCE SEGMENT MATRIX RECOVERY
# CORRECTED VERSION — ANALYSIS ONLY
# NO FILE SAVING
# ============================================================

print("=" * 68)
print("STEP 12.27C — CORRECTED REFERENCE MATRIX RECOVERY")
print("=" * 68)

# ------------------------------------------------------------
# 1. Validate required sources
# ------------------------------------------------------------

required = [
    "selected_result",
    "segments_df",
    "semantic_evidence_df",
    "focal_df"
]

for name in required:
    assert name in globals(), f"Missing object: {name}"

seg = segments_df.reset_index(drop=True).copy()

N = len(seg)
K = 7

assert N == 2344
assert "speech_id" in seg.columns

reference_shares = np.asarray(
    selected_result["speech_shares"],
    dtype=float
)

assert reference_shares.shape == (162, K)

model = selected_result["model"]

assert hasattr(model, "transform")
assert hasattr(model, "components_")
assert model.components_.shape[0] == K

n_features = model.components_.shape[1]

print("\nReference model:")
print("Number of topics:", K)
print("Segment count:", N)
print("Vocabulary size:", n_features)
print("Topic-term matrix:", selected_result["H"].shape)
print("Speech shares:", reference_shares.shape)

# ------------------------------------------------------------
# 2. Discover existing matrices
# ------------------------------------------------------------

def is_matrix(value):
    return (
        isinstance(value, np.ndarray)
        or sparse.issparse(value)
    ) and hasattr(value, "shape")

candidates_W = []
candidates_X = []

seen = set()

def inspect_object(name, value, depth=0):

    if id(value) in seen:
        return

    seen.add(id(value))

    if is_matrix(value):

        shape = tuple(value.shape)

        if shape == (N, K):
            candidates_W.append((name, value))

        elif shape == (N, n_features):
            candidates_X.append((name, value))

        return

    if isinstance(value, dict) and depth < 3:

        if len(value) > 100:
            return

        for key, item in value.items():

            inspect_object(
                f"{name}[{repr(key)}]",
                item,
                depth + 1
            )

    elif isinstance(value, (list, tuple)) and depth < 2:

        if len(value) > 30:
            return

        for idx, item in enumerate(value):

            inspect_object(
                f"{name}[{idx}]",
                item,
                depth + 1
            )

for name, value in list(globals().items()):

    if name.startswith("_"):
        continue

    if name in {
        "candidates_W",
        "candidates_X",
        "seen",
        "reference_shares"
    }:
        continue

    inspect_object(name, value)

print("\nExisting segment-topic candidates:")
for name, _ in candidates_W:
    print(" -", name)

print("\nExisting segment-feature candidates:")
for name, _ in candidates_X:
    print(" -", name)

# ------------------------------------------------------------
# 3. Validate candidates against reference speech shares
# ------------------------------------------------------------

speech_ids = seg["speech_id"].to_numpy(dtype=int)

assert speech_ids.min() >= 0
assert speech_ids.max() < 162

def evaluate_weights(W_candidate):

    if sparse.issparse(W_candidate):
        W_candidate = W_candidate.toarray()

    W_candidate = np.asarray(
        W_candidate,
        dtype=float
    )

    if W_candidate.shape != (N, K):
        return None

    if not np.isfinite(W_candidate).all():
        return None

    if (W_candidate < -1e-8).any():
        return None

    W_candidate = np.maximum(W_candidate, 0)

    # Aggregation A: sum weights before normalization
    speech_weights = np.zeros(
        (162, K),
        dtype=float
    )

    np.add.at(
        speech_weights,
        speech_ids,
        W_candidate
    )

    totals = speech_weights.sum(axis=1)

    if (totals <= 0).any():
        return None

    shares_a = speech_weights / totals[:, None]

    # Aggregation B: average normalized segment shares
    row_totals = W_candidate.sum(axis=1)

    segment_shares = np.divide(
        W_candidate,
        row_totals[:, None],
        out=np.zeros_like(W_candidate),
        where=row_totals[:, None] > 0
    )

    shares_b = np.zeros((162, K))

    np.add.at(
        shares_b,
        speech_ids,
        segment_shares
    )

    counts = np.bincount(
        speech_ids,
        minlength=162
    )

    shares_b /= counts[:, None]

    mae_a = np.mean(
        np.abs(shares_a - reference_shares)
    )

    mae_b = np.mean(
        np.abs(shares_b - reference_shares)
    )

    return {
        "weights": W_candidate,
        "mae": float(min(mae_a, mae_b)),
        "aggregation": (
            "normalized summed weights"
            if mae_a <= mae_b
            else "mean normalized segment weights"
        )
    }

evaluations = []

# Prefer exact cached segment-topic weights
for name, value in candidates_W:

    result = evaluate_weights(value)

    if result is not None:

        evaluations.append({
            "source": name,
            "method": "cached",
            **result
        })

evaluations.sort(key=lambda x: x["mae"])

# If no sufficiently matching cached matrix exists,
# reconstruct weights from the fitted model.
if (
    not evaluations
    or evaluations[0]["mae"] > 0.01
):

    print("\nTesting fitted-model projections...")

    for name, X_candidate in candidates_X:

        try:

            reconstructed_W = model.transform(
                X_candidate
            )

            result = evaluate_weights(
                reconstructed_W
            )

            if result is not None:

                evaluations.append({
                    "source": name,
                    "method": "model.transform",
                    **result
                })

                print(
                    f"{name}: "
                    f"reference MAE = {result['mae']:.6f}"
                )

        except (ValueError, TypeError) as error:

            print(
                f"Skipped {name}: "
                f"{str(error)[:120]}"
            )

evaluations.sort(key=lambda x: x["mae"])

# ------------------------------------------------------------
# 4. Source audit
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("REFERENCE MATCHING DIAGNOSTICS")
print("=" * 68)

if not evaluations:

    print("No valid matrix was identified.")

    print("\nAvailable matrix candidates:")
    print("Segment-topic:", len(candidates_W))
    print("Segment-feature:", len(candidates_X))

    raise RuntimeError(
        "The reference segment weights are not "
        "available from the inspected notebook objects. "
        "Review the diagnostics before continuing."
    )

audit = pd.DataFrame([
    {
        "source": item["source"],
        "method": item["method"],
        "reference_mae": item["mae"],
        "aggregation": item["aggregation"]
    }
    for item in evaluations
])

display(audit)

best = evaluations[0]

print("\nBest matching source:", best["source"])
print("Recovery method:", best["method"])
print("Reference MAE:", round(best["mae"], 8))
print("Aggregation:", best["aggregation"])

# Do not silently accept a different model
if best["mae"] > 0.01:

    print("\nREFERENCE VALIDATION: REVIEW REQUIRED")
    print("Files saved: NONE")

    raise RuntimeError(
        "Recovered segment weights do not adequately "
        "reproduce the reference speech shares. "
        "Inspect the candidate matrix audit."
    )

# ------------------------------------------------------------
# 5. Prepare validated reference weights
# ------------------------------------------------------------

W_reference_27c = best["weights"].copy()

row_totals = W_reference_27c.sum(axis=1)

shares_reference_27c = np.divide(
    W_reference_27c,
    row_totals[:, None],
    out=np.zeros_like(W_reference_27c),
    where=row_totals[:, None] > 0
)

seg["topic03_share"] = shares_reference_27c[:, 2]

seg["topic03_weight"] = W_reference_27c[:, 2]

seg["dominant_topic"] = np.where(
    row_totals > 0,
    np.argmax(W_reference_27c, axis=1) + 1,
    0
)

assert W_reference_27c.shape == (2344, 7)

print("\n" + "=" * 68)
print("TOPIC 03 — REFERENCE SEGMENT DIAGNOSTICS")
print("=" * 68)

print("Validated segment-topic matrix:",
      W_reference_27c.shape)

print(
    "Segments with Topic 03 dominant:",
    int((seg["dominant_topic"] == 3).sum())
)

print(
    "Segments with Topic 03 share >= 0.50:",
    int((seg["topic03_share"] >= 0.50).sum())
)

print(
    "Segments with Topic 03 share >= 0.20:",
    int((seg["topic03_share"] >= 0.20).sum())
)

print("\nTopic 03 distribution by country:")

display(
    seg.groupby("country_code")[
        "topic03_share"
    ].agg(
        ["count", "mean", "median", "max"]
    ).round(4)
)

# ------------------------------------------------------------
# 6. Final status
# ------------------------------------------------------------

segment_reference_27c = seg

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Reference model identification: PASSED")
print("Reference topic matrix recovery: PASSED")
print("Speech-share consistency: PASSED")
print("Segment-topic dimensions: PASSED")
print("Files saved: NONE")
print("\nSTEP 12.27C MATRIX RECOVERY: PASSED")



### Step 12.27D — Diachronic Semantic Validation of Topic 03

**Objective:** Examine whether Topic 03 consistently represents Kampuchea and regional conflicts across three historical periods (1972–1990, 1991–2010, and 2011–2025).

**Procedure:**
1. Retrieve the six previously sampled Chinese text segments from their original sources.
2. Link each segment to its corresponding Topic 03 weight in the recovered reference matrix.
3. Inspect the complete source text and identify its substantive geopolitical content.
4. Prepare a structured human-review template for assessing thematic consistency and possible semantic drift.

**Important:** Lexical keyword occurrences are descriptive aids, not evidence of semantic validity on their own. Final topic interpretation requires contextual review.

**Output:** Six source-verified segments, historical diagnostics, and an in-memory semantic review template. No files are saved in this step.


In [ ]:

# ============================================================
# STEP 12.27D — DIACHRONIC SEMANTIC VALIDATION
# TOPIC 03 — CHINESE HISTORICAL CONTEXTS
# ANALYSIS ONLY — NO FILE SAVING
# ============================================================

print("=" * 68)
print("TOPIC 03 — DIACHRONIC SEMANTIC VALIDATION")
print("=" * 68)

# ------------------------------------------------------------
# 1. Source validation
# ------------------------------------------------------------

required_objects = [
    "semantic_evidence_df",
    "segment_reference_27c",
    "W_reference_27c",
    "selected_result"
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Missing notebook objects: {missing_objects}"
    )

review = semantic_evidence_df.copy()
segments = segment_reference_27c.copy()

required_review_cols = [
    "topic_id", "country_code", "period",
    "year", "speech_id", "segment_id"
]

required_segment_cols = [
    "speech_id", "segment_id",
    "country_code", "year",
    "segment_text", "topic03_share"
]

for name, frame, required_cols in [
    ("semantic_evidence_df", review, required_review_cols),
    ("segment_reference_27c", segments, required_segment_cols)
]:
    missing = [
        col for col in required_cols
        if col not in frame.columns
    ]
    if missing:
        raise RuntimeError(
            f"{name} missing columns: {missing}. "
            f"Available: {list(frame.columns)}"
        )

assert np.asarray(W_reference_27c).shape == (2344, 7)

# ------------------------------------------------------------
# 2. Select six previously sampled Chinese records
# ------------------------------------------------------------

selected = review.loc[
    (pd.to_numeric(review["topic_id"]) == 3)
    & (review["country_code"] == "CHN")
].copy()

assert len(selected) == 6, (
    f"Expected 6 records; found {len(selected)}"
)

assert selected["period"].nunique() == 3

period_counts = selected.groupby("period").size()

assert (period_counts == 2).all()

for col in ["speech_id", "segment_id"]:
    selected[col] = pd.to_numeric(
        selected[col], errors="raise"
    ).astype(int)

# ------------------------------------------------------------
# 3. Join to exact source segment
# ------------------------------------------------------------

source_lookup = segments[
    [
        "speech_id",
        "segment_id",
        "country_code",
        "year",
        "segment_text",
        "topic03_share"
    ]
].copy()

source_lookup = source_lookup.rename(
    columns={
        "country_code": "verified_country",
        "year": "verified_year",
        "segment_text": "verified_segment_text",
        "topic03_share": "reference_segment_share"
    }
)

assert not source_lookup.duplicated(
    ["speech_id", "segment_id"]
).any(), "Duplicate segment identifiers detected."

matched = selected.merge(
    source_lookup,
    on=["speech_id", "segment_id"],
    how="left",
    validate="one_to_one",
    indicator=True
)

assert matched["_merge"].eq("both").all(), (
    "Some review segments could not be matched."
)

assert (
    matched["country_code"] ==
    matched["verified_country"]
).all(), "Country mismatch."

assert (
    pd.to_numeric(matched["year"]) ==
    pd.to_numeric(matched["verified_year"])
).all(), "Year mismatch."

assert matched["verified_segment_text"].notna().all()

assert matched["reference_segment_share"].between(
    0, 1
).all()

# ------------------------------------------------------------
# 4. Validate historical period mapping
# ------------------------------------------------------------

def historical_period(year):
    year = int(year)

    if 1972 <= year <= 1990:
        return "1972-1990"

    if 1991 <= year <= 2010:
        return "1991-2010"

    if 2011 <= year <= 2025:
        return "2011-2025"

    return "OUT_OF_RANGE"

matched["computed_period"] = (
    matched["year"].map(historical_period)
)

def normalized_period(value):
    return str(value).replace("–", "-").replace("—", "-")

assert (
    matched["computed_period"] ==
    matched["period"].map(normalized_period)
).all(), "Historical period classification mismatch."

# ------------------------------------------------------------
# 5. Descriptive lexical diagnostics
# ------------------------------------------------------------

keyword_patterns = {
    "kampuchea_cambodia":
        r"\b(?:kampuchea\w*|cambodia\w*)\b",

    "vietnam":
        r"\bviet(?:nam|namese)\b",

    "afghanistan":
        r"\bafghan\w*\b",

    "soviet":
        r"\bsoviet\w*\b",

    "middle_east":
        r"\bmiddle[\s-]+east\b",

    "palestine_israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "conflict_terms":
        r"\b(?:conflict\w*|war|wars|"
        r"aggress\w*|invasion\w*)\b",

    "settlement_terms":
        r"\b(?:settlement\w*|negotiat\w*|"
        r"resolution\w*)\b"
}

for label, pattern in keyword_patterns.items():

    matched[label] = matched[
        "verified_segment_text"
    ].apply(
        lambda text: len(
            re.findall(
                pattern,
                str(text),
                flags=re.IGNORECASE
            )
        )
    )

# ------------------------------------------------------------
# 6. Compact historical overview
# ------------------------------------------------------------

sort_cols = ["period", "year"]

matched = matched.sort_values(
    sort_cols
).reset_index(drop=True)

print("\nSOURCE-VERIFIED REVIEW RECORDS")

overview_cols = [
    "year",
    "period",
    "reference_segment_share",
    "kampuchea_cambodia",
    "vietnam",
    "afghanistan",
    "middle_east",
    "palestine_israel",
    "conflict_terms",
    "settlement_terms"
]

if "review_id" in matched.columns:
    overview_cols.insert(0, "review_id")

display(
    matched[overview_cols].round(4)
)

# ------------------------------------------------------------
# 7. Historical summary
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("HISTORICAL SAMPLE DIAGNOSTICS")
print("=" * 68)

period_summary = (
    matched.groupby("period", observed=True)
    .agg(
        n_segments=("segment_id", "size"),
        mean_segment_topic_share=(
            "reference_segment_share", "mean"
        ),
        kampuchea_mentions=(
            "kampuchea_cambodia", "sum"
        ),
        conflict_mentions=(
            "conflict_terms", "sum"
        ),
        settlement_mentions=(
            "settlement_terms", "sum"
        )
    )
    .reset_index()
)

display(period_summary.round(4))

# ------------------------------------------------------------
# 8. Complete source text for human assessment
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("COMPLETE SOURCE SEGMENTS")
print("=" * 68)

for _, row in matched.iterrows():

    print("\n" + "#" * 68)

    if "review_id" in row.index:
        print("Review ID:", row["review_id"])

    print("Period:", row["period"])
    print("Year:", row["year"])

    print(
        "Topic 03 segment share:",
        f"{row['reference_segment_share']:.4f}"
    )

    print("Speech ID:", row["speech_id"])
    print("Segment ID:", row["segment_id"])

    if "archive_path" in row.index:
        print("Source:", row["archive_path"])

    print("\nCOMPLETE SEGMENT TEXT:\n")

    clean_text = re.sub(
        r"\s+",
        " ",
        str(row["verified_segment_text"])
    ).strip()

    print(
        textwrap.fill(
            clean_text,
            width=105,
            break_long_words=False,
            break_on_hyphens=False
        )
    )

# ------------------------------------------------------------
# 9. Prepare structured human review template
# ------------------------------------------------------------

review_columns = [
    col for col in [
        "review_id",
        "country_code",
        "year",
        "period",
        "speech_id",
        "segment_id",
        "reference_segment_share"
    ]
    if col in matched.columns
]

semantic_review_template_27d = matched[
    review_columns
].copy()

semantic_review_template_27d[
    "primary_geographic_focus"
] = pd.NA

semantic_review_template_27d[
    "conflict_or_policy_context"
] = pd.NA

semantic_review_template_27d[
    "topic_relevance"
] = pd.NA

semantic_review_template_27d[
    "supports_kampuchea_label"
] = pd.NA

semantic_review_template_27d[
    "supporting_text"
] = pd.NA

semantic_review_template_27d[
    "reviewer_notes"
] = pd.NA

print("\n" + "=" * 68)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 68)

display(semantic_review_template_27d)

# ------------------------------------------------------------
# 10. Final computational validation
# ------------------------------------------------------------

semantic_china_27d = matched.copy()

assert len(semantic_china_27d) == 6
assert semantic_china_27d["period"].nunique() == 3
assert semantic_china_27d["verified_segment_text"].notna().all()

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Reference segment linkage: PASSED")
print("Country and year consistency: PASSED")
print("Historical coverage: PASSED")
print("Complete text retrieval: PASSED")
print("Human review template: PREPARED")
print("Human semantic validation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27D: COMPUTATION COMPLETED")



### Step 12.27E — Within-Speech Targeted Semantic Validation

**Objective:** Verify the historical interpretation of Topic 03 by locating the strongest topic-bearing segment within each of the six previously sampled Chinese speeches.

**Method:**
1. Preserve the original stratified sample of six speeches across three historical periods.
2. Identify the highest-scoring Topic 03 segment within each selected speech.
3. Compare the originally sampled segment with the targeted segment.
4. Examine changes in geographic references and conflict-related vocabulary.
5. Retrieve complete targeted segments for subsequent qualitative review.

**Methodological safeguards:**
- No source speeches are replaced.
- Segment selection is explicitly targeted, not random.
- Topic weights are based on the recovered segment-level matrix, whose aggregation is approximate rather than an exact reproduction of speech-level weights.
- Keyword counts do not determine semantic validity.
- Human interpretation remains pending.

**Output:** In-memory targeted sample, diagnostic tables, and semantic review template. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27E — WITHIN-SPEECH TARGETED SEMANTIC VALIDATION
# ANALYSIS ONLY — NO FILE SAVING
# ============================================================

print("=" * 68)
print("TOPIC 03 — TARGETED WITHIN-SPEECH VALIDATION")
print("=" * 68)

# ------------------------------------------------------------
# 1. Validate existing sources
# ------------------------------------------------------------

assert "semantic_china_27d" in globals()
assert "segment_reference_27c" in globals()

original = semantic_china_27d.copy()
segments = segment_reference_27c.copy()

required_original = [
    "review_id", "speech_id", "segment_id",
    "country_code", "year", "period",
    "reference_segment_share"
]

required_segments = [
    "speech_id", "segment_id", "country_code",
    "year", "segment_text", "topic03_share"
]

assert set(required_original).issubset(original.columns)
assert set(required_segments).issubset(segments.columns)

assert len(original) == 6
assert original["speech_id"].nunique() == 6
assert original["period"].nunique() == 3

# ------------------------------------------------------------
# 2. Normalize identifiers and topic weights
# ------------------------------------------------------------

for frame in [original, segments]:
    frame["speech_id"] = pd.to_numeric(
        frame["speech_id"], errors="raise"
    ).astype(int)

    frame["segment_id"] = pd.to_numeric(
        frame["segment_id"], errors="raise"
    ).astype(int)

    frame["year"] = pd.to_numeric(
        frame["year"], errors="raise"
    ).astype(int)

segments["topic03_share"] = pd.to_numeric(
    segments["topic03_share"], errors="raise"
)

assert segments["topic03_share"].between(0, 1).all()

assert not segments.duplicated(
    ["speech_id", "segment_id"]
).any()

# ------------------------------------------------------------
# 3. Find strongest Topic 03 segment in each original speech
# ------------------------------------------------------------

candidate_segments = segments.loc[
    (segments["country_code"] == "CHN")
    & (
        segments["speech_id"].isin(
            original["speech_id"]
        )
    )
].copy()

candidate_segments = candidate_segments.sort_values(
    ["speech_id", "topic03_share", "segment_id"],
    ascending=[True, False, True]
)

strongest = (
    candidate_segments
    .drop_duplicates("speech_id", keep="first")
    [
        [
            "speech_id", "segment_id", "year",
            "topic03_share", "segment_text"
        ]
    ]
    .rename(
        columns={
            "segment_id": "target_segment_id",
            "year": "target_year",
            "topic03_share": "target_topic_share",
            "segment_text": "target_text"
        }
    )
)

assert len(strongest) == 6

# ------------------------------------------------------------
# 4. Preserve original selections for comparison
# ------------------------------------------------------------

original = original.rename(
    columns={
        "segment_id": "original_segment_id",
        "reference_segment_share": "original_topic_share"
    }
)

targeted = original.merge(
    strongest,
    on="speech_id",
    how="left",
    validate="one_to_one"
)

assert targeted["target_text"].notna().all()

assert (
    targeted["year"] == targeted["target_year"]
).all()

assert (
    targeted["target_topic_share"].to_numpy()
    >= targeted["original_topic_share"].to_numpy() - 1e-10
).all()

targeted["share_increase"] = (
    targeted["target_topic_share"]
    - targeted["original_topic_share"]
)

targeted["segment_changed"] = (
    targeted["original_segment_id"]
    != targeted["target_segment_id"]
)

# ------------------------------------------------------------
# 5. Descriptive lexical evidence
# ------------------------------------------------------------

keyword_patterns = {
    "kampuchea_cambodia":
        r"\b(?:kampuchea\w*|cambodia\w*)\b",

    "afghanistan":
        r"\bafghan\w*\b",

    "vietnam":
        r"\bviet(?:nam|namese)\b",

    "middle_east":
        r"\bmiddle[\s-]+east\b",

    "palestine_israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "conflict_terms":
        r"\b(?:conflict\w*|war|wars|"
        r"aggress\w*|invasion\w*)\b",

    "settlement_terms":
        r"\b(?:settlement\w*|negotiat\w*|"
        r"resolution\w*)\b"
}

for label, pattern in keyword_patterns.items():
    targeted[label] = targeted["target_text"].apply(
        lambda text: len(
            re.findall(
                pattern,
                str(text),
                flags=re.IGNORECASE
            )
        )
    )

# ------------------------------------------------------------
# 6. Compare original and targeted segments
# ------------------------------------------------------------

period_order = [
    "1972-1990",
    "1991-2010",
    "2011-2025"
]

targeted["period"] = (
    targeted["period"]
    .astype(str)
    .str.replace("–", "-", regex=False)
)

targeted["period"] = pd.Categorical(
    targeted["period"],
    categories=period_order,
    ordered=True
)

targeted = targeted.sort_values(
    ["period", "year"]
).reset_index(drop=True)

print("\nORIGINAL VS TARGETED SEGMENT COMPARISON")

display(
    targeted[
        [
            "review_id",
            "year",
            "period",
            "original_segment_id",
            "target_segment_id",
            "original_topic_share",
            "target_topic_share",
            "share_increase",
            "segment_changed"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 7. Historical vocabulary diagnostics
# ------------------------------------------------------------

print("\nTARGETED SEGMENT KEYWORD DIAGNOSTICS")

display(
    targeted[
        [
            "review_id", "year",
            "target_topic_share",
            "kampuchea_cambodia",
            "afghanistan",
            "vietnam",
            "middle_east",
            "palestine_israel",
            "conflict_terms",
            "settlement_terms"
        ]
    ].round(4)
)

print("\nHISTORICAL PERIOD SUMMARY")

period_summary_27e = (
    targeted.groupby(
        "period", observed=True
    )
    .agg(
        n_speeches=("speech_id", "nunique"),
        mean_original_share=(
            "original_topic_share", "mean"
        ),
        mean_targeted_share=(
            "target_topic_share", "mean"
        ),
        changed_segments=("segment_changed", "sum"),
        kampuchea_mentions=(
            "kampuchea_cambodia", "sum"
        ),
        afghanistan_mentions=(
            "afghanistan", "sum"
        ),
        middle_east_mentions=(
            "middle_east", "sum"
        )
    )
    .reset_index()
)

display(period_summary_27e.round(4))

# ------------------------------------------------------------
# 8. Prepare human-review data
# ------------------------------------------------------------

semantic_china_targeted_27e = targeted.copy()

semantic_review_template_27e = targeted[
    [
        "review_id", "year", "period",
        "speech_id", "target_segment_id",
        "target_topic_share"
    ]
].copy()

semantic_review_template_27e[
    "primary_geographic_focus"
] = pd.NA

semantic_review_template_27e[
    "substantive_conflict_or_policy"
] = pd.NA

semantic_review_template_27e[
    "supports_topic_label"
] = pd.NA

semantic_review_template_27e[
    "semantic_drift_notes"
] = pd.NA

semantic_review_template_27e[
    "supporting_evidence"
] = pd.NA

print("\nHUMAN REVIEW TEMPLATE")
display(semantic_review_template_27e)

# ------------------------------------------------------------
# 9. Final validation
# ------------------------------------------------------------

assert len(semantic_china_targeted_27e) == 6
assert semantic_china_targeted_27e["period"].nunique() == 3
assert (
    semantic_china_targeted_27e.groupby(
        "period", observed=True
    ).size() == 2
).all()

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Original speech sample preserved: PASSED")
print("Within-speech segment matching: PASSED")
print("Historical stratification: PASSED")
print("Topic-weight comparison: PASSED")
print("Source text availability: PASSED")
print("Human semantic interpretation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27E: COMPUTATION COMPLETED")


In [ ]:

# ============================================================
# STEP 12.27E — COMPLETE TEXT INSPECTION
# READ ONLY — NO FILE SAVING
# ============================================================


# Change to "1991-2010" or "2011-2025" for other periods
TARGET_PERIOD = "1972-1990"

assert "semantic_china_targeted_27e" in globals()

review_period = semantic_china_targeted_27e.loc[
    semantic_china_targeted_27e["period"].astype(str)
    == TARGET_PERIOD
].sort_values("year")

assert len(review_period) == 2

print("=" * 68)
print("TOPIC 03 — COMPLETE TARGETED SOURCE TEXTS")
print("Historical period:", TARGET_PERIOD)
print("=" * 68)

for _, record in review_period.iterrows():

    print("\n" + "=" * 68)
    print("Review ID:", record["review_id"])
    print("Year:", record["year"])
    print("Speech ID:", record["speech_id"])
    print("Target segment:", record["target_segment_id"])
    print(
        "Topic 03 share:",
        f"{record['target_topic_share']:.4f}"
    )
    print("=" * 68)

    clean_text = " ".join(
        str(record["target_text"]).split()
    )

    print(
        textwrap.fill(
            clean_text,
            width=100,
            break_long_words=False
        )
    )

print("\nCOMPLETE TEXT INSPECTION: FINISHED")
print("Files saved: NONE")



### Step 12.27F — Diachronic Semantic Interpretation of Topic 03

**Objective:** Evaluate whether Topic 03 represents a coherent regional-conflict discourse across historical periods or combines distinct geopolitical themes.

**Reference:** Balanced Chinese speech cohort (1972–2025), K=7 NMF, reference seed 11.

**Procedure:**
1. Retain all six previously selected Chinese speeches.
2. Examine the highest-scoring Topic 03 segment within each speech.
3. Review the complete source text, emphasizing the four post-1990 records.
4. Identify geographic references, conflict narratives, and diplomatic-settlement language.
5. Distinguish historical shifts in geographic focus from potential topic-model confounding.
6. Record qualitative judgments without automatically assigning semantic validity.

**Interpretation rule:** Geographic keyword frequencies are descriptive indicators, not independent evidence that a topic is semantically coherent.

**Output:** In-memory semantic review dataset, contextual keyword evidence, and complete source text for the selected review record.

**Saving:** Disabled. Publication-ready exports will be handled in a separate cell after semantic validation.


In [ ]:

# ============================================================
# STEP 12.27F — DIACHRONIC SEMANTIC REVIEW
# COMPUTATION AND INSPECTION ONLY — NO FILE SAVING
# ============================================================


print("=" * 68)
print("TOPIC 03 — DIACHRONIC SEMANTIC REVIEW")
print("=" * 68)

# ------------------------------------------------------------
# 1. Validate previous analytical results
# ------------------------------------------------------------

assert "semantic_china_targeted_27e" in globals(), (
    "Run Step 12.27E first."
)

semantic_27f = semantic_china_targeted_27e.copy()

required = [
    "review_id",
    "speech_id",
    "year",
    "period",
    "target_segment_id",
    "target_topic_share",
    "target_text"
]

assert set(required).issubset(semantic_27f.columns)
assert len(semantic_27f) == 6
assert semantic_27f["speech_id"].nunique() == 6
assert semantic_27f["target_text"].notna().all()

# ------------------------------------------------------------
# 2. Historical review inventory
# ------------------------------------------------------------

print("\nHISTORICAL REVIEW INVENTORY")

inventory = semantic_27f[
    [
        "review_id",
        "year",
        "period",
        "speech_id",
        "target_segment_id",
        "target_topic_share"
    ]
].copy()

display(inventory.round(4))

# ------------------------------------------------------------
# 3. Select post-1990 evidence requiring detailed review
# ------------------------------------------------------------

post_1990 = semantic_27f.loc[
    semantic_27f["year"] >= 1991
].copy()

assert len(post_1990) == 4

print("\nPOST-1990 SEMANTIC REVIEW PRIORITIES")

priority = post_1990[
    [
        "review_id",
        "year",
        "target_topic_share",
        "kampuchea_cambodia",
        "afghanistan",
        "middle_east",
        "palestine_israel",
        "conflict_terms",
        "settlement_terms"
    ]
].copy()

display(priority.round(4))

# ------------------------------------------------------------
# 4. Specify geographic and substantive evidence patterns
# ------------------------------------------------------------

patterns_27f = {
    "Cambodia / Kampuchea":
        r"\b(?:cambodia\w*|kampuchea\w*)\b",

    "Afghanistan":
        r"\bafghan\w*\b",

    "Vietnam":
        r"\bviet(?:nam|namese)\b",

    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "Middle East":
        r"\bmiddle[\s-]+east\b",

    "Military conflict":
        r"\b(?:war|wars|conflict\w*|"
        r"aggress\w*|invasion\w*|"
        r"occupation\w*|military)\b",

    "Diplomatic settlement":
        r"\b(?:settlement\w*|negotiat\w*|"
        r"peace\w*|resolution\w*|"
        r"diplomac\w*|dialogue)\b"
}

# ------------------------------------------------------------
# 5. Review one complete segment per execution
# ------------------------------------------------------------

# Change this identifier and rerun the cell:
# SV-015 : China, 1991
# SV-016 : China, 1999
# SV-017 : China, 2014
# SV-018 : China, 2022
#
# Earlier reference cases:
# SV-013 : China, 1981
# SV-014 : China, 1988

REVIEW_ID = "SV-018"

selected = semantic_27f.loc[
    semantic_27f["review_id"] == REVIEW_ID
]

assert len(selected) == 1, "Invalid review identifier."

record = selected.iloc[0]

text = str(record["target_text"])

assert text.strip(), "Selected segment has no text."

print("\n" + "=" * 68)
print("COMPLETE SOURCE SEGMENT")
print("=" * 68)

print("Review ID:", record["review_id"])
print("Country:", record["country_code"])
print("Year:", record["year"])
print("Historical period:", record["period"])
print("Speech ID:", record["speech_id"])
print("Segment ID:", record["target_segment_id"])
print(
    "Topic 03 share:",
    f"{record['target_topic_share']:.4f}"
)

print("\nCOMPLETE TEXT:\n")

print(
    textwrap.fill(
        " ".join(text.split()),
        width=105,
        break_long_words=False
    )
)

# ------------------------------------------------------------
# 6. Keyword-in-context evidence
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("KEYWORD-IN-CONTEXT EVIDENCE")
print("=" * 68)

evidence_rows = []

for category, pattern in patterns_27f.items():

    matches = list(
        re.finditer(
            pattern,
            text,
            flags=re.IGNORECASE
        )
    )

    evidence_rows.append({
        "category": category,
        "n_mentions": len(matches)
    })

    print(f"\n{category}: {len(matches)} occurrence(s)")

    # Display a maximum of three context excerpts per category.
    for match in matches[:3]:

        left = max(0, match.start() - 100)
        right = min(len(text), match.end() + 100)

        excerpt = " ".join(
            text[left:right].split()
        )

        print("  ...", excerpt, "...")

keyword_evidence_27f = pd.DataFrame(evidence_rows)

print("\nKEYWORD EVIDENCE SUMMARY")
display(keyword_evidence_27f)

# ------------------------------------------------------------
# 7. Prepare structured human semantic review
# ------------------------------------------------------------

human_review_27f = semantic_27f[
    [
        "review_id",
        "country_code",
        "year",
        "period",
        "speech_id",
        "target_segment_id",
        "target_topic_share"
    ]
].copy()

human_review_27f["geographic_focus"] = pd.NA
human_review_27f["conflict_or_policy_context"] = pd.NA
human_review_27f["supports_broad_topic_label"] = pd.NA
human_review_27f["semantic_consistency"] = pd.NA
human_review_27f["supporting_quote"] = pd.NA
human_review_27f["reviewer_notes"] = pd.NA

print("\nHUMAN REVIEW STATUS")
print("Total records:", len(human_review_27f))
print("Qualitative judgments: PENDING")
print("No automated semantic decisions assigned.")

# ------------------------------------------------------------
# 8. Final checks
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Original speech sample preserved: PASSED")
print("Targeted source retrieval: PASSED")
print("Keyword-context extraction: PASSED")
print("Historical review inventory: PASSED")
print("Human semantic validation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27F: INSPECTION READY")



### Step 12.27G — Cross-Country Semantic Validation of Topic 03

**Objective:** Extend the diachronic semantic validation of Topic 03 beyond China by systematically sampling Iranian and United States speeches.

**Design:**
- Countries: Iran and United States.
- Historical periods: 1972–1990, 1991–2010, and 2011–2025.
- Two source segments per country-period stratum.
- Selection: highest topic-contributing speech and a reproducibly sampled comparison speech.
- Reference specification: selected seven-topic segment-level NMF model.

**Validation:** Verify the recovered segment-topic matrix against previously inspected Chinese reference records before drawing additional samples.

**Interpretation:** Automated keyword diagnostics are exploratory and do not constitute human semantic validation. Topic labels remain provisional.

**Output:** Cross-country review inventory, keyword diagnostics, and a human-review template.

**File saving:** Disabled in this computational cell.


In [ ]:

print("=" * 65)
print("STEP 12.27G — CROSS-COUNTRY SEMANTIC VALIDATION")
print("=" * 65)

# ---------------------------------------------------------
# 1. Validate notebook sources
# ---------------------------------------------------------

required_objects = ["segments_df", "component_contributions"]

missing = [
    name for name in required_objects
    if name not in globals()
]

if missing:
    raise RuntimeError(
        f"Required notebook objects not found: {missing}"
    )

source_segments = segments_df.copy()

required_cols = [
    "speech_id", "segment_id", "country_code",
    "year", "segment_text"
]

missing_cols = [
    c for c in required_cols
    if c not in source_segments.columns
]

if missing_cols:
    raise RuntimeError(
        f"Missing segment metadata: {missing_cols}"
    )

reference_segment_matrix = np.asarray(
    component_contributions, dtype=float
)

assert reference_segment_matrix.shape == (
    len(source_segments), 7
), "Incorrect reference segment-topic dimensions."

assert np.isfinite(reference_segment_matrix).all()
assert (reference_segment_matrix >= -1e-10).all()

row_totals = reference_segment_matrix.sum(axis=1)

assert (row_totals > 0).all(), (
    "Zero-weight reference segment detected."
)

reference_segment_matrix = (
    reference_segment_matrix / row_totals[:, None]
)

source_segments["topic03_share"] = (
    reference_segment_matrix[:, 2]
)

for col in ["speech_id", "segment_id", "year"]:
    source_segments[col] = pd.to_numeric(
        source_segments[col], errors="raise"
    ).astype(int)

assert not source_segments.duplicated(
    ["speech_id", "segment_id"]
).any(), "Duplicate speech-segment identifiers."

# ---------------------------------------------------------
# 2. Reference verification using Chinese samples
# ---------------------------------------------------------

reference_checks = [
    (27, 4, 1.0000),
    (48, 9, 1.0000),
    (57, 5, 0.9438),
    (81, 5, 0.4368),
    (126, 6, 0.4735),
    (150, 9, 0.3396),
]

verification_rows = []

for speech_id, segment_id, expected in reference_checks:

    match = source_segments.loc[
        (source_segments["speech_id"] == speech_id) &
        (source_segments["segment_id"] == segment_id) &
        (source_segments["country_code"] == "CHN")
    ]

    assert len(match) == 1, (
        f"Missing Chinese reference: "
        f"{speech_id}/{segment_id}"
    )

    observed = float(match.iloc[0]["topic03_share"])

    verification_rows.append({
        "speech_id": speech_id,
        "segment_id": segment_id,
        "expected_share": expected,
        "observed_share": observed,
        "absolute_error": abs(observed - expected)
    })

reference_check_df = pd.DataFrame(verification_rows)

print("\nCHINESE REFERENCE VERIFICATION")
display(reference_check_df.round(5))

max_error = reference_check_df["absolute_error"].max()

if max_error > 0.01:
    raise RuntimeError(
        "Reference topic weights do not adequately match "
        "the previously verified Chinese records. "
        "Stop and inspect model identity/alignment."
    )

print("REFERENCE VERIFICATION: PASSED")

# ---------------------------------------------------------
# 3. Historical stratification
# ---------------------------------------------------------

source_segments = source_segments.loc[
    source_segments["year"].between(1972, 2025)
].copy()

source_segments["period"] = pd.cut(
    source_segments["year"],
    bins=[1971, 1990, 2010, 2025],
    labels=[
        "1972-1990",
        "1991-2010",
        "2011-2025"
    ]
).astype(str)

countries = ["IRN", "USA"]

periods = [
    "1972-1990",
    "1991-2010",
    "2011-2025"
]

# ---------------------------------------------------------
# 4. Reproducible speech-level sampling
# ---------------------------------------------------------

selected_records = []

for country in countries:
    for period in periods:

        subset = source_segments.loc[
            (source_segments["country_code"] == country) &
            (source_segments["period"] == period)
        ].copy()

        # Retain the strongest Topic 03 segment
        # from each distinct speech.

        candidates = (
            subset.sort_values(
                ["topic03_share", "year", "segment_id"],
                ascending=[False, True, True]
            )
            .drop_duplicates("speech_id")
            .reset_index(drop=True)
        )

        if len(candidates) < 2:
            raise RuntimeError(
                f"Insufficient distinct speeches: "
                f"{country}, {period}"
            )

        highest = candidates.iloc[0]

        # Reproducibly select a different speech.

        comparison = candidates.iloc[1:].sample(
            n=1,
            random_state=73
        ).iloc[0]

        for selection_type, record in [
            ("Highest contribution", highest),
            ("Random comparison", comparison)
        ]:
            selected_records.append({
                "country_code": country,
                "period": period,
                "selection_type": selection_type,
                "year": int(record["year"]),
                "speech_id": int(record["speech_id"]),
                "segment_id": int(record["segment_id"]),
                "topic03_share": float(
                    record["topic03_share"]
                ),
                "archive_path": record.get(
                    "archive_path", ""
                ),
                "segment_text": str(
                    record["segment_text"]
                )
            })

semantic_crosscountry_df = pd.DataFrame(
    selected_records
)

semantic_crosscountry_df.insert(
    0,
    "review_id",
    [
        f"SV-X{i:03d}"
        for i in range(1, len(semantic_crosscountry_df) + 1)
    ]
)

assert len(semantic_crosscountry_df) == 12

assert semantic_crosscountry_df.groupby(
    ["country_code", "period"]
).size().eq(2).all()

# ---------------------------------------------------------
# 5. Exploratory keyword diagnostics
# ---------------------------------------------------------

keyword_patterns = {
    "cambodia_kampuchea":
        r"\b(?:cambodia\w*|kampuchea\w*)\b",

    "afghanistan":
        r"\bafghan\w*\b",

    "palestine_israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "kosovo_yugoslavia":
        r"\b(?:kosovo|yugoslav\w*)\b",

    "syria":
        r"\bsyria\w*\b",

    "korean_peninsula":
        r"\b(?:korea\w*|peninsula)\b",

    "diplomatic_settlement":
        r"\b(?:peace|settle\w*|negotiat\w*|"
        r"dialogue|mediat\w*|consultation\w*|"
        r"ceasefire)\b",

    "sovereignty_intervention":
        r"\b(?:sovereign\w*|independen\w*|"
        r"intervention|non.interference)\b"
}

semantic_keyword_df = semantic_crosscountry_df[
    [
        "review_id", "country_code",
        "period", "year", "topic03_share"
    ]
].copy()

for category, pattern in keyword_patterns.items():

    semantic_keyword_df[category] = [
        len(re.findall(pattern, text, flags=re.I))
        for text in semantic_crosscountry_df[
            "segment_text"
        ]
    ]

# ---------------------------------------------------------
# 6. Human-review template
# ---------------------------------------------------------

semantic_crosscountry_review_df = (
    semantic_crosscountry_df.copy()
)

human_fields = [
    "primary_geographic_focus",
    "substantive_conflict_or_policy",
    "supports_broad_topic_label",
    "semantic_drift_assessment",
    "supporting_evidence",
    "reviewer_notes"
]

for field in human_fields:
    semantic_crosscountry_review_df[field] = pd.NA

# ---------------------------------------------------------
# 7. Display results (no files saved)
# ---------------------------------------------------------

print("\n" + "=" * 65)
print("CROSS-COUNTRY SEMANTIC REVIEW INVENTORY")
print("=" * 65)

inventory_cols = [
    "review_id", "country_code", "period",
    "selection_type", "year", "speech_id",
    "segment_id", "topic03_share"
]

display(
    semantic_crosscountry_df[
        inventory_cols
    ].round(4)
)

print("\n" + "=" * 65)
print("KEYWORD DIAGNOSTICS")
print("=" * 65)

display(
    semantic_keyword_df.round(4)
)

print("\n" + "=" * 65)
print("SOURCE TEXT PREVIEWS")
print("=" * 65)

for _, record in semantic_crosscountry_df.iterrows():

    preview = re.sub(
        r"\s+", " ", record["segment_text"]
    )[:350]

    print(
        f"\n{record['review_id']} | "
        f"{record['country_code']} | "
        f"{record['year']} | "
        f"Topic share: {record['topic03_share']:.4f}"
    )

    print(preview + "...")

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Reference matrix validation: PASSED")
print("Historical stratification: PASSED")
print("Distinct speech selection: PASSED")
print("Source records prepared:", len(semantic_crosscountry_df))
print("Human semantic validation: PENDING")
print("Files saved: NONE")
print("STEP 12.27G: COMPUTATION COMPLETED")



### Step 12.27H — Cross-Country Qualitative Semantic Inspection

**Objective:** Examine original source segments from Iran and the United States to evaluate the substantive meaning and historical consistency of Topic 03.

**Reference model:** Seven-topic, speech-balanced segment-level NMF.

**Review procedure:**
1. Inspect complete source segments without truncation.
2. Identify the principal geopolitical conflict or policy issue.
3. Distinguish geographic focus from broader diplomatic themes.
4. Assess whether the proposed broad topic label accurately represents the text.
5. Document evidence of semantic continuity or historical change.

**Review strategy:** Examine two records at a time, grouped by country and historical period, preserving the original sampling design.

**Quality control:** Automated keyword matches are descriptive diagnostics, not substitutes for contextual human interpretation.

**Output:** Complete source texts, review metadata, keyword evidence, and human-review fields.

**Saving:** No files are written in this computational cell.


In [ ]:

print("=" * 70)
print("STEP 12.27H — CROSS-COUNTRY SEMANTIC INSPECTION")
print("=" * 70)

# =========================================================
# 1. REVIEW CONFIGURATION
# =========================================================

REVIEW_COUNTRY = "IRN"
REVIEW_PERIOD = "1972-1990"

VALID_COUNTRIES = ["IRN", "USA"]

VALID_PERIODS = [
    "1972-1990",
    "1991-2010",
    "2011-2025"
]

assert REVIEW_COUNTRY in VALID_COUNTRIES
assert REVIEW_PERIOD in VALID_PERIODS

# =========================================================
# 2. VERIFY EXISTING SOURCE RECORDS
# =========================================================

assert "semantic_crosscountry_df" in globals(), (
    "Run Step 12.27G first."
)

source = semantic_crosscountry_df.copy()

required_columns = [
    "review_id",
    "country_code",
    "period",
    "year",
    "speech_id",
    "segment_id",
    "topic03_share",
    "segment_text"
]

missing_columns = [
    col for col in required_columns
    if col not in source.columns
]

assert not missing_columns, (
    f"Missing columns: {missing_columns}"
)

assert len(source) == 12
assert source["review_id"].is_unique
assert source["segment_text"].notna().all()

# =========================================================
# 3. SELECT TWO RECORDS
# =========================================================

review_subset = source.loc[
    (source["country_code"] == REVIEW_COUNTRY) &
    (source["period"] == REVIEW_PERIOD)
].copy()

review_subset = review_subset.sort_values(
    ["selection_type", "year"]
).reset_index(drop=True)

assert len(review_subset) == 2, (
    "Expected exactly two records for this stratum."
)

assert review_subset["speech_id"].nunique() == 2, (
    "Selected records must represent distinct speeches."
)

print("\nREVIEW CONFIGURATION")
print("Country:", REVIEW_COUNTRY)
print("Historical period:", REVIEW_PERIOD)
print("Selected records:", len(review_subset))

# =========================================================
# 4. REVIEW INVENTORY
# =========================================================

print("\n" + "=" * 70)
print("SOURCE RECORD INVENTORY")
print("=" * 70)

display(
    review_subset[
        [
            "review_id",
            "year",
            "selection_type",
            "speech_id",
            "segment_id",
            "topic03_share"
        ]
    ].round(5)
)

# =========================================================
# 5. KEYWORD DIAGNOSTICS
# =========================================================

keyword_categories = {
    "Cambodia / Kampuchea":
        r"\b(?:cambodia\w*|kampuchea\w*)\b",

    "Afghanistan":
        r"\bafghan\w*\b",

    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "Middle East":
        r"\bmiddle[\s-]+east\b",

    "South Africa / Apartheid":
        r"\b(?:apartheid|south[\s-]+africa\w*)\b",

    "Sovereignty / Intervention":
        r"\b(?:sovereign\w*|intervention|"
        r"non.interference|independen\w*)\b",

    "Diplomatic Settlement":
        r"\b(?:peace|settle\w*|negotiat\w*|"
        r"dialogue|mediat\w*|ceasefire|"
        r"consultation\w*)\b",

    "Armed Conflict":
        r"\b(?:war|wars|armed|aggression|"
        r"hostilities|invasion|conflict\w*)\b"
}

# =========================================================
# 6. COMPLETE SOURCE INSPECTION
# =========================================================

inspection_rows = []

for _, record in review_subset.iterrows():

    text = str(record["segment_text"])

    print("\n" + "#" * 70)

    print("REVIEW ID:", record["review_id"])
    print("Country:", record["country_code"])
    print("Year:", record["year"])
    print("Selection:", record["selection_type"])
    print("Speech ID:", record["speech_id"])
    print("Segment ID:", record["segment_id"])

    print(
        "Reference Topic 03 share:",
        f"{float(record['topic03_share']):.5f}"
    )

    print("Source:", record.get("archive_path", ""))

    print("\nCOMPLETE ORIGINAL SEGMENT")
    print("-" * 70)

    # Preserve the complete text while wrapping for readability.
    paragraphs = text.splitlines()

    for paragraph in paragraphs:
        if paragraph.strip():
            print(
                textwrap.fill(
                    paragraph,
                    width=100,
                    break_long_words=False,
                    replace_whitespace=False
                )
            )
        else:
            print()

    print("\nKEYWORD EVIDENCE")
    print("-" * 70)

    diagnostics = {
        "review_id": record["review_id"],
        "country_code": record["country_code"],
        "year": int(record["year"]),
        "topic03_share": float(record["topic03_share"])
    }

    for category, pattern in keyword_categories.items():

        matches = list(
            re.finditer(pattern, text, flags=re.I)
        )

        diagnostics[category] = len(matches)

        print(f"{category}: {len(matches)}")

        # Up to two illustrative contexts per category.
        for match in matches[:2]:

            start = max(0, match.start() - 65)
            end = min(len(text), match.end() + 65)

            context = re.sub(
                r"\s+", " ", text[start:end]
            )

            print("  ...", context, "...")

    inspection_rows.append(diagnostics)

# =========================================================
# 7. STRUCTURED REVIEW OUTPUT
# =========================================================

inspection_diagnostics_df = pd.DataFrame(
    inspection_rows
)

print("\n" + "=" * 70)
print("COMPARATIVE KEYWORD EVIDENCE")
print("=" * 70)

display(inspection_diagnostics_df.round(5))

human_review_columns = [
    "primary_geographic_focus",
    "principal_conflict_or_policy",
    "sovereignty_or_intervention_theme",
    "diplomatic_settlement_theme",
    "supports_broad_topic_label",
    "semantic_drift_assessment",
    "supporting_text",
    "reviewer_notes"
]

inspection_review_template = review_subset[
    [
        "review_id",
        "country_code",
        "period",
        "year",
        "speech_id",
        "segment_id",
        "topic03_share"
    ]
].copy()

for column in human_review_columns:
    inspection_review_template[column] = pd.NA

print("\n" + "=" * 70)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 70)

display(inspection_review_template)

# =========================================================
# 8. FINAL VALIDATION
# =========================================================

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

assert len(inspection_diagnostics_df) == 2

assert set(inspection_diagnostics_df["review_id"]) == (
    set(review_subset["review_id"])
)

print("Source linkage: PASSED")
print("Distinct speech selection: PASSED")
print("Complete text retrieval: PASSED")
print("Keyword diagnostics: PASSED")
print("Human semantic judgement: PENDING")
print("Files saved: NONE")
print("\nSTEP 12.27H: INSPECTION READY")


In [ ]:

# ============================================================
# STEP 12.27H.2 — IRAN, 1991–2010
# CROSS-COUNTRY SEMANTIC INSPECTION
# No files are saved.
# ============================================================


REVIEW_COUNTRY = "IRN"
REVIEW_PERIOD = "1991-2010"

assert "semantic_crosscountry_df" in globals()

source = semantic_crosscountry_df.copy()

required = [
    "review_id", "country_code", "period",
    "year", "speech_id", "segment_id",
    "topic03_share", "segment_text"
]

assert set(required).issubset(source.columns)

review_subset = source.loc[
    (source["country_code"] == REVIEW_COUNTRY) &
    (source["period"] == REVIEW_PERIOD)
].copy().sort_values("review_id")

assert len(review_subset) == 2
assert review_subset["speech_id"].nunique() == 2
assert review_subset["segment_text"].notna().all()

print("=" * 70)
print("STEP 12.27H.2 — IRAN HISTORICAL SEMANTIC REVIEW")
print("=" * 70)

display(
    review_subset[
        [
            "review_id", "year", "selection_type",
            "speech_id", "segment_id", "topic03_share"
        ]
    ].reset_index(drop=True)
)

# ------------------------------------------------------------
# Geographic and thematic evidence
# ------------------------------------------------------------

patterns = {
    "Cambodia / Kampuchea": r"\b(?:cambodia\w*|kampuchea\w*)\b",
    "Afghanistan": r"\bafghan\w*\b",
    "Palestine / Israel": r"\b(?:palestin\w*|israel\w*)\b",
    "Middle East": r"\bmiddle[\s-]+east\b",
    "Iraq": r"\biraq\w*\b",
    "Kosovo / Yugoslavia": r"\b(?:kosovo|yugoslav\w*)\b",
    "South Africa / Apartheid": r"\b(?:apartheid|south[\s-]+africa\w*)\b",
    "Sovereignty / Intervention":
        r"\b(?:sovereign\w*|intervention|non.interference|independen\w*)\b",
    "Diplomatic Settlement":
        r"\b(?:peace|settle\w*|negotiat\w*|dialogue|mediat\w*|ceasefire)\b",
    "Armed Conflict":
        r"\b(?:war|wars|armed|aggression|invasion|conflict\w*)\b"
}

diagnostic_rows = []

for _, record in review_subset.iterrows():

    content = str(record["segment_text"])

    print("\n" + "=" * 70)
    print("REVIEW ID:", record["review_id"])
    print("Country:", record["country_code"])
    print("Year:", record["year"])
    print("Speech ID:", record["speech_id"])
    print("Segment ID:", record["segment_id"])
    print(f"Topic 03 share: {record['topic03_share']:.5f}")
    print("Source:", record.get("archive_path", ""))

    print("\nCOMPLETE ORIGINAL SOURCE TEXT")
    print("-" * 70)

    for paragraph in content.splitlines():
        if paragraph.strip():
            print(textwrap.fill(
                paragraph,
                width=100,
                break_long_words=False,
                replace_whitespace=False
            ))
        else:
            print()

    row = {
        "review_id": record["review_id"],
        "year": record["year"],
        "topic03_share": record["topic03_share"]
    }

    print("\nKEYWORD EVIDENCE")
    print("-" * 70)

    for label, pattern in patterns.items():
        matches = list(re.finditer(pattern, content, re.I))
        row[label] = len(matches)
        print(f"{label}: {len(matches)}")

    diagnostic_rows.append(row)

iran_middle_semantic_diagnostics = pd.DataFrame(
    diagnostic_rows
)

print("\n" + "=" * 70)
print("COMPARATIVE SEMANTIC DIAGNOSTICS")
print("=" * 70)

display(iran_middle_semantic_diagnostics)

# ------------------------------------------------------------
# Human review template — no automatic semantic decisions
# ------------------------------------------------------------

iran_middle_review_template = review_subset[
    [
        "review_id", "country_code", "period",
        "year", "speech_id", "segment_id",
        "topic03_share"
    ]
].copy()

for col in [
    "primary_geographic_focus",
    "principal_conflict_or_policy",
    "supports_broad_topic_label",
    "semantic_drift_assessment",
    "supporting_text",
    "reviewer_notes"
]:
    iran_middle_review_template[col] = pd.NA

print("\nHUMAN REVIEW TEMPLATE")
display(iran_middle_review_template)

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)
print("Source records:", len(review_subset))
print("Distinct speeches:", review_subset["speech_id"].nunique())
print("Complete text retrieval: PASSED")
print("Keyword diagnostics: PASSED")
print("Human semantic validation: PENDING")
print("Files saved: NONE")
print("\nSTEP 12.27H.2: INSPECTION READY")


In [ ]:

# ============================================================
# STEP 12.27H.3
# IRAN — LATE-PERIOD SEMANTIC VALIDATION
# Additional verification: 1992, 2016, 2025
# NO FILES SAVED
# ============================================================

assert "semantic_crosscountry_df" in globals(), (
    "Cross-country semantic evidence is unavailable."
)

source = semantic_crosscountry_df.copy()

required_columns = [
    "review_id", "country_code", "year", "period",
    "speech_id", "segment_id", "topic03_share",
    "segment_text"
]

assert set(required_columns).issubset(source.columns)

REVIEW_IDS = ["SV-X004", "SV-X005", "SV-X006"]

review_df = source.loc[
    source["review_id"].isin(REVIEW_IDS)
].copy()

assert len(review_df) == 3
assert set(review_df["review_id"]) == set(REVIEW_IDS)
assert review_df["country_code"].eq("IRN").all()
assert review_df["speech_id"].nunique() == 3
assert review_df["segment_text"].notna().all()

review_df["review_id"] = pd.Categorical(
    review_df["review_id"],
    categories=REVIEW_IDS,
    ordered=True
)

review_df = review_df.sort_values(
    "review_id"
).reset_index(drop=True)

print("=" * 72)
print("STEP 12.27H.3 — IRAN SEMANTIC VALIDATION")
print("=" * 72)

print("\nSOURCE RECORD INVENTORY")

display(
    review_df[
        [
            "review_id", "year", "period",
            "speech_id", "segment_id",
            "topic03_share"
        ]
    ]
)

# ------------------------------------------------------------
# Keyword-context categories
# Descriptive evidence only; not semantic classification.
# ------------------------------------------------------------

patterns = {
    "Cambodia / Kampuchea":
        r"\b(?:cambodia\w*|kampuchea\w*)\b",

    "Afghanistan":
        r"\bafghan\w*\b",

    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "Middle East":
        r"\bmiddle[\s-]+east\b",

    "Iraq / Syria":
        r"\b(?:iraq\w*|syria\w*)\b",

    "Sovereignty / Intervention":
        r"\b(?:sovereign\w*|interven\w*|interferen\w*)\b",

    "Diplomatic Settlement":
        r"\b(?:peace|settle\w*|negotiat\w*|dialogue|mediat\w*|ceasefire)\b",

    "Armed Conflict":
        r"\b(?:war|wars|armed|aggression|invasion|conflict\w*)\b",

    "Humanitarian Crisis":
        r"\b(?:humanitarian|genocide|starvation|refugee\w*|civilian\w*)\b",

    "Sanctions":
        r"\bsanction\w*\b"
}

diagnostics = []

for _, record in review_df.iterrows():

    content = str(record["segment_text"])

    row = {
        "review_id": str(record["review_id"]),
        "year": int(record["year"]),
        "topic03_share": float(record["topic03_share"])
    }

    for category, pattern in patterns.items():
        row[category] = len(
            list(re.finditer(pattern, content, flags=re.I))
        )

    diagnostics.append(row)

iran_late_semantic_diagnostics = pd.DataFrame(
    diagnostics
)

print("\n" + "=" * 72)
print("COMPARATIVE KEYWORD DIAGNOSTICS")
print("=" * 72)

display(iran_late_semantic_diagnostics)

# ------------------------------------------------------------
# Complete source texts
# Printed individually without excerpt truncation.
# ------------------------------------------------------------

for _, record in review_df.iterrows():

    print("\n" + "=" * 72)
    print("COMPLETE ORIGINAL SOURCE SEGMENT")
    print("=" * 72)

    print("Review ID:", record["review_id"])
    print("Country:", record["country_code"])
    print("Year:", record["year"])
    print("Period:", record["period"])
    print("Speech ID:", record["speech_id"])
    print("Segment ID:", record["segment_id"])
    print(
        "Topic 03 share:",
        f"{float(record['topic03_share']):.4f}"
    )

    if "archive_path" in review_df.columns:
        print("Source:", record["archive_path"])

    print("\nCOMPLETE TEXT:\n")

    content = str(record["segment_text"])

    for paragraph in content.splitlines():
        if paragraph.strip():
            print(
                textwrap.fill(
                    paragraph,
                    width=100,
                    break_long_words=False,
                    break_on_hyphens=False,
                    replace_whitespace=False
                )
            )
        else:
            print()

    print("\nKEYWORD EVIDENCE:\n")

    evidence = iran_late_semantic_diagnostics.loc[
        iran_late_semantic_diagnostics["review_id"]
        == str(record["review_id"])
    ].iloc[0]

    for category in patterns:
        print(f"{category}: {int(evidence[category])}")

# ------------------------------------------------------------
# Human semantic review template
# No automatic qualitative judgments.
# ------------------------------------------------------------

iran_late_review_template = review_df[
    [
        "review_id", "year", "period",
        "speech_id", "segment_id",
        "topic03_share"
    ]
].copy()

for column in [
    "primary_geographic_focus",
    "principal_conflict_or_policy",
    "supports_broad_topic_label",
    "semantic_drift_assessment",
    "supporting_evidence",
    "reviewer_notes"
]:
    iran_late_review_template[column] = pd.NA

print("\n" + "=" * 72)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 72)

display(iran_late_review_template)

# ------------------------------------------------------------
# Validation
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

print("Source records:", len(review_df))
print("Distinct speeches:", review_df["speech_id"].nunique())
print("Original source text retrieval: PASSED")
print("Keyword-context diagnostics: PASSED")
print("Historical-period coverage: PASSED")
print("Human semantic interpretation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.3: INSPECTION READY")



### Step 12.27H.4 — United States Historical Semantic Validation

**Objective:** Assess the historical and substantive meaning of Topic 03 in United States General Debate speeches across 1972–2025.

The analysis examines six source-linked segments, with two observations from each historical period (1972–1990, 1991–2010, and 2011–2025).

The audit evaluates:

1. Geographic and conflict-specific references.
2. Diplomatic settlement and sovereignty-related discourse.
3. Changes in the substantive contexts associated with Topic 03.
4. Whether a broader thematic label is supported by the underlying texts.

Keyword frequencies are descriptive screening indicators, not automated semantic judgments. Final topic interpretation requires qualitative inspection of original source passages.

No files are written during this computational step.


In [ ]:

# ============================================================
# STEP 12.27H.4
# UNITED STATES — HISTORICAL SEMANTIC VALIDATION
# NO FILES SAVED
# ============================================================


# ------------------------------------------------------------
# 1. Source validation
# ------------------------------------------------------------

assert "semantic_crosscountry_df" in globals(), (
    "Run Step 12.27G before this cell."
)

source = semantic_crosscountry_df.copy()

required = [
    "review_id",
    "country_code",
    "year",
    "period",
    "speech_id",
    "segment_id",
    "topic03_share",
    "segment_text"
]

assert set(required).issubset(source.columns)

USA_IDS = [
    "SV-X007",
    "SV-X008",
    "SV-X009",
    "SV-X010",
    "SV-X011",
    "SV-X012"
]

usa_review_df = source.loc[
    source["review_id"].isin(USA_IDS)
].copy()

assert len(usa_review_df) == 6
assert usa_review_df["review_id"].is_unique
assert set(usa_review_df["review_id"]) == set(USA_IDS)
assert usa_review_df["country_code"].eq("USA").all()
assert usa_review_df["speech_id"].nunique() == 6
assert usa_review_df["segment_text"].notna().all()

assert usa_review_df.groupby("period").size().to_dict() == {
    "1972-1990": 2,
    "1991-2010": 2,
    "2011-2025": 2
}

usa_review_df = usa_review_df.sort_values(
    ["year", "review_id"]
).reset_index(drop=True)

shares = pd.to_numeric(
    usa_review_df["topic03_share"]
)

assert shares.between(0, 1).all()

print("=" * 72)
print("STEP 12.27H.4 — UNITED STATES SEMANTIC VALIDATION")
print("=" * 72)

print("\nHISTORICAL SOURCE INVENTORY")

display(
    usa_review_df[
        [
            "review_id",
            "year",
            "period",
            "speech_id",
            "segment_id",
            "topic03_share"
        ]
    ]
)

# ------------------------------------------------------------
# 2. Geographic and thematic screening vocabulary
# ------------------------------------------------------------

usa_patterns = {
    "Cambodia / Kampuchea":
        r"\b(?:cambodia\w*|kampuchea\w*)\b",

    "Vietnam":
        r"\bviet\s?nam\w*\b",

    "Afghanistan":
        r"\bafghan\w*\b",

    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*)\b",

    "Middle East":
        r"\bmiddle[\s-]+east\b",

    "Korean Peninsula":
        r"\b(?:korea\w*|pyongyang)\b",

    "Iraq / Syria":
        r"\b(?:iraq\w*|syria\w*)\b",

    "Bosnia / Kosovo / Yugoslavia":
        r"\b(?:bosni\w*|kosov\w*|yugoslav\w*)\b",

    "South Africa / Apartheid":
        r"\b(?:apartheid|south\s+africa\w*)\b",

    "Soviet Union / Russia":
        r"\b(?:soviet\w*|russia\w*|ussr)\b",

    "Sovereignty / Intervention":
        r"\b(?:sovereign\w*|interven\w*|interferen\w*)\b",

    "Diplomatic Settlement":
        r"\b(?:peace|settle\w*|negotiat\w*|dialogue|"
        r"dialog|mediat\w*|ceasefire|agreement\w*|"
        r"conference\w*|reconcil\w*)\b",

    "Armed Conflict":
        r"\b(?:war|wars|armed|aggression|invasion|"
        r"conflict\w*|hostilit\w*)\b",

    "Human Rights":
        r"\b(?:human\s+rights|civil\s+rights|freedom\w*|"
        r"democra\w*|libert\w*)\b",

    "Humanitarian Crisis":
        r"\b(?:humanitarian|genocide|starvation|"
        r"refugee\w*|civilian\w*)\b"
}

# ------------------------------------------------------------
# 3. All six records: keyword diagnostics
# ------------------------------------------------------------

diagnostics = []

for _, record in usa_review_df.iterrows():

    content = str(record["segment_text"])

    row = {
        "review_id": record["review_id"],
        "year": int(record["year"]),
        "period": record["period"],
        "topic03_share": float(record["topic03_share"])
    }

    for category, pattern in usa_patterns.items():
        row[category] = len(
            re.findall(pattern, content, flags=re.I)
        )

    diagnostics.append(row)

usa_semantic_diagnostics = pd.DataFrame(diagnostics)

print("\n" + "=" * 72)
print("CROSS-PERIOD KEYWORD DIAGNOSTICS")
print("=" * 72)

display(usa_semantic_diagnostics)

# ------------------------------------------------------------
# 4. Geographic evidence by historical period
# ------------------------------------------------------------

geographic_categories = [
    "Cambodia / Kampuchea",
    "Vietnam",
    "Afghanistan",
    "Palestine / Israel",
    "Middle East",
    "Korean Peninsula",
    "Iraq / Syria",
    "Bosnia / Kosovo / Yugoslavia",
    "South Africa / Apartheid",
    "Soviet Union / Russia"
]

usa_period_geography = (
    usa_semantic_diagnostics
    .groupby("period", sort=False)[geographic_categories]
    .sum()
    .reset_index()
)

print("\n" + "=" * 72)
print("GEOGRAPHIC REFERENCES BY HISTORICAL PERIOD")
print("=" * 72)

display(usa_period_geography)

# ------------------------------------------------------------
# 5. Full source inspection
# Select two early-period records.
# ------------------------------------------------------------

INSPECT_IDS = ["SV-X007", "SV-X008"]

assert set(INSPECT_IDS).issubset(
    set(usa_review_df["review_id"])
)

for review_id in INSPECT_IDS:

    record = usa_review_df.loc[
        usa_review_df["review_id"] == review_id
    ].iloc[0]

    print("\n" + "=" * 72)
    print("COMPLETE ORIGINAL SOURCE SEGMENT")
    print("=" * 72)

    print("Review ID:", record["review_id"])
    print("Country:", record["country_code"])
    print("Year:", record["year"])
    print("Period:", record["period"])
    print("Speech ID:", record["speech_id"])
    print("Segment ID:", record["segment_id"])
    print(
        "Topic 03 share:",
        f"{record['topic03_share']:.4f}"
    )

    if "archive_path" in usa_review_df.columns:
        print("Source:", record["archive_path"])

    print("\nCOMPLETE TEXT:\n")

    content = str(record["segment_text"])

    for paragraph in content.splitlines():

        if paragraph.strip():
            print(
                textwrap.fill(
                    paragraph,
                    width=95,
                    break_long_words=False,
                    break_on_hyphens=False
                )
            )
        else:
            print()

    print("\nKEYWORD EVIDENCE SUMMARY")

    evidence = usa_semantic_diagnostics.loc[
        usa_semantic_diagnostics["review_id"] == review_id
    ].iloc[0]

    for category in usa_patterns:

        count = int(evidence[category])

        if count > 0:
            print(f"{category}: {count}")

# ------------------------------------------------------------
# 6. Human review template
# ------------------------------------------------------------

usa_semantic_review_template = usa_review_df[
    [
        "review_id",
        "year",
        "period",
        "speech_id",
        "segment_id",
        "topic03_share"
    ]
].copy()

for column in [
    "primary_geographic_focus",
    "principal_conflict_or_policy",
    "sovereignty_or_intervention_theme",
    "diplomatic_settlement_theme",
    "supports_broad_topic_label",
    "semantic_drift_assessment",
    "supporting_evidence",
    "reviewer_notes"
]:
    usa_semantic_review_template[column] = pd.NA

print("\n" + "=" * 72)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 72)

display(usa_semantic_review_template)

# ------------------------------------------------------------
# 7. Validation
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

print("Original source records:", len(usa_review_df))
print(
    "Distinct speeches:",
    usa_review_df["speech_id"].nunique()
)
print(
    "Historical periods:",
    usa_review_df["period"].nunique()
)
print("Original source alignment: PASSED")
print("Historical stratification: PASSED")
print("Keyword-context screening: PASSED")
print("Original text retrieval: PASSED")
print("Human semantic validation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.4: INSPECTION READY")



### Step 12.27H.5 — Detailed US Source Inspection

**Objective:** Conduct source-level semantic inspection of selected United States speeches associated with Topic 03.

This step examines complete original text, geographic references, diplomatic settlement language, and contextual keyword evidence.

**Validation priorities:**
- Identify the principal geographic and political context.
- Determine whether the segment concerns armed conflict, diplomacy, sovereignty, or other international issues.
- Evaluate whether the provisional Kampuchea-specific label adequately represents the text.
- Document possible historical semantic drift.

Keyword screening is descriptive and does not constitute human semantic validation.

**Output policy:** Computational results only. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.5
# DETAILED UNITED STATES SOURCE INSPECTION
# NO FILES SAVED
# ============================================================

# ------------------------------------------------------------
# 1. Select the source record
# ------------------------------------------------------------

REVIEW_ID = "SV-X008"  # USA, 1975

assert "usa_review_df" in globals(), (
    "Run Step 12.27H.4 first."
)

assert "usa_patterns" in globals(), (
    "Keyword definitions from Step 12.27H.4 are missing."
)

matches = usa_review_df.loc[
    usa_review_df["review_id"] == REVIEW_ID
]

assert len(matches) == 1, (
    "Selected source record is missing or duplicated."
)

record = matches.iloc[0]

assert record["country_code"] == "USA"
assert pd.notna(record["segment_text"])

content = str(record["segment_text"])

assert len(content.strip()) > 0

# ------------------------------------------------------------
# 2. Source identification
# ------------------------------------------------------------

print("=" * 72)
print("STEP 12.27H.5 — DETAILED US SOURCE INSPECTION")
print("=" * 72)

print("\nSOURCE IDENTIFICATION")
print("-" * 60)

for column in [
    "review_id",
    "country_code",
    "year",
    "period",
    "speech_id",
    "segment_id",
    "topic03_share"
]:
    print(f"{column}: {record[column]}")

if "archive_path" in record.index:
    print("Source:", record["archive_path"])

print("\n" + "=" * 72)
print("COMPLETE ORIGINAL SEGMENT")
print("=" * 72)

for paragraph in content.splitlines():

    if paragraph.strip():
        print(
            textwrap.fill(
                paragraph,
                width=95,
                break_long_words=False,
                break_on_hyphens=False
            )
        )
    else:
        print()

# ------------------------------------------------------------
# 3. Geographic and thematic evidence
# ------------------------------------------------------------

evidence_rows = []

for category, pattern in usa_patterns.items():

    occurrences = list(
        re.finditer(
            pattern,
            content,
            flags=re.IGNORECASE
        )
    )

    evidence_rows.append({
        "category": category,
        "mentions": len(occurrences)
    })

evidence_df = pd.DataFrame(evidence_rows)

evidence_df = evidence_df.sort_values(
    ["mentions", "category"],
    ascending=[False, True]
).reset_index(drop=True)

print("\n" + "=" * 72)
print("GEOGRAPHIC AND THEMATIC KEYWORD EVIDENCE")
print("=" * 72)

display(evidence_df)

# ------------------------------------------------------------
# 4. Keyword-in-context inspection
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("KEYWORD-IN-CONTEXT EVIDENCE")
print("=" * 72)

CONTEXT_CHARS = 100
MAX_EXAMPLES_PER_CATEGORY = 3

for category, pattern in usa_patterns.items():

    occurrences = list(
        re.finditer(
            pattern,
            content,
            flags=re.IGNORECASE
        )
    )

    if not occurrences:
        continue

    print(f"\n{category}: {len(occurrences)} occurrence(s)")

    for match in occurrences[:MAX_EXAMPLES_PER_CATEGORY]:

        start = max(0, match.start() - CONTEXT_CHARS)
        end = min(
            len(content),
            match.end() + CONTEXT_CHARS
        )

        excerpt = content[start:end]

        excerpt = re.sub(r"\s+", " ", excerpt).strip()

        print(
            textwrap.fill(
                f"... {excerpt} ...",
                width=95
            )
        )
        print()

# ------------------------------------------------------------
# 5. Human semantic review fields
# ------------------------------------------------------------

review_fields = pd.DataFrame([
    {
        "review_id": REVIEW_ID,
        "year": int(record["year"]),
        "topic03_share": float(record["topic03_share"]),
        "primary_geographic_focus": pd.NA,
        "principal_conflict_or_policy": pd.NA,
        "diplomatic_settlement_theme": pd.NA,
        "supports_broad_topic_label": pd.NA,
        "semantic_drift_assessment": pd.NA,
        "supporting_evidence": pd.NA
    }
])

print("\n" + "=" * 72)
print("HUMAN SEMANTIC REVIEW RECORD")
print("=" * 72)

display(review_fields)

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

print("Source record:", REVIEW_ID)
print("Original country:", record["country_code"])
print("Original year:", record["year"])
print("Complete source retrieval: PASSED")
print("Keyword-context inspection: PASSED")
print("Source metadata consistency: PASSED")
print("Human semantic judgment: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.5: INSPECTION READY")



### Step 12.27H.6 — US Historical Semantic Drift Assessment

**Objective:** Assess temporal variation in the substantive meaning of Topic 03 across United States General Debate speeches.

**Sample:** Four source segments from 1995, 1997, 2011, and 2025.

**Analytical procedure:**
1. Verify source metadata and reference topic contributions.
2. Compare geographic and thematic keyword evidence.
3. Retrieve and inspect the complete selected source segment.
4. Examine whether Topic 03 consistently represents international conflicts, diplomatic settlement, and related political issues.
5. Identify possible diachronic semantic drift.

**Important:** Keyword frequencies are descriptive indicators, not definitive semantic classifications. Human interpretation remains necessary.

**Output policy:** In-memory diagnostics only. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.6
# US HISTORICAL SEMANTIC DRIFT ASSESSMENT
# COMPUTATION ONLY — NO FILES SAVED
# ============================================================


# ------------------------------------------------------------
# 1. Configuration
# ------------------------------------------------------------

REVIEW_IDS = [
    "SV-X010",  # 1995
    "SV-X009",  # 1997
    "SV-X011",  # 2011
    "SV-X012"   # 2025
]

# Change this ID to inspect a different complete source.
REVIEW_ID = "SV-X010"

assert "usa_review_df" in globals()
assert "usa_patterns" in globals()
assert isinstance(usa_patterns, dict)

required_cols = [
    "review_id",
    "country_code",
    "year",
    "period",
    "speech_id",
    "segment_id",
    "topic03_share",
    "segment_text"
]

missing = [
    col for col in required_cols
    if col not in usa_review_df.columns
]

assert not missing, f"Missing columns: {missing}"

records = usa_review_df.loc[
    usa_review_df["review_id"].isin(REVIEW_IDS)
].copy()

assert len(records) == 4
assert records["review_id"].is_unique
assert set(records["country_code"]) == {"USA"}
assert set(records["year"].astype(int)) == {
    1995, 1997, 2011, 2025
}

records = records.sort_values("year").reset_index(drop=True)

# ------------------------------------------------------------
# 2. Comparative keyword audit
# ------------------------------------------------------------

keyword_rows = []

for _, row in records.iterrows():

    source_text = str(row["segment_text"])

    assert source_text.strip()
    assert source_text.lower() != "nan"

    result = {
        "review_id": row["review_id"],
        "year": int(row["year"]),
        "period": row["period"],
        "topic03_share": float(row["topic03_share"])
    }

    for category, pattern in usa_patterns.items():

        result[category] = len(
            list(
                re.finditer(
                    pattern,
                    source_text,
                    flags=re.IGNORECASE
                )
            )
        )

    keyword_rows.append(result)

usa_semantic_comparison_df = pd.DataFrame(keyword_rows)

print("=" * 72)
print("STEP 12.27H.6 — US HISTORICAL SEMANTIC DRIFT")
print("=" * 72)

print("\nSOURCE INVENTORY")

display(
    records[
        [
            "review_id",
            "year",
            "period",
            "speech_id",
            "segment_id",
            "topic03_share"
        ]
    ].round(4)
)

# ------------------------------------------------------------
# 3. Historical keyword comparison
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("HISTORICAL GEOGRAPHIC AND THEMATIC COMPARISON")
print("=" * 72)

keyword_columns = list(usa_patterns.keys())

comparison_long = usa_semantic_comparison_df.melt(
    id_vars=[
        "review_id",
        "year",
        "period",
        "topic03_share"
    ],
    value_vars=keyword_columns,
    var_name="category",
    value_name="mentions"
)

positive_evidence = comparison_long.loc[
    comparison_long["mentions"] > 0
].copy()

positive_evidence = positive_evidence.sort_values(
    ["year", "mentions", "category"],
    ascending=[True, False, True]
)

display(
    positive_evidence[
        ["year", "category", "mentions"]
    ].reset_index(drop=True)
)

# ------------------------------------------------------------
# 4. Compact cross-period diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("DOMINANT KEYWORD CATEGORIES BY YEAR")
print("=" * 72)

summary_rows = []

for _, row in usa_semantic_comparison_df.iterrows():

    counts = sorted(
        [
            (category, int(row[category]))
            for category in keyword_columns
            if row[category] > 0
        ],
        key=lambda x: (-x[1], x[0])
    )

    summary_rows.append({
        "year": int(row["year"]),
        "topic03_share": float(row["topic03_share"]),
        "top_keyword_categories": "; ".join(
            f"{name} ({count})"
            for name, count in counts[:4]
        ) if counts else "No dictionary matches"
    })

usa_semantic_summary_df = pd.DataFrame(summary_rows)

display(usa_semantic_summary_df)

# ------------------------------------------------------------
# 5. Full selected source inspection
# ------------------------------------------------------------

selected = records.loc[
    records["review_id"] == REVIEW_ID
]

assert len(selected) == 1, (
    "Selected REVIEW_ID is not part of the target sample."
)

source = selected.iloc[0]
source_text = str(source["segment_text"])

print("\n" + "=" * 72)
print("COMPLETE SELECTED ORIGINAL SOURCE SEGMENT")
print("=" * 72)

print("Review ID:", source["review_id"])
print("Country:", source["country_code"])
print("Year:", source["year"])
print("Period:", source["period"])
print("Speech ID:", source["speech_id"])
print("Segment ID:", source["segment_id"])
print(
    "Topic 03 share:",
    round(float(source["topic03_share"]), 4)
)

if "archive_path" in source.index:
    print("Source:", source["archive_path"])

print("\nORIGINAL TEXT:\n")

for paragraph in source_text.splitlines():

    if paragraph.strip():

        print(
            textwrap.fill(
                paragraph,
                width=95,
                break_long_words=False,
                break_on_hyphens=False
            )
        )

    else:
        print()

# ------------------------------------------------------------
# 6. Selected-source keyword-in-context evidence
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("SELECTED-SOURCE KEYWORD-IN-CONTEXT EVIDENCE")
print("=" * 72)

selected_categories = []

for category, pattern in usa_patterns.items():

    matches = list(
        re.finditer(
            pattern,
            source_text,
            flags=re.IGNORECASE
        )
    )

    if matches:
        selected_categories.append(
            (category, len(matches), matches)
        )

selected_categories.sort(
    key=lambda x: (-x[1], x[0])
)

for category, count, matches in selected_categories:

    print(f"\n{category}: {count} occurrence(s)")

    for match in matches[:2]:

        start = max(0, match.start() - 90)
        end = min(
            len(source_text),
            match.end() + 90
        )

        excerpt = re.sub(
            r"\s+",
            " ",
            source_text[start:end]
        ).strip()

        print(
            textwrap.fill(
                f"... {excerpt} ...",
                width=95
            )
        )

# ------------------------------------------------------------
# 7. Human review template
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 72)

usa_selected_human_review_df = pd.DataFrame([
    {
        "review_id": source["review_id"],
        "year": int(source["year"]),
        "topic03_share": float(source["topic03_share"]),
        "primary_geographic_focus": pd.NA,
        "principal_conflict_or_policy": pd.NA,
        "diplomatic_settlement_theme": pd.NA,
        "supports_broad_topic_label": pd.NA,
        "semantic_drift_assessment": pd.NA,
        "supporting_evidence": pd.NA,
        "reviewer_notes": pd.NA
    }
])

display(usa_selected_human_review_df)

# ------------------------------------------------------------
# 8. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

assert len(usa_semantic_comparison_df) == 4
assert set(records["country_code"]) == {"USA"}
assert len(usa_semantic_summary_df) == 4
assert len(usa_selected_human_review_df) == 1

print("United States source records: 4")
print("Historical years verified: 4")
print("Source metadata integrity: PASSED")
print("Keyword comparison: PASSED")
print("Selected text retrieval: PASSED")
print("Human semantic validation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.6: INSPECTION READY")



### Step 12.27H.6A — Source Completeness and Keyword Revalidation

**Objective:** Verify that the four selected US historical examples use complete, correctly matched original segments.

**Procedure:**
1. Match review records to the original segmented corpus using speech and segment identifiers.
2. Audit source-text completeness and detect possible truncation.
3. Recalculate geographic and thematic keyword frequencies from the recovered source texts.
4. Display the full original 1995 segment for qualitative inspection.

**Validation rule:** Do not finalize semantic interpretations when the original source text is incomplete or cannot be uniquely matched.

**Output:** In-memory diagnostic tables and original source text. No files are saved.

In [ ]:

# ============================================================
# STEP 12.27H.6A
# ORIGINAL SOURCE COMPLETENESS AUDIT
# NO FILES SAVED
# ============================================================

review_ids = ["SV-X010", "SV-X009", "SV-X011", "SV-X012"]

assert "usa_review_df" in globals()
assert "segments_df" in globals()
assert "usa_patterns" in globals()

keys = ["speech_id", "segment_id", "country_code", "year"]

required_segment_cols = keys + ["segment_text"]
assert all(c in segments_df.columns for c in required_segment_cols), (
    "Original segments_df does not contain all required columns."
)

sample = usa_review_df.loc[
    usa_review_df["review_id"].isin(review_ids)
].copy()

assert len(sample) == 4
assert sample["review_id"].is_unique

verified = []
audit = []

def normalize_text(s):
    return re.sub(r"\s+", " ", str(s)).strip()

for _, r in sample.iterrows():

    mask = pd.Series(True, index=segments_df.index)

    for key in keys:
        mask &= segments_df[key].eq(r[key])

    matched = segments_df.loc[mask]

    if len(matched) != 1:
        raise ValueError(
            f"{r['review_id']}: Expected exactly one original "
            f"segment; found {len(matched)}."
        )

    original = matched.iloc[0]

    preview = str(r["segment_text"])
    full_text = str(original["segment_text"])

    if not full_text.strip() or full_text.strip().lower() == "nan":
        raise ValueError(
            f"Invalid original text for {r['review_id']}"
        )

    old_norm = normalize_text(preview)
    new_norm = normalize_text(full_text)

    prefix_agrees = (
        old_norm[:min(80, len(old_norm))]
        == new_norm[:min(80, len(old_norm))]
    )

    rec = r.to_dict()
    rec["verified_segment_text"] = full_text
    verified.append(rec)

    audit.append({
        "review_id": r["review_id"],
        "year": int(r["year"]),
        "previous_chars": len(preview),
        "source_chars": len(full_text),
        "previous_ends_ellipsis": bool(
            re.search(r"\.\.\.\s*$|…\s*$", preview)
        ),
        "source_ends_ellipsis": bool(
            re.search(r"\.\.\.\s*$|…\s*$", full_text)
        ),
        "prefix_agrees": prefix_agrees
    })

usa_verified_sources_df = pd.DataFrame(verified)
usa_source_completeness_audit_df = pd.DataFrame(audit)

print("=" * 70)
print("STEP 12.27H.6A — ORIGINAL SOURCE COMPLETENESS")
print("=" * 70)

display(
    usa_source_completeness_audit_df
    .sort_values("year")
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Recalculate keyword frequencies from original sources
# ------------------------------------------------------------

rows = []

for _, r in usa_verified_sources_df.iterrows():

    text = r["verified_segment_text"]

    result = {
        "review_id": r["review_id"],
        "year": int(r["year"]),
        "topic03_share": float(r["topic03_share"])
    }

    for category, pattern in usa_patterns.items():
        result[category] = len(
            list(re.finditer(
                pattern,
                text,
                flags=re.IGNORECASE
            ))
        )

    rows.append(result)

usa_verified_keywords_df = pd.DataFrame(rows)

long_df = usa_verified_keywords_df.melt(
    id_vars=["review_id", "year", "topic03_share"],
    value_vars=list(usa_patterns),
    var_name="category",
    value_name="mentions"
)

long_df = (
    long_df.loc[long_df["mentions"] > 0]
    .sort_values(
        ["year", "mentions", "category"],
        ascending=[True, False, True]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("REVALIDATED HISTORICAL KEYWORD EVIDENCE")
print("=" * 70)

display(long_df)

# ------------------------------------------------------------
# Full 1995 original source
# ------------------------------------------------------------

record = usa_verified_sources_df.loc[
    usa_verified_sources_df["review_id"] == "SV-X010"
].iloc[0]

print("\n" + "=" * 70)
print("1995 — VERIFIED ORIGINAL SEGMENT")
print("=" * 70)

print("Review ID:", record["review_id"])
print("Year:", record["year"])
print("Speech ID:", record["speech_id"])
print("Segment ID:", record["segment_id"])
print("Topic 03 share:", round(record["topic03_share"], 4))

if "archive_path" in record.index:
    print("Source:", record["archive_path"])

print("\nCOMPLETE SOURCE TEXT:\n")

full_text = record["verified_segment_text"]

for paragraph in full_text.splitlines():
    if paragraph.strip():
        print(textwrap.fill(
            paragraph,
            width=100,
            break_long_words=False,
            break_on_hyphens=False
        ))
    else:
        print()

# ------------------------------------------------------------
# Final audit
# ------------------------------------------------------------

source_match_ok = (
    len(usa_verified_sources_df) == 4
    and usa_source_completeness_audit_df["prefix_agrees"].all()
)

source_ellipsis = (
    usa_source_completeness_audit_df["source_ends_ellipsis"].any()
)

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Original source matches:", len(usa_verified_sources_df))
print("Metadata/prefix verification:",
      "PASSED" if source_match_ok else "REVIEW REQUIRED")

print("Unresolved terminal ellipsis:",
      "YES — REVIEW REQUIRED" if source_ellipsis else "NO")

print("Keyword recalculation: COMPLETED")
print("Human semantic interpretation: PENDING")
print("Files saved: NONE")

if not source_match_ok or source_ellipsis:
    print("\nSOURCE COMPLETENESS: REVIEW REQUIRED")
else:
    print("\nSOURCE COMPLETENESS AUDIT: PASSED")

print("STEP 12.27H.6A: COMPUTATION COMPLETED")



### Step 12.27H.6B — US Historical Semantic Review: 1997

**Objective:** Examine the complete 1997 US speech segment associated with Topic 03 to assess its substantive meaning beyond keyword frequencies.

**Analytical tasks:**
1. Verify the selected source record and reference topic contribution.
2. Inspect the complete segment and its final text boundaries.
3. Identify geographic conflicts, diplomatic processes, and sovereignty-related arguments.
4. Extract contextual evidence for the strongest thematic keyword categories.
5. Prepare a human-review record without automatically assigning semantic judgments.

**Interpretive focus:** Assess whether Topic 03 represents Kampuchea-specific discourse or a broader dimension of international disputes and diplomatic settlements.

**Output:** Source audit, keyword evidence, complete segment, and an in-memory review template.

**Data policy:** No files are saved in this computational cell.


In [ ]:

# ============================================================
# STEP 12.27H.6B
# US HISTORICAL SEMANTIC REVIEW — 1997
# COMPUTATION ONLY — NO FILES SAVED
# ============================================================


REVIEW_ID = "SV-X009"

assert "usa_verified_sources_df" in globals()
assert "usa_patterns" in globals()

records = usa_verified_sources_df.loc[
    usa_verified_sources_df["review_id"] == REVIEW_ID
].copy()

assert len(records) == 1, (
    f"Expected exactly one record for {REVIEW_ID}"
)

r = records.iloc[0]
text = str(r["verified_segment_text"])

assert text.strip()
assert r["country_code"] == "USA"
assert int(r["year"]) == 1997

print("=" * 65)
print("STEP 12.27H.6B — US HISTORICAL SEMANTIC REVIEW")
print("=" * 65)

print("\nSOURCE IDENTIFICATION")
print("-" * 65)

for field in [
    "review_id",
    "country_code",
    "year",
    "period",
    "speech_id",
    "segment_id"
]:
    if field in r.index:
        print(f"{field}: {r[field]}")

print(f"Topic 03 share: {float(r['topic03_share']):.6f}")

if "archive_path" in r.index:
    print("Source:", r["archive_path"])

print("\nSOURCE BOUNDARY AUDIT")
print("-" * 65)

print("Character count:", len(text))
print("Word count:", len(text.split()))
print("Terminal ellipsis:", bool(
    re.search(r"(?:\.{3}|…)\s*$", text)
))
print("Literal ellipsis occurrences:",
      len(re.findall(r"\.{3}|…", text)))

print("\nLAST 300 SOURCE CHARACTERS:")
print(text[-300:])

# ------------------------------------------------------------
# Keyword evidence
# ------------------------------------------------------------

evidence = []

for category, pattern in usa_patterns.items():

    matches = list(
        re.finditer(pattern, text, flags=re.IGNORECASE)
    )

    evidence.append({
        "category": category,
        "mentions": len(matches)
    })

usa_1997_keyword_evidence_df = (
    pd.DataFrame(evidence)
    .sort_values(
        ["mentions", "category"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 65)
print("GEOGRAPHIC AND THEMATIC KEYWORD EVIDENCE")
print("=" * 65)

display(usa_1997_keyword_evidence_df)

# ------------------------------------------------------------
# Keyword-in-context evidence
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("KEYWORD-IN-CONTEXT EXCERPTS")
print("=" * 65)

for category, pattern in usa_patterns.items():

    matches = list(
        re.finditer(pattern, text, flags=re.IGNORECASE)
    )

    if not matches:
        continue

    print(f"\n{category}: {len(matches)} mention(s)")

    for match in matches[:2]:

        left = max(0, match.start() - 90)
        right = min(len(text), match.end() + 90)

        excerpt = re.sub(
            r"\s+", " ", text[left:right]
        ).strip()

        print("  ...", excerpt, "...")

# ------------------------------------------------------------
# Complete original source segment
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMPLETE ORIGINAL 1997 SOURCE SEGMENT")
print("=" * 65)

for paragraph in text.splitlines():
    if paragraph.strip():
        print(
            textwrap.fill(
                paragraph,
                width=95,
                break_long_words=False,
                break_on_hyphens=False
            )
        )
    else:
        print()

# ------------------------------------------------------------
# Human semantic review template
# ------------------------------------------------------------

usa_1997_semantic_review_df = pd.DataFrame([{
    "review_id": r["review_id"],
    "year": int(r["year"]),
    "topic03_share": float(r["topic03_share"]),
    "primary_geographic_focus": pd.NA,
    "principal_conflict_or_policy": pd.NA,
    "diplomatic_settlement_theme": pd.NA,
    "supports_broad_topic_label": pd.NA,
    "semantic_drift_assessment": pd.NA,
    "supporting_evidence": pd.NA,
    "reviewer_notes": pd.NA
}])

print("\n" + "=" * 65)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 65)

display(usa_1997_semantic_review_df)

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Unique source match: PASSED")
print("Reference topic share: VERIFIED")
print("Full cached segment retrieval: PASSED")
print("Keyword-context extraction: PASSED")
print("Independent raw-file verification: NOT PERFORMED")
print("Human semantic interpretation: PENDING")
print("Files saved: NONE")
print("\nSTEP 12.27H.6B: INSPECTION READY")



### Step 12.27H.6C — Contemporary US Semantic Validation (2011 and 2025)

**Objective:** Examine the substantive content of Topic 03 in two contemporary United States General Debate speeches (2011 and 2025), using the previously validated reference NMF solution.

**Analytical procedure:**
1. Verify the identities, years, and reference topic shares of the selected speech segments.
2. Inspect geographic and thematic keyword evidence.
3. Retrieve the complete cached segment texts and examine their boundaries.
4. Compare the semantic contexts of the 2011 and 2025 records.
5. Prepare a structured template for subsequent human semantic assessment.

**Interpretive focus:** Determine whether the contemporary American documents reflect international conflict resolution, diplomatic negotiation, regional crises, or other substantive themes.

**Methodological caution:** Keyword counts are exploratory indicators rather than independent semantic classifications. Retrieved segments must not automatically be treated as complete original speeches.

**Output:** Comparative diagnostic tables, source texts, boundary checks, and a human-review template.

**Data policy:** Computation only. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.6C
# CONTEMPORARY US SEMANTIC VALIDATION — 2011 & 2025
# COMPUTATION ONLY — NO FILES SAVED
# ============================================================


print("=" * 68)
print("STEP 12.27H.6C — CONTEMPORARY US SEMANTIC REVIEW")
print("=" * 68)

# ------------------------------------------------------------
# 1. Source validation
# ------------------------------------------------------------

assert "usa_verified_sources_df" in globals()
assert "usa_patterns" in globals()

required_columns = [
    "review_id", "country_code", "year", "period",
    "speech_id", "segment_id", "topic03_share",
    "verified_segment_text"
]

missing = [
    c for c in required_columns
    if c not in usa_verified_sources_df.columns
]

assert not missing, f"Missing columns: {missing}"

target_ids = ["SV-X011", "SV-X012"]

usa_contemporary_df = (
    usa_verified_sources_df.loc[
        usa_verified_sources_df["review_id"].isin(target_ids)
    ]
    .copy()
    .sort_values("year")
    .reset_index(drop=True)
)

assert len(usa_contemporary_df) == 2
assert usa_contemporary_df["review_id"].is_unique
assert set(usa_contemporary_df["year"].astype(int)) == {
    2011, 2025
}
assert (usa_contemporary_df["country_code"] == "USA").all()

print("\nSOURCE RECORD INVENTORY")

display(
    usa_contemporary_df[
        [
            "review_id", "year", "period",
            "speech_id", "segment_id", "topic03_share"
        ]
    ]
)

# ------------------------------------------------------------
# 2. Text boundary diagnostics
# ------------------------------------------------------------

boundary_records = []

for _, r in usa_contemporary_df.iterrows():

    source_text = str(r["verified_segment_text"])

    assert source_text.strip(), (
        f"Empty source text: {r['review_id']}"
    )

    stripped = source_text.rstrip()

    boundary_records.append({
        "review_id": r["review_id"],
        "year": int(r["year"]),
        "characters": len(source_text),
        "words": len(source_text.split()),
        "ends_sentence_punctuation": bool(
            re.search(r'[.!?]["\']?$', stripped)
        ),
        "terminal_ellipsis": bool(
            re.search(r"(?:\.{3}|…)$", stripped)
        ),
        "last_100_characters": stripped[-100:]
    })

usa_contemporary_boundaries_df = pd.DataFrame(
    boundary_records
)

print("\n" + "=" * 68)
print("CACHED SEGMENT BOUNDARY AUDIT")
print("=" * 68)

display(usa_contemporary_boundaries_df)

# ------------------------------------------------------------
# 3. Comparative keyword diagnostics
# ------------------------------------------------------------

keyword_records = []

for _, r in usa_contemporary_df.iterrows():

    source_text = str(r["verified_segment_text"])

    for category, pattern in usa_patterns.items():

        matches = list(
            re.finditer(
                pattern,
                source_text,
                flags=re.IGNORECASE
            )
        )

        keyword_records.append({
            "review_id": r["review_id"],
            "year": int(r["year"]),
            "category": category,
            "mentions": len(matches)
        })

usa_contemporary_keywords_df = pd.DataFrame(
    keyword_records
)

comparison = (
    usa_contemporary_keywords_df
    .pivot_table(
        index="category",
        columns="year",
        values="mentions",
        aggfunc="sum",
        fill_value=0
    )
    .reset_index()
)

comparison.columns = [
    str(c) for c in comparison.columns
]

comparison["total_mentions"] = (
    comparison["2011"] + comparison["2025"]
)

comparison = (
    comparison
    .sort_values(
        ["total_mentions", "category"],
        ascending=[False, True]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 68)
print("COMPARATIVE GEOGRAPHIC AND THEMATIC EVIDENCE")
print("=" * 68)

display(comparison)

# ------------------------------------------------------------
# 4. Full cached segments + contextual evidence
# ------------------------------------------------------------

for _, r in usa_contemporary_df.iterrows():

    source_text = str(r["verified_segment_text"])

    print("\n" + "=" * 68)
    print(
        f"ORIGINAL CACHED SEGMENT — USA {int(r['year'])}"
    )
    print("=" * 68)

    print("Review ID:", r["review_id"])
    print("Speech ID:", r["speech_id"])
    print("Segment ID:", r["segment_id"])
    print(
        "Topic 03 share:",
        f"{float(r['topic03_share']):.6f}"
    )

    if "archive_path" in r.index:
        print("Source:", r["archive_path"])

    print("\nSOURCE TEXT:\n")

    for paragraph in source_text.splitlines():

        if paragraph.strip():
            print(
                textwrap.fill(
                    paragraph,
                    width=95,
                    break_long_words=False,
                    break_on_hyphens=False
                )
            )
        else:
            print()

    print("\nTOP KEYWORD-IN-CONTEXT EVIDENCE")
    print("-" * 68)

    year_keywords = (
        usa_contemporary_keywords_df.loc[
            (usa_contemporary_keywords_df["year"] == int(r["year"]))
            & (usa_contemporary_keywords_df["mentions"] > 0)
        ]
        .sort_values("mentions", ascending=False)
    )

    for _, item in year_keywords.head(6).iterrows():

        category = item["category"]
        pattern = usa_patterns[category]

        matches = list(
            re.finditer(
                pattern,
                source_text,
                flags=re.IGNORECASE
            )
        )

        print(
            f"\n{category}: {len(matches)} occurrence(s)"
        )

        for match in matches[:2]:

            left = max(0, match.start() - 80)
            right = min(
                len(source_text),
                match.end() + 80
            )

            excerpt = re.sub(
                r"\s+",
                " ",
                source_text[left:right]
            ).strip()

            print("...", excerpt, "...")

# ------------------------------------------------------------
# 5. Human semantic review template
# ------------------------------------------------------------

review_records = []

for _, r in usa_contemporary_df.iterrows():

    review_records.append({
        "review_id": r["review_id"],
        "country_code": r["country_code"],
        "year": int(r["year"]),
        "period": r["period"],
        "speech_id": r["speech_id"],
        "segment_id": r["segment_id"],
        "topic03_share": float(r["topic03_share"]),
        "primary_geographic_focus": pd.NA,
        "principal_conflict_or_policy": pd.NA,
        "diplomatic_settlement_theme": pd.NA,
        "supports_broad_topic_label": pd.NA,
        "semantic_drift_assessment": pd.NA,
        "supporting_evidence": pd.NA,
        "reviewer_notes": pd.NA
    })

usa_contemporary_review_df = pd.DataFrame(
    review_records
)

print("\n" + "=" * 68)
print("HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 68)

display(usa_contemporary_review_df)

# ------------------------------------------------------------
# 6. Final computational audit
# ------------------------------------------------------------

assert len(usa_contemporary_keywords_df) == (
    2 * len(usa_patterns)
)

assert usa_contemporary_boundaries_df[
    "characters"
].gt(0).all()

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Reference source identification: PASSED")
print("Contemporary year coverage: PASSED")
print("Cached segment retrieval: PASSED")
print("Keyword-context extraction: PASSED")
print("Source boundary diagnostics: COMPLETED")
print("Independent raw-file verification: NOT PERFORMED")
print("Human semantic interpretation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.6C: INSPECTION READY")



### Step 12.27H.7 — Cross-Country Semantic Consolidation of Topic 03

**Objective:** Consolidate the previously inspected Topic 03 source segments from Iran, the United States, and China across three historical periods (1972–1990, 1991–2010, and 2011–2025).

**Analytical procedure:**
1. Reidentify the 18 previously selected source segments using their original speech and segment identifiers.
2. Verify country, year, and source-text consistency.
3. Recover reference Topic 03 contributions for descriptive comparison.
4. Produce harmonized geographic and thematic keyword diagnostics.
5. Construct an integrated, source-linked human semantic review inventory.
6. Distinguish evidence for the original Kampuchea-specific interpretation from evidence for a broader international-conflict and diplomacy interpretation.

**Key methodological distinction:** Semantic variation within a fixed NMF component does not establish statistical topic drift. Human assessment, temporal representativeness, and broader corpus-level evidence are necessary before assigning a final topic label.

**Validation policy:** Missing source matches, inconsistent metadata, or incompatible model dimensions must be reported rather than silently corrected.

**Outputs:** Cross-country review inventory, comparative thematic diagnostics, and an uncompleted human-coding template.

**Data policy:** Computation only. File export will be handled in an independent cell after validation.


In [ ]:

# ============================================================
# STEP 12.27H.7
# CROSS-COUNTRY SEMANTIC CONSOLIDATION — TOPIC 03
# COMPUTATION ONLY — NO FILES SAVED
# ============================================================

print("=" * 70)
print("STEP 12.27H.7 — CROSS-COUNTRY SEMANTIC CONSOLIDATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate original segment data
# ------------------------------------------------------------

assert "segments_df" in globals(), "segments_df not found."
assert "component_contributions" in globals(), (
    "Recovered component_contributions not found."
)

required = [
    "speech_id", "segment_id", "country_code",
    "year", "segment_text"
]

missing = [c for c in required if c not in segments_df.columns]
assert not missing, f"Missing columns: {missing}"

source = segments_df.reset_index(drop=True).copy()

source["speech_id"] = source["speech_id"].astype(int)
source["segment_id"] = source["segment_id"].astype(int)
source["year"] = source["year"].astype(int)

assert not source.duplicated(
    ["speech_id", "segment_id"]
).any(), "Duplicate segment identifiers found."

W = np.asarray(component_contributions, dtype=float)

assert W.shape == (len(source), 7), (
    f"Unexpected contribution matrix: {W.shape}"
)
assert np.isfinite(W).all()
assert W.min() >= -1e-10

W = np.maximum(W, 0)

row_totals = W.sum(axis=1)

assert np.all(row_totals > 0), (
    "Zero-contribution rows found."
)

source["topic03_share_recovered"] = (
    W[:, 2] / row_totals
)

# ------------------------------------------------------------
# 2. Previously selected 18 review segments
# ------------------------------------------------------------

sample_records = [
    # Iran
    ("IRN", 1977, 16, 2, "SV-X001"),
    ("IRN", 1981, 28, 23, "SV-X002"),
    ("IRN", 1991, 58, 12, "SV-X003"),
    ("IRN", 1992, 61, 9, "SV-X004"),
    ("IRN", 2025, 160, 5, "SV-X005"),
    ("IRN", 2016, 133, 2, "SV-X006"),

    # United States
    ("USA", 1972, 2, 6, "SV-X007"),
    ("USA", 1975, 11, 13, "SV-X008"),
    ("USA", 1995, 71, 5, "SV-X010"),
    ("USA", 1997, 77, 4, "SV-X009"),
    ("USA", 2011, 119, 11, "SV-X011"),
    ("USA", 2025, 161, 4, "SV-X012"),

    # China
    ("CHN", 1981, 27, 4, "SV-013"),
    ("CHN", 1988, 48, 9, "SV-014"),
    ("CHN", 1991, 57, 5, "SV-015"),
    ("CHN", 1999, 81, 5, "SV-016"),
    ("CHN", 2014, 126, 6, "SV-017"),
    ("CHN", 2022, 150, 9, "SV-018"),
]

samples = pd.DataFrame(
    sample_records,
    columns=[
        "country_code", "year", "speech_id",
        "segment_id", "review_id"
    ]
)

matched = samples.merge(
    source[
        [
            "country_code", "year",
            "speech_id", "segment_id",
            "segment_text",
            "topic03_share_recovered"
        ] + (
            ["archive_path"]
            if "archive_path" in source.columns else []
        )
    ],
    on=["speech_id", "segment_id"],
    how="left",
    suffixes=("_expected", ""),
    validate="one_to_one",
    indicator=True
)

assert (matched["_merge"] == "both").all(), (
    "One or more original segments could not be recovered."
)

assert (
    matched["country_code_expected"]
    == matched["country_code"]
).all(), "Country mismatch detected."

assert (
    matched["year_expected"]
    == matched["year"]
).all(), "Year mismatch detected."

matched["country_code"] = matched[
    "country_code_expected"
]

matched = matched.drop(
    columns=[
        "country_code_expected",
        "year_expected",
        "_merge"
    ]
)

assert matched["segment_text"].notna().all()

matched["period"] = pd.cut(
    matched["year"],
    bins=[1971, 1990, 2010, 2025],
    labels=[
        "1972-1990",
        "1991-2010",
        "2011-2025"
    ]
)

assert matched["period"].notna().all()

assert len(matched) == 18

# ------------------------------------------------------------
# 3. Review source inventory
# ------------------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-COUNTRY SOURCE INVENTORY")
print("=" * 70)

display(
    matched[
        [
            "review_id",
            "country_code",
            "period",
            "year",
            "speech_id",
            "segment_id",
            "topic03_share_recovered"
        ]
    ].sort_values(
        ["country_code", "year"]
    ).reset_index(drop=True)
)

coverage = (
    matched.groupby(
        ["country_code", "period"],
        observed=True
    )
    .agg(
        n_records=("review_id", "count"),
        n_speeches=("speech_id", "nunique"),
        mean_topic_share=(
            "topic03_share_recovered", "mean"
        )
    )
    .reset_index()
)

assert len(coverage) == 9
assert (coverage["n_records"] == 2).all()

print("\nHISTORICAL SAMPLING COVERAGE")
display(coverage)

# ------------------------------------------------------------
# 4. Harmonized keyword categories
# ------------------------------------------------------------

semantic_patterns = {
    "Cambodia / Kampuchea":
        r"\b(?:cambodia|kampuchea|kampuchean)\b",

    "Afghanistan":
        r"\b(?:afghanistan|afghan)\b",

    "Palestine / Israel":
        r"\b(?:palestine|palestinian|israel|israeli)\w*\b",

    "Korean Peninsula":
        r"\b(?:korea|korean)\b",

    "Vietnam":
        r"\b(?:vietnam|vietnamese)\b",

    "Middle East":
        r"\bmiddle[\s-]+east\b",

    "Bosnia / Kosovo / Yugoslavia":
        r"\b(?:bosnia|bosnian|kosovo|yugoslavia|yugoslav)\w*\b",

    "South Africa / Apartheid":
        r"\b(?:south africa|apartheid)\b",

    "Sovereignty / Intervention":
        r"\b(?:sovereignty|sovereign|intervention|interference)\b",

    "Diplomatic Settlement":
        r"\b(?:peace|settlement|negotiat\w*|dialogue|"
        r"dialog|mediat\w*|diplomac\w*|ceasefire)\b",

    "Armed Conflict":
        r"\b(?:war|warfare|armed|aggression|"
        r"military|combat|hostilities)\b",

    "Humanitarian Context":
        r"\b(?:humanitarian|refugee\w*|"
        r"civilian\w*|famine|starvation)\b",
}

keyword_records = []

for _, row in matched.iterrows():

    content = str(row["segment_text"])

    for category, pattern in semantic_patterns.items():

        mentions = len(
            re.findall(
                pattern,
                content,
                flags=re.IGNORECASE
            )
        )

        keyword_records.append({
            "review_id": row["review_id"],
            "country_code": row["country_code"],
            "year": int(row["year"]),
            "period": str(row["period"]),
            "category": category,
            "mentions": mentions
        })

semantic_keyword_long_df = pd.DataFrame(
    keyword_records
)

# ------------------------------------------------------------
# 5. Country-period comparison
# ------------------------------------------------------------

country_period_keywords = (
    semantic_keyword_long_df
    .groupby(
        ["country_code", "period", "category"],
        as_index=False
    )["mentions"]
    .sum()
)

semantic_comparison_df = (
    country_period_keywords
    .pivot_table(
        index=["country_code", "period"],
        columns="category",
        values="mentions",
        fill_value=0,
        observed=True
    )
    .reset_index()
)

semantic_comparison_df.columns.name = None

print("\n" + "=" * 70)
print("CROSS-COUNTRY AND HISTORICAL KEYWORD DIAGNOSTICS")
print("=" * 70)

display(semantic_comparison_df)

# ------------------------------------------------------------
# 6. Integrated manual-review inventory
# ------------------------------------------------------------

semantic_master_review_df = matched[
    [
        "review_id",
        "country_code",
        "year",
        "period",
        "speech_id",
        "segment_id",
        "topic03_share_recovered"
    ]
].copy()

manual_fields = [
    "primary_geographic_focus",
    "principal_conflict_or_policy",
    "diplomatic_or_peace_process",
    "supports_kampuchea_specific_label",
    "supports_broad_conflict_peace_label",
    "semantic_interpretation",
    "supporting_evidence",
    "reviewer_notes"
]

for field in manual_fields:
    semantic_master_review_df[field] = pd.NA

semantic_master_review_df = (
    semantic_master_review_df
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("INTEGRATED HUMAN SEMANTIC REVIEW TEMPLATE")
print("=" * 70)

display(semantic_master_review_df)

# ------------------------------------------------------------
# 7. Inspect one original cached segment
# ------------------------------------------------------------

# Change this ID to inspect any of the 18 records.
REVIEW_ID_TO_INSPECT = "SV-X012"

selected = matched.loc[
    matched["review_id"] == REVIEW_ID_TO_INSPECT
]

assert len(selected) == 1

r = selected.iloc[0]
content = str(r["segment_text"])

print("\n" + "=" * 70)
print("SELECTED ORIGINAL CACHED SEGMENT")
print("=" * 70)

print("Review ID:", r["review_id"])
print("Country:", r["country_code"])
print("Year:", r["year"])
print("Speech ID:", r["speech_id"])
print("Segment ID:", r["segment_id"])

if "archive_path" in r.index:
    print("Source:", r["archive_path"])

print("\nCACHED SEGMENT TEXT:\n")

for paragraph in content.splitlines():
    if paragraph.strip():
        print(
            textwrap.fill(
                paragraph,
                width=95,
                break_long_words=False,
                break_on_hyphens=False
            )
        )
    else:
        print()

print("\nSEGMENT BOUNDARY DIAGNOSTICS")
print("Characters:", len(content))
print("Words:", len(content.split()))
print("First 100 characters:", repr(content[:100]))
print("Last 100 characters:", repr(content[-100:]))

# ------------------------------------------------------------
# 8. Final audit
# ------------------------------------------------------------

assert semantic_master_review_df["review_id"].is_unique
assert len(semantic_master_review_df) == 18
assert len(semantic_keyword_long_df) == (
    18 * len(semantic_patterns)
)

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Original segment identification: PASSED")
print("Country-year linkage: PASSED")
print("Historical coverage: PASSED")
print("Keyword diagnostics: PASSED")
print("Manual review inventory: PREPARED")
print("Human semantic judgments: PENDING")
print("Original full-speech verification: NOT PERFORMED")
print("Files saved: NONE")

print("\nSTEP 12.27H.7: COMPUTATION COMPLETED")



### Step 12.27H.8 — Semantic Coherence Audit of Topic 03

**Objective:** Evaluate whether Topic 03 retains a coherent substantive interpretation across countries and historical periods.

**Dataset:** 18 previously verified source segments, stratified by country (Iran, United States, China) and historical period (1972–1990, 1991–2010, 2011–2025).

**Procedure:**
1. Revalidate all 18 source identifiers and recovered Topic 03 contributions.
2. Examine the presence of Kampuchea-specific, regional-conflict, sovereignty, and diplomatic-settlement vocabulary.
3. Identify potentially contradictory or weakly relevant source segments for qualitative inspection.
4. Retrieve complete cached source segments for case-by-case human review.
5. Prepare a structured semantic adjudication template without assigning automatic qualitative judgments.

**Coding dimensions:**
- Principal geographic focus
- Principal substantive issue
- Diplomatic or peace-settlement context
- Support for the Kampuchea-specific label
- Support for a broader international-conflict and diplomacy label
- Potential semantic heterogeneity
- Direct supporting textual evidence

**Interpretation:** Keyword frequencies are descriptive screening indicators, not validated semantic classifications. Selected segments are not a representative random sample of the entire corpus. Statistical historical contrasts remain unchanged.

**Decision rule:** No final topic label will be assigned before reviewing the substantive source evidence.

**Data policy:** Computation and inspection only. No files will be saved in this cell.


In [ ]:

# ============================================================
# STEP 12.27H.8
# TOPIC 03 — SEMANTIC COHERENCE AND HUMAN REVIEW AUDIT
# COMPUTATION ONLY — NO FILES SAVED
# ============================================================


print("=" * 70)
print("STEP 12.27H.8 — SEMANTIC COHERENCE AUDIT")
print("=" * 70)

# ------------------------------------------------------------
# 1. Validate previous outputs
# ------------------------------------------------------------

assert "matched" in globals()
assert "semantic_keyword_long_df" in globals()

review_source = matched.copy()
keyword_source = semantic_keyword_long_df.copy()

assert len(review_source) == 18
assert review_source["review_id"].is_unique
assert review_source["segment_text"].notna().all()

assert set(review_source["country_code"]) == {
    "IRN", "USA", "CHN"
}

print("\nSOURCE VALIDATION: PASSED")
print("Review records:", len(review_source))
print("Countries:", review_source["country_code"].nunique())
print("Historical periods:", review_source["period"].nunique())

# ------------------------------------------------------------
# 2. Assemble comparable evidence categories
# ------------------------------------------------------------

selected_categories = [
    "Cambodia / Kampuchea",
    "Palestine / Israel",
    "Korean Peninsula",
    "Middle East",
    "Diplomatic Settlement",
    "Armed Conflict",
    "Sovereignty / Intervention"
]

assert set(selected_categories).issubset(
    set(keyword_source["category"])
)

keyword_wide = (
    keyword_source[
        keyword_source["category"].isin(selected_categories)
    ]
    .pivot(
        index="review_id",
        columns="category",
        values="mentions"
    )
    .fillna(0)
    .astype(int)
    .reset_index()
)

semantic_case_audit_df = review_source[
    [
        "review_id",
        "country_code",
        "period",
        "year",
        "speech_id",
        "segment_id",
        "topic03_share_recovered"
    ]
].merge(
    keyword_wide,
    on="review_id",
    how="left",
    validate="one_to_one"
)

assert semantic_case_audit_df[
    selected_categories
].notna().all().all()

semantic_case_audit_df = (
    semantic_case_audit_df
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("SOURCE-LEVEL THEMATIC EVIDENCE")
print("=" * 70)

display(
    semantic_case_audit_df[
        [
            "review_id",
            "country_code",
            "year",
            "topic03_share_recovered"
        ] + selected_categories
    ]
)

# ------------------------------------------------------------
# 3. Kampuchea-specific label screening
# ------------------------------------------------------------

label_screening_df = (
    semantic_case_audit_df
    .groupby(
        ["country_code", "period"],
        observed=True
    )
    .agg(
        n_segments=("review_id", "count"),
        kampuchea_mentions=(
            "Cambodia / Kampuchea", "sum"
        ),
        kampuchea_positive_segments=(
            "Cambodia / Kampuchea",
            lambda x: int((x > 0).sum())
        ),
        palestine_mentions=(
            "Palestine / Israel", "sum"
        ),
        diplomacy_mentions=(
            "Diplomatic Settlement", "sum"
        ),
        conflict_mentions=(
            "Armed Conflict", "sum"
        )
    )
    .reset_index()
)

print("\n" + "=" * 70)
print("TOPIC-LABEL SCREENING BY COUNTRY AND PERIOD")
print("=" * 70)

display(label_screening_df)

# ------------------------------------------------------------
# 4. Source-level cases needing qualitative attention
# ------------------------------------------------------------

semantic_case_audit_df[
    "kampuchea_keyword_present"
] = (
    semantic_case_audit_df[
        "Cambodia / Kampuchea"
    ] > 0
)

# Keyword absence is a review flag, not a semantic judgment.
semantic_case_audit_df[
    "requires_substantive_review"
] = True

print("\n" + "=" * 70)
print("KAMPUCHEA KEYWORD COVERAGE")
print("=" * 70)

print(
    "Segments with a Kampuchea/Cambodia mention:",
    int(
        semantic_case_audit_df[
            "kampuchea_keyword_present"
        ].sum()
    ),
    "/ 18"
)

print(
    "Segments without a Kampuchea/Cambodia mention:",
    int(
        (
            ~semantic_case_audit_df[
                "kampuchea_keyword_present"
            ]
        ).sum()
    ),
    "/ 18"
)

print(
    "\nNote: This is lexical screening only; "
    "absence of a keyword does not establish "
    "absence of substantive relevance."
)

# ------------------------------------------------------------
# 5. Prepare human semantic adjudication template
# ------------------------------------------------------------

semantic_adjudication_df = review_source[
    [
        "review_id",
        "country_code",
        "year",
        "period",
        "speech_id",
        "segment_id",
        "topic03_share_recovered",
        "segment_text"
    ]
].copy()

coding_fields = [
    "principal_geographic_focus",
    "principal_substantive_issue",
    "diplomatic_or_peace_process",
    "kampuchea_label_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "supporting_text",
    "reviewer_notes"
]

for col in coding_fields:
    semantic_adjudication_df[col] = pd.NA

semantic_adjudication_df = (
    semantic_adjudication_df
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)

print("\n" + "=" * 70)
print("HUMAN ADJUDICATION INVENTORY")
print("=" * 70)

display(
    semantic_adjudication_df.drop(
        columns=["segment_text"]
    )
)

# ------------------------------------------------------------
# 6. Inspect selected original segments
# ------------------------------------------------------------

# Change this list to inspect other cases.
# Maximum two records recommended per execution
# to avoid notebook output truncation.

REVIEW_IDS_TO_INSPECT = [
    "SV-X002",  # Iran 1981
    "SV-X012"   # United States 2025
]

assert len(REVIEW_IDS_TO_INSPECT) == len(
    set(REVIEW_IDS_TO_INSPECT)
)

assert set(REVIEW_IDS_TO_INSPECT).issubset(
    set(review_source["review_id"])
)

for review_id in REVIEW_IDS_TO_INSPECT:

    row = review_source.loc[
        review_source["review_id"] == review_id
    ].iloc[0]

    content = str(row["segment_text"])

    print("\n" + "=" * 70)
    print("ORIGINAL CACHED SOURCE SEGMENT")
    print("=" * 70)

    print("Review ID:", review_id)
    print("Country:", row["country_code"])
    print("Year:", row["year"])
    print("Period:", row["period"])
    print("Speech ID:", row["speech_id"])
    print("Segment ID:", row["segment_id"])

    print(
        "Topic 03 share:",
        round(
            float(row["topic03_share_recovered"]),
            6
        )
    )

    if "archive_path" in row.index:
        print("Source:", row["archive_path"])

    print("\nORIGINAL CACHED TEXT:\n")

    for paragraph in content.splitlines():
        if paragraph.strip():
            print(
                textwrap.fill(
                    paragraph,
                    width=100,
                    break_long_words=False,
                    break_on_hyphens=False
                )
            )
        else:
            print()

    print("\nSOURCE DIAGNOSTICS")
    print("Characters:", len(content))
    print("Words:", len(content.split()))
    print("Cached segment recovered: YES")
    print("Independent raw-file verification: NOT PERFORMED")

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

assert len(semantic_case_audit_df) == 18
assert len(semantic_adjudication_df) == 18
assert semantic_adjudication_df["review_id"].is_unique

assert all(
    semantic_adjudication_df[col].isna().all()
    for col in coding_fields
)

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Cross-country source coverage: PASSED")
print("Country-period stratification: PASSED")
print("Reference segment retrieval: PASSED")
print("Keyword screening: PASSED")
print("Qualitative review template: PREPARED")
print("Human semantic validation: PENDING")
print("Final topic label: NOT ASSIGNED")
print("Existing statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.8: COMPUTATION COMPLETED")



### Step 12.27H.9 — Targeted Human Semantic Adjudication

**Objective:** Examine substantively divergent source segments assigned to Topic 03 and evaluate the adequacy of its provisional label.

**Priority cases:**
- China, 1999 (SV-016): Sovereignty, intervention, and international conflict resolution.
- Iran, 1981 (SV-X002): South African apartheid and Namibia.
- United States, 2025 (SV-X012): Contemporary foreign policy and international security.

**Review dimensions:**
1. Principal geographic focus and substantive issue.
2. Relevance to a Kampuchea-specific topic label.
3. Relevance to a broader international conflict, sovereignty, and diplomacy construct.
4. Evidence of semantic heterogeneity.
5. Direct textual evidence supporting the interpretation.

**Methodological safeguards:**
- Examine original cached source segments without altering them.
- Distinguish lexical evidence from substantive interpretation.
- Do not automatically assign human qualitative judgments.
- Do not generalize three case studies to the full historical corpus.
- Preserve all existing NMF solutions and statistical estimates.

**Expected output:** Three complete source excerpts, associated diagnostic evidence, and a blank human adjudication table.

**Data policy:** No files are saved. Export will be handled in a separate cell after review.


In [ ]:

# ============================================================
# STEP 12.27H.9
# TARGETED HUMAN SEMANTIC ADJUDICATION
# COMPUTATION ONLY — NO FILES SAVED
# ============================================================

print("=" * 70)
print("STEP 12.27H.9 — HUMAN SEMANTIC ADJUDICATION")
print("=" * 70)

# ------------------------------------------------------------
# 1. Verify previous analysis objects
# ------------------------------------------------------------

required_objects = [
    "semantic_adjudication_df",
    "semantic_case_audit_df"
]

for name in required_objects:
    assert name in globals(), (
        f"Missing object: {name}. "
        "Run Step 12.27H.8 first."
    )

source_df = semantic_adjudication_df.copy()
evidence_df = semantic_case_audit_df.copy()

assert len(source_df) == 18
assert source_df["review_id"].is_unique
assert source_df["segment_text"].notna().all()

# ------------------------------------------------------------
# 2. Select three priority cases
# ------------------------------------------------------------

REVIEW_CASES = [
    "SV-016",   # China, 1999
    "SV-X002",  # Iran, 1981
    "SV-X012"   # United States, 2025
]

assert len(REVIEW_CASES) == len(set(REVIEW_CASES))
assert set(REVIEW_CASES).issubset(
    set(source_df["review_id"])
)

selected = (
    source_df
    .set_index("review_id")
    .loc[REVIEW_CASES]
    .reset_index()
)

print("\nPRIORITY SOURCE INVENTORY")

display(
    selected[
        [
            "review_id",
            "country_code",
            "year",
            "period",
            "speech_id",
            "segment_id",
            "topic03_share_recovered"
        ]
    ]
)

# ------------------------------------------------------------
# 3. Comparative keyword evidence
# ------------------------------------------------------------

keyword_columns = [
    "Cambodia / Kampuchea",
    "Palestine / Israel",
    "Korean Peninsula",
    "Middle East",
    "Diplomatic Settlement",
    "Armed Conflict",
    "Sovereignty / Intervention"
]

comparison = (
    evidence_df
    .set_index("review_id")
    .loc[REVIEW_CASES]
    .reset_index()
)

print("\n" + "=" * 70)
print("COMPARATIVE KEYWORD EVIDENCE")
print("=" * 70)

display(
    comparison[
        ["review_id", "country_code", "year"]
        + keyword_columns
    ]
)

# ------------------------------------------------------------
# 4. Display complete cached source segments
# ------------------------------------------------------------

for _, row in selected.iterrows():

    content = str(row["segment_text"])

    print("\n" + "=" * 70)
    print("ORIGINAL CACHED SOURCE SEGMENT")
    print("=" * 70)

    print("Review ID:", row["review_id"])
    print("Country:", row["country_code"])
    print("Year:", row["year"])
    print("Historical period:", row["period"])
    print("Speech ID:", row["speech_id"])
    print("Segment ID:", row["segment_id"])
    print(
        "Topic 03 share:",
        f"{float(row['topic03_share_recovered']):.6f}"
    )

    print("\nCOMPLETE CACHED SEGMENT TEXT:\n")

    for paragraph in content.splitlines():
        if paragraph.strip():
            print(
                textwrap.fill(
                    paragraph,
                    width=105,
                    break_long_words=False,
                    break_on_hyphens=False
                )
            )
        else:
            print()

    print("\nTEXT DIAGNOSTICS")
    print("Characters:", len(content))
    print("Words:", len(content.split()))
    print("Cached segment retrieved: YES")
    print("Independent raw-file verification: NOT PERFORMED")
    print("Human semantic judgment: PENDING")

# ------------------------------------------------------------
# 5. Prepare blank human adjudication records
# ------------------------------------------------------------

coding_fields = [
    "principal_geographic_focus",
    "principal_substantive_issue",
    "diplomatic_or_peace_process",
    "kampuchea_label_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "supporting_text",
    "reviewer_notes"
]

semantic_review_batch_129 = selected[
    [
        "review_id",
        "country_code",
        "year",
        "period",
        "topic03_share_recovered"
    ]
].copy()

for field in coding_fields:
    semantic_review_batch_129[field] = pd.NA

print("\n" + "=" * 70)
print("HUMAN SEMANTIC CODING TEMPLATE")
print("=" * 70)

display(semantic_review_batch_129)

# ------------------------------------------------------------
# 6. Validation
# ------------------------------------------------------------

assert len(selected) == 3
assert selected["review_id"].is_unique

assert all(
    semantic_review_batch_129[field].isna().all()
    for field in coding_fields
)

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Source record matching: PASSED")
print("Cached text retrieval: PASSED")
print("Comparative keyword evidence: PASSED")
print("Human adjudication template: PREPARED")
print("Automatic qualitative classifications: NONE")
print("Final Topic 03 label: PENDING")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.9: INSPECTION READY")



### Step 12.27H.10 — Structured Human Semantic Adjudication

**Objective:** Prepare a reproducible, source-grounded human review of Topic 03 across Iran, the United States, and China.

This step will:

1. Recover the 18 previously selected semantic review records.
2. Match each record to its original segment text.
3. Verify the country, year, speech, and segment identifiers.
4. Establish a standardized semantic adjudication framework.
5. Prioritize three potentially heterogeneous records (CHN 1999, IRN 1981, USA 2025).
6. Prepare explicit human coding fields without automatically assigning semantic judgments.

**Methodological safeguards**

- Topic shares are statistical model outputs, not direct evidence of semantic validity.
- Keyword matches are screening evidence, not human semantic classifications.
- Kampuchea-specific relevance must be distinguished from broader conflict, sovereignty, and diplomatic discourse.
- Existing NMF solutions and inferential results remain unchanged.
- Human semantic validation remains pending until the source texts are adjudicated.

**Output:** Verified source inventory, standardized review template, and priority source evidence.

**Saving:** Disabled. Export will be handled in a separate cell after validation.


In [ ]:


print("=" * 70)
print("STEP 12.27H.10 — STRUCTURED HUMAN SEMANTIC ADJUDICATION")
print("=" * 70)

# --------------------------------------------------
# 1. Locate the complete semantic-review inventory
# --------------------------------------------------

expected_ids = {
    "SV-013", "SV-014", "SV-015",
    "SV-016", "SV-017", "SV-018",
    *[f"SV-X{i:03d}" for i in range(1, 13)]
}

required = {
    "review_id", "country_code", "year",
    "speech_id", "segment_id"
}

candidates = []

for name, obj in list(globals().items()):
    if not isinstance(obj, pd.DataFrame):
        continue

    if not required.issubset(obj.columns):
        continue

    ids = set(obj["review_id"].dropna().astype(str))

    if not expected_ids.issubset(ids):
        continue

    priority = (
        10 * ("semantic" in name.lower())
        + 5 * ("review" in name.lower())
        + 2 * ("topic03_share_recovered" in obj.columns)
        + int(len(obj) == 18)
    )

    candidates.append((priority, name, obj))

if not candidates:
    raise RuntimeError(
        "Complete 18-record review inventory not found. "
        "Check that Step 12.27H.8 was executed."
    )

candidates.sort(key=lambda x: (-x[0], x[1]))

source_name = candidates[0][1]
source_df = candidates[0][2].copy()

review_df_12_27h10 = source_df[
    source_df["review_id"].astype(str).isin(expected_ids)
].copy()

review_df_12_27h10 = (
    review_df_12_27h10
    .drop_duplicates(subset=["review_id"])
    .sort_values(["country_code", "year", "review_id"])
    .reset_index(drop=True)
)

assert len(review_df_12_27h10) == 18
assert set(review_df_12_27h10["review_id"]) == expected_ids

print("\nReview inventory:", source_name)
print("Verified records:", len(review_df_12_27h10))

# --------------------------------------------------
# 2. Recover original source segments
# --------------------------------------------------

if "segments_df" not in globals():
    raise RuntimeError("segments_df is unavailable.")

segment_source = segments_df.copy()

text_options = [
    "segment_text", "text", "clean_text",
    "original_text", "segment_content"
]

text_column = next(
    (c for c in text_options if c in segment_source.columns),
    None
)

if text_column is None:
    raise RuntimeError(
        "Original segment text column not found. "
        f"Available columns: {list(segment_source.columns)}"
    )

keys = ["speech_id", "segment_id"]

assert all(c in segment_source.columns for c in keys)

segment_source = segment_source[
    keys + [text_column]
].copy()

for c in keys:
    review_df_12_27h10[c] = pd.to_numeric(
        review_df_12_27h10[c], errors="raise"
    ).astype(int)

    segment_source[c] = pd.to_numeric(
        segment_source[c], errors="raise"
    ).astype(int)

if segment_source.duplicated(keys).any():
    raise RuntimeError("Duplicate segment identifiers detected.")

review_df_12_27h10 = review_df_12_27h10.merge(
    segment_source,
    on=keys,
    how="left",
    validate="one_to_one"
)

review_df_12_27h10 = review_df_12_27h10.rename(
    columns={text_column: "original_segment_text"}
)

assert review_df_12_27h10["original_segment_text"].notna().all()

assert (
    review_df_12_27h10["original_segment_text"]
    .astype(str).str.strip().str.len() > 100
).all()

print("Original segment linkage: PASSED")

# --------------------------------------------------
# 3. Prepare standardized human coding fields
# --------------------------------------------------

coding_fields = [
    "principal_geographic_focus",
    "principal_substantive_issue",
    "diplomatic_or_peace_process",
    "kampuchea_specific_relevance",
    "broad_conflict_diplomacy_relevance",
    "semantic_drift_assessment",
    "supporting_text",
    "reviewer_notes"
]

semantic_adjudication_12_27h10 = review_df_12_27h10[
    [
        "review_id", "country_code", "year",
        "speech_id", "segment_id",
        "original_segment_text"
    ]
].copy()

share_column = next(
    (
        c for c in [
            "topic03_share_recovered",
            "topic03_share",
            "target_topic_share"
        ]
        if c in review_df_12_27h10.columns
    ),
    None
)

if share_column is not None:
    semantic_adjudication_12_27h10[
        "topic03_share"
    ] = review_df_12_27h10[share_column].values

for field in coding_fields:
    semantic_adjudication_12_27h10[field] = pd.NA

semantic_adjudication_12_27h10[
    "human_review_status"
] = "PENDING"

# --------------------------------------------------
# 4. Priority records requiring careful review
# --------------------------------------------------

priority_ids = [
    "SV-016",   # China, 1999
    "SV-X002",  # Iran, 1981
    "SV-X012"   # United States, 2025
]

priority_df_12_27h10 = (
    semantic_adjudication_12_27h10[
        semantic_adjudication_12_27h10[
            "review_id"
        ].isin(priority_ids)
    ]
    .set_index("review_id")
    .loc[priority_ids]
    .reset_index()
)

print("\n" + "=" * 70)
print("PRIORITY HUMAN ADJUDICATION RECORDS")
print("=" * 70)

display(
    priority_df_12_27h10[
        [
            "review_id", "country_code",
            "year", "speech_id", "segment_id",
            "human_review_status"
        ]
    ]
)

# --------------------------------------------------
# 5. Display original texts for qualitative review
# --------------------------------------------------

for _, record in priority_df_12_27h10.iterrows():

    print("\n" + "#" * 70)
    print("Review ID:", record["review_id"])
    print("Country:", record["country_code"])
    print("Year:", record["year"])
    print("Speech ID:", record["speech_id"])
    print("Segment ID:", record["segment_id"])
    print("-" * 70)

    source_text = str(record["original_segment_text"])

    print(
        textwrap.fill(
            source_text,
            width=105,
            replace_whitespace=False
        )
    )

# --------------------------------------------------
# 6. Final inventory and review-status audit
# --------------------------------------------------

print("\n" + "=" * 70)
print("CROSS-COUNTRY REVIEW COVERAGE")
print("=" * 70)

coverage = (
    semantic_adjudication_12_27h10
    .groupby("country_code")
    .agg(
        n_records=("review_id", "count"),
        n_years=("year", "nunique")
    )
)

display(coverage)

assert len(coverage) == 3
assert (coverage["n_records"] == 6).all()

print("\n" + "=" * 70)
print("FINAL VALIDATION")
print("=" * 70)

print("Total review records:", len(semantic_adjudication_12_27h10))
print("Countries:", len(coverage))
print("Source text recovery: PASSED")
print("Record identification: PASSED")
print("Human coding template: PREPARED")
print("Human semantic validation: PENDING")
print("Existing statistical results modified: NO")
print("Files saved: NONE")
print("\nSTEP 12.27H.10: COMPUTATION COMPLETED")



### Step 12.27H.11 — Priority Semantic Evidence Assessment

**Objective:** Examine three potentially heterogeneous Topic 03 records using their complete original source segments.

**Priority cases**
- China, 1999 (SV-016): Kosovo, sovereignty, and intervention.
- Iran, 1981 (SV-X002): South African apartheid and Namibia.
- United States, 2025 (SV-X012): Immigration, security alliances, and international relations.

**Procedure**
1. Verify the source text for each priority record.
2. Normalize text formatting for complete and readable inspection.
3. Extract contextual evidence around case-specific expressions.
4. Prepare provisional thematic interpretations.
5. Distinguish Kampuchea-specific relevance from broader international conflict and diplomatic discourse.
6. Preserve human adjudication as pending until the evidence has been reviewed.

**Scientific restriction:** Provisional interpretations are not validated human annotations. No NMF model, topic weight, statistical effect, or significance result will be changed.

**Saving:** None. Any approved outputs will be exported through a separate code cell.


In [ ]:

print("=" * 72)
print("STEP 12.27H.11 — PRIORITY SEMANTIC EVIDENCE ASSESSMENT")
print("=" * 72)

# --------------------------------------------------
# 1. Verify previous adjudication inventory
# --------------------------------------------------

source_name = "semantic_adjudication_12_27h10"

if source_name not in globals():
    raise RuntimeError(
        "Step 12.27H.10 must be executed first."
    )

df = semantic_adjudication_12_27h10.copy()

required_columns = {
    "review_id",
    "country_code",
    "year",
    "speech_id",
    "segment_id",
    "original_segment_text",
    "human_review_status"
}

missing = required_columns - set(df.columns)

if missing:
    raise RuntimeError(
        f"Required columns missing: {sorted(missing)}"
    )

assert len(df) == 18
assert df["review_id"].is_unique
assert df["original_segment_text"].notna().all()

print("\nSource inventory: VERIFIED")
print("Total review records:", len(df))

# --------------------------------------------------
# 2. Define priority cases and inspection criteria
# --------------------------------------------------

priority_cases = {
    "SV-016": {
        "expected_country": "CHN",
        "expected_year": 1999,
        "thematic_hypothesis":
            "Kosovo, sovereignty, and non-intervention",
        "terms": [
            r"\bkosovo\b",
            r"\bsovereig\w*\b",
            r"\binterven\w*\b",
            r"\bkampuchea\b",
            r"\bcambodia\b"
        ]
    },
    "SV-X002": {
        "expected_country": "IRN",
        "expected_year": 1981,
        "thematic_hypothesis":
            "Apartheid, South Africa, and Namibia",
        "terms": [
            r"\bapartheid\b",
            r"\bsouth africa\b",
            r"\bnamibia\b",
            r"\bkampuchea\b",
            r"\bcambodia\b"
        ]
    },
    "SV-X012": {
        "expected_country": "USA",
        "expected_year": 2025,
        "thematic_hypothesis":
            "Immigration, alliances, and international security",
        "terms": [
            r"\bimmigra\w*\b",
            r"\bnato\b",
            r"\bmiddle east\b",
            r"\bsecurity\b",
            r"\bkampuchea\b",
            r"\bcambodia\b"
        ]
    }
}

# --------------------------------------------------
# 3. Verify original record identities
# --------------------------------------------------

selected_rows = []

for review_id, configuration in priority_cases.items():

    match = df.loc[df["review_id"] == review_id]

    if len(match) != 1:
        raise RuntimeError(
            f"Invalid source match for {review_id}"
        )

    row = match.iloc[0].copy()

    assert row["country_code"] == configuration[
        "expected_country"
    ]

    assert int(row["year"]) == configuration[
        "expected_year"
    ]

    selected_rows.append(row)

priority_source_df_12_27h11 = pd.DataFrame(
    selected_rows
).reset_index(drop=True)

print("\n" + "=" * 72)
print("VERIFIED PRIORITY SOURCES")
print("=" * 72)

display(
    priority_source_df_12_27h11[
        [
            "review_id",
            "country_code",
            "year",
            "speech_id",
            "segment_id",
            "human_review_status"
        ]
    ]
)

# --------------------------------------------------
# 4. Recover and inspect complete source texts
# --------------------------------------------------

def normalize_source_text(value):
    return re.sub(r"\s+", " ", str(value)).strip()


def extract_keyword_context(text, patterns, radius=90):

    evidence = []

    for pattern in patterns:

        matches = list(
            re.finditer(
                pattern,
                text,
                flags=re.IGNORECASE
            )
        )

        contexts = []

        for match in matches[:3]:

            left = max(0, match.start() - radius)
            right = min(len(text), match.end() + radius)

            excerpt = text[left:right]

            if left > 0:
                excerpt = "..." + excerpt

            if right < len(text):
                excerpt += "..."

            contexts.append(excerpt)

        evidence.append({
            "keyword_pattern": pattern,
            "mentions": len(matches),
            "sample_contexts": contexts
        })

    return evidence


evidence_records = []

for _, record in priority_source_df_12_27h11.iterrows():

    review_id = record["review_id"]
    configuration = priority_cases[review_id]

    text = normalize_source_text(
        record["original_segment_text"]
    )

    print("\n" + "#" * 72)
    print("REVIEW:", review_id)
    print("COUNTRY:", record["country_code"])
    print("YEAR:", record["year"])
    print("THEMATIC HYPOTHESIS:")
    print(configuration["thematic_hypothesis"])
    print("#" * 72)

    print("\nCOMPLETE ORIGINAL SEGMENT:\n")

    print(
        textwrap.fill(
            text,
            width=105,
            break_long_words=False,
            break_on_hyphens=False
        )
    )

    print("\nKEYWORD-IN-CONTEXT EVIDENCE")
    print("-" * 72)

    keyword_evidence = extract_keyword_context(
        text,
        configuration["terms"]
    )

    for item in keyword_evidence:

        pattern = item["keyword_pattern"]

        print(
            f"\nPattern: {pattern} | "
            f"Occurrences: {item['mentions']}"
        )

        for context in item["sample_contexts"]:
            print(
                textwrap.fill(
                    context,
                    width=105,
                    initial_indent="  ",
                    subsequent_indent="  "
                )
            )

        evidence_records.append({
            "review_id": review_id,
            "country_code": record["country_code"],
            "year": int(record["year"]),
            "keyword_pattern": pattern,
            "mentions": item["mentions"]
        })

keyword_evidence_df_12_27h11 = pd.DataFrame(
    evidence_records
)

# --------------------------------------------------
# 5. Provisional interpretations
# --------------------------------------------------

provisional_cases = [
    {
        "review_id": "SV-016",
        "provisional_theme":
            "Kosovo and sovereignty/non-intervention",
        "semantic_question":
            "Does this represent broad conflict and "
            "international-order discourse rather than "
            "Kampuchea-specific discourse?"
    },
    {
        "review_id": "SV-X002",
        "provisional_theme":
            "Apartheid and Southern African politics",
        "semantic_question":
            "Does this indicate a distinct anti-apartheid "
            "dimension within Topic 03?"
    },
    {
        "review_id": "SV-X012",
        "provisional_theme":
            "Contemporary US domestic and foreign policy",
        "semantic_question":
            "Is this source substantively coherent with "
            "a broad conflict/diplomacy topic?"
    }
]

provisional_review_df_12_27h11 = pd.DataFrame(
    provisional_cases
)

provisional_review_df_12_27h11[
    "adjudication_status"
] = "PENDING HUMAN CONFIRMATION"

print("\n" + "=" * 72)
print("PROVISIONAL SEMANTIC INTERPRETATIONS")
print("=" * 72)

display(provisional_review_df_12_27h11)

# --------------------------------------------------
# 6. Check whether manual judgments were assigned
# --------------------------------------------------

human_fields = [
    "principal_geographic_focus",
    "principal_substantive_issue",
    "kampuchea_specific_relevance",
    "broad_conflict_diplomacy_relevance",
    "semantic_drift_assessment"
]

human_fields = [
    c for c in human_fields if c in df.columns
]

completed_records = 0

if human_fields:
    completed_records = int(
        df[human_fields].notna().all(axis=1).sum()
    )

print("\n" + "=" * 72)
print("SEMANTIC REVIEW STATUS")
print("=" * 72)

print("Total records:", len(df))
print("Priority cases inspected:", len(priority_cases))
print("Fully populated human coding records:", completed_records)
print("Provisional evidence assessment: COMPLETED")
print("Final human adjudication: PENDING")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.11: COMPUTATION COMPLETED")



### Step 12.27H.12 — Priority Semantic Adjudication and Topic-Label Assessment

**Objective:** Evaluate three priority sources for semantic consistency and assess whether Topic 03 represents Kampuchea-specific discourse or a broader international conflict, sovereignty, and diplomacy theme.

**Cases:** SV-016 (China, 1999), SV-X002 (Iran, 1981), and SV-X012 (United States, 2025).

**Analytical procedure:**
1. Recover the three original cached source segments.
2. Extract traceable source evidence for each provisional interpretation.
3. Assess narrow-label and broad-label relevance separately.
4. Prepare provisional human-review decisions.
5. Require explicit human confirmation before treating any decision as validated.

**Interpretation rule:** Semantic coding must be based on substantive text rather than keyword counts alone. The three priority cases cannot establish topic-wide validity without reviewing the remaining sampled records.

**Safeguards:** No NMF refitting, topic reassignment, effect modification, inferential recomputation, or automatic human approval.

**Output:** Provisional adjudication table, traceable supporting excerpts, and human-confirmation audit.

**Saving:** Disabled. Export will be handled in a separate cell only after review.


In [ ]:


print("=" * 72)
print("STEP 12.27H.12 — PRIORITY SEMANTIC ADJUDICATION")
print("=" * 72)

# --------------------------------------------------
# 1. Verify source
# --------------------------------------------------

assert "semantic_adjudication_12_27h10" in globals()

source = semantic_adjudication_12_27h10.copy()

required = [
    "review_id", "country_code", "year",
    "original_segment_text"
]

assert all(c in source.columns for c in required)
assert len(source) == 18
assert source["review_id"].is_unique

# --------------------------------------------------
# 2. Provisional coding proposals
# --------------------------------------------------

proposals = {
    "SV-016": {
        "geographic_focus": "Kosovo / former Yugoslavia",
        "substantive_issue": "Sovereignty and humanitarian intervention",
        "kampuchea_specific_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_interpretation": "Geographic and thematic heterogeneity",
        "patterns": [
            r"\bKosovo\b",
            r"\bsovereignty\b",
            r"\bhumanitarian intervention\b"
        ]
    },
    "SV-X002": {
        "geographic_focus": "South Africa / Namibia",
        "substantive_issue": "Apartheid, racial domination and decolonization",
        "kampuchea_specific_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_interpretation": "Distinct anti-apartheid discourse",
        "patterns": [
            r"\bapartheid\b",
            r"\bNamibia\b",
            r"\bSouth Africa\b"
        ]
    },
    "SV-X012": {
        "geographic_focus": "United States / international",
        "substantive_issue": "Immigration, security alliances and foreign policy",
        "kampuchea_specific_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_interpretation": "Mixed domestic and international discourse",
        "patterns": [
            r"\bimmigra\w*\b",
            r"\bNATO\b",
            r"\bMiddle East\b"
        ]
    }
}

# --------------------------------------------------
# 3. Human confirmation controls
#
# Change False to True ONLY after personally
# checking the source and proposed interpretation.
# --------------------------------------------------

human_confirmations = {
    "SV-016": False,
    "SV-X002": False,
    "SV-X012": False
}

# --------------------------------------------------
# 4. Source-based evidence extraction
# --------------------------------------------------

def extract_context(text, pattern, radius=105):

    match = re.search(pattern, text, re.IGNORECASE)

    if match is None:
        return None

    start = max(0, match.start() - radius)
    end = min(len(text), match.end() + radius)

    return text[start:end].strip()


review_rows = []
evidence_rows = []

for review_id, proposal in proposals.items():

    matched = source.loc[
        source["review_id"] == review_id
    ]

    assert len(matched) == 1

    record = matched.iloc[0]

    text = re.sub(
        r"\s+", " ",
        str(record["original_segment_text"])
    ).strip()

    assert len(text) > 100

    for pattern in proposal["patterns"]:

        excerpt = extract_context(text, pattern)

        evidence_rows.append({
            "review_id": review_id,
            "evidence_pattern": pattern,
            "found": excerpt is not None,
            "source_excerpt": excerpt
        })

    confirmed = bool(human_confirmations[review_id])

    review_rows.append({
        "review_id": review_id,
        "country_code": record["country_code"],
        "year": int(record["year"]),
        "geographic_focus": proposal["geographic_focus"],
        "substantive_issue": proposal["substantive_issue"],
        "kampuchea_specific_fit":
            proposal["kampuchea_specific_fit"],
        "broad_conflict_diplomacy_fit":
            proposal["broad_conflict_diplomacy_fit"],
        "semantic_interpretation":
            proposal["semantic_interpretation"],
        "human_review_status":
            "CONFIRMED" if confirmed else "PENDING"
    })

priority_adjudication_12_27h12 = pd.DataFrame(review_rows)

priority_evidence_12_27h12 = pd.DataFrame(evidence_rows)

# --------------------------------------------------
# 5. Display provisional adjudication
# --------------------------------------------------

print("\nPROVISIONAL SEMANTIC ADJUDICATION")
print("-" * 72)

display(priority_adjudication_12_27h12)

print("\nTRACEABLE SOURCE EVIDENCE")
print("-" * 72)

display(priority_evidence_12_27h12)

# --------------------------------------------------
# 6. Semantic-label comparison
# --------------------------------------------------

label_comparison_12_27h12 = (
    priority_adjudication_12_27h12[
        [
            "review_id",
            "kampuchea_specific_fit",
            "broad_conflict_diplomacy_fit",
            "human_review_status"
        ]
    ].copy()
)

print("\nNARROW VS BROAD TOPIC-LABEL ASSESSMENT")
print("-" * 72)

display(label_comparison_12_27h12)

# --------------------------------------------------
# 7. Validation
# --------------------------------------------------

assert len(priority_adjudication_12_27h12) == 3
assert len(priority_evidence_12_27h12) == 9

n_confirmed = int(
    priority_adjudication_12_27h12[
        "human_review_status"
    ].eq("CONFIRMED").sum()
)

print("\n" + "=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

print("Original source records:", len(source))
print("Priority cases evaluated:", 3)
print("Evidence checks:", len(priority_evidence_12_27h12))
print(
    "Patterns matched:",
    int(priority_evidence_12_27h12["found"].sum())
)
print("Human-confirmed cases:", n_confirmed)
print("Human-pending cases:", 3 - n_confirmed)
print("Provisional coding preparation: PASSED")
print("Topic-wide semantic validation: PENDING")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.12: COMPUTATION COMPLETED")



### Step 12.27H.13 — Full-Sample Semantic Evidence Audit

**Objective:** Assess the historical and cross-country semantic consistency of Topic 03 using all 18 sampled source segments.

**Procedure:**
1. Verify the integrity and coverage of the 18 source records.
2. Identify geographic and substantive themes through reproducible lexical screening.
3. Compare thematic signals across China, Iran, and the United States.
4. Retain the three priority adjudications as provisional.
5. Inspect additional complete cached source segments to support qualitative interpretation.

**Important distinction:** Keyword matches are screening evidence, not human semantic validation. Topic labels must not be finalized from lexical counts alone.

**Expected output:** An 18-record semantic inventory, thematic evidence summary, and three complete source segments for qualitative review.

**Safeguards:** No automatic semantic approval, statistical recomputation, or changes to the original NMF model.

**Saving:** Disabled. Any subsequent export must use a separate code cell.


In [ ]:


print("=" * 72)
print("STEP 12.27H.13 — FULL-SAMPLE SEMANTIC EVIDENCE AUDIT")
print("=" * 72)

# --------------------------------------------------
# 1. Source verification
# --------------------------------------------------

assert "semantic_adjudication_12_27h10" in globals()
assert "priority_adjudication_12_27h12" in globals()

source = semantic_adjudication_12_27h10.copy()
priority = priority_adjudication_12_27h12.copy()

required = [
    "review_id",
    "country_code",
    "year",
    "original_segment_text"
]

assert all(c in source.columns for c in required)
assert len(source) == 18
assert source["review_id"].is_unique

expected_countries = {"CHN": 6, "IRN": 6, "USA": 6}

assert source["country_code"].value_counts().to_dict() == expected_countries

assert source["original_segment_text"].notna().all()

# --------------------------------------------------
# 2. Lexical screening dictionary
# --------------------------------------------------

patterns = {
    "Kampuchea/Cambodia":
        r"\b(?:kampuchea|cambodia|cambodian)\b",

    "Palestine/Israel":
        r"\b(?:palestin\w*|israel\w*|jerusalem|gaza)\b",

    "Korean Peninsula":
        r"\b(?:korea|korean)\b",

    "South Africa/Namibia":
        r"\b(?:apartheid|namibia|south africa)\b",

    "Kosovo/Bosnia":
        r"\b(?:kosovo|bosnia|yugoslavia)\b",

    "Afghanistan":
        r"\b(?:afghanistan|afghan)\b",

    "Middle East":
        r"\bmiddle east\b",

    "Sovereignty/Intervention":
        r"\b(?:sovereignty|sovereign|intervention)\b",

    "Diplomacy/Peace":
        r"\b(?:peace|negotiat\w*|settlement|dialogue|diplomacy)\b",

    "Immigration/NATO":
        r"\b(?:immigra\w*|nato)\b"
}

# --------------------------------------------------
# 3. Complete thematic screening
# --------------------------------------------------

records = []

for _, row in source.iterrows():

    text = str(row["original_segment_text"])

    result = {
        "review_id": row["review_id"],
        "country_code": row["country_code"],
        "year": int(row["year"])
    }

    if "period" in source.columns:
        result["period"] = row["period"]

    for label, pattern in patterns.items():

        result[label] = len(
            re.findall(pattern, text, flags=re.IGNORECASE)
        )

    records.append(result)

semantic_theme_counts_12_27h13 = pd.DataFrame(records)

# --------------------------------------------------
# 4. Geographic evidence profile
# --------------------------------------------------

geo_labels = [
    "Kampuchea/Cambodia",
    "Palestine/Israel",
    "Korean Peninsula",
    "South Africa/Namibia",
    "Kosovo/Bosnia",
    "Afghanistan",
    "Middle East"
]

def top_geographies(row):

    matches = [
        (label, int(row[label]))
        for label in geo_labels
        if row[label] > 0
    ]

    matches.sort(key=lambda x: -x[1])

    if not matches:
        return "No geographic dictionary match"

    return "; ".join(
        f"{label} ({count})"
        for label, count in matches[:3]
    )

inventory = semantic_theme_counts_12_27h13[
    ["review_id", "country_code", "year"] +
    (["period"] if "period" in semantic_theme_counts_12_27h13.columns else [])
].copy()

inventory["leading_geographic_signals"] = (
    semantic_theme_counts_12_27h13.apply(
        top_geographies, axis=1
    )
)

inventory["kampuchea_mentions"] = (
    semantic_theme_counts_12_27h13["Kampuchea/Cambodia"]
)

inventory["diplomacy_mentions"] = (
    semantic_theme_counts_12_27h13["Diplomacy/Peace"]
)

priority_ids = set(priority["review_id"])

inventory["review_priority"] = inventory["review_id"].apply(
    lambda x: "Previously assessed" if x in priority_ids
    else "Remaining review"
)

inventory["human_review_status"] = "PENDING"

semantic_inventory_12_27h13 = (
    inventory.sort_values(
        ["country_code", "year", "review_id"]
    ).reset_index(drop=True)
)

print("\nFULL SEMANTIC INVENTORY")
print("-" * 72)

display(semantic_inventory_12_27h13)

# --------------------------------------------------
# 5. Cross-country thematic evidence
# --------------------------------------------------

thematic_columns = list(patterns.keys())

country_summary = (
    semantic_theme_counts_12_27h13
    .groupby("country_code")[thematic_columns]
    .sum()
)

print("\nCROSS-COUNTRY THEMATIC EVIDENCE")
print("-" * 72)

display(country_summary)

# --------------------------------------------------
# 6. Topic-label screening
# --------------------------------------------------

n_kampuchea = int(
    (semantic_theme_counts_12_27h13[
        "Kampuchea/Cambodia"
    ] > 0).sum()
)

print("\nTOPIC-LABEL COVERAGE")
print("-" * 72)

print("Total sampled segments:", len(source))
print("Segments mentioning Kampuchea/Cambodia:", n_kampuchea)
print("Segments without such mentions:", len(source) - n_kampuchea)

print(
    "\nInterpretation: Lexical coverage alone does not "
    "establish or refute semantic topic relevance."
)

# --------------------------------------------------
# 7. Priority adjudication linkage
# --------------------------------------------------

print("\nPREVIOUS PRIORITY ADJUDICATIONS")
print("-" * 72)

display(
    priority[
        [
            "review_id",
            "geographic_focus",
            "substantive_issue",
            "kampuchea_specific_fit",
            "broad_conflict_diplomacy_fit",
            "human_review_status"
        ]
    ]
)

# --------------------------------------------------
# 8. Additional qualitative source inspection
#
# These three cases cover different countries
# and historical contexts.
# --------------------------------------------------

INSPECT_IDS = [
    "SV-014",   # China, 1988
    "SV-X004",  # Iran, 1992
    "SV-X011"   # United States, 2011
]

assert len(INSPECT_IDS) == len(set(INSPECT_IDS))
assert set(INSPECT_IDS).issubset(set(source["review_id"]))

for review_id in INSPECT_IDS:

    row = source.loc[
        source["review_id"] == review_id
    ].iloc[0]

    original_text = str(row["original_segment_text"])

    print("\n" + "=" * 72)
    print("QUALITATIVE SOURCE INSPECTION")
    print("=" * 72)

    print("Review ID:", review_id)
    print("Country:", row["country_code"])
    print("Year:", row["year"])
    print("\nCACHED SOURCE SEGMENT:\n")

    print(
        textwrap.fill(
            original_text,
            width=100,
            replace_whitespace=True
        )
    )

    print("\nHuman semantic judgment: PENDING")

# --------------------------------------------------
# 9. Final validation
# --------------------------------------------------

assert len(semantic_inventory_12_27h13) == 18
assert len(semantic_theme_counts_12_27h13) == 18
assert len(priority) == 3

print("\n" + "=" * 72)
print("FINAL VALIDATION")
print("=" * 72)

print("Source records verified: 18")
print("Country coverage: 3/3")
print("Previously assessed priority cases: 3")
print("Additional segments inspected: 3")
print("Full thematic screening: PASSED")
print("Final human semantic validation: PENDING")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.13: COMPUTATION COMPLETED")



### Step 12.27H.14 — Reference NMF Topic Vocabulary and Historical Semantic Triangulation

**Objective:** Evaluate whether the current Topic 03 label adequately represents its underlying NMF vocabulary and observed historical thematic diversity.

**Analysis:**
1. Recover the reference seven-topic NMF vocabulary inventory.
2. Inspect Topic 03 and neighboring topic vocabularies.
3. Examine historical topic contribution diagnostics.
4. Compare geographic and thematic keyword coverage across nine country-period strata.
5. Identify potential topic-label limitations and semantic heterogeneity.

**Interpretation:** Lexical screening and NMF word weights provide supporting evidence, not definitive semantic classification.

**Safeguards:** Preserve all original NMF outputs, statistical estimates, and existing publication tables.

**Saving:** Disabled. Human semantic confirmation remains pending.


In [ ]:

print("=" * 72)
print("STEP 12.27H.14 — NMF SEMANTIC TRIANGULATION")
print("=" * 72)

# --------------------------------------------------
# 1. Verify existing analytical objects
# --------------------------------------------------

required_objects = [
    "topic_inventory_df",
    "semantic_theme_counts_12_27h13",
    "semantic_adjudication_12_27h10"
]

missing = [x for x in required_objects if x not in globals()]

if missing:
    raise RuntimeError(f"Missing required objects: {missing}")

topic_inventory = topic_inventory_df.copy()
theme_counts = semantic_theme_counts_12_27h13.copy()
source = semantic_adjudication_12_27h10.copy()

assert len(theme_counts) == 18
assert len(source) == 18
assert theme_counts["review_id"].is_unique

print("\nSOURCE VALIDATION: PASSED")

# --------------------------------------------------
# 2. Recover reference NMF topic vocabulary
# --------------------------------------------------

print("\n" + "=" * 72)
print("REFERENCE NMF TOPIC VOCABULARY")
print("=" * 72)

print("Inventory columns:", list(topic_inventory.columns))

if "n_topics" in topic_inventory.columns:
    k_col = "n_topics"
elif "K" in topic_inventory.columns:
    k_col = "K"
else:
    raise ValueError("NMF topic-count column not found.")

assert "topic_id" in topic_inventory.columns

reference_topics = topic_inventory.loc[
    pd.to_numeric(topic_inventory[k_col], errors="coerce").eq(7)
].copy()

assert not reference_topics.empty, (
    "No seven-topic NMF vocabulary records were found."
)

reference_topics["topic_id_numeric"] = pd.to_numeric(
    reference_topics["topic_id"], errors="coerce"
)

reference_topics = reference_topics.sort_values(
    "topic_id_numeric"
)

term_columns = [
    c for c in reference_topics.columns
    if "term" in c.lower() or "word" in c.lower()
]

display_columns = (
    ["topic_id"] +
    [c for c in ["reference_seed"] if c in reference_topics.columns] +
    term_columns
)

display(reference_topics[display_columns])

topic03_vocabulary = reference_topics.loc[
    reference_topics["topic_id_numeric"].eq(3)
]

assert not topic03_vocabulary.empty, (
    "Topic 03 is missing from the seven-topic inventory."
)

print("\nTOPIC 03 — MODEL VOCABULARY")
display(topic03_vocabulary[display_columns])

# --------------------------------------------------
# 3. Historical model-level diagnostics
# --------------------------------------------------

print("\n" + "=" * 72)
print("HISTORICAL TOPIC CONTRIBUTION DIAGNOSTICS")
print("=" * 72)

if "topic_historical_df" in globals():

    historical = topic_historical_df.copy()

    if "K" in historical.columns:
        historical = historical.loc[
            pd.to_numeric(historical["K"], errors="coerce").eq(7)
        ]

    if "topic_id" in historical.columns:
        historical = historical.loc[
            pd.to_numeric(
                historical["topic_id"], errors="coerce"
            ).eq(3)
        ]

    if historical.empty:
        print("No matching Topic 03 historical records found.")
    else:
        display(historical.reset_index(drop=True))

else:
    print("Historical topic diagnostics unavailable.")

# --------------------------------------------------
# 4. Reconstruct historical strata
# --------------------------------------------------

print("\n" + "=" * 72)
print("COUNTRY-PERIOD SEMANTIC EVIDENCE")
print("=" * 72)

theme_counts["year"] = pd.to_numeric(
    theme_counts["year"], errors="raise"
)

theme_counts["period"] = pd.cut(
    theme_counts["year"],
    bins=[1971, 1990, 2010, 2025],
    labels=["1972-1990", "1991-2010", "2011-2025"]
)

assert theme_counts["period"].notna().all()

geo_columns = [
    "Kampuchea/Cambodia",
    "Palestine/Israel",
    "Korean Peninsula",
    "South Africa/Namibia",
    "Kosovo/Bosnia",
    "Afghanistan",
    "Middle East"
]

for c in geo_columns:
    assert c in theme_counts.columns

# Count sampled segments mentioning each category,
# rather than counting repeated word occurrences.

presence = theme_counts.copy()

for c in geo_columns:
    presence[c] = presence[c].gt(0).astype(int)

country_period_summary = (
    presence.groupby(
        ["country_code", "period"],
        observed=True
    )[geo_columns]
    .sum()
    .reset_index()
)

sample_counts = (
    presence.groupby(
        ["country_code", "period"],
        observed=True
    )
    .size()
    .rename("sampled_segments")
    .reset_index()
)

country_period_summary = sample_counts.merge(
    country_period_summary,
    on=["country_code", "period"],
    validate="one_to_one"
)

assert len(country_period_summary) == 9
assert country_period_summary["sampled_segments"].eq(2).all()

display(country_period_summary)

# --------------------------------------------------
# 5. Narrow-label coverage audit
# --------------------------------------------------

print("\n" + "=" * 72)
print("NARROW TOPIC-LABEL COVERAGE")
print("=" * 72)

coverage = (
    presence.groupby("country_code")[
        "Kampuchea/Cambodia"
    ]
    .agg(
        sampled_segments="count",
        keyword_positive_segments="sum"
    )
    .reset_index()
)

coverage["keyword_negative_segments"] = (
    coverage["sampled_segments"] -
    coverage["keyword_positive_segments"]
)

display(coverage)

print(
    "\nTotal keyword-positive sampled segments:",
    int(coverage["keyword_positive_segments"].sum())
)

# --------------------------------------------------
# 6. Semantic interpretation guardrails
# --------------------------------------------------

print("\n" + "=" * 72)
print("SEMANTIC INTERPRETATION STATUS")
print("=" * 72)

print("Reference NMF vocabulary: RETRIEVED")
print("Country-period strata: 9/9")
print("Sampled records: 18/18")
print("Historical lexical screening: COMPLETED")
print("Narrow Topic 03 label: REQUIRES REVIEW")
print("Alternative topic labels: NOT YET FINALIZED")
print("Human semantic adjudication: PENDING")
print("Original NMF model modified: NO")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.14: COMPUTATION COMPLETED")



### Step 12.27H.15 — Reference NMF Topic Identity and Vocabulary Audit

**Objective:** Verify the identity and vocabulary of Topic 03 in the actual reference NMF model before finalizing its semantic label.

**Procedure:**
1. Verify the reference NMF model, random state, and topic-term matrix.
2. Compare reference-model metadata with the topic inventory.
3. Identify available vocabulary sources consistent with the reference feature dimension.
4. Extract the highest-weighted terms of the actual reference Topic 03, if vocabulary alignment can be established.
5. Inspect all seven reference topics for potential component-label mismatches.

**Decision rule:** Do not transfer topic labels between NMF solutions based solely on topic ID.

**Safeguards:** No model refitting, relabeling, modification of statistical findings, or file saving.

**Status:** Final semantic labeling pending identity verification.


In [ ]:


print("=" * 72)
print("STEP 12.27H.15 — REFERENCE NMF TOPIC IDENTITY AUDIT")
print("=" * 72)

# --------------------------------------------------
# 1. Recover the actual reference model
# --------------------------------------------------

assert "selected_result" in globals()
assert "topic_inventory_df" in globals()

ref_result = selected_result
ref_model = ref_result["model"]

assert hasattr(ref_model, "components_"), (
    "Selected reference model has no fitted components_."
)

H_model = np.asarray(ref_model.components_, dtype=float)

assert H_model.ndim == 2
assert H_model.shape[0] == 7

print("\nREFERENCE MODEL METADATA")
print("-" * 55)

print("Model class:", type(ref_model).__name__)
print("Number of topics:", H_model.shape[0])
print("Vocabulary dimension:", H_model.shape[1])
print("Random state:", getattr(ref_model, "random_state", None))
print("NMF components shape:", H_model.shape)

if "H" in ref_result:
    H_stored = np.asarray(ref_result["H"], dtype=float)

    same_H = (
        H_stored.shape == H_model.shape
        and np.allclose(
            H_stored,
            H_model,
            rtol=1e-8,
            atol=1e-10
        )
    )

    print("Stored H matches fitted model:", same_H)

    if not same_H:
        raise RuntimeError(
            "Stored H does not match fitted reference model."
        )

# --------------------------------------------------
# 2. Audit the seven-topic inventory
# --------------------------------------------------

print("\n" + "=" * 72)
print("REFERENCE SEED AND TOPIC INVENTORY")
print("=" * 72)

inventory = topic_inventory_df.copy()

k_column = (
    "n_topics" if "n_topics" in inventory.columns
    else "K"
)

inventory = inventory.loc[
    pd.to_numeric(
        inventory[k_column], errors="coerce"
    ).eq(7)
].copy()

inventory["topic_id"] = pd.to_numeric(
    inventory["topic_id"], errors="raise"
).astype(int)

assert inventory["topic_id"].nunique() == 7

display(
    inventory[
        ["topic_id", "reference_seed", "top_terms"]
    ].sort_values("topic_id")
)

print("\nFULL INVENTORY VOCABULARY — TOPIC 03")

for _, row in inventory.loc[
    inventory["topic_id"].eq(3)
].iterrows():
    print("Seed:", row["reference_seed"])
    print("Terms:", row["top_terms"])

model_seed = getattr(ref_model, "random_state", None)

inventory_seeds = inventory["reference_seed"].dropna().unique()

print("\nModel random state:", model_seed)
print("Inventory reference seeds:", inventory_seeds.tolist())

seed_match = (
    len(inventory_seeds) == 1
    and str(inventory_seeds[0]) == str(model_seed)
)

print("Seed metadata agrees:", seed_match)

if not seed_match:
    print(
        "CAUTION: Inventory and model seed metadata differ. "
        "Direct topic-ID correspondence is not established."
    )

# --------------------------------------------------
# 3. Discover candidate vocabulary sources
# --------------------------------------------------

print("\n" + "=" * 72)
print("VOCABULARY SOURCE DISCOVERY")
print("=" * 72)

n_features = H_model.shape[1]

vocabulary_candidates = []

for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    lower_name = name.lower()

    if not any(
        token in lower_name
        for token in ["vector", "vocab", "feature", "term", "tfidf"]
    ):
        continue

    words = None

    try:
        if hasattr(obj, "get_feature_names_out"):
            words = np.asarray(
                obj.get_feature_names_out(),
                dtype=str
            )

        elif (
            isinstance(obj, dict)
            and len(obj) == n_features
            and all(isinstance(k, str) for k in obj)
            and all(
                isinstance(v, (int, np.integer))
                for v in obj.values()
            )
        ):
            indices = list(obj.values())

            if set(indices) == set(range(n_features)):
                words = np.empty(n_features, dtype=object)

                for word, index in obj.items():
                    words[index] = word

                words = np.asarray(words, dtype=str)

        elif isinstance(obj, (list, tuple, np.ndarray, pd.Index)):
            if len(obj) == n_features:
                sample = list(obj[:min(5, n_features)])

                if all(isinstance(x, str) for x in sample):
                    words = np.asarray(obj, dtype=str)

        if words is not None and len(words) == n_features:
            vocabulary_candidates.append((name, words))

    except (TypeError, ValueError, AttributeError):
        continue

print("Matching vocabulary candidates:")

if vocabulary_candidates:
    for name, words in vocabulary_candidates:
        print(f"  {name}: {len(words)} terms")
else:
    print("  NONE")

# --------------------------------------------------
# 4. Check candidate agreement
# --------------------------------------------------

print("\n" + "=" * 72)
print("VOCABULARY ALIGNMENT AUDIT")
print("=" * 72)

reference_words = None

if vocabulary_candidates:

    first_words = vocabulary_candidates[0][1]

    all_agree = all(
        np.array_equal(first_words, words)
        for _, words in vocabulary_candidates
    )

    print("Candidate vocabulary ordering agrees:", all_agree)

    if all_agree:
        reference_words = first_words
        print(
            "Single candidate vocabulary ordering recovered."
        )
        print(
            "Training-pipeline provenance still requires verification."
        )
    else:
        print(
            "WARNING: Conflicting feature orders detected."
        )
        print(
            "Topic-term interpretation is blocked until "
            "the training vocabulary is identified."
        )

# --------------------------------------------------
# 5. Inspect actual model topic terms
# --------------------------------------------------

print("\n" + "=" * 72)
print("ACTUAL REFERENCE NMF TOPIC VOCABULARY")
print("=" * 72)

if reference_words is not None:

    topic_records = []

    for topic_index in range(H_model.shape[0]):

        weights = H_model[topic_index]

        top_indices = np.argsort(weights)[::-1][:20]

        topic_records.append({
            "topic_id": topic_index + 1,
            "top_20_terms": ", ".join(
                reference_words[top_indices]
            )
        })

    reference_topic_vocabulary_audit = pd.DataFrame(
        topic_records
    )

    for _, row in reference_topic_vocabulary_audit.iterrows():
        print(f"\nTOPIC {int(row['topic_id']):02d}")
        print(row["top_20_terms"])

    print("\n" + "=" * 72)
    print("TOPIC 03 — TOP 40 WEIGHTED TERMS")
    print("=" * 72)

    topic03_weights = H_model[2]

    indices = np.argsort(topic03_weights)[::-1][:40]

    topic03_reference_terms = pd.DataFrame({
        "rank": np.arange(1, len(indices) + 1),
        "term": reference_words[indices],
        "nmf_weight": topic03_weights[indices]
    })

    display(topic03_reference_terms)

else:
    print(
        "Reference topic-term matrix is available, "
        "but a unique matching vocabulary was not recovered."
    )

# --------------------------------------------------
# 6. Final interpretation status
# --------------------------------------------------

print("\n" + "=" * 72)
print("FINAL IDENTITY AUDIT")
print("=" * 72)

print("Reference NMF model identification: PASSED")
print("Reference topic-term matrix: VERIFIED")

print(
    "Vocabulary candidates resolved:",
    reference_words is not None
)

print("Seed metadata agreement:", seed_match)

print(
    "Topic identity across NMF solutions:",
    "NOT YET VERIFIED" if not seed_match
    else "SEED AGREEMENT ONLY"
)

print("Topic 03 semantic label: PENDING REVIEW")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.15: DIAGNOSTIC COMPLETED")



### Step 12.27H.16 — Cross-Model NMF Topic Correspondence Audit

**Objective:** Investigate whether Topic 03 in the reference NMF model corresponds to the same semantic component in other fitted NMF solutions.

**Procedure:**
1. Identify fitted NMF models retained in notebook memory.
2. Inspect topic counts, random states, and feature dimensions.
3. Compare topic-term weight vectors using cosine similarity.
4. Calculate optimal one-to-one component assignments where possible.
5. Investigate the reference-seed mismatch (11 versus 73).
6. Identify unresolved feature-alignment and model-provenance issues.

**Interpretation:** Topic numbers are arbitrary across independently fitted NMF models. Similarity-based component matching is diagnostic and requires verification of feature ordering before it can establish topic equivalence.

**Safeguards:** No refitting, relabeling, modification of published estimates, or file saving.

**Status:** Cross-model topic correspondence pending audit.


In [ ]:


print("=" * 72)
print("STEP 12.27H.16 — CROSS-MODEL TOPIC CORRESPONDENCE AUDIT")
print("=" * 72)

# --------------------------------------------------
# 1. Reference model
# --------------------------------------------------

assert "selected_result" in globals()

reference_model = selected_result["model"]
H_reference = np.asarray(reference_model.components_, dtype=float)

assert H_reference.ndim == 2
assert H_reference.shape[0] == 7

reference_seed = getattr(reference_model, "random_state", None)
n_features = H_reference.shape[1]

print("\nREFERENCE MODEL")
print("Topics:", H_reference.shape[0])
print("Features:", n_features)
print("Random state:", reference_seed)
print("Reference Topic:", 3)

# --------------------------------------------------
# 2. Discover existing fitted NMF models
# --------------------------------------------------

print("\n" + "=" * 72)
print("FITTED MODEL DISCOVERY")
print("=" * 72)

discovered_models = {}
visited_containers = set()

def scan_nmf_object(obj, path, depth=0):

    if depth > 4:
        return

    if (
        hasattr(obj, "components_")
        and type(obj).__name__ == "NMF"
    ):
        try:
            H = np.asarray(obj.components_, dtype=float)

            if (
                H.ndim == 2
                and H.shape[1] == n_features
                and np.isfinite(H).all()
                and np.all(H >= 0)
            ):
                identifier = id(obj)

                if identifier not in discovered_models:
                    discovered_models[identifier] = {
                        "path": path,
                        "model": obj,
                        "H": H,
                        "seed": getattr(
                            obj, "random_state", None
                        ),
                        "K": H.shape[0]
                    }
        except (TypeError, ValueError, AttributeError):
            pass

        return

    if not isinstance(obj, (dict, list, tuple)):
        return

    identifier = id(obj)

    if identifier in visited_containers:
        return

    visited_containers.add(identifier)

    if isinstance(obj, dict):
        items = list(obj.items())[:250]

        for key, value in items:
            scan_nmf_object(
                value,
                f"{path}[{str(key)!r}]",
                depth + 1
            )

    else:
        for i, value in enumerate(obj[:250]):
            scan_nmf_object(
                value,
                f"{path}[{i}]",
                depth + 1
            )

root_names = []

for name, obj in list(globals().items()):

    if name.startswith("_"):
        continue

    lower = name.lower()

    relevant = any(
        term in lower
        for term in [
            "model", "result", "fitted",
            "ablation", "solution", "nmf", "fit"
        ]
    )

    if relevant:
        root_names.append(name)

for name in sorted(root_names):
    scan_nmf_object(globals()[name], name)

# Guarantee inclusion of the selected reference model
scan_nmf_object(
    reference_model,
    "selected_result['model']"
)

model_inventory = pd.DataFrame([
    {
        "source": item["path"],
        "K": item["K"],
        "random_state": item["seed"],
        "n_features": item["H"].shape[1],
        "is_reference": identifier == id(reference_model)
    }
    for identifier, item in discovered_models.items()
])

if not model_inventory.empty:
    display(model_inventory)
else:
    print("No compatible fitted models discovered.")

print(
    "\nCompatible fitted model objects:",
    len(discovered_models)
)

# --------------------------------------------------
# 3. Cosine similarity and topic alignment
# --------------------------------------------------

print("\n" + "=" * 72)
print("REFERENCE TOPIC 03 — CROSS-MODEL MATCHING")
print("=" * 72)

def normalize_components(H):
    norms = np.linalg.norm(H, axis=1, keepdims=True)

    if np.any(norms <= 0):
        raise ValueError(
            "At least one topic has zero component norm."
        )

    return H / norms

H_ref_normalized = normalize_components(H_reference)

alignment_records = []
mapping_records = []

for identifier, item in discovered_models.items():

    H_other = item["H"]
    H_other_normalized = normalize_components(H_other)

    similarity = (
        H_ref_normalized @ H_other_normalized.T
    )

    # Highest-similarity component for reference Topic 03
    local_match = int(np.argmax(similarity[2]))

    # Optimal global one-to-one topic assignment
    row_ind, col_ind = linear_sum_assignment(
        -similarity
    )

    assignment = dict(zip(
        row_ind.tolist(),
        col_ind.tolist()
    ))

    globally_matched = assignment.get(2)

    for reference_idx, candidate_idx in zip(
        row_ind, col_ind
    ):
        mapping_records.append({
            "source": item["path"],
            "seed": item["seed"],
            "K": item["K"],
            "reference_topic": int(reference_idx + 1),
            "candidate_topic": int(candidate_idx + 1),
            "cosine_similarity": float(
                similarity[reference_idx, candidate_idx]
            )
        })

    alignment_records.append({
        "source": item["path"],
        "seed": item["seed"],
        "K": item["K"],
        "best_topic_for_ref_T03": local_match + 1,
        "best_cosine_similarity": float(
            similarity[2, local_match]
        ),
        "global_assignment_T03": (
            int(globally_matched + 1)
            if globally_matched is not None else pd.NA
        ),
        "global_assignment_similarity": (
            float(similarity[2, globally_matched])
            if globally_matched is not None else np.nan
        ),
        "is_reference": identifier == id(reference_model)
    })

topic_correspondence_diagnostic_df = pd.DataFrame(
    alignment_records
)

topic_mapping_diagnostic_df = pd.DataFrame(
    mapping_records
)

if not topic_correspondence_diagnostic_df.empty:

    display(
        topic_correspondence_diagnostic_df.round(5)
    )

else:
    print("No topic alignment results available.")

# --------------------------------------------------
# 4. Inspect inventory seed 73 availability
# --------------------------------------------------

print("\n" + "=" * 72)
print("INVENTORY SEED 73 INVESTIGATION")
print("=" * 72)

inventory_seeds = []

if "topic_inventory_df" in globals():
    if "reference_seed" in topic_inventory_df.columns:
        inventory_seeds = (
            topic_inventory_df["reference_seed"]
            .dropna()
            .unique()
            .tolist()
        )

print("Inventory seed(s):", inventory_seeds)
print("Reference model seed:", reference_seed)

seed73_models = [
    item
    for item in discovered_models.values()
    if str(item["seed"]) == "73"
]

print(
    "Compatible fitted seed-73 models discovered:",
    len(seed73_models)
)

if seed73_models:

    seed73_paths = [m["path"] for m in seed73_models]

    print("\nSeed-73 source paths:")

    for path in seed73_paths:
        print(" -", path)

else:
    print(
        "Seed-73 model not recovered from the "
        "searched notebook objects."
    )

# --------------------------------------------------
# 5. Diagnostic conclusions
# --------------------------------------------------

print("\n" + "=" * 72)
print("FINAL CROSS-MODEL AUDIT")
print("=" * 72)

print("Reference component matrix: VERIFIED")
print(
    "Fitted compatible NMF models:",
    len(discovered_models)
)

print(
    "Reference seed-73 metadata agreement:",
    str(reference_seed) == "73"
)

print(
    "Feature dimension compatibility:",
    "PASSED FOR DISCOVERED MODELS"
)

print(
    "Identical feature ordering across models:",
    "NOT INDEPENDENTLY VERIFIED"
)

print(
    "Cosine component matching:",
    "COMPLETED AS DIAGNOSTIC"
)

print(
    "Six-solution robustness correspondence:",
    "PENDING SOURCE-LEVEL VERIFICATION"
)

print("Original statistical results modified: NO")
print("Files saved: NONE")
print("\nSTEP 12.27H.16: DIAGNOSTIC COMPLETED")



### Step 12.27H.17 — Reference Topic Identity and Seed-73 Inventory Reconciliation

**Objective:** Verify topic correspondence among the three K=7 NMF solutions (seeds 11, 42, and 73), and determine whether the existing topic vocabulary inventory corresponds to the fitted seed-73 model.

**Procedure:**
1. Validate reference model dimensions and available vocabulary.
2. Compute cosine similarity between reference and alternative NMF topic components.
3. Obtain optimal one-to-one topic assignments and inspect matching ambiguity.
4. Reconcile the seed-73 vocabulary inventory with fitted topic components.
5. Inspect the reference Topic 03 and its matched counterparts.

**Scientific safeguards:** Topic identifiers are not assumed interchangeable across model solutions. Matching is treated as a diagnostic until feature-order provenance and substantive topic correspondence are verified. Statistical estimates and existing publication outputs remain unchanged.

**Expected output:** Cross-seed correspondence tables, inventory reconciliation results, and a topic-identity audit.

**Export policy:** No files are saved in this computational step.


In [ ]:

# STEP 12.27H.17 — TOPIC IDENTITY AND INVENTORY RECONCILIATION
# Computational audit only. No imports, exports, or file modifications.

print("=" * 72)
print("STEP 12.27H.17 — TOPIC IDENTITY RECONCILIATION")
print("=" * 72)

# ------------------------------------------------------------
# 1. Validate required notebook objects
# ------------------------------------------------------------

required_objects = [
    "model",
    "nmf_selection_bank",
    "topic_inventory_df",
    "focal_vectorizer",
]

missing = [name for name in required_objects if name not in globals()]

if missing:
    raise RuntimeError(f"Missing notebook objects: {missing}")

reference_model_h17 = model
feature_names_h17 = np.asarray(
    focal_vectorizer.get_feature_names_out()
)

H_reference_h17 = np.asarray(
    reference_model_h17.components_, dtype=float
)

assert H_reference_h17.shape == (7, 18981)
assert len(feature_names_h17) == 18981
assert int(reference_model_h17.random_state) == 11

print("\nREFERENCE MODEL")
print("Topics:", H_reference_h17.shape[0])
print("Features:", H_reference_h17.shape[1])
print("Reference seed:", reference_model_h17.random_state)

# ------------------------------------------------------------
# 2. Recover fitted K=7 models
# ------------------------------------------------------------

models_h17 = {11: reference_model_h17}

for seed in [42, 73]:
    key = (7, seed)

    if key not in nmf_selection_bank:
        raise KeyError(f"Missing fitted NMF model: {key}")

    entry = nmf_selection_bank[key]

    fitted = entry["model"] if isinstance(entry, dict) else entry

    if np.asarray(fitted.components_).shape != H_reference_h17.shape:
        raise ValueError(f"Incompatible component matrix: {key}")

    if int(fitted.random_state) != seed:
        raise ValueError(f"Unexpected random state: {key}")

    models_h17[seed] = fitted

print("\nFITTED MODEL VALIDATION")
for seed, fitted in models_h17.items():
    print(
        f"Seed {seed}: "
        f"K={fitted.n_components}, "
        f"H={fitted.components_.shape}"
    )

# Check whether the reference is also stored in the model bank.

if (7, 11) in nmf_selection_bank:
    entry = nmf_selection_bank[(7, 11)]
    bank_reference = entry["model"] if isinstance(entry, dict) else entry

    reference_error = float(
        np.max(
            np.abs(
                H_reference_h17 -
                np.asarray(bank_reference.components_)
            )
        )
    )

    print("\nReference-versus-bank component error:", reference_error)
else:
    print("\nSeed-11 model-bank duplicate: NOT AVAILABLE")

# ------------------------------------------------------------
# 3. Normalize topic components
# ------------------------------------------------------------

def normalize_topic_rows_h17(H):
    H = np.asarray(H, dtype=float)

    if not np.isfinite(H).all():
        raise ValueError("Non-finite topic weights detected.")

    norms = np.linalg.norm(H, axis=1)

    if np.any(norms <= 0):
        raise ValueError("Zero-norm topic component detected.")

    return H / norms[:, None]


def optimal_assignment_h17(similarity):
    """
    Exact maximum-weight one-to-one assignment.
    Uses dynamic programming for K=7.
    """
    n = similarity.shape[0]

    if similarity.shape != (n, n):
        raise ValueError("Assignment requires a square matrix.")

    states = {0: (0.0, ())}

    for i in range(n):
        next_states = {}

        for mask, (score, assigned) in states.items():
            for j in range(n):
                if mask & (1 << j):
                    continue

                new_mask = mask | (1 << j)
                new_score = score + float(similarity[i, j])

                if (
                    new_mask not in next_states
                    or new_score > next_states[new_mask][0]
                ):
                    next_states[new_mask] = (
                        new_score,
                        assigned + (j,)
                    )

        states = next_states

    return states[(1 << n) - 1][1]


H_reference_normalized_h17 = normalize_topic_rows_h17(
    H_reference_h17
)

alignment_rows_h17 = []
similarity_matrices_h17 = {}

for seed, fitted in models_h17.items():

    H_target = normalize_topic_rows_h17(fitted.components_)

    similarity = H_reference_normalized_h17 @ H_target.T

    similarity_matrices_h17[seed] = similarity

    assigned_topics = optimal_assignment_h17(similarity)

    for reference_index in range(7):

        similarities = similarity[reference_index]

        ranked = np.argsort(similarities)[::-1]

        best_index = int(ranked[0])
        second_index = int(ranked[1])
        assigned_index = int(assigned_topics[reference_index])

        alignment_rows_h17.append({
            "target_seed": seed,
            "reference_topic": reference_index + 1,
            "local_best_topic": best_index + 1,
            "local_best_cosine": float(similarities[best_index]),
            "second_best_cosine": float(similarities[second_index]),
            "local_margin": float(
                similarities[best_index] -
                similarities[second_index]
            ),
            "global_assigned_topic": assigned_index + 1,
            "global_cosine": float(similarities[assigned_index]),
            "local_global_agree": best_index == assigned_index,
        })

topic_alignment_h17_df = pd.DataFrame(alignment_rows_h17)

print("\n" + "=" * 72)
print("CROSS-SEED TOPIC CORRESPONDENCE")
print("=" * 72)

display(
    topic_alignment_h17_df.round(5)
)

# ------------------------------------------------------------
# 4. Inspect reference Topic 03 across K=7 solutions
# ------------------------------------------------------------

topic03_alignment_h17_df = (
    topic_alignment_h17_df
    .loc[lambda d: d["reference_topic"] == 3]
    .copy()
)

print("\n" + "=" * 72)
print("REFERENCE TOPIC 03 — MATCHING DIAGNOSTICS")
print("=" * 72)

display(topic03_alignment_h17_df.round(6))

def top_terms_h17(fitted, topic_index, n_terms=12):

    component = np.asarray(
        fitted.components_[topic_index], dtype=float
    )

    indices = np.argsort(component)[::-1][:n_terms]

    return [str(feature_names_h17[i]) for i in indices]


topic03_vocabulary_rows_h17 = []

for row in topic03_alignment_h17_df.itertuples(index=False):

    seed = int(row.target_seed)
    matched_index = int(row.global_assigned_topic) - 1

    terms = top_terms_h17(
        models_h17[seed],
        matched_index,
        n_terms=12
    )

    topic03_vocabulary_rows_h17.append({
        "seed": seed,
        "matched_topic": matched_index + 1,
        "cosine_similarity": row.global_cosine,
        "top_terms": ", ".join(terms)
    })

topic03_vocabulary_h17_df = pd.DataFrame(
    topic03_vocabulary_rows_h17
)

print("\nREFERENCE TOPIC 03 — MATCHED VOCABULARIES")
display(topic03_vocabulary_h17_df)

# ------------------------------------------------------------
# 5. Reconcile the seed-73 vocabulary inventory
# ------------------------------------------------------------

inventory_h17 = topic_inventory_df.copy()

required_inventory_columns = {
    "n_topics", "reference_seed", "topic_id", "top_terms"
}

if not required_inventory_columns.issubset(inventory_h17.columns):
    raise ValueError(
        "Inventory columns are incompatible with this audit."
    )

inventory73_h17 = inventory_h17.loc[
    (pd.to_numeric(inventory_h17["n_topics"]) == 7) &
    (pd.to_numeric(inventory_h17["reference_seed"]) == 73)
].copy()

if len(inventory73_h17) != 7:
    raise ValueError(
        f"Expected seven seed-73 inventory topics; "
        f"found {len(inventory73_h17)}."
    )

def parse_inventory_terms_h17(value):

    if isinstance(value, (list, tuple, np.ndarray)):
        values = value
    else:
        values = str(value).split(",")

    return [
        " ".join(str(term).lower().strip().split())
        for term in values
        if str(term).strip()
    ]


inventory_rows_h17 = []

seed73_model_h17 = models_h17[73]

for _, row in inventory73_h17.iterrows():

    inventory_topic = int(row["topic_id"])

    inventory_terms = parse_inventory_terms_h17(
        row["top_terms"]
    )

    inventory_set = set(inventory_terms[:12])

    match_scores = []

    for model_topic_index in range(7):

        actual_terms = top_terms_h17(
            seed73_model_h17,
            model_topic_index,
            n_terms=12
        )

        actual_set = set(
            " ".join(term.lower().split())
            for term in actual_terms
        )

        match_scores.append(
            len(inventory_set.intersection(actual_set))
        )

    best_index = int(np.argmax(match_scores))

    inventory_rows_h17.append({
        "inventory_topic": inventory_topic,
        "same_id_overlap": match_scores[inventory_topic - 1],
        "best_matching_model_topic": best_index + 1,
        "best_top12_overlap": match_scores[best_index],
        "topic_id_agrees": (
            inventory_topic == best_index + 1
        ),
    })

inventory_reconciliation_h17_df = pd.DataFrame(
    inventory_rows_h17
).sort_values("inventory_topic").reset_index(drop=True)

print("\n" + "=" * 72)
print("SEED-73 INVENTORY RECONCILIATION")
print("=" * 72)

display(inventory_reconciliation_h17_df)

# ------------------------------------------------------------
# 6. Final diagnostic summary
# ------------------------------------------------------------

print("\n" + "=" * 72)
print("FINAL TOPIC IDENTITY AUDIT")
print("=" * 72)

print("Reference NMF identification: PASSED")
print("K=7 fitted model recovery: PASSED")
print("Component dimensions: PASSED")
print("One-to-one topic matching: COMPUTED")

seed73_topic03 = topic03_alignment_h17_df.loc[
    topic03_alignment_h17_df["target_seed"] == 73
].iloc[0]

print(
    "\nReference T03 matches seed-73 topic:",
    int(seed73_topic03["global_assigned_topic"])
)

print(
    "Correspondence cosine:",
    round(float(seed73_topic03["global_cosine"]), 6)
)

print(
    "Seed-73 inventory same-ID best matches:",
    int(inventory_reconciliation_h17_df["topic_id_agrees"].sum()),
    "/7"
)

print(
    "\nFeature-order provenance across fitted models: "
    "REQUIRES INDEPENDENT VERIFICATION"
)

print(
    "Semantic equivalence across model solutions: "
    "PENDING REVIEW"
)

print("Original statistical results modified: NO")
print("Files saved: NONE")
print("\nSTEP 12.27H.17: DIAGNOSTIC COMPLETED")



### Step 12.27H.18 — Topic Inventory Provenance and Vocabulary Reconciliation

**Objective:** Investigate the disagreement between the stored topic inventory and the fitted NMF reference model.

This diagnostic will:

1. Verify the K=7 NMF models fitted with random seeds 11, 42, and 73.
2. Check consistency among available feature-vocabulary candidates.
3. Reconstruct the leading weighted terms directly from fitted NMF components.
4. Compare the stored seed-73 topic inventory with each fitted model using top-term overlap.
5. Apply one-to-one topic matching to distinguish topic permutation from substantive vocabulary disagreement.
6. Inspect the reference Topic 03 across models.

**Interpretation:** Matching topic identifiers are not assumed to represent equivalent semantic content. Vocabulary overlap is a diagnostic rather than proof of topic equivalence.

**Output policy:** Diagnostics only. No files are saved, and existing statistical estimates remain unchanged.


In [ ]:

# ============================================================
# STEP 12.27H.18
# TOPIC INVENTORY PROVENANCE AND VOCABULARY RECONCILIATION
# ============================================================

print("=" * 68)
print("STEP 12.27H.18 — TOPIC INVENTORY PROVENANCE AUDIT")
print("=" * 68)

# ------------------------------------------------------------
# 1. Verify required notebook objects
# ------------------------------------------------------------

required_objects = [
    "model",
    "nmf_selection_bank",
    "topic_inventory_df",
    "linear_sum_assignment",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

if missing_objects:
    raise RuntimeError(
        f"Missing notebook objects: {missing_objects}"
    )

SEEDS_H18 = (11, 42, 73)
K_H18 = 7
TOP_N_H18 = 12

models_h18 = {}

for seed in SEEDS_H18:
    key = (K_H18, seed)

    if key not in nmf_selection_bank:
        raise RuntimeError(
            f"Required fitted model is missing: {key}"
        )

    candidate = nmf_selection_bank[key]["model"]

    H = np.asarray(candidate.components_, dtype=float)

    if H.ndim != 2 or H.shape[0] != K_H18:
        raise ValueError(
            f"Invalid NMF component shape for seed {seed}: "
            f"{H.shape}"
        )

    models_h18[seed] = candidate

reference_H_h18 = np.asarray(model.components_, dtype=float)

bank_reference_H_h18 = np.asarray(
    models_h18[11].components_,
    dtype=float
)

assert reference_H_h18.shape == bank_reference_H_h18.shape

reference_error_h18 = float(
    np.max(
        np.abs(
            reference_H_h18 - bank_reference_H_h18
        )
    )
)

print("\nREFERENCE MODEL VERIFICATION")
print("Reference shape:", reference_H_h18.shape)
print("Reference-versus-bank error:", reference_error_h18)

assert reference_error_h18 < 1e-10

n_features_h18 = reference_H_h18.shape[1]

for seed, fitted_model in models_h18.items():
    assert fitted_model.components_.shape == (
        K_H18, n_features_h18
    )

print("MODEL MATRIX VALIDATION: PASSED")


# ------------------------------------------------------------
# 2. Audit available vocabulary orderings
# ------------------------------------------------------------

vocabulary_candidates_h18 = {}

for name in [
    "focal_features",
    "segment_features",
    "feature_names",
]:
    if name in globals():
        candidate = np.asarray(
            globals()[name],
            dtype=str
        ).ravel()

        if len(candidate) == n_features_h18:
            vocabulary_candidates_h18[name] = candidate

if "focal_vectorizer" in globals():
    candidate = np.asarray(
        focal_vectorizer.get_feature_names_out(),
        dtype=str
    )

    if len(candidate) == n_features_h18:
        vocabulary_candidates_h18[
            "focal_vectorizer"
        ] = candidate

if not vocabulary_candidates_h18:
    raise RuntimeError(
        "No dimension-compatible vocabulary was found."
    )

vocabulary_names_h18 = list(
    vocabulary_candidates_h18.keys()
)

vocabulary_h18 = vocabulary_candidates_h18[
    vocabulary_names_h18[0]
]

vocabulary_audit_h18 = pd.DataFrame([
    {
        "source": name,
        "n_features": len(values),
        "ordering_agrees": bool(
            np.array_equal(values, vocabulary_h18)
        ),
    }
    for name, values in vocabulary_candidates_h18.items()
])

print("\nVOCABULARY ORDERING AUDIT")
display(vocabulary_audit_h18)

if not vocabulary_audit_h18["ordering_agrees"].all():
    raise RuntimeError(
        "Vocabulary ordering disagreement detected. "
        "Topic-term interpretation must stop."
    )

assert len(np.unique(vocabulary_h18)) == n_features_h18

print("AVAILABLE VOCABULARY ORDERINGS: CONSISTENT")
print(
    "Independent training-pipeline provenance: "
    "NOT YET VERIFIED"
)


# ------------------------------------------------------------
# 3. Reconstruct top terms from fitted NMF models
# ------------------------------------------------------------

model_terms_h18 = {}
weighted_terms_records_h18 = []

for seed, fitted_model in models_h18.items():

    H = np.asarray(
        fitted_model.components_,
        dtype=float
    )

    for topic_idx in range(K_H18):

        ranked_indices = np.argsort(
            -H[topic_idx],
            kind="stable"
        )[:TOP_N_H18]

        terms = vocabulary_h18[ranked_indices].tolist()

        model_terms_h18[
            (seed, topic_idx + 1)
        ] = terms

        for rank, feature_idx in enumerate(
            ranked_indices,
            start=1
        ):
            weighted_terms_records_h18.append({
                "seed": seed,
                "topic_id": topic_idx + 1,
                "rank": rank,
                "term": str(vocabulary_h18[feature_idx]),
                "nmf_weight": float(
                    H[topic_idx, feature_idx]
                ),
            })

model_weighted_terms_h18 = pd.DataFrame(
    weighted_terms_records_h18
)

assert len(model_weighted_terms_h18) == (
    len(SEEDS_H18) * K_H18 * TOP_N_H18
)

print("\nREFERENCE TOPIC 03 — FITTED MODEL VOCABULARY")

display(
    model_weighted_terms_h18.loc[
        (model_weighted_terms_h18["seed"] == 11)
        & (model_weighted_terms_h18["topic_id"] == 3)
    ].reset_index(drop=True).round(6)
)


# ------------------------------------------------------------
# 4. Retrieve the original seed-73 topic inventory
# ------------------------------------------------------------

inventory_h18 = topic_inventory_df.copy()

required_columns_h18 = {
    "n_topics",
    "reference_seed",
    "topic_id",
    "top_terms",
}

assert required_columns_h18.issubset(
    inventory_h18.columns
)

inventory_h18 = inventory_h18.loc[
    (
        pd.to_numeric(
            inventory_h18["n_topics"],
            errors="coerce"
        ) == K_H18
    )
    & (
        pd.to_numeric(
            inventory_h18["reference_seed"],
            errors="coerce"
        ) == 73
    )
].copy()

inventory_h18["topic_id"] = pd.to_numeric(
    inventory_h18["topic_id"]
).astype(int)

inventory_h18 = inventory_h18.sort_values(
    "topic_id"
).reset_index(drop=True)

assert len(inventory_h18) == K_H18
assert inventory_h18["topic_id"].tolist() == list(
    range(1, K_H18 + 1)
)


def normalize_inventory_terms_h18(value):
    if isinstance(value, (list, tuple, np.ndarray)):
        tokens = value
    else:
        tokens = str(value).split(",")

    return [
        str(term).strip().lower()
        for term in tokens
        if str(term).strip()
    ]


inventory_terms_h18 = {
    int(row.topic_id): normalize_inventory_terms_h18(
        row.top_terms
    )[:TOP_N_H18]
    for row in inventory_h18.itertuples(index=False)
}

assert all(
    len(terms) > 0
    for terms in inventory_terms_h18.values()
)

print("\nSTORED SEED-73 INVENTORY")

display(
    inventory_h18[
        ["topic_id", "reference_seed", "top_terms"]
    ]
)


# ------------------------------------------------------------
# 5. Compare inventory terms against fitted models
# ------------------------------------------------------------

matching_records_h18 = []
matching_details_h18 = {}

for seed in SEEDS_H18:

    overlap_matrix = np.zeros(
        (K_H18, K_H18),
        dtype=int
    )

    for inventory_topic in range(1, K_H18 + 1):

        inventory_set = set(
            inventory_terms_h18[inventory_topic]
        )

        for fitted_topic in range(1, K_H18 + 1):

            fitted_set = set(
                term.lower()
                for term in model_terms_h18[
                    (seed, fitted_topic)
                ]
            )

            overlap_matrix[
                inventory_topic - 1,
                fitted_topic - 1
            ] = len(
                inventory_set.intersection(fitted_set)
            )

    row_ind, col_ind = linear_sum_assignment(
        -overlap_matrix
    )

    matches = {
        int(r + 1): int(c + 1)
        for r, c in zip(row_ind, col_ind)
    }

    same_id_overlap = int(
        np.trace(overlap_matrix)
    )

    assigned_overlap = int(
        overlap_matrix[row_ind, col_ind].sum()
    )

    same_id_matches = sum(
        matches[topic_id] == topic_id
        for topic_id in range(1, K_H18 + 1)
    )

    matching_details_h18[seed] = {
        "overlap_matrix": overlap_matrix,
        "assignment": matches,
        "total_assigned_overlap": assigned_overlap,
    }

    matching_records_h18.append({
        "model_seed": seed,
        "same_id_total_overlap": same_id_overlap,
        "optimal_total_overlap": assigned_overlap,
        "same_id_assignments": same_id_matches,
        "topics_compared": K_H18,
    })

inventory_matching_summary_h18 = pd.DataFrame(
    matching_records_h18
).sort_values(
    "optimal_total_overlap",
    ascending=False
).reset_index(drop=True)

print("\nINVENTORY-TO-MODEL MATCHING SUMMARY")
display(inventory_matching_summary_h18)


# ------------------------------------------------------------
# 6. Inspect each inventory topic against seed-73 model
# ------------------------------------------------------------

seed73_matches_h18 = matching_details_h18[
    73
]["assignment"]

seed73_matrix_h18 = matching_details_h18[
    73
]["overlap_matrix"]

seed73_details_h18 = []

for inventory_topic in range(1, K_H18 + 1):

    fitted_topic = seed73_matches_h18[
        inventory_topic
    ]

    seed73_details_h18.append({
        "inventory_topic": inventory_topic,
        "matched_fitted_topic": fitted_topic,
        "same_id_overlap": int(
            seed73_matrix_h18[
                inventory_topic - 1,
                inventory_topic - 1
            ]
        ),
        "matched_overlap": int(
            seed73_matrix_h18[
                inventory_topic - 1,
                fitted_topic - 1
            ]
        ),
        "topic_id_agrees": (
            inventory_topic == fitted_topic
        ),
        "inventory_terms": ", ".join(
            inventory_terms_h18[inventory_topic][:6]
        ),
        "matched_model_terms": ", ".join(
            model_terms_h18[(73, fitted_topic)][:6]
        ),
    })

seed73_reconciliation_h18 = pd.DataFrame(
    seed73_details_h18
)

print("\nSEED-73 INVENTORY RECONCILIATION")
display(seed73_reconciliation_h18)


# ------------------------------------------------------------
# 7. Direct comparison of reference Topic 03
# ------------------------------------------------------------

topic03_comparison_h18 = []

reference_topic_h18 = 3

matched_topic_by_seed_h18 = {
    11: 3,
    42: 2,
    73: 2,
}

for seed, topic_id in matched_topic_by_seed_h18.items():

    H = np.asarray(
        models_h18[seed].components_,
        dtype=float
    )

    reference_vector = reference_H_h18[
        reference_topic_h18 - 1
    ]

    target_vector = H[topic_id - 1]

    denominator = (
        np.linalg.norm(reference_vector)
        * np.linalg.norm(target_vector)
    )

    cosine = (
        float(
            np.dot(
                reference_vector,
                target_vector
            ) / denominator
        )
        if denominator > 0 else np.nan
    )

    topic03_comparison_h18.append({
        "seed": seed,
        "matched_topic": topic_id,
        "cosine_similarity": cosine,
        "top_terms": ", ".join(
            model_terms_h18[(seed, topic_id)]
        ),
    })

topic03_vocabulary_audit_h18 = pd.DataFrame(
    topic03_comparison_h18
)

print("\nREFERENCE TOPIC 03 — CROSS-SEED VOCABULARY")
display(topic03_vocabulary_audit_h18)


# ------------------------------------------------------------
# 8. Final diagnostic status
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FINAL INVENTORY PROVENANCE DIAGNOSTICS")
print("=" * 68)

print("Reference model recovery: PASSED")
print("Fitted model dimensions: PASSED")
print("Available vocabulary ordering: CONSISTENT")
print("Inventory matching: COMPUTED")
print("Original inventory reconstruction: NOT YET VERIFIED")
print("Independent feature-order provenance: PENDING")
print("Cross-model semantic equivalence: PENDING")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.18: DIAGNOSTIC COMPLETED")



### Step 12.27H.19 — Stored Model Lineage and Inventory Origin Audit

**Objective:** Determine whether the stored K=7, seed-73 topic inventory originated from any available fitted NMF model or from a different analytical pipeline.

**Procedure:**
1. Discover compatible cached NMF models.
2. Compare inventory vocabulary against fitted components using top-12, top-40, and top-100 term overlap.
3. Apply optimal one-to-one matching to K=7 models.
4. Identify possible model-source and topic-identity inconsistencies.
5. Inspect available notebook execution history for the original inventory construction.

**Scientific safeguards:** Term overlap is diagnostic, not proof of model provenance. No topic labels, statistical results, or fitted models are modified.

**Saving policy:** No files are saved in this step.


In [ ]:

# ============================================================
# STEP 12.27H.19
# STORED MODEL LINEAGE AND INVENTORY ORIGIN AUDIT
# ============================================================

print("=" * 68)
print("STEP 12.27H.19 — MODEL LINEAGE AND INVENTORY ORIGIN")
print("=" * 68)

# ------------------------------------------------------------
# 1. Validate previous diagnostic objects
# ------------------------------------------------------------

required_h19 = [
    "inventory_terms_h18",
    "vocabulary_h18",
    "models_h18",
    "linear_sum_assignment",
]

missing_h19 = [
    name for name in required_h19
    if name not in globals()
]

if missing_h19:
    raise RuntimeError(
        f"Run Step 12.27H.18 first: {missing_h19}"
    )

vocab_h19 = np.asarray(vocabulary_h18, dtype=str)
n_features_h19 = len(vocab_h19)

inventory_sets_h19 = [
    set(
        str(term).strip().lower()
        for term in inventory_terms_h18[topic]
    )
    for topic in range(1, 8)
]

inventory_term_total_h19 = sum(
    len(terms) for terms in inventory_sets_h19
)

# ------------------------------------------------------------
# 2. Discover available fitted NMF models
# ------------------------------------------------------------

candidate_models_h19 = []
seen_models_h19 = set()
seen_containers_h19 = set()


def discover_models_h19(obj, source, depth=0):

    if depth > 5:
        return

    if hasattr(obj, "components_"):

        H = np.asarray(obj.components_)

        if (
            H.ndim == 2
            and H.shape[1] == n_features_h19
            and H.shape[0] >= 7
            and id(obj) not in seen_models_h19
        ):
            candidate_models_h19.append(
                (source, obj)
            )
            seen_models_h19.add(id(obj))

        return

    if id(obj) in seen_containers_h19:
        return

    if isinstance(obj, dict):

        seen_containers_h19.add(id(obj))

        for key, value in obj.items():
            discover_models_h19(
                value,
                f"{source}[{repr(key)}]",
                depth + 1
            )

    elif isinstance(obj, (list, tuple)):

        seen_containers_h19.add(id(obj))

        for index, value in enumerate(obj):
            discover_models_h19(
                value,
                f"{source}[{index}]",
                depth + 1
            )


candidate_roots_h19 = [
    "model",
    "nmf_selection_bank",
    "nmf_ablation_bank",
    "ablation_models",
    "segment_nmf_model",
    "selected_model",
    "reference_model",
    "nmf_model_bank",
]

for root_name in candidate_roots_h19:
    if root_name in globals():
        discover_models_h19(
            globals()[root_name],
            root_name
        )

if not candidate_models_h19:
    raise RuntimeError(
        "No compatible fitted models were discovered."
    )

model_registry_h19 = pd.DataFrame([
    {
        "source": source,
        "K": obj.components_.shape[0],
        "seed": str(getattr(
            obj, "random_state", "unknown"
        )),
        "n_features": obj.components_.shape[1],
    }
    for source, obj in candidate_models_h19
])

print("\nCOMPATIBLE FITTED MODEL INVENTORY")
display(model_registry_h19)

# ------------------------------------------------------------
# 3. Compare inventory terms with every fitted model
# ------------------------------------------------------------

matching_results_h19 = []
assignment_records_h19 = []

for source, fitted in candidate_models_h19:

    H = np.asarray(
        fitted.components_,
        dtype=float
    )

    K = H.shape[0]

    ranked_terms = {}

    for topic_index in range(K):

        ranked_indices = np.argsort(
            -H[topic_index],
            kind="stable"
        )[:100]

        ranked_terms[topic_index] = [
            str(term).lower()
            for term in vocab_h19[ranked_indices]
        ]

    overlap_matrices = {}

    for cutoff in (12, 40, 100):

        overlap = np.zeros((7, K), dtype=int)

        for i in range(7):
            for j in range(K):
                overlap[i, j] = len(
                    inventory_sets_h19[i].intersection(
                        ranked_terms[j][:cutoff]
                    )
                )

        overlap_matrices[cutoff] = overlap

    scores = {}

    for cutoff, matrix in overlap_matrices.items():

        row_idx, col_idx = linear_sum_assignment(
            -matrix
        )

        scores[cutoff] = int(
            matrix[row_idx, col_idx].sum()
        )

    # Use top-12 matching as the primary correspondence.
    matrix12 = overlap_matrices[12]

    row_idx, col_idx = linear_sum_assignment(
        -matrix12
    )

    for i, j in zip(row_idx, col_idx):

        assignment_records_h19.append({
            "source": source,
            "K": K,
            "inventory_topic": int(i + 1),
            "matched_model_topic": int(j + 1),
            "overlap_at_12": int(
                overlap_matrices[12][i, j]
            ),
            "overlap_at_40": int(
                overlap_matrices[40][i, j]
            ),
            "overlap_at_100": int(
                overlap_matrices[100][i, j]
            ),
        })

    exact_matches = sum(
        matrix12[i, j] == len(inventory_sets_h19[i])
        for i, j in zip(row_idx, col_idx)
    )

    matching_results_h19.append({
        "source": source,
        "K": K,
        "seed": str(getattr(
            fitted, "random_state", "unknown"
        )),
        "optimal_overlap_12": scores[12],
        "optimal_overlap_40": scores[40],
        "optimal_overlap_100": scores[100],
        "exact_top12_matches": int(exact_matches),
    })

model_lineage_audit_h19 = pd.DataFrame(
    matching_results_h19
)

model_lineage_audit_h19 = (
    model_lineage_audit_h19.sort_values(
        [
            "optimal_overlap_12",
            "optimal_overlap_40"
        ],
        ascending=False
    ).reset_index(drop=True)
)

model_topic_assignments_h19 = pd.DataFrame(
    assignment_records_h19
)

print("\nINVENTORY-TO-MODEL LINEAGE COMPARISON")
print(
    "Maximum available inventory-term matches:",
    inventory_term_total_h19
)

display(model_lineage_audit_h19)

# ------------------------------------------------------------
# 4. Focus on all fitted K=7 solutions
# ------------------------------------------------------------

k7_lineage_h19 = model_lineage_audit_h19.loc[
    model_lineage_audit_h19["K"] == 7
].copy()

if k7_lineage_h19.empty:
    raise RuntimeError(
        "No fitted K=7 model was discovered."
    )

print("\nK=7 FITTED MODEL COMPARISON")
display(k7_lineage_h19)

best_k7_source_h19 = k7_lineage_h19.iloc[0]["source"]

best_k7_matches_h19 = model_topic_assignments_h19.loc[
    model_topic_assignments_h19["source"]
    == best_k7_source_h19
].sort_values(
    "inventory_topic"
).reset_index(drop=True)

print("\nBEST K=7 INVENTORY CORRESPONDENCE")
print("Source:", best_k7_source_h19)
display(best_k7_matches_h19)

# ------------------------------------------------------------
# 5. Check available notebook construction history
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("INVENTORY CONSTRUCTION HISTORY")
print("=" * 68)

history_matches_h19 = []

try:
    ipython_h19 = get_ipython()

    history_h19 = (
        ipython_h19.history_manager.input_hist_raw
    )

    for cell_number, cell_source in enumerate(history_h19):

        lines = cell_source.splitlines()

        for line_index, line in enumerate(lines):

            compact = line.replace(" ", "")

            if (
                "topic_inventory_df=" in compact
                or "topic_inventory_df=pd." in compact
            ):
                start = max(0, line_index - 4)
                end = min(
                    len(lines),
                    line_index + 12
                )

                history_matches_h19.append({
                    "cell_number": cell_number,
                    "excerpt": "\n".join(
                        lines[start:end]
                    ),
                })

except (NameError, AttributeError, TypeError):
    print("Notebook execution history unavailable.")

if history_matches_h19:

    print(
        "Possible construction records:",
        len(history_matches_h19)
    )

    for item in history_matches_h19[-5:]:
        print("\nCell:", item["cell_number"])
        print(item["excerpt"])

else:
    print(
        "No inventory-construction assignment "
        "found in available execution history."
    )

# ------------------------------------------------------------
# 6. Final scientific diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FINAL MODEL LINEAGE AUDIT")
print("=" * 68)

print(
    "Compatible fitted models:",
    len(candidate_models_h19)
)

print(
    "Compatible K=7 models:",
    len(k7_lineage_h19)
)

print(
    "Best K=7 top-12 overlap:",
    int(k7_lineage_h19.iloc[0]["optimal_overlap_12"]),
    "/",
    inventory_term_total_h19
)

print(
    "Exact inventory-topic reconstructions:",
    int(k7_lineage_h19.iloc[0]["exact_top12_matches"]),
    "/ 7"
)

print("Model lineage comparison: COMPLETED")
print("Inventory provenance: REQUIRES INTERPRETATION")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.19: DIAGNOSTIC COMPLETED")



### Step 12.27H.20 — Original Topic Inventory Construction Audit

**Objective:** Trace the original construction of the stored topic inventory and identify the model, random seed, vocabulary source, and component-ranking procedure.

**Procedure:**
1. Locate the original inventory-construction code in notebook execution history.
2. Recover the source statements responsible for topic ranking and inventory creation.
3. Examine the available inventory metadata and its relationship to the fitted reference model.
4. Identify unresolved provenance dependencies before rebuilding or interpreting topic labels.

**Scientific safeguards:** This is a read-only diagnostic. No model refitting, topic relabeling, statistical modifications, or file exports are performed.


In [ ]:

# ============================================================
# STEP 12.27H.20
# ORIGINAL TOPIC INVENTORY CONSTRUCTION AUDIT
# ============================================================

print("=" * 68)
print("STEP 12.27H.20 — INVENTORY CONSTRUCTION PROVENANCE")
print("=" * 68)

# ------------------------------------------------------------
# 1. Verify required objects
# ------------------------------------------------------------

assert "topic_inventory_df" in globals()
assert "model_lineage_audit_h19" in globals()

history_h20 = list(
    get_ipython().history_manager.input_hist_raw
)

# ------------------------------------------------------------
# 2. Locate the original inventory-producing cell
# ------------------------------------------------------------

producer_cells_h20 = []

for cell_number, source in enumerate(history_h20):

    if "STEP 12.27H.20" in source:
        continue

    source_lines = source.splitlines()

    production_lines = [
        i for i, line in enumerate(source_lines)
        if line.strip().startswith((
            "topic_inventory_records =",
            "topic_inventory_records=",
            "topic_inventory_records.append(",
        ))
    ]

    if production_lines:
        producer_cells_h20.append({
            "cell_number": cell_number,
            "source": source,
            "production_lines": production_lines,
        })

print("\nORIGINAL INVENTORY PRODUCER DISCOVERY")
print("Candidate producer cells:", len(producer_cells_h20))

if not producer_cells_h20:
    raise RuntimeError(
        "Original inventory construction was not found "
        "in the available execution history."
    )

for item in producer_cells_h20:
    print(
        "Execution cell:",
        item["cell_number"],
        "| Construction references:",
        len(item["production_lines"])
    )

# ------------------------------------------------------------
# 3. Recover relevant construction statements
# ------------------------------------------------------------

source_h20 = producer_cells_h20[0]["source"]
lines_h20 = source_h20.splitlines()

search_terms_h20 = (
    "topic_inventory_records",
    "topic_inventory_df",
    "reference_seed",
    "random_state",
    "components_",
    "top_terms",
    "get_feature_names_out",
    "feature_names",
    "vectorizer",
    "nmf_selection_bank",
    "nmf_ablation_bank",
    "reference_model",
    "topic_model",
)

matching_lines_h20 = [
    i for i, line in enumerate(lines_h20)
    if any(
        term in line
        for term in search_terms_h20
    )
]

included_lines_h20 = set()

for index in matching_lines_h20:
    included_lines_h20.update(
        range(
            max(0, index - 5),
            min(len(lines_h20), index + 9)
        )
    )

print("\n" + "=" * 68)
print("ORIGINAL INVENTORY CONSTRUCTION CODE")
print("=" * 68)

print(
    "Source execution cell:",
    producer_cells_h20[0]["cell_number"]
)

print("Total source lines:", len(lines_h20))

previous_line_h20 = -2

for index in sorted(included_lines_h20):

    if index > previous_line_h20 + 1:
        print("\n    ...\n")

    print(
        f"{index + 1:04d} | {lines_h20[index]}"
    )

    previous_line_h20 = index

# ------------------------------------------------------------
# 4. Inspect inventory metadata
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("STORED INVENTORY METADATA")
print("=" * 68)

print("Inventory shape:", topic_inventory_df.shape)
print("Columns:", list(topic_inventory_df.columns))

metadata_columns_h20 = [
    col for col in [
        "n_topics",
        "reference_seed",
        "topic_id",
        "top_terms",
    ]
    if col in topic_inventory_df.columns
]

display(
    topic_inventory_df[
        metadata_columns_h20
    ]
)

# ------------------------------------------------------------
# 5. Confirm unresolved lineage discrepancy
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("K=7 MODEL LINEAGE STATUS")
print("=" * 68)

display(
    model_lineage_audit_h19.loc[
        model_lineage_audit_h19["K"].eq(7)
    ].reset_index(drop=True)
)

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Original construction history: RECOVERED")
print("Inventory metadata: INSPECTED")
print("Model provenance: PENDING SOURCE INTERPRETATION")
print("Original NMF models modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.20: DIAGNOSTIC COMPLETED")



### Step 12.27H.21 — Inventory Source and Model Selection Verification

**Objective:** Identify the exact model-selection and vocabulary-generation procedures used to construct the original topic inventory.

**Checks:**
1. Inspect the original inventory-construction code.
2. Trace model-selection variables and fitted-model references.
3. Examine cached selected models and their random seeds.
4. Identify whether the inventory can be reconciled with the statistical reference model without refitting.

**Safeguards:** Read-only inspection. No statistical results, fitted models, or publication files will be modified.


In [ ]:

# ============================================================
# STEP 12.27H.21
# INVENTORY SOURCE AND MODEL SELECTION VERIFICATION
# ============================================================

assert "source_h20" in globals()
assert "topic_inventory_df" in globals()

print("=" * 65)
print("STEP 12.27H.21 — INVENTORY SOURCE VERIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Inspect the original model-selection code
# ------------------------------------------------------------

source_lines_h21 = source_h20.splitlines()

keywords_h21 = (
    "candidate_k",
    "selected_topic_models",
    "reference_seed",
    "random_state",
    "nmf_selection_bank",
    "nmf_ablation_bank",
    "fit_transform",
    ".fit(",
    "components_",
    "topic_inventory_records",
    "topic_inventory_df",
    "top_terms",
    "get_feature_names_out",
    "feature_names",
    "vectorizer",
    "top_indices",
    "argsort",
)

matched_indices_h21 = [
    i for i, line in enumerate(source_lines_h21)
    if any(
        keyword in line.lower()
        for keyword in keywords_h21
    )
]

context_indices_h21 = set()

for i in matched_indices_h21:
    context_indices_h21.update(
        range(
            max(0, i - 2),
            min(len(source_lines_h21), i + 4)
        )
    )

print("\nORIGINAL MODEL-SELECTION SOURCE")
print("Total source lines:", len(source_lines_h21))
print("Relevant source lines:", len(context_indices_h21))

previous_h21 = -2

for i in sorted(context_indices_h21):
    if i > previous_h21 + 1:
        print("\n...")

    print(f"{i + 1:03d} | {source_lines_h21[i]}")
    previous_h21 = i

# ------------------------------------------------------------
# 2. Inspect cached selected model objects
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SELECTED TOPIC MODEL OBJECTS")
print("=" * 65)

selected_models_h21 = globals().get(
    "selected_topic_models", None
)

selected_model_records_h21 = []

if isinstance(selected_models_h21, dict):

    print("Selected model keys:",
          list(selected_models_h21.keys()))

    for key, entry in selected_models_h21.items():

        fitted = entry

        if isinstance(entry, dict):
            fitted = entry.get("model", None)

        components = getattr(
            fitted, "components_", None
        )

        selected_model_records_h21.append({
            "selection_key": str(key),
            "object_type": type(fitted).__name__,
            "random_state": getattr(
                fitted, "random_state", None
            ),
            "component_shape": (
                tuple(components.shape)
                if components is not None
                else None
            ),
        })

    display(pd.DataFrame(selected_model_records_h21))

else:
    print("Selected topic model cache: NOT AVAILABLE")

# ------------------------------------------------------------
# 3. Summarize stored inventory configurations
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ORIGINAL INVENTORY CONFIGURATIONS")
print("=" * 65)

inventory_config_h21 = (
    topic_inventory_df
    .groupby(
        ["n_topics", "reference_seed"],
        dropna=False
    )
    .agg(
        n_records=("topic_id", "size"),
        n_unique_topics=("topic_id", "nunique"),
    )
    .reset_index()
    .sort_values(["n_topics", "reference_seed"])
)

display(inventory_config_h21)

assert inventory_config_h21["n_records"].sum() == len(
    topic_inventory_df
)

# ------------------------------------------------------------
# 4. Detect original fitted-model dependencies
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MODEL PROVENANCE DEPENDENCIES")
print("=" * 65)

dependency_names_h21 = [
    "selected_topic_models",
    "nmf_selection_bank",
    "nmf_ablation_bank",
    "focal_vectorizer",
    "focal_features",
    "segment_features",
    "feature_names",
    "model",
]

dependency_records_h21 = []

for name in dependency_names_h21:

    available = name in globals()

    dependency_records_h21.append({
        "object": name,
        "available": available,
        "object_type": (
            type(globals()[name]).__name__
            if available else None
        ),
    })

display(pd.DataFrame(dependency_records_h21))

# ------------------------------------------------------------
# 5. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Original source code inspection: COMPLETED")
print("Inventory configurations: VERIFIED")
print("Selected model cache: INSPECTED")
print("Inventory-to-model lineage: PENDING FINAL REVIEW")
print("Original fitted models modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.21: DIAGNOSTIC COMPLETED")



### Step 12.27H.22 — Original Model Selection and Inventory Lineage Audit

**Objective:** Trace the original topic inventory to its actual fitted NMF model and verify the correspondence between stored topic terms and model components.

**Checks:**
1. Inspect the original model-selection procedure.
2. Identify the structure and contents of selected model entries.
3. Verify the cached K=7 models across random seeds.
4. Compare stored inventory terms against the selected fitted model.

**Safeguards:** Diagnostic only. No model refitting, statistical recomputation, or file exports.


In [ ]:

# ============================================================
# STEP 12.27H.22
# ORIGINAL MODEL SELECTION AND INVENTORY LINEAGE AUDIT
# ============================================================

assert "source_h20" in globals()
assert "selected_topic_models" in globals()
assert "nmf_selection_bank" in globals()
assert "topic_inventory_df" in globals()

print("=" * 65)
print("STEP 12.27H.22 — INVENTORY LINEAGE AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Inspect original model-selection procedure
# ------------------------------------------------------------

source_lines_h22 = source_h20.splitlines()

print("\n" + "=" * 65)
print("ORIGINAL MODEL SELECTION CODE")
print("=" * 65)

for i in range(14, min(80, len(source_lines_h22))):
    print(f"{i + 1:03d} | {source_lines_h22[i]}")

# Locate the original inventory producer.

producer_terms_h22 = (
    "topic_inventory_records.append",
    "selected_topic_models[",
    "top_terms",
)

producer_indices_h22 = [
    i for i, line in enumerate(source_lines_h22)
    if any(
        term in line.lower()
        for term in producer_terms_h22
    )
]

print("\n" + "=" * 65)
print("INVENTORY CONSTRUCTION REFERENCES")
print("=" * 65)

for i in producer_indices_h22:
    print(f"{i + 1:03d} | {source_lines_h22[i]}")

# ------------------------------------------------------------
# 2. Inspect selected model entries correctly
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SELECTED MODEL ENTRY STRUCTURES")
print("=" * 65)

def describe_object_h22(obj):

    if hasattr(obj, "components_"):
        return (
            f"Fitted NMF | shape={obj.components_.shape} | "
            f"seed={getattr(obj, 'random_state', None)}"
        )

    if isinstance(obj, np.ndarray):
        return f"ndarray | shape={obj.shape}"

    if isinstance(obj, dict):
        return f"dict | keys={list(obj.keys())}"

    if isinstance(obj, (list, tuple)):
        return f"{type(obj).__name__} | length={len(obj)}"

    return f"{type(obj).__name__} | value={str(obj)[:100]}"


for k, entry in sorted(selected_topic_models.items()):

    print(f"\nK={k}")
    print("Entry:", describe_object_h22(entry))

    if isinstance(entry, dict):
        for key, value in entry.items():
            print(
                f"  {key}: {describe_object_h22(value)}"
            )

# ------------------------------------------------------------
# 3. Recover fitted models without modifying them
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("CACHED K=7 MODEL VERIFICATION")
print("=" * 65)

def recover_model_h22(entry):

    if hasattr(entry, "components_"):
        return entry

    if isinstance(entry, dict):

        for value in entry.values():

            if hasattr(value, "components_"):
                return value

    return None


selected_k7_h22 = recover_model_h22(
    selected_topic_models.get(7)
)

bank_models_h22 = {}

for seed in (11, 42, 73):

    entry = nmf_selection_bank.get((7, seed))
    fitted = recover_model_h22(entry)

    bank_models_h22[seed] = fitted

    if fitted is not None:

        print(
            f"Seed {seed}: FITTED | "
            f"shape={fitted.components_.shape}"
        )

    else:
        print(
            f"Seed {seed}: "
            f"FITTED MODEL NOT FOUND AT EXPECTED KEY"
        )

if selected_k7_h22 is not None:

    print("\nSelected K=7 fitted model: RECOVERED")
    print(
        "Selected seed:",
        selected_k7_h22.random_state
    )

    for seed, fitted in bank_models_h22.items():

        if fitted is not None:

            same = np.allclose(
                selected_k7_h22.components_,
                fitted.components_,
                rtol=1e-8,
                atol=1e-10,
            )

            print(
                f"Selected K=7 equals bank seed {seed}: {same}"
            )

else:
    print(
        "\nSelected K=7 fitted model: "
        "REQUIRES ADDITIONAL STRUCTURE INSPECTION"
    )

# ------------------------------------------------------------
# 4. Compare original inventory with selected K=7 model
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("INVENTORY-TO-MODEL VOCABULARY CHECK")
print("=" * 65)

inventory_k7_h22 = topic_inventory_df.loc[
    topic_inventory_df["n_topics"].eq(7)
].sort_values("topic_id").copy()

features_h22 = np.asarray(
    focal_vectorizer.get_feature_names_out()
)

comparison_records_h22 = []

if (
    selected_k7_h22 is not None
    and selected_k7_h22.components_.shape[1]
        == len(features_h22)
):

    H_h22 = selected_k7_h22.components_

    for row in inventory_k7_h22.itertuples(index=False):

        topic_index = int(row.topic_id) - 1

        stored_terms = [
            term.strip().lower()
            for term in str(row.top_terms).split(",")
            if term.strip()
        ]

        fitted_indices = np.argsort(
            H_h22[topic_index]
        )[::-1][:len(stored_terms)]

        fitted_terms = [
            str(features_h22[j]).lower()
            for j in fitted_indices
        ]

        comparison_records_h22.append({
            "topic_id": int(row.topic_id),
            "inventory_seed": int(row.reference_seed),
            "selected_model_seed":
                selected_k7_h22.random_state,
            "stored_terms": ", ".join(
                stored_terms[:5]
            ),
            "fitted_terms": ", ".join(
                fitted_terms[:5]
            ),
            "ordered_terms_agree":
                stored_terms == fitted_terms,
            "term_overlap": len(
                set(stored_terms) & set(fitted_terms)
            ),
            "terms_compared": len(stored_terms),
        })

    comparison_h22 = pd.DataFrame(
        comparison_records_h22
    )

    display(comparison_h22)

    print(
        "\nExactly matching topic vocabularies:",
        int(comparison_h22["ordered_terms_agree"].sum()),
        "/",
        len(comparison_h22)
    )

else:
    print(
        "Direct comparison pending: selected model "
        "or compatible feature matrix unavailable."
    )

# ------------------------------------------------------------
# 5. Final diagnostic summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL LINEAGE DIAGNOSTICS")
print("=" * 65)

print("Original source inspection: COMPLETED")
print("Selected model structure inspection: COMPLETED")
print("K=7 bank model recovery: COMPLETED")
print("Inventory origin: PENDING SCIENTIFIC REVIEW")
print("Original models refitted: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.22: DIAGNOSTIC COMPLETED")



### Step 12.27H.23 — Reference-Aligned Topic Inventory Reconstruction

**Objective:** Reconstruct the complete seven-topic vocabulary inventory directly from the fitted reference NMF model used in the statistical analyses.

This step will:

1. Verify the provenance and dimensions of the original topic inventory.
2. Reproduce its stored vocabulary using the original model matrices.
3. Extract topic-specific vocabulary from the actual reference model.
4. Identify differences between the two modeling pipelines.
5. Prepare a corrected reference-aligned topic inventory for subsequent human semantic validation.

**Scientific safeguard:** Topic identifiers from different NMF models are not assumed to have equivalent meanings. Statistical estimates remain unchanged.

No files are saved in this computational cell.


In [ ]:

# ============================================================
# STEP 12.27H.23
# REFERENCE-ALIGNED TOPIC INVENTORY RECONSTRUCTION
# ============================================================

required_objects = [
    "selected_topic_models",
    "topic_inventory_df",
    "topic_features",
    "df_nlp",
    "model",
    "focal_features",
]

missing_objects = [
    name for name in required_objects
    if name not in globals()
]

assert not missing_objects, (
    f"Missing required objects: {missing_objects}"
)

print("=" * 65)
print("STEP 12.27H.23 — REFERENCE TOPIC INVENTORY")
print("=" * 65)


# ------------------------------------------------------------
# 1. Verify the original inventory model
# ------------------------------------------------------------

original_entry = selected_topic_models[7]

assert isinstance(original_entry, dict)
assert all(
    key in original_entry
    for key in ["seed", "W", "H"]
)

original_seed = int(original_entry["seed"])
original_W = np.asarray(original_entry["W"])
original_H = np.asarray(original_entry["H"])
original_features = np.asarray(topic_features)

assert original_seed == 73
assert original_W.shape[1] == original_H.shape[0] == 7
assert original_H.shape[1] == len(original_features)
assert original_W.shape[0] == len(df_nlp)

print("\nORIGINAL INVENTORY SOURCE")
print("-" * 65)

print("Random seed:", original_seed)
print("Document count:", original_W.shape[0])
print("Feature count:", original_H.shape[1])
print("Component shape:", original_H.shape)

print("\nOriginal source columns:")
print(list(df_nlp.columns))


# ------------------------------------------------------------
# 2. Reproduce the stored K=7 inventory
# ------------------------------------------------------------

original_inventory_23 = topic_inventory_df.loc[
    (topic_inventory_df["n_topics"] == 7)
    & (topic_inventory_df["reference_seed"] == 73)
].sort_values("topic_id").copy()

assert len(original_inventory_23) == 7
assert original_inventory_23["topic_id"].is_unique

inventory_checks = []

for topic_id in range(1, 8):

    weights = original_H[topic_id - 1]

    top_indices = np.argsort(weights)[::-1][:12]

    recovered_terms = [
        str(term).strip()
        for term in original_features[top_indices]
    ]

    stored_row = original_inventory_23.loc[
        original_inventory_23["topic_id"] == topic_id
    ].iloc[0]

    stored_terms = [
        term.strip()
        for term in str(stored_row["top_terms"]).split(",")
    ]

    inventory_checks.append({
        "topic_id": topic_id,
        "exact_top12_match": recovered_terms == stored_terms,
        "recovered_top_terms": ", ".join(recovered_terms),
    })

inventory_verification_23 = pd.DataFrame(inventory_checks)

print("\n" + "=" * 65)
print("ORIGINAL INVENTORY REPRODUCIBILITY")
print("=" * 65)

display(inventory_verification_23)

original_inventory_reproduced_23 = bool(
    inventory_verification_23["exact_top12_match"].all()
)

print(
    "\nOriginal inventory exactly reproduced:",
    original_inventory_reproduced_23
)


# ------------------------------------------------------------
# 3. Verify the actual reference model
# ------------------------------------------------------------

reference_H_23 = np.asarray(model.components_)
reference_features_23 = np.asarray(focal_features)

assert reference_H_23.ndim == 2
assert reference_H_23.shape[0] == 7
assert reference_H_23.shape[1] == len(reference_features_23)

reference_seed_23 = int(model.random_state)

assert reference_seed_23 == 11

print("\n" + "=" * 65)
print("ACTUAL REFERENCE MODEL")
print("=" * 65)

print("Random seed:", reference_seed_23)
print("Number of topics:", reference_H_23.shape[0])
print("Number of features:", reference_H_23.shape[1])


# ------------------------------------------------------------
# 4. Reconstruct the correct reference vocabulary
# ------------------------------------------------------------

reference_topic_records_23 = []

for topic_id in range(1, 8):

    weights = reference_H_23[topic_id - 1]

    ranked_indices = np.argsort(weights)[::-1][:20]

    ranked_terms = reference_features_23[ranked_indices]

    reference_topic_records_23.append({
        "topic_id": topic_id,
        "reference_seed": reference_seed_23,
        "top_12_terms": ", ".join(
            str(term) for term in ranked_terms[:12]
        ),
        "top_20_terms": ", ".join(
            str(term) for term in ranked_terms
        ),
        "top_weight": float(weights[ranked_indices[0]]),
        "semantic_label_status": "PENDING HUMAN VALIDATION",
    })

reference_topic_inventory_corrected_23 = pd.DataFrame(
    reference_topic_records_23
)

assert len(reference_topic_inventory_corrected_23) == 7

print("\n" + "=" * 65)
print("CORRECTED REFERENCE TOPIC INVENTORY")
print("=" * 65)

for row in reference_topic_inventory_corrected_23.itertuples(
    index=False
):

    print(f"\nTOPIC {row.topic_id:02d}")
    print("-" * 65)
    print("Top 20 terms:")
    print(row.top_20_terms)
    print("Semantic status:", row.semantic_label_status)


# ------------------------------------------------------------
# 5. Summarize pipeline differences
# ------------------------------------------------------------

model_lineage_comparison_23 = pd.DataFrame([
    {
        "pipeline": "Original topic inventory",
        "random_seed": original_seed,
        "n_topics": original_H.shape[0],
        "n_features": original_H.shape[1],
        "n_matrix_rows": original_W.shape[0],
    },
    {
        "pipeline": "Statistical reference NMF",
        "random_seed": reference_seed_23,
        "n_topics": reference_H_23.shape[0],
        "n_features": reference_H_23.shape[1],
        "n_matrix_rows": np.nan,
    },
])

print("\n" + "=" * 65)
print("MODEL LINEAGE COMPARISON")
print("=" * 65)

display(model_lineage_comparison_23)

print(
    "\nSame feature-space dimension:",
    original_H.shape[1] == reference_H_23.shape[1]
)

print(
    "Same random seed:",
    original_seed == reference_seed_23
)


# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print(
    "Original inventory reproduction:",
    "PASSED" if original_inventory_reproduced_23
    else "REQUIRES INVESTIGATION"
)

print("Reference model identity: VERIFIED")
print("Reference vocabulary extraction: COMPLETED")
print("Corrected reference inventory: PREPARED")
print("Semantic interpretation: PENDING HUMAN REVIEW")
print("Original statistical results modified: NO")
print("Original models refitted: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.23: COMPUTATION COMPLETED")



### Step 12.27H.24 — Reference-Aligned Semantic Label Audit

**Objective:** Evaluate the semantic consistency of all seven reference NMF topics and identify publication labels requiring revision.

This step will:

1. Verify the corrected reference topic inventory.
2. Compare previously used labels with reference-derived terminology.
3. Examine lexical evidence supporting each provisional label.
4. Identify potentially heterogeneous or procedural topics.
5. Audit the topic identifiers used in the primary publication tables.
6. Prepare a provisional semantic labeling framework for final validation.

**Important:** Vocabulary-based evidence supports preliminary labeling but does not establish complete semantic coherence or cross-period stability.

No statistical results are modified. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.24
# REFERENCE-ALIGNED SEMANTIC LABEL AUDIT
# ============================================================

assert "reference_topic_inventory_corrected_23" in globals()
assert "paper_results_all18" in globals()
assert "paper_results_main5" in globals()

reference_inventory_24 = (
    reference_topic_inventory_corrected_23.copy()
)

assert len(reference_inventory_24) == 7
assert reference_inventory_24["topic_id"].is_unique


# ------------------------------------------------------------
# 1. Define provisional labels and diagnostic vocabulary
# ------------------------------------------------------------

topic_label_config_24 = {
    1: {
        "previous": "Economic development and cooperation",
        "proposed": "Economic development and global cooperation",
        "anchors": [
            "developing countries", "trade",
            "economy", "sustainable development",
            "globalization",
        ],
        "risk": "Low",
    },
    2: {
        "previous": "China and peaceful development",
        "proposed": "China and peaceful development",
        "anchors": [
            "china", "chinese", "peace development",
            "taiwan", "cooperation",
        ],
        "risk": "Country-specific",
    },
    3: {
        "previous": "Kampuchea and regional conflicts",
        "proposed": "Regional conflicts and diplomatic settlement",
        "anchors": [
            "kampuchea", "settlement", "aggression",
            "middle east", "palestinian",
        ],
        "risk": "High semantic heterogeneity",
    },
    4: {
        "previous": "Iran and Islamic Republic",
        "proposed": "Iran and regional security",
        "anchors": [
            "iran", "iraq", "region",
            "terrorism", "sanctions",
        ],
        "risk": "Country-specific",
    },
    5: {
        "previous": "Human rights and freedom discourse",
        "proposed": "Human rights and freedom discourse",
        "anchors": [
            "human rights", "freedom",
            "united states", "democracy", "justice",
        ],
        "risk": "Country-rhetoric overlap",
    },
    6: {
        "previous": "Not included in primary contrasts",
        "proposed": "UN General Assembly procedural language",
        "anchors": [
            "session", "general assembly",
            "presidency", "congratulate", "election",
        ],
        "risk": "Procedural / boilerplate",
    },
    7: {
        "previous": "Nuclear arms and disarmament",
        "proposed": "Nuclear arms and disarmament",
        "anchors": [
            "nuclear", "nuclear weapons",
            "disarmament", "proliferation", "treaty",
        ],
        "risk": "Low",
    },
}


# ------------------------------------------------------------
# 2. Construct the semantic label audit
# ------------------------------------------------------------

semantic_label_records_24 = []

for row in reference_inventory_24.itertuples(index=False):

    topic_id = int(row.topic_id)
    config = topic_label_config_24[topic_id]

    top_terms = [
        term.strip().lower()
        for term in row.top_20_terms.split(",")
    ]

    matched_anchors = [
        anchor
        for anchor in config["anchors"]
        if anchor in top_terms
    ]

    semantic_label_records_24.append({
        "topic_id": topic_id,
        "previous_label": config["previous"],
        "proposed_label": config["proposed"],
        "matched_anchors": ", ".join(matched_anchors),
        "n_matched_anchors": len(matched_anchors),
        "semantic_risk": config["risk"],
        "label_changed": (
            config["previous"].lower()
            != config["proposed"].lower()
        ),
        "human_validation": "PENDING",
    })

reference_semantic_label_review_df_24 = pd.DataFrame(
    semantic_label_records_24
)

assert len(reference_semantic_label_review_df_24) == 7

print("=" * 65)
print("REFERENCE-ALIGNED SEMANTIC LABEL AUDIT")
print("=" * 65)

display(reference_semantic_label_review_df_24)


# ------------------------------------------------------------
# 3. Verify publication topic identifiers
# ------------------------------------------------------------

all18_topics_24 = set(
    paper_results_all18["topic_id"].astype(int)
)

main5_topics_24 = set(
    paper_results_main5["topic_id"].astype(int)
)

reference_topics_24 = set(
    reference_inventory_24["topic_id"].astype(int)
)

assert all18_topics_24.issubset(reference_topics_24)
assert main5_topics_24.issubset(all18_topics_24)

print("\n" + "=" * 65)
print("PUBLICATION TOPIC CONSISTENCY")
print("=" * 65)

print("Reference topics:", sorted(reference_topics_24))
print("Primary test topics:", sorted(all18_topics_24))
print("Main finding topics:", sorted(main5_topics_24))
print("Excluded reference topics:",
      sorted(reference_topics_24 - all18_topics_24))


# ------------------------------------------------------------
# 4. Prepare publication labeling review
# ------------------------------------------------------------

publication_topic_review_df_24 = (
    reference_semantic_label_review_df_24.loc[
        reference_semantic_label_review_df_24[
            "topic_id"
        ].isin(all18_topics_24)
    ].copy()
)

publication_topic_review_df_24["in_main_table"] = (
    publication_topic_review_df_24["topic_id"].isin(
        main5_topics_24
    )
)

publication_topic_review_df_24["review_priority"] = (
    publication_topic_review_df_24.apply(
        lambda row: (
            "HIGH"
            if row["topic_id"] == 3
            else "MODERATE"
            if row["semantic_risk"] != "Low"
            else "STANDARD"
        ),
        axis=1,
    )
)

print("\n" + "=" * 65)
print("PUBLICATION LABEL REVIEW")
print("=" * 65)

display(
    publication_topic_review_df_24[
        [
            "topic_id",
            "previous_label",
            "proposed_label",
            "semantic_risk",
            "in_main_table",
            "review_priority",
        ]
    ]
)


# ------------------------------------------------------------
# 5. Final scientific validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Reference topics evaluated:", len(reference_inventory_24))
print("Primary topics evaluated:", len(all18_topics_24))
print("Main-table topics evaluated:", len(main5_topics_24))

print(
    "Changed provisional labels:",
    int(reference_semantic_label_review_df_24[
        "label_changed"
    ].sum())
)

print("Reference inventory linkage: PASSED")
print("Publication topic identifiers: VERIFIED")
print("Vocabulary-based screening: COMPLETED")
print("Final human semantic validation: PENDING")
print("Existing statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.24: COMPUTATION COMPLETED")



### Step 12.27H.25 — Publication Label Impact and Integrity Audit

**Objective:** Identify which publication results are affected by the corrected reference-model topic labels, while preserving the original statistical evidence.

This step will:

1. Verify the semantic label mapping against the reference NMF model.
2. Identify the affected rows in Main Table 1 and Supplementary Table S2.
3. Prepare provisional label-corrected publication previews.
4. Highlight Topic 03 and its implications for historical interpretation.
5. Verify that all original statistical results remain unchanged.

**Scientific safeguards:**
- Proposed labels remain provisional until semantic adjudication.
- Topic identifiers, effects, confidence intervals, and adjusted p-values must remain unchanged.
- The original publication DataFrames will not be overwritten.
- No tables or figures will be saved in this computation cell.

**Expected output:** Publication label impact summary, provisional publication previews, and numerical integrity validation.


In [ ]:

# ============================================================
# STEP 12.27H.25
# PUBLICATION LABEL IMPACT AND INTEGRITY AUDIT
# ============================================================

required_objects_25 = [
    "reference_semantic_label_review_df_24",
    "paper_results_all18",
    "paper_results_main5",
]

for name in required_objects_25:
    assert name in globals(), f"Missing required object: {name}"

# ------------------------------------------------------------
# 1. Preserve and validate source tables
# ------------------------------------------------------------

original_main5_25 = paper_results_main5.copy(deep=True)
original_all18_25 = paper_results_all18.copy(deep=True)

label_mapping_25 = (
    reference_semantic_label_review_df_24.copy(deep=True)
    .set_index("topic_id", verify_integrity=True)
)

assert len(original_main5_25) == 5
assert len(original_all18_25) == 18
assert len(label_mapping_25) == 7

for source in [original_main5_25, original_all18_25]:
    assert {"pair_label", "topic_id"}.issubset(source.columns)
    assert source["topic_id"].isin(label_mapping_25.index).all()

print("=" * 65)
print("STEP 12.27H.25 — PUBLICATION LABEL IMPACT AUDIT")
print("=" * 65)

print("Reference topics:", len(label_mapping_25))
print("Main-table contrasts:", len(original_main5_25))
print("Supplementary contrasts:", len(original_all18_25))


# ------------------------------------------------------------
# 2. Build publication previews without modifying sources
# ------------------------------------------------------------

def build_semantic_preview_25(source_df):

    result = source_df.copy(deep=True)

    original_columns = list(source_df.columns)

    result["previous_semantic_label"] = (
        result["topic_id"].map(
            label_mapping_25["previous_label"]
        )
    )

    result["proposed_semantic_label"] = (
        result["topic_id"].map(
            label_mapping_25["proposed_label"]
        )
    )

    result["semantic_risk"] = (
        result["topic_id"].map(
            label_mapping_25["semantic_risk"]
        )
    )

    result["label_change_proposed"] = (
        result["topic_id"].map(
            label_mapping_25["label_changed"]
        )
    )

    result["semantic_validation_status"] = (
        result["topic_id"].map(
            label_mapping_25["human_validation"]
        )
    )

    assert result["proposed_semantic_label"].notna().all()

    pd.testing.assert_frame_equal(
        source_df,
        result[original_columns],
        check_dtype=True,
        check_exact=True,
    )

    return result


paper_main5_semantic_preview_25 = (
    build_semantic_preview_25(original_main5_25)
)

paper_all18_semantic_preview_25 = (
    build_semantic_preview_25(original_all18_25)
)


# ------------------------------------------------------------
# 3. Identify publication rows affected by label revisions
# ------------------------------------------------------------

impact_records_25 = []

for topic_id in sorted(label_mapping_25.index):

    main_count = int(
        original_main5_25["topic_id"].eq(topic_id).sum()
    )

    supplementary_count = int(
        original_all18_25["topic_id"].eq(topic_id).sum()
    )

    impact_records_25.append({
        "topic_id": topic_id,
        "proposed_label":
            label_mapping_25.loc[topic_id, "proposed_label"],
        "main_table_rows": main_count,
        "supplementary_rows": supplementary_count,
        "label_change_proposed": bool(
            label_mapping_25.loc[topic_id, "label_changed"]
        ),
        "semantic_risk":
            label_mapping_25.loc[topic_id, "semantic_risk"],
    })

publication_label_impact_df_25 = pd.DataFrame(
    impact_records_25
)

print("\n" + "=" * 65)
print("PUBLICATION LABEL IMPACT SUMMARY")
print("=" * 65)

display(publication_label_impact_df_25)


# ------------------------------------------------------------
# 4. Inspect the main publication table
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("MAIN TABLE 1 — PROVISIONAL SEMANTIC PREVIEW")
print("=" * 65)

main_display_columns_25 = [
    "pair_label",
    "topic_id",
    "effect_pp",
    "holm_adjusted_p",
    "previous_semantic_label",
    "proposed_semantic_label",
    "semantic_risk",
    "semantic_validation_status",
]

main_display_columns_25 = [
    col for col in main_display_columns_25
    if col in paper_main5_semantic_preview_25.columns
]

display(
    paper_main5_semantic_preview_25[
        main_display_columns_25
    ]
)


# ------------------------------------------------------------
# 5. Identify every Topic 03 contrast
# ------------------------------------------------------------

topic03_publication_audit_df_25 = (
    paper_all18_semantic_preview_25.loc[
        paper_all18_semantic_preview_25["topic_id"].eq(3)
    ].copy()
)

assert len(topic03_publication_audit_df_25) == 3

topic03_columns_25 = [
    "pair_label",
    "topic_id",
    "effect_pp",
    "holm_adjusted_p",
    "previous_semantic_label",
    "proposed_semantic_label",
    "semantic_risk",
]

topic03_columns_25 = [
    col for col in topic03_columns_25
    if col in topic03_publication_audit_df_25.columns
]

print("\n" + "=" * 65)
print("TOPIC 03 — PUBLICATION INTERPRETATION AUDIT")
print("=" * 65)

display(
    topic03_publication_audit_df_25[
        topic03_columns_25
    ]
)


# ------------------------------------------------------------
# 6. Numerical and source integrity checks
# ------------------------------------------------------------

pd.testing.assert_frame_equal(
    paper_results_main5,
    original_main5_25,
    check_exact=True,
)

pd.testing.assert_frame_equal(
    paper_results_all18,
    original_all18_25,
    check_exact=True,
)

assert len(paper_main5_semantic_preview_25) == 5
assert len(paper_all18_semantic_preview_25) == 18

assert (
    paper_main5_semantic_preview_25[
        "semantic_validation_status"
    ].eq("PENDING").all()
)

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Topic-to-label linkage: PASSED")
print("Main table integrity: PASSED")
print("Supplementary table integrity: PASSED")

print(
    "Main-table rows with proposed label changes:",
    int(
        paper_main5_semantic_preview_25[
            "label_change_proposed"
        ].sum()
    )
)

print(
    "Supplementary rows with proposed label changes:",
    int(
        paper_all18_semantic_preview_25[
            "label_change_proposed"
        ].sum()
    )
)

print("Original numerical results modified: NO")
print("Final human semantic validation: PENDING")
print("Publication label replacement: NOT APPLIED")
print("Files saved: NONE")

print("\nSTEP 12.27H.25: COMPUTATION COMPLETED")



### Step 12.27H.26 — Final Topic 03 Semantic Evidence Review

**Objective:** Assess whether the proposed label "Regional conflicts and diplomatic settlement" adequately represents the reference NMF Topic 03 across countries and historical periods.

**Procedure:**
1. Verify the existing 18-record stratified semantic review inventory.
2. Recover original segment texts using exact speech and segment identifiers.
3. Examine historical and geographic coverage.
4. Prepare detailed evidence for three previously identified priority cases.
5. Preserve the distinction between lexical screening and human semantic adjudication.

**Scientific safeguards:**
- The 18 selected segments are diagnostic samples, not a representative prevalence estimate.
- Topic 03 cannot be characterized exclusively by Kampuchea references.
- No semantic verdict will be assigned automatically.
- Original fitted models and statistical results remain unchanged.
- No files will be saved in this computation cell.

**Expected outcome:** A source-linked semantic evidence packet for final qualitative adjudication.


In [ ]:

# ============================================================
# STEP 12.27H.26
# FINAL TOPIC 03 SEMANTIC EVIDENCE REVIEW
# ============================================================

assert "semantic_master_review_df" in globals()

review_26 = semantic_master_review_df.copy(deep=True)

required_26 = [
    "review_id",
    "country_code",
    "year",
    "period",
    "speech_id",
    "segment_id",
    "topic03_share_recovered",
]

assert set(required_26).issubset(review_26.columns)
assert len(review_26) == 18
assert review_26["review_id"].is_unique

assert (
    review_26.groupby(["country_code", "period"])
    .size()
    .eq(2)
    .all()
)

print("=" * 65)
print("STEP 12.27H.26 — SEMANTIC EVIDENCE REVIEW")
print("=" * 65)

print("Review records:", len(review_26))
print("Countries:", review_26["country_code"].nunique())
print("Historical periods:", review_26["period"].nunique())


# ------------------------------------------------------------
# 1. Discover complete source-text candidates
# ------------------------------------------------------------

text_columns_26 = [
    "segment_text",
    "original_segment_text",
    "source_text",
    "cached_segment_text",
    "complete_text",
    "text",
]

source_candidates_26 = []

reference_keys_26 = review_26[
    ["review_id", "speech_id", "segment_id"]
].copy()

for key in ["speech_id", "segment_id"]:
    reference_keys_26[key] = pd.to_numeric(
        reference_keys_26[key], errors="coerce"
    )

for object_name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    if not {"speech_id", "segment_id"}.issubset(obj.columns):
        continue

    available_text_columns = [
        col for col in text_columns_26
        if col in obj.columns
    ]

    for text_col in available_text_columns:

        candidate = obj[
            ["speech_id", "segment_id", text_col]
        ].copy()

        for key in ["speech_id", "segment_id"]:
            candidate[key] = pd.to_numeric(
                candidate[key], errors="coerce"
            )

        candidate = candidate.dropna(
            subset=["speech_id", "segment_id"]
        )

        if candidate.duplicated(
            ["speech_id", "segment_id"]
        ).any():
            continue

        try:
            matched = reference_keys_26.merge(
                candidate,
                on=["speech_id", "segment_id"],
                how="left",
                validate="one_to_one",
            )
        except (ValueError, KeyError):
            continue

        texts = matched[text_col].fillna("").astype(str)

        lengths = texts.str.len()

        coverage = int((lengths >= 150).sum())

        if coverage == 18:
            source_candidates_26.append({
                "object_name": object_name,
                "text_column": text_col,
                "median_characters": float(lengths.median()),
                "texts": texts.tolist(),
            })


# ------------------------------------------------------------
# 2. Select a complete linked source
# ------------------------------------------------------------

if not source_candidates_26:
    raise RuntimeError(
        "No complete 18-record segment-text source was found. "
        "Inspect the existing source DataFrame names and columns "
        "before proceeding. Do not substitute speech-level text."
    )

source_candidates_26.sort(
    key=lambda item: (
        item["text_column"] == "segment_text",
        item["median_characters"],
    ),
    reverse=True,
)

selected_source_26 = source_candidates_26[0]

review_26["verified_source_text"] = (
    selected_source_26["texts"]
)

assert review_26["verified_source_text"].notna().all()

print("\nSOURCE TEXT RECOVERY")
print("Source object:", selected_source_26["object_name"])
print("Text column:", selected_source_26["text_column"])
print("Recovered segments:", len(review_26))


# ------------------------------------------------------------
# 3. Prepare the historical inspection inventory
# ------------------------------------------------------------

semantic_evidence_packet_26 = review_26[
    required_26 + ["verified_source_text"]
].copy()

semantic_evidence_packet_26["source_characters"] = (
    semantic_evidence_packet_26[
        "verified_source_text"
    ].str.len()
)

semantic_evidence_packet_26["source_preview"] = (
    semantic_evidence_packet_26[
        "verified_source_text"
    ]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 160)
)

semantic_evidence_packet_26 = (
    semantic_evidence_packet_26.sort_values(
        ["country_code", "year", "review_id"]
    ).reset_index(drop=True)
)

print("\n" + "=" * 65)
print("HISTORICAL SEMANTIC EVIDENCE INVENTORY")
print("=" * 65)

display(
    semantic_evidence_packet_26[
        [
            "review_id",
            "country_code",
            "year",
            "period",
            "topic03_share_recovered",
            "source_characters",
            "source_preview",
        ]
    ]
)


# ------------------------------------------------------------
# 4. Select previously identified priority cases
# ------------------------------------------------------------

priority_ids_26 = [
    "SV-016",   # China, 1999
    "SV-X002",  # Iran, 1981
    "SV-X012",  # United States, 2025
]

priority_evidence_26 = (
    semantic_evidence_packet_26.loc[
        semantic_evidence_packet_26[
            "review_id"
        ].isin(priority_ids_26)
    ].copy()
)

assert len(priority_evidence_26) == 3

print("\n" + "=" * 65)
print("PRIORITY SEMANTIC ADJUDICATION")
print("=" * 65)

for review_id in priority_ids_26:

    record = priority_evidence_26.loc[
        priority_evidence_26["review_id"].eq(review_id)
    ].iloc[0]

    print("\n" + "#" * 65)
    print("REVIEW ID:", review_id)
    print("Country:", record["country_code"])
    print("Year:", record["year"])
    print(
        "Topic 03 share:",
        round(record["topic03_share_recovered"], 6)
    )
    print("-" * 65)
    print("ORIGINAL SOURCE SEGMENT")
    print("-" * 65)
    print(record["verified_source_text"])


# ------------------------------------------------------------
# 5. Initialize the qualitative adjudication register
# ------------------------------------------------------------

semantic_adjudication_register_26 = (
    semantic_evidence_packet_26[
        [
            "review_id",
            "country_code",
            "year",
            "period",
            "topic03_share_recovered",
        ]
    ].copy()
)

semantic_adjudication_register_26[
    "primary_substantive_issue"
] = pd.NA

semantic_adjudication_register_26[
    "conflict_or_diplomacy_fit"
] = pd.NA

semantic_adjudication_register_26[
    "kampuchea_specific_fit"
] = pd.NA

semantic_adjudication_register_26[
    "semantic_heterogeneity"
] = pd.NA

semantic_adjudication_register_26[
    "supporting_evidence"
] = pd.NA

semantic_adjudication_register_26[
    "human_review_status"
] = "PENDING"

print("\n" + "=" * 65)
print("FINAL SEMANTIC REVIEW STATUS")
print("=" * 65)

print("Records available:", len(semantic_adjudication_register_26))
print("Priority source texts reviewed for presentation:", 3)
print("Human-coded records:", 0)
print("Topic 03 final label: PENDING ADJUDICATION")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.26: EVIDENCE PACKET PREPARED")



### Step 12.27H.27 — Complete Cross-Country Semantic Adjudication

**Objective:** Complete a transparent qualitative assessment of Topic 03 using the original source segments from China, Iran, and the United States.

**Assessment dimensions:**
1. Primary geographic and substantive focus.
2. Representation of international conflict or disputes.
3. Evidence of sovereignty, intervention, or self-determination.
4. Evidence of diplomatic settlement or peace processes.
5. Fit with the proposed broad topic label.
6. Evidence of semantic heterogeneity.

Three previously inspected priority cases are excluded from the remaining-review queue, but retained in the complete 18-record inventory.

Review the original texts country by country. Do not infer semantic validity from NMF weights or keyword frequencies alone.

This cell performs no automated qualitative adjudication, model modification, statistical recalculation, or file saving.


In [ ]:

# ============================================================
# STEP 12.27H.27
# COMPLETE CROSS-COUNTRY SEMANTIC ADJUDICATION
# ============================================================

assert "semantic_evidence_packet_26" in globals()

REVIEW_COUNTRY_27 = "CHN"

assert REVIEW_COUNTRY_27 in {"CHN", "IRN", "USA"}

already_inspected_27 = {
    "SV-016",
    "SV-X002",
    "SV-X012",
}

full_inventory_27 = semantic_evidence_packet_26.copy()

assert len(full_inventory_27) == 18
assert full_inventory_27["review_id"].is_unique

assert full_inventory_27.groupby(
    "country_code"
).size().eq(6).all()

remaining_inventory_27 = full_inventory_27.loc[
    ~full_inventory_27["review_id"].isin(
        already_inspected_27
    )
].copy()

assert len(remaining_inventory_27) == 15

country_review_27 = (
    remaining_inventory_27.loc[
        remaining_inventory_27["country_code"].eq(
            REVIEW_COUNTRY_27
        )
    ]
    .sort_values(["year", "review_id"])
    .reset_index(drop=True)
)

assert len(country_review_27) == 5

print("=" * 65)
print("STEP 12.27H.27 — SEMANTIC ADJUDICATION")
print("=" * 65)

print("Country:", REVIEW_COUNTRY_27)
print("Total evidence records:", len(full_inventory_27))
print("Previously inspected cases:", 3)
print("Remaining review records:", len(remaining_inventory_27))
print("Current country records:", len(country_review_27))

print("\nSOURCE INVENTORY")

display(
    country_review_27[
        [
            "review_id",
            "year",
            "period",
            "topic03_share_recovered",
            "source_characters",
        ]
    ].round(6)
)

# ------------------------------------------------------------
# Present complete original source texts
# ------------------------------------------------------------

for record in country_review_27.itertuples(index=False):

    print("\n" + "=" * 65)
    print("REVIEW ID:", record.review_id)
    print("COUNTRY:", record.country_code)
    print("YEAR:", record.year)
    print("PERIOD:", record.period)

    print(
        "TOPIC 03 SHARE:",
        round(record.topic03_share_recovered, 6)
    )

    print("-" * 65)
    print("COMPLETE SOURCE SEGMENT")
    print("-" * 65)

    source_text = str(
        record.verified_source_text
    ).strip()

    assert len(source_text) >= 150

    # Display wrapping only; original text remains unchanged.
    print(
        pd.Series([source_text])
        .str.wrap(width=105)
        .iloc[0]
    )

    print("\nQUALITATIVE REVIEW DIMENSIONS")
    print("1. Geographic and substantive focus")
    print("2. International conflict or disputes")
    print("3. Sovereignty and non-intervention")
    print("4. Diplomatic or peace processes")
    print("5. Broad topic-label fit")
    print("6. Semantic heterogeneity")

print("\n" + "=" * 65)
print("REVIEW COMPLETENESS")
print("=" * 65)

print("Source records available:", 18)
print("Priority cases previously inspected:", 3)
print("Additional records presented:", len(country_review_27))
print("Human-coded decisions: PENDING")
print("Final Topic 03 label: PENDING")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.27: INSPECTION READY")



### Step 12.27H.28 — Iran Historical Semantic Validation

**Objective:** Independently examine the semantic consistency of Reference NMF Topic 03 across Iranian General Debate speeches from 1972 to 2025.

This step will:

1. Recover all six previously selected Iranian source segments.
2. Verify speech identifiers, segment identifiers, and reference topic weights.
3. Examine geographic, conflict-related, sovereignty, and diplomatic vocabulary.
4. Retrieve complete original segment texts for qualitative interpretation.
5. Prepare a traceable human-review template without assigning automatic semantic judgments.

**Scientific safeguards:**
- Preserve the original stratified sampling design.
- Use the verified reference NMF topic assignments.
- Distinguish lexical evidence from qualitative semantic validation.
- Do not modify statistical models, topic assignments, or publication estimates.
- Perform all file exports in a separate cell.

**Expected outputs:** Iranian source evidence, thematic diagnostics, and an independent qualitative coding template.


In [ ]:

# ============================================================
# STEP 12.27H.28
# IRAN HISTORICAL SEMANTIC VALIDATION
# ============================================================

print("=" * 65)
print("STEP 12.27H.28 — IRAN HISTORICAL SEMANTIC VALIDATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Validate existing research objects
# ------------------------------------------------------------

assert "semantic_master_review_df" in globals()
assert "segments" in globals()

required_review = {
    "review_id", "country_code", "year", "period",
    "speech_id", "segment_id", "topic03_share_recovered"
}

assert required_review.issubset(
    semantic_master_review_df.columns
)

iran_review_28 = (
    semantic_master_review_df.loc[
        semantic_master_review_df["country_code"].eq("IRN")
    ]
    .copy()
    .sort_values(["year", "review_id"])
    .reset_index(drop=True)
)

assert len(iran_review_28) == 6
assert iran_review_28["review_id"].is_unique
assert iran_review_28["year"].nunique() == 6
assert iran_review_28["period"].nunique() == 3

# ------------------------------------------------------------
# 2. Recover complete source segments
# ------------------------------------------------------------

source_28 = segments.copy()

keys_28 = ["speech_id", "segment_id"]

if not set(keys_28).issubset(source_28.columns):
    source_28 = source_28.reset_index()

required_source = set(keys_28 + ["segment_text"])
assert required_source.issubset(source_28.columns)

for key in keys_28:
    iran_review_28[key] = pd.to_numeric(
        iran_review_28[key], errors="raise"
    ).astype(int)

    source_28[key] = pd.to_numeric(
        source_28[key], errors="raise"
    ).astype(int)

assert not source_28.duplicated(keys_28).any()

iran_source_28 = iran_review_28.merge(
    source_28[keys_28 + ["segment_text"]],
    on=keys_28,
    how="left",
    validate="one_to_one",
    indicator=True
)

assert iran_source_28["_merge"].eq("both").all()

iran_source_28 = iran_source_28.drop(columns="_merge")
iran_source_28["segment_text"] = (
    iran_source_28["segment_text"].astype("string")
)

assert iran_source_28["segment_text"].notna().all()
assert iran_source_28["segment_text"].str.strip().ne("").all()

iran_source_28["source_characters"] = (
    iran_source_28["segment_text"].str.len()
)

print("\nSOURCE INVENTORY")

display(
    iran_source_28[
        [
            "review_id", "year", "period",
            "speech_id", "segment_id",
            "topic03_share_recovered",
            "source_characters"
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 3. Lexical evidence diagnostics
# ------------------------------------------------------------

patterns_28 = {
    "Kampuchea / Cambodia":
        r"\b(?:kampuchea|kampuchean|cambodia|cambodian)\b",
    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*)\b",
    "South Africa / Apartheid":
        r"\b(?:south africa|apartheid|namibia)\b",
    "Bosnia / Kosovo":
        r"\b(?:bosnia|kosovo|yugoslavia)\b",
    "Afghanistan":
        r"\b(?:afghanistan|afghan)\b",
    "Iraq / Syria":
        r"\b(?:iraq\w*|syria\w*)\b",
    "Diplomatic Settlement":
        r"\b(?:peace|settlement|negotiat\w*|dialogue|conference)\b",
    "Sovereignty / Intervention":
        r"\b(?:sovereignty|intervention|non.interference)\b"
}

iran_lexical_28 = iran_source_28[
    ["review_id", "year", "period", "topic03_share_recovered"]
].copy()

for category, pattern in patterns_28.items():
    iran_lexical_28[category] = (
        iran_source_28["segment_text"]
        .str.lower()
        .str.count(pattern)
        .astype(int)
    )

print("\nHISTORICAL THEMATIC EVIDENCE")
display(iran_lexical_28.round(5))

print(
    "\nNOTE: Keyword counts are screening indicators, "
    "not human semantic judgments."
)

# ------------------------------------------------------------
# 4. Prepare the independent human coding template
# ------------------------------------------------------------

iran_human_review_28 = iran_source_28[
    [
        "review_id", "country_code", "year",
        "period", "speech_id", "segment_id",
        "topic03_share_recovered"
    ]
].copy()

coding_fields_28 = [
    "primary_geographic_focus",
    "principal_substantive_issue",
    "sovereignty_or_intervention_theme",
    "diplomatic_or_peace_process",
    "kampuchea_specific_label_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "supporting_text",
    "reviewer_notes"
]

for field in coding_fields_28:
    iran_human_review_28[field] = pd.NA

iran_human_review_28["human_review_status"] = "PENDING"

print("\nHUMAN SEMANTIC CODING TEMPLATE")
display(iran_human_review_28)

# ------------------------------------------------------------
# 5. Print complete source evidence
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMPLETE IRANIAN SOURCE SEGMENTS")
print("=" * 65)

for record in iran_source_28.itertuples(index=False):

    print("\n" + "#" * 65)
    print("Review ID:", record.review_id)
    print("Country:", record.country_code)
    print("Year:", record.year)
    print("Period:", record.period)
    print("Speech ID:", record.speech_id)
    print("Segment ID:", record.segment_id)
    print("Topic 03 share:",
          round(record.topic03_share_recovered, 6))
    print("-" * 65)
    print(record.segment_text)

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

assert len(iran_source_28) == 6
assert len(iran_lexical_28) == 6
assert len(iran_human_review_28) == 6

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Iranian records:", len(iran_source_28))
print("Distinct speeches:", iran_source_28["speech_id"].nunique())
print("Historical periods:", iran_source_28["period"].nunique())
print("Complete source retrieval: PASSED")
print("Lexical diagnostics: PASSED")
print("Human coding template: PREPARED")
print("Human semantic adjudication: PENDING")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.28: COMPUTATION COMPLETED")



### Step 12.27H.28A — Iranian Source Integrity and Lexical Specificity Audit

**Objective:** Verify potentially malformed historical text and distinguish geographic references from thematic terminology before finalizing the Iranian semantic evidence.

**Procedures:**
1. Screen all six Iranian source segments for identified textual discontinuities.
2. Compare the 1991 cached segment with available speech-level source records.
3. Distinguish South Africa/Namibia references from general apartheid terminology.
4. Preserve the original statistical estimates and source records.

**Decision rule:** Potential source corruption must be resolved or explicitly documented before the semantic evidence is considered publication-ready.

**Saving:** No files are created in this diagnostic cell.


In [ ]:

# ============================================================
# STEP 12.27H.28A
# IRANIAN SOURCE INTEGRITY AND LEXICAL SPECIFICITY AUDIT
# ============================================================

assert "iran_source_28" in globals()
assert len(iran_source_28) == 6

print("=" * 65)
print("STEP 12.27H.28A — IRAN SOURCE INTEGRITY AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Detect identified textual discontinuities
# ------------------------------------------------------------

iran_integrity_28a = iran_source_28[
    ["review_id", "year", "source_characters"]
].copy()

suspect_fragments_28a = [
    "onourable settlement",
    "Palestine.ression"
]

iran_integrity_28a["identified_discontinuities"] = (
    iran_source_28["segment_text"].apply(
        lambda text: sum(
            fragment.lower() in str(text).lower()
            for fragment in suspect_fragments_28a
        )
    )
)

iran_integrity_28a["screening_status"] = np.where(
    iran_integrity_28a["identified_discontinuities"] > 0,
    "FLAGGED FOR SOURCE VERIFICATION",
    "NO IDENTIFIED DISCONTINUITY"
)

print("\nSOURCE INTEGRITY SCREENING")
display(iran_integrity_28a)

# ------------------------------------------------------------
# 2. Compare 1991 with available speech-level records
# ------------------------------------------------------------

target_28a = iran_source_28.loc[
    iran_source_28["review_id"].eq("SV-X003")
].iloc[0]

print("\n1991 SEGMENT: IDENTIFIED FRAGMENTS")

for fragment in suspect_fragments_28a:
    text = str(target_28a["segment_text"])
    position = text.lower().find(fragment.lower())

    print("\nFragment:", fragment)
    print("Present:", position >= 0)

    if position >= 0:
        print(
            text[max(0, position - 120):
                 min(len(text), position + 200)]
        )

print("\nAVAILABLE ORIGINAL SPEECH COMPARISON")

original_sources_28a = []

for object_name in [
    "df_nlp", "df", "df_clean", "df_prepared", "df_raw"
]:
    obj = globals().get(object_name)

    if not isinstance(obj, pd.DataFrame):
        continue

    if not {"country_code", "year"}.issubset(obj.columns):
        continue

    subset = obj.loc[
        obj["country_code"].astype(str).eq("IRN")
        & pd.to_numeric(
            obj["year"], errors="coerce"
        ).eq(1991)
    ]

    for text_col in [
        "text_raw", "text_clean", "speech_text", "text"
    ]:
        if text_col not in subset.columns:
            continue

        for value in subset[text_col].dropna():
            original_sources_28a.append({
                "object": object_name,
                "text_column": text_col,
                "text": str(value)
            })

print("Speech-level source versions found:",
      len(original_sources_28a))

for record in original_sources_28a[:6]:

    text = record["text"]
    lower = text.lower()

    print("\nSource:", record["object"])
    print("Column:", record["text_column"])
    print("Characters:", len(text))

    for fragment in suspect_fragments_28a:
        print(
            f"  '{fragment}' present:",
            fragment.lower() in lower
        )

    for anchor in ["palestinian question", "intifadah"]:
        position = lower.find(anchor)

        if position >= 0:
            print("\nSource context:")
            print(
                text[max(0, position - 180):
                     min(len(text), position + 300)]
            )
            break

if not original_sources_28a:
    print(
        "Independent speech-level text unavailable "
        "in the inspected memory objects."
    )

# ------------------------------------------------------------
# 3. Disaggregate geographic and thematic evidence
# ------------------------------------------------------------

iran_lexical_refined_28a = iran_source_28[
    ["review_id", "year", "period"]
].copy()

refined_patterns_28a = {
    "South Africa / Namibia":
        r"\b(?:south africa|namibia|namibian)\b",
    "Apartheid terminology":
        r"\bapartheid\b",
    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*)\b",
    "Kampuchea / Cambodia":
        r"\b(?:kampuchea|kampuchean|cambodia|cambodian)\b"
}

for label, pattern in refined_patterns_28a.items():
    iran_lexical_refined_28a[label] = (
        iran_source_28["segment_text"]
        .str.lower()
        .str.count(pattern)
        .astype(int)
    )

print("\nREFINED HISTORICAL LEXICAL EVIDENCE")
display(iran_lexical_refined_28a)

# ------------------------------------------------------------
# 4. Final diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL DIAGNOSTICS")
print("=" * 65)

print("Iranian source records:", len(iran_source_28))
print(
    "Records with identified discontinuities:",
    int(
        iran_integrity_28a[
            "identified_discontinuities"
        ].gt(0).sum()
    )
)

print(
    "Speech-level source versions examined:",
    len(original_sources_28a)
)

print("Original statistical estimates modified: NO")
print("Human semantic adjudication: PENDING")
print("Files saved: NONE")
print("\nSTEP 12.27H.28A: DIAGNOSTIC COMPLETED")



### Step 12.27H.29 — United States Historical Semantic Validation

**Objective:** Evaluate the historical and semantic coherence of reference NMF Topic 03 across the six sampled United States speech segments.

**Analysis:**
1. Verify source-to-review linkage and historical coverage.
2. Examine complete cached segments and their topic contributions.
3. Separate geographic references from diplomatic, military, institutional, and domestic-policy terminology.
4. Identify possible semantic heterogeneity across historical periods.
5. Prepare traceable evidence for final human adjudication.

**Scientific safeguards:**
- Lexical frequency is treated as screening evidence, not semantic classification.
- The six selected segments are not treated as a representative population sample for estimating historical prevalence.
- Original NMF components, topic weights, statistical contrasts, and significance tests remain unchanged.
- Independent original-file verification and final human adjudication remain separate requirements.

**Output:** Diagnostic tables and source evidence only. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.29
# UNITED STATES HISTORICAL SEMANTIC VALIDATION
# ============================================================

assert "semantic_master_review_df" in globals()

required_29 = {
    "review_id", "country_code", "year", "period",
    "speech_id", "segment_id", "topic03_share_recovered"
}

assert required_29.issubset(
    semantic_master_review_df.columns
), "Required review metadata are missing."

usa_review_29 = (
    semantic_master_review_df.loc[
        semantic_master_review_df["country_code"].eq("USA")
    ]
    .copy()
    .sort_values(["year", "review_id"])
    .reset_index(drop=True)
)

assert len(usa_review_29) == 6
assert usa_review_29["review_id"].is_unique

assert (
    usa_review_29.groupby("period")
    .size()
    .eq(2)
    .all()
)

# ------------------------------------------------------------
# 1. Recover source-linked segments
# ------------------------------------------------------------

if (
    "segment_text" in usa_review_29.columns
    and usa_review_29["segment_text"].notna().all()
):
    usa_source_29 = usa_review_29.copy()

else:
    assert "segments" in globals(), (
        "The cached segment source is unavailable."
    )

    segment_source_29 = (
        segments.copy()
        if isinstance(segments, pd.DataFrame)
        else pd.DataFrame(segments)
    )

    required_segments_29 = {
        "speech_id", "segment_id", "segment_text"
    }

    assert required_segments_29.issubset(
        segment_source_29.columns
    )

    assert not segment_source_29.duplicated(
        ["speech_id", "segment_id"]
    ).any()

    usa_source_29 = (
        usa_review_29.drop(
            columns=["segment_text"], errors="ignore"
        )
        .merge(
            segment_source_29[
                ["speech_id", "segment_id", "segment_text"]
            ],
            on=["speech_id", "segment_id"],
            how="left",
            validate="one_to_one"
        )
    )

assert usa_source_29["segment_text"].notna().all()

usa_source_29["segment_text"] = (
    usa_source_29["segment_text"].astype(str)
)

usa_source_29["source_characters"] = (
    usa_source_29["segment_text"].str.len()
)

assert usa_source_29["source_characters"].gt(100).all()

usa_source_29["source_preview"] = (
    usa_source_29["segment_text"]
    .str.replace(r"\s+", " ", regex=True)
    .str.slice(0, 180)
)

print("=" * 65)
print("STEP 12.27H.29 — US SEMANTIC VALIDATION")
print("=" * 65)

print("\nSOURCE RECORD INVENTORY")

display(
    usa_source_29[
        [
            "review_id", "year", "period",
            "speech_id", "segment_id",
            "topic03_share_recovered",
            "source_characters", "source_preview"
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 2. Geographic and thematic lexical screening
# ------------------------------------------------------------

patterns_29 = {
    "Korean Peninsula":
        r"\b(?:korea|korean)\b",

    "Palestine / Israel":
        r"\b(?:palestin\w*|israel\w*|gaza|jerusalem)\b",

    "Middle East / Gulf":
        r"\b(?:middle east|persian gulf|saudi arabia|qatar|uae)\b",

    "Cambodia / Kampuchea":
        r"\b(?:cambodia\w*|kampuchea\w*)\b",

    "Diplomacy / Peace":
        r"\b(?:peace|peaceful|negotiat\w*|"
        r"dialog\w*|settlement|conference|"
        r"agreement|reconcil\w*|ceasefire|"
        r"mediat\w*|diplomac\w*)\b",

    "UN / Peacekeeping":
        r"\b(?:united nations|peacekeeping|"
        r"blue helmets|security council)\b",

    "Armed Conflict":
        r"\b(?:war|wars|warfare|conflict\w*|"
        r"military|troops|aggression|combat)\b",

    "Immigration / Domestic":
        r"\b(?:immigra\w*|deport\w*|"
        r"criminal\w*|border\w*)\b",

    "NATO / Alliances":
        r"\b(?:nato|alliance|alliances|"
        r"allies|defense spending)\b"
}

usa_lexical_29 = usa_source_29[
    [
        "review_id", "year", "period",
        "topic03_share_recovered"
    ]
].copy()

for category, pattern in patterns_29.items():
    usa_lexical_29[category] = (
        usa_source_29["segment_text"]
        .str.count(pattern, flags=0)
        .astype(int)
    )

print("\n" + "=" * 65)
print("GEOGRAPHIC AND THEMATIC LEXICAL EVIDENCE")
print("=" * 65)

display(usa_lexical_29.round(5))

# ------------------------------------------------------------
# 3. Historical coverage diagnostics
# ------------------------------------------------------------

usa_period_29 = (
    usa_source_29.groupby("period", as_index=False)
    .agg(
        n_segments=("review_id", "size"),
        n_speeches=("speech_id", "nunique"),
        mean_sampled_topic_share=(
            "topic03_share_recovered", "mean"
        )
    )
)

print("\n" + "=" * 65)
print("HISTORICAL SAMPLE COVERAGE")
print("=" * 65)

display(usa_period_29.round(5))

# ------------------------------------------------------------
# 4. Identify potentially heterogeneous sources
# ------------------------------------------------------------

usa_priority_29 = usa_lexical_29[
    [
        "review_id", "year",
        "Korean Peninsula",
        "Palestine / Israel",
        "Diplomacy / Peace",
        "UN / Peacekeeping",
        "Immigration / Domestic",
        "NATO / Alliances"
    ]
].copy()

print("\n" + "=" * 65)
print("SEMANTIC HETEROGENEITY SCREENING")
print("=" * 65)

display(usa_priority_29)

# ------------------------------------------------------------
# 5. Inspect the contemporary US source
# ------------------------------------------------------------

TARGET_REVIEW_ID_29 = "SV-X012"

target_29 = usa_source_29.loc[
    usa_source_29["review_id"].eq(TARGET_REVIEW_ID_29)
]

assert len(target_29) == 1

target_row_29 = target_29.iloc[0]

print("\n" + "=" * 65)
print("SELECTED ORIGINAL CACHED SEGMENT")
print("=" * 65)

print("Review ID:", target_row_29["review_id"])
print("Country:", target_row_29["country_code"])
print("Year:", target_row_29["year"])
print("Speech ID:", target_row_29["speech_id"])
print("Segment ID:", target_row_29["segment_id"])
print(
    "Topic 03 share:",
    round(target_row_29["topic03_share_recovered"], 6)
)

print("\nCOMPLETE CACHED TEXT:\n")
print(target_row_29["segment_text"])

# ------------------------------------------------------------
# 6. Prepare the human-review inventory
# ------------------------------------------------------------

usa_human_review_29 = usa_source_29[
    [
        "review_id", "country_code", "year",
        "period", "speech_id", "segment_id",
        "topic03_share_recovered"
    ]
].copy()

for field in [
    "primary_geographic_focus",
    "principal_substantive_issue",
    "diplomatic_or_peace_process",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "supporting_evidence",
    "reviewer_notes"
]:
    usa_human_review_29[field] = pd.NA

usa_human_review_29["human_review_status"] = "PENDING"

print("\n" + "=" * 65)
print("HUMAN SEMANTIC CODING INVENTORY")
print("=" * 65)

display(usa_human_review_29)

# ------------------------------------------------------------
# 7. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("United States source records:", len(usa_source_29))
print("Historical periods:", usa_source_29["period"].nunique())
print("Distinct speeches:", usa_source_29["speech_id"].nunique())
print("Cached source linkage: PASSED")
print("Lexical screening: COMPLETED")
print("Independent raw-file verification: NOT PERFORMED")
print("Human semantic adjudication: PENDING")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.29: COMPUTATION COMPLETED")



### Step 12.27H.29A — Case-Insensitive Lexical Screening Correction

**Objective:** Correct the capitalization sensitivity in the United States geographic and thematic keyword diagnostics.

This step preserves the original cached source segments and NMF topic contributions while recomputing lexical counts using lowercase-normalized text.

**Validation criteria:**
1. Confirm that geographic references are recovered from known source passages.
2. Compare corrected counts against the previous case-sensitive counts.
3. Verify that corrected counts never decrease.
4. Preserve all original statistical estimates and model parameters.

**Output:** Corrected lexical evidence and reproducibility diagnostics. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.29A
# CASE-INSENSITIVE LEXICAL SCREENING CORRECTION
# ============================================================

assert "usa_source_29" in globals()
assert "usa_lexical_29" in globals()
assert "patterns_29" in globals()

# Preserve the previous diagnostic results.
usa_lexical_case_sensitive_29 = usa_lexical_29.copy(deep=True)

# ------------------------------------------------------------
# 1. Normalize text for case-insensitive matching
# ------------------------------------------------------------

normalized_text_29 = (
    usa_source_29["segment_text"]
    .astype("string")
    .str.lower()
)

categories_29 = list(patterns_29.keys())

usa_lexical_corrected_29 = usa_source_29[
    [
        "review_id",
        "year",
        "period",
        "topic03_share_recovered"
    ]
].copy()

for category, pattern in patterns_29.items():
    usa_lexical_corrected_29[category] = (
        normalized_text_29
        .str.count(pattern)
        .fillna(0)
        .astype(int)
    )

# ------------------------------------------------------------
# 2. Compare old and corrected counts
# ------------------------------------------------------------

previous_29 = (
    usa_lexical_case_sensitive_29
    .set_index("review_id")[categories_29]
)

corrected_29 = (
    usa_lexical_corrected_29
    .set_index("review_id")[categories_29]
)

assert previous_29.index.is_unique
assert corrected_29.index.is_unique
assert previous_29.index.equals(corrected_29.index)

count_changes_29 = corrected_29 - previous_29

assert count_changes_29.ge(0).all().all(), (
    "Corrected lexical counts unexpectedly decreased."
)

print("=" * 65)
print("STEP 12.27H.29A — LEXICAL CORRECTION AUDIT")
print("=" * 65)

print("\nCATEGORY-LEVEL CORRECTION SUMMARY")

correction_summary_29 = pd.DataFrame({
    "category": categories_29,
    "previous_mentions": previous_29.sum().to_numpy(),
    "corrected_mentions": corrected_29.sum().to_numpy()
})

correction_summary_29["additional_mentions"] = (
    correction_summary_29["corrected_mentions"]
    - correction_summary_29["previous_mentions"]
)

display(correction_summary_29)

# ------------------------------------------------------------
# 3. Validate known geographic references
# ------------------------------------------------------------

expected_evidence_29 = {
    ("SV-X007", "Korean Peninsula"): 1,
    ("SV-X008", "Korean Peninsula"): 1,
    ("SV-X011", "Palestine / Israel"): 1,
    ("SV-X012", "Middle East / Gulf"): 1,
    ("SV-X012", "Korean Peninsula"): 1
}

validation_records_29 = []

for (review_id, category), minimum in expected_evidence_29.items():
    observed = int(corrected_29.loc[review_id, category])

    validation_records_29.append({
        "review_id": review_id,
        "category": category,
        "expected_minimum": minimum,
        "observed_mentions": observed,
        "passed": observed >= minimum
    })

lexical_validation_29 = pd.DataFrame(validation_records_29)

print("\n" + "=" * 65)
print("SOURCE-BASED GEOGRAPHIC VERIFICATION")
print("=" * 65)

display(lexical_validation_29)

assert lexical_validation_29["passed"].all(), (
    "One or more expected geographic references are missing."
)

# ------------------------------------------------------------
# 4. Update the working diagnostic table
# ------------------------------------------------------------

usa_lexical_29 = usa_lexical_corrected_29.copy()

print("\n" + "=" * 65)
print("CORRECTED UNITED STATES LEXICAL EVIDENCE")
print("=" * 65)

display(usa_lexical_29.round(6))

# ------------------------------------------------------------
# 5. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Source records:", len(usa_source_29))
print("Case-insensitive lexical correction: PASSED")
print("Known geographic references: VERIFIED")
print("Previous diagnostic table preserved: YES")
print("Original source segments modified: NO")
print("Original NMF model modified: NO")
print("Statistical results modified: NO")
print("Human semantic adjudication: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.29A: VALIDATION COMPLETED")



### Step 12.27H.30 — Harmonized Cross-Country Semantic Evidence

**Objective:** Establish a reproducible, harmonized lexical evidence framework for Topic 03 across China, Iran, and the United States.

**Procedure:**
1. Retrieve all 18 selected source segments from the original segment inventory.
2. Verify unique source identifiers and complete text linkage.
3. Apply identical case-insensitive geographic and thematic dictionaries across countries.
4. Compute raw lexical counts and length-adjusted frequencies per 1,000 words.
5. Summarize the three historical periods separately for each country.
6. Prepare a traceable qualitative coding worksheet without automatically assigning human judgments.

**Scientific limitations:** Lexical frequency does not establish substantive topic relevance. Geographic references may occur incidentally, and thematic dictionaries may capture contextually different meanings. Historical samples are diagnostic rather than representative.

**Output:** Cross-country evidence tables and an unfilled human-adjudication worksheet. Original model results remain unchanged. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.30
# HARMONIZED CROSS-COUNTRY SEMANTIC EVIDENCE
# ============================================================

assert "semantic_master_review_df" in globals()
assert "segments" in globals()

keys_30 = [
    "country_code",
    "year",
    "speech_id",
    "segment_id"
]

review_30 = semantic_master_review_df.copy()
source_30 = pd.DataFrame(segments).copy()

required_review_30 = keys_30 + [
    "review_id",
    "period",
    "topic03_share_recovered"
]

required_source_30 = keys_30 + ["segment_text"]

assert set(required_review_30).issubset(review_30.columns)
assert set(required_source_30).issubset(source_30.columns)

# ------------------------------------------------------------
# 1. Standardize source identifiers
# ------------------------------------------------------------

for frame in (review_30, source_30):
    frame["country_code"] = (
        frame["country_code"]
        .astype(str)
        .str.strip()
        .str.upper()
    )

    for column in ["year", "speech_id", "segment_id"]:
        frame[column] = pd.to_numeric(
            frame[column],
            errors="raise"
        ).astype(int)

assert len(review_30) == 18
assert review_30["review_id"].is_unique
assert not review_30.duplicated(keys_30).any()
assert not source_30.duplicated(keys_30).any()

assert set(review_30["country_code"]) == {
    "CHN", "IRN", "USA"
}

# ------------------------------------------------------------
# 2. Recover the original source texts
# ------------------------------------------------------------

topic03_evidence_30 = review_30.merge(
    source_30[required_source_30],
    on=keys_30,
    how="left",
    validate="one_to_one"
)

assert len(topic03_evidence_30) == 18
assert topic03_evidence_30["segment_text"].notna().all()

topic03_evidence_30["segment_text"] = (
    topic03_evidence_30["segment_text"].astype(str)
)

topic03_evidence_30["n_words"] = (
    topic03_evidence_30["segment_text"]
    .str.split()
    .str.len()
)

topic03_evidence_30["n_characters"] = (
    topic03_evidence_30["segment_text"].str.len()
)

assert topic03_evidence_30["n_words"].gt(0).all()

# ------------------------------------------------------------
# 3. Define a common lexical screening dictionary
# ------------------------------------------------------------

patterns_30 = {
    "Cambodia / Kampuchea":
        r"\b(?:cambodia(?:n)?|kampuchea(?:n)?)\b",

    "Vietnam":
        r"\b(?:viet[\s-]?nam(?:ese)?|vietnam(?:ese)?)\b",

    "Afghanistan":
        r"\b(?:afghan(?:istan|s)?)\b",

    "Palestine / Israel":
        r"\b(?:palestin(?:e|ian|ians)|israel(?:i|is)?|"
        r"gaza|jerusalem|west[\s-]+bank)\b",

    "Middle East / Gulf":
        r"\b(?:middle[\s-]+east|gulf|"
        r"saudi[\s-]+arabia|qatar|uae|"
        r"united[\s-]+arab[\s-]+emirates)\b",

    "Korean Peninsula":
        r"\b(?:korea(?:n)?|seoul|pyongyang)\b",

    "South Africa / Apartheid":
        r"\b(?:south[\s-]+africa(?:n)?|"
        r"apartheid|namibia(?:n)?|swapo)\b",

    "Bosnia / Kosovo / Yugoslavia":
        r"\b(?:bosnia(?:n)?|kosovo|yugoslavia(?:n)?|"
        r"herzegovina|balkans?)\b",

    "Iraq / Syria":
        r"\b(?:iraq(?:i)?|syria(?:n)?)\b",

    "Diplomacy / Peace":
        r"\b(?:peace(?:ful|fully)?|pacific|"
        r"dialog(?:ue)?|negotiat\w*|mediati\w*|"
        r"settlement|ceasefire|reconcil\w*|"
        r"conference|consultation\w*|diplomac\w*)\b",

    "UN / Peacekeeping":
        r"\b(?:united[\s-]+nations|"
        r"security[\s-]+council|"
        r"general[\s-]+assembly|"
        r"peace[\s-]*keep\w*|"
        r"blue[\s-]+helmets?)\b",

    "Armed Conflict":
        r"\b(?:war|wars|armed|aggression|aggressors?|"
        r"attacks?|attacked|troops?|invasion|"
        r"military|conflicts?|violence|combat)\b",

    "Sovereignty / Non-interference":
        r"\b(?:sovereign(?:ty)?|"
        r"non[\s-]*interference|intervention|"
        r"independen(?:ce|t)|"
        r"self[\s-]determination)\b",

    "Immigration / Domestic":
        r"\b(?:immigrat\w*|migrant\w*|"
        r"deport\w*|asylum)\b",

    "NATO / Alliances":
        r"\b(?:nato|allianc(?:e|es)|allies|"
        r"defen[sc]e[\s-]+spending)\b",

    "Trade / Economy":
        r"\b(?:trade|economy|economic|"
        r"development|gdp|investment|financial)\b"
}

normalized_30 = (
    topic03_evidence_30["segment_text"]
    .str.lower()
)

for category, pattern in patterns_30.items():
    topic03_evidence_30[category] = (
        normalized_30.str.count(pattern).astype(int)
    )

lexical_columns_30 = list(patterns_30)

assert topic03_evidence_30[
    lexical_columns_30
].ge(0).all().all()

# ------------------------------------------------------------
# 4. Verify country-period sampling coverage
# ------------------------------------------------------------

group_keys_30 = ["country_code", "period"]

coverage_30 = (
    topic03_evidence_30
    .groupby(group_keys_30)
    .agg(
        n_segments=("review_id", "size"),
        n_speeches=("speech_id", "nunique"),
        total_words=("n_words", "sum"),
        mean_topic_share=(
            "topic03_share_recovered", "mean"
        )
    )
    .reset_index()
)

assert len(coverage_30) == 9
assert coverage_30["n_segments"].eq(2).all()
assert coverage_30["n_speeches"].eq(2).all()

print("=" * 65)
print("STEP 12.27H.30 — CROSS-COUNTRY VALIDATION")
print("=" * 65)

print("\nSOURCE COVERAGE")
display(coverage_30.round(5))

# ------------------------------------------------------------
# 5. Compute harmonized country-period frequencies
# ------------------------------------------------------------

lexical_totals_30 = (
    topic03_evidence_30
    .groupby(group_keys_30)[lexical_columns_30]
    .sum()
    .reset_index()
)

country_period_evidence_30 = coverage_30.merge(
    lexical_totals_30,
    on=group_keys_30,
    validate="one_to_one"
)

for category in lexical_columns_30:
    country_period_evidence_30[
        category + " (per 1,000 words)"
    ] = (
        1000
        * country_period_evidence_30[category]
        / country_period_evidence_30["total_words"]
    )

geographic_categories_30 = [
    "Cambodia / Kampuchea",
    "Palestine / Israel",
    "Korean Peninsula",
    "South Africa / Apartheid",
    "Bosnia / Kosovo / Yugoslavia",
    "Afghanistan",
    "Middle East / Gulf"
]

thematic_categories_30 = [
    "Diplomacy / Peace",
    "UN / Peacekeeping",
    "Armed Conflict",
    "Sovereignty / Non-interference",
    "Immigration / Domestic",
    "NATO / Alliances",
    "Trade / Economy"
]

print("\n" + "=" * 65)
print("HARMONIZED GEOGRAPHIC EVIDENCE")
print("MENTIONS PER 1,000 SOURCE WORDS")
print("=" * 65)

display(
    country_period_evidence_30[
        group_keys_30 + [
            c + " (per 1,000 words)"
            for c in geographic_categories_30
        ]
    ].round(3)
)

print("\n" + "=" * 65)
print("HARMONIZED THEMATIC EVIDENCE")
print("MENTIONS PER 1,000 SOURCE WORDS")
print("=" * 65)

display(
    country_period_evidence_30[
        group_keys_30 + [
            c + " (per 1,000 words)"
            for c in thematic_categories_30
        ]
    ].round(3)
)

# ------------------------------------------------------------
# 6. Prepare traceable source-level evidence
# ------------------------------------------------------------

def summarize_lexical_signals_30(row):
    ranked = sorted(
        (
            (category, int(row[category]))
            for category in lexical_columns_30
            if int(row[category]) > 0
        ),
        key=lambda item: (-item[1], item[0])
    )

    return "; ".join(
        f"{category} ({count})"
        for category, count in ranked[:4]
    )

topic03_evidence_30["leading_lexical_signals"] = (
    topic03_evidence_30.apply(
        summarize_lexical_signals_30,
        axis=1
    )
)

source_inventory_30 = topic03_evidence_30[
    [
        "review_id",
        "country_code",
        "year",
        "period",
        "topic03_share_recovered",
        "n_words",
        "leading_lexical_signals"
    ]
].sort_values(
    ["country_code", "year"]
).reset_index(drop=True)

print("\n" + "=" * 65)
print("SOURCE-LEVEL SEMANTIC SCREENING INVENTORY")
print("=" * 65)

display(source_inventory_30)

# ------------------------------------------------------------
# 7. Prepare manual adjudication worksheet
# ------------------------------------------------------------

topic03_review_sheet_30 = topic03_evidence_30[
    [
        "review_id",
        "country_code",
        "year",
        "period",
        "speech_id",
        "segment_id",
        "topic03_share_recovered",
        "leading_lexical_signals",
        "segment_text"
    ]
].copy()

manual_fields_30 = [
    "primary_geographic_focus",
    "principal_substantive_issue",
    "diplomacy_or_peace_process",
    "kampuchea_specific_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "supporting_evidence",
    "reviewer_notes"
]

for field in manual_fields_30:
    topic03_review_sheet_30[field] = pd.NA

topic03_review_sheet_30["human_review_status"] = "PENDING"

# Flag the previously identified source-integrity issue.
topic03_review_sheet_30["source_integrity_note"] = (
    topic03_review_sheet_30["review_id"].map({
        "SV-X003":
        "Earlier text discontinuity flagged; source check required."
    }).fillna("No previously flagged discontinuity")
)

assert len(topic03_review_sheet_30) == 18
assert topic03_review_sheet_30["human_review_status"].eq(
    "PENDING"
).all()

# ------------------------------------------------------------
# 8. Final reproducibility diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Countries verified:",
      topic03_evidence_30["country_code"].nunique())

print("Country-period strata:", len(coverage_30))
print("Source segments recovered:", len(topic03_evidence_30))
print("Harmonized lexical categories:", len(patterns_30))
print("Manual coding records:", len(topic03_review_sheet_30))

print("Source identification: PASSED")
print("Case normalization: PASSED")
print("Historical coverage: PASSED")
print("Length-adjusted diagnostics: COMPLETED")
print("Human semantic validation: PENDING")
print("Original NMF model modified: NO")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.30: COMPUTATION COMPLETED")



### Step 12.27H.31 — Source-Level Semantic Adjudication

**Objective:** Conduct traceable qualitative review of the 18 selected Topic 03 segments, using complete cached source texts rather than lexical frequencies alone.

**Review protocol:**
1. Organize the 18 source records into six reproducible batches.
2. Display three complete source segments per batch.
3. Inspect the primary geographic focus and substantive policy issue.
4. Distinguish diplomatic settlement, conflict, sovereignty, and unrelated discourse.
5. Evaluate whether each segment supports the narrow Kampuchea label, the broader conflict-and-diplomacy label, or neither.
6. Record proposed interpretations separately from confirmed human judgments.

**Important:** No automatic human coding or label replacement is performed in this step. The original NMF model and statistical estimates remain unchanged.

**Output:** Full-text qualitative review packets. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.31
# SOURCE-LEVEL SEMANTIC ADJUDICATION
# CORRECTED VERSION
# ============================================================

assert "topic03_review_sheet_30" in globals()
assert isinstance(topic03_review_sheet_30, pd.DataFrame)

# ------------------------------------------------------------
# 1. Recover and validate the review inventory
# ------------------------------------------------------------

review_source_31 = topic03_review_sheet_30.copy(deep=True)

required_columns_31 = [
    "review_id",
    "country_code",
    "year",
    "segment_text",
]

missing_columns_31 = [
    col for col in required_columns_31
    if col not in review_source_31.columns
]

assert not missing_columns_31, (
    f"Required columns missing: {missing_columns_31}"
)

assert len(review_source_31) == 18
assert review_source_31["review_id"].is_unique
assert review_source_31["segment_text"].notna().all()

# ------------------------------------------------------------
# 2. Construct missing descriptive fields safely
# ------------------------------------------------------------

review_source_31["source_word_count_31"] = (
    review_source_31["segment_text"]
    .astype(str)
    .str.split()
    .str.len()
)

review_source_31["source_characters_31"] = (
    review_source_31["segment_text"]
    .astype(str)
    .str.len()
)

if "period" not in review_source_31.columns:
    years_31 = pd.to_numeric(
        review_source_31["year"],
        errors="raise"
    )

    review_source_31["period"] = np.select(
        [
            years_31.between(1972, 1990),
            years_31.between(1991, 2010),
            years_31.between(2011, 2025),
        ],
        [
            "1972-1990",
            "1991-2010",
            "2011-2025",
        ],
        default="OUTSIDE_STUDY_PERIOD",
    )

assert review_source_31["source_word_count_31"].gt(0).all()
assert review_source_31["source_characters_31"].gt(0).all()

if "human_review_status" not in review_source_31.columns:
    review_source_31["human_review_status"] = "PENDING"

if "topic03_share_recovered" not in review_source_31.columns:
    review_source_31["topic03_share_recovered"] = np.nan

# ------------------------------------------------------------
# 3. Build reproducible review batches
# ------------------------------------------------------------

REVIEW_BATCH_31 = 1
BATCH_SIZE_31 = 3

review_source_31 = (
    review_source_31
    .sort_values(["country_code", "year", "review_id"])
    .reset_index(drop=True)
)

n_batches_31 = int(
    np.ceil(len(review_source_31) / BATCH_SIZE_31)
)

assert 1 <= REVIEW_BATCH_31 <= n_batches_31

start_31 = (
    (REVIEW_BATCH_31 - 1) * BATCH_SIZE_31
)

end_31 = min(
    start_31 + BATCH_SIZE_31,
    len(review_source_31)
)

selected_review_31 = review_source_31.iloc[
    start_31:end_31
].copy()

assert len(selected_review_31) == 3

# ------------------------------------------------------------
# 4. Display the selected source inventory
# ------------------------------------------------------------

print("=" * 68)
print("STEP 12.27H.31 — SOURCE-LEVEL SEMANTIC ADJUDICATION")
print("=" * 68)

print(f"Review batch: {REVIEW_BATCH_31}/{n_batches_31}")
print(f"Selected source records: {len(selected_review_31)}")

inventory_columns_31 = [
    "review_id",
    "country_code",
    "year",
    "period",
    "topic03_share_recovered",
    "source_word_count_31",
    "source_characters_31",
    "human_review_status",
]

print("\nSOURCE RECORD INVENTORY")

display(
    selected_review_31[
        inventory_columns_31
    ].round(6)
)

# ------------------------------------------------------------
# 5. Display complete cached segment texts
# ------------------------------------------------------------

for _, row_31 in selected_review_31.iterrows():

    print("\n" + "#" * 68)

    print("REVIEW ID:", row_31["review_id"])
    print("COUNTRY:", row_31["country_code"])
    print("YEAR:", row_31["year"])
    print("PERIOD:", row_31["period"])

    print(
        "TOPIC 03 SHARE:",
        row_31["topic03_share_recovered"]
    )

    print(
        "SOURCE WORD COUNT:",
        row_31["source_word_count_31"]
    )

    print(
        "SOURCE CHARACTER COUNT:",
        row_31["source_characters_31"]
    )

    print("\nCOMPLETE CACHED SOURCE SEGMENT")
    print("-" * 68)

    print(row_31["segment_text"])

    print("-" * 68)

    # Optional metadata: display only when available.
    if (
        "source_integrity_note"
        in selected_review_31.columns
        and pd.notna(row_31["source_integrity_note"])
    ):
        print("\nSOURCE INTEGRITY NOTE")
        print(row_31["source_integrity_note"])

    if (
        "leading_lexical_signals"
        in selected_review_31.columns
        and pd.notna(row_31["leading_lexical_signals"])
    ):
        print("\nPRELIMINARY LEXICAL EVIDENCE")
        print(row_31["leading_lexical_signals"])

    print("\nQUALITATIVE REVIEW QUESTIONS")

    questions_31 = [
        "What is the principal substantive issue?",
        "What is the primary geographic focus?",
        "Is conflict or diplomacy substantively discussed?",
        "Does the narrow Kampuchea label fit?",
        "Does the broader Topic 03 label fit?",
        "Is substantial semantic heterogeneity present?",
    ]

    for question_number_31, question_31 in enumerate(
        questions_31, start=1
    ):
        print(f"{question_number_31}. {question_31}")

    print(
        "\nHUMAN REVIEW STATUS:",
        row_31["human_review_status"]
    )

# ------------------------------------------------------------
# 6. Final validation
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print("Total source records:", len(review_source_31))
print("Review batches:", n_batches_31)
print("Records presented:", len(selected_review_31))

print(
    "All selected texts recovered:",
    bool(selected_review_31["segment_text"].notna().all())
)

print(
    "Source word counts computed:",
    bool(selected_review_31["source_word_count_31"].gt(0).all())
)

print("Original source dataframe modified: NO")
print("Original NMF model modified: NO")
print("Statistical results modified: NO")
print("Human semantic confirmation: PENDING")
print("Files saved: NONE")

print("\nSTEP 12.27H.31: REVIEW PACKET PREPARED")



### Step 12.27H.31A — Source Integrity and Text-Order Verification

**Objective:** Verify the completeness, ordering, and source correspondence of the 1991 Chinese segment (SV-015) before final semantic adjudication.

This diagnostic compares the cached review text with the original segment and, where available, the speech-level source. It checks whether the apparent discontinuity arises from notebook output truncation or the underlying source data.

No source records, fitted models, statistical estimates, or publication files are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31A
# SOURCE INTEGRITY AND TEXT-ORDER VERIFICATION
# ============================================================

assert "review_source_31" in globals()

target_31a = review_source_31.loc[
    review_source_31["review_id"].eq("SV-015")
]

assert len(target_31a) == 1

record_31a = target_31a.iloc[0]
text_31a = str(record_31a["segment_text"])

print("=" * 65)
print("STEP 12.27H.31A — SOURCE INTEGRITY VERIFICATION")
print("=" * 65)

print("Review ID:", record_31a["review_id"])
print("Country:", record_31a["country_code"])
print("Year:", record_31a["year"])
print("Speech ID:", record_31a["speech_id"])
print("Segment ID:", record_31a["segment_id"])

print("\nCACHED TEXT METRICS")
print("Characters:", len(text_31a))
print("Words:", len(text_31a.split()))

print("\nCACHED TEXT — COMPLETE")
print("-" * 65)
print(text_31a)
print("-" * 65)

# ------------------------------------------------------------
# 1. Identify key textual markers
# ------------------------------------------------------------

markers_31a = {
    "Occupied territories opening":
        "the occupied arab territories should be returned",
    "North-South economic discrepancy":
        "the discrepancies between north and south",
    "Middle East peace process":
        "useful with a view to accelerating",
    "Arab-Jewish peaceful coexistence":
        "peaceful coexistence of the arab and the jewish",
}

print("\nTEXTUAL MARKER POSITIONS")

for label_31a, phrase_31a in markers_31a.items():
    position_31a = text_31a.lower().find(phrase_31a)

    print(
        f"{label_31a}: "
        f"{position_31a if position_31a >= 0 else 'NOT FOUND'}"
    )

# ------------------------------------------------------------
# 2. Compare with original segment source
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ORIGINAL SEGMENT COMPARISON")
print("=" * 65)

if (
    "segments" in globals()
    and isinstance(segments, pd.DataFrame)
    and all(
        column_31a in segments.columns
        for column_31a in [
            "speech_id",
            "segment_id",
            "segment_text",
        ]
    )
):

    original_match_31a = segments.loc[
        pd.to_numeric(
            segments["speech_id"], errors="coerce"
        ).eq(int(record_31a["speech_id"]))
        &
        pd.to_numeric(
            segments["segment_id"], errors="coerce"
        ).eq(int(record_31a["segment_id"]))
    ]

    print("Matching original segments:", len(original_match_31a))

    if len(original_match_31a) == 1:
        original_text_31a = str(
            original_match_31a.iloc[0]["segment_text"]
        )

        print(
            "Original characters:",
            len(original_text_31a)
        )

        print(
            "Original words:",
            len(original_text_31a.split())
        )

        print(
            "Exact cached-source agreement:",
            text_31a == original_text_31a
        )

    else:
        print("Unique original segment match unavailable.")

else:
    print("Original segment table unavailable for comparison.")

# ------------------------------------------------------------
# 3. Inspect speech-level evidence when available
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SPEECH-LEVEL SOURCE ORDER")
print("=" * 65)

if (
    "df_nlp" in globals()
    and isinstance(df_nlp, pd.DataFrame)
    and all(
        column_31a in df_nlp.columns
        for column_31a in ["country_code", "year", "text_raw"]
    )
):

    original_speech_31a = df_nlp.loc[
        df_nlp["country_code"].astype(str).eq("CHN")
        &
        pd.to_numeric(
            df_nlp["year"], errors="coerce"
        ).eq(1991)
    ]

    if "speech_id" in original_speech_31a.columns:
        original_speech_31a = original_speech_31a.loc[
            pd.to_numeric(
                original_speech_31a["speech_id"],
                errors="coerce"
            ).eq(int(record_31a["speech_id"]))
        ]

    print("Matching speeches:", len(original_speech_31a))

    if len(original_speech_31a) == 1:

        speech_text_31a = str(
            original_speech_31a.iloc[0]["text_raw"]
        )

        print("Full speech characters:", len(speech_text_31a))

        for label_31a, phrase_31a in markers_31a.items():
            position_31a = speech_text_31a.lower().find(
                phrase_31a
            )

            print(
                f"{label_31a}: "
                f"{position_31a if position_31a >= 0 else 'NOT FOUND'}"
            )

    else:
        print("Unique speech-level match unavailable.")

else:
    print("Speech-level raw text unavailable.")

# ------------------------------------------------------------
# 4. Final verification status
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL DIAGNOSTICS")
print("=" * 65)

print("Cached record identified: PASSED")
print("Cached text inspected: PASSED")
print("Source order: REQUIRES INTERPRETATION")
print("Human semantic adjudication: PENDING")
print("Original source records modified: NO")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31A: DIAGNOSTIC COMPLETED")



### Step 12.27H.31B — Exact Source Alignment Verification

**Objective:** Establish whether the cached 1991 Chinese segment is an exact, contiguous substring of the original speech.

The audit also identifies control characters and displays the source text in short, escaped chunks to prevent misleading notebook rendering.

**Decision rule:** Exact substring agreement and consistent source boundaries are required before completing semantic adjudication.

No original data, NMF components, statistical estimates, or publication outputs are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31B
# EXACT SOURCE ALIGNMENT VERIFICATION
# ============================================================

assert "review_source_31" in globals()
assert "df_nlp" in globals()

record_31b = review_source_31.loc[
    review_source_31["review_id"].eq("SV-015")
]

assert len(record_31b) == 1

record_31b = record_31b.iloc[0]
cached_text_31b = str(record_31b["segment_text"])

speech_match_31b = df_nlp.loc[
    df_nlp["country_code"].eq("CHN")
    & pd.to_numeric(
        df_nlp["year"], errors="coerce"
    ).eq(1991)
]

if "speech_id" in speech_match_31b.columns:
    speech_match_31b = speech_match_31b.loc[
        pd.to_numeric(
            speech_match_31b["speech_id"], errors="coerce"
        ).eq(int(record_31b["speech_id"]))
    ]

assert len(speech_match_31b) == 1, (
    "Unique original speech could not be established."
)

speech_text_31b = str(
    speech_match_31b.iloc[0]["text_raw"]
)

print("=" * 65)
print("STEP 12.27H.31B — EXACT SOURCE VERIFICATION")
print("=" * 65)

# ------------------------------------------------------------
# 1. Verify exact contiguous source correspondence
# ------------------------------------------------------------

source_start_31b = speech_text_31b.find(cached_text_31b)

exact_match_31b = source_start_31b >= 0

print("\nSOURCE ALIGNMENT")

print("Cached characters:", len(cached_text_31b))
print("Full speech characters:", len(speech_text_31b))
print("Exact contiguous substring:", exact_match_31b)

if exact_match_31b:
    source_end_31b = (
        source_start_31b + len(cached_text_31b)
    )

    print("Source start:", source_start_31b)
    print("Source end (exclusive):", source_end_31b)
    print(
        "Exact source slice agreement:",
        speech_text_31b[
            source_start_31b:source_end_31b
        ] == cached_text_31b
    )

else:
    print(
        "Cached prefix location:",
        speech_text_31b.find(cached_text_31b[:80])
    )
    print(
        "Cached suffix location:",
        speech_text_31b.find(cached_text_31b[-80:])
    )

# ------------------------------------------------------------
# 2. Detect potentially disruptive control characters
# ------------------------------------------------------------

control_chars_31b = [
    {
        "position": index_31b,
        "unicode": f"U+{ord(char_31b):04X}",
        "representation": ascii(char_31b),
    }
    for index_31b, char_31b in enumerate(cached_text_31b)
    if (
        (ord(char_31b) < 32 and char_31b not in "\n\t")
        or ord(char_31b) == 127
        or char_31b in "\u2028\u2029"
    )
]

print("\nCONTROL CHARACTER DIAGNOSTICS")
print("Potentially disruptive characters:", len(control_chars_31b))

if control_chars_31b:
    display(pd.DataFrame(control_chars_31b))
else:
    print("No disruptive control characters identified.")

# ------------------------------------------------------------
# 3. Render the full cached text safely
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("ESCAPED SOURCE TEXT — NUMBERED CHUNKS")
print("=" * 65)

chunk_size_31b = 90

for offset_31b in range(
    0, len(cached_text_31b), chunk_size_31b
):
    chunk_31b = cached_text_31b[
        offset_31b:offset_31b + chunk_size_31b
    ]

    print(
        f"{offset_31b:04d} | {ascii(chunk_31b)}"
    )

# ------------------------------------------------------------
# 4. Final audit decision
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print(
    "Exact source correspondence:",
    "PASSED" if exact_match_31b else "NOT VERIFIED"
)

print(
    "Control-character screening:",
    "PASSED" if not control_chars_31b else "FLAGGED"
)

print("Semantic adjudication: PENDING")
print("Original source records modified: NO")
print("Original NMF model modified: NO")
print("Statistical estimates modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31B: DIAGNOSTIC COMPLETED")



### Step 12.27H.31C — Evidence-Based Chinese Semantic Coding

**Objective:** Document provisional qualitative interpretations of three Chinese source segments (1981, 1988, and 1991).

**Assessment criteria:**
- Primary geographic and substantive focus
- Evidence of diplomatic settlement or peace processes
- Fit with the narrow Kampuchea-specific topic label
- Fit with the broader regional conflict and diplomatic settlement label
- Within-segment semantic heterogeneity

Every interpretation must be linked to exact source text.

All coding decisions remain provisional until researcher confirmation. Original texts, model components, statistical estimates, and publication tables remain unchanged.


In [ ]:

# ============================================================
# STEP 12.27H.31C
# EVIDENCE-BASED CHINESE SEMANTIC CODING
# ============================================================

assert "review_source_31" in globals()

required_columns_31c = {
    "review_id", "country_code", "year", "segment_text"
}

assert required_columns_31c.issubset(
    review_source_31.columns
)

# Provisional researcher-review proposals.
# These are NOT confirmed human coding decisions.

coding_proposals_31c = [
    {
        "review_id": "SV-013",
        "primary_geographic_focus":
            "Kampuchea/Cambodia and Afghanistan",
        "principal_substantive_issue":
            "Foreign intervention, independence and sovereignty",
        "diplomatic_or_peace_process":
            "Yes - political settlement discussed alongside struggle",
        "kampuchea_specific_label_fit": "Partial",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - two connected regional conflicts",
        "evidence_anchor":
            "a political settlement of the Afghan and Kampuchean issues",
        "reviewer_notes":
            "Kampuchea is central but not the sole geographic focus.",
    },
    {
        "review_id": "SV-014",
        "primary_geographic_focus":
            "Kampuchea/Cambodia; secondary Korean and Afghan references",
        "principal_substantive_issue":
            "Vietnamese military withdrawal and non-interference",
        "diplomatic_or_peace_process":
            "Yes - withdrawal, settlement and dialogue",
        "kampuchea_specific_label_fit": "Yes",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - secondary regional references",
        "evidence_anchor":
            "Vietnamese troop withdrawal from Kampuchea",
        "reviewer_notes":
            "Kampuchea is the dominant substantive issue.",
    },
    {
        "review_id": "SV-015",
        "primary_geographic_focus":
            "Palestine/Israel and the Middle East",
        "principal_substantive_issue":
            "Negotiated peace and mutual recognition",
        "diplomatic_or_peace_process":
            "Yes - international conference and peace negotiations",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Limited - final North-South economic transition",
        "evidence_anchor":
            "Genuine and lasting peace can be achieved only through mutual recognition",
        "reviewer_notes":
            "Exact original source alignment verified; three CR characters.",
    },
]

proposal_df_31c = pd.DataFrame(coding_proposals_31c)

# ------------------------------------------------------------
# Verify every supporting excerpt against the source
# ------------------------------------------------------------

records_31c = []

for proposal_31c in coding_proposals_31c:

    review_id_31c = proposal_31c["review_id"]

    match_31c = review_source_31.loc[
        review_source_31["review_id"].eq(review_id_31c)
    ]

    assert len(match_31c) == 1

    source_31c = match_31c.iloc[0]
    text_31c = str(source_31c["segment_text"])
    anchor_31c = proposal_31c["evidence_anchor"]

    evidence_position_31c = text_31c.find(anchor_31c)

    assert evidence_position_31c >= 0, (
        f"Supporting evidence not found: {review_id_31c}"
    )

    assert source_31c["country_code"] == "CHN"

    record_31c = {
        "country_code": source_31c["country_code"],
        "year": int(source_31c["year"]),
        **proposal_31c,
        "evidence_start": evidence_position_31c,
        "evidence_verified": True,
        "human_review_status": "PENDING",
    }

    records_31c.append(record_31c)

china_semantic_proposals_31c = pd.DataFrame(records_31c)

# ------------------------------------------------------------
# Display provisional coding and traceable evidence
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31C — CHINESE SEMANTIC CODING")
print("=" * 65)

display(
    china_semantic_proposals_31c[
        [
            "review_id",
            "year",
            "primary_geographic_focus",
            "principal_substantive_issue",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "human_review_status",
        ]
    ]
)

print("\n" + "=" * 65)
print("SOURCE-VERIFIED SUPPORTING EVIDENCE")
print("=" * 65)

for row_31c in china_semantic_proposals_31c.itertuples(
    index=False
):
    print(f"\n{row_31c.review_id} | {row_31c.year}")
    print("Evidence:", row_31c.evidence_anchor)
    print("Verified:", row_31c.evidence_verified)
    print("Interpretation:", row_31c.principal_substantive_issue)
    print("Status:", row_31c.human_review_status)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

assert len(china_semantic_proposals_31c) == 3
assert china_semantic_proposals_31c["review_id"].is_unique
assert china_semantic_proposals_31c["evidence_verified"].all()
assert china_semantic_proposals_31c[
    "human_review_status"
].eq("PENDING").all()

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Source records reviewed:", 3)
print("Evidence verification: PASSED")
print("Provisional coding records:", 3)
print("Human confirmation: PENDING")
print("Original source data modified: NO")
print("Original NMF model modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")
print("\nSTEP 12.27H.31C: PROVISIONAL CODING PREPARED")



### Step 12.27H.31D — Remaining Chinese Semantic Evidence

**Objective:** Prepare traceable provisional semantic coding for the remaining Chinese records: SV-016 (1999), SV-017 (2014), and SV-018 (2022).

Each interpretation is evaluated against two source-text anchors, with whitespace-normalized matching to accommodate archived line breaks.

The new proposals are combined with Step 12.27H.31C to form a six-record Chinese review inventory.

**Validation policy:** Source evidence verification is computational. Semantic interpretations remain provisional pending researcher confirmation.

No original texts, NMF models, statistical results, or publication files are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31D
# REMAINING CHINESE SEMANTIC ADJUDICATION
# ============================================================

assert "review_source_31" in globals()
assert "china_semantic_proposals_31c" in globals()

required_columns_31d = {
    "review_id", "country_code", "year", "segment_text"
}

assert required_columns_31d.issubset(
    review_source_31.columns
)

# ------------------------------------------------------------
# 1. Provisional semantic coding proposals
# ------------------------------------------------------------

coding_proposals_31d = [
    {
        "review_id": "SV-016",
        "primary_geographic_focus":
            "Kosovo / former Yugoslavia; international legal principles",
        "principal_substantive_issue":
            "Sovereignty, non-intervention and humanitarian intervention",
        "diplomatic_or_peace_process":
            "Yes - peaceful settlement of international disputes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - Kosovo and general international legal norms",
        "evidence_anchor":
            "The Kosovo crisis has proved",
        "secondary_evidence_anchor":
            "pacific settlement of international disputes",
        "reviewer_notes":
            "Conflict-related international law and diplomatic settlement.",
    },
    {
        "review_id": "SV-017",
        "primary_geographic_focus":
            "Syria, Palestine/Israel and the Middle East",
        "principal_substantive_issue":
            "Regional conflict resolution and diplomatic negotiations",
        "diplomatic_or_peace_process":
            "Yes - mediation, ceasefire and peace negotiations",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - several linked Middle Eastern disputes",
        "evidence_anchor":
            "The Palestinian issue is an open wound",
        "secondary_evidence_anchor":
            "Negotiations on the Iranian nuclear issue",
        "reviewer_notes":
            "Syria, Palestine/Israel and Iranian nuclear diplomacy.",
    },
    {
        "review_id": "SV-018",
        "primary_geographic_focus":
            "Palestine, Korean Peninsula and Afghanistan",
        "principal_substantive_issue":
            "Multiple regional disputes, sanctions and sovereignty",
        "diplomatic_or_peace_process":
            "Yes - dialogue and negotiated political arrangements",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_heterogeneity":
            "High - multiple regions and additional policy themes",
        "evidence_anchor":
            "two-State solution is crucial",
        "secondary_evidence_anchor":
            "Afghanistan is in a critical transition from chaos to order",
        "reviewer_notes":
            "Also discusses sanctions, Cuba and firearm-related concerns.",
    },
]

# ------------------------------------------------------------
# 2. Verify source records and textual evidence
# ------------------------------------------------------------

records_31d = []
evidence_records_31d = []

for proposal in coding_proposals_31d:

    review_id = proposal["review_id"]

    source_match = review_source_31.loc[
        review_source_31["review_id"].eq(review_id)
    ]

    assert len(source_match) == 1, (
        f"Source record mismatch: {review_id}"
    )

    source_row = source_match.iloc[0]

    assert source_row["country_code"] == "CHN"

    # Normalize whitespace for evidence matching only.
    # The original source text remains unchanged.
    source_text = str(source_row["segment_text"])
    normalized_text = " ".join(source_text.split())

    anchor_positions = []

    for anchor_type, anchor in [
        ("Primary", proposal["evidence_anchor"]),
        ("Secondary", proposal["secondary_evidence_anchor"]),
    ]:

        position = normalized_text.casefold().find(
            anchor.casefold()
        )

        assert position >= 0, (
            f"{review_id}: {anchor_type} evidence not found: "
            f"{anchor}"
        )

        anchor_positions.append(position)

        context_start = max(0, position - 85)
        context_end = min(
            len(normalized_text),
            position + len(anchor) + 85,
        )

        evidence_records_31d.append({
            "review_id": review_id,
            "year": int(source_row["year"]),
            "anchor_type": anchor_type,
            "source_anchor": anchor,
            "source_context": normalized_text[
                context_start:context_end
            ],
            "verified": True,
        })

    records_31d.append({
        "country_code": source_row["country_code"],
        "year": int(source_row["year"]),
        **proposal,
        "evidence_start": anchor_positions[0],
        "evidence_verified": True,
        "secondary_evidence_verified": True,
        "human_review_status": "PENDING",
    })

china_semantic_proposals_31d = pd.DataFrame(records_31d)

china_evidence_31d = pd.DataFrame(evidence_records_31d)

# ------------------------------------------------------------
# 3. Create combined Chinese review inventory
# ------------------------------------------------------------

china_semantic_proposals_all_31d = pd.concat(
    [
        china_semantic_proposals_31c,
        china_semantic_proposals_31d,
    ],
    ignore_index=True,
    sort=False,
).sort_values(
    ["year", "review_id"]
).reset_index(drop=True)

assert len(china_semantic_proposals_all_31d) == 6

assert china_semantic_proposals_all_31d[
    "review_id"
].is_unique

assert china_semantic_proposals_all_31d[
    "evidence_verified"
].all()

assert china_semantic_proposals_all_31d[
    "human_review_status"
].eq("PENDING").all()

# ------------------------------------------------------------
# 4. Display provisional coding
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31D — CHINESE SEMANTIC ADJUDICATION")
print("=" * 65)

print("\nREMAINING CHINESE SOURCE CODING")

display(
    china_semantic_proposals_31d[
        [
            "review_id",
            "year",
            "primary_geographic_focus",
            "principal_substantive_issue",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "human_review_status",
        ]
    ]
)

print("\n" + "=" * 65)
print("TRACEABLE SOURCE EVIDENCE")
print("=" * 65)

for row in china_evidence_31d.itertuples(index=False):
    print(
        f"\n{row.review_id} | {row.year} | "
        f"{row.anchor_type}"
    )
    print("Anchor:", row.source_anchor)
    print("Context:", row.source_context)
    print("Verified:", row.verified)

# ------------------------------------------------------------
# 5. Combined coverage and final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COMBINED CHINESE SEMANTIC REVIEW")
print("=" * 65)

display(
    china_semantic_proposals_all_31d[
        [
            "review_id",
            "year",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "human_review_status",
        ]
    ]
)

assert len(china_evidence_31d) == 6
assert china_evidence_31d["verified"].all()

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Chinese records evaluated:", 6)
print("New records evaluated:", 3)
print("New source anchors verified:", len(china_evidence_31d))
print("Evidence verification: PASSED")
print("Provisional source coding: PREPARED")
print("Human semantic confirmation: PENDING")
print("Original sources modified: NO")
print("Original NMF models modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31D: CODING PROPOSALS PREPARED")



### Step 12.27H.31E — Iran Source-Level Semantic Adjudication

**Objective:** Evaluate all six Iranian source segments for the semantic coherence and historical scope of reference NMF Topic 03.

The review covers 1977, 1981, 1991, 1992, 2016, and 2025.

**Methodology:**
1. Recover the original cached source segments.
2. Verify two textual evidence anchors for each record.
3. Prepare source-grounded provisional thematic classifications.
4. Assess the Kampuchea-specific and broader conflict/diplomacy labels separately.
5. Preserve the previously identified 1991 source-integrity warning.
6. Maintain an auditable coding inventory for subsequent human review.

**Important:** Computational evidence verification does not constitute independent human semantic confirmation. All classifications remain provisional.

No original source data, NMF models, statistical estimates, or publication files will be modified.


In [ ]:

# ============================================================
# STEP 12.27H.31E
# IRAN SOURCE-LEVEL SEMANTIC ADJUDICATION
# ============================================================

assert "review_source_31" in globals()

required_columns_31e = {
    "review_id",
    "country_code",
    "year",
    "period",
    "segment_text",
}

assert required_columns_31e.issubset(
    review_source_31.columns
)

# ------------------------------------------------------------
# 1. Source-grounded provisional coding proposals
# ------------------------------------------------------------

iran_coding_proposals_31e = [
    {
        "review_id": "SV-X001",
        "primary_geographic_focus":
            "Palestine/Israel and the Middle East",
        "principal_substantive_issue":
            "Arab-Israeli conflict and negotiated peace settlement",
        "diplomatic_or_peace_process": "Yes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Low to moderate - Middle East peace and related disputes",
        "evidence_anchor":
            "Security Council resolutions 242",
        "secondary_evidence_anchor":
            "legitimate rights of the Palestinian people",
        "reviewer_notes":
            "Peace conference, territorial occupation and Palestinian rights.",
    },
    {
        "review_id": "SV-X002",
        "primary_geographic_focus":
            "South Africa, Namibia and Southern Africa",
        "principal_substantive_issue":
            "Apartheid, colonial domination and anti-imperialism",
        "diplomatic_or_peace_process": "Partial",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_heterogeneity":
            "Moderate - apartheid, decolonization and military aggression",
        "evidence_anchor":
            "apartheid in South Africa",
        "secondary_evidence_anchor":
            "occupation of Namibia",
        "reviewer_notes":
            "Substantively distinct anti-apartheid discourse.",
    },
    {
        "review_id": "SV-X003",
        "primary_geographic_focus":
            "Palestine/Israel and the Middle East",
        "principal_substantive_issue":
            "Palestinian self-determination and criticism of peace initiatives",
        "diplomatic_or_peace_process": "Yes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - diplomatic criticism and liberation discourse",
        "evidence_anchor":
            "peace plan for the Middle East",
        "secondary_evidence_anchor":
            "Palestinian people",
        "reviewer_notes":
            "Prior source-integrity flag requires separate resolution.",
    },
    {
        "review_id": "SV-X004",
        "primary_geographic_focus":
            "Palestine and Bosnia/Herzegovina",
        "principal_substantive_issue":
            "Self-determination, regional conflict and territorial sovereignty",
        "diplomatic_or_peace_process": "Partial",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - Palestinian and Balkan conflicts",
        "evidence_anchor":
            "inalienable rights of the Palestinian people",
        "secondary_evidence_anchor":
            "Republic of Bosnia and Herzegovina",
        "reviewer_notes":
            "Emphasis on political rights, sovereignty and conflict.",
    },
    {
        "review_id": "SV-X006",
        "primary_geographic_focus":
            "Middle East, Iraq, Libya and wider regional security",
        "principal_substantive_issue":
            "Terrorism, regional insecurity and transnational conflict",
        "diplomatic_or_peace_process": "Partial",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_heterogeneity":
            "Moderate - terrorism and interconnected regional insecurity",
        "evidence_anchor":
            "security in one region at the cost of insecurity in others",
        "secondary_evidence_anchor":
            "Levant, Iraq and Libya",
        "reviewer_notes":
            "Security-focused discourse with limited direct peace-process evidence.",
    },
    {
        "review_id": "SV-X005",
        "primary_geographic_focus":
            "Palestine/Israel and the wider Middle East",
        "principal_substantive_issue":
            "Palestinian humanitarian crisis, occupation and regional aggression",
        "diplomatic_or_peace_process": "Partial",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_heterogeneity":
            "Low to moderate - concentrated conflict-related discourse",
        "evidence_anchor":
            "mass starvation",
        "secondary_evidence_anchor":
            "scheme of a greater Israel",
        "reviewer_notes":
            "Conflict and humanitarian rhetoric; limited direct diplomacy.",
    },
]

# ------------------------------------------------------------
# 2. Verify original sources and evidence anchors
# ------------------------------------------------------------

iran_records_31e = []
iran_evidence_records_31e = []

for proposal in iran_coding_proposals_31e:

    review_id = proposal["review_id"]

    matched_source = review_source_31.loc[
        review_source_31["review_id"].eq(review_id)
    ]

    assert len(matched_source) == 1, (
        f"Non-unique or missing source: {review_id}"
    )

    source_row = matched_source.iloc[0]

    assert source_row["country_code"] == "IRN"

    original_text = source_row["segment_text"]

    assert isinstance(original_text, str)
    assert original_text.strip()

    normalized_text = " ".join(original_text.split())

    for anchor_type, anchor in [
        ("Primary", proposal["evidence_anchor"]),
        ("Secondary", proposal["secondary_evidence_anchor"]),
    ]:

        position = normalized_text.casefold().find(
            anchor.casefold()
        )

        assert position >= 0, (
            f"{review_id}: {anchor_type} anchor not found: "
            f"{anchor}"
        )

        context_start = max(0, position - 90)
        context_end = min(
            len(normalized_text),
            position + len(anchor) + 90,
        )

        iran_evidence_records_31e.append({
            "review_id": review_id,
            "year": int(source_row["year"]),
            "anchor_type": anchor_type,
            "source_anchor": anchor,
            "source_context": normalized_text[
                context_start:context_end
            ],
            "verified": True,
        })

    # Preserve the unresolved flag from the prior
    # source-integrity audit without modifying source text.
    integrity_status = (
        "PRIOR_FLAG_REQUIRES_REVIEW"
        if review_id == "SV-X003"
        else "NO_PREVIOUSLY_FLAGGED_DISCONTINUITY"
    )

    iran_records_31e.append({
        "review_id": review_id,
        "country_code": "IRN",
        "year": int(source_row["year"]),
        "period": source_row["period"],
        "source_word_count": len(original_text.split()),
        "source_character_count": len(original_text),
        "source_integrity_status": integrity_status,
        "primary_geographic_focus":
            proposal["primary_geographic_focus"],
        "principal_substantive_issue":
            proposal["principal_substantive_issue"],
        "diplomatic_or_peace_process":
            proposal["diplomatic_or_peace_process"],
        "kampuchea_specific_label_fit":
            proposal["kampuchea_specific_label_fit"],
        "broad_conflict_diplomacy_fit":
            proposal["broad_conflict_diplomacy_fit"],
        "semantic_heterogeneity":
            proposal["semantic_heterogeneity"],
        "supporting_text":
            proposal["evidence_anchor"],
        "reviewer_notes":
            proposal["reviewer_notes"],
        "evidence_verified": True,
        "human_review_status": "PENDING",
    })

# ------------------------------------------------------------
# 3. Construct documented coding inventories
# ------------------------------------------------------------

iran_semantic_proposals_31e = pd.DataFrame(
    iran_records_31e
).sort_values(
    ["year", "review_id"]
).reset_index(drop=True)

iran_evidence_31e = pd.DataFrame(
    iran_evidence_records_31e
)

assert len(iran_semantic_proposals_31e) == 6
assert len(iran_evidence_31e) == 12

assert iran_semantic_proposals_31e[
    "review_id"
].is_unique

assert iran_evidence_31e["verified"].all()

assert iran_semantic_proposals_31e[
    "human_review_status"
].eq("PENDING").all()

assert (
    iran_semantic_proposals_31e.loc[
        iran_semantic_proposals_31e["review_id"].eq(
            "SV-X003"
        ),
        "source_integrity_status",
    ].iloc[0]
    == "PRIOR_FLAG_REQUIRES_REVIEW"
)

# ------------------------------------------------------------
# 4. Display provisional thematic classification
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31E — IRAN SEMANTIC ADJUDICATION")
print("=" * 65)

print("\nPROVISIONAL SOURCE-LEVEL CODING")

display(
    iran_semantic_proposals_31e[
        [
            "review_id",
            "year",
            "primary_geographic_focus",
            "principal_substantive_issue",
            "diplomatic_or_peace_process",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "human_review_status",
        ]
    ]
)

# ------------------------------------------------------------
# 5. Display independently traceable text anchors
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SOURCE-VERIFIED SUPPORTING EVIDENCE")
print("=" * 65)

for row in iran_evidence_31e.itertuples(index=False):

    print(
        f"\n{row.review_id} | {row.year} | "
        f"{row.anchor_type}"
    )

    print("Anchor:", row.source_anchor)
    print("Source context:", row.source_context)
    print("Verified:", row.verified)

# ------------------------------------------------------------
# 6. Summarize proposed topic-label fit
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PROVISIONAL TOPIC-LABEL FIT")
print("=" * 65)

display(
    iran_semantic_proposals_31e.groupby(
        [
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
        ],
        dropna=False,
    ).size().reset_index(
        name="n_review_records"
    )
)

# ------------------------------------------------------------
# 7. Integrity and final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SOURCE INTEGRITY REVIEW")
print("=" * 65)

display(
    iran_semantic_proposals_31e[
        [
            "review_id",
            "year",
            "source_character_count",
            "source_integrity_status",
        ]
    ]
)

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Iranian records evaluated:", 6)
print("Verified evidence anchors:", len(iran_evidence_31e))
print("Evidence verification: PASSED")
print("Provisional semantic coding: PREPARED")
print("Previously flagged source records:", 1)
print("Source integrity resolution: PENDING")
print("Human semantic confirmation: PENDING")
print("Original source data modified: NO")
print("Original NMF model modified: NO")
print("Statistical estimates modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31E: REVIEW PROPOSALS PREPARED")



### Step 12.27H.31F — United States Source-Level Semantic Adjudication

**Objective:** Evaluate the six US source segments across three historical periods to assess the semantic coherence of reference NMF Topic 03.

**Historical coverage:** 1972, 1975, 1995, 1997, 2011, and 2025.

**Methodology:**

1. Recover the six original cached source segments without modifying them.
2. Verify two source-text anchors for each segment.
3. Record provisional geographic and substantive classifications.
4. Distinguish diplomatic conflict-resolution discourse from UN peacekeeping, security alliances, and domestic political rhetoric.
5. Evaluate the narrow Kampuchea-specific label against the broader regional conflict and diplomacy label.
6. Preserve all interpretations as provisional pending human confirmation.

**Scientific safeguards:** Lexical evidence is not treated as automatic semantic validation. The six purposively selected segments cannot independently establish topic prevalence across the full corpus.

No statistical estimates, source data, fitted models, or publication files will be modified.


In [ ]:

# ============================================================
# STEP 12.27H.31F
# UNITED STATES SOURCE-LEVEL SEMANTIC ADJUDICATION
# ============================================================

assert "review_source_31" in globals()

required_columns_31f = {
    "review_id",
    "country_code",
    "year",
    "period",
    "segment_text",
}

assert required_columns_31f.issubset(
    review_source_31.columns
)

# ------------------------------------------------------------
# 1. Provisional semantic coding proposals
# ------------------------------------------------------------

us_coding_proposals_31f = [
    {
        "review_id": "SV-X007",
        "primary_geographic_focus":
            "Korean Peninsula, Middle East and international disputes",
        "principal_substantive_issue":
            "Negotiated settlement and reconciliation of interstate disputes",
        "diplomatic_or_peace_process": "Yes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - several regional disputes under a common diplomatic framework",
        "evidence_anchor":
            "When North Korea can talk to South Korea",
        "secondary_evidence_anchor":
            "Middle East should be no exception",
        "reviewer_notes":
            "Diplomatic negotiation is the dominant unifying theme.",
    },
    {
        "review_id": "SV-X008",
        "primary_geographic_focus":
            "Korean Peninsula",
        "principal_substantive_issue":
            "Korean armistice, diplomatic negotiations and conflict settlement",
        "diplomatic_or_peace_process": "Yes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Low - concentrated Korean Peninsula peace process",
        "evidence_anchor":
            "lasting solution of the Korean problem",
        "secondary_evidence_anchor":
            "Armistice Agreement",
        "reviewer_notes":
            "Direct substantive evidence of negotiated conflict resolution.",
    },
    {
        "review_id": "SV-X010",
        "primary_geographic_focus":
            "International - Korea, Persian Gulf and Haiti",
        "principal_substantive_issue":
            "United Nations peacekeeping and international security",
        "diplomatic_or_peace_process": "Partial",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_heterogeneity":
            "Moderate - UN institutional reform and peacekeeping mandates",
        "evidence_anchor":
            "From Korea to the Persian Gulf to Haiti",
        "secondary_evidence_anchor":
            "Blue Helmets",
        "reviewer_notes":
            "Peacekeeping and institutional mandates extend beyond narrow diplomatic settlement.",
    },
    {
        "review_id": "SV-X009",
        "primary_geographic_focus":
            "International - Middle East, Bosnia and other conflicts",
        "principal_substantive_issue":
            "Peacekeeping, conflict resolution and international cooperation",
        "diplomatic_or_peace_process": "Yes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - multiple peacekeeping and post-conflict settings",
        "evidence_anchor":
            "peace process in the Middle East",
        "secondary_evidence_anchor":
            "United Nations helicopter in Bosnia",
        "reviewer_notes":
            "Peacekeeping and conflict-resolution discourse across regions.",
    },
    {
        "review_id": "SV-X011",
        "primary_geographic_focus":
            "Palestine/Israel, Northern Ireland and Sudan",
        "principal_substantive_issue":
            "Negotiated peace, compromise and political settlement",
        "diplomatic_or_peace_process": "Yes",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Yes",
        "semantic_heterogeneity":
            "Moderate - multiple examples linked by negotiated settlement",
        "evidence_anchor":
            "Israelis and the Palestinians",
        "secondary_evidence_anchor":
            "peace depends upon compromise",
        "reviewer_notes":
            "Substantial consistency with diplomatic conflict settlement.",
    },
    {
        "review_id": "SV-X012",
        "primary_geographic_focus":
            "United States, Middle East and international alliances",
        "principal_substantive_issue":
            "Immigration, NATO, regional partnerships and trade",
        "diplomatic_or_peace_process": "Partial",
        "kampuchea_specific_label_fit": "No",
        "broad_conflict_diplomacy_fit": "Partial",
        "semantic_heterogeneity":
            "High - domestic policy, alliances, trade and international relations",
        "evidence_anchor":
            "immigration",
        "secondary_evidence_anchor":
            "NATO summit in June",
        "reviewer_notes":
            "Mixed domestic and foreign-policy discourse; broad topic only partially applicable.",
    },
]

# ------------------------------------------------------------
# 2. Recover source records and verify text evidence
# ------------------------------------------------------------

us_records_31f = []
us_evidence_records_31f = []

for proposal in us_coding_proposals_31f:

    review_id = proposal["review_id"]

    matched = review_source_31.loc[
        review_source_31["review_id"].eq(review_id)
    ]

    assert len(matched) == 1, (
        f"Missing or duplicated source: {review_id}"
    )

    source_row = matched.iloc[0]

    assert source_row["country_code"] == "USA"

    original_text = source_row["segment_text"]

    assert isinstance(original_text, str)
    assert original_text.strip()

    normalized_text = " ".join(original_text.split())

    # Two independent textual anchors per source.
    for anchor_type, anchor in [
        ("Primary", proposal["evidence_anchor"]),
        ("Secondary", proposal["secondary_evidence_anchor"]),
    ]:

        position = normalized_text.casefold().find(
            anchor.casefold()
        )

        verified = position >= 0

        if verified:
            context_start = max(0, position - 90)
            context_end = min(
                len(normalized_text),
                position + len(anchor) + 90,
            )
            context = normalized_text[
                context_start:context_end
            ]
        else:
            context = None

        us_evidence_records_31f.append({
            "review_id": review_id,
            "year": int(source_row["year"]),
            "anchor_type": anchor_type,
            "source_anchor": anchor,
            "source_context": context,
            "verified": verified,
        })

    us_records_31f.append({
        "review_id": review_id,
        "country_code": "USA",
        "year": int(source_row["year"]),
        "period": source_row["period"],
        "source_word_count": len(original_text.split()),
        "source_character_count": len(original_text),
        "primary_geographic_focus":
            proposal["primary_geographic_focus"],
        "principal_substantive_issue":
            proposal["principal_substantive_issue"],
        "diplomatic_or_peace_process":
            proposal["diplomatic_or_peace_process"],
        "kampuchea_specific_label_fit":
            proposal["kampuchea_specific_label_fit"],
        "broad_conflict_diplomacy_fit":
            proposal["broad_conflict_diplomacy_fit"],
        "semantic_heterogeneity":
            proposal["semantic_heterogeneity"],
        "supporting_text":
            proposal["evidence_anchor"],
        "reviewer_notes":
            proposal["reviewer_notes"],
        "human_review_status": "PENDING",
        "raw_file_verification":
            "NOT_INDEPENDENTLY_VERIFIED",
    })

# ------------------------------------------------------------
# 3. Construct reproducible audit tables
# ------------------------------------------------------------

us_semantic_proposals_31f = pd.DataFrame(
    us_records_31f
).sort_values(
    ["year", "review_id"]
).reset_index(drop=True)

us_evidence_31f = pd.DataFrame(
    us_evidence_records_31f
)

assert len(us_semantic_proposals_31f) == 6
assert len(us_evidence_31f) == 12

assert us_semantic_proposals_31f[
    "review_id"
].is_unique

assert us_semantic_proposals_31f[
    "year"
].nunique() == 6

assert set(us_semantic_proposals_31f["period"]) == {
    "1972-1990",
    "1991-2010",
    "2011-2025",
}

assert us_semantic_proposals_31f.groupby(
    "period"
).size().eq(2).all()

# ------------------------------------------------------------
# 4. Display provisional semantic classifications
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31F — US SEMANTIC ADJUDICATION")
print("=" * 65)

print("\nPROVISIONAL SOURCE-LEVEL CODING")

display(
    us_semantic_proposals_31f[
        [
            "review_id",
            "year",
            "primary_geographic_focus",
            "principal_substantive_issue",
            "diplomatic_or_peace_process",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "human_review_status",
        ]
    ]
)

# ------------------------------------------------------------
# 5. Display verifiable textual evidence
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SOURCE-VERIFIED SUPPORTING EVIDENCE")
print("=" * 65)

for row in us_evidence_31f.itertuples(index=False):

    print(
        f"\n{row.review_id} | {row.year} | "
        f"{row.anchor_type}"
    )

    print("Anchor:", row.source_anchor)
    print("Verified:", row.verified)

    if row.verified:
        print("Context:", row.source_context)
    else:
        print(
            "WARNING: Anchor not found in the cached source."
        )

# ------------------------------------------------------------
# 6. Historical thematic comparison
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("HISTORICAL SEMANTIC COMPARISON")
print("=" * 65)

display(
    us_semantic_proposals_31f[
        [
            "review_id",
            "period",
            "principal_substantive_issue",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
        ]
    ]
)

# ------------------------------------------------------------
# 7. Topic-label fit summary
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("PROVISIONAL TOPIC-LABEL FIT")
print("=" * 65)

us_label_fit_summary_31f = (
    us_semantic_proposals_31f.groupby(
        [
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
        ],
        dropna=False,
    )
    .size()
    .reset_index(name="n_review_records")
)

display(us_label_fit_summary_31f)

# ------------------------------------------------------------
# 8. Evidence completeness and final validation
# ------------------------------------------------------------

verified_count_31f = int(
    us_evidence_31f["verified"].sum()
)

failed_evidence_31f = us_evidence_31f.loc[
    ~us_evidence_31f["verified"]
].copy()

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("United States records evaluated:", 6)
print("Distinct historical years:", 6)
print("Historical periods:", 3)

print(
    "Verified source-text anchors:",
    f"{verified_count_31f}/12",
)

if failed_evidence_31f.empty:
    print("Cached-source evidence verification: PASSED")
else:
    print("Cached-source evidence verification: REQUIRES REVIEW")
    display(
        failed_evidence_31f[
            ["review_id", "anchor_type", "source_anchor"]
        ]
    )

print("Provisional semantic coding: PREPARED")
print("Independent raw-file verification: NOT PERFORMED")
print("Human semantic confirmation: PENDING")
print("Original source data modified: NO")
print("Original NMF model modified: NO")
print("Statistical estimates modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31F: REVIEW PROPOSALS PREPARED")



### Step 12.27H.31G — Cross-Country Semantic Integration Audit

**Objective:** Integrate the provisional qualitative coding of China, Iran, and the United States into a single traceable semantic review inventory for reference NMF Topic 03.

**Validation strategy:**
1. Identify the existing country-specific coding tables.
2. Reconcile all 18 review IDs against the verified source inventory.
3. Check country and historical-period coverage.
4. Compare narrow Kampuchea-specific and broad conflict-diplomacy label fit.
5. Document source-integrity qualifications separately from semantic interpretation.
6. Preserve human confirmation as pending until an explicit review is completed.

**Limitations:** These purposively selected segments support qualitative interpretation but do not constitute a representative estimate of corpus-wide thematic prevalence.

**Reproducibility:** No source records, NMF components, numerical estimates, or publication files are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31G
# CROSS-COUNTRY SEMANTIC INTEGRATION AUDIT
# ============================================================

assert "review_source_31" in globals()
assert "us_semantic_proposals_31f" in globals()

EXPECTED_IDS_31G = {
    "CHN": {f"SV-{i:03d}" for i in range(13, 19)},
    "IRN": {f"SV-X{i:03d}" for i in range(1, 7)},
    "USA": {f"SV-X{i:03d}" for i in range(7, 13)},
}

REQUIRED_CODING_31G = {
    "review_id",
    "primary_geographic_focus",
    "principal_substantive_issue",
    "kampuchea_specific_label_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
}

REQUIRED_SOURCE_31G = {
    "review_id",
    "country_code",
    "year",
    "period",
    "topic03_share_recovered",
    "segment_text",
}

assert REQUIRED_SOURCE_31G.issubset(
    review_source_31.columns
)

source_master_31g = review_source_31.copy(deep=True)

assert len(source_master_31g) == 18
assert source_master_31g["review_id"].is_unique

# ------------------------------------------------------------
# 1. Discover previously prepared coding proposals
# ------------------------------------------------------------

def find_country_coding_31g(country_code):

    expected_ids = EXPECTED_IDS_31G[country_code]
    candidates = []

    preferred_step = {
        "CHN": "31d",
        "IRN": "31e",
        "USA": "31f",
    }[country_code]

    for name, value in list(globals().items()):

        if not isinstance(value, pd.DataFrame):
            continue

        if not REQUIRED_CODING_31G.issubset(value.columns):
            continue

        if not value["review_id"].is_unique:
            continue

        matched = value.loc[
            value["review_id"].astype(str).isin(expected_ids)
        ].copy()

        if len(matched) != 6:
            continue

        if set(matched["review_id"].astype(str)) != expected_ids:
            continue

        content_columns = list(
            REQUIRED_CODING_31G - {"review_id"}
        )

        if matched[content_columns].isna().any().any():
            continue

        if "country_code" in matched.columns:
            if not matched["country_code"].eq(
                country_code
            ).all():
                continue

        score = 0

        if len(value) == 6:
            score += 100

        if preferred_step in name.lower():
            score += 50

        if any(
            word in name.lower()
            for word in ("semantic", "coding", "proposal")
        ):
            score += 20

        candidates.append(
            (score, name, matched)
        )

    if not candidates:
        raise RuntimeError(
            f"No complete coding table found for {country_code}. "
            "Check the corresponding country-level coding step."
        )

    candidates.sort(
        key=lambda item: (-item[0], item[1])
    )

    _, selected_name, selected_df = candidates[0]

    return selected_name, selected_df.copy(deep=True)


country_coding_parts_31g = []
selected_sources_31g = []

for country in ("CHN", "IRN", "USA"):

    selected_name, selected_df = (
        find_country_coding_31g(country)
    )

    selected_sources_31g.append({
        "country_code": country,
        "coding_object": selected_name,
        "records": len(selected_df),
    })

    coding_columns = list(
        REQUIRED_CODING_31G
    )

    if "human_review_status" in selected_df.columns:
        coding_columns.append("human_review_status")

    selected_df = selected_df[
        coding_columns
    ].copy()

    if "human_review_status" not in selected_df.columns:
        selected_df["human_review_status"] = "PENDING"

    country_coding_parts_31g.append(selected_df)

coding_combined_31g = pd.concat(
    country_coding_parts_31g,
    ignore_index=True,
)

assert len(coding_combined_31g) == 18
assert coding_combined_31g["review_id"].is_unique

# ------------------------------------------------------------
# 2. Reconcile coding with original source metadata
# ------------------------------------------------------------

source_columns_31g = [
    "review_id",
    "country_code",
    "year",
    "period",
    "topic03_share_recovered",
]

semantic_master_proposals_31g = (
    source_master_31g[source_columns_31g]
    .merge(
        coding_combined_31g,
        on="review_id",
        how="left",
        validate="one_to_one",
        indicator=True,
    )
)

assert semantic_master_proposals_31g[
    "_merge"
].eq("both").all()

semantic_master_proposals_31g = (
    semantic_master_proposals_31g
    .drop(columns="_merge")
    .sort_values(["country_code", "year"])
    .reset_index(drop=True)
)

assert len(semantic_master_proposals_31g) == 18

assert semantic_master_proposals_31g.groupby(
    "country_code"
).size().eq(6).all()

assert (
    semantic_master_proposals_31g.groupby(
        ["country_code", "period"]
    ).size().eq(2).all()
)

# ------------------------------------------------------------
# 3. Preserve known source-integrity qualifications
# ------------------------------------------------------------

semantic_master_proposals_31g[
    "source_integrity_note"
] = "Cached source recovered; independent raw-file check pending"

semantic_master_proposals_31g.loc[
    semantic_master_proposals_31g["review_id"].eq("SV-X003"),
    "source_integrity_note",
] = "Previously flagged source discontinuity; unresolved"

semantic_master_proposals_31g.loc[
    semantic_master_proposals_31g["review_id"].eq("SV-015"),
    "source_integrity_note",
] = "Carriage returns identified; exact speech-source alignment verified"

# ------------------------------------------------------------
# 4. Verify original coding table discovery
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31G — CROSS-COUNTRY SEMANTIC AUDIT")
print("=" * 65)

print("\nCODING TABLES IDENTIFIED")
display(pd.DataFrame(selected_sources_31g))

# ------------------------------------------------------------
# 5. Evaluate country-period completeness
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("COUNTRY-PERIOD REVIEW COVERAGE")
print("=" * 65)

coverage_31g = (
    semantic_master_proposals_31g
    .groupby(
        ["country_code", "period"],
        as_index=False,
    )
    .agg(
        n_records=("review_id", "size"),
        n_years=("year", "nunique"),
        mean_sampled_topic_share=(
            "topic03_share_recovered",
            "mean",
        ),
    )
)

display(coverage_31g.round(5))

# ------------------------------------------------------------
# 6. Compare semantic label fit across countries
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("BROAD TOPIC-LABEL FIT BY COUNTRY")
print("=" * 65)

broad_fit_31g = (
    semantic_master_proposals_31g
    .groupby(
        ["country_code", "broad_conflict_diplomacy_fit"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=["Yes", "Partial", "No"],
        fill_value=0,
    )
)

display(broad_fit_31g)

print("\nNARROW KAMPUCHEA LABEL FIT BY COUNTRY")

narrow_fit_31g = (
    semantic_master_proposals_31g
    .groupby(
        ["country_code", "kampuchea_specific_label_fit"]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        columns=["Yes", "Partial", "No"],
        fill_value=0,
    )
)

display(narrow_fit_31g)

# ------------------------------------------------------------
# 7. Show the integrated qualitative inventory
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("INTEGRATED SEMANTIC REVIEW INVENTORY")
print("=" * 65)

display(
    semantic_master_proposals_31g[
        [
            "review_id",
            "country_code",
            "year",
            "principal_substantive_issue",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "human_review_status",
        ]
    ]
)

# ------------------------------------------------------------
# 8. Show source-integrity exceptions
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("SOURCE-INTEGRITY QUALIFICATIONS")
print("=" * 65)

display(
    semantic_master_proposals_31g.loc[
        semantic_master_proposals_31g[
            "review_id"
        ].isin(["SV-X003", "SV-015"]),
        [
            "review_id",
            "country_code",
            "year",
            "source_integrity_note",
        ],
    ]
)

# ------------------------------------------------------------
# 9. Final scientific validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL SEMANTIC INTEGRATION DIAGNOSTICS")
print("=" * 65)

print(
    "Total integrated records:",
    len(semantic_master_proposals_31g),
)

print(
    "Countries:",
    semantic_master_proposals_31g[
        "country_code"
    ].nunique(),
)

print(
    "Country-period strata:",
    len(coverage_31g),
)

print(
    "Broad-label full-fit proposals:",
    int(
        semantic_master_proposals_31g[
            "broad_conflict_diplomacy_fit"
        ].eq("Yes").sum()
    ),
)

print(
    "Broad-label partial-fit proposals:",
    int(
        semantic_master_proposals_31g[
            "broad_conflict_diplomacy_fit"
        ].eq("Partial").sum()
    ),
)

print(
    "Human-pending records:",
    int(
        semantic_master_proposals_31g[
            "human_review_status"
        ].eq("PENDING").sum()
    ),
)

print("\nSource metadata reconciliation: PASSED")
print("Historical sampling coverage: PASSED")
print("Semantic coding integration: PREPARED")
print("Final human adjudication: PENDING")
print("Source-integrity exceptions: RETAINED")
print("Corpus-wide semantic prevalence: NOT ESTIMATED")
print("Original NMF models modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31G: INTEGRATION COMPLETED")



### Step 12.27H.31H — Semantic Decision Readiness and Sensitivity Audit

**Objective:** Assess whether the integrated source-level evidence supports a defensible provisional label for reference NMF Topic 03.

**Analytical tasks:**
1. Verify the completeness and consistency of all 18 source-linked coding records.
2. Quantify full, partial, and absent label fit in the purposive review sample.
3. Flag records requiring particular attention because of partial semantic fit or source-integrity qualifications.
4. Evaluate the descriptive sensitivity of the results to excluding the previously flagged Iranian source.
5. Prepare an auditable review table without automatically converting provisional judgments into human-confirmed decisions.

**Interpretive safeguards:** Qualitative proportions describe only the selected review sample. They are not population estimates, statistical validation metrics, or evidence of topic prevalence across the full corpus.

**Preservation policy:** No original texts, fitted NMF models, statistical estimates, or publication files are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31H
# SEMANTIC DECISION READINESS AND SENSITIVITY AUDIT
# ============================================================

assert "semantic_master_proposals_31g" in globals()
assert "review_source_31" in globals()

required_columns_31h = {
    "review_id",
    "country_code",
    "year",
    "period",
    "topic03_share_recovered",
    "primary_geographic_focus",
    "principal_substantive_issue",
    "kampuchea_specific_label_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "human_review_status",
    "source_integrity_note",
}

assert required_columns_31h.issubset(
    semantic_master_proposals_31g.columns
)

assert {
    "review_id",
    "segment_text",
}.issubset(review_source_31.columns)

# ------------------------------------------------------------
# 1. Construct a source-linked working audit table
# ------------------------------------------------------------

semantic_review_31h = (
    semantic_master_proposals_31g
    .copy(deep=True)
    .merge(
        review_source_31[
            ["review_id", "segment_text"]
        ],
        on="review_id",
        how="left",
        validate="one_to_one",
    )
    .sort_values(
        ["country_code", "year"]
    )
    .reset_index(drop=True)
)

assert len(semantic_review_31h) == 18
assert semantic_review_31h["review_id"].is_unique
assert semantic_review_31h["segment_text"].notna().all()
assert semantic_review_31h[
    "segment_text"
].astype(str).str.len().gt(0).all()

fit_values_31h = {"Yes", "Partial", "No"}

for column in [
    "kampuchea_specific_label_fit",
    "broad_conflict_diplomacy_fit",
]:
    assert set(
        semantic_review_31h[column].dropna()
    ).issubset(fit_values_31h)

assert semantic_review_31h[
    "human_review_status"
].eq("PENDING").all()

# ------------------------------------------------------------
# 2. Identify cases requiring priority review
# ------------------------------------------------------------

SOURCE_EXCEPTIONS_31H = {
    "SV-X003",
    "SV-015",
}

semantic_review_31h[
    "partial_broad_fit"
] = semantic_review_31h[
    "broad_conflict_diplomacy_fit"
].eq("Partial")

semantic_review_31h[
    "source_exception"
] = semantic_review_31h[
    "review_id"
].isin(SOURCE_EXCEPTIONS_31H)

semantic_review_31h[
    "priority_review"
] = (
    semantic_review_31h["partial_broad_fit"]
    | semantic_review_31h["source_exception"]
)

def review_reason_31h(row):
    reasons = []

    if row["partial_broad_fit"]:
        reasons.append("Partial broad-label fit")

    if row["source_exception"]:
        reasons.append("Source-integrity qualification")

    if not reasons:
        return "Standard review"

    return "; ".join(reasons)


semantic_review_31h[
    "review_reason"
] = semantic_review_31h.apply(
    review_reason_31h,
    axis=1,
)

priority_review_31h = (
    semantic_review_31h.loc[
        semantic_review_31h["priority_review"]
    ]
    .copy()
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 3. Summarize label fit descriptively
# ------------------------------------------------------------

def summarize_fit_31h(frame, scenario):

    n = len(frame)

    broad_counts = frame[
        "broad_conflict_diplomacy_fit"
    ].value_counts()

    narrow_counts = frame[
        "kampuchea_specific_label_fit"
    ].value_counts()

    return {
        "scenario": scenario,
        "n_segments": n,
        "broad_yes": int(broad_counts.get("Yes", 0)),
        "broad_partial": int(
            broad_counts.get("Partial", 0)
        ),
        "broad_no": int(broad_counts.get("No", 0)),
        "broad_full_fit_pct": (
            100 * broad_counts.get("Yes", 0) / n
        ),
        "narrow_yes": int(
            narrow_counts.get("Yes", 0)
        ),
        "narrow_partial": int(
            narrow_counts.get("Partial", 0)
        ),
        "narrow_no": int(
            narrow_counts.get("No", 0)
        ),
    }


descriptive_sensitivity_31h = pd.DataFrame([
    summarize_fit_31h(
        semantic_review_31h,
        "All reviewed segments",
    ),
    summarize_fit_31h(
        semantic_review_31h.loc[
            semantic_review_31h["review_id"]
            != "SV-X003"
        ],
        "Exclude flagged Iranian segment",
    ),
    summarize_fit_31h(
        semantic_review_31h.loc[
            ~semantic_review_31h[
                "source_exception"
            ]
        ],
        "Exclude both source-qualified segments",
    ),
])

# ------------------------------------------------------------
# 4. Country and historical-period interpretation
# ------------------------------------------------------------

country_period_fit_31h = (
    semantic_review_31h
    .groupby(
        ["country_code", "period"],
        as_index=False,
    )
    .agg(
        n_segments=("review_id", "size"),
        broad_yes=(
            "broad_conflict_diplomacy_fit",
            lambda s: int(s.eq("Yes").sum()),
        ),
        broad_partial=(
            "broad_conflict_diplomacy_fit",
            lambda s: int(s.eq("Partial").sum()),
        ),
        broad_no=(
            "broad_conflict_diplomacy_fit",
            lambda s: int(s.eq("No").sum()),
        ),
    )
)

assert len(country_period_fit_31h) == 9
assert country_period_fit_31h[
    "n_segments"
].eq(2).all()

# ------------------------------------------------------------
# 5. Report the completed diagnostics
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31H — SEMANTIC DECISION READINESS")
print("=" * 65)

print("\n" + "=" * 65)
print("DESCRIPTIVE LABEL-FIT SENSITIVITY")
print("=" * 65)

display(descriptive_sensitivity_31h.round(2))

print("\n" + "=" * 65)
print("COUNTRY-PERIOD SEMANTIC FIT")
print("=" * 65)

display(country_period_fit_31h)

print("\n" + "=" * 65)
print("PRIORITY RECORDS FOR FINAL ADJUDICATION")
print("=" * 65)

display(
    priority_review_31h[
        [
            "review_id",
            "country_code",
            "year",
            "principal_substantive_issue",
            "broad_conflict_diplomacy_fit",
            "review_reason",
            "human_review_status",
        ]
    ]
)

print("\n" + "=" * 65)
print("SOURCE-INTEGRITY QUALIFICATIONS")
print("=" * 65)

display(
    semantic_review_31h.loc[
        semantic_review_31h["source_exception"],
        [
            "review_id",
            "country_code",
            "year",
            "source_integrity_note",
        ],
    ]
)

# ------------------------------------------------------------
# 6. Preserve an auditable provisional publication table
# ------------------------------------------------------------

publication_semantic_audit_31h = (
    semantic_review_31h[
        [
            "review_id",
            "country_code",
            "year",
            "period",
            "topic03_share_recovered",
            "primary_geographic_focus",
            "principal_substantive_issue",
            "kampuchea_specific_label_fit",
            "broad_conflict_diplomacy_fit",
            "semantic_heterogeneity",
            "source_integrity_note",
            "review_reason",
            "human_review_status",
        ]
    ]
    .copy(deep=True)
)

print("\n" + "=" * 65)
print("FINAL DIAGNOSTICS")
print("=" * 65)

print("Source-linked records:", len(semantic_review_31h))
print("Historical strata:", len(country_period_fit_31h))
print("Priority review records:", len(priority_review_31h))

print(
    "Human-confirmed records:",
    int(
        semantic_review_31h[
            "human_review_status"
        ].eq("CONFIRMED").sum()
    ),
)

print("\nSource-record linkage: PASSED")
print("Semantic consistency screening: COMPLETED")
print("Descriptive sensitivity audit: COMPLETED")
print("Final human adjudication: PENDING")
print("Final publication label: NOT YET LOCKED")
print("Original NMF models modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31H: DIAGNOSTIC COMPLETED")



### Step 12.27H.31I — Targeted Source Verification and Human Adjudication Register

**Objective:** Prepare a reproducible final semantic review register for all 18 selected Topic 03 segments.

This step will:

1. Recheck the two source-qualified segments against available speech-level texts.
2. Distinguish exact source correspondence from independent archival verification.
3. Identify the eight priority adjudication cases.
4. Construct a source-linked decision register for all 18 records.
5. Preserve provisional semantic judgments separately from human-confirmed decisions.

No human confirmation is inferred from automated text matching.

No original source data, NMF models, statistical results, or publication files will be modified.


In [ ]:

# ============================================================
# STEP 12.27H.31I
# TARGETED SOURCE VERIFICATION AND ADJUDICATION REGISTER
# ============================================================

assert "semantic_review_31h" in globals()
assert "df_nlp" in globals()

required_source_columns_31i = {
    "country_code",
    "year",
    "text_raw",
}

assert required_source_columns_31i.issubset(df_nlp.columns)

assert {
    "review_id",
    "country_code",
    "year",
    "segment_text",
    "priority_review",
    "broad_conflict_diplomacy_fit",
    "human_review_status",
}.issubset(semantic_review_31h.columns)

assert len(semantic_review_31h) == 18
assert semantic_review_31h["review_id"].is_unique

# ------------------------------------------------------------
# 1. Verify the two source-qualified records
# ------------------------------------------------------------

source_check_ids_31i = ["SV-015", "SV-X003"]

source_check_records_31i = []

text_columns_31i = [
    col for col in ["text_raw", "text_clean"]
    if col in df_nlp.columns
]

for review_id in source_check_ids_31i:

    record = semantic_review_31h.loc[
        semantic_review_31h["review_id"] == review_id
    ]

    assert len(record) == 1

    record = record.iloc[0]

    cached_text = str(record["segment_text"])

    matches = df_nlp.loc[
        (df_nlp["country_code"] == record["country_code"])
        & (df_nlp["year"] == record["year"])
    ]

    exact_locations = []

    for source_index, speech in matches.iterrows():
        for text_column in text_columns_31i:

            if pd.isna(speech[text_column]):
                continue

            full_text = str(speech[text_column])
            start = full_text.find(cached_text)

            if start >= 0:
                exact_locations.append({
                    "source_index": str(source_index),
                    "text_column": text_column,
                    "start": start,
                    "end": start + len(cached_text),
                })

    source_check_records_31i.append({
        "review_id": review_id,
        "country_code": record["country_code"],
        "year": int(record["year"]),
        "candidate_speeches": len(matches),
        "cached_characters": len(cached_text),
        "exact_speech_match": len(exact_locations) > 0,
        "matching_text_columns": ", ".join(
            sorted(set(
                location["text_column"]
                for location in exact_locations
            ))
        ),
        "carriage_returns": cached_text.count("\r"),
        "replacement_characters": cached_text.count("\ufffd"),
        "source_integrity_note": record.get(
            "source_integrity_note", pd.NA
        ),
    })

source_verification_31i = pd.DataFrame(
    source_check_records_31i
)

# ------------------------------------------------------------
# 2. Build a separate human decision register
# ------------------------------------------------------------

register_columns_31i = [
    "review_id",
    "country_code",
    "year",
    "period",
    "topic03_share_recovered",
    "primary_geographic_focus",
    "principal_substantive_issue",
    "kampuchea_specific_label_fit",
    "broad_conflict_diplomacy_fit",
    "semantic_heterogeneity",
    "source_integrity_note",
    "review_reason",
    "priority_review",
]

human_adjudication_register_31i = (
    semantic_review_31h[
        register_columns_31i
    ]
    .copy(deep=True)
)

human_adjudication_register_31i[
    "reviewer_broad_fit"
] = pd.Series(
    pd.NA,
    index=human_adjudication_register_31i.index,
    dtype="string",
)

human_adjudication_register_31i[
    "reviewer_source_decision"
] = pd.Series(
    pd.NA,
    index=human_adjudication_register_31i.index,
    dtype="string",
)

human_adjudication_register_31i[
    "reviewer_justification"
] = pd.Series(
    pd.NA,
    index=human_adjudication_register_31i.index,
    dtype="string",
)

human_adjudication_register_31i[
    "human_review_status"
] = "PENDING"

# ------------------------------------------------------------
# 3. Prepare priority evidence for adjudication
# ------------------------------------------------------------

priority_cases_31i = (
    semantic_review_31h.loc[
        semantic_review_31h["priority_review"]
    ]
    .copy()
    .reset_index(drop=True)
)

priority_cases_31i["source_preview"] = (
    priority_cases_31i["segment_text"]
    .astype(str)
    .map(lambda text: " ".join(text.split())[:320])
)

assert len(priority_cases_31i) == 8

# ------------------------------------------------------------
# 4. Report the verification and decision status
# ------------------------------------------------------------

print("=" * 65)
print("STEP 12.27H.31I — FINAL SEMANTIC REVIEW PREPARATION")
print("=" * 65)

print("\n" + "=" * 65)
print("TARGETED SPEECH-LEVEL SOURCE VERIFICATION")
print("=" * 65)

display(source_verification_31i)

print("\n" + "=" * 65)
print("EIGHT PRIORITY ADJUDICATION RECORDS")
print("=" * 65)

display(
    priority_cases_31i[
        [
            "review_id",
            "country_code",
            "year",
            "broad_conflict_diplomacy_fit",
            "review_reason",
            "source_preview",
        ]
    ]
)

print("\n" + "=" * 65)
print("HUMAN ADJUDICATION REGISTER")
print("=" * 65)

print("Total records:", len(human_adjudication_register_31i))
print(
    "Priority records:",
    int(
        human_adjudication_register_31i[
            "priority_review"
        ].sum()
    ),
)

print(
    "Human-confirmed records:",
    int(
        human_adjudication_register_31i[
            "human_review_status"
        ].eq("CONFIRMED").sum()
    ),
)

print("\nProvisional broad-label distribution:")

display(
    human_adjudication_register_31i[
        "broad_conflict_diplomacy_fit"
    ]
    .value_counts()
    .rename_axis("provisional_fit")
    .reset_index(name="n_records")
)

print("\n" + "=" * 65)
print("FINAL VALIDATION")
print("=" * 65)

print("Source records:", len(human_adjudication_register_31i))
print("Source exceptions examined:", len(source_verification_31i))
print("Adjudication register: PREPARED")
print("Human adjudication: PENDING")
print("Independent archival verification: NOT PERFORMED")
print("Final Topic 03 label: NOT YET LOCKED")
print("Original source data modified: NO")
print("Original NMF models modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31I: DIAGNOSTIC COMPLETED")



### Step 12.27H.31J — Researcher-Confirmed Semantic Adjudication

**Objective:** Record the researcher's explicit approval of the 18 source-level Topic 03 semantic-fit classifications.

The reference NMF Topic 03 is described as **Regional conflicts and diplomatic settlement**. Twelve sampled segments were provisionally classified as fully consistent with this broad label, while six exhibited partial semantic fit.

This step:

1. Reconciles the three country-specific coding registers.
2. Verifies the approved 12/6 classification against individual source records.
3. Records the researcher's confirmation in a separate auditable dataset.
4. Preserves source-integrity qualifications for the two previously flagged 1991 records.
5. Produces descriptive summaries for subsequent manuscript preparation.

**Interpretive limitations:** The sample is a purposively structured semantic-validation sample, not an estimate of corpus-wide topic-label prevalence. Researcher confirmation is not independent intercoder validation. Exact correspondence with cached speech texts does not constitute independent archival verification.

The original source data, fitted NMF models, topic contributions, statistical estimates, and existing publication tables remain unchanged.


In [ ]:

# ============================================================
# STEP 12.27H.31J
# RESEARCHER-CONFIRMED SEMANTIC ADJUDICATION
# ============================================================

print("=" * 68)
print("STEP 12.27H.31J — SEMANTIC ADJUDICATION")
print("=" * 68)

# ------------------------------------------------------------
# 1. Recover the country-specific proposal registers
# ------------------------------------------------------------

source_registers_31j = {
    "CHN": "china_semantic_proposals_all_31d",
    "IRN": "iran_semantic_proposals_31e",
    "USA": "us_semantic_proposals_31f",
}

frames_31j = []

for country, object_name in source_registers_31j.items():

    assert object_name in globals(), (
        f"Missing source register: {object_name}"
    )

    frame = globals()[object_name].copy(deep=True)

    required = {
        "review_id",
        "year",
        "kampuchea_specific_label_fit",
        "broad_conflict_diplomacy_fit",
    }

    missing = required.difference(frame.columns)

    assert not missing, (
        f"{object_name} is missing columns: {missing}"
    )

    if "country_code" in frame.columns:
        assert frame["country_code"].astype(
            str
        ).str.upper().eq(country).all()
    else:
        frame["country_code"] = country

    assert len(frame) == 6, (
        f"Unexpected record count for {country}"
    )

    frames_31j.append(frame)


semantic_adjudication_final_31j = pd.concat(
    frames_31j,
    ignore_index=True,
)

assert len(semantic_adjudication_final_31j) == 18

assert (
    semantic_adjudication_final_31j["review_id"]
    .is_unique
)

# ------------------------------------------------------------
# 2. Validate the researcher-approved classifications
# ------------------------------------------------------------

approved_partial_ids_31j = {
    "SV-018",
    "SV-X002",
    "SV-X006",
    "SV-X005",
    "SV-X010",
    "SV-X012",
}

observed_partial_ids_31j = set(
    semantic_adjudication_final_31j.loc[
        semantic_adjudication_final_31j[
            "broad_conflict_diplomacy_fit"
        ].eq("Partial"),
        "review_id",
    ]
)

assert observed_partial_ids_31j == approved_partial_ids_31j, (
    "The source-level partial-fit classifications "
    "do not match the researcher-approved records."
)

fit_counts_31j = (
    semantic_adjudication_final_31j[
        "broad_conflict_diplomacy_fit"
    ]
    .value_counts()
)

assert fit_counts_31j.get("Yes", 0) == 12
assert fit_counts_31j.get("Partial", 0) == 6
assert fit_counts_31j.get("No", 0) == 0

assert fit_counts_31j.sum() == 18

# ------------------------------------------------------------
# 3. Record researcher confirmation without modifying
#    the original proposal registers
# ------------------------------------------------------------

if "human_review_status" in (
    semantic_adjudication_final_31j.columns
):
    semantic_adjudication_final_31j[
        "previous_review_status_31j"
    ] = semantic_adjudication_final_31j[
        "human_review_status"
    ].copy()

semantic_adjudication_final_31j[
    "human_review_status"
] = "RESEARCHER_CONFIRMED"

semantic_adjudication_final_31j[
    "confirmation_basis"
] = "Explicit researcher approval"

semantic_adjudication_final_31j[
    "confirmation_date"
] = "2026-10-16"

semantic_adjudication_final_31j[
    "independent_second_coder"
] = False

semantic_adjudication_final_31j[
    "topic03_publication_label"
] = "Regional conflicts and diplomatic settlement"

semantic_adjudication_final_31j[
    "label_scope"
] = "Broad semantic descriptor; heterogeneous subthemes"

# ------------------------------------------------------------
# 4. Preserve unresolved source-integrity qualifications
# ------------------------------------------------------------

source_exceptions_31j = {
    "SV-015": (
        "Exact cached speech correspondence established; "
        "carriage returns identified; independent archival "
        "verification remains pending."
    ),
    "SV-X003": (
        "Cached speech correspondence established; "
        "previously flagged source discontinuity requires "
        "independent archival verification."
    ),
}

semantic_adjudication_final_31j[
    "source_integrity_qualification"
] = semantic_adjudication_final_31j[
    "review_id"
].map(source_exceptions_31j).fillna(
    "No specific exception retained in current audit; "
    "independent archival verification not established."
)

semantic_adjudication_final_31j[
    "independent_archival_verification"
] = "NOT ESTABLISHED"

semantic_adjudication_final_31j[
    "source_exception_flag"
] = semantic_adjudication_final_31j[
    "review_id"
].isin(source_exceptions_31j)

assert (
    semantic_adjudication_final_31j[
        "source_exception_flag"
    ].sum() == 2
)

# ------------------------------------------------------------
# 5. Country-level confirmation summary
# ------------------------------------------------------------

semantic_summary_31j = (
    semantic_adjudication_final_31j
    .groupby(
        [
            "country_code",
            "broad_conflict_diplomacy_fit",
        ]
    )
    .size()
    .unstack(fill_value=0)
    .reindex(
        index=["CHN", "IRN", "USA"],
        columns=["Yes", "Partial", "No"],
        fill_value=0,
    )
)

semantic_summary_31j["Total"] = (
    semantic_summary_31j[
        ["Yes", "Partial", "No"]
    ].sum(axis=1)
)

assert semantic_summary_31j["Total"].eq(6).all()

expected_by_country_31j = {
    "CHN": (5, 1, 0),
    "IRN": (3, 3, 0),
    "USA": (4, 2, 0),
}

for country, expected in expected_by_country_31j.items():

    observed = tuple(
        semantic_summary_31j.loc[
            country, ["Yes", "Partial", "No"]
        ].astype(int)
    )

    assert observed == expected, (
        f"Country-level classification mismatch: {country}"
    )

print("\n" + "=" * 68)
print("RESEARCHER-CONFIRMED SEMANTIC FIT")
print("=" * 68)

display(semantic_summary_31j)

# ------------------------------------------------------------
# 6. Produce an auditable review register
# ------------------------------------------------------------

audit_columns_31j = [
    "review_id",
    "country_code",
    "year",
    "principal_substantive_issue",
    "kampuchea_specific_label_fit",
    "broad_conflict_diplomacy_fit",
    "human_review_status",
    "source_exception_flag",
]

available_columns_31j = [
    col for col in audit_columns_31j
    if col in semantic_adjudication_final_31j.columns
]

semantic_audit_register_31j = (
    semantic_adjudication_final_31j[
        available_columns_31j
    ]
    .sort_values(
        ["country_code", "year", "review_id"]
    )
    .reset_index(drop=True)
)

print("\n" + "=" * 68)
print("CONFIRMED SOURCE-LEVEL REVIEW REGISTER")
print("=" * 68)

display(semantic_audit_register_31j)

# ------------------------------------------------------------
# 7. Report source-integrity exceptions
# ------------------------------------------------------------

source_exception_register_31j = (
    semantic_adjudication_final_31j.loc[
        semantic_adjudication_final_31j[
            "source_exception_flag"
        ],
        [
            "review_id",
            "country_code",
            "year",
            "source_integrity_qualification",
            "independent_archival_verification",
        ],
    ]
    .reset_index(drop=True)
)

print("\n" + "=" * 68)
print("RETAINED SOURCE-INTEGRITY QUALIFICATIONS")
print("=" * 68)

display(source_exception_register_31j)

# ------------------------------------------------------------
# 8. Final reproducibility and interpretation checks
# ------------------------------------------------------------

assert semantic_adjudication_final_31j[
    "human_review_status"
].eq("RESEARCHER_CONFIRMED").all()

assert semantic_adjudication_final_31j[
    "independent_second_coder"
].eq(False).all()

assert semantic_adjudication_final_31j[
    "topic03_publication_label"
].nunique() == 1

print("\n" + "=" * 68)
print("FINAL SEMANTIC ADJUDICATION DIAGNOSTICS")
print("=" * 68)

print("Total source records:", len(
    semantic_adjudication_final_31j
))

print("Researcher-confirmed records: 18")
print("Broad-label full fit:", 12)
print("Broad-label partial fit:", 6)
print("Broad-label no fit:", 0)

print(
    "Full-fit percentage in reviewed sample:",
    round(100 * 12 / 18, 2)
)

print("Retained source exceptions:", 2)

print("\nResearcher fit confirmation: RECORDED")
print("Independent second-coder validation: NOT PERFORMED")
print("Independent archival validation: NOT ESTABLISHED")
print("Corpus-wide semantic prevalence: NOT ESTIMATED")

print("\nOriginal NMF model modified: NO")
print("Original statistical estimates modified: NO")
print("Original proposal registers modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31J: COMPUTATION COMPLETED")



### Step 12.27H.31K — Publication Results and Semantic Label Reconciliation

**Objective:** Reconcile the researcher-approved Topic 03 semantic interpretation with the five primary findings and eighteen supplementary statistical contrasts.

This step verifies:

1. Complete country-period coverage of the 18 reviewed source segments.
2. Consistency between the primary and supplementary statistical estimates.
3. Correct mapping of reference NMF topic identifiers to semantic labels.
4. The impact of the revised Topic 03 label on publication tables.
5. Remaining source-integrity and semantic-validation limitations.

The reference Topic 03 label is **Regional conflicts and diplomatic settlement**.

This label is supported by source-level qualitative evidence but remains subject to documented semantic heterogeneity and archival qualifications.

All publication previews are produced as independent DataFrame copies. Original estimates, fitted models, and existing manuscript results remain unchanged.

No files or figures are saved in this step.


In [ ]:

# ============================================================
# STEP 12.27H.31K
# PUBLICATION RESULTS AND SEMANTIC LABEL RECONCILIATION
# ============================================================

print("=" * 68)
print("STEP 12.27H.31K — PUBLICATION RECONCILIATION")
print("=" * 68)

# ------------------------------------------------------------
# 1. Verify required analytical objects
# ------------------------------------------------------------

required_objects_31k = [
    "semantic_adjudication_final_31j",
    "paper_results_main5",
    "paper_results_all18",
]

for object_name in required_objects_31k:
    assert object_name in globals(), (
        f"Required object missing: {object_name}"
    )

review_31k = semantic_adjudication_final_31j.copy(deep=True)
main_31k = paper_results_main5.copy(deep=True)
supplement_31k = paper_results_all18.copy(deep=True)

assert len(review_31k) == 18
assert len(main_31k) == 5
assert len(supplement_31k) == 18

assert review_31k["review_id"].is_unique

assert review_31k["human_review_status"].eq(
    "RESEARCHER_CONFIRMED"
).all()

print("\nINPUT DATA VALIDATION: PASSED")

# ------------------------------------------------------------
# 2. Verify country-period sampling coverage
# ------------------------------------------------------------

year_31k = review_31k["year"].astype(int)

review_31k["verified_period_31k"] = np.select(
    [
        year_31k.between(1972, 1990),
        year_31k.between(1991, 2010),
        year_31k.between(2011, 2025),
    ],
    [
        "1972-1990",
        "1991-2010",
        "2011-2025",
    ],
    default="OUTSIDE_STUDY_PERIOD",
)

assert not review_31k[
    "verified_period_31k"
].eq("OUTSIDE_STUDY_PERIOD").any()

coverage_31k = (
    review_31k
    .groupby(
        ["country_code", "verified_period_31k"]
    )
    .agg(
        n_segments=("review_id", "nunique"),
        full_fit=(
            "broad_conflict_diplomacy_fit",
            lambda x: int(x.eq("Yes").sum()),
        ),
        partial_fit=(
            "broad_conflict_diplomacy_fit",
            lambda x: int(x.eq("Partial").sum()),
        ),
    )
    .reset_index()
)

assert len(coverage_31k) == 9
assert coverage_31k["n_segments"].eq(2).all()

print("\n" + "=" * 68)
print("COUNTRY-PERIOD SEMANTIC COVERAGE")
print("=" * 68)

display(coverage_31k)

# ------------------------------------------------------------
# 3. Validate the statistical result structures
# ------------------------------------------------------------

required_result_columns_31k = {
    "pair_label",
    "topic_id",
    "effect_pp",
    "holm_adjusted_p",
}

for name, dataframe in [
    ("Main", main_31k),
    ("Supplementary", supplement_31k),
]:
    missing = required_result_columns_31k.difference(
        dataframe.columns
    )
    assert not missing, (
        f"{name} results missing columns: {missing}"
    )

    assert not dataframe.duplicated(
        ["pair_label", "topic_id"]
    ).any()

# ------------------------------------------------------------
# 4. Verify full numerical consistency
# ------------------------------------------------------------

comparison_31k = main_31k[
    [
        "pair_label",
        "topic_id",
        "effect_pp",
        "holm_adjusted_p",
    ]
].merge(
    supplement_31k[
        [
            "pair_label",
            "topic_id",
            "effect_pp",
            "holm_adjusted_p",
        ]
    ],
    on=["pair_label", "topic_id"],
    how="left",
    suffixes=("_main", "_supplement"),
    validate="one_to_one",
    indicator=True,
)

assert len(comparison_31k) == 5

assert comparison_31k["_merge"].eq(
    "both"
).all()

effect_difference_31k = (
    comparison_31k["effect_pp_main"]
    - comparison_31k["effect_pp_supplement"]
).abs()

p_difference_31k = (
    comparison_31k["holm_adjusted_p_main"]
    - comparison_31k["holm_adjusted_p_supplement"]
).abs()

assert np.allclose(
    comparison_31k["effect_pp_main"],
    comparison_31k["effect_pp_supplement"],
    rtol=1e-10,
    atol=1e-10,
)

assert np.allclose(
    comparison_31k["holm_adjusted_p_main"],
    comparison_31k["holm_adjusted_p_supplement"],
    rtol=1e-10,
    atol=1e-10,
)

print("\n" + "=" * 68)
print("MAIN VS SUPPLEMENTARY NUMERICAL INTEGRITY")
print("=" * 68)

print(
    "Maximum effect discrepancy:",
    effect_difference_31k.max()
)

print(
    "Maximum adjusted p-value discrepancy:",
    p_difference_31k.max()
)

print("NUMERICAL RECONCILIATION: PASSED")

# ------------------------------------------------------------
# 5. Construct reference-aligned semantic labels
# ------------------------------------------------------------

topic_labels_31k = {
    1: "Economic development and global cooperation",
    2: "China and peaceful development",
    3: "Regional conflicts and diplomatic settlement",
    4: "Iran and regional security",
    5: "Human rights and freedom discourse",
    6: "UN General Assembly procedural language",
    7: "Nuclear arms and disarmament",
}

assert set(
    main_31k["topic_id"]
).issubset(topic_labels_31k)

assert set(
    supplement_31k["topic_id"]
).issubset(topic_labels_31k)

publication_main_preview_31k = main_31k.copy(deep=True)
publication_supp_preview_31k = supplement_31k.copy(deep=True)

for dataframe in [
    publication_main_preview_31k,
    publication_supp_preview_31k,
]:

    dataframe["proposed_semantic_label"] = (
        dataframe["topic_id"].map(topic_labels_31k)
    )

    dataframe["semantic_label_status"] = np.where(
        dataframe["topic_id"].eq(3),
        "RESEARCHER_APPROVED_WITH_QUALIFICATIONS",
        "PROVISIONAL_REFERENCE_LABEL",
    )

assert (
    publication_main_preview_31k[
        "proposed_semantic_label"
    ].notna().all()
)

assert (
    publication_supp_preview_31k[
        "proposed_semantic_label"
    ].notna().all()
)

# ------------------------------------------------------------
# 6. Main manuscript table preview
# ------------------------------------------------------------

main_preview_display_31k = (
    publication_main_preview_31k[
        [
            "pair_label",
            "topic_id",
            "proposed_semantic_label",
            "effect_pp",
            "holm_adjusted_p",
            "semantic_label_status",
        ]
    ]
    .copy()
)

main_preview_display_31k["effect_pp"] = (
    main_preview_display_31k["effect_pp"].round(4)
)

main_preview_display_31k["holm_adjusted_p"] = (
    main_preview_display_31k[
        "holm_adjusted_p"
    ].map(lambda x: f"{x:.3e}")
)

print("\n" + "=" * 68)
print("MAIN MANUSCRIPT RESULTS — SEMANTIC PREVIEW")
print("=" * 68)

display(main_preview_display_31k)

# ------------------------------------------------------------
# 7. Topic 03 across all supplementary contrasts
# ------------------------------------------------------------

topic03_publication_review_31k = (
    publication_supp_preview_31k.loc[
        publication_supp_preview_31k["topic_id"].eq(3),
        [
            "pair_label",
            "topic_id",
            "proposed_semantic_label",
            "effect_pp",
            "holm_adjusted_p",
        ],
    ]
    .copy()
)

assert len(topic03_publication_review_31k) == 3

topic03_publication_review_31k["effect_pp"] = (
    topic03_publication_review_31k[
        "effect_pp"
    ].round(4)
)

topic03_publication_review_31k[
    "holm_adjusted_p"
] = topic03_publication_review_31k[
    "holm_adjusted_p"
].map(lambda x: f"{x:.3e}")

print("\n" + "=" * 68)
print("TOPIC 03 — ALL BILATERAL CONTRASTS")
print("=" * 68)

display(topic03_publication_review_31k)

# ------------------------------------------------------------
# 8. Confirm source-level evidence and qualifications
# ------------------------------------------------------------

fit_counts_31k = review_31k[
    "broad_conflict_diplomacy_fit"
].value_counts()

n_yes_31k = int(fit_counts_31k.get("Yes", 0))
n_partial_31k = int(fit_counts_31k.get("Partial", 0))
n_no_31k = int(fit_counts_31k.get("No", 0))

assert (n_yes_31k, n_partial_31k, n_no_31k) == (
    12, 6, 0
)

source_exception_count_31k = int(
    review_31k["source_exception_flag"].sum()
)

assert source_exception_count_31k == 2

# ------------------------------------------------------------
# 9. Final publication-readiness diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("PUBLICATION READINESS DIAGNOSTICS")
print("=" * 68)

print("Reference NMF topics:", 7)
print("Main manuscript contrasts:", len(main_31k))
print("Supplementary contrasts:", len(supplement_31k))
print("Reviewed Topic 03 segments:", len(review_31k))
print("Country-period sampling strata:", len(coverage_31k))

print("\nTopic 03 full fit:", n_yes_31k)
print("Topic 03 partial fit:", n_partial_31k)
print("Topic 03 no fit:", n_no_31k)

print(
    "Full-fit proportion in reviewed sample:",
    f"{100 * n_yes_31k / len(review_31k):.2f}%"
)

print("Retained source exceptions:", source_exception_count_31k)

print("\nMain/supplement numerical consistency: PASSED")
print("Reference topic-ID mapping: PASSED")
print("Researcher batch confirmation: RECORDED")
print("Independent second-coder validation: NOT PERFORMED")
print("Independent archival verification: NOT ESTABLISHED")
print("Non-Topic-03 labels: PROVISIONAL")

print("\nOriginal statistical estimates modified: NO")
print("Original NMF models modified: NO")
print("Existing publication tables modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31K: COMPUTATION COMPLETED")



### Step 12.27H.31L — Final Robustness and Publication Readiness Audit

**Objective:** Evaluate the readiness of the five primary statistical findings for manuscript reporting.

The audit integrates:

1. Previously reconciled main and supplementary results.
2. Researcher-confirmed semantic assessments across 18 sampled segments.
3. Available leave-one-year-out influence diagnostics.
4. Outstanding archival and independent-coding limitations.
5. Publication-readiness decisions.

**Important:** This audit does not independently re-estimate the original statistical models. If leave-one-year-out diagnostics are unavailable, robustness remains pending.

Original datasets, fitted models, statistical estimates, and publication tables are preserved. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.31L
# FINAL ROBUSTNESS AND PUBLICATION READINESS AUDIT
# ============================================================

print("=" * 68)
print("STEP 12.27H.31L — FINAL PUBLICATION READINESS")
print("=" * 68)

# ------------------------------------------------------------
# 1. Recover previously verified results
# ------------------------------------------------------------

required_31l = [
    "publication_main_preview_31k",
    "publication_supp_preview_31k",
    "semantic_adjudication_final_31j",
]

for name in required_31l:
    assert name in globals(), f"Missing object: {name}"

main_31l = publication_main_preview_31k.copy(deep=True)
supp_31l = publication_supp_preview_31k.copy(deep=True)
review_31l = semantic_adjudication_final_31j.copy(deep=True)

assert len(main_31l) == 5
assert len(supp_31l) == 18
assert len(review_31l) == 18

assert review_31l["human_review_status"].eq(
    "RESEARCHER_CONFIRMED"
).all()

# ------------------------------------------------------------
# 2. Validate primary statistical findings
# ------------------------------------------------------------

required_numeric_31l = [
    "effect_pp",
    "holm_adjusted_p",
]

for col in required_numeric_31l:
    values = pd.to_numeric(main_31l[col], errors="coerce")
    assert np.isfinite(values.to_numpy()).all()

assert main_31l["holm_adjusted_p"].between(0, 1).all()

main_31l["holm_significant_31l"] = (
    main_31l["holm_adjusted_p"] < 0.05
)

print("\nPRIMARY STATISTICAL RESULTS")
display(
    main_31l[
        [
            "pair_label",
            "topic_id",
            "proposed_semantic_label",
            "effect_pp",
            "holm_adjusted_p",
            "holm_significant_31l",
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 3. Inspect leave-one-year-out stability if available
# ------------------------------------------------------------

loyo_required_31l = {
    "pair_label",
    "topic_id",
    "same_direction_years",
    "holm_significant_years",
    "maximum_absolute_shift_pp",
    "most_influential_year",
}

loyo_available_31l = (
    "loyo_summary_df" in globals()
    and isinstance(globals().get("loyo_summary_df"), pd.DataFrame)
    and loyo_required_31l.issubset(
        globals()["loyo_summary_df"].columns
    )
)

primary_robustness_31l = main_31l[
    [
        "pair_label",
        "topic_id",
        "proposed_semantic_label",
        "effect_pp",
        "holm_adjusted_p",
    ]
].copy()

if loyo_available_31l:

    loyo_check_31l = loyo_summary_df.copy(deep=True)

    assert not loyo_check_31l.duplicated(
        ["pair_label", "topic_id"]
    ).any()

    primary_robustness_31l = primary_robustness_31l.merge(
        loyo_check_31l[
            [
                "pair_label",
                "topic_id",
                "same_direction_years",
                "holm_significant_years",
                "maximum_absolute_shift_pp",
                "most_influential_year",
            ]
        ],
        on=["pair_label", "topic_id"],
        how="left",
        validate="one_to_one",
        indicator=True,
    )

    assert primary_robustness_31l["_merge"].eq(
        "both"
    ).all(), "LOYO results do not cover all five primary findings."

    primary_robustness_31l.drop(
        columns="_merge", inplace=True
    )

    primary_robustness_31l["loyo_stable_31l"] = (
        primary_robustness_31l["same_direction_years"].eq(54)
        & primary_robustness_31l[
            "holm_significant_years"
        ].eq(54)
    )

    loyo_status_31l = "AVAILABLE"

else:

    primary_robustness_31l["loyo_stable_31l"] = pd.NA
    loyo_status_31l = "PENDING_REPRODUCTION"

print("\n" + "=" * 68)
print("FIVE PRIMARY FINDINGS — ROBUSTNESS AUDIT")
print("=" * 68)

display(primary_robustness_31l.round(6))

print("\nLOYO diagnostic status:", loyo_status_31l)

if loyo_available_31l:
    print(
        "Findings stable across all 54 exclusions:",
        int(primary_robustness_31l["loyo_stable_31l"].sum()),
        "/ 5"
    )
else:
    print(
        "LOYO results unavailable in current notebook state."
    )

# ------------------------------------------------------------
# 4. Register source-integrity qualifications
# ------------------------------------------------------------

assert "source_exception_flag" in review_31l.columns

exception_mask_31l = (
    review_31l["source_exception_flag"]
    .fillna(False)
    .astype(bool)
)

source_exceptions_31l = review_31l.loc[
    exception_mask_31l,
    [
        "review_id",
        "country_code",
        "year",
        "broad_conflict_diplomacy_fit",
        "human_review_status",
    ]
].copy()

assert len(source_exceptions_31l) == 2

print("\n" + "=" * 68)
print("RETAINED SOURCE-INTEGRITY QUALIFICATIONS")
print("=" * 68)

display(source_exceptions_31l)

# ------------------------------------------------------------
# 5. Assess reviewed-sample semantic fit
# ------------------------------------------------------------

semantic_counts_31l = (
    review_31l["broad_conflict_diplomacy_fit"]
    .value_counts()
)

n_full_31l = int(semantic_counts_31l.get("Yes", 0))
n_partial_31l = int(semantic_counts_31l.get("Partial", 0))
n_no_31l = int(semantic_counts_31l.get("No", 0))

assert (n_full_31l, n_partial_31l, n_no_31l) == (
    12, 6, 0
)

# ------------------------------------------------------------
# 6. Produce final audit register
# ------------------------------------------------------------

audit_register_31l = pd.DataFrame([
    {
        "audit": "Main/supplement numerical agreement",
        "status": "PASSED",
    },
    {
        "audit": "Primary statistical significance",
        "status": (
            f"{int(main_31l['holm_significant_31l'].sum())}/5"
        ),
    },
    {
        "audit": "LOYO robustness",
        "status": loyo_status_31l,
    },
    {
        "audit": "Researcher semantic confirmation",
        "status": "18/18 RECORDED",
    },
    {
        "audit": "Topic 03 broad-label full fit",
        "status": "12/18 SAMPLED SEGMENTS",
    },
    {
        "audit": "Topic 03 broad-label partial fit",
        "status": "6/18 SAMPLED SEGMENTS",
    },
    {
        "audit": "Independent second-coder assessment",
        "status": "NOT PERFORMED",
    },
    {
        "audit": "Outstanding archival qualifications",
        "status": "2 RECORDS",
    },
    {
        "audit": "Corpus-wide semantic prevalence",
        "status": "NOT ESTIMATED",
    },
    {
        "audit": "Non-Topic-03 semantic labels",
        "status": "PROVISIONAL",
    },
])

print("\n" + "=" * 68)
print("FINAL SCIENTIFIC AUDIT REGISTER")
print("=" * 68)

display(audit_register_31l)

# ------------------------------------------------------------
# 7. Final readiness classification
# ------------------------------------------------------------

statistically_consistent_31l = (
    len(main_31l) == 5
    and len(supp_31l) == 18
    and main_31l["holm_significant_31l"].all()
)

if loyo_available_31l:
    robust_count_31l = int(
        primary_robustness_31l["loyo_stable_31l"].sum()
    )
else:
    robust_count_31l = None

print("\n" + "=" * 68)
print("FINAL VALIDATION")
print("=" * 68)

print(
    "Statistical table consistency:",
    "PASSED" if statistically_consistent_31l else "REVIEW"
)

print("LOYO availability:", loyo_status_31l)
print("LOYO fully stable findings:", robust_count_31l)

print("Topic 03 semantic interpretation: APPROVED WITH QUALIFICATIONS")
print("Independent archival verification: NOT ESTABLISHED")
print("Independent second-coder validation: NOT PERFORMED")
print("Source-level sample representativeness: NOT CLAIMED")

print("\nOriginal statistical estimates modified: NO")
print("Original NMF models modified: NO")
print("Publication tables modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31L: AUDIT COMPLETED")



### Step 12.27H.31M — Publication Table Construction

**Objective:** Construct publication-ready statistical tables from the validated NMF topic contrasts and leave-one-year-out diagnostics.

**Outputs:**
- Table 1: Five primary bilateral findings, including HAC 95% confidence intervals and Holm-adjusted p-values.
- Supplementary Table S1: All 18 bilateral topic contrasts.
- Supplementary Table S2: Leave-one-year-out robustness for the five primary findings.

The effects represent changes in annual first-country-minus-second-country topic-share differences for 2011–2025 relative to 1972–1990, measured in percentage points.

Confidence intervals are retrieved from the independently recomputed full-sample HAC estimates and checked against the publication results before use.

Topic labels remain provisional where independent semantic validation has not been completed. No files are saved in this step.


In [ ]:

# ============================================================
# STEP 12.27H.31M
# PUBLICATION TABLE CONSTRUCTION
# ============================================================

print("=" * 68)
print("STEP 12.27H.31M — PUBLICATION TABLE CONSTRUCTION")
print("=" * 68)

# ------------------------------------------------------------
# 1. Verify source objects
# ------------------------------------------------------------

required_objects_31m = [
    "main_31l",
    "supp_31l",
    "loyo_baseline_df",
    "loyo_summary_df",
]

for name in required_objects_31m:
    assert name in globals(), f"Missing source object: {name}"

main_source_31m = main_31l.copy(deep=True)
supp_source_31m = supp_31l.copy(deep=True)
hac_source_31m = loyo_baseline_df.copy(deep=True)
loyo_source_31m = loyo_summary_df.copy(deep=True)

keys_31m = ["pair_label", "topic_id"]

assert len(main_source_31m) == 5
assert len(supp_source_31m) == 18
assert len(hac_source_31m) == 18
assert len(loyo_source_31m) == 5

for frame in [
    main_source_31m,
    supp_source_31m,
    hac_source_31m,
    loyo_source_31m,
]:
    assert not frame.duplicated(keys_31m).any()

required_hac_31m = [
    "effect_pp",
    "ci_lower_pp",
    "ci_upper_pp",
    "holm_adjusted_p",
]

assert set(required_hac_31m).issubset(
    hac_source_31m.columns
)

# ------------------------------------------------------------
# 2. Verify numerical equivalence
# ------------------------------------------------------------

comparison_31m = supp_source_31m.merge(
    hac_source_31m[
        keys_31m + [
            "effect_pp",
            "holm_adjusted_p",
        ]
    ],
    on=keys_31m,
    suffixes=("_publication", "_recomputed"),
    validate="one_to_one",
)

assert len(comparison_31m) == 18

effect_difference_31m = (
    comparison_31m["effect_pp_publication"]
    - comparison_31m["effect_pp_recomputed"]
).abs().max()

p_difference_31m = (
    comparison_31m["holm_adjusted_p_publication"]
    - comparison_31m["holm_adjusted_p_recomputed"]
).abs().max()

print("\nNUMERICAL REPRODUCIBILITY")

print(
    "Maximum effect difference:",
    f"{effect_difference_31m:.10g}"
)

print(
    "Maximum adjusted p-value difference:",
    f"{p_difference_31m:.10g}"
)

assert effect_difference_31m < 1e-6, (
    "HAC effect estimates require reconciliation."
)

assert p_difference_31m < 1e-6, (
    "HAC adjusted p-values require reconciliation."
)

print("HAC NUMERICAL REPRODUCIBILITY: PASSED")


# ------------------------------------------------------------
# 3. Attach verified HAC confidence intervals
#    Safely resolve pre-existing CI columns
# ------------------------------------------------------------

ci_columns_31m = [
    "ci_lower_pp",
    "ci_upper_pp",
]

ci_source_31m = hac_source_31m[
    keys_31m + ci_columns_31m
].copy()

assert not ci_source_31m.duplicated(keys_31m).any()

# Remove existing CI columns before merging.
# The recomputed HAC baseline is the authoritative CI source.

main_clean_31m = main_source_31m.drop(
    columns=ci_columns_31m,
    errors="ignore",
).copy()

supp_clean_31m = supp_source_31m.drop(
    columns=ci_columns_31m,
    errors="ignore",
).copy()

table1_numeric_31m = main_clean_31m.merge(
    ci_source_31m,
    on=keys_31m,
    how="left",
    validate="one_to_one",
)

table_s1_numeric_31m = supp_clean_31m.merge(
    ci_source_31m,
    on=keys_31m,
    how="left",
    validate="one_to_one",
)

# Validate column uniqueness and confidence intervals.

for frame in [
    table1_numeric_31m,
    table_s1_numeric_31m,
]:

    assert frame.columns.is_unique, (
        "Duplicate column names detected."
    )

    assert set(ci_columns_31m).issubset(
        frame.columns
    )

    assert frame[ci_columns_31m].notna().all().all(), (
        "Missing confidence interval values."
    )

    assert (
        frame["ci_lower_pp"] <= frame["effect_pp"]
    ).all(), (
        "Lower CI exceeds the effect estimate."
    )

    assert (
        frame["effect_pp"] <= frame["ci_upper_pp"]
    ).all(), (
        "Upper CI is below the effect estimate."
    )

assert len(table1_numeric_31m) == 5
assert len(table_s1_numeric_31m) == 18

print("=" * 68)
print("HAC CONFIDENCE INTERVAL RECONCILIATION")
print("=" * 68)

print("Main table rows:", len(table1_numeric_31m))
print("Supplementary rows:", len(table_s1_numeric_31m))

print("Main table CI columns:",
      table1_numeric_31m[ci_columns_31m].columns.tolist())

print("Supplementary CI columns:",
      table_s1_numeric_31m[ci_columns_31m].columns.tolist())

print("\nCI RECONCILIATION: PASSED")


# ------------------------------------------------------------
# 4. Publication formatting
# ------------------------------------------------------------

def format_publication_table_31m(frame):

    formatted = frame.copy(deep=True)

    formatted["Effect (pp)"] = formatted[
        "effect_pp"
    ].map(lambda x: f"{x:+.2f}")

    formatted["95% HAC CI (pp)"] = [
        f"[{lower:+.2f}, {upper:+.2f}]"
        for lower, upper in zip(
            formatted["ci_lower_pp"],
            formatted["ci_upper_pp"],
        )
    ]

    formatted["Holm-adjusted p"] = formatted[
        "holm_adjusted_p"
    ].map(lambda x: f"{x:.3e}")

    formatted = formatted.rename(
        columns={
            "pair_label": "Country contrast",
            "topic_id": "Topic",
            "proposed_semantic_label": "Semantic label",
        }
    )

    return formatted[
        [
            "Country contrast",
            "Topic",
            "Semantic label",
            "Effect (pp)",
            "95% HAC CI (pp)",
            "Holm-adjusted p",
        ]
    ]

table1_preview_31m = format_publication_table_31m(
    table1_numeric_31m
)

table_s1_preview_31m = format_publication_table_31m(
    table_s1_numeric_31m
)

print("\n" + "=" * 68)
print("TABLE 1 — FIVE PRIMARY BILATERAL FINDINGS")
print("=" * 68)

display(table1_preview_31m)

print("\n" + "=" * 68)
print("SUPPLEMENTARY TABLE S1 — ALL 18 CONTRASTS")
print("=" * 68)

display(table_s1_preview_31m)

# ------------------------------------------------------------
# 5. Construct LOYO robustness table
# ------------------------------------------------------------

loyo_columns_31m = [
    "pair_label",
    "topic_id",
    "baseline_effect_pp",
    "maximum_absolute_shift_pp",
    "most_influential_year",
    "same_direction_years",
    "holm_significant_years",
    "maximum_holm_adjusted_p",
    "loyo_stable",
]

assert set(loyo_columns_31m).issubset(
    loyo_source_31m.columns
)

table_s2_numeric_31m = main_source_31m[
    keys_31m + ["proposed_semantic_label"]
].merge(
    loyo_source_31m[loyo_columns_31m],
    on=keys_31m,
    validate="one_to_one",
)

assert len(table_s2_numeric_31m) == 5

assert table_s2_numeric_31m[
    "same_direction_years"
].eq(54).all()

assert table_s2_numeric_31m[
    "holm_significant_years"
].eq(54).all()

assert table_s2_numeric_31m[
    "loyo_stable"
].eq(True).all()

table_s2_preview_31m = table_s2_numeric_31m.rename(
    columns={
        "pair_label": "Country contrast",
        "topic_id": "Topic",
        "baseline_effect_pp": "Baseline effect (pp)",
        "maximum_absolute_shift_pp": "Maximum shift (pp)",
        "most_influential_year": "Most influential year",
        "same_direction_years": "Direction retained",
        "holm_significant_years": "Holm significance retained",
    }
)[
    [
        "Country contrast",
        "Topic",
        "Baseline effect (pp)",
        "Maximum shift (pp)",
        "Most influential year",
        "Direction retained",
        "Holm significance retained",
    ]
].copy()

for col in [
    "Baseline effect (pp)",
    "Maximum shift (pp)",
]:
    table_s2_preview_31m[col] = (
        table_s2_preview_31m[col].round(3)
    )

print("\n" + "=" * 68)
print("SUPPLEMENTARY TABLE S2 — LOYO ROBUSTNESS")
print("=" * 68)

display(table_s2_preview_31m)

# ------------------------------------------------------------
# 6. Final publication-table validation
# ------------------------------------------------------------

print("\n" + "=" * 68)
print("FINAL TABLE VALIDATION")
print("=" * 68)

print("Main table rows:", len(table1_preview_31m))
print("Supplementary Table S1 rows:", len(table_s1_preview_31m))
print("Supplementary Table S2 rows:", len(table_s2_preview_31m))

print("HAC confidence intervals: VERIFIED")
print("Holm-adjusted p-values: VERIFIED")
print("LOYO robustness: 5/5 FULLY STABLE")

print("Topic 03 semantic label: APPROVED WITH QUALIFICATIONS")
print("Other semantic labels: PROVISIONAL")
print("Independent archival qualifications: RETAINED")

print("\nOriginal statistical estimates modified: NO")
print("Original fitted NMF models modified: NO")
print("Source data modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31M: TABLES PREPARED FOR VISUAL QC")



### Step 12.27H.31N — Complete Robustness and Primary-Finding Selection Audit

**Objective:** Evaluate all 18 bilateral topic contrasts using the existing leave-one-year-out results and identify whether statistically significant, fully stable findings have been omitted from the primary manuscript table.

This step:
- Preserves all previously estimated HAC effects and Holm-adjusted p-values.
- Verifies the consistency of the five primary findings.
- Evaluates effect direction and Holm significance across all 54 annual exclusions.
- Identifies additional robust findings requiring a transparent reporting decision.
- Does not modify the original models, statistical estimates, or publication tables.


In [ ]:

# ============================================================
# STEP 12.27H.31N
# COMPLETE ROBUSTNESS AND PRIMARY-SELECTION AUDIT
# ============================================================

required_31n = [
    "table1_numeric_31m",
    "table_s1_numeric_31m",
    "loyo_results_df",
]

for name in required_31n:
    assert name in globals(), f"Missing object: {name}"

keys_31n = ["pair_label", "topic_id"]

main_31n = table1_numeric_31m.copy()
supp_31n = table_s1_numeric_31m.copy()
loyo_31n = loyo_results_df.copy()

assert len(main_31n) == 5
assert len(supp_31n) == 18
assert len(loyo_31n) == 54 * 18

assert not main_31n.duplicated(keys_31n).any()
assert not supp_31n.duplicated(keys_31n).any()

assert not loyo_31n.duplicated(
    keys_31n + ["excluded_year"]
).any()

# ------------------------------------------------------------
# 1. Reconcile primary and supplementary estimates
# ------------------------------------------------------------

comparison_31n = main_31n[
    keys_31n + [
        "effect_pp",
        "holm_adjusted_p",
        "ci_lower_pp",
        "ci_upper_pp",
    ]
].merge(
    supp_31n[
        keys_31n + [
            "effect_pp",
            "holm_adjusted_p",
            "ci_lower_pp",
            "ci_upper_pp",
        ]
    ],
    on=keys_31n,
    suffixes=("_main", "_supp"),
    validate="one_to_one",
)

assert len(comparison_31n) == 5

for field in [
    "effect_pp",
    "holm_adjusted_p",
    "ci_lower_pp",
    "ci_upper_pp",
]:
    discrepancy = (
        comparison_31n[f"{field}_main"]
        - comparison_31n[f"{field}_supp"]
    ).abs().max()

    assert discrepancy < 1e-10, (
        f"Main/supplement discrepancy: {field}"
    )

print("=" * 65)
print("STEP 12.27H.31N — FULL ROBUSTNESS AUDIT")
print("=" * 65)
print("Main/supplement numerical reconciliation: PASSED")

# ------------------------------------------------------------
# 2. Attach full-sample reference estimates
# ------------------------------------------------------------

baseline_31n = supp_31n[
    keys_31n + ["effect_pp", "holm_adjusted_p"]
].rename(
    columns={
        "effect_pp": "baseline_effect_pp",
        "holm_adjusted_p": "baseline_holm_p",
    }
)

loyo_31n = loyo_31n.merge(
    baseline_31n,
    on=keys_31n,
    how="left",
    validate="many_to_one",
)

assert loyo_31n[
    ["baseline_effect_pp", "baseline_holm_p"]
].notna().all().all()

loyo_31n["same_direction"] = (
    np.sign(loyo_31n["effect_pp"])
    == np.sign(loyo_31n["baseline_effect_pp"])
)

loyo_31n["retains_holm_significance"] = (
    loyo_31n["holm_adjusted_p"] < 0.05
)

loyo_31n["absolute_effect_shift_pp"] = (
    loyo_31n["effect_pp"]
    - loyo_31n["baseline_effect_pp"]
).abs()

# ------------------------------------------------------------
# 3. Aggregate all 18 LOYO trajectories
# ------------------------------------------------------------

audit_31n = loyo_31n.groupby(
    keys_31n,
    as_index=False,
).agg(
    n_exclusions=("excluded_year", "nunique"),
    same_direction_years=("same_direction", "sum"),
    holm_significant_years=(
        "retains_holm_significance", "sum"
    ),
    maximum_absolute_shift_pp=(
        "absolute_effect_shift_pp", "max"
    ),
)

audit_31n = audit_31n.merge(
    baseline_31n,
    on=keys_31n,
    validate="one_to_one",
)

assert len(audit_31n) == 18
assert audit_31n["n_exclusions"].eq(54).all()

audit_31n["baseline_significant"] = (
    audit_31n["baseline_holm_p"] < 0.05
)

audit_31n["loyo_stable"] = (
    audit_31n["same_direction_years"].eq(54)
    & audit_31n["holm_significant_years"].eq(54)
)

main_keys_31n = main_31n[keys_31n].copy()
main_keys_31n["in_main_table"] = True

audit_31n = audit_31n.merge(
    main_keys_31n,
    on=keys_31n,
    how="left",
    validate="one_to_one",
)

audit_31n["in_main_table"] = (
    audit_31n["in_main_table"]
    .fillna(False)
    .astype(bool)
)

audit_31n["additional_robust_finding"] = (
    audit_31n["baseline_significant"]
    & audit_31n["loyo_stable"]
    & ~audit_31n["in_main_table"]
)

# ------------------------------------------------------------
# 4. Inspect all Holm-significant findings
# ------------------------------------------------------------

significant_31n = audit_31n.loc[
    audit_31n["baseline_significant"]
].sort_values(
    ["pair_label", "topic_id"]
).reset_index(drop=True)

print("\n" + "=" * 65)
print("ALL HOLM-SIGNIFICANT CONTRASTS")
print("=" * 65)

display(
    significant_31n[
        [
            "pair_label",
            "topic_id",
            "baseline_effect_pp",
            "baseline_holm_p",
            "same_direction_years",
            "holm_significant_years",
            "maximum_absolute_shift_pp",
            "loyo_stable",
            "in_main_table",
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 5. Identify additional robust findings
# ------------------------------------------------------------

additional_31n = audit_31n.loc[
    audit_31n["additional_robust_finding"]
].copy()

print("\n" + "=" * 65)
print("ADDITIONAL ROBUST FINDINGS OUTSIDE TABLE 1")
print("=" * 65)

if additional_31n.empty:
    print("No additional fully stable findings.")
else:
    display(
        additional_31n[
            [
                "pair_label",
                "topic_id",
                "baseline_effect_pp",
                "baseline_holm_p",
                "same_direction_years",
                "holm_significant_years",
                "maximum_absolute_shift_pp",
            ]
        ].round(6)
    )

# ------------------------------------------------------------
# 6. Final scientific diagnostics
# ------------------------------------------------------------

assert audit_31n.loc[
    audit_31n["in_main_table"], "loyo_stable"
].all(), "At least one primary finding is not LOYO-stable."

print("\n" + "=" * 65)
print("FINAL PRIMARY-SELECTION DIAGNOSTICS")
print("=" * 65)

print("Total contrasts:", len(audit_31n))

print(
    "Holm-significant contrasts:",
    int(audit_31n["baseline_significant"].sum())
)

print(
    "LOYO-stable contrasts:",
    int(audit_31n["loyo_stable"].sum())
)

print(
    "Primary findings:",
    int(audit_31n["in_main_table"].sum())
)

print(
    "Additional robust findings:",
    int(audit_31n["additional_robust_finding"].sum())
)

print("\nOriginal statistical estimates modified: NO")
print("Original NMF models modified: NO")
print("Publication tables modified: NO")
print("Files saved: NONE")
print("\nSTEP 12.27H.31N: AUDIT COMPLETED")



### Step 12.27H.31O — Publication Table Selection and Robustness Preview

**Objective:** Prepare and visually inspect an expanded primary-results table containing all nine Holm-significant, leave-one-year-out-stable contrasts.

The existing five-result Table 1 remains unchanged.

This diagnostic also:
- Prepares the nine-result LOYO sensitivity table.
- Identifies significant results that do not satisfy full LOYO stability.
- Preserves pointwise 95% HAC confidence intervals and Holm-adjusted p-values.
- Flags country-specific and semantically heterogeneous topic interpretations.

No statistical estimates, fitted models, publication tables, or source data are overwritten. No files are saved during this step.


In [ ]:

# ============================================================
# STEP 12.27H.31O
# PUBLICATION TABLE SELECTION AND ROBUSTNESS PREVIEW
# ============================================================

required_31o = [
    "audit_31n",
    "loyo_31n",
    "table1_numeric_31m",
    "table_s1_numeric_31m",
]

for name in required_31o:
    assert name in globals(), f"Missing object: {name}"

keys_31o = ["pair_label", "topic_id"]

labels_31o = {
    1: "Economic development and global cooperation",
    2: "China and peaceful development",
    3: "Regional conflicts and diplomatic settlement",
    4: "Iran and regional security",
    5: "Human rights and freedom discourse",
    7: "Nuclear arms and disarmament",
}

# ------------------------------------------------------------
# 1. Prepare the nine-result candidate table
# ------------------------------------------------------------

stable_keys_31o = audit_31n.loc[
    audit_31n["baseline_significant"]
    & audit_31n["loyo_stable"],
    keys_31o,
].drop_duplicates()

candidate_main9_numeric_31o = (
    table_s1_numeric_31m.merge(
        stable_keys_31o,
        on=keys_31o,
        how="inner",
        validate="one_to_one",
    )
    .sort_values(keys_31o)
    .reset_index(drop=True)
)

assert len(candidate_main9_numeric_31o) == 9

candidate_main9_numeric_31o["semantic_label_31o"] = (
    candidate_main9_numeric_31o["topic_id"]
    .map(labels_31o)
)

assert candidate_main9_numeric_31o[
    "semantic_label_31o"
].notna().all()

# Publication-style preview only.
candidate_main9_preview_31o = pd.DataFrame({
    "Country contrast":
        candidate_main9_numeric_31o["pair_label"],

    "Topic":
        candidate_main9_numeric_31o["topic_id"],

    "Semantic label":
        candidate_main9_numeric_31o["semantic_label_31o"],

    "Effect (pp)":
        candidate_main9_numeric_31o["effect_pp"].map(
            lambda x: f"{x:+.2f}"
        ),

    "95% HAC CI (pp)": [
        f"[{lower:+.2f}, {upper:+.2f}]"
        for lower, upper in zip(
            candidate_main9_numeric_31o["ci_lower_pp"],
            candidate_main9_numeric_31o["ci_upper_pp"],
        )
    ],

    "Holm-adjusted p":
        candidate_main9_numeric_31o["holm_adjusted_p"].map(
            lambda x: f"{x:.3e}"
        ),
})

print("=" * 65)
print("STEP 12.27H.31O — PUBLICATION TABLE SELECTION")
print("=" * 65)

print("\n" + "=" * 65)
print("CANDIDATE TABLE 1 — NINE FULLY STABLE RESULTS")
print("=" * 65)

display(candidate_main9_preview_31o)

# ------------------------------------------------------------
# 2. Identify the most influential excluded years
# ------------------------------------------------------------

influential_indices_31o = (
    loyo_31n.groupby(keys_31o)[
        "absolute_effect_shift_pp"
    ].idxmax()
)

influential_31o = loyo_31n.loc[
    influential_indices_31o,
    keys_31o + ["excluded_year"],
].rename(
    columns={
        "excluded_year": "most_influential_year"
    }
)

# ------------------------------------------------------------
# 3. Construct an expanded LOYO audit table
# ------------------------------------------------------------

candidate_loyo9_numeric_31o = (
    audit_31n.loc[
        audit_31n["baseline_significant"]
        & audit_31n["loyo_stable"]
    ]
    .merge(
        influential_31o,
        on=keys_31o,
        validate="one_to_one",
    )
    .sort_values(keys_31o)
    .reset_index(drop=True)
)

assert len(candidate_loyo9_numeric_31o) == 9

candidate_loyo9_preview_31o = (
    candidate_loyo9_numeric_31o[
        [
            "pair_label",
            "topic_id",
            "baseline_effect_pp",
            "maximum_absolute_shift_pp",
            "most_influential_year",
            "same_direction_years",
            "holm_significant_years",
        ]
    ].copy()
)

candidate_loyo9_preview_31o.columns = [
    "Country contrast",
    "Topic",
    "Baseline effect (pp)",
    "Maximum shift (pp)",
    "Most influential year",
    "Direction retained",
    "Holm significance retained",
]

print("\n" + "=" * 65)
print("CANDIDATE SUPPLEMENTARY LOYO TABLE — NINE RESULTS")
print("=" * 65)

display(candidate_loyo9_preview_31o.round(3))

# ------------------------------------------------------------
# 4. Identify significant but less stable findings
# ------------------------------------------------------------

less_stable_31o = audit_31n.loc[
    audit_31n["baseline_significant"]
    & ~audit_31n["loyo_stable"]
].sort_values(keys_31o)

print("\n" + "=" * 65)
print("SIGNIFICANT FINDINGS NOT FULLY LOYO-STABLE")
print("=" * 65)

display(
    less_stable_31o[
        [
            "pair_label",
            "topic_id",
            "baseline_effect_pp",
            "baseline_holm_p",
            "same_direction_years",
            "holm_significant_years",
            "maximum_absolute_shift_pp",
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 5. Compare existing and candidate selections
# ------------------------------------------------------------

existing_keys_31o = set(
    map(
        tuple,
        table1_numeric_31m[keys_31o].to_numpy()
    )
)

candidate_keys_31o = set(
    map(
        tuple,
        candidate_main9_numeric_31o[
            keys_31o
        ].to_numpy()
    )
)

assert existing_keys_31o.issubset(candidate_keys_31o)
assert len(candidate_keys_31o - existing_keys_31o) == 4

print("\n" + "=" * 65)
print("FINAL SELECTION DIAGNOSTICS")
print("=" * 65)

print("Existing primary findings:", len(existing_keys_31o))
print("Candidate primary findings:", len(candidate_keys_31o))

print(
    "Additional fully stable findings:",
    len(candidate_keys_31o - existing_keys_31o)
)

print(
    "Significant but not fully stable:",
    len(less_stable_31o)
)

print(
    "All candidate results LOYO-stable:",
    bool(candidate_loyo9_numeric_31o["loyo_stable"].all())
)

print("\nImportant interpretation qualifications:")
print("- Topic 2: Country-specific semantic content.")
print("- Topic 3: High semantic heterogeneity.")
print("- HAC confidence intervals are pointwise.")
print("- Holm p-values are multiplicity-adjusted.")
print("- Primary selection requires transparent reporting.")

print("\nExisting Table 1 modified: NO")
print("Existing Supplementary Tables modified: NO")
print("Original NMF models modified: NO")
print("Statistical estimates modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31O: PREVIEW COMPLETED")



### Step 12.27H.31P — Final Manuscript Tables and Integrity Check

**Objective:** Prepare three publication-ready tables using the validated numerical results.

- **Table 1:** Nine Holm-significant contrasts that retain significance and effect direction in all 54 leave-one-year-out analyses.
- **Supplementary Table S1:** All 18 bilateral topic contrasts.
- **Supplementary Table S2:** Leave-one-year-out diagnostics for all 18 contrasts, including significant results that are not fully stable.

**Interpretation:** Effects represent changes in annual bilateral topic-share differences between 2011–2025 and the reference period 1972–1990. They are descriptive historical contrasts, not causal effects.

The nine-result main-table selection is post-analysis and must be transparently disclosed. Confidence intervals are pointwise HAC intervals; Holm adjustment applies to the reported p-values.

No models are refitted. No original tables or estimates are overwritten. No files are saved in this cell.


In [ ]:

# ============================================================
# STEP 12.27H.31P
# FINAL MANUSCRIPT TABLES AND INTEGRITY CHECK
# ============================================================

required_31p = [
    "candidate_main9_numeric_31o",
    "table_s1_numeric_31m",
    "audit_31n",
    "loyo_31n",
]

for name in required_31p:
    assert name in globals(), f"Missing object: {name}"

keys_31p = ["pair_label", "topic_id"]

labels_31p = {
    1: "Economic development and global cooperation",
    2: "China and peaceful development",
    3: "Regional conflicts and diplomatic settlement",
    4: "Iran and regional security",
    5: "Human rights and freedom discourse",
    7: "Nuclear arms and disarmament",
}


# ------------------------------------------------------------
# 1. Prepare numerical manuscript tables
# ------------------------------------------------------------

final_main9_numeric_31p = (
    candidate_main9_numeric_31o.copy()
    .sort_values(keys_31p)
    .reset_index(drop=True)
)

final_s1_numeric_31p = (
    table_s1_numeric_31m.copy()
    .sort_values(keys_31p)
    .reset_index(drop=True)
)

for frame in [
    final_main9_numeric_31p,
    final_s1_numeric_31p,
]:
    frame["semantic_label"] = (
        frame["topic_id"].map(labels_31p)
    )

    assert frame["semantic_label"].notna().all()

assert len(final_main9_numeric_31p) == 9
assert len(final_s1_numeric_31p) == 18


# ------------------------------------------------------------
# 2. Reconcile all main-table numerical estimates
# ------------------------------------------------------------

reconciliation_31p = final_main9_numeric_31p.merge(
    final_s1_numeric_31p,
    on=keys_31p,
    suffixes=("_main", "_supp"),
    validate="one_to_one",
)

assert len(reconciliation_31p) == 9

numeric_fields_31p = [
    "effect_pp",
    "ci_lower_pp",
    "ci_upper_pp",
    "holm_adjusted_p",
]

for field in numeric_fields_31p:
    assert np.allclose(
        reconciliation_31p[f"{field}_main"],
        reconciliation_31p[f"{field}_supp"],
        rtol=1e-10,
        atol=1e-12,
    ), f"Numerical mismatch: {field}"

print("=" * 65)
print("STEP 12.27H.31P — FINAL MANUSCRIPT TABLES")
print("=" * 65)

print("\nMain/supplement numerical agreement: PASSED")


# ------------------------------------------------------------
# 3. Prepare comprehensive LOYO audit
# ------------------------------------------------------------

influential_rows_31p = (
    loyo_31n.groupby(keys_31p)[
        "absolute_effect_shift_pp"
    ].idxmax()
)

influential_years_31p = (
    loyo_31n.loc[
        influential_rows_31p,
        keys_31p + ["excluded_year"],
    ]
    .rename(columns={
        "excluded_year": "most_influential_year"
    })
)

final_s2_numeric_31p = (
    audit_31n.merge(
        influential_years_31p,
        on=keys_31p,
        validate="one_to_one",
    )
    .sort_values(keys_31p)
    .reset_index(drop=True)
)

assert len(final_s2_numeric_31p) == 18
assert loyo_31n["excluded_year"].nunique() == 54

assert int(
    final_s2_numeric_31p["baseline_significant"].sum()
) == 10

assert int(
    final_s2_numeric_31p["loyo_stable"].sum()
) == 9


# ------------------------------------------------------------
# 4. Construct publication-formatted tables
# ------------------------------------------------------------

def format_result_table_31p(frame):
    return pd.DataFrame({
        "Country contrast": frame["pair_label"],
        "Topic": frame["topic_id"],
        "Semantic label": frame["semantic_label"],
        "Effect (pp)": frame["effect_pp"].map(
            lambda x: f"{x:+.2f}"
        ),
        "95% HAC CI (pp)": [
            f"[{low:+.2f}, {high:+.2f}]"
            for low, high in zip(
                frame["ci_lower_pp"],
                frame["ci_upper_pp"],
            )
        ],
        "Holm-adjusted p": frame[
            "holm_adjusted_p"
        ].map(lambda x: f"{x:.3e}"),
    })


final_table1_31p = format_result_table_31p(
    final_main9_numeric_31p
)

final_table_s1_31p = format_result_table_31p(
    final_s1_numeric_31p
)

final_table_s2_31p = pd.DataFrame({
    "Country contrast":
        final_s2_numeric_31p["pair_label"],

    "Topic":
        final_s2_numeric_31p["topic_id"],

    "Baseline effect (pp)":
        final_s2_numeric_31p["baseline_effect_pp"].round(3),

    "Baseline Holm-adjusted p":
        final_s2_numeric_31p["baseline_holm_p"].map(
            lambda x: f"{x:.3e}"
        ),

    "Maximum absolute shift (pp)":
        final_s2_numeric_31p[
            "maximum_absolute_shift_pp"
        ].round(3),

    "Most influential year":
        final_s2_numeric_31p["most_influential_year"],

    "Direction retained (of 54)":
        final_s2_numeric_31p["same_direction_years"],

    "Holm significance retained (of 54)":
        final_s2_numeric_31p["holm_significant_years"],

    "Fully LOYO-stable":
        final_s2_numeric_31p["loyo_stable"],
})


# ------------------------------------------------------------
# 5. Visual quality control
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("TABLE 1 — NINE ROBUST BILATERAL CONTRASTS")
print("=" * 65)

display(final_table1_31p)

print("\n" + "=" * 65)
print("SUPPLEMENTARY TABLE S1 — ALL 18 CONTRASTS")
print("=" * 65)

display(final_table_s1_31p)

print("\n" + "=" * 65)
print("SUPPLEMENTARY TABLE S2 — FULL LOYO AUDIT")
print("=" * 65)

display(final_table_s2_31p)


# ------------------------------------------------------------
# 6. Final scientific safeguards
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL SCIENTIFIC INTEGRITY CHECK")
print("=" * 65)

assert final_table1_31p.shape[0] == 9
assert final_table_s1_31p.shape[0] == 18
assert final_table_s2_31p.shape[0] == 18

assert set(
    map(tuple, final_main9_numeric_31p[keys_31p].to_numpy())
) == set(
    map(
        tuple,
        final_s2_numeric_31p.loc[
            final_s2_numeric_31p["loyo_stable"],
            keys_31p,
        ].to_numpy(),
    )
)

print("Table 1 rows:", len(final_table1_31p))
print("Table S1 rows:", len(final_table_s1_31p))
print("Table S2 rows:", len(final_table_s2_31p))

print("Holm-significant full-sample contrasts: 10")
print("Fully LOYO-stable contrasts: 9")
print("Significant but not fully stable: 1")
print("Numerical consistency: PASSED")
print("Table selection consistency: PASSED")

print("\nPublication qualifications:")
print("- Main-table stability screening is post-analysis.")
print("- Topic 2 has country-specific semantic content.")
print("- Topic 3 has semantic heterogeneity.")
print("- HAC confidence intervals are pointwise.")
print("- Independent archival validation is incomplete.")
print("- Sampled semantic fit is not corpus-wide prevalence.")

print("\nOriginal statistical estimates modified: NO")
print("Original NMF models modified: NO")
print("Original publication tables overwritten: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31P: TABLES READY FOR FINAL QC")



### Step 12.27H.31Q — Export Publication Tables

**Objective:** Export the validated main and supplementary manuscript tables.

**Outputs**
- Table 1: Nine fully LOYO-stable bilateral contrasts.
- Supplementary Table S1: All 18 bilateral contrasts.
- Supplementary Table S2: Full LOYO robustness diagnostics for all 18 contrasts.
- Excel workbook: Publication tables, full-precision numerical data, and methodological notes.

Files are saved separately from the analytical computation. Existing files are protected against accidental overwriting.

No statistical models are refitted and no original results are modified.


In [ ]:

# # ============================================================
# # STEP 12.27H.31Q
# # EXPORT FINAL PUBLICATION TABLES
# # ============================================================

# required_31q = [
#     "final_table1_31p",
#     "final_table_s1_31p",
#     "final_table_s2_31p",
#     "final_main9_numeric_31p",
#     "final_s1_numeric_31p",
#     "final_s2_numeric_31p",
# ]

# for name in required_31q:
#     assert name in globals(), f"Missing object: {name}"

# assert len(final_table1_31p) == 9
# assert len(final_table_s1_31p) == 18
# assert len(final_table_s2_31p) == 18

# # Locate the project directory.
# current_dir_31q = Path.cwd().resolve()

# project_root_31q = next(
#     (
#         path
#         for path in [
#             current_dir_31q,
#             *current_dir_31q.parents,
#         ]
#         if path.name == "7-masoud"
#     ),
#     current_dir_31q,
# )

# table_output_dir_31q = (
#     project_root_31q / "results" / "tables"
# )

# table_output_dir_31q.mkdir(
#     parents=True,
#     exist_ok=True,
# )

# # ------------------------------------------------------------
# # 1. Define descriptive filenames
# # ------------------------------------------------------------

# csv_files_31q = {
#     "Table_1_Nine_Robust_Bilateral_Contrasts.csv":
#         final_table1_31p,

#     "Supplementary_Table_S1_All_18_Contrasts.csv":
#         final_table_s1_31p,

#     "Supplementary_Table_S2_Full_LOYO_Audit.csv":
#         final_table_s2_31p,
# }

# workbook_path_31q = (
#     table_output_dir_31q
#     / "Manuscript_Tables_T1_S1_S2.xlsx"
# )

# all_paths_31q = [
#     table_output_dir_31q / filename
#     for filename in csv_files_31q
# ] + [workbook_path_31q]

# existing_paths_31q = [
#     path for path in all_paths_31q
#     if path.exists()
# ]

# assert not existing_paths_31q, (
#     "Export file already exists. "
#     "Review existing files before overwriting: "
#     + ", ".join(map(str, existing_paths_31q))
# )

# # ------------------------------------------------------------
# # 2. Export manuscript-formatted CSV tables
# # ------------------------------------------------------------

# for filename, frame in csv_files_31q.items():
#     frame.to_csv(
#         table_output_dir_31q / filename,
#         index=False,
#         encoding="utf-8-sig",
#     )

# # ------------------------------------------------------------
# # 3. Create methodological documentation
# # ------------------------------------------------------------

# method_notes_31q = pd.DataFrame(
#     [
#         (
#             "Analytical scope",
#             "Three countries: China, Iran, and United States."
#         ),
#         (
#             "Historical comparison",
#             "2011-2025 versus 1972-1990."
#         ),
#         (
#             "Effect definition",
#             "Change in the first-minus-second bilateral "
#             "annual topic-share gap, in percentage points."
#         ),
#         (
#             "Statistical inference",
#             "HAC covariance estimation with lag 5."
#         ),
#         (
#             "Confidence intervals",
#             "Pointwise 95% HAC confidence intervals; "
#             "not multiplicity-adjusted intervals."
#         ),
#         (
#             "Multiple testing",
#             "Holm adjustment across 18 bilateral contrasts."
#         ),
#         (
#             "Main table selection",
#             "Nine post-analysis selected results retaining "
#             "effect direction and Holm significance "
#             "across all 54 leave-one-year-out analyses."
#         ),
#         (
#             "Additional significant result",
#             "Iran - United States, Topic 3: "
#             "baseline significant but not fully LOYO-stable."
#         ),
#         (
#             "Topic 2 interpretation",
#             "Country-specific China-related vocabulary."
#         ),
#         (
#             "Topic 3 interpretation",
#             "Regional conflicts and diplomatic settlement; "
#             "semantic heterogeneity requires qualification."
#         ),
#         (
#             "Semantic validation",
#             "Researcher-confirmed screening of 18 selected "
#             "segments; not a corpus-wide prevalence estimate."
#         ),
#         (
#             "Archival qualification",
#             "Independent archival verification remains "
#             "incomplete for two source records."
#         ),
#         (
#             "Numerical precision",
#             "Publication tables contain formatted values; "
#             "numeric worksheets retain underlying precision."
#         ),
#     ],
#     columns=["Item", "Documentation"],
# )

# # ------------------------------------------------------------
# # 4. Export a consolidated Excel workbook
# # ------------------------------------------------------------

# excel_sheets_31q = {
#     "Table 1": final_table1_31p,
#     "Table S1": final_table_s1_31p,
#     "Table S2": final_table_s2_31p,
#     "Numeric T1": final_main9_numeric_31p,
#     "Numeric S1": final_s1_numeric_31p,
#     "Numeric S2": final_s2_numeric_31p,
#     "Method Notes": method_notes_31q,
# }

# excel_saved_31q = False

# try:
#     with pd.ExcelWriter(
#         workbook_path_31q,
#         engine="openpyxl",
#     ) as writer:

#         for sheet_name, frame in excel_sheets_31q.items():

#             frame.to_excel(
#                 writer,
#                 sheet_name=sheet_name,
#                 index=False,
#             )

#             ws = writer.sheets[sheet_name]

#             ws.freeze_panes = "A2"
#             ws.auto_filter.ref = ws.dimensions
#             ws.sheet_view.showGridLines = False

#             for col_index, column_name in enumerate(
#                 frame.columns,
#                 start=1,
#             ):
#                 values_width = (
#                     frame[column_name]
#                     .astype(str)
#                     .map(len)
#                     .max()
#                 )

#                 width = min(
#                     55,
#                     max(
#                         14,
#                         len(str(column_name)) + 3,
#                         int(values_width) + 3,
#                     ),
#                 )

#                 letter = chr(64 + col_index)

#                 ws.column_dimensions[letter].width = width

#     excel_saved_31q = True

# except ImportError:
#     print(
#         "Excel export skipped: openpyxl is unavailable. "
#         "The three CSV files were saved successfully."
#     )

# # ------------------------------------------------------------
# # 5. Verify saved CSV records
# # ------------------------------------------------------------

# for filename, original_frame in csv_files_31q.items():

#     saved_path = table_output_dir_31q / filename

#     assert saved_path.is_file()

#     verified_frame = pd.read_csv(
#         saved_path,
#         dtype=str,
#         keep_default_na=False,
#     )

#     assert len(verified_frame) == len(original_frame)

#     assert (
#         verified_frame.columns.tolist()
#         == original_frame.columns.tolist()
#     )

# # ------------------------------------------------------------
# # 6. Final export report
# # ------------------------------------------------------------

# print("=" * 65)
# print("STEP 12.27H.31Q — PUBLICATION TABLE EXPORT")
# print("=" * 65)

# print("\nProject root:", project_root_31q)
# print("Output directory:", table_output_dir_31q)

# print("\nSAVED CSV FILES")

# for filename, frame in csv_files_31q.items():
#     print(f"  {filename}: {len(frame)} records")

# print("\nExcel workbook saved:", excel_saved_31q)

# if excel_saved_31q:
#     assert workbook_path_31q.is_file()

#     print("Workbook:", workbook_path_31q)
#     print("Worksheets:", len(excel_sheets_31q))

# print("\nCSV integrity verification: PASSED")
# print("Original dataframes modified: NO")
# print("Original statistical results modified: NO")
# print("Original NMF models modified: NO")

# print("\nSTEP 12.27H.31Q: EXPORT COMPLETED")


### Step 12.27H.31W — Preview Figure 4

Generate a preview-only version of the main manuscript contrast figure using the final
Table 1 results (nine fully LOYO-stable bilateral contrasts).

This cell:
1. locates the final Table 1 dataset,
2. standardizes the required columns,
3. reconstructs 95% HAC confidence intervals if needed,
4. generates a publication-style horizontal contrast plot,
5. does **not** save any files.

No statistical results are modified.

In [ ]:
# ============================================================
# STEP 12.27H.31W — PREVIEW FIGURE 4
# ============================================================


print("=" * 65)
print("STEP 12.27H.31W — PREVIEW FIGURE 4")
print("=" * 65)

# ============================================================
# STEP 12.27H.31W — REPLACE ITS SECTIONS 1 AND 2
# Replace from "# 1. Resolve project paths" to just BEFORE
# "# 3. Standardize required columns".
# Keep the rest of Step 31W unchanged.
# ============================================================

required_concepts_31w = {
    "pair": ["pair_label", "Country contrast", "country_contrast"],
    "topic": ["topic_id", "Topic", "topic"],
    "label": ["proposed_semantic_label", "Semantic label", "semantic_label"],
    "effect": ["effect_pp", "Effect (pp)", "effect"],
    "p": ["holm_adjusted_p", "Holm-adjusted p", "holm_p"],
}

assert "final_main9_numeric_31p" in globals(), (
    "Execute Step 31P before plotting."
)
table1_31w = final_main9_numeric_31p.copy()
table1_source_31w = "final_main9_numeric_31p"
table1_31w.columns = [str(c).strip() for c in table1_31w.columns]
assert len(table1_31w) == 9
assert not table1_31w.duplicated(["pair_label", "topic_id"]).any()
assert {"pair_label", "topic_id", "semantic_label", "effect_pp",
        "ci_lower_pp", "ci_upper_pp", "holm_adjusted_p"}.issubset(table1_31w.columns)
print("Figure source:", table1_source_31w)


# ------------------------------------------------------------
# 3. Standardize required columns
# ------------------------------------------------------------

def first_existing_column_31w(df, candidates):
    for col in candidates:
        if col in df.columns:
            return col
    return None

pair_col_31w = first_existing_column_31w(table1_31w, required_concepts_31w["pair"])
topic_col_31w = first_existing_column_31w(table1_31w, required_concepts_31w["topic"])
label_col_31w = first_existing_column_31w(table1_31w, required_concepts_31w["label"])
effect_col_31w = first_existing_column_31w(table1_31w, required_concepts_31w["effect"])
p_col_31w = first_existing_column_31w(table1_31w, required_concepts_31w["p"])

assert all([pair_col_31w, topic_col_31w, label_col_31w, effect_col_31w, p_col_31w]), (
    "One or more required columns are missing."
)

plot_df_31w = pd.DataFrame({
    "pair_label": table1_31w[pair_col_31w].astype(str).str.strip(),
    "topic_id": pd.to_numeric(table1_31w[topic_col_31w], errors="coerce"),
    "semantic_label": table1_31w[label_col_31w].astype(str).str.strip(),
    "effect_pp": pd.to_numeric(table1_31w[effect_col_31w], errors="coerce"),
    "holm_adjusted_p": pd.to_numeric(table1_31w[p_col_31w], errors="coerce"),
})

# ------------------------------------------------------------
# 4. Resolve confidence intervals
# ------------------------------------------------------------

ci_lower_col_31w = first_existing_column_31w(
    table1_31w, ["ci_lower_pp", "CI lower", "ci_lower"]
)
ci_upper_col_31w = first_existing_column_31w(
    table1_31w, ["ci_upper_pp", "CI upper", "ci_upper"]
)
ci_text_col_31w = first_existing_column_31w(
    table1_31w, ["95% HAC CI (pp)", "95% CI", "hac_ci", "ci_text"]
)

if ci_lower_col_31w and ci_upper_col_31w:
    plot_df_31w["ci_lower_pp"] = pd.to_numeric(table1_31w[ci_lower_col_31w], errors="coerce")
    plot_df_31w["ci_upper_pp"] = pd.to_numeric(table1_31w[ci_upper_col_31w], errors="coerce")

elif ci_text_col_31w:
    ci_text_series_31w = table1_31w[ci_text_col_31w].astype(str)

    def parse_ci_31w(text):
        numbers = re.findall(r"[-+]?\d+(?:\.\d+)?", text)
        if len(numbers) >= 2:
            return float(numbers[0]), float(numbers[1])
        return np.nan, np.nan

    parsed_ci_31w = ci_text_series_31w.apply(parse_ci_31w)
    plot_df_31w["ci_lower_pp"] = [x[0] for x in parsed_ci_31w]
    plot_df_31w["ci_upper_pp"] = [x[1] for x in parsed_ci_31w]

else:
    raise AssertionError(
        "No confidence interval columns found. "
        "Expected ci_lower_pp/ci_upper_pp or a CI text column."
    )

# ------------------------------------------------------------
# 5. Final validation
# ------------------------------------------------------------

assert len(plot_df_31w) == 9, f"Expected 9 rows, found {len(plot_df_31w)}."

required_numeric_31w = [
    "topic_id", "effect_pp", "holm_adjusted_p", "ci_lower_pp", "ci_upper_pp"
]
assert plot_df_31w[required_numeric_31w].notna().all().all(), (
    "Missing numeric values detected in plotting dataframe."
)

assert (plot_df_31w["ci_lower_pp"] <= plot_df_31w["effect_pp"]).all()
assert (plot_df_31w["effect_pp"] <= plot_df_31w["ci_upper_pp"]).all()

# ------------------------------------------------------------
# 6. Sort rows in manuscript-friendly order
# ------------------------------------------------------------

pair_order_31w = {
    "Iran - China": 0,
    "Iran – China": 0,
    "Iran - United States": 1,
    "Iran – United States": 1,
    "United States - China": 2,
    "United States – China": 2,
}

plot_df_31w["pair_order"] = plot_df_31w["pair_label"].map(pair_order_31w)
assert plot_df_31w["pair_order"].notna().all(), (
    "Unrecognized pair labels detected."
)

plot_df_31w = (
    plot_df_31w
    .sort_values(["pair_order", "topic_id", "semantic_label"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# 7. Build display labels and p-value text
# ------------------------------------------------------------

def topic_tag_31w(x):
    return f"T{int(x):02d}"

def pretty_p_31w(p):
    if p < 1e-4:
        return "p < 0.0001"
    return f"p = {p:.4f}"

plot_df_31w["display_label"] = (
    plot_df_31w["pair_label"]
    + "  |  "
    + plot_df_31w["topic_id"].apply(topic_tag_31w)
    + ": "
    + plot_df_31w["semantic_label"]
)

plot_df_31w["p_text"] = plot_df_31w["holm_adjusted_p"].apply(pretty_p_31w)

# ------------------------------------------------------------
# 8. Compute plotting coordinates
# ------------------------------------------------------------

n_rows_31w = len(plot_df_31w)
y_positions_31w = np.arange(n_rows_31w)[::-1]

plot_df_31w["y"] = y_positions_31w
plot_df_31w["err_left"] = plot_df_31w["effect_pp"] - plot_df_31w["ci_lower_pp"]
plot_df_31w["err_right"] = plot_df_31w["ci_upper_pp"] - plot_df_31w["effect_pp"]

xmin_31w = float(plot_df_31w["ci_lower_pp"].min())
xmax_31w = float(plot_df_31w["ci_upper_pp"].max())
xrange_31w = xmax_31w - xmin_31w
padding_31w = max(3.0, xrange_31w * 0.15)

x_left_31w = min(-1.0, xmin_31w - padding_31w)
x_right_31w = xmax_31w + padding_31w * 1.8
p_text_x_31w = xmax_31w + padding_31w * 1.05

# ------------------------------------------------------------
# 9. Generate preview figure
# ------------------------------------------------------------

fig_height_31w = max(7.5, n_rows_31w * 0.72 + 2.4)

fig, ax = plt.subplots(figsize=(14, fig_height_31w))

# Use default matplotlib color cycle without manually specifying colors.
unique_topics_31w = sorted(plot_df_31w["topic_id"].unique())

for topic_value in unique_topics_31w:
    sub = plot_df_31w[plot_df_31w["topic_id"] == topic_value]

    ax.errorbar(
        sub["effect_pp"],
        sub["y"],
        xerr=[sub["err_left"], sub["err_right"]],
        fmt="o",
        capsize=6,
        elinewidth=2,
        markersize=10,
        linewidth=0,
    )

ax.axvline(
    0,
    linestyle="--",
    linewidth=1.5,
)

for _, row in plot_df_31w.iterrows():
    ax.text(
        p_text_x_31w,
        row["y"],
        row["p_text"],
        va="center",
        fontsize=11,
    )

ax.set_yticks(plot_df_31w["y"])
ax.set_yticklabels(plot_df_31w["display_label"], fontsize=11)
ax.set_xlabel(
    "Change in between-country topic-share gap (percentage points)",
    fontsize=13
)
ax.set_title(
    "Nine Fully LOYO-Stable Bilateral Topic Contrasts",
    fontsize=18,
    weight="bold",
    pad=18,
)

ax.set_xlim(x_left_31w, x_right_31w)
ax.set_ylim(-1, n_rows_31w)
ax.grid(axis="x", alpha=0.3)

ax.spines["top"].set_visible(False)
ax.spines["right"].set_visible(False)

footnote_31w = (
    "Dots indicate estimated effects; horizontal lines represent 95% HAC confidence intervals.\n"
    "Reported p-values are Holm-adjusted across 18 bilateral contrasts; confidence intervals are pointwise.\n"
    "This preview is generated from the final Table 1 dataset and is not saved in this step."
)

fig.text(
    0.01,
    0.01,
    footnote_31w,
    ha="left",
    va="bottom",
    fontsize=10,
)

plt.tight_layout(rect=[0, 0.06, 1, 0.95])
plt.show()

# ------------------------------------------------------------
# 10. Preview audit table
# ------------------------------------------------------------

preview_audit_31w = plot_df_31w[
    [
        "pair_label",
        "topic_id",
        "semantic_label",
        "effect_pp",
        "ci_lower_pp",
        "ci_upper_pp",
        "holm_adjusted_p",
    ]
].copy()

print("\nPREVIEW AUDIT TABLE")
display(preview_audit_31w)

print("\nFINAL VALIDATION")
print("Source:", table1_source_31w)
print("Rows plotted:", len(plot_df_31w))
print("Figure saved: NO")
print("Original statistical results modified: NO")
print("STEP 12.27H.31W: PREVIEW COMPLETED")



### Step 12.27H.31Y — Final Figure 4 Visual Refinement

Refine the publication figure displaying nine fully LOYO-stable bilateral topic contrasts.

- Preserve all original estimates, confidence intervals, and adjusted p-values.
- Group observations by country pair.
- Clarify the historical comparison and contrast direction.
- Improve labels, spacing, and statistical annotations.
- Display the figure for visual approval without saving.

No statistical computations or source data are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31Y — FIGURE 4 VISUAL REFINEMENT
# PREVIEW ONLY — NO FILES SAVED
# ============================================================

assert "plot_df_31w" in globals()

figure4_data_31y = plot_df_31w.copy()

required_31y = [
    "pair_label",
    "topic_id",
    "semantic_label",
    "effect_pp",
    "ci_lower_pp",
    "ci_upper_pp",
    "holm_adjusted_p",
]

assert figure4_data_31y[required_31y].notna().all().all()
assert len(figure4_data_31y) == 9

expected_31y = {
    ("Iran - China", 1),
    ("Iran - China", 2),
    ("Iran - China", 3),
    ("Iran - China", 7),
    ("Iran - United States", 7),
    ("United States - China", 1),
    ("United States - China", 2),
    ("United States - China", 3),
    ("United States - China", 5),
}

actual_31y = set(
    figure4_data_31y[
        ["pair_label", "topic_id"]
    ].itertuples(index=False, name=None)
)

assert actual_31y == expected_31y
assert figure4_data_31y["holm_adjusted_p"].lt(0.05).all()

pair_order_31y = [
    "Iran - China",
    "Iran - United States",
    "United States - China",
]

figure4_data_31y["pair_label"] = pd.Categorical(
    figure4_data_31y["pair_label"],
    categories=pair_order_31y,
    ordered=True,
)

figure4_data_31y = (
    figure4_data_31y
    .sort_values(["pair_label", "topic_id"])
    .reset_index(drop=True)
)

# ------------------------------------------------------------
# Construct country-group positions
# ------------------------------------------------------------

cursor_31y = 11.0
row_positions_31y = []
group_headers_31y = []

for pair in pair_order_31y:

    indices = figure4_data_31y.index[
        figure4_data_31y["pair_label"] == pair
    ].tolist()

    group_headers_31y.append(
        (pair.replace(" - ", " – "), cursor_31y + 0.78)
    )

    for idx in indices:
        row_positions_31y.append((idx, cursor_31y))
        cursor_31y -= 1.0

    cursor_31y -= 1.10

position_map_31y = dict(row_positions_31y)
figure4_data_31y["y"] = (
    figure4_data_31y.index.map(position_map_31y)
)

# ------------------------------------------------------------
# Colors: consistent topic encoding
# ------------------------------------------------------------

topic_ids_31y = sorted(
    figure4_data_31y["topic_id"].unique()
)

palette_31y = plt.rcParams[
    "axes.prop_cycle"
].by_key()["color"]

topic_colors_31y = {
    topic: palette_31y[i]
    for i, topic in enumerate(topic_ids_31y)
}

# ------------------------------------------------------------
# Figure
# ------------------------------------------------------------

fig_31y, ax_31y = plt.subplots(
    figsize=(13.5, 8.8)
)

for row in figure4_data_31y.itertuples():

    left_error = row.effect_pp - row.ci_lower_pp
    right_error = row.ci_upper_pp - row.effect_pp

    ax_31y.errorbar(
        row.effect_pp,
        row.y,
        xerr=np.array([
            [left_error],
            [right_error],
        ]),
        fmt="o",
        markersize=8,
        capsize=5,
        capthick=1.7,
        elinewidth=2.2,
        color=topic_colors_31y[row.topic_id],
        zorder=3,
    )

# Neutral zero-reference line
ax_31y.axvline(
    0,
    linestyle="--",
    linewidth=1.2,
    color="0.35",
    zorder=1,
)

# Topic names, without repeated country names
ax_31y.set_yticks(
    figure4_data_31y["y"].to_numpy()
)

ax_31y.set_yticklabels(
    [
        f"T{int(row.topic_id):02d}  {row.semantic_label}"
        for row in figure4_data_31y.itertuples()
    ],
    fontsize=10,
)

ax_31y.tick_params(
    axis="y",
    length=0,
    pad=10,
)

# Country-pair headings
for pair_name, header_y in group_headers_31y:

    ax_31y.text(
        -0.02,
        header_y,
        pair_name,
        transform=ax_31y.get_yaxis_transform(),
        ha="right",
        va="center",
        fontsize=11,
        fontweight="bold",
        color="0.18",
    )

# Adjusted p-value column
ax_31y.text(
    1.025,
    group_headers_31y[0][1] + 0.55,
    "Holm-adjusted p",
    transform=ax_31y.get_yaxis_transform(),
    ha="left",
    va="center",
    fontsize=10,
    fontweight="bold",
)

for row in figure4_data_31y.itertuples():

    p_text = (
        f"{row.holm_adjusted_p:.2e}"
        if row.holm_adjusted_p < 0.001
        else f"{row.holm_adjusted_p:.4f}"
    )

    ax_31y.text(
        1.025,
        row.y,
        p_text,
        transform=ax_31y.get_yaxis_transform(),
        ha="left",
        va="center",
        fontsize=10,
    )

# ------------------------------------------------------------
# Axes, title and layout
# ------------------------------------------------------------

lower_31y = figure4_data_31y["ci_lower_pp"].min()
upper_31y = figure4_data_31y["ci_upper_pp"].max()

ax_31y.set_xlim(
    min(-48, lower_31y - 3),
    max(49, upper_31y + 3),
)

ax_31y.set_ylim(
    figure4_data_31y["y"].min() - 0.75,
    group_headers_31y[0][1] + 1.0,
)

ax_31y.set_xlabel(
    "Change in bilateral topic-share gap (percentage points)",
    fontsize=11,
    labelpad=13,
)

ax_31y.set_title(
    "Historical Changes in Bilateral Topic Gaps",
    fontsize=16,
    fontweight="bold",
    pad=22,
)

ax_31y.grid(
    axis="x",
    alpha=0.18,
    linewidth=0.8,
)

ax_31y.set_axisbelow(True)

ax_31y.spines["top"].set_visible(False)
ax_31y.spines["right"].set_visible(False)
ax_31y.spines["left"].set_visible(False)

fig_31y.subplots_adjust(
    left=0.43,
    right=0.83,
    top=0.89,
    bottom=0.20,
)

# ------------------------------------------------------------
# Scientific notes
# ------------------------------------------------------------

fig_31y.text(
    0.035,
    0.125,
    "Effects represent the 2011–2025 minus 1972–1990 "
    "change in the first-country minus second-country topic gap.",
    fontsize=9.5,
    ha="left",
)

fig_31y.text(
    0.035,
    0.092,
    "Points are estimates; horizontal bars are 95% "
    "pointwise HAC confidence intervals.",
    fontsize=9.5,
    ha="left",
)

fig_31y.text(
    0.035,
    0.059,
    "Holm correction covers 18 contrasts; confidence "
    "intervals are not multiplicity-adjusted. "
    "Nine contrasts passed post hoc LOYO stability screening.",
    fontsize=9.5,
    ha="left",
)

plt.show()

print("=" * 65)
print("FIGURE 4 — REFINED VISUAL PREVIEW")
print("=" * 65)
print("Contrasts:", len(figure4_data_31y))
print("Country pairs:", 3)
print("Unique topics:", len(topic_ids_31y))
print("Original numerical results modified: NO")
print("Figure saved: NO")
print("Visual approval: PENDING")



### Step 12.27H.31Z — Export Final Figure 4

Export the visually approved Figure 4 showing nine fully LOYO-stable bilateral topic contrasts.

Outputs:
- Vector PDF for manuscript submission.
- High-resolution 600 DPI PNG for visual inspection and publication.

The original fitted models, statistical estimates, and publication tables remain unchanged. Existing figure files are protected from accidental overwriting.


In [ ]:

# # ============================================================
# # STEP 12.27H.31Z — FINAL FIGURE 4 EXPORT
# # ============================================================

# assert "fig_31y" in globals(), (
#     "Run Step 12.27H.31Y before exporting."
# )

# assert "figure4_data_31y" in globals()

# assert len(figure4_data_31y) == 9

# assert (
#     figure4_data_31y["holm_adjusted_p"] < 0.05
# ).all()

# assert (
#     figure4_data_31y["ci_lower_pp"]
#     <= figure4_data_31y["effect_pp"]
# ).all()

# assert (
#     figure4_data_31y["effect_pp"]
#     <= figure4_data_31y["ci_upper_pp"]
# ).all()

# # ------------------------------------------------------------
# # 1. Define final publication directory
# # ------------------------------------------------------------

# project_root_31z = Path(
#     globals().get(
#         "root_31w",
#         Path.home() / "Projects" / "7-masoud"
#     )
# ).resolve()

# figure_dir_31z = (
#     project_root_31z
#     / "results"
#     / "publication"
#     / "figures"
# )

# figure_dir_31z.mkdir(
#     parents=True,
#     exist_ok=True
# )

# # ------------------------------------------------------------
# # 2. Define publication filenames
# # ------------------------------------------------------------

# figure_stem_31z = (
#     "Figure_04_Nine_LOYO_Stable_Bilateral_Topic_Contrasts"
# )

# pdf_path_31z = (
#     figure_dir_31z / f"{figure_stem_31z}.pdf"
# )

# png_path_31z = (
#     figure_dir_31z / f"{figure_stem_31z}.png"
# )

# # Protect previously exported files.
# assert not pdf_path_31z.exists(), (
#     f"PDF already exists: {pdf_path_31z}"
# )

# assert not png_path_31z.exists(), (
#     f"PNG already exists: {png_path_31z}"
# )

# # ------------------------------------------------------------
# # 3. Export approved figure
# # ------------------------------------------------------------

# try:
#     fig_31y.savefig(
#         pdf_path_31z,
#         format="pdf",
#         bbox_inches="tight",
#         pad_inches=0.25,
#         facecolor="white",
#     )

#     fig_31y.savefig(
#         png_path_31z,
#         format="png",
#         dpi=600,
#         bbox_inches="tight",
#         pad_inches=0.25,
#         facecolor="white",
#     )

#     # --------------------------------------------------------
#     # 4. Verify exported files
#     # --------------------------------------------------------

#     assert pdf_path_31z.is_file()
#     assert png_path_31z.is_file()

#     assert (
#         pdf_path_31z.read_bytes()[:4] == b"%PDF"
#     ), "Invalid PDF signature."

#     assert (
#         png_path_31z.read_bytes()[:8]
#         == b"\x89PNG\r\n\x1a\n"
#     ), "Invalid PNG signature."

#     assert pdf_path_31z.stat().st_size > 1000
#     assert png_path_31z.stat().st_size > 1000

# except Exception:
#     print(
#         "Export validation failed. Inspect the generated "
#         "files before rerunning."
#     )
#     raise

# # ------------------------------------------------------------
# # 5. Final report
# # ------------------------------------------------------------

# print("=" * 65)
# print("STEP 12.27H.31Z — FINAL FIGURE 4 EXPORT")
# print("=" * 65)

# print("\nOUTPUT DIRECTORY")
# print(figure_dir_31z)

# print("\nSAVED FILES")

# for path in [pdf_path_31z, png_path_31z]:
#     size_mb = path.stat().st_size / (1024 ** 2)

#     print(
#         f"{path.name} | "
#         f"{size_mb:.3f} MB | VERIFIED"
#     )

# print("\nSCIENTIFIC VALIDATION")
# print("Number of contrasts:", len(figure4_data_31y))
# print("Statistically significant:", 9)
# print("LOYO-stable contrasts:", 9)
# print("Statistical estimates modified: NO")
# print("NMF models modified: NO")
# print("Existing publication tables modified: NO")

# print("\nFIGURE 4 EXPORT: COMPLETED")



### Step 12.27H.31AA — Supplementary Figure S1 Numerical Audit

Independently reproduce the moving five-year block deletion sensitivity analysis for Iran versus the United States, Topic 7.

Objectives:
1. Reconstruct all 18 bilateral contrasts from the validated annual topic shares.
2. Exclude each possible consecutive five-year window.
3. Re-estimate historical effects using the original HAC specification.
4. Apply Holm correction across the same 18 contrasts for each window.
5. Identify windows that do not retain adjusted statistical significance.
6. Compare the results with the existing Supplementary Figure S1.

This step performs diagnostics only. No files are saved or overwritten.


In [ ]:

# ============================================================
# STEP 12.27H.31AA — SUPPLEMENTARY FIGURE S1 AUDIT
# FIVE-YEAR MOVING BLOCK DELETION
# NO FILES SAVED
# ============================================================

print("=" * 65)
print("STEP 12.27H.31AA — SUPPLEMENTARY FIGURE S1 AUDIT")
print("=" * 65)

# ------------------------------------------------------------
# 1. Validate existing analysis dependencies
# ------------------------------------------------------------

required_objects_31aa = [
    "loyo_annual_df",
    "estimate_historical_effect",
    "multipletests",
]

missing_31aa = [
    name for name in required_objects_31aa
    if name not in globals()
]

assert not missing_31aa, (
    "Required earlier analysis objects are missing: "
    + ", ".join(missing_31aa)
)

required_columns_31aa = {
    "country_code",
    "year",
    "topic_id",
    "topic_share",
}

assert required_columns_31aa.issubset(
    loyo_annual_df.columns
)

assert int(HAC_LAG_18) == 5

# ------------------------------------------------------------
# 2. Reconstruct annual topic gaps
# ------------------------------------------------------------

years_31aa = np.arange(1972, 2026)

topics_31aa = [1, 2, 3, 4, 5, 7]

pairs_31aa = {
    "Iran - China": ("IRN", "CHN"),
    "Iran - United States": ("IRN", "USA"),
    "United States - China": ("USA", "CHN"),
}

source_31aa = loyo_annual_df.loc[
    loyo_annual_df["year"].isin(years_31aa)
    & loyo_annual_df["topic_id"].isin(topics_31aa)
    & loyo_annual_df["country_code"].isin(
        ["IRN", "USA", "CHN"]
    )
].copy()

assert not source_31aa.duplicated(
    ["country_code", "year", "topic_id"]
).any()

assert len(source_31aa) == 54 * 3 * 6

assert source_31aa["topic_share"].notna().all()

annual_matrix_31aa = source_31aa.pivot(
    index=["country_code", "year"],
    columns="topic_id",
    values="topic_share",
)

gap_series_31aa = {}

for pair_label, (first, second) in pairs_31aa.items():

    for topic_id in topics_31aa:

        first_series = annual_matrix_31aa.loc[
            first, topic_id
        ].reindex(years_31aa)

        second_series = annual_matrix_31aa.loc[
            second, topic_id
        ].reindex(years_31aa)

        gap = first_series - second_series

        assert gap.notna().all()

        gap_series_31aa[(pair_label, topic_id)] = gap

assert len(gap_series_31aa) == 18

# ------------------------------------------------------------
# 3. Reproduce the full-sample baseline
# ------------------------------------------------------------

def fit_contrast_family_31aa(excluded_years=None):

    records = []

    for (pair_label, topic_id), gap in (
        gap_series_31aa.items()
    ):

        if excluded_years is None:
            selected = gap
        else:
            selected = gap.drop(
                index=list(excluded_years)
            )

        fit = estimate_historical_effect(
            selected.index.to_numpy(),
            selected.to_numpy(),
        )

        records.append({
            "pair_label": pair_label,
            "topic_id": topic_id,
            "n_years": len(selected),
            **fit,
        })

    result = pd.DataFrame(records)

    assert len(result) == 18
    assert result["p_value"].notna().all()

    result["holm_adjusted_p"] = multipletests(
        result["p_value"].to_numpy(),
        method="holm",
    )[1]

    result["holm_significant"] = (
        result["holm_adjusted_p"] < 0.05
    )

    return result


baseline_31aa = fit_contrast_family_31aa()

target_pair_31aa = "Iran - United States"
target_topic_31aa = 7

target_baseline_31aa = baseline_31aa.loc[
    (baseline_31aa["pair_label"] == target_pair_31aa)
    & (baseline_31aa["topic_id"] == target_topic_31aa)
].iloc[0]

assert np.isclose(
    target_baseline_31aa["effect_pp"],
    10.213846,
    atol=0.05,
), "Baseline effect differs from the validated manuscript result."

assert np.isclose(
    target_baseline_31aa["holm_adjusted_p"],
    0.00017905,
    atol=0.00001,
), "Baseline adjusted p-value was not reproduced."

print("\nFULL-SAMPLE BASELINE")
print(
    "Effect (pp):",
    round(target_baseline_31aa["effect_pp"], 6)
)
print(
    "Holm-adjusted p:",
    target_baseline_31aa["holm_adjusted_p"]
)
print("Baseline reproduction: PASSED")

# ------------------------------------------------------------
# 4. Remove consecutive five-year blocks
# ------------------------------------------------------------

block_records_31aa = []

for start_year in range(1972, 2022):

    excluded_years = np.arange(
        start_year,
        start_year + 5
    )

    family = fit_contrast_family_31aa(
        excluded_years=excluded_years
    )

    family["block_start"] = start_year
    family["block_end"] = start_year + 4

    block_records_31aa.append(family)

block_results_31aa = pd.concat(
    block_records_31aa,
    ignore_index=True,
)

assert len(block_results_31aa) == 50 * 18
assert block_results_31aa["n_years"].eq(49).all()

# ------------------------------------------------------------
# 5. Extract the Iran-US Topic 7 series
# ------------------------------------------------------------

s1_audit_31aa = block_results_31aa.loc[
    (
        block_results_31aa["pair_label"]
        == target_pair_31aa
    )
    & (
        block_results_31aa["topic_id"]
        == target_topic_31aa
    )
].sort_values("block_start").reset_index(drop=True)

assert len(s1_audit_31aa) == 50

s1_audit_31aa["effect_shift_pp"] = (
    s1_audit_31aa["effect_pp"]
    - target_baseline_31aa["effect_pp"]
)

s1_audit_31aa["same_direction"] = (
    np.sign(s1_audit_31aa["effect_pp"])
    == np.sign(target_baseline_31aa["effect_pp"])
)

non_significant_31aa = s1_audit_31aa.loc[
    ~s1_audit_31aa["holm_significant"]
].copy()

print("\n" + "=" * 65)
print("MOVING FIVE-YEAR BLOCK DELETION")
print("=" * 65)

print("Deletion windows:", len(s1_audit_31aa))
print("Years removed per window:", 5)
print("Remaining years per fit:", 49)
print("Total fitted contrasts:", len(block_results_31aa))

print(
    "Windows retaining effect direction:",
    int(s1_audit_31aa["same_direction"].sum())
)

print(
    "Windows retaining Holm significance:",
    int(s1_audit_31aa["holm_significant"].sum())
)

print(
    "Windows losing Holm significance:",
    len(non_significant_31aa)
)

print("\nNON-SIGNIFICANT DELETION WINDOWS")

display(
    non_significant_31aa[
        [
            "block_start",
            "block_end",
            "effect_pp",
            "ci_lower_pp",
            "ci_upper_pp",
            "holm_adjusted_p",
        ]
    ].round(6)
)

# ------------------------------------------------------------
# 6. Identify the most influential window
# ------------------------------------------------------------

most_influential_31aa = s1_audit_31aa.loc[
    s1_audit_31aa["effect_shift_pp"].abs().idxmax()
]

print("\nMOST INFLUENTIAL WINDOW")
print(
    "Deleted years:",
    int(most_influential_31aa["block_start"]),
    "-",
    int(most_influential_31aa["block_end"])
)
print(
    "Absolute effect shift (pp):",
    round(
        abs(most_influential_31aa["effect_shift_pp"]),
        5
    )
)

# ------------------------------------------------------------
# 7. Verify the existing publication figure
# ------------------------------------------------------------

root_31aa = Path(
    globals().get(
        "project_root_31z",
        Path.home() / "Projects" / "7-masoud"
    )
).resolve()

figure_dir_31aa = (
    root_31aa
    / "results"
    / "publication"
    / "figures"
)

stem_31aa = "Figure_S1_Moving_Block_Deletion_Iran_US_T07"

print("\nEXISTING FIGURE FILES")

for extension in [".pdf", ".png", ".tiff"]:
    path = figure_dir_31aa / f"{stem_31aa}{extension}"

    print(
        f"{path.name}:",
        "FOUND" if path.is_file() else "NOT FOUND"
    )

# ------------------------------------------------------------
# 8. Final scientific diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL SCIENTIFIC AUDIT")
print("=" * 65)

print("Original baseline reproduced: YES")
print("Holm adjustment family: 18 contrasts")
print("Moving-block windows evaluated: 50")

print(
    "Direction retained:",
    int(s1_audit_31aa["same_direction"].sum()),
    "/ 50"
)

print(
    "Holm significance retained:",
    int(s1_audit_31aa["holm_significant"].sum()),
    "/ 50"
)

print(
    "Potential discrepancy with previous S1 preview:",
    "NO" if len(non_significant_31aa) == 2
    else "REQUIRES REVIEW"
)

print("\nOriginal statistical results modified: NO")
print("Original NMF models modified: NO")
print("Existing figure files modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31AA: DIAGNOSTIC COMPLETED")



### Step 12.27H.31AB — Speech Length Distribution Audit

Verify the underlying speech-length data for Supplementary Figure S2.

The audit examines:
- Country-level speech counts and historical coverage.
- Median, interquartile range, and minimum/maximum speech lengths.
- Availability of the existing publication figure.

This is a descriptive quality-control step. It does not reproduce the previously saved plot or establish numerical equivalence with its original plotting dataset.

No statistical models are refitted and no files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.31AB — SPEECH LENGTH DISTRIBUTION AUDIT
# ============================================================

print("=" * 65)
print("STEP 12.27H.31AB — SPEECH LENGTH AUDIT")
print("=" * 65)

assert "df_nlp" in globals()

source_31ab = df_nlp.copy()

assert {"country_code", "year"}.issubset(
    source_31ab.columns
)

countries_31ab = ["IRN", "USA", "CHN"]

source_31ab = source_31ab.loc[
    source_31ab["country_code"].isin(countries_31ab)
].copy()

# Identify the available speech-length measurement.
word_candidates_31ab = [
    "word_count",
    "clean_word_count_check",
    "raw_word_count_check",
]

word_col_31ab = next(
    (
        col for col in word_candidates_31ab
        if col in source_31ab.columns
    ),
    None,
)

assert word_col_31ab is not None, (
    "No previously calculated word-count column found. "
    "Inspect df_nlp.columns before proceeding."
)

source_31ab["speech_words_31ab"] = pd.to_numeric(
    source_31ab[word_col_31ab],
    errors="coerce",
)

source_31ab["year"] = pd.to_numeric(
    source_31ab["year"],
    errors="coerce",
)

assert source_31ab["year"].notna().all()

print("\nWORD COUNT SOURCE:", word_col_31ab)

# ------------------------------------------------------------
# 1. Country-level descriptive statistics
# ------------------------------------------------------------

summary_31ab = (
    source_31ab
    .groupby("country_code")["speech_words_31ab"]
    .agg(
        n_speeches="count",
        mean_words="mean",
        median_words="median",
        minimum_words="min",
        maximum_words="max",
    )
)

quartiles_31ab = (
    source_31ab
    .groupby("country_code")["speech_words_31ab"]
    .quantile([0.25, 0.75])
    .unstack()
)

quartiles_31ab.columns = ["q1_words", "q3_words"]

summary_31ab = summary_31ab.join(
    quartiles_31ab
)

summary_31ab["IQR_words"] = (
    summary_31ab["q3_words"]
    - summary_31ab["q1_words"]
)

print("\nSPEECH LENGTH DESCRIPTIVE STATISTICS")

display(
    summary_31ab.round(2)
)

# ------------------------------------------------------------
# 2. Historical coverage
# ------------------------------------------------------------

coverage_31ab = (
    source_31ab
    .groupby("country_code")
    .agg(
        first_year=("year", "min"),
        last_year=("year", "max"),
        n_records=("year", "size"),
        n_unique_years=("year", "nunique"),
    )
)

print("\nCOUNTRY HISTORICAL COVERAGE")

display(coverage_31ab)

# ------------------------------------------------------------
# 3. Missing and invalid measurements
# ------------------------------------------------------------

invalid_31ab = (
    source_31ab["speech_words_31ab"].isna()
    | (source_31ab["speech_words_31ab"] <= 0)
)

print("\nDATA QUALITY")
print("Total source records:", len(source_31ab))
print("Invalid or missing word counts:", int(invalid_31ab.sum()))

# ------------------------------------------------------------
# 4. Verify existing figure files
# ------------------------------------------------------------

project_31ab = Path(
    globals().get(
        "project_root_31z",
        Path.home() / "Projects" / "7-masoud"
    )
).resolve()

figure_dir_31ab = (
    project_31ab
    / "results"
    / "publication"
    / "figures"
)

stem_31ab = "Figure_S2_Speech_Length_Distribution"

print("\nPUBLICATION FIGURE FILES")

for ext in [".pdf", ".png"]:
    path = figure_dir_31ab / f"{stem_31ab}{ext}"

    print(
        path.name,
        "FOUND" if path.is_file() else "NOT FOUND"
    )

# ------------------------------------------------------------
# 5. Final diagnostics
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL FIGURE S2 AUDIT")
print("=" * 65)

print("Countries:", source_31ab["country_code"].nunique())
print("Source word-count field:", word_col_31ab)
print("Speech length summary: COMPUTED")
print("Historical coverage: COMPUTED")
print("Original figure/data numerical equivalence: NOT YET VERIFIED")
print("Original statistical results modified: NO")
print("Existing figure files modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31AB: DIAGNOSTIC COMPLETED")



### Step 12.27H.31AD — Final Publication Table Audit

**Objective:** Verify the existence, dimensions, column structure, and basic integrity of the final manuscript tables.

**Tables examined:**
- Table 1: Nine LOYO-stable bilateral contrasts.
- Supplementary Table S1: All 18 bilateral contrasts.
- Supplementary Table S2: Complete LOYO robustness audit.

This step is read-only. No statistical estimates, models, or existing publication files are modified.


In [ ]:

# ============================================================
# STEP 12.27H.31AD — FINAL PUBLICATION TABLE AUDIT
# ============================================================

project_31ad = Path.home() / "Projects" / "7-masoud"

table_dir_31ad = (
    project_31ad / "results" / "publication" / "tables"
)

files_31ad = {
    "Table 1": (
        "Table_1_Nine_Robust_Bilateral_Contrasts.csv",
        9,
    ),
    "Supplementary Table S1": (
        "Supplementary_Table_S1_All_18_Contrasts.csv",
        18,
    ),
    "Supplementary Table S2": (
        "Supplementary_Table_S2_Full_LOYO_Audit.csv",
        18,
    ),
}

print("=" * 65)
print("STEP 12.27H.31AD — PUBLICATION TABLE AUDIT")
print("=" * 65)

tables_31ad = {}

for label, (filename, expected_rows) in files_31ad.items():

    path = table_dir_31ad / filename

    assert path.is_file(), f"Missing file: {path}"

    frame = pd.read_csv(path)

    tables_31ad[label] = frame

    print("\n" + "=" * 65)
    print(label.upper())
    print("=" * 65)

    print("File:", filename)
    print("Rows:", len(frame))
    print("Columns:", len(frame.columns))
    print("Expected rows:", expected_rows)
    print("Column names:", frame.columns.tolist())

    assert len(frame) == expected_rows, (
        f"Unexpected record count in {label}"
    )

    assert frame.columns.is_unique, (
        f"Duplicate column names in {label}"
    )

    print("\nPREVIEW")
    display(frame.head(5))

    print("Basic structural validation: PASSED")

# ------------------------------------------------------------
# Excel workbook availability
# ------------------------------------------------------------

workbook_31ad = (
    table_dir_31ad / "Manuscript_Tables_T1_S1_S2.xlsx"
)

print("\n" + "=" * 65)
print("EXCEL WORKBOOK")
print("=" * 65)

print("Workbook exists:", workbook_31ad.is_file())

if workbook_31ad.is_file():
    workbook_info_31ad = pd.ExcelFile(workbook_31ad)

    print("Worksheets:")
    for sheet in workbook_info_31ad.sheet_names:
        print(" -", sheet)

# ------------------------------------------------------------
# Final validation
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FINAL DIAGNOSTICS")
print("=" * 65)

print("Table 1: VERIFIED — 9 rows")
print("Supplementary Table S1: VERIFIED — 18 rows")
print("Supplementary Table S2: VERIFIED — 18 rows")
print("Detailed numerical reconciliation: PENDING REVIEW")
print("Original statistical results modified: NO")
print("Files saved: NONE")

print("\nSTEP 12.27H.31AD: AUDIT COMPLETED")



### Step 12.27H.31AE — Corpus Characteristics and Historical Coverage

**Objective:** Construct Supplementary Table S4 using the verified speech-level source data.

This table summarizes:
- Number of speeches and observed years for each country.
- Historical coverage from 1946 to 2025.
- Availability of speeches during the 1972–2025 inferential window.
- Speech length distributions.

The output distinguishes corpus-level speech coverage from the annual observations used in the inferential models.

This step generates a publication-ready table preview and preserves full-precision diagnostic data. No files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.31AE — CORPUS CHARACTERISTICS
# SUPPLEMENTARY TABLE S4
# ============================================================

assert "df_nlp" in globals(), "df_nlp is unavailable."

required_31ae = {
    "country_code", "year", "word_count"
}

assert required_31ae.issubset(df_nlp.columns), (
    f"Missing columns: {required_31ae - set(df_nlp.columns)}"
)

country_names_31ae = {
    "IRN": "Iran",
    "USA": "United States",
    "CHN": "China",
}

source_31ae = df_nlp.loc[
    df_nlp["country_code"].isin(country_names_31ae),
    ["country_code", "year", "word_count"]
].copy()

source_31ae["year"] = pd.to_numeric(
    source_31ae["year"], errors="raise"
).astype(int)

source_31ae["word_count"] = pd.to_numeric(
    source_31ae["word_count"], errors="raise"
)

assert source_31ae["word_count"].notna().all()
assert source_31ae["word_count"].gt(0).all()

assert source_31ae["year"].between(1946, 2025).all()

assert not source_31ae.duplicated(
    ["country_code", "year"]
).any(), "Duplicate country-year observations detected."

expected_31ae = {
    "IRN": 74,
    "USA": 80,
    "CHN": 79,
}

observed_31ae = (
    source_31ae.groupby("country_code").size().to_dict()
)

assert observed_31ae == expected_31ae, (
    f"Unexpected country counts: {observed_31ae}"
)

records_31ae = []
coverage_records_31ae = []

for code in ["IRN", "USA", "CHN"]:

    group = source_31ae.loc[
        source_31ae["country_code"] == code
    ].copy()

    words = group["word_count"]
    years = set(group["year"].astype(int))

    inferential_years = set(range(1972, 2026))
    covered = sorted(years.intersection(inferential_years))
    missing = sorted(inferential_years - years)

    q1 = float(words.quantile(0.25))
    q3 = float(words.quantile(0.75))
    median = float(words.median())

    records_31ae.append({
        "Country": country_names_31ae[code],
        "Speeches (n)": len(group),
        "Observed years (n)": len(years),
        "First-last year":
            f"{min(years)}-{max(years)}",
        "Observed years (1972-2025)":
            f"{len(covered)}/54",
        "Mean words": f"{words.mean():,.1f}",
        "Median [Q1-Q3]":
            f"{median:,.1f} [{q1:,.1f}-{q3:,.1f}]",
        "Range (words)":
            f"{int(words.min()):,}-{int(words.max()):,}",
    })

    coverage_records_31ae.append({
        "country_code": code,
        "n_speeches": len(group),
        "observed_inferential_years": len(covered),
        "missing_inferential_years": len(missing),
        "missing_year_list": missing,
        "mean_words_unrounded": float(words.mean()),
        "median_words_unrounded": median,
        "q1_words": q1,
        "q3_words": q3,
    })

supp_table_s4 = pd.DataFrame(records_31ae)
coverage_audit_s4 = pd.DataFrame(coverage_records_31ae)

assert len(supp_table_s4) == 3
assert len(source_31ae) == 233

print("=" * 65)
print("SUPPLEMENTARY TABLE S4")
print("CORPUS CHARACTERISTICS AND HISTORICAL COVERAGE")
print("=" * 65)

display(supp_table_s4)

print("\n" + "=" * 65)
print("HISTORICAL COVERAGE DIAGNOSTICS")
print("=" * 65)

display(coverage_audit_s4)

print("\nTotal speeches:", len(source_31ae))
print("Total speech words:", int(source_31ae["word_count"].sum()))

print("\nVALIDATION")
print("Source counts: PASSED")
print("Country-year uniqueness: PASSED")
print("Speech lengths: PASSED")
print("Inferential coverage: DESCRIPTIVE AUDIT COMPLETED")
print("Original data modified: NO")
print("Files saved: NONE")



### Step 12.27H.31AF — Reference NMF Topic Inventory

**Objective:** Construct Supplementary Table S5 directly from the verified reference NMF model.

The table includes all seven components, their interpretive labels, representative vocabulary, inferential inclusion status, and semantic qualifications.

**Reference configuration:** K = 7, random seed = 11.

Topic 6 is reported for transparency but is excluded from the substantive bilateral contrasts because it primarily captures procedural language.

No model is refitted, and no files are saved.


In [ ]:

# ============================================================
# STEP 12.27H.31AF — REFERENCE NMF TOPIC INVENTORY
# SUPPLEMENTARY TABLE S5
# ============================================================

assert "model" in globals()
assert hasattr(model, "components_")

H_31af = np.asarray(model.components_, dtype=float)

assert H_31af.shape[0] == 7
assert H_31af.shape[1] == 18981
assert np.isfinite(H_31af).all()
assert np.all(H_31af >= 0)

assert getattr(model, "random_state", None) == 11, (
    "The active model is not the seed-11 reference."
)

assert "focal_features" in globals(), (
    "The verified reference feature ordering is unavailable."
)

terms_31af = np.asarray(focal_features).astype(str)

assert len(terms_31af) == H_31af.shape[1]
assert len(np.unique(terms_31af)) == len(terms_31af)

# Verify vectorizer agreement when available.
if "focal_vectorizer" in globals():
    vectorizer_terms_31af = np.asarray(
        focal_vectorizer.get_feature_names_out()
    ).astype(str)

    assert np.array_equal(
        terms_31af, vectorizer_terms_31af
    ), "Reference vocabulary ordering mismatch."

topic_labels_31af = {
    1: "Economic development and global cooperation",
    2: "China and peaceful development",
    3: "Regional conflicts and diplomatic settlement",
    4: "Iran and regional security",
    5: "Human rights and freedom discourse",
    6: "UN General Assembly procedural language",
    7: "Nuclear arms and disarmament",
}

topic_cautions_31af = {
    1: "Provisional semantic label",
    2: "Country-specific vocabulary",
    3: "Researcher-approved; heterogeneous regional content",
    4: "Country-specific vocabulary",
    5: "Potential country-rhetoric overlap",
    6: "Procedural and formulaic language",
    7: "Provisional semantic label",
}

records_31af = []

for topic_id in range(1, 8):

    weights = H_31af[topic_id - 1]

    top_indices = np.argsort(weights)[::-1][:12]
    top_terms = terms_31af[top_indices].tolist()

    records_31af.append({
        "Topic": f"T{topic_id:02d}",
        "Semantic label": topic_labels_31af[topic_id],
        "Top 12 representative terms":
            ", ".join(top_terms),
        "Bilateral analysis":
            "Excluded" if topic_id == 6 else "Included",
        "Interpretive qualification":
            topic_cautions_31af[topic_id],
    })

supp_table_s5 = pd.DataFrame(records_31af)

assert len(supp_table_s5) == 7
assert supp_table_s5["Topic"].is_unique
assert supp_table_s5["Bilateral analysis"].eq(
    "Excluded"
).sum() == 1

# Confirm that leading terms match previously inspected
# reference-component anchor vocabulary.
expected_anchors_31af = {
    1: {"developing", "global"},
    2: {"china", "chinese"},
    3: {"settlement", "kampuchea"},
    4: {"iran", "islamic"},
    5: {"human", "freedom"},
    6: {"session", "general assembly"},
    7: {"nuclear", "weapons"},
}

for topic_id, anchors in expected_anchors_31af.items():
    extracted = set(
        terms_31af[
            np.argsort(H_31af[topic_id - 1])[::-1][:12]
        ]
    )

    assert anchors.issubset(extracted), (
        f"Reference anchor mismatch for Topic {topic_id}"
    )

print("=" * 65)
print("SUPPLEMENTARY TABLE S5")
print("REFERENCE NMF TOPIC INVENTORY")
print("=" * 65)

display(supp_table_s5)

print("\n" + "=" * 65)
print("REFERENCE MODEL VALIDATION")
print("=" * 65)

print("Number of topics:", H_31af.shape[0])
print("Number of features:", H_31af.shape[1])
print("Random seed:", model.random_state)
print("Included substantive topics: 6")
print("Excluded procedural topics: 1")
print("Reference vocabulary anchors: PASSED")
print("Original NMF model modified: NO")
print("Statistical results modified: NO")
print("Files saved: NONE")


In [ ]:

# ============================================================
# STEP 12.27H.31AG — BLOCK-DELETION P-VALUE AUDIT
# Read-only search for the original fitted results.
# ============================================================

required_31ag = {
    "block_start",
    "block_end",
    "holm_adjusted_p",
}

matches_31ag = []

for name, obj in list(globals().items()):

    if not isinstance(obj, pd.DataFrame):
        continue

    if not required_31ag.issubset(obj.columns):
        continue

    candidate = obj.loc[
        pd.to_numeric(
            obj["block_start"], errors="coerce"
        ).isin([1976, 1977])
        &
        pd.to_numeric(
            obj["block_end"], errors="coerce"
        ).isin([1980, 1981])
    ].copy()

    if "topic_id" in candidate.columns:
        candidate = candidate.loc[
            pd.to_numeric(
                candidate["topic_id"], errors="coerce"
            ).eq(7)
        ]

    if "pair_label" in candidate.columns:
        candidate = candidate.loc[
            candidate["pair_label"].astype(str).eq(
                "Iran - United States"
            )
        ]

    if len(candidate):
        matches_31ag.append((name, candidate))

print("=" * 65)
print("BLOCK-DELETION SOURCE RECONCILIATION")
print("=" * 65)

if not matches_31ag:
    print("Original block results not identified.")
    print("Supplementary Table S3 audit: PENDING")
else:
    for name, candidate in matches_31ag:
        print("\nSource object:", name)
        display(candidate[
            [
                c for c in [
                    "pair_label", "topic_id",
                    "block_start", "block_end",
                    "effect_pp", "holm_adjusted_p"
                ] if c in candidate.columns
            ]
        ])

print("\nOriginal calculations modified: NO")
print("Files saved: NONE")


In [ ]:

# ============================================================
# STEP 12.27H.31AH — BLOCK-DELETION RECONCILIATION
# ============================================================

keys_31ah = [
    "pair_label", "topic_id", "block_start", "block_end"
]

required_31ah = keys_31ah + ["effect_pp", "holm_adjusted_p"]

for name in ["moving_block_hac_df", "block_results_31aa"]:
    assert name in globals(), f"Missing: {name}"
    assert set(required_31ah).issubset(globals()[name].columns)

original_31ah = moving_block_hac_df.copy()
recomputed_31ah = block_results_31aa.copy()

assert not original_31ah.duplicated(keys_31ah).any()
assert not recomputed_31ah.duplicated(keys_31ah).any()

optional_31ah = ["p_value", "ci_lower_pp", "ci_upper_pp"]

selected_31ah = required_31ah + [
    c for c in optional_31ah
    if c in original_31ah.columns
    and c in recomputed_31ah.columns
]

comparison_31ah = original_31ah[selected_31ah].merge(
    recomputed_31ah[selected_31ah],
    on=keys_31ah,
    suffixes=("_original", "_recomputed"),
    validate="one_to_one",
    how="outer",
    indicator=True
)

assert comparison_31ah["_merge"].eq("both").all()

comparison_31ah["adjusted_p_difference"] = (
    comparison_31ah["holm_adjusted_p_original"]
    - comparison_31ah["holm_adjusted_p_recomputed"]
).abs()

print("=" * 65)
print("FULL BLOCK-DELETION RECONCILIATION")
print("=" * 65)

print("Matched comparisons:", len(comparison_31ah))
print(
    "Maximum adjusted p discrepancy:",
    comparison_31ah["adjusted_p_difference"].max()
)

print("\nLARGEST DISCREPANCIES")
display(
    comparison_31ah.sort_values(
        "adjusted_p_difference", ascending=False
    ).head(12)
)

print("\nIRAN–US TOPIC 7 CRITICAL WINDOWS")
display(
    comparison_31ah.loc[
        comparison_31ah["pair_label"].eq("Iran - United States")
        & comparison_31ah["topic_id"].eq(7)
        & comparison_31ah["block_start"].isin([1976, 1977])
    ].sort_values("block_start")
)

print("\nOriginal models modified: NO")
print("Files saved: NONE")



### Step 12.27H.31AE — Unified Calendar-Aware HAC Reproducibility Audit

**Objective:** Independently reconcile all 54 leave-one-year-out
analyses and all 50 moving five-year block-deletion analyses using
a single calendar-aware HAC estimator.

**Validation:** Compare the nine selected primary contrasts,
the complete LOYO robustness table, and the original five-year
block-deletion results.

**Output:** Diagnostic tables only. No files are saved, and no
original model estimates or fitted NMF objects are modified.


In [ ]:
loyo_frames_31ae = []
for year in YEARS_1220:
    family = fit_family_31ae([int(year)])
    family['excluded_year'] = int(year)
    loyo_frames_31ae.append(family)
loyo_calendar_31ae = pd.concat(loyo_frames_31ae, ignore_index=True)
assert len(loyo_calendar_31ae) == 972
assert loyo_calendar_31ae['n_years'].eq(53).all()

summary_rows_31ae = []
for (pair, topic), group in loyo_calendar_31ae.groupby(keys_31ae, sort=False):
    original = baseline_31ae.loc[
        (baseline_31ae['pair_label'] == pair)
        & (baseline_31ae['topic_id'] == topic), 'effect_pp'
    ].iloc[0]
    max_row = group.loc[(group['effect_pp'] - original).abs().idxmax()]
    n_direction = int((np.sign(group['effect_pp']) == np.sign(original)).sum())
    n_significant = int(group['holm_significant'].sum())
    summary_rows_31ae.append({
        'pair_label': pair,
        'topic_id': int(topic),
        'baseline_effect_pp': float(original),
        'same_direction_years': n_direction,
        'holm_significant_years': n_significant,
        'maximum_absolute_shift_pp': float(abs(max_row['effect_pp'] - original)),
        'most_influential_year': int(max_row['excluded_year']),
        'loyo_stable': n_direction == 54 and n_significant == 54,
    })
loyo_summary_calendar_31ae = pd.DataFrame(summary_rows_31ae).sort_values(keys_31ae).reset_index(drop=True)
assert len(loyo_summary_calendar_31ae) == 18

original_s2_31ae = final_s2_numeric_31p[keys_31ae + [
    'same_direction_years', 'holm_significant_years', 'loyo_stable',
    'maximum_absolute_shift_pp', 'most_influential_year'
]].copy()
compare_s2_31ae = original_s2_31ae.merge(
    loyo_summary_calendar_31ae, on=keys_31ae,
    validate='one_to_one', suffixes=('_existing', '_calendar')
)
for field in ['same_direction_years', 'holm_significant_years',
              'loyo_stable', 'most_influential_year']:
    compare_s2_31ae[f'{field}_changed'] = (
        compare_s2_31ae[f'{field}_existing'].to_numpy()
        != compare_s2_31ae[f'{field}_calendar'].to_numpy()
    )
compare_s2_31ae['max_shift_difference_pp'] = (
    compare_s2_31ae['maximum_absolute_shift_pp_calendar']
    - compare_s2_31ae['maximum_absolute_shift_pp_existing']
)

old_main_keys_31ae = set(map(tuple, final_main9_numeric_31p[keys_31ae].to_numpy()))
new_main_keys_31ae = set(map(tuple, loyo_summary_calendar_31ae.loc[
    loyo_summary_calendar_31ae['loyo_stable'], keys_31ae
].to_numpy()))

# Independently reproduce 50 x 18 calendar-aware five-year deletions.
block_frames_31ae = []
for start in range(1972, 2022):
    family = fit_family_31ae(range(start, start + 5))
    family['block_start'] = start
    family['block_end'] = start + 4
    block_frames_31ae.append(family)
block_calendar_31ae = pd.concat(block_frames_31ae, ignore_index=True)
assert len(block_calendar_31ae) == 900
block_compare_31ae = moving_block_hac_df.merge(
    block_calendar_31ae,
    on=keys_31ae + ['block_start', 'block_end'],
    validate='one_to_one', suffixes=('_existing', '_calendar')
)
block_errors_31ae = {
    field: float((
        block_compare_31ae[f'{field}_existing']
        - block_compare_31ae[f'{field}_calendar']
    ).abs().max())
    for field in ['effect_pp', 'ci_lower_pp', 'ci_upper_pp', 'holm_adjusted_p']
}

critical_31ae = block_calendar_31ae.loc[
    (block_calendar_31ae['pair_label'] == 'Iran - United States')
    & (block_calendar_31ae['topic_id'] == 7)
    & (block_calendar_31ae['block_start'].isin([1976, 1977])),
    ['block_start', 'block_end', 'effect_pp', 'holm_adjusted_p']
].sort_values('block_start')

print('=' * 70)
print('STEP 12.27H.31AE — UNIFIED CALENDAR-AWARE HAC REPRODUCIBILITY')
print('=' * 70)
print('Full-sample maximum discrepancies:', baseline_errors_31ae)
print('Calendar LOYO fits:', len(loyo_calendar_31ae))
print('Existing LOYO-stable results:', len(old_main_keys_31ae))
print('Calendar-aware LOYO-stable results:', len(new_main_keys_31ae))
print('Newly stable:', sorted(new_main_keys_31ae - old_main_keys_31ae))
print('No longer stable:', sorted(old_main_keys_31ae - new_main_keys_31ae))
print('Five-year block maximum discrepancies:', block_errors_31ae)
print('\nLOYO COUNTS / STABILITY CHANGES')
changed_mask_31ae = compare_s2_31ae[
    [f'{f}_changed' for f in [
        'same_direction_years', 'holm_significant_years',
        'loyo_stable', 'most_influential_year'
    ]]
].any(axis=1) | compare_s2_31ae['max_shift_difference_pp'].abs().gt(1e-8)
if changed_mask_31ae.any():
    display(compare_s2_31ae.loc[changed_mask_31ae])
else:
    print('No differences from the existing Table S2.')
print('\nCRITICAL FIVE-YEAR WINDOWS — IRAN/US TOPIC 7')
display(critical_31ae)
print('\nFINAL STATUS')
print('LOYO selection agreement:', 'PASS' if old_main_keys_31ae == new_main_keys_31ae else 'REVIEW REQUIRED')
print('Existing 50-window calendar HAC agreement:', 'PASS' if max(block_errors_31ae.values()) < 1e-8 else 'REVIEW REQUIRED')
print('Original models/data/tables changed: NO')
print('Files saved: NONE')
